# Part 1 — packaged source code

This notebook exposes the Python implementations behind the four reader notebooks, including frozen training, graph/pseudo-label methods, loss functions, calibration, experiment runners, and verification. The packaged files retain their original hashes. `%%writefile` cells write review copies under `_source_export/`; they do not execute every historical experiment or overwrite the frozen resources.

The four finalized notebooks are portable. Historical runners and source notebooks remain provenance: some refer to omitted intermediate models/matrices or historical worktrees and may need the full research checkout or regeneration of those inputs. They are not presented as portable one-click replays. The winner's fresh training and the representative feature experiments have dedicated, tested portable paths in notebooks 01 and 02.
Run with Python 3.12 and the pinned `requirements.txt` beside this notebook. In Colab, extract the complete ZIP, change into `midterm/part1`, and install those dependencies before running. The supplied inputs and selected evidence are included; no Git checkout, private path, external trial data, or model cache is required. Notebook 05 contains the complete packaged Python source.

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

# Works from the extracted part1 directory or from the repository root.
options = [Path.cwd(), Path.cwd() / "submission/midterm/part1", Path.cwd() / "midterm/part1"]
HOME = next((p for p in options if (p / "resources/handin.py").is_file()), None)
if HOME is None:
    raise RuntimeError("Extract the complete Part 1 ZIP and open this notebook beside its resources folder.")
RES = HOME / "resources"
OUT = HOME / "generated"
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(RES))
import handin as h
P1, LABELS, EMB = h.P1, h.LABELS, h.EMB
# Stable numerical behavior; release automatically when the kernel exits.
thread_limit = threadpool_limits(limits=1)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
display(h.verify_resources())

{'files': 763, 'hashes_match': True, 'external_checkout_required': False}

In [2]:
source_names = ['part1/anchored_solution.py', 'part1/experiments/anchored_graph_components_v1/models.py', 'part1/experiments/anchored_graph_components_v1/pseudo_anchored.py', 'part1/experiments/anchored_graph_components_v1/run.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/__init__.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/blocks.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/cache.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/classifiers.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/configs.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/context.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/data.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/evaluate.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/features.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/graph.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/head.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/labelers.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/pipeline.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/search.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/submit.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/synth.py', 'part1/experiments/anchored_graph_components_v1/vendor/p1lib/viz.py', 'part1/experiments/anchored_graph_components_v1/vendor/strict_adapter.py', 'part1/experiments/anchored_graph_components_v1/verify.py', 'part1/experiments/blend_refinement_v1/run.py', 'part1/experiments/blend_refinement_v2/run.py', 'part1/experiments/blend_refinement_v2/verify.py', 'part1/experiments/cand2_followup_v1/analyze.py', 'part1/experiments/cand2_followup_v1/export_blend.py', 'part1/experiments/cand2_followup_v1/run.py', 'part1/experiments/cand2_followup_v1/source/p1lib/__init__.py', 'part1/experiments/cand2_followup_v1/source/p1lib/blocks.py', 'part1/experiments/cand2_followup_v1/source/p1lib/cache.py', 'part1/experiments/cand2_followup_v1/source/p1lib/classifiers.py', 'part1/experiments/cand2_followup_v1/source/p1lib/configs.py', 'part1/experiments/cand2_followup_v1/source/p1lib/context.py', 'part1/experiments/cand2_followup_v1/source/p1lib/data.py', 'part1/experiments/cand2_followup_v1/source/p1lib/evaluate.py', 'part1/experiments/cand2_followup_v1/source/p1lib/features.py', 'part1/experiments/cand2_followup_v1/source/p1lib/graph.py', 'part1/experiments/cand2_followup_v1/source/p1lib/head.py', 'part1/experiments/cand2_followup_v1/source/p1lib/labelers.py', 'part1/experiments/cand2_followup_v1/source/p1lib/pipeline.py', 'part1/experiments/cand2_followup_v1/source/p1lib/search.py', 'part1/experiments/cand2_followup_v1/source/p1lib/submit.py', 'part1/experiments/cand2_followup_v1/source/p1lib/synth.py', 'part1/experiments/cand2_followup_v1/source/p1lib/viz.py', 'part1/experiments/cand2_followup_v1/strict_adapter.py', 'part1/experiments/cand2_followup_v1/verify_retained.py', 'part1/experiments/cand2_submission_v1/build.py', 'part1/experiments/cand2_submission_v1/check_export.py', 'part1/experiments/cand2_submission_v1/source/p1lib/__init__.py', 'part1/experiments/cand2_submission_v1/source/p1lib/blocks.py', 'part1/experiments/cand2_submission_v1/source/p1lib/cache.py', 'part1/experiments/cand2_submission_v1/source/p1lib/classifiers.py', 'part1/experiments/cand2_submission_v1/source/p1lib/configs.py', 'part1/experiments/cand2_submission_v1/source/p1lib/context.py', 'part1/experiments/cand2_submission_v1/source/p1lib/data.py', 'part1/experiments/cand2_submission_v1/source/p1lib/evaluate.py', 'part1/experiments/cand2_submission_v1/source/p1lib/features.py', 'part1/experiments/cand2_submission_v1/source/p1lib/graph.py', 'part1/experiments/cand2_submission_v1/source/p1lib/head.py', 'part1/experiments/cand2_submission_v1/source/p1lib/labelers.py', 'part1/experiments/cand2_submission_v1/source/p1lib/pipeline.py', 'part1/experiments/cand2_submission_v1/source/p1lib/search.py', 'part1/experiments/cand2_submission_v1/source/p1lib/submit.py', 'part1/experiments/cand2_submission_v1/source/p1lib/synth.py', 'part1/experiments/cand2_submission_v1/source/p1lib/viz.py', 'part1/experiments/cand2_submission_v1/strict_adapter.py', 'part1/experiments/current_leaders_grouped_v1/snapshots/anchored_solution.py', 'part1/experiments/current_leaders_grouped_v1/snapshots/compare_current_leaders.py', 'part1/experiments/current_leaders_grouped_v1/snapshots/paired_models.py', 'part1/experiments/derived_benchmark_audit_v1/analyze.py', 'part1/experiments/derived_benchmark_audit_v1/build_report.py', 'part1/experiments/derived_benchmark_audit_v1/source_snapshot/benchmark_run.py', 'part1/experiments/derived_benchmark_audit_v1/source_snapshot/context.py', 'part1/experiments/derived_benchmark_audit_v1/source_snapshot/data.py', 'part1/experiments/derived_benchmark_audit_v1/source_snapshot/evaluate.py', 'part1/experiments/derived_benchmark_audit_v1/study.py', 'part1/experiments/derived_benchmark_audit_v1/verify.py', 'part1/research/anchored_neighbor_blend_v1/check_neighbor_math.py', 'part1/research/anchored_neighbor_blend_v1/evidence.py', 'part1/research/anchored_neighbor_blend_v1/neighbor_models.py', 'part1/research/anchored_neighbor_blend_v1/study.py', 'part1/research/anchored_transductive_v1/models.py', 'part1/research/anchored_transductive_v1/run.py', 'part1/research/anchored_transductive_v1/verify_retained.py', 'part1/research/complementary_heads_v1/check_complement.py', 'part1/research/complementary_heads_v1/complement.py', 'part1/research/complementary_heads_v1/head_study.py', 'part1/research/complementary_heads_v1/retained_evidence.py', 'part1/research/complementary_heads_v1/verify_heads.py', 'part1/research/complementary_heads_v2/check_strong.py', 'part1/research/complementary_heads_v2/evidence.py', 'part1/research/complementary_heads_v2/strong_kernel.py', 'part1/research/complementary_heads_v2/strong_study.py', 'part1/research/complementary_heads_v2/verify_strong.py', 'part1/research/other_combiner_v1/run.py', 'part1/research/other_combiner_v1/study_models.py', 'part1/research/other_combiner_v2/diagnose.py', 'part1/research/other_combiner_v2/retain.py', 'part1/research/other_combiner_v2/run.py', 'part1/research/other_combiner_v2/source/p1lib/__init__.py', 'part1/research/other_combiner_v2/source/p1lib/blocks.py', 'part1/research/other_combiner_v2/source/p1lib/cache.py', 'part1/research/other_combiner_v2/source/p1lib/classifiers.py', 'part1/research/other_combiner_v2/source/p1lib/configs.py', 'part1/research/other_combiner_v2/source/p1lib/context.py', 'part1/research/other_combiner_v2/source/p1lib/data.py', 'part1/research/other_combiner_v2/source/p1lib/evaluate.py', 'part1/research/other_combiner_v2/source/p1lib/features.py', 'part1/research/other_combiner_v2/source/p1lib/graph.py', 'part1/research/other_combiner_v2/source/p1lib/head.py', 'part1/research/other_combiner_v2/source/p1lib/labelers.py', 'part1/research/other_combiner_v2/source/p1lib/pipeline.py', 'part1/research/other_combiner_v2/source/p1lib/search.py', 'part1/research/other_combiner_v2/source/p1lib/submit.py', 'part1/research/other_combiner_v2/source/p1lib/synth.py', 'part1/research/other_combiner_v2/source/p1lib/viz.py', 'part1/research/other_combiner_v2/strict_adapter.py', 'part1/research/other_combiner_v2/study_models.py', 'part1/research/other_combiner_v2/verify.py', 'part1/research/other_combiner_v2/verify_retained.py', 'part1/research/partial_secondary_v1/check_math.py', 'part1/research/partial_secondary_v1/evidence_partial.py', 'part1/research/partial_secondary_v1/partial_model.py', 'part1/research/partial_secondary_v1/study_run.py', 'part1/research/partial_secondary_v1/verify_partial.py', 'part1/research/pysr_blend_submission_v1/export.py', 'part1/research/pysr_blend_submission_v1/verify.py', 'part1/research/pysr_blend_submission_v1/verify_retained.py', 'part1/research/pysr_v1/build_controls.py', 'part1/research/pysr_v1/check_geometry.py', 'part1/research/pysr_v1/common.py', 'part1/research/pysr_v1/model.py', 'part1/research/pysr_v1/retained.py', 'part1/research/pysr_v1/study.py', 'handin.py']
EXPORT=HOME / "_source_export"
for name in source_names:
    (EXPORT/name).parent.mkdir(parents=True,exist_ok=True)
# Relative writefile destinations below are rooted beside this notebook.
import os
os.chdir(HOME)

## part1/anchored_solution.py

In [3]:
%%writefile _source_export/part1/anchored_solution.py
"""Current Ani with three small changes: anchored weights, binary votes, weaker GMM.

Fit receives only permitted rows, in original CSV order. Fully missing labels
are -100; query/test rows must never be passed to fit. Prediction uses only the
150 stopping-reason embedding columns, ordered emb_0 through emb_149.
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, normalize


def select_reasons(scores):
    result=(scores>=.3).astype(int)
    result[np.arange(len(scores)),scores.argmax(axis=1)]=1
    return result


class AnchoredReasonClassifier:
    def fit(self, embeddings, labels):
        X,Y=np.asarray(embeddings,float),np.asarray(labels,int)
        if X.ndim!=2 or X.shape[1]!=150 or Y.shape!=(len(X),5):
            raise ValueError('Expected 150 embeddings and five labels per row.')
        known=np.isin(Y,[0,1]).all(axis=1)
        unknown=(Y==-100).all(axis=1)
        if not (known|unknown).all() or (Y[known].sum(axis=1)==0).any() or (Y[known].sum(axis=0)==0).any():
            raise ValueError('Use complete nonempty known labels or five -100 values; all five reasons must be represented.')
        self.scaler_=StandardScaler().fit(X)
        Z=self.scaler_.transform(X)
        self.pca_=PCA(30,random_state=0).fit(Z)
        self.gmm_=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(self.pca_.transform(Z))
        assert self.gmm_.converged_
        K,V=Z[known],Z[unknown];targets=Y[known]
        Rk=self.gmm_.predict_proba(self.pca_.transform(K))
        Ru=self.gmm_.predict_proba(self.pca_.transform(V))
        counts=(targets/targets.sum(axis=1,keepdims=True)).sum(axis=0)
        self.class_weights_=counts.mean()/counts
        Za,Ya,Ra=K,targets,Rk
        self.round_audit_=[]
        for step in range(3):
            row,reason=np.nonzero(Ya)
            weights=self.class_weights_[reason]/Ya.sum(axis=1)[row]
            weights*=np.where(row<len(K),1.,.25)
            self.logistic_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(Za[row],reason,sample_weight=weights)
            self.lda_=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(5)/5).fit(Za[row],reason)
            self.neighbor_X_=normalize(Za)
            self.neighbor_Y_=Ya.copy()
            self.round_audit_.append(dict(step=step,known=len(K),generated=len(Za)-len(K),
                                          known_weight=float(weights[row<len(K)].sum()),
                                          generated_weight=float(weights[row>=len(K)].sum())))
            if step==2:break
            T=Ya/Ya.sum(axis=1,keepdims=True)
            rates=(Ra.T@T+T.mean(axis=0))/(Ra.sum(axis=0)[:,None]+1)
            scores=.95*self._scores(V)+.05*(Ru@rates)
            self.generated_mask_=scores.max(axis=1)>=.8
            self.generated_labels_=select_reasons(scores[self.generated_mask_])
            Za=np.vstack([K,V[self.generated_mask_]])
            Ya=np.vstack([targets,self.generated_labels_])
            Ra=np.vstack([Rk,Ru[self.generated_mask_]])
        return self

    def _scores(self,Z):
        logistic=np.zeros((len(Z),5))
        logistic[:,self.logistic_.classes_]=self.logistic_.predict_proba(Z)
        similarity=normalize(Z)@self.neighbor_X_.T
        neighbors=np.argsort(-similarity,axis=1)[:,:10]
        weights=np.take_along_axis(similarity,neighbors,axis=1).clip(0)**2
        local=(weights[:,:,None]*self.neighbor_Y_[neighbors]).sum(axis=1)
        local/=weights.sum(axis=1,keepdims=True)+1e-9
        lda=np.zeros_like(logistic)
        lda[:,self.lda_.classes_]=self.lda_.predict_proba(Z)
        return (1-.3-.2)*logistic+.3*local+.2*lda

    def predict_proba(self, embeddings):
        """Selection scores, not calibrated marginal probabilities."""
        return self._scores(self.scaler_.transform(np.asarray(embeddings,float)))

    def predict(self, embeddings):
        return select_reasons(self.predict_proba(embeddings))

Overwriting _source_export/part1/anchored_solution.py


## part1/experiments/anchored_graph_components_v1/models.py

In [4]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/models.py
"""Frozen, inductive C&S component experiments around the submitted anchored model."""
from pathlib import Path
import sys

HERE = Path(__file__).resolve().parent
sys.path[:0] = [str(HERE / 'vendor'), str(HERE.parents[1])]

import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize

from anchored_solution import AnchoredReasonClassifier, select_reasons
from pseudo_anchored import PseudoTeacherAnchored
from p1lib import cache
from p1lib.data import Data, LABELS, EMB, BITS
from p1lib.graph import Graph
import strict_adapter as strict

GRAPH = dict(space=dict(scale=True, pca=64, l2=True), k=10, s=1., sym='max')
CS = dict(type='cs', graph=GRAPH, a1=.8, a2=.6382007398005838,
          base=dict(type='lr', C=.1), base_features=dict(scale=True))
VARIANTS = ['cs_pseudo', 'graph_residual_raw', 'graph_residual_autoscale']


def fitting_context(frame):
    """The caller supplies eligible fitting rows only; no outer query is stored."""
    frame = frame.reset_index(drop=True).copy()
    Y = frame[LABELS].to_numpy(dtype=int, copy=True)
    known = (Y >= 0).all(1)
    assert (known | (Y == -100).all(1)).all()
    Y[~known] = -1
    bitmask = Y[known] @ BITS
    combos = np.unique(bitmask)
    CM = ((combos[:, None] & BITS[None, :]) > 0).astype(int)
    y = np.full(len(frame), -1)
    y[known] = np.searchsorted(combos, bitmask)
    train, pool = np.flatnonzero(known), np.flatnonzero(~known)
    D = Data(frame, frame[EMB].to_numpy(float), Y, y, combos, CM,
             idx=dict(lab=train, unl=pool, train=np.arange(len(frame)),
                      labval=train, val=np.array([], int), test=np.array([], int)))
    return strict.context(D, train, np.array([], int), pool, seed=0)


def fit_pseudo(frame):
    cache.clear()
    strict.AUDIT.clear()
    ctx = fitting_context(frame)
    Q = strict.cs_probabilities(CS, ctx, ctx.pool)
    assert np.isfinite(Q).all() and np.allclose(Q.sum(1), 1.)
    teacher = Q @ ctx.D.CM
    m = PseudoTeacherAnchored().fit(ctx.D.emb, frame[LABELS].to_numpy(int), teacher)
    m.component_audit_ = dict(teacher='C&S powerset LR, grouped inner OOF',
        teacher_generated=int(m.generated_mask_.sum()),
        generated_positive_counts=m.generated_labels_.sum(0).tolist(),
        inner=list(strict.AUDIT), rounds=m.round_audit_)
    return m


class ResidualCorrector:
    """Signed five-label residual field; shares one fit between the two scales.

    Inner anchored fits produce OOF errors. The graph sees fitting inputs only.
    Prediction extends that field to new rows and clips each label independently.
    There is deliberately no cross-label normalization or final smoothing stage.
    """
    def fit(self, frame, parent):
        strict.AUDIT.clear()
        ctx = fitting_context(frame)
        X = ctx.D.emb
        original_Y = frame[LABELS].to_numpy(int)
        oof = np.zeros((len(ctx.train), 5))
        coverage = np.zeros(len(ctx.train), int)
        for tr, te in strict.inner_folds(ctx):
            sub = strict.subcontext(ctx, ctx.train[tr], ctx.train[te])
            assert (original_Y[sub.train].sum(0) > 0).all()
            inner = AnchoredReasonClassifier().fit(X[sub.unsup], original_Y[sub.unsup])
            oof[te] = inner.predict_proba(X[sub.eval])
            coverage[te] += 1
        assert (coverage == 1).all()
        self.scaler = StandardScaler().fit(X)
        self.pca = PCA(64, random_state=0).fit(self.scaler.transform(X))
        self.graph = Graph(self.transform(X), np.arange(len(X)), len(X), 10, 1., 'max')
        E0 = np.zeros((len(X), 5))
        E0[ctx.train] = original_Y[ctx.train] - oof
        raw = self.graph.propagate(E0, .8)
        sigma = np.abs(E0[ctx.train]).sum(1).mean()
        scale = sigma / np.maximum(np.abs(raw).sum(1, keepdims=True), 1e-12)
        self.fields = dict(raw=raw, autoscale=scale * raw)
        self.parent = parent
        self.audit = dict(inner=list(strict.AUDIT), inner_coverage_exactly_once=True,
            inner_oof_sample_f1=float((2 * (select_reasons(oof) * original_Y[ctx.train]).sum(1)
                / (select_reasons(oof).sum(1) + original_Y[ctx.train].sum(1))).mean()),
            residual_mean_l1=float(sigma),
            scale_quantiles=np.quantile(scale, [0, .25, .5, .75, 1]).tolist(),
            raw_field_abs_quantiles=np.quantile(np.abs(raw), [0, .5, .9, 1]).tolist())
        return self

    def transform(self, X):
        return normalize(self.pca.transform(self.scaler.transform(X)))

    def predict_proba(self, X, mode):
        return np.clip(self.parent.predict_proba(X) + self.correction(X, mode), 0, 1)

    def correction(self, X, mode):
        return self.graph.extend(self.transform(X), self.fields[mode])

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/models.py


## part1/experiments/anchored_graph_components_v1/pseudo_anchored.py

In [5]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/pseudo_anchored.py
"""Current Ani with three small changes: anchored weights, binary votes, weaker GMM.

Fit receives only permitted rows, in original CSV order. Fully missing labels
are -100; query/test rows must never be passed to fit. Prediction uses only the
150 stopping-reason embedding columns, ordered emb_0 through emb_149.
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, normalize


def select_reasons(scores):
    result=(scores>=.3).astype(int)
    result[np.arange(len(scores)),scores.argmax(axis=1)]=1
    return result


class PseudoTeacherAnchored:
    def fit(self, embeddings, labels, teacher_scores=None):
        X,Y=np.asarray(embeddings,float),np.asarray(labels,int)
        if X.ndim!=2 or X.shape[1]!=150 or Y.shape!=(len(X),5):
            raise ValueError('Expected 150 embeddings and five labels per row.')
        known=np.isin(Y,[0,1]).all(axis=1)
        unknown=(Y==-100).all(axis=1)
        if not (known|unknown).all() or (Y[known].sum(axis=1)==0).any() or (Y[known].sum(axis=0)==0).any():
            raise ValueError('Use complete nonempty known labels or five -100 values; all five reasons must be represented.')
        self.scaler_=StandardScaler().fit(X)
        Z=self.scaler_.transform(X)
        self.pca_=PCA(30,random_state=0).fit(Z)
        self.gmm_=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(self.pca_.transform(Z))
        assert self.gmm_.converged_
        K,V=Z[known],Z[unknown];targets=Y[known]
        Rk=self.gmm_.predict_proba(self.pca_.transform(K))
        Ru=self.gmm_.predict_proba(self.pca_.transform(V))
        counts=(targets/targets.sum(axis=1,keepdims=True)).sum(axis=0)
        self.class_weights_=counts.mean()/counts
        Za,Ya,Ra=K,targets,Rk
        self.round_audit_=[]
        for step in range(3):
            row,reason=np.nonzero(Ya)
            weights=self.class_weights_[reason]/Ya.sum(axis=1)[row]
            weights*=np.where(row<len(K),1.,.25)
            self.logistic_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(Za[row],reason,sample_weight=weights)
            self.lda_=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(5)/5).fit(Za[row],reason)
            self.neighbor_X_=normalize(Za)
            self.neighbor_Y_=Ya.copy()
            self.round_audit_.append(dict(step=step,known=len(K),generated=len(Za)-len(K),
                                          known_weight=float(weights[row<len(K)].sum()),
                                          generated_weight=float(weights[row>=len(K)].sum())))
            if step==2:break
            T=Ya/Ya.sum(axis=1,keepdims=True)
            rates=(Ra.T@T+T.mean(axis=0))/(Ra.sum(axis=0)[:,None]+1)
            scores=(.95*self._scores(V)+.05*(Ru@rates) if teacher_scores is None
                    else np.asarray(teacher_scores, float))
            assert scores.shape == (len(V), 5)
            self.generated_mask_=scores.max(axis=1)>=.8
            self.generated_labels_=select_reasons(scores[self.generated_mask_])
            Za=np.vstack([K,V[self.generated_mask_]])
            Ya=np.vstack([targets,self.generated_labels_])
            Ra=np.vstack([Rk,Ru[self.generated_mask_]])
        return self

    def _scores(self,Z):
        logistic=np.zeros((len(Z),5))
        logistic[:,self.logistic_.classes_]=self.logistic_.predict_proba(Z)
        similarity=normalize(Z)@self.neighbor_X_.T
        neighbors=np.argsort(-similarity,axis=1)[:,:10]
        weights=np.take_along_axis(similarity,neighbors,axis=1).clip(0)**2
        local=(weights[:,:,None]*self.neighbor_Y_[neighbors]).sum(axis=1)
        local/=weights.sum(axis=1,keepdims=True)+1e-9
        lda=np.zeros_like(logistic)
        lda[:,self.lda_.classes_]=self.lda_.predict_proba(Z)
        return (1-.3-.2)*logistic+.3*local+.2*lda

    def predict_proba(self, embeddings):
        """Selection scores, not calibrated marginal probabilities."""
        return self._scores(self.scaler_.transform(np.asarray(embeddings,float)))

    def predict(self, embeddings):
        return select_reasons(self.predict_proba(embeddings))

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/pseudo_anchored.py


## part1/experiments/anchored_graph_components_v1/run.py

In [6]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/run.py
"""Replay the registered screen, then only qualifying confirmations. No test CSV read."""
import os
for key in ['OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS',
            'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[key] = '1'

import argparse
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import platform
import time

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support
from threadpoolctl import threadpool_limits

from models import (HERE, LABELS, EMB, CS, VARIANTS, AnchoredReasonClassifier,
                    select_reasons, fit_pseudo, ResidualCorrector, fitting_context, strict)
from pseudo_anchored import PseudoTeacherAnchored

P1 = HERE.parents[1]
REF = P1 / 'experiments/current_leaders_grouped_v1'
SNAP = REF / 'snapshots'


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')


def protected_paths():
    return ([P1 / 'anchored_solution.py', P1 / 'TESTING_POLICY.md',
             P1 / 'data/part1_train.csv', P1 / 'data/part1_val.csv']
            + list(SNAP.glob('*.json')) + list(REF.glob('anchored_*.npz')))


def freeze():
    target = HERE / 'protocol.json'
    assert not target.exists(), 'Do not overwrite a frozen study.'
    source = [HERE / 'run.py', HERE / 'models.py', HERE / 'pseudo_anchored.py']
    source += sorted((HERE / 'vendor').rglob('*.py'))
    p = dict(version='anchored-graph-components-v1', created_at=datetime.now(timezone.utc).isoformat(),
        base_main='256fa8dabb2cd243f1162a0356e2c55b7a369a59', parent='anchored 0.89636',
        source_sha256={str(f.relative_to(HERE)): digest(f) for f in source},
        protected_sha256={str(f.relative_to(P1)): digest(f) for f in protected_paths()},
        python=platform.python_version(), requirements_sha256=digest(HERE / 'requirements-lock.txt'),
        hypotheses=dict(cs_pseudo='Graph teacher labels the masked fitting pool more usefully than anchored self-training.',
            graph_residual_raw='Nearby fitting rows have transferable anchored OOF errors.',
            graph_residual_autoscale='C&S automatic residual magnitude scaling makes that correction useful.'),
        variants=VARIANTS, cs_teacher=CS,
        cs_parent_settings_preserved=dict(pseudo_confidence=.8, pseudo_decode='.3 + argmax',
            pseudo_weight=.25, rounds=3, blend=[.5, .3, .2]),
        cs_teacher_schedule='One fit from original fitting gold only; identical teacher on both refreshes.',
        residual=dict(inner_folds=5, inner_seed=0, group_purges=True, alpha=.8,
            graph=CS['graph'], modes=['raw', 'autoscale'],
            autoscale='mean gold OOF residual L1 / node propagated residual L1, floor 1e-12',
            extension='RBF weighted nearest fitting graph nodes',
            output='clip(base + field,0,1) independently by label; no row normalization or smoothing'),
        fixed='Parent blend, classifiers, weights and .3+argmax decoder unchanged; no parameter search.',
        access='Inductive; only eligible fitting inputs; Train157 excludes validation entirely; no test inputs.',
        inner='All five inner anchored fits refit preprocessing, clusters, pseudo labels and classifiers; same group purge.',
        screen='First three saved repeats, BOTH pools: positive mean paired delta and >=2/3 wins each.',
        confirm='Only screen passers: five extra saved repeats; positive mean and >=5/8 wins BOTH pools over all eight.',
        supplementary='Only shortlisted recipes: original validation plus three saved chronology windows.',
        diagnostic_exceptions=[], seeds='Parent0, graph PCA0, teacher/inner0; saved outer assignments unchanged.',
        metrics='Complete OOF repeat sample F1; mean, population SD; paired wins/ties/losses; macro and label PRF/support.',
        limitation='Repeated development labels; no untouched holdout or Kaggle outcome. Overlapping repeats are descriptive.')
    write_json(target, p)
    print('Frozen', target, digest(target), flush=True)


def check_frozen():
    p = json.loads((HERE / 'protocol.json').read_text())
    for name, sha in p['source_sha256'].items():
        assert digest(HERE / name) == sha, name
    for name, sha in p['protected_sha256'].items():
        assert digest(P1 / name) == sha, name
    return p


def group_hash(frame):
    return pd.util.hash_pandas_object(frame[EMB].round(10), index=False).to_numpy()


def cases(stage):
    train = pd.read_csv(P1 / 'data/part1_train.csv')
    val = pd.read_csv(P1 / 'data/part1_val.csv')
    both = pd.concat([train, val], ignore_index=True)
    if stage == 'supplementary':
        yield dict(pool='validation', seed=0, fold=0, fit=train, query=val, positions=np.arange(len(val)))
        for c in json.loads((SNAP / 'chronology.json').read_text())['chronological_splits']:
            fit = both[both.ID.isin(c['training_ids'])].copy()
            query = both.set_index('ID').loc[c['heldout_ids']].reset_index()
            assert not np.intersect1d(group_hash(fit), group_hash(query)).size
            yield dict(pool=c['name'], seed=0, fold=0, fit=fit, query=query, positions=np.arange(len(query)))
        return
    for pool, frame in [('train', train), ('pooled', both)]:
        plan = json.loads((SNAP / f'{pool}{3 if stage == "screen" else 5}.json').read_text())
        known = (frame[LABELS].to_numpy() >= 0).all(1)
        gold = np.flatnonzero(known)
        assert frame.iloc[gold].ID.tolist() == plan['gold_ids']
        groups = group_hash(frame)
        for fold in plan['folds']:
            query_pos = gold[fold['heldout_gold_positions']]
            allowed_masked = (~known) & (~np.isin(groups, groups[query_pos]))
            assert int((~known).sum() - allowed_masked.sum()) == fold['masked_duplicates_excluded']
            fit_pos = np.sort(np.r_[gold[fold['train_gold_positions']], np.flatnonzero(allowed_masked)])
            assert not np.intersect1d(groups[fit_pos], groups[query_pos]).size
            yield dict(pool=pool, seed=fold['seed'], fold=fold['fold'], fit=frame.iloc[fit_pos].copy(),
                query=frame.iloc[query_pos].copy(), positions=np.array(fold['heldout_gold_positions']))


def parent_reference(case, scores):
    if case['pool'] not in ['train', 'pooled']:
        return None
    r = np.load(REF / f'anchored_{case["pool"]}.npz')
    seeds = [20261126, 20261127, 20261128, 20261129, 20261130, 20261201, 20261202, 20261203]
    ix = seeds.index(case['seed'])
    positions = case['positions']
    assert r['ids'][positions].tolist() == case['query'].ID.tolist()
    assert np.array_equal(r['truth'][positions], case['query'][LABELS].to_numpy(int))
    delta = float(np.abs(scores - r['scores'][ix, positions]).max())
    assert delta < 1e-10, delta
    assert np.array_equal(select_reasons(scores), r['predictions'][ix, positions])
    return delta


def fit_case(case, variants, output, preflight=False):
    with threadpool_limits(limits=1):
        start = time.monotonic()
        frame, query = case['fit'], case['query']
        X, Y, Xq = frame[EMB].to_numpy(float), frame[LABELS].to_numpy(int), query[EMB].to_numpy(float)
        frame_before = frame.copy(deep=True)
        parent = AnchoredReasonClassifier().fit(X, Y)
        scores = dict(anchored=parent.predict_proba(Xq))
        audit = dict(pool=case['pool'], seed=case['seed'], fold=case['fold'],
            gold=int((Y >= 0).all(1).sum()), masked=int((Y == -100).all(1).sum()),
            fit_ids_sha256=hashlib.sha256('\n'.join(frame.ID).encode()).hexdigest(),
            parent_rounds=parent.round_audit_,
            reference_score_max_abs=parent_reference(case, scores['anchored']))
        if preflight:
            clone = PseudoTeacherAnchored().fit(X, Y)
            np.testing.assert_array_equal(clone.predict_proba(Xq), scores['anchored'])
            audit['no_teacher_clone_exact'] = True
        if 'cs_pseudo' in variants:
            cs = fit_pseudo(frame)
            scores['cs_pseudo'] = cs.predict_proba(Xq)
            audit['cs_pseudo'] = cs.component_audit_
            if preflight:
                np.testing.assert_allclose(cs.predict_proba(Xq[:2]), scores['cs_pseudo'][:2], atol=1e-12, rtol=0)
        if any(v.startswith('graph_residual_') for v in variants):
            residual = ResidualCorrector().fit(frame, parent)
            audit['residual'] = residual.audit
            for name in variants:
                if not name.startswith('graph_residual_'):
                    continue
                mode = name.removeprefix('graph_residual_')
                correction = residual.correction(Xq, mode)
                scores[name] = residual.predict_proba(Xq, mode)
                audit[name] = dict(correction_abs_mean=float(np.abs(correction).mean()),
                    clipped_fraction=float(((scores['anchored'] + correction < 0) |
                                            (scores['anchored'] + correction > 1)).mean()),
                    changed_rows=int(np.any(select_reasons(scores[name]) != select_reasons(scores['anchored']), axis=1).sum()))
                if preflight:
                    np.testing.assert_allclose(residual.predict_proba(Xq[:2], mode), scores[name][:2], atol=1e-12, rtol=0)
            if preflight:
                residual.fields['zero'] = np.zeros_like(residual.fields['raw'])
                np.testing.assert_array_equal(residual.predict_proba(Xq, 'zero'), scores['anchored'])
                audit['zero_residual_identity'] = True
                audit['query_batch_independence'] = True
        pd.testing.assert_frame_equal(frame, frame_before)
        audit['gold_unchanged'] = True
        audit['elapsed_seconds'] = time.monotonic() - start
        key = f'{case["pool"]}_{case["seed"]}_{case["fold"]}'
        arrays = dict(ids=query.ID.to_numpy(str), truth=query[LABELS].to_numpy(int), positions=case['positions'])
        for name, s in scores.items():
            assert np.isfinite(s).all()
            arrays[f'scores_{name}'] = s
            arrays[f'predictions_{name}'] = select_reasons(s)
        output = Path(output)
        np.savez_compressed(output / f'{key}.npz', **arrays)
        write_json(output / f'{key}.json', audit)
        return key, audit['elapsed_seconds']


def summarize(output, stages):
    """Independent sklearn metric computation after prediction files are written."""
    output = Path(output)
    rows, label_rows, audits = [], [], []
    for stage in stages:
        by_group = {}
        for p in sorted((output / stage).glob('*.npz')):
            a = json.loads(p.with_suffix('.json').read_text()); audits.append(a)
            by_group.setdefault((a['pool'], a['seed']), []).append(p)
        for (pool, seed), paths in by_group.items():
            arrays = [np.load(p) for p in paths]
            truth = np.concatenate([a['truth'] for a in arrays])
            ids = np.concatenate([a['ids'] for a in arrays])
            assert len(np.unique(ids)) == len(ids)
            if pool in ['train', 'pooled']:
                expected = json.loads((SNAP / f'{pool}3.json').read_text())['gold_ids']
                assert len(paths) == 5 and set(ids) == set(expected)
            for key in arrays[0].files:
                if not key.startswith('predictions_'): continue
                name = key.removeprefix('predictions_')
                pred = np.concatenate([a[key] for a in arrays])
                assert np.isin(pred, [0, 1]).all() and (pred.sum(1) > 0).all()
                row = dict(pool=pool, seed=seed, recipe=name, n=len(ids),
                    sample_f1=f1_score(truth, pred, average='samples'),
                    macro_f1=f1_score(truth, pred, average='macro', zero_division=0))
                rows.append(row)
                pr, re, f, support = precision_recall_fscore_support(truth, pred, zero_division=0)
                for j, label in enumerate(LABELS):
                    label_rows.append(dict(pool=pool, seed=seed, recipe=name, label=label,
                        precision=pr[j], recall=re[j], f1=f[j], support=int(support[j])))
    repeats = pd.DataFrame(rows)
    labels = pd.DataFrame(label_rows)
    summary = []
    for (pool, name), group in repeats.groupby(['pool', 'recipe']):
        parent = repeats[(repeats.pool == pool) & (repeats.recipe == 'anchored')].set_index('seed')
        delta = group.sample_f1.to_numpy() - parent.loc[group.seed, 'sample_f1'].to_numpy()
        summary.append(dict(pool=pool, recipe=name, repeats=len(group),
            sample_f1_mean=group.sample_f1.mean(), sample_f1_population_sd=group.sample_f1.std(ddof=0),
            macro_f1_mean=group.macro_f1.mean(), paired_delta_mean=float(delta.mean()),
            wins=int((delta > 1e-12).sum()), ties=int((np.abs(delta) <= 1e-12).sum()), losses=int((delta < -1e-12).sum())))
    summary = pd.DataFrame(summary)
    repeats.to_csv(output / 'repeat_scores.csv', index=False)
    labels.to_csv(output / 'per_label_repeat_scores.csv', index=False)
    labels.groupby(['pool', 'recipe', 'label'])[['precision', 'recall', 'f1', 'support']].mean().to_csv(output / 'per_label_summary.csv')
    summary.to_csv(output / 'summary.csv', index=False)
    write_json(output / 'fit_audits.json', audits)
    if stages == ['screen']:
        passes = []
        for v in VARIANTS:
            sub = summary[(summary.recipe == v) & summary.pool.isin(['train', 'pooled'])]
            if len(sub) == 2 and ((sub.paired_delta_mean > 0) & (sub.wins >= 2)).all(): passes.append(v)
        write_json(output / 'screen_gate.json', dict(passers=passes, rule='positive mean + 2/3 wins in BOTH pools'))
    print(summary.to_string(index=False), flush=True)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('action', choices=['freeze', 'preflight', 'screen', 'confirm', 'supplementary', 'summarize'])
    p.add_argument('--output', type=Path)
    p.add_argument('--workers', type=int, default=4)
    args = p.parse_args()
    if args.action == 'freeze': freeze(); return
    check_frozen()
    assert args.output is not None
    args.output.mkdir(parents=True, exist_ok=True)
    if args.action == 'summarize':
        summarize(args.output, [s for s in ['screen', 'confirm', 'supplementary'] if (args.output / s).exists()]); return
    stage = 'screen' if args.action == 'preflight' else args.action
    folder = args.output / stage
    folder.mkdir(exist_ok=True)
    variants = VARIANTS
    if stage != 'screen':
        variants = json.loads((args.output / 'screen_gate.json').read_text())['passers']
        assert variants, 'No screen passers; stop under the registered rule.'
    jobs = list(cases(stage))
    if args.action == 'preflight':
        print(fit_case(jobs[0], variants, folder, preflight=True), flush=True)
        return
    with ProcessPoolExecutor(max_workers=args.workers) as executor:
        pending = {}
        for c in jobs:
            key = f'{c["pool"]}_{c["seed"]}_{c["fold"]}'
            if (folder / f'{key}.npz').exists() and (folder / f'{key}.json').exists(): continue
            pending[executor.submit(fit_case, c, variants, folder)] = key
        for future in as_completed(pending):
            print('Completed', *future.result(), flush=True)
    summarize(args.output, [s for s in ['screen', 'confirm', 'supplementary'] if (args.output / s).exists()])


if __name__ == '__main__':
    main()

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/run.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/__init__.py

In [7]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/__init__.py
"""p1lib: composable semi-supervised experimentation framework for CS5785 midterm Part 1."""
from .data import LABELS, load, metrics, row_f1, write_submission  # noqa: F401
from .evaluate import evaluate, make_folds, score, fold_outputs, summary_row, paired_se  # noqa: F401
from .pipeline import run_model, validate  # noqa: F401

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/__init__.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/blocks.py

In [8]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/blocks.py
"""Optional non-embedding feature blocks (each a toggle in the features config).

Every block is label-free: vocabularies / IDF / SVD are fit only on the rows the fold allows
(`fit_rows`), then applied to all rows.  Output is a dense (n_all, d) array, standardized on
`fit_rows`.

Blocks
  criteria  : TF-IDF of eligibility criteria  -> TruncatedSVD(d)
  diseases  : TF-IDF of diseases_cleaned      -> TruncatedSVD(d)
  drugs     : multi-hot matched_drugs         -> TruncatedSVD(d)
  pathways  : multi-hot drug_pathways (optionally only Reactome level <= max_level) -> SVD(d)
  morgan    : RDKit Morgan fingerprint (radius 2, 1024 bits, max over drugs) -> SVD(d)
  drugmeta  : [has_matched_drug, log1p(#matched drugs)]
  year      : standardized start_year  (sampling artifact; ablation only)
"""
from __future__ import annotations

import numpy as np
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from .cache import memo

_MORGAN = None


def _split_semicolon(s):
    if not isinstance(s, str) or not s.strip():
        return []
    return [t.strip().lower() for t in s.split(";") if t.strip()]


def _morgan_matrix(D):
    global _MORGAN
    if _MORGAN is not None:
        return _MORGAN
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog("rdApp.*")
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    out = np.zeros((D.n, 1024), dtype=np.float32)
    for i, s in enumerate(D.df["drug_smiles"].to_numpy()):
        if not isinstance(s, str):
            continue
        fps = [gen.GetFingerprintAsNumPy(m) for t in s.split(";") if (m := Chem.MolFromSmiles(t)) is not None]
        if fps:
            out[i] = np.max(fps, axis=0)
    _MORGAN = out
    return out


def _svd(M, fit_rows, d, seed=0):
    d = int(min(d, M.shape[1] - 1, len(fit_rows) - 1))
    svd = TruncatedSVD(n_components=max(d, 1), random_state=seed).fit(M[fit_rows])
    return svd.transform(M)


def _standardize(Z, fit_rows):
    mu = Z[fit_rows].mean(0)
    sd = Z[fit_rows].std(0)
    sd[sd < 1e-12] = 1.0
    return (Z - mu) / sd


def build_block(D, name: str, cfg: dict, fit_rows: np.ndarray) -> np.ndarray:
    cfg = dict(cfg or {})
    key = {"block": name, **cfg}
    return memo("block", key, fit_rows, lambda: _build(D, name, cfg, fit_rows))


def _build(D, name, cfg, fit_rows):
    d = int(cfg.get("d", 32))
    df = D.df
    if name == "criteria" or name == "diseases":
        col = "criteria" if name == "criteria" else "diseases_cleaned"
        text = df[col].fillna("").astype(str).to_numpy()
        vec = TfidfVectorizer(max_features=int(cfg.get("max_features", 5000)), min_df=3,
                              sublinear_tf=True, ngram_range=(1, int(cfg.get("ngram", 1))),
                              stop_words="english")
        vec.fit(text[fit_rows])
        M = vec.transform(text)
        Z = _svd(M, fit_rows, d)
    elif name in ("drugs", "pathways"):
        col = "matched_drugs" if name == "drugs" else "drug_pathways"
        toks = [_split_semicolon(s) for s in df[col].to_numpy()]
        if name == "pathways" and cfg.get("max_level") is not None:
            lvl = dict(zip(D.catalog["reactome_id"].str.lower(), D.catalog["pathway_level"]))
            ml = int(cfg["max_level"])
            toks = [[t for t in ts if lvl.get(t, 99) <= ml] for ts in toks]
        docs = [" ".join(t.replace(" ", "_") for t in ts) for ts in toks]
        vec = CountVectorizer(binary=True, min_df=3, token_pattern=r"[^ ]+")
        try:
            vec.fit([docs[i] for i in fit_rows])
            M = vec.transform(docs).astype(float)
        except ValueError:            # empty vocabulary
            M = sparse.csr_matrix((D.n, 1))
        Z = _svd(M, fit_rows, d) if M.shape[1] > 2 else M.toarray()
    elif name == "morgan":
        M = _morgan_matrix(D)
        Z = _svd(sparse.csr_matrix(M), fit_rows, d)
    elif name == "drugmeta":
        toks = [_split_semicolon(s) for s in df["matched_drugs"].to_numpy()]
        Z = np.c_[[len(t) > 0 for t in toks], np.log1p([len(t) for t in toks])].astype(float)
    elif name == "year":
        Z = df["start_year"].to_numpy(float)[:, None]
    else:
        raise ValueError(f"unknown block {name}")
    return _standardize(np.asarray(Z, float), fit_rows) * float(cfg.get("weight", 1.0))

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/blocks.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/cache.py

In [9]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/cache.py
"""Tiny in-process memo for label-free fits (scalers, PCA, clusterers, graphs).

A label-free object depends only on (its config, the set of rows whose features it was fit on).
In transductive mode that row set is the same (all 1,561 rows) for every fold, so one fit is
reused across folds and trials; in inductive mode the row set changes per fold and the cache
simply misses.  Nothing that touches labels is ever cached here.
"""
from __future__ import annotations

import hashlib
import json

import numpy as np

_CACHE: dict = {}
_MAX = 4000


def cfg_key(obj) -> str:
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]


def rows_key(rows: np.ndarray) -> str:
    return hashlib.md5(np.sort(np.asarray(rows)).astype(np.int64).tobytes()).hexdigest()[:16]


def memo(kind: str, cfg, rows, fn):
    key = (kind, cfg_key(cfg), rows_key(rows))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val


def clear():
    _CACHE.clear()


def memo_ctx(kind: str, cfg, ctx, fn, extra=None):
    """Memo for LABEL-DEPENDENT results: keyed by the exact training rows (so a result is only
    reused for the identical fold), the feature rows, the pool and the requested rows."""
    key = (kind, cfg_key(cfg), rows_key(ctx.train), rows_key(ctx.unsup), rows_key(ctx.pool),
           ctx.seed, None if extra is None else cfg_key(extra))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/cache.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/classifiers.py

In [10]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/classifiers.py
"""Classifiers over the K label combinations.  Common interface:

    clf = make_classifier(cfg, K, CM)
    clf.fit(X, y, w)            # y: int combo ids, w: sample weights (soft labels are
                                #    passed as replicated rows, one per combo, weight lam*q)
    P = clf.predict_proba(X)    # (n, K), columns aligned to all K combos
    clf.prior_                  # effective source prior the model was trained under
                                # (needed by SLD; uniform when class_weight='balanced')

cfg["type"] in {"lr", "ovr", "lda", "knn", "hgb", "nb"}.
"""
from __future__ import annotations

import warnings

import numpy as np
from sklearn.covariance import ledoit_wolf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


def _weighted_prior(y, w, K):
    p = np.bincount(y, weights=w, minlength=K).astype(float)
    return p / p.sum()


def _balance(y, w, K):
    """class_weight='balanced' computed from the weighted label mass."""
    mass = np.bincount(y, weights=w, minlength=K)
    present = mass > 0
    cw = np.zeros(K)
    cw[present] = w.sum() / (present.sum() * mass[present])
    return w * cw[y], present


class _Base:
    def __init__(self, cfg, K, CM):
        self.cfg = cfg
        self.K = K
        self.CM = CM

    def _prep(self, y, w):
        w = np.ones(len(y)) if w is None else np.asarray(w, float)
        if self.cfg.get("class_weight") == "balanced":
            w, present = _balance(y, w, self.K)
            self.prior_ = present / present.sum()
        else:
            self.prior_ = _weighted_prior(y, w, self.K)
        return w

    def _full(self, Pc, classes):
        P = np.zeros((Pc.shape[0], self.K))
        P[:, classes] = Pc
        P = P + 1e-9
        return P / P.sum(1, keepdims=True)


class LRPowerset(_Base):
    """Multinomial logistic regression on the label powerset (T1)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.m = LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                    tol=1e-5).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class OvRLR(_Base):
    """One-vs-rest LR on the 5 labels; combo probabilities = product of marginals renormalized
    over the K allowed combinations (so the output is always a valid label set)."""

    def fit(self, X, y, w=None):
        w0 = np.ones(len(y)) if w is None else np.asarray(w, float)
        self.prior_ = _weighted_prior(y, w0, self.K)
        B = self.CM[y]
        self.ms = []
        for j in range(5):
            yj = B[:, j]
            wj = w0.copy()
            if self.cfg.get("class_weight") == "balanced":
                pos = wj[yj == 1].sum()
                neg = wj[yj == 0].sum()
                if pos > 0 and neg > 0:
                    wj = np.where(yj == 1, wj * (pos + neg) / (2 * pos), wj * (pos + neg) / (2 * neg))
            if yj.min() == yj.max():
                self.ms.append(float(yj[0]))
                continue
            self.ms.append(LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                              tol=1e-5).fit(X, yj, sample_weight=wj))
        if self.cfg.get("class_weight") == "balanced":
            self.prior_ = np.ones(self.K) / self.K
        return self

    def marginals(self, X):
        M = np.zeros((len(X), 5))
        for j, m in enumerate(self.ms):
            M[:, j] = m if isinstance(m, float) else m.predict_proba(X)[:, 1]
        return np.clip(M, 1e-6, 1 - 1e-6)

    def predict_proba(self, X):
        M = self.marginals(X)
        L = np.log(M) @ self.CM.T + np.log(1 - M) @ (1 - self.CM).T
        L -= L.max(1, keepdims=True)
        P = np.exp(L)
        return P / P.sum(1, keepdims=True)


class ShrinkLDA(_Base):
    """Shrinkage LDA (tied Gaussian classes) with sample-weight support.  Follows sklearn's
    solver='lsqr', shrinkage='auto': per-class Ledoit-Wolf covariances (on standardized
    features) averaged with the class priors."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        K, d = self.K, X.shape[1]
        self.classes_ = np.unique(y)
        means = np.zeros((K, d))
        S = np.zeros((d, d))
        prior = self.prior_
        for k in self.classes_:
            m = y == k
            wk = w[m]
            Xk = X[m]
            mu = (wk[:, None] * Xk).sum(0) / wk.sum()
            means[k] = mu
            Xc = (Xk - mu) * np.sqrt(wk / wk.mean())[:, None]
            if len(Xk) < 2:
                continue
            sd = Xc.std(0)
            sd[sd < 1e-12] = 1.0
            C = ledoit_wolf(Xc / sd, assume_centered=True)[0]
            S += prior[k] * (sd[:, None] * C * sd[None, :])
        S += 1e-8 * np.eye(d)
        self.means_ = means
        self.coef_ = np.linalg.solve(S, means[self.classes_].T).T
        self.intercept_ = -0.5 * (self.coef_ * means[self.classes_]).sum(1) + np.log(prior[self.classes_])
        return self

    def predict_proba(self, X):
        Lg = X @ self.coef_.T + self.intercept_
        Lg -= Lg.max(1, keepdims=True)
        P = np.exp(Lg)
        P /= P.sum(1, keepdims=True)
        return self._full(P, self.classes_)


class KNNClf(_Base):
    """Cosine kNN over combos with distance weighting and soft targets; smoothed toward the
    training prior with beta pseudo-counts so probabilities are never exactly 0."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        self.Yw = np.eye(self.K)[y] * w[:, None]
        self.w = w
        return self

    def predict_proba(self, X, exclude_self=False):
        k = int(self.cfg.get("k", 10))
        Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        S = Xn @ self.Xn.T
        if exclude_self:
            np.fill_diagonal(S, -np.inf)
        k = min(k, S.shape[1] - (1 if exclude_self else 0))
        nn = np.argpartition(-S, k - 1, axis=1)[:, :k]
        sims = np.take_along_axis(S, nn, 1)
        if self.cfg.get("weights", "distance") == "distance":
            wt = np.exp(sims / float(self.cfg.get("tau", 0.1)))
        else:
            wt = np.ones_like(sims)
        votes = np.einsum("nk,nkc->nc", wt, self.Yw[nn])
        mass = np.einsum("nk,nk->n", wt, self.w[nn])
        beta = float(self.cfg.get("beta", 1.0))
        P = (votes + beta * self.prior_) / (mass + beta)[:, None]
        return P / P.sum(1, keepdims=True)


class HGB(_Base):
    """HistGradientBoosting over combos (mainly as a stacker on meta-features)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        c = self.cfg
        self.m = HistGradientBoostingClassifier(
            learning_rate=float(c.get("learning_rate", 0.05)), max_iter=int(c.get("max_iter", 100)),
            max_leaf_nodes=int(c.get("max_leaf_nodes", 6)), max_depth=c.get("max_depth", 3),
            min_samples_leaf=int(c.get("min_samples_leaf", 20)),
            l2_regularization=float(c.get("l2_regularization", 1.0)),
            max_features=float(c.get("max_features", 1.0)), early_stopping=False,
            random_state=int(c.get("seed", 0))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class GaussNB(_Base):
    """Gaussian naive Bayes with variance smoothing (cheap, high-bias baseline)."""

    def fit(self, X, y, w=None):
        from sklearn.naive_bayes import GaussianNB
        w = self._prep(y, w)
        self.m = GaussianNB(var_smoothing=float(self.cfg.get("var_smoothing", 1e-2))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class PriorClf(_Base):
    """Predicts the training prior for every row (MAP -> always the majority combo)."""

    def fit(self, X, y, w=None):
        self._prep(y, w)
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))


REGISTRY = {"prior": PriorClf, "lr": LRPowerset, "ovr": OvRLR, "lda": ShrinkLDA, "knn": KNNClf, "hgb": HGB, "nb": GaussNB}


def make_classifier(cfg, K, CM):
    return REGISTRY[cfg["type"]](cfg, K, CM)

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/classifiers.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/configs.py

In [11]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/configs.py
"""Named reference configurations (the report's five techniques, baselines and ensembles).

Every entry is a full experiment config: {"model": ..., "head": ..., "transductive": ...}.
Build new ones by copying and editing; `exp(model, head)` is a small helper.
"""
from __future__ import annotations

import copy

STD = {"scale": True}
PCA64 = {"scale": True, "pca": 64}
GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}

NO_HEAD = {"prior": "none", "decode": "map"}
SLD_HEAD = {"prior": "sld", "sld_a": 100, "decode": "map"}


def exp(model, head=None, transductive=True):
    return {"model": copy.deepcopy(model), "head": copy.deepcopy(head or NO_HEAD), "transductive": transductive}


def single(clf, features=STD, synth=None, labeler=None, temperature=False):
    return {"features": copy.deepcopy(features), "synth": copy.deepcopy(synth or []),
            "labeler": copy.deepcopy(labeler), "clf": copy.deepcopy(clf), "temperature": temperature}


LR = {"type": "lr", "C": 0.1}
LDA = {"type": "lda"}
OVR = {"type": "ovr", "C": 0.1}
KNN = {"type": "knn", "k": 10, "weights": "distance", "tau": 0.1, "beta": 1.0}

M_LR = single(LR)
M_LDA = single(LDA)
M_OVR = single(OVR)
M_KNN = single(KNN, features={"scale": True, "pca": 64, "l2": True})
M_CS = single({"type": "direct", "labeler": {"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5}})
M_SPREAD = single({"type": "direct", "labeler": {"type": "spread", "graph": GRAPH, "alpha": 0.5}})
M_CLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "kmeans", "k": 24, "m": 4}})
M_GMMCLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "gmm", "k": 16, "cov": "diag", "m": 4}})

# T1 with SLD-filled pseudo-labels for the unlabeled pool
M_LR_SLDPL = single(LR, labeler={"type": "sld", "base": LR, "a": 100, "lam": 0.1})
# T2 C&S pseudo-labels feeding LR
M_LR_CSPL = single(LR, labeler={"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5, "lam": 0.1})
# T4 cluster-then-label pseudo-labels feeding LR
M_LR_CLPL = single(LR, labeler={"type": "cluster", "algo": "kmeans", "k": 24, "m": 4, "lam": 0.1})
# T3 guarded self-training
M_LR_ST = single(LR, labeler={"type": "selftrain", "members": ["lr", "lda", "cs"], "rounds": 2, "tau": 0.75,
                              "lam": 0.3, "ramp": True, "align": "sld", "weighting": "softmatch",
                              "cluster_filter": True})
M_LR_ST_NAIVE = single(LR, labeler={"type": "selftrain", "members": ["lr"], "rounds": 3, "tau": 0.75,
                                    "lam": 0.5, "align": "none", "hard": True})
# synthetic-feature variants
M_LR_SYN = single(LR, synth=[{"type": "gmm", "k": 16, "cov": "diag"}, {"type": "knn_vote", "k": 10},
                             {"type": "graph", "mode": "spread", "graph": GRAPH, "alpha": 0.5},
                             {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}])
# T5 HGB stacker on nested-OOF meta-features
META = [{"type": "oof_model", "model": M_LR}, {"type": "oof_model", "model": M_LDA},
        {"type": "graph", "mode": "cs", "graph": GRAPH, "alpha": 0.5},
        {"type": "knn_vote", "k": 10}, {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}]
HGB = {"type": "hgb", "learning_rate": 0.05, "max_iter": 100, "max_leaf_nodes": 4, "max_depth": 2,
       "min_samples_leaf": 20, "l2_regularization": 1.0}
M_STACK_HGB = single(HGB, features=None, synth=META)
M_STACK_LR = single({"type": "lr", "C": 0.1}, features=None, synth=META)
M_HGB_RAW = single(HGB, features=STD)

# report's recommended ensemble: T1-LR + T1-LDA + T2-C&S, equal weights
M_ENS3 = {"ensemble": [M_LR, M_LDA, M_CS], "weights": None}

M_PRIOR = single({"type": "prior"})
M_NB = single({"type": "nb"}, features=PCA64)
SLD_EF1 = {"prior": "sld", "sld_a": 25, "decode": "expf1"}

REFERENCE = {
    "Majority": exp(M_PRIOR),
    "T1_LR": exp(M_LR),
    "T1_LR+SLD": exp(M_LR, SLD_HEAD),
    "T1_LDA": exp(M_LDA),
    "T1_LDA+SLD": exp(M_LDA, SLD_HEAD),
    "OvR_LR": exp(M_OVR),
    "kNN": exp(M_KNN),
    "T2_CS": exp(M_CS),
    "T2_CS+SLD": exp(M_CS, SLD_HEAD),
    "T2_spread": exp(M_SPREAD),
    "T4_kmeans_direct": exp(M_CLUSTER),
    "T4_gmm_direct": exp(M_GMMCLUSTER),
    "T1_LR+SLDpseudo": exp(M_LR_SLDPL),
    "T1_LR+SLDpseudo+SLD": exp(M_LR_SLDPL, SLD_HEAD),
    "T2_LR+CSpseudo": exp(M_LR_CSPL),
    "T4_LR+clusterpseudo": exp(M_LR_CLPL),
    "T3_selftrain_guarded": exp(M_LR_ST),
    "T3_selftrain_naive": exp(M_LR_ST_NAIVE),
    "LR+synth": exp(M_LR_SYN),
    "T5_stack_HGB": exp(M_STACK_HGB),
    "T5_stack_HGB+SLD": exp(M_STACK_HGB, SLD_HEAD),
    "stack_LR": exp(M_STACK_LR),
    "HGB_raw": exp(M_HGB_RAW),
    "ENS3": exp(M_ENS3),
    "ENS3+SLD": exp(M_ENS3, SLD_HEAD),
}

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/configs.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/context.py

In [12]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/context.py
"""Fold context: which rows are labeled-for-training, which are predicted, which are unlabeled.

Every component receives a `Ctx` and may only read labels of `ctx.train`.  Features of
`ctx.unsup` rows may be used by label-free steps (scaler, PCA, clustering, graph nodes).

transductive=True  -> unsup = all 1,561 rows (train + val + test features, no labels)
transductive=False -> unsup = all rows except the eval rows and the test rows
"""
from __future__ import annotations

from dataclasses import dataclass, replace

import numpy as np


@dataclass
class Ctx:
    D: object
    train: np.ndarray            # global idx of labeled training rows (labels visible)
    eval: np.ndarray             # global idx of rows to predict (labels hidden)
    pool: np.ndarray             # global idx of the pseudo-label pool (default: 1,014 unlabeled)
    unsup: np.ndarray            # global idx of rows whose features label-free steps may use
    transductive: bool = True
    seed: int = 0
    name: str = ""
    depth: int = 0               # nesting depth (0 = outer fold)

    @property
    def y_train(self):
        return self.D.y[self.train]

    @property
    def sld_target(self):
        """Rows SLD estimates the target prior on: the eval batch when transductive, else the
        unlabeled pool (inductive: we may not look at the eval/test features)."""
        return self.eval if self.transductive else self.pool

    def sub(self, train, eval_, name=None):
        """Inner context for nested OOF: same unlabeled pool / feature rows, fewer labels."""
        return replace(self, train=np.asarray(train), eval=np.asarray(eval_),
                       name=name or self.name + "/inner", depth=self.depth + 1)


def make_ctx(D, train, eval_, transductive=True, pool=None, seed=0, name=""):
    train = np.asarray(train)
    eval_ = np.asarray(eval_)
    pool = D.idx["unl"] if pool is None else np.asarray(pool)
    if transductive:
        unsup = np.arange(D.n)
    else:
        banned = np.zeros(D.n, bool)
        banned[eval_] = True
        banned[D.idx["test"]] = True
        unsup = np.where(~banned)[0]
    assert len(np.intersect1d(train, eval_)) == 0, "train/eval overlap"
    assert (D.y[train] >= 0).all(), "training rows must be labeled"
    return Ctx(D=D, train=train, eval=eval_, pool=pool, unsup=unsup,
               transductive=transductive, seed=seed, name=name)

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/context.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/data.py

In [13]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/data.py
"""Data loading, label-powerset encoding and metrics for Part 1.

Global row order used everywhere in this package:
    [ part1_train (1171 rows, file order) | part1_val (78) | part1_test (312) ]  -> 1561 rows.
`Data.y` holds the label-combo id (0..7) for labeled rows and -1 for masked/test rows.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field

import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

LABELS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMB = [f"emb_{i}" for i in range(150)]
BITS = np.array([1, 2, 4, 8, 16])

DEFAULT_DATA_DIR = os.environ.get(
    "P1_DATA_DIR",
    os.path.join(os.path.dirname(__file__), "..", "..", "..", "part1", "data"),
)


@dataclass
class Data:
    df: pd.DataFrame                 # all 1561 rows, global order, with a 'split' column
    emb: np.ndarray                  # (1561, 150) raw embeddings
    Y: np.ndarray                    # (1561, 5) labels, -1 where unknown
    y: np.ndarray                    # (1561,) combo id, -1 where unknown
    combos: np.ndarray               # (K,) bitmask of each allowed combo
    CM: np.ndarray                   # (K, 5) binary label matrix of each combo
    idx: dict = field(default_factory=dict)   # named index groups
    year: np.ndarray = None
    catalog: pd.DataFrame = None

    @property
    def K(self) -> int:
        return len(self.combos)

    @property
    def n(self) -> int:
        return len(self.df)

    def combo_names(self):
        return ["+".join(LABELS[j][0] for j in range(5) if self.CM[k, j]) for k in range(self.K)]


def load(data_dir: str | None = None) -> Data:
    d = os.path.abspath(data_dir or DEFAULT_DATA_DIR)
    tr = pd.read_csv(os.path.join(d, "part1_train.csv"))
    va = pd.read_csv(os.path.join(d, "part1_val.csv"))
    te = pd.read_csv(os.path.join(d, "part1_test.csv"))
    cat = pd.read_csv(os.path.join(d, "pathway_catalog.csv"))
    te = pd.concat([te, pd.DataFrame(-1, index=te.index, columns=LABELS)], axis=1)
    tr = tr.assign(split="train")
    va = va.assign(split="val")
    te = te.assign(split="test")
    df = pd.concat([tr, va, te], ignore_index=True).copy()
    Y = df[LABELS].to_numpy().astype(int)
    Y[Y == -100] = -1
    known = (Y >= 0).all(1)
    mask = (Y.clip(0) * BITS).sum(1)
    combos = np.unique(mask[known])
    y = np.full(len(df), -1)
    y[known] = np.searchsorted(combos, mask[known])
    CM = np.array([[(c >> j) & 1 for j in range(5)] for c in combos])
    split = df["split"].to_numpy()
    idx = {
        "train": np.where(split == "train")[0],
        "lab": np.where((split == "train") & known)[0],
        "unl": np.where((split == "train") & ~known)[0],
        "val": np.where(split == "val")[0],
        "test": np.where(split == "test")[0],
    }
    idx["labval"] = np.r_[idx["lab"], idx["val"]]
    D = Data(df=df, emb=df[EMB].to_numpy(float), Y=Y, y=y, combos=combos, CM=CM, idx=idx,
             year=df["start_year"].to_numpy(int), catalog=cat)
    assert len(idx["lab"]) == 157 and len(idx["unl"]) == 1014 and len(idx["val"]) == 78 and len(idx["test"]) == 312
    assert (Y[idx["lab"]].sum(1) >= 1).all(), "labeled rows are never all-zero"
    return D


# ----------------------------------------------------------------------------- metrics
def row_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-row F1 between binary label sets (both non-empty here)."""
    inter = (Ytrue * Ypred).sum(1)
    denom = Ytrue.sum(1) + Ypred.sum(1)
    out = np.where(denom > 0, 2 * inter / np.maximum(denom, 1), 1.0)
    return out


def label_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-label (column) binary F1; F1 = 0 when a label has no positives and none predicted."""
    tp = (Ytrue * Ypred).sum(0)
    fp = ((1 - Ytrue) * Ypred).sum(0)
    fn = (Ytrue * (1 - Ypred)).sum(0)
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)


def metrics(Ytrue: np.ndarray, Ypred: np.ndarray) -> dict:
    rf = row_f1(Ytrue, Ypred)
    lf = label_f1(Ytrue, Ypred)
    return {
        "sample_f1": float(rf.mean()),
        "sample_f1_se": float(rf.std(ddof=1) / np.sqrt(len(rf))),
        "macro_f1": float(lf.mean()),
        **{f"f1_{LABELS[j]}": float(lf[j]) for j in range(5)},
        "n": int(len(rf)),
    }


def bootstrap_se(Ytrue: np.ndarray, Ypreds: list[np.ndarray], B: int = 300, seed: int = 0) -> dict:
    """Bootstrap SE (over rows) of macro and per-label F1, averaged over a list of prediction
    matrices for the same rows (e.g. CV repeats) using shared resamples."""
    rng = np.random.default_rng(seed)
    n = len(Ytrue)
    macro, per = [], []
    for _ in range(B):
        b = rng.integers(0, n, n)
        lfs = np.mean([label_f1(Ytrue[b], P[b]) for P in Ypreds], axis=0)
        per.append(lfs)
        macro.append(lfs.mean())
    per = np.array(per)
    return {"macro_f1_se": float(np.std(macro, ddof=1)),
            **{f"f1_{LABELS[j]}_se": float(per[:, j].std(ddof=1)) for j in range(5)}}


def combos_to_labels(D: Data, kidx: np.ndarray) -> np.ndarray:
    return D.CM[kidx]


def write_submission(D: Data, Ypred: np.ndarray, path: str, ids=None) -> pd.DataFrame:
    """Kaggle format: ID,Predicted with space-separated reason names (always >=1)."""
    assert (Ypred.sum(1) >= 1).all(), "every test row needs at least one predicted reason"
    ids = D.df["ID"].to_numpy()[D.idx["test"]] if ids is None else ids
    pred = [" ".join(LABELS[j] for j in range(5) if row[j]) for row in Ypred]
    sub = pd.DataFrame({"ID": ids, "Predicted": pred})
    sub.to_csv(path, index=False, lineterminator="\n")
    return sub

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/data.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/evaluate.py

In [14]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/evaluate.py
"""Leak-free evaluation harness.

Protocols (all over the 235 labeled rows = 157 labeled-train + 78 val):
  A : RepeatedStratifiedKFold(5 x R) stratified on the label-combo id         (no shift)
  B : chronological holdout: train start_year < cut, evaluate >= cut         (shift, cut=2005)
  C : official split: train on the 157 labeled-train rows, evaluate on val   (tie-breaker)
Every fold builds a fresh Ctx; everything label-dependent is fit inside `run_model` from
ctx.train labels only.  Label-free fits are reused across folds only when their feature-row
set is identical (transductive mode).

Fold outputs (calibrated combo probabilities BEFORE the head) are cached on disk keyed by the
model part of the config, so SLD strength / decoding are post-hoc and free to vary.
"""
from __future__ import annotations

import hashlib
import json
import os
import time

import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold

from .context import make_ctx
from .data import LABELS, bootstrap_se, label_f1, metrics, row_f1
from .head import apply_head
from .pipeline import model_part, run_model, validate

# fold-output cache lives OUTSIDE the midterm/ folder (regenerable, ~0.5 GB); override with P1_CACHE
CACHE_DIR = os.environ.get("P1_CACHE", os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "..", ".p1_cache", "folds")))
CHRONO_CUT = 2005
CHRONO_CUTS = (2004, 2005, 2006)   # objective uses the mean over these cuts


def cfg_hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:20]


# ----------------------------------------------------------------------------- folds
def make_folds(D, protocols=("A", "B", "C"), R=4, seed=0, rows=None, chrono_cuts=CHRONO_CUTS):
    """rows: restrict to a subset of labeled rows (used by the nested search-overfitting check)."""
    labval = D.idx["labval"] if rows is None else np.asarray(rows)
    y = D.y[labval]
    folds = []
    if "A" in protocols:
        rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=R, random_state=seed)
        for i, (tr, te) in enumerate(rskf.split(np.zeros(len(y)), y)):
            folds.append({"name": f"A{i // 5}_{i % 5}", "protocol": "A", "repeat": i // 5,
                          "train": labval[tr], "eval": labval[te]})
    if "B" in protocols:
        for cut in chrono_cuts:
            yr = D.year[labval]
            folds.append({"name": f"B{cut}", "protocol": "B", "cut": cut,
                          "train": labval[yr < cut], "eval": labval[yr >= cut]})
    if "C" in protocols:
        tr = np.intersect1d(labval, D.idx["lab"])
        te = np.intersect1d(labval, D.idx["val"])
        if len(tr) and len(te):
            folds.append({"name": "C", "protocol": "C", "train": tr, "eval": te})
    return folds


def fold_tag(fold):
    h = hashlib.md5(np.r_[fold["train"], [-1], fold["eval"]].astype(np.int64).tobytes()).hexdigest()[:10]
    return f"{fold['name']}_{h}"


# ----------------------------------------------------------------------------- one fold
def run_fold(cfg, D, fold, use_cache=True):
    mp = model_part(cfg)
    key = cfg_hash(mp)
    path = os.path.join(CACHE_DIR, key, fold_tag(fold) + ".npz")
    if use_cache and os.path.exists(path):
        try:
            z = np.load(path, allow_pickle=False)
            out = {k: z[k] for k in z.files}
            out["T"] = out["T"].tolist()
            return out
        except Exception:
            pass
    tr = cfg.get("transductive", True)
    ctx = make_ctx(D, fold["train"], fold["eval"], transductive=tr, seed=0, name=fold["name"])
    t0 = time.time()
    rows = fold["eval"] if tr else np.r_[fold["eval"], ctx.pool]
    o = run_model(cfg["model"], ctx, rows)
    ne = len(fold["eval"])
    out = {"P_eval": o["P"][:ne].astype(np.float32), "prior": np.asarray(o["prior"], float),
           "T": np.atleast_1d(np.asarray(o["T"], float)), "secs": np.float32(time.time() - t0)}
    if not tr:
        out["P_target"] = o["P"][ne:].astype(np.float32)
    if use_cache:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + f".{os.getpid()}.tmp.npz"
        np.savez_compressed(tmp, **out)
        os.replace(tmp, path)
        mpath = os.path.join(CACHE_DIR, key, "config.json")
        if not os.path.exists(mpath):
            with open(mpath, "w") as f:
                json.dump(mp, f, indent=1, default=str)
    out["T"] = out["T"].tolist()
    return out


def fold_outputs(cfg, D, folds, n_jobs=1, use_cache=True):
    if n_jobs == 1:
        return [run_fold(cfg, D, f, use_cache) for f in folds]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=n_jobs, backend="loky")(delayed(run_fold)(cfg, D, f, use_cache) for f in folds)


# ----------------------------------------------------------------------------- scoring
def score(cfg, D, folds, outs, bootstrap=True):
    """Apply the head to cached fold outputs and compute A/B/C metrics with SEs."""
    head = cfg.get("head", {})
    res = {}
    per = {}
    for f, o in zip(folds, outs):
        P = o["P_eval"].astype(float)
        P = P / P.sum(1, keepdims=True)
        Pt = o.get("P_target")
        Pt = None if Pt is None else Pt.astype(float) / Pt.astype(float).sum(1, keepdims=True)
        Yp, Q, pi = apply_head(P, o["prior"], D.CM, head, Pt)
        per[f["name"]] = (f, Yp, pi)
    # ---- A ----
    A = [v for v in per.values() if v[0]["protocol"] == "A"]
    if A:
        rows = np.unique(np.concatenate([f["eval"] for f, _, _ in A]))
        pos = {r: i for i, r in enumerate(rows)}
        reps = sorted({f["repeat"] for f, _, _ in A})
        Ytrue = D.Y[rows]
        Ypreds = []
        for r in reps:
            Yp = np.zeros((len(rows), 5), int)
            for f, yp, _ in A:
                if f["repeat"] == r:
                    Yp[[pos[i] for i in f["eval"]]] = yp
            Ypreds.append(Yp)
        rep_metrics = [metrics(Ytrue, Yp) for Yp in Ypreds]
        rf = np.mean([row_f1(Ytrue, Yp) for Yp in Ypreds], axis=0)
        a = {k: float(np.mean([m[k] for m in rep_metrics])) for k in rep_metrics[0] if k not in ("n",)}
        a["sample_f1_se"] = float(rf.std(ddof=1) / np.sqrt(len(rf)))
        a["sample_f1_rep_sd"] = float(np.std([m["sample_f1"] for m in rep_metrics], ddof=1)) if len(reps) > 1 else 0.0
        a["n"] = int(len(rows))
        a["R"] = len(reps)
        if bootstrap:
            a.update(bootstrap_se(Ytrue, Ypreds, B=200))
        res["A"] = a
        res["rowf1_A"] = rf.tolist()
        res["rows_A"] = rows.tolist()
    # ---- B ----  (one or several chronological cuts; B = mean over cuts, B_se = mean per-cut SE
    #               because the cuts share most rows and are highly correlated)
    Bs = [v for v in per.values() if v[0]["protocol"] == "B"]
    if Bs:
        bm = []
        for f, yp, pi in Bs:
            Yt = D.Y[f["eval"]]
            m = metrics(Yt, yp)
            if bootstrap:
                m.update(bootstrap_se(Yt, [yp], B=200))
            m["cut"] = f["cut"]
            if pi is not None:
                m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
            m["true_prior_labels"] = Yt.mean(0).round(3).tolist()
            bm.append(m)
            res.setdefault("rowf1_B", []).extend(row_f1(Yt, yp).tolist())
        res["B_ncuts"] = len(bm)
        if len(bm) == 1:
            res["B"] = bm[0]
        else:
            agg = {k: float(np.mean([m[k] for m in bm])) for k in bm[0]
                   if isinstance(bm[0][k], (int, float)) and k != "cut"}
            agg["cuts"] = bm
            res["B"] = agg
        for m in bm:
            if m["cut"] == CHRONO_CUT:
                res["B2005"] = {k: v for k, v in m.items()}
    # ---- C ----
    Cs = [v for v in per.values() if v[0]["protocol"] == "C"]
    if Cs:
        f, yp, pi = Cs[0]
        Yt = D.Y[f["eval"]]
        m = metrics(Yt, yp)
        if bootstrap:
            m.update(bootstrap_se(Yt, [yp], B=200))
        if pi is not None:
            m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
        res["C"] = m
        res["rowf1_C"] = row_f1(Yt, yp).tolist()
    # ---- robust objective ----
    if "A" in res and "B" in res:
        res["J"] = 0.5 * res["A"]["sample_f1"] + 0.5 * res["B"]["sample_f1"]
        res["J_se"] = 0.5 * float(np.hypot(res["A"]["sample_f1_se"], res["B"]["sample_f1_se"]))
    res["secs"] = float(sum(float(o["secs"]) for o in outs))
    res["T_mean"] = float(np.mean([np.mean(o["T"]) for o in outs]))
    return res


def evaluate(cfg, D, protocols=("A", "B", "C"), R=4, n_jobs=1, seed=0, use_cache=True, rows=None,
             bootstrap=True, chrono_cuts=CHRONO_CUTS):
    validate(cfg)
    folds = make_folds(D, protocols, R, seed, rows=rows, chrono_cuts=chrono_cuts)
    outs = fold_outputs(cfg, D, folds, n_jobs, use_cache)
    return score(cfg, D, folds, outs, bootstrap=bootstrap)


def summary_row(name, res):
    r = {"name": name}
    for p in ("A", "B", "B2005", "C"):
        if p in res:
            r[f"{p}"] = res[p]["sample_f1"]
            r[f"{p}_se"] = res[p]["sample_f1_se"]
            r[f"{p}_macro"] = res[p]["macro_f1"]
    if "J" in res:
        r["J"] = res["J"]
        r["J_se"] = res["J_se"]
    r["secs"] = res.get("secs")
    return r


def paired_se(res1, res2, w=(0.5, 0.5)):
    """SE of the difference in J between two configs using per-row paired differences."""
    dA = np.asarray(res1["rowf1_A"]) - np.asarray(res2["rowf1_A"])
    dB = np.asarray(res1["rowf1_B"]) - np.asarray(res2["rowf1_B"])
    nB = len(dB) / res1.get("B_ncuts", 1)          # cuts overlap: effective n ~ one cut
    return float(np.sqrt((w[0] ** 2) * dA.var(ddof=1) / len(dA) + (w[1] ** 2) * dB.var(ddof=1) / nB))

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/evaluate.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/features.py

In [15]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/features.py
"""Preprocessors: standardize -> [PCA] -> [L2-normalize] on the embeddings, plus optional
non-embedding blocks.  All fitting uses only `ctx.unsup` rows (label-free), so the same
object is refit per fold when the feature-row set changes (inductive) and reused otherwise.

features config (all keys optional):
    {"scale": True, "pca": None|int, "whiten": False, "l2": False,
     "blocks": {"criteria": {"d": 32, "weight": 1.0}, "pathways": {...}, ...}}
"""
from __future__ import annotations

import numpy as np
from sklearn.decomposition import PCA

from .blocks import build_block
from .cache import memo

DEFAULT_FEATURES = {"scale": True, "pca": None, "l2": False, "blocks": {}}


def norm_features_cfg(cfg):
    c = dict(DEFAULT_FEATURES)
    c.update(cfg or {})
    c["blocks"] = dict(c.get("blocks") or {})
    return c


def _embed(D, cfg, rows):
    X = D.emb.copy()
    if cfg["scale"]:
        mu = X[rows].mean(0)
        sd = X[rows].std(0)
        X = (X - mu) / sd
    if cfg["pca"]:
        p = PCA(n_components=int(cfg["pca"]), whiten=bool(cfg.get("whiten", False)), random_state=0).fit(X[rows])
        X = p.transform(X)
    if cfg["l2"]:
        X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    return X


def featurize(ctx, cfg) -> np.ndarray:
    """Return the (n_all, d) representation for every global row."""
    cfg = norm_features_cfg(cfg)
    emb_cfg = {k: cfg[k] for k in ("scale", "pca", "l2", "whiten") if k in cfg}
    X = memo("embed", emb_cfg, ctx.unsup, lambda: _embed(ctx.D, cfg, ctx.unsup))
    if not cfg["blocks"]:
        return X
    parts = [X] if cfg.get("use_emb", True) else []
    for name in sorted(cfg["blocks"]):
        parts.append(build_block(ctx.D, name, cfg["blocks"][name], ctx.unsup))
    return np.hstack(parts)

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/features.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/graph.py

In [16]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/graph.py
"""Symmetric RBF-weighted kNN graph over the feature rows a fold may use, and
normalized-Laplacian diffusion (Zhou et al. 2004 label spreading; Huang et al. 2021 C&S).

graph config: {"space": features-cfg, "k": 10, "s": 1.0, "sym": "max"|"min"}
Weights w_ij = exp(-d_ij^2 / (s * median kNN distance)^2) on cosine-normalized vectors,
symmetrized with max (union) or min (mutual kNN), S = D^-1/2 W D^-1/2.
Diffusion solves (I - alpha S) F = (1 - alpha) F0 exactly (LU factor cached per alpha).
"""
from __future__ import annotations

import numpy as np
import scipy.sparse as sp
from scipy.linalg import lu_factor, lu_solve
from sklearn.neighbors import NearestNeighbors

from .cache import memo
from .features import featurize

DEFAULT_GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}


class Graph:
    def __init__(self, Z_nodes, nodes, n_all, k, s, sym):
        self.nodes = nodes
        self.pos = np.full(n_all, -1)
        self.pos[nodes] = np.arange(len(nodes))
        self.Zn = Z_nodes / np.maximum(np.linalg.norm(Z_nodes, axis=1, keepdims=True), 1e-12)
        self.nn = NearestNeighbors(n_neighbors=k + 1).fit(self.Zn)
        dist, ind = self.nn.kneighbors(self.Zn)
        dist, ind = dist[:, 1:], ind[:, 1:]
        self.sigma = s * np.median(dist)
        n = len(nodes)
        rows = np.repeat(np.arange(n), k)
        W = sp.csr_matrix((np.exp(-(dist.ravel() ** 2) / self.sigma ** 2), (rows, ind.ravel())), shape=(n, n))
        W = W.maximum(W.T) if sym == "max" else W.minimum(W.T)
        deg = np.asarray(W.sum(1)).ravel()
        iso = deg <= 0
        if iso.any():                      # isolated nodes (possible with mutual kNN): self-loop
            W = W + sp.diags(iso.astype(float))
            deg = np.asarray(W.sum(1)).ravel()
        Dm = sp.diags(1 / np.sqrt(deg))
        self.S = (Dm @ W @ Dm).toarray()
        self.k = k
        self._lu = {}

    def propagate(self, F0, alpha):
        if alpha <= 0:
            return F0.copy()
        if alpha not in self._lu:
            self._lu[alpha] = lu_factor(np.eye(len(self.S)) - alpha * self.S)
        return (1 - alpha) * lu_solve(self._lu[alpha], F0)

    def extend(self, Z_rows, F_nodes):
        """Inductive extension for rows that are not graph nodes: RBF-weighted kNN average."""
        Zn = Z_rows / np.maximum(np.linalg.norm(Z_rows, axis=1, keepdims=True), 1e-12)
        dist, ind = self.nn.kneighbors(Zn, n_neighbors=self.k)
        w = np.exp(-dist ** 2 / self.sigma ** 2) + 1e-12
        return np.einsum("nk,nkc->nc", w, F_nodes[ind]) / w.sum(1, keepdims=True)


def get_graph(ctx, gcfg) -> Graph:
    g = dict(DEFAULT_GRAPH)
    g.update(gcfg or {})
    Z = featurize(ctx, g["space"])

    def build():
        return Graph(Z[ctx.unsup], ctx.unsup, ctx.D.n, int(g["k"]), float(g["s"]), g["sym"])

    return memo("graph", {k: g[k] for k in ("space", "k", "s", "sym")}, ctx.unsup, build), Z


def node_values(graph, Z, F_nodes, rows):
    """Read propagated values for arbitrary global rows (extend for non-nodes)."""
    out = np.zeros((len(rows), F_nodes.shape[1]))
    p = graph.pos[rows]
    m = p >= 0
    out[m] = F_nodes[p[m]]
    if (~m).any():
        out[~m] = graph.extend(Z[rows[~m]], F_nodes)
    return out


def label_spread(ctx, gcfg, alpha, train_rows, y_train, K, out_rows, base_all=None):
    """Pure label spreading (base_all=None) or the Smooth step of C&S (base_all = (n_all,K)
    base probabilities).  Training nodes are clamped to their one-hot labels in F0."""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    F0 = np.zeros((n, K))
    if base_all is not None:
        F0 = base_all[graph.nodes].copy()
    p = graph.pos[train_rows]
    keep = p >= 0
    F0[p[keep]] = np.eye(K)[y_train[keep]]
    F = graph.propagate(F0, alpha)
    return node_values(graph, Z, F, out_rows)


def correct_and_smooth(ctx, gcfg, a1, a2, train_rows, y_train, P_oof_train, P_all, K, out_rows,
                       scale_mode="autoscale"):
    """Correct & Smooth (Huang et al. 2021).
    P_oof_train : base probabilities of the training rows, OUT-OF-FOLD (in-sample residuals ~0)
    P_all       : base probabilities for every global row (fit on all training rows)"""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    p = graph.pos[train_rows]
    keep = p >= 0
    Ytr = np.eye(K)[y_train]
    base = P_all[graph.nodes].copy()
    if a1 > 0:
        E0 = np.zeros((n, K))
        E0[p[keep]] = (Ytr - P_oof_train)[keep]
        E = graph.propagate(E0, a1)
        if scale_mode == "autoscale":
            sigma = np.abs(E0[p[keep]]).sum(1).mean()
            scale = sigma / np.maximum(np.abs(E).sum(1, keepdims=True), 1e-12)
            base = base + scale * E
        else:
            base = base + float(scale_mode) * E
        base = np.clip(base, 0, None)
        base = base / np.maximum(base.sum(1, keepdims=True), 1e-12)
    F0 = base.copy()
    F0[p[keep]] = Ytr[keep]
    F = graph.propagate(F0, a2)
    out = node_values(graph, Z, F, out_rows)
    out = np.clip(out, 0, None) + 1e-9
    return out / out.sum(1, keepdims=True)

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/graph.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/head.py

In [17]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/head.py
"""Shared head: temperature calibration, ONE prior correction (SLD-EM, MAP-stabilized),
and threshold-free decoding to an allowed label set with >= 1 label.

All probabilities are over the K allowed label combinations (label powerset).
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax

EPS = 1e-6


def safe_log(P):
    return np.log(np.clip(P, EPS, None))


def renorm(P):
    P = np.clip(P, 0, None) + 1e-12
    return P / P.sum(1, keepdims=True)


# ----------------------------------------------------------------------------- temperature
def fit_temperature(P_oof: np.ndarray, y: np.ndarray, w: np.ndarray | None = None,
                    bounds=(0.05, 20.0)) -> float:
    """One-parameter temperature T minimizing NLL of softmax(log P / T) on OOF probabilities."""
    L = safe_log(P_oof)
    w = np.ones(len(y)) if w is None else w

    def nll(logT):
        T = np.exp(logT)
        lp = log_softmax(L / T, axis=1)
        return -(w * lp[np.arange(len(y)), y]).sum() / w.sum()

    r = minimize_scalar(nll, bounds=(np.log(bounds[0]), np.log(bounds[1])), method="bounded",
                        options={"xatol": 1e-4})
    return float(np.exp(r.x))


def apply_temperature(P: np.ndarray, T: float) -> np.ndarray:
    if T is None or T == 1.0:
        return P
    return np.exp(log_softmax(safe_log(P) / T, axis=1))


# ----------------------------------------------------------------------------- SLD-EM
def sld_em(P: np.ndarray, prior_src: np.ndarray, a: float | None = 0.0, floor: float = 0.25,
           tol: float = 1e-8, max_iter: int = 1000, cap: float | None = None):
    """Saerens-Latinne-Decaestecker EM for the target label prior, with a Dirichlet(a*pi_s)
    pseudo-count toward the source prior (MAP) and a floor pi >= floor*pi_s.

    cap: optional ceiling pi_k <= cap * pi_s,k (guards the degenerate MLE in which a rare
    'bridge' combo such as Safety+Efficacy absorbs the mass of its parents).
    Returns (Q, pi_hat): prior-corrected posteriors and the estimated target prior.
    a=None or a=inf means 'off' (returns P unchanged)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6            # combos absent from training stay at exactly 0
    ps = np.where(present, prior_src, 0.0)
    ps = ps / ps.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ratio_den = np.where(present, ps, 1.0)
    pi = ps.copy()
    n = len(P)
    for _ in range(max_iter):
        Q = P * (pi / ratio_den)
        Q /= Q.sum(1, keepdims=True)
        new = (Q.sum(0) + a * ps) / (n + a)
        new = np.maximum(new, floor * ps)
        if cap:
            new = np.minimum(new, cap * ps)
        new /= new.sum()
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * (pi / ratio_den)
    Q /= Q.sum(1, keepdims=True)
    return Q, pi


def sld_em_factorized(P: np.ndarray, prior_src: np.ndarray, CM: np.ndarray, a: float | None = 0.0,
                      floor: float = 0.25, tol: float = 1e-8, max_iter: int = 1000):
    """Factorized SLD: the target/source combo-prior ratio is constrained to a product of
    per-label ratios, w_k = prod_j (pi_j/ps_j)^{CM_kj} ((1-pi_j)/(1-ps_j))^{1-CM_kj}.
    Only 5 free parameters (one per reason) instead of K-1, so a rare 'bridge' combo
    (Safety+Efficacy) cannot absorb the mass of its parents.  EM-style fixed point: E-step
    reweights combos by w, M-step sets each label prior to the mean marginal of Q (with the same
    Dirichlet pseudo-count a toward the source marginal and floor)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6
    ps_k = np.where(present, prior_src, 0.0)
    ps_k = ps_k / ps_k.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ps = np.clip(ps_k @ CM, 1e-6, 1 - 1e-6)            # source label marginals
    pi = ps.copy()
    n = len(P)

    def weights(pi):
        lw = np.log(pi / ps) @ CM.T + np.log((1 - pi) / (1 - ps)) @ (1 - CM).T
        return np.exp(lw - lw.max())

    for _ in range(max_iter):
        Q = P * weights(pi)
        Q /= Q.sum(1, keepdims=True)
        m = Q @ CM                                          # (n, 5) label marginals
        new = (m.sum(0) + a * ps) / (n + a)
        new = np.clip(np.maximum(new, floor * ps), 1e-6, 1 - 1e-6)
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * weights(pi)
    Q /= Q.sum(1, keepdims=True)
    pi_k = ps_k * weights(pi)
    return Q, pi_k / pi_k.sum()


# ----------------------------------------------------------------------------- decoding
_SUBSETS = np.array([[(s >> j) & 1 for j in range(5)] for s in range(1, 32)])


def _f1_matrix(CM):
    inter = _SUBSETS @ CM.T                                   # (31, K)
    return 2 * inter / (_SUBSETS.sum(1)[:, None] + CM.sum(1)[None, :])


def decode(P: np.ndarray, CM: np.ndarray, method: str = "map") -> np.ndarray:
    """Map combo posteriors to a binary label matrix (n, 5); never empty.
    map   : the most probable allowed combination
    expf1 : Bayes-optimal subset for expected sample-F1 over the 31 non-empty subsets"""
    if method == "map":
        return CM[P.argmax(1)]
    if method == "expf1":
        F = _f1_matrix(CM)
        return _SUBSETS[(P @ F.T).argmax(1)]
    raise ValueError(method)


def apply_head(P_eval, prior_src, CM, head: dict, P_target=None):
    """SLD (optional) + decoding.  P_target: posteriors of the SLD target rows (defaults to the
    eval rows, i.e. transductive).  Returns (Ypred, Q_eval, pi_hat)."""
    head = head or {}
    pi_hat = None
    Q = P_eval
    if head.get("prior", "none") == "sld":
        a = head.get("sld_a", 0.0)
        floor = head.get("sld_floor", 0.25)
        cap = head.get("sld_cap")
        if head.get("sld_mode", "combo") == "factorized":
            em = lambda P_: sld_em_factorized(P_, prior_src, CM, a=a, floor=floor)
        else:
            em = lambda P_: sld_em(P_, prior_src, a=a, floor=floor, cap=cap)
        if P_target is None or P_target is P_eval:
            Q, pi_hat = em(P_eval)
        else:
            _, pi_hat = em(P_target)
            present = prior_src > 1e-6
            Q = renorm(np.where(present, P_eval * (pi_hat / np.where(present, prior_src, 1.0)), 0.0))
    return decode(Q, CM, head.get("decode", "map")), Q, pi_hat

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/head.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/labelers.py

In [18]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/labelers.py
"""Labelers: fill the missing labels of the unlabeled pool with soft combo distributions.

Every labeler type exposes
    soft_labels(lcfg, ctx, rows) -> Q (len(rows), K)      for rows NOT in ctx.train
and, where cheap and exact, a leave-one-out version for training rows (used when the labeler's
output is a synthetic feature or a direct predictor that needs OOF scores):
    soft_labels_train_oof(lcfg, ctx) -> Q (len(ctx.train), K)

`pseudo_labels(lcfg, ctx, clf_cfg)` turns soft labels into weighted pseudo-rows
(rows, Q, w) for the pipeline's classifier (soft-label replication, weight lam * q_ik).

Types
  cluster   : cluster-then-label (kmeans | gmm | seeded_kmeans | hdbscan), m-smoothed map   [T4]
  spread    : label spreading on the kNN graph (Zhou et al. 2004)                            [T2]
  cs        : Correct & Smooth on the kNN graph (Huang et al. 2021)                          [T2]
  sld       : calibrated base classifier + SLD-EM on the pool (Saerens et al. 2002)          [T1]
  selftrain : guarded self-training (agreement, alignment, soft weights, ramp)               [T3]
"""
from __future__ import annotations

import numpy as np
from sklearn.cluster import HDBSCAN, KMeans, kmeans_plusplus
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold

from .cache import memo, memo_ctx, rows_key
from .classifiers import make_classifier
from .features import featurize
from .graph import correct_and_smooth, label_spread
from .head import apply_temperature, fit_temperature, sld_em, sld_em_factorized

DEFAULT_CLUSTER_SPACE = {"scale": True, "pca": 32, "l2": True}


# ============================================================================ clustering
def _cluster_fit(D, algo, k, Z, rows, cfg, seed=0):
    info = {}
    if algo == "kmeans":
        km = KMeans(n_clusters=k, n_init=int(cfg.get("n_init", 10)), max_iter=300, tol=1e-4,
                    random_state=seed).fit(Z[rows])
        dist = km.transform(Z)
        info.update(inertia=float(km.inertia_), n_iter=int(km.n_iter_))
        return {"dist": dist, "labels": km.predict(Z), "info": info}
    if algo == "gmm":
        g = GaussianMixture(n_components=k, covariance_type=cfg.get("cov", "diag"), reg_covar=1e-3,
                            n_init=int(cfg.get("n_init", 3)), max_iter=500, tol=1e-4,
                            init_params="k-means++", random_state=seed).fit(Z[rows])
        info.update(converged=bool(g.converged_), n_iter=int(g.n_iter_), bic=float(g.bic(Z[rows])),
                    lower_bound=float(g.lower_bound_))
        return {"resp": g.predict_proba(Z), "labels": g.predict(Z), "info": info}
    if algo == "hdbscan":
        h = HDBSCAN(min_cluster_size=int(cfg.get("min_cluster_size", 10)),
                    min_samples=cfg.get("min_samples")).fit(Z[rows])
        lab = np.full(D.n, -1)
        lab[rows] = h.labels_
        others = np.setdiff1d(np.arange(D.n), rows)
        if len(others):                     # inductive: attach to the nearest fitted point
            Zr = Z[rows]
            nn = np.argmax(Z[others] @ Zr.T / (np.linalg.norm(Z[others], axis=1)[:, None] *
                                                np.linalg.norm(Zr, axis=1)[None] + 1e-12), axis=1)
            lab[others] = h.labels_[nn]
        info.update(n_clusters=int(h.labels_.max() + 1), noise_frac=float((h.labels_ < 0).mean()))
        return {"labels": lab, "info": info}
    raise ValueError(algo)


def cluster_resp(lcfg, ctx):
    """(n_all, C) cluster membership (hard one-hot, kmeans-soft, or GMM responsibilities)."""
    algo = lcfg.get("algo", "kmeans")
    k = int(lcfg.get("k", 24))
    space = lcfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    if algo == "seeded_kmeans":                 # label-dependent init -> not shared across folds
        y = ctx.y_train
        cls = np.unique(y)
        seeds = np.array([Z[ctx.train][y == c].mean(0) for c in cls])
        extra = max(k - len(cls), 0)
        if extra:
            more, _ = kmeans_plusplus(Z[ctx.unsup], n_clusters=extra, random_state=ctx.seed)
            seeds = np.vstack([seeds, more])
        km = KMeans(n_clusters=len(seeds), init=seeds, n_init=1, max_iter=300).fit(Z[ctx.unsup])
        lab = km.predict(Z)
        return np.eye(len(seeds))[lab]
    cfg_key = {"algo": algo, "k": k, "space": space, **{kk: lcfg[kk] for kk in ("cov", "n_init", "min_cluster_size", "min_samples") if kk in lcfg}}
    fit = memo("cluster", cfg_key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, lcfg))
    if algo == "gmm":
        return fit["resp"]
    if algo == "kmeans" and lcfg.get("soft_temp"):
        d2 = fit["dist"] ** 2
        T = float(lcfg["soft_temp"]) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    lab = fit["labels"]
    C = int(max(lab.max() + 1, 1))
    R = np.zeros((ctx.D.n, C))
    R[lab >= 0, lab[lab >= 0]] = 1.0          # hdbscan noise rows keep all-zero membership
    return R


def _cluster_map(R_train, y, K, m, prior):
    N = R_train.T @ np.eye(K)[y]                    # (C, K) label mass per cluster
    return N, (N + m * prior) / (N.sum(1, keepdims=True) + m)


def cluster_soft(lcfg, ctx, rows):
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)
    y = ctx.y_train
    prior = np.bincount(y, minlength=K) / len(y)
    _, pi = _cluster_map(R[ctx.train], y, K, float(lcfg.get("m", 4)), prior)
    Q = R[rows] @ pi
    mass = R[rows].sum(1, keepdims=True)
    Q = Q + (1 - np.clip(mass, 0, 1)) * prior        # noise / empty -> global prior
    return Q / Q.sum(1, keepdims=True)


def cluster_soft_train_oof(lcfg, ctx):
    """Exact leave-one-out cluster map for training rows."""
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)[ctx.train]
    y = ctx.y_train
    m = float(lcfg.get("m", 4))
    prior = np.bincount(y, minlength=K) / len(y)
    N, _ = _cluster_map(R, y, K, m, prior)
    nc = N.sum(1)                                    # (C,)
    out = np.zeros((len(y), K))
    for i in range(len(y)):
        Ni = N - np.outer(R[i], np.eye(K)[y[i]])
        pi = (Ni + m * prior) / (nc - R[i] + m)[:, None]
        q = R[i] @ pi
        q = q + (1 - min(R[i].sum(), 1)) * prior
        out[i] = q / q.sum()
    return out


# ============================================================================ graph labelers
def spread_soft(lcfg, ctx, rows):
    K = ctx.D.K
    F = label_spread(ctx, lcfg.get("graph"), float(lcfg.get("alpha", 0.5)), ctx.train, ctx.y_train, K, rows)
    prior = np.bincount(ctx.y_train, minlength=K) / len(ctx.train)
    F = np.clip(F, 0, None)
    s = F.sum(1, keepdims=True)
    F = np.where(s > 1e-12, F / np.maximum(s, 1e-12), prior)   # unreachable rows -> prior
    beta = float(lcfg.get("beta", 0.01))
    F = (F + beta * prior) / (1 + beta)
    return F


def _base_oof_and_all(base_cfg, ctx, X, n_inner=5):
    """Base classifier probabilities: OOF for training rows, full-fit for every other row."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    P_oof = np.zeros((len(y), K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
        P_oof[te] = m.predict_proba(X[ctx.train[te]])
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P_all = m.predict_proba(X)
    return P_oof, P_all


def cs_soft(lcfg, ctx, rows):
    K = ctx.D.K
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    P_oof, P_all = _base_oof_and_all(base_cfg, ctx, X)
    return correct_and_smooth(ctx, lcfg.get("graph"), float(lcfg.get("a1", 0.5)), float(lcfg.get("a2", 0.5)),
                              ctx.train, ctx.y_train, P_oof, P_all, K, rows)


# ============================================================================ SLD labeler
def calibrated_base(base_cfg, ctx, X, rows, temperature=True):
    """Fit base classifier on ctx.train, temperature-scale with inner OOF, predict rows."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P = m.predict_proba(X[rows])
    T = 1.0
    if temperature:
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            mm = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
            P_oof[te] = mm.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    return P, m.prior_, T


def sld_soft(lcfg, ctx, rows):
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    P, prior, _ = calibrated_base(base_cfg, ctx, X, rows, lcfg.get("temperature", False))
    if lcfg.get("mode", "combo") == "factorized":
        Q, pi = sld_em_factorized(P, prior, ctx.D.CM, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    else:
        Q, pi = sld_em(P, prior, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    return Q


# ============================================================================ self-training
def _sinkhorn(P, target, iters=200):
    Q = P.copy()
    n = len(Q)
    for _ in range(iters):
        col = Q.sum(0)
        Q = Q * (target * n / np.maximum(col, 1e-12))
        Q = Q / Q.sum(1, keepdims=True)
    return Q


def _softmatch_weight(conf):
    mu, sd = conf.mean(), conf.std() + 1e-6
    return np.where(conf >= mu, 1.0, np.exp(-((conf - mu) ** 2) / (2 * sd ** 2)))


def selftrain_pseudo(lcfg, ctx, clf_cfg, X, pool_rows):
    """Guarded self-training.  Returns (rows, Q, w, info) for the final pseudo-labeled set."""
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    Ytr = np.eye(K)[y]
    R = int(lcfg.get("rounds", 2))
    lam_max = float(lcfg.get("lam", 0.3))
    members = lcfg.get("members", ["lr"])
    align = lcfg.get("align", "none")
    tau = lcfg.get("tau", 0.75)
    top_p = lcfg.get("top_p")
    hard = bool(lcfg.get("hard", False))
    weighting = lcfg.get("weighting", "hard")
    # members that do not use pseudo-labels are computed once
    static = []
    if "lda" in members:
        static.append(make_classifier({"type": "lda"}, K, CM).fit(X[ctx.train], y).predict_proba(X[pool_rows]))
    if "cs" in members:
        static.append(cs_soft({"a1": 0.5, "a2": 0.5}, ctx, pool_rows))
    cl_major = None
    if lcfg.get("cluster_filter"):
        cl_major = cluster_soft({"algo": "kmeans", "k": int(lcfg.get("cluster_k", 24)), "m": 1}, ctx, pool_rows).argmax(1)
    sel_rows, Q, w = pool_rows[:0], np.zeros((0, K)), np.zeros(0)
    prev_lab = None
    info = {}
    for r in range(1, R + 1):
        lam_r = lam_max * (r / R if lcfg.get("ramp", False) else 1.0)
        Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[sel_rows], Q, w)
        main = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
        Pm_list = [main.predict_proba(X[pool_rows])] + static
        Pm = np.mean(Pm_list, axis=0)
        prior_src = np.bincount(y, minlength=K) / len(y)
        if align == "sld":
            Pm, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
        elif align == "sinkhorn":
            _, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
            Pm = _sinkhorn(Pm, pi)
        conf = Pm.max(1)
        lab = Pm.argmax(1)
        if top_p:
            keep = np.zeros(len(lab), bool)
            for c in range(K):
                mc = np.where(lab == c)[0]
                if len(mc):
                    thr = np.quantile(conf[mc], 1 - float(top_p))
                    keep[mc[conf[mc] >= thr]] = True
        else:
            keep = conf >= float(tau)
        if len(Pm_list) > 1:
            agree = np.sum([P.argmax(1) == lab for P in Pm_list], axis=0)
            keep &= agree >= 2
        if cl_major is not None:
            keep &= cl_major == lab
        g = _softmatch_weight(conf) if weighting == "softmatch" else np.ones(len(conf))
        sel_rows = pool_rows[keep]
        Q = np.eye(K)[lab[keep]] if hard else Pm[keep]
        w = lam_r * g[keep]
        info = {"round": r, "n_selected": int(keep.sum()), "lam": lam_r}
        if prev_lab is not None and np.mean(prev_lab != lab) < 0.01:
            break
        prev_lab = lab
    return sel_rows, Q, w, info


# ============================================================================ dispatch
SOFT = {"cluster": cluster_soft, "spread": spread_soft, "cs": cs_soft, "sld": sld_soft}


_POST = ("lam", "tau", "hard", "pool")      # applied after soft labels -> not part of the memo key


def soft_labels(lcfg, ctx, rows):
    rows = np.asarray(rows)
    core = {k: v for k, v in lcfg.items() if k not in _POST}
    return memo_ctx("soft", core, ctx, lambda: SOFT[lcfg["type"]](lcfg, ctx, rows), extra=rows.tolist())


def soft_labels_train_oof(lcfg, ctx, n_inner=5):
    """OOF soft labels for the training rows (exact LOO for cluster; inner K-fold otherwise)."""
    if lcfg["type"] == "cluster":
        core = {k: v for k, v in lcfg.items() if k not in _POST}
        return memo_ctx("soft_oof", core, ctx, lambda: cluster_soft_train_oof(lcfg, ctx))
    from .pipeline import inner_folds
    y = ctx.y_train
    out = np.zeros((len(y), ctx.D.K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        sub = ctx.sub(ctx.train[tr], ctx.train[te])
        out[te] = soft_labels(lcfg, sub, ctx.train[te])
    return out


def pool_rows_for(lcfg, ctx):
    if lcfg.get("pool", "unl") == "all" and ctx.transductive:
        rows = np.union1d(np.union1d(ctx.pool, ctx.eval), ctx.D.idx["test"])
        return np.setdiff1d(rows, ctx.train)
    return np.setdiff1d(ctx.pool, ctx.train)


def pseudo_labels(lcfg, ctx, clf_cfg, X):
    """Weighted pseudo-rows (rows, Q, w, info) for the pipeline classifier."""
    rows = pool_rows_for(lcfg, ctx)
    if lcfg["type"] == "selftrain":
        return selftrain_pseudo(lcfg, ctx, clf_cfg, X, rows)
    Q = soft_labels(lcfg, ctx, rows)
    conf = Q.max(1)
    keep = conf >= float(lcfg.get("tau", 0.0))
    if lcfg.get("hard", False):
        Q = np.eye(ctx.D.K)[Q.argmax(1)]
    lam = float(lcfg.get("lam", 0.1))
    return rows[keep], Q[keep], np.full(keep.sum(), lam), {"n_selected": int(keep.sum())}


def expand(X_l, Y_l, w_l, X_u, Q_u, w_u, min_q=0.05):
    """Soft-label replication: one copy of each row per combo with weight w * q (exact
    expected cross-entropy for LR).  Y_l may be one-hot or soft."""
    Xs, ys, ws = [], [], []
    for Xb, Yb, wb in ((X_l, Y_l, w_l), (X_u, Q_u, w_u)):
        if len(Xb) == 0:
            continue
        i, k = np.where(Yb >= min_q)
        Xs.append(Xb[i])
        ys.append(k)
        ws.append(wb[i] * Yb[i, k])
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ws)

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/labelers.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/pipeline.py

In [19]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/pipeline.py
"""Pipeline composition from config dicts.

A *single* pipeline config:
    {
      "features":   {"scale": True, "pca": 64, "l2": False, "blocks": {...}} | None,
      "synth":      [ {"type": "gmm", "k": 16}, {"type": "knn_vote", "k": 10}, ... ],
      "labeler":    {"type": "cluster"|"spread"|"cs"|"sld"|"selftrain", "lam": 0.1, ...} | None,
      "clf":        {"type": "lr"|"ovr"|"lda"|"knn"|"hgb"|"nb"|"direct", ...},
      "temperature": True,
    }
An *ensemble*:  {"ensemble": [single, single, ...], "weights": None | [w1, w2, ...]}
A *stack* is a single pipeline whose synth list holds {"type": "oof_model", "model": single}
entries (nested OOF meta-features) and whose features are None (or a small base block).

A full experiment config adds the head and the transductive toggle:
    {"model": <single|ensemble>, "head": {"prior": "none"|"sld", "sld_a": 100,
     "decode": "map"|"expf1"}, "transductive": True}

run_model(model_cfg, ctx, rows) -> {"P": calibrated combo probs for rows, "prior": pi_s, ...}
Only labels of ctx.train are ever read.
"""
from __future__ import annotations

import copy
import warnings

import numpy as np
from sklearn.model_selection import StratifiedKFold

from .classifiers import make_classifier
from .features import featurize
from .head import apply_temperature, fit_temperature


def _mask_absent(P, prior):
    """Combos with zero training mass keep probability exactly 0 (temperature/log clipping
    would otherwise resurrect them and SLD would then amplify them)."""
    present = np.asarray(prior) > 1e-6
    if present.all():
        return P
    P = np.where(present[None, :], P, 0.0)
    return P / np.maximum(P.sum(1, keepdims=True), 1e-300)
from .labelers import expand, pseudo_labels, soft_labels, soft_labels_train_oof
from .synth import build_synth_block

warnings.filterwarnings("ignore", message="The least populated class")


def inner_folds(y, n_splits=5, seed=0):
    n_splits = int(min(n_splits, max(2, np.bincount(y).max())))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(y)), y))


def _design(cfg, ctx):
    """Base features + synthetic features for every global row."""
    parts = []
    if cfg.get("features") is not None:
        parts.append(featurize(ctx, cfg["features"]))
    S = build_synth_block(cfg.get("synth"), ctx)
    if S is not None:
        parts.append(S)
    if not parts:
        raise ValueError("pipeline has neither base features nor synthetic features")
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def run_single(cfg, ctx, rows):
    rows = np.asarray(rows)
    D = ctx.D
    K, CM = D.K, D.CM
    y = ctx.y_train
    clf_cfg = cfg["clf"]
    info = {}

    # ---- direct labeler-as-predictor (cluster / spread / cs / sld) ----
    if clf_cfg["type"] == "direct":
        lcfg = clf_cfg["labeler"]
        P = soft_labels(lcfg, ctx, rows)
        prior = np.bincount(y, minlength=K) / len(y)
        T = 1.0
        if cfg.get("temperature", True):
            P_oof = soft_labels_train_oof(lcfg, ctx)
            T = fit_temperature(P_oof, y)
            P = apply_temperature(P, T)
        return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}

    X = _design(cfg, ctx)
    Ytr = np.eye(K)[y]
    # ---- labeler -> weighted pseudo rows from the unlabeled pool ----
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    if cfg.get("labeler"):
        pr, Q, w, linfo = pseudo_labels(cfg["labeler"], ctx, clf_cfg, X)
        info["labeler"] = linfo
    Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[pr], Q, w)
    clf = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
    P = clf.predict_proba(X[rows])
    prior = clf.prior_
    T = 1.0
    if cfg.get("temperature", True):
        # classifier-stage OOF: pseudo rows fixed, inner folds over the labeled rows
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            Xi, yi, wi = expand(X[ctx.train[tr]], Ytr[tr], np.ones(len(tr)), X[pr], Q, w)
            m = make_classifier(clf_cfg, K, CM).fit(Xi, yi, wi)
            P_oof[te] = m.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    info["n_pseudo_rows"] = int(len(pr))
    return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}


def run_model(model_cfg, ctx, rows):
    if "ensemble" in model_cfg:
        members = model_cfg["ensemble"]
        wts = np.asarray(model_cfg.get("weights") or [1.0] * len(members), float)
        wts = wts / wts.sum()
        outs = [run_single(m, ctx, rows) for m in members]
        P = sum(wi * o["P"] for wi, o in zip(wts, outs))
        prior = sum(wi * o["prior"] for wi, o in zip(wts, outs))
        return {"P": P, "prior": prior, "T": [o["T"] for o in outs], "info": {"members": [o["info"] for o in outs]},
                "member_P": [o["P"] for o in outs], "member_prior": [o["prior"] for o in outs]}
    return run_single(model_cfg, ctx, rows)


# ----------------------------------------------------------------------------- validation
def final_stage_singles(model_cfg):
    return model_cfg["ensemble"] if "ensemble" in model_cfg else [model_cfg]


def validate(cfg):
    """Enforce the 'only ONE prior correction' rule and basic sanity."""
    model = cfg["model"]
    head = cfg.get("head", {})
    corrections = set()
    for s in final_stage_singles(model):
        clf = s["clf"]
        if clf.get("class_weight") == "balanced":
            corrections.add("class_weight=balanced")
        lab = s.get("labeler") or {}
        if lab.get("type") == "selftrain" and lab.get("align", "none") != "none" and clf.get("class_weight") == "balanced":
            raise ValueError("self-training alignment + balanced class weights = double prior correction")
    if head.get("prior", "none") == "sld":
        corrections.add("sld")
    if len(corrections) > 1:
        raise ValueError(f"more than one prior correction: {sorted(corrections)}")
    return sorted(corrections)


def model_part(cfg):
    """Config without the post-hoc head (SLD / decoding) -> cache key for fold outputs."""
    return {"model": cfg["model"], "transductive": cfg.get("transductive", True)}


def deep(cfg):
    return copy.deepcopy(cfg)

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/pipeline.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/search.py

In [20]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/search.py
"""Optuna TPE search over the composed config space, robust objective, 1-SE selection and the
nested search-overfitting check.

Objective  J = 0.5 * A + 0.5 * B   (A = repeated stratified CV sample-F1, B = mean chronological
holdout sample-F1 over cuts 2004/2005/2006).  C (official val) is logged, never optimized.
Selection  = one-standard-error rule on J, then the SIMPLEST config (fewest active components),
then a guard: B must not be worse than the plain-LR baseline's B by more than one SE.
"""
from __future__ import annotations

import copy
import fcntl
import json
import os
import time
import traceback

import numpy as np

from .evaluate import evaluate, make_folds, fold_outputs, score, CHRONO_CUTS
from .pipeline import validate

GRAPH_SPACE = {"scale": True, "pca": 64, "l2": True}


# ============================================================================ search space
def suggest_config(t, allow_blocks=True, allow_stack=False, require_unsup=False):
    """Define-by-run space.  Returns a full experiment config."""
    pca = t.suggest_categorical("pca", [0, 32, 64, 96])
    l2 = t.suggest_categorical("l2", [False, True])
    feats = {"scale": True, "pca": pca or None, "l2": l2, "blocks": {}}
    if allow_blocks:
        if t.suggest_categorical("blk_criteria", [False, True]):
            feats["blocks"]["criteria"] = {"d": 16}
        if t.suggest_categorical("blk_drug", [False, True]):
            feats["blocks"]["pathways"] = {"d": 16, "max_level": 2}
            feats["blocks"]["drugmeta"] = {}

    prior = t.suggest_categorical("prior", ["none", "sld"])
    head = {"prior": prior, "decode": t.suggest_categorical("decode", ["map", "expf1"])}
    if prior == "sld":
        head["sld_a"] = t.suggest_categorical("sld_a", [0, 25, 100, 400])
        head["sld_floor"] = 0.25

    clf_t = t.suggest_categorical("clf", ["lr", "ovr", "lda", "knn"] if require_unsup else ["lr", "ovr", "lda", "knn", "cs"])
    temperature = t.suggest_categorical("temperature", [False, True])
    if clf_t == "cs":       # Correct & Smooth as the predictor (T2); base = powerset LR
        gk = t.suggest_categorical("cs_graph_k", [5, 10, 20])
        lcfg = {"type": "cs", "graph": {"space": GRAPH_SPACE, "k": gk, "s": 1.0, "sym": "max"},
                "a1": t.suggest_categorical("cs_a1", [0.0, 0.5, 0.8]),
                "a2": t.suggest_float("cs_a2", 0.1, 0.9),
                "base": {"type": "lr", "C": t.suggest_float("C", 0.01, 3.0, log=True)},
                "base_features": {"scale": True, "pca": pca or None}}
        model = {"features": None, "synth": [], "labeler": None,
                 "clf": {"type": "direct", "labeler": lcfg}, "temperature": temperature}
        return {"model": model, "head": head, "transductive": True}

    clf = {"type": clf_t}
    if clf_t in ("lr", "ovr"):
        clf["C"] = t.suggest_float("C", 0.01, 3.0, log=True)
    if clf_t == "knn":
        clf.update(k=t.suggest_categorical("knn_k", [3, 5, 10, 20]), weights="distance",
                   tau=t.suggest_categorical("knn_tau", [0.05, 0.1, 0.3]), beta=1.0)
    if prior != "sld" and clf_t in ("lr", "ovr", "lda"):
        if t.suggest_categorical("class_weight", ["none", "balanced"]) == "balanced":
            clf["class_weight"] = "balanced"

    synth = []
    if t.suggest_categorical("syn_gmm", [False, True]):
        synth.append({"type": "gmm", "k": t.suggest_categorical("syn_gmm_k", [8, 16, 32]), "cov": "diag",
                      "space": {"scale": True, "pca": 32, "l2": True}})
    if t.suggest_categorical("syn_kmeans", [False, True]):
        synth.append({"type": "kmeans", "k": t.suggest_categorical("syn_kmeans_k", [8, 16, 32]),
                      "out": t.suggest_categorical("syn_kmeans_out", ["dist", "soft"])})
    if t.suggest_categorical("syn_knn", [False, True]):
        synth.append({"type": "knn_vote", "k": t.suggest_categorical("syn_knn_k", [5, 10, 20])})
    if t.suggest_categorical("syn_graph", [False, True]):
        synth.append({"type": "graph", "mode": t.suggest_categorical("syn_graph_mode", ["spread", "cs"]),
                      "graph": {"space": GRAPH_SPACE, "k": 10, "s": 1.0, "sym": "max"},
                      "alpha": t.suggest_float("syn_graph_alpha", 0.2, 0.9)})
    if t.suggest_categorical("syn_clmap", [False, True]):
        synth.append({"type": "cluster_map", "algo": "kmeans", "k": t.suggest_categorical("syn_clmap_k", [8, 16, 32]), "m": 4})
    if t.suggest_categorical("syn_oof", [False, True]):
        synth.append({"type": "oof_model", "model": {"features": {"scale": True}, "clf": {"type": "lda"}, "temperature": True}})

    lab_t = t.suggest_categorical("labeler", ["cluster", "spread", "cs", "selftrain"] if require_unsup else
                                  ["none", "cluster", "spread", "cs", "sld", "selftrain"])
    labeler = None
    if lab_t != "none":
        lam = t.suggest_float("lam", 0.03, 0.5, log=True)
        if lab_t == "cluster":
            labeler = {"type": "cluster", "algo": t.suggest_categorical("cl_algo", ["kmeans", "gmm", "seeded_kmeans"]),
                       "k": t.suggest_categorical("cl_k", [8, 16, 24, 32, 48]), "m": t.suggest_categorical("cl_m", [1, 4]),
                       "lam": lam}
        elif lab_t in ("spread", "cs"):
            g = {"space": GRAPH_SPACE, "k": t.suggest_categorical("lab_graph_k", [5, 10, 20]), "s": 1.0, "sym": "max"}
            labeler = {"type": lab_t, "graph": g, "lam": lam}
            if lab_t == "spread":
                labeler["alpha"] = t.suggest_float("lab_alpha", 0.2, 0.9)
            else:
                labeler.update(a1=t.suggest_categorical("lab_a1", [0.0, 0.5, 0.8]), a2=t.suggest_float("lab_a2", 0.2, 0.9))
        elif lab_t == "sld":
            labeler = {"type": "sld", "base": {"type": "lr", "C": 0.1}, "a": t.suggest_categorical("lab_sld_a", [25, 100, 400]),
                       "lam": lam}
        elif lab_t == "selftrain":
            labeler = {"type": "selftrain", "lam": lam,
                       "members": t.suggest_categorical("st_members", ["lr", "lr+lda+cs"]).split("+"),
                       "rounds": t.suggest_categorical("st_rounds", [1, 2, 3]),
                       "tau": t.suggest_categorical("st_tau", [0.6, 0.75, 0.9]),
                       "align": t.suggest_categorical("st_align", ["none", "sld"]) if clf.get("class_weight") != "balanced" else "none",
                       "weighting": t.suggest_categorical("st_weighting", ["hard", "softmatch"]),
                       "ramp": t.suggest_categorical("st_ramp", [False, True]),
                       "cluster_filter": t.suggest_categorical("st_cluster_filter", [False, True])}
            if require_unsup and "cs" not in labeler["members"]:
                labeler["cluster_filter"] = True
        if lab_t != "selftrain" and t.suggest_categorical("lab_hard", [False, True]):
            labeler["hard"] = True
            labeler["tau"] = t.suggest_categorical("lab_tau", [0.5, 0.7, 0.9])
    model = {"features": feats, "synth": synth, "labeler": labeler, "clf": clf, "temperature": temperature}
    return {"model": model, "head": head, "transductive": True}


# ============================================================================ complexity
CLF_COST = {"lr": 0.0, "lda": 0.0, "ovr": 0.25, "nb": 0.25, "knn": 0.5, "direct": 1.0, "hgb": 2.0}


def complexity(cfg) -> float:
    """Number of active components (+ small costs for extra knobs).  Lower = simpler."""
    m = cfg["model"]
    singles = m["ensemble"] if "ensemble" in m else [m]
    c = 0.0
    for s in singles:
        c += len(s.get("synth") or [])
        for sy in s.get("synth") or []:
            if sy["type"] == "oof_model":
                c += complexity({"model": sy["model"], "head": {}})
        lab = s.get("labeler")
        if lab:
            c += 1.0 + (0.5 * (len(lab.get("members", [])) > 1) + 0.25 * (lab.get("rounds", 1) - 1)
                        if lab["type"] == "selftrain" else 0.0)
        f = s.get("features") or {}
        c += len(f.get("blocks") or {})
        clf = s["clf"]
        c += CLF_COST.get(clf["type"], 1.0)
        if clf.get("class_weight") == "balanced":
            c += 0.5
        if s.get("temperature"):
            c += 0.25
    if "ensemble" in m:
        c += len(singles) - 1
    if cfg.get("head", {}).get("prior") == "sld":
        c += 0.5
    return c


def unsup_labeling(cfg) -> bool:
    """Assignment rule: an UNSUPERVISED method must take part in filling the missing labels.
    True if the label-filling path uses clustering or a similarity graph: a cluster / label-
    spreading / C&S labeler, or guarded self-training with the K-means consistency filter or a
    C&S member, or a direct cluster/graph predictor."""
    m = cfg["model"]
    for s in (m["ensemble"] if "ensemble" in m else [m]):
        lab = s.get("labeler") or {}
        if lab.get("type") in ("cluster", "spread", "cs"):
            return True
        if lab.get("type") == "selftrain" and (lab.get("cluster_filter") or "cs" in lab.get("members", [])):
            return True
        if s["clf"]["type"] == "direct" and s["clf"]["labeler"]["type"] in ("cluster", "spread", "cs"):
            return True
    return False


# ============================================================================ objective
def append_jsonl(path, rec):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    line = json.dumps(rec, default=float) + "\n"
    with open(path, "a") as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        f.write(line)
        f.flush()
        fcntl.flock(f, fcntl.LOCK_UN)


def make_objective(D, log_path, R=4, rows=None, chrono_cuts=CHRONO_CUTS, prune_margin=0.05,
                   allow_blocks=True, cv_seed=0, tag="main", require_unsup=False):
    import optuna

    def objective(trial):
        cfg = suggest_config(trial, allow_blocks=allow_blocks, require_unsup=require_unsup)
        validate(cfg)
        t0 = time.time()
        # stage 1: chrono cuts + val + first CV repeat; prune clearly bad configs
        folds1 = make_folds(D, ("A", "B", "C"), R=R, seed=cv_seed, rows=rows, chrono_cuts=chrono_cuts)
        first = [f for f in folds1 if f["protocol"] != "A" or f["repeat"] == 0]
        rest = [f for f in folds1 if f["protocol"] == "A" and f["repeat"] > 0]
        try:
            outs1 = fold_outputs(cfg, D, first, n_jobs=1)
            r1 = score(cfg, D, first, outs1, bootstrap=False)
            j1 = r1.get("J", np.nan)
            trial.set_user_attr("J_stage1", float(j1))
            try:
                best = trial.study.best_value
            except ValueError:
                best = -np.inf
            if np.isfinite(best) and j1 < best - prune_margin:
                append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "pruned", "J_stage1": j1,
                                        "params": trial.params, "config": cfg, "secs": time.time() - t0})
                raise optuna.TrialPruned()
            outs2 = fold_outputs(cfg, D, rest, n_jobs=1)
            res = score(cfg, D, first + rest, outs1 + outs2, bootstrap=False)
        except optuna.TrialPruned:
            raise
        except Exception as e:
            append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "fail", "error": repr(e),
                                    "tb": traceback.format_exc()[-2000:], "params": trial.params, "config": cfg})
            raise
        cx = complexity(cfg)
        attrs = {"A": res["A"]["sample_f1"], "A_se": res["A"]["sample_f1_se"], "A_macro": res["A"]["macro_f1"],
                 "B": res["B"]["sample_f1"], "B_se": res["B"]["sample_f1_se"], "B_macro": res["B"]["macro_f1"],
                 "J": res["J"], "J_se": res["J_se"], "complexity": cx, "secs": time.time() - t0}
        if "B2005" in res:
            attrs["B2005"] = res["B2005"]["sample_f1"]
        if "C" in res:
            attrs.update(C=res["C"]["sample_f1"], C_se=res["C"]["sample_f1_se"], C_macro=res["C"]["macro_f1"])
        for k, v in attrs.items():
            trial.set_user_attr(k, float(v))
        trial.set_user_attr("config", json.dumps(cfg))
        append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "complete", **attrs, "params": trial.params,
                                "config": cfg, "rowf1_A": res["rowf1_A"], "rowf1_B": res["rowf1_B"],
                                "rowf1_C": res.get("rowf1_C"), "B_ncuts": res.get("B_ncuts", 1)})
        return res["J"]

    return objective


def run_worker(storage_path, study_name, n_trials, log_path, seed, R=4, rows=None, chrono_cuts=CHRONO_CUTS,
               allow_blocks=True, tag="main", timeout=None, sampler_name="tpe", cv_seed=0, require_unsup=False):
    """One search worker process (shares the study through a journal file)."""
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    from .data import load
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    D = load()
    storage = JournalStorage(JournalFileBackend(storage_path))
    if sampler_name == "random":
        sampler = optuna.samplers.RandomSampler(seed=seed)
    else:
        sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True, group=True, constant_liar=True,
                                             n_startup_trials=40)
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize",
                                sampler=sampler, load_if_exists=True)
    obj = make_objective(D, log_path, R=R, rows=rows, chrono_cuts=chrono_cuts, allow_blocks=allow_blocks, tag=tag,
                         cv_seed=cv_seed, prune_margin=(np.inf if sampler_name == "random" else 0.05),
                         require_unsup=require_unsup)
    study.optimize(obj, n_trials=n_trials, timeout=timeout, catch=(Exception,))


def launch(storage_path, study_name, n_workers, trials_per_worker, log_path, R=4, rows=None,
           chrono_cuts=CHRONO_CUTS, allow_blocks=True, tag="main", timeout=None, base_seed=0, enqueue=None,
           sampler_name="tpe", cv_seed=0, require_unsup=False):
    """Run n_workers processes in parallel (spawn) against one journal-backed study."""
    import multiprocessing as mp
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    os.makedirs(os.path.dirname(storage_path), exist_ok=True)
    storage = JournalStorage(JournalFileBackend(storage_path))
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize", load_if_exists=True)
    for params in enqueue or []:
        study.enqueue_trial(params, skip_if_exists=True)
    ctx = mp.get_context("spawn")
    procs = []
    for w in range(n_workers):
        p = ctx.Process(target=run_worker, args=(storage_path, study_name, trials_per_worker, log_path,
                                                 base_seed + w, R, None if rows is None else list(map(int, rows)),
                                                 chrono_cuts, allow_blocks, tag, timeout, sampler_name, cv_seed,
                                                 require_unsup))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
    return study


# ============================================================================ selection
def load_log(log_path, tag=None, state="complete"):
    recs = []
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            if (tag is None or r.get("tag") == tag) and (state is None or r.get("state") == state):
                recs.append(r)
    return recs


def select_one_se(recs, baseline=None, key="J", se_key="J_se"):
    """1-SE rule: candidates with J >= J_best - SE(J_best); choose minimal complexity, ties by J.
    Guard: if a baseline record is given, the choice must have B >= B_base - B_se_base."""
    if not recs:
        return None, []
    best = max(recs, key=lambda r: r[key])
    thr = best[key] - best[se_key]
    cands = [r for r in recs if r[key] >= thr]
    cands.sort(key=lambda r: (r["complexity"], -r[key]))
    for r in cands:
        if baseline is None or r["B"] >= baseline["B"] - baseline["B_se"]:
            return r, cands
    return cands[0], cands


def dedupe(recs):
    """Keep one record per distinct config (TPE re-samples identical categorical configs)."""
    seen = {}
    for r in recs:
        k = json.dumps(r["config"], sort_keys=True)
        if k not in seen or r[ "J"] > seen[k]["J"]:
            seen[k] = r
    return list(seen.values())

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/search.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/submit.py

In [21]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/submit.py
"""Final refit on all 235 labeled rows, test predictions, Kaggle CSV + manifest, filled train labels."""
from __future__ import annotations

import datetime
import hashlib
import json
import os

import numpy as np
import pandas as pd

from .context import make_ctx
from .data import LABELS, write_submission
from .head import apply_head, decode, sld_em
from .pipeline import run_model, validate


def final_fit(cfg, D, n_seeds=1):
    """Refit on labeled-train + val (235 rows); return calibrated probs for test and the pool."""
    validate(cfg)
    test, unl = D.idx["test"], D.idx["unl"]
    rows = np.r_[test, unl]
    Ps, priors, Ts = [], [], []
    for s in range(n_seeds):
        ctx = make_ctx(D, D.idx["labval"], test, transductive=cfg.get("transductive", True), seed=s, name="final")
        o = run_model(cfg["model"], ctx, rows)
        Ps.append(o["P"]); priors.append(np.asarray(o["prior"])); Ts.append(o["T"])
    P = np.mean(Ps, axis=0)
    prior = np.mean(priors, axis=0)
    return {"P_test": P[: len(test)], "P_pool": P[len(test):], "prior": prior, "T": Ts}


def predict_test(cfg, D, fit):
    head = cfg.get("head", {})
    tr = cfg.get("transductive", True)
    target = None if tr else fit["P_pool"]          # inductive: SLD prior estimated on the unlabeled pool
    Y, Q, pi = apply_head(fit["P_test"], fit["prior"], D.CM, head, target)
    return Y, Q, pi


def fill_pool_labels(cfg, D, fit):
    """Hard labels for the 1,014 masked train rows from the refit model (SLD toward the pool's own
    prior when the head uses SLD)."""
    head = cfg.get("head", {})
    P = fit["P_pool"]
    if head.get("prior") == "sld":
        P, _ = sld_em(P, fit["prior"], a=head.get("sld_a", 0), floor=head.get("sld_floor", 0.25))
    return decode(P, D.CM, head.get("decode", "map")), P


def write_candidate(cfg, D, out_dir, name, scores=None, n_seeds=1, sample_path=None):
    os.makedirs(out_dir, exist_ok=True)
    fit = final_fit(cfg, D, n_seeds=n_seeds)
    Y, Q, pi = predict_test(cfg, D, fit)
    path = os.path.join(out_dir, f"{name}.csv")
    sub = write_submission(D, Y, path)
    if sample_path:                                  # exact format / order check vs the sample file
        sample = pd.read_csv(sample_path)
        assert list(sub.columns) == list(sample.columns) and (sub["ID"].values == sample["ID"].values).all()
    sha = hashlib.sha1(open(path, "rb").read()).hexdigest()
    manifest = {
        "name": name, "csv": os.path.abspath(path), "sha1": sha,
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "config": cfg, "transductive_test_features_used": bool(cfg.get("transductive", True)),
        "trained_on": "157 labeled train rows + 78 val rows (235); 1,014 unlabeled train rows as pool",
        "n_seeds": n_seeds,
        "scores": scores or {},
        "test_pred_label_rate": dict(zip(LABELS, Y.mean(0).round(3).tolist())),
        "test_pred_cardinality": float(Y.sum(1).mean()),
        "test_pred_combo_counts": {"+".join(LABELS[j] for j in range(5) if r[j]): int(c)
                                   for r, c in zip(*np.unique(Y, axis=0, return_counts=True))},
        "sld_test_prior_labels": None if pi is None else dict(zip(LABELS, (pi @ D.CM).round(3).tolist())),
        "train_prior_labels": dict(zip(LABELS, (fit["prior"] @ D.CM).round(3).tolist())),
        "temperature": fit["T"],
    }
    with open(os.path.join(out_dir, f"{name}.manifest.json"), "w") as f:
        json.dump(manifest, f, indent=1, default=float)
    return manifest, fit, Y


def write_filled_train(cfg, D, fit, path):
    Yp, P = fill_pool_labels(cfg, D, fit)
    tr = D.df.iloc[D.idx["train"]].copy()
    tr = tr.drop(columns=["split"])
    pos = {g: i for i, g in enumerate(D.idx["unl"])}
    lab = tr[LABELS].to_numpy().copy()
    src = np.array(["given"] * len(tr), dtype=object)
    for i, g in enumerate(D.idx["train"]):
        if g in pos:
            lab[i] = Yp[pos[g]]
            src[i] = "filled"
    tr[LABELS] = lab
    tr["label_source"] = src
    tr["fill_confidence"] = np.where(src == "filled", [P[pos[g]].max() if g in pos else 1.0 for g in D.idx["train"]], 1.0)
    tr.to_csv(path, index=False)
    return tr

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/submit.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/synth.py

In [22]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/synth.py
"""Synthetic features: unsupervised / semi-supervised model outputs turned into new columns
that downstream classifiers (or a stacker) consume.

    synth_features(scfg, ctx) -> (n_all, m) array for every global row

Label-free generators (fit on ctx.unsup features only, cached):
  gmm        : GMM posterior responsibilities (or log-responsibilities)
  kmeans     : distances to K-means centroids, or soft assignments softmax(-d^2/T)
Label-dependent generators (training rows get OUT-OF-FOLD values, so no row sees its own label):
  knn_vote   : cosine kNN label-vote fractions over the K combos (exact leave-one-out)
  graph      : label-spreading or C&S scores (inner K-fold for training rows)
  cluster_map: cluster -> label-combo mapped probabilities (exact leave-one-out)
  oof_model  : out-of-fold probabilities of any single pipeline config (inner K-fold = nested OOF)
Label-dependent outputs are returned as log-probabilities (K columns).
"""
from __future__ import annotations

import numpy as np

from .cache import memo, memo_ctx
from .classifiers import KNNClf
from .features import featurize
from .head import safe_log
from .labelers import DEFAULT_CLUSTER_SPACE, _cluster_fit, soft_labels, soft_labels_train_oof

LABEL_FREE = {"gmm", "kmeans"}


def _label_free(scfg, ctx):
    algo = scfg["type"]
    k = int(scfg.get("k", 16))
    space = scfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    key = {"algo": algo, "k": k, "space": space, **{kk: scfg[kk] for kk in ("cov", "n_init") if kk in scfg}}
    fit = memo("cluster", key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, scfg))
    if algo == "gmm":
        R = fit["resp"]
        return np.log(np.clip(R, 1e-6, None)) if scfg.get("out", "resp") == "logresp" else R
    d = fit["dist"]
    if scfg.get("out", "dist") == "soft":
        d2 = d ** 2
        T = float(scfg.get("temp", 1.0)) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    return d


def _knn_vote(scfg, ctx):
    K = ctx.D.K
    Z = featurize(ctx, scfg.get("space", {"scale": True, "l2": True}))
    y = ctx.y_train
    m = KNNClf({"type": "knn", "k": int(scfg.get("k", 10)), "weights": scfg.get("weights", "uniform"),
                "beta": float(scfg.get("beta", 1.0))}, K, ctx.D.CM).fit(Z[ctx.train], y)
    out = m.predict_proba(Z)
    out[ctx.train] = m.predict_proba(Z[ctx.train], exclude_self=True)
    return safe_log(out)


def _labeler_feature(lcfg, ctx):
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out = np.zeros((ctx.D.n, ctx.D.K))
    out[rows] = soft_labels(lcfg, ctx, rows)
    out[ctx.train] = soft_labels_train_oof(lcfg, ctx)
    return safe_log(out)


def _oof_model(scfg, ctx):
    from .pipeline import inner_folds, run_single
    sub = scfg["model"]
    y = ctx.y_train
    out = np.zeros((ctx.D.n, ctx.D.K))
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out[rows] = run_single(sub, ctx, rows)["P"]
    for tr, te in inner_folds(y, int(scfg.get("n_inner", 5)), ctx.seed + 17):
        inner = ctx.sub(ctx.train[tr], ctx.train[te])
        out[ctx.train[te]] = run_single(sub, inner, ctx.train[te])["P"]
    return safe_log(out)


def synth_features(scfg, ctx) -> np.ndarray:
    if scfg["type"] in LABEL_FREE:
        return _synth(scfg, ctx)
    return memo_ctx("synth", scfg, ctx, lambda: _synth(scfg, ctx))


def _synth(scfg, ctx) -> np.ndarray:
    t = scfg["type"]
    if t in LABEL_FREE:
        return _label_free(scfg, ctx)
    if t == "knn_vote":
        return _knn_vote(scfg, ctx)
    if t == "graph":
        mode = scfg.get("mode", "spread")
        lcfg = {"type": "spread" if mode == "spread" else "cs", "graph": scfg.get("graph"),
                "alpha": scfg.get("alpha", 0.5), "a1": scfg.get("a1", 0.5), "a2": scfg.get("alpha", 0.5)}
        return _labeler_feature(lcfg, ctx)
    if t == "cluster_map":
        lcfg = {"type": "cluster", **{k: v for k, v in scfg.items() if k != "type"}}
        return _labeler_feature(lcfg, ctx)
    if t == "oof_model":
        return _oof_model(scfg, ctx)
    raise ValueError(t)


def build_synth_block(synth_list, ctx):
    """Concatenate all synthetic features and standardize them on train+pool rows."""
    if not synth_list:
        return None
    parts = [synth_features(s, ctx) for s in synth_list]
    S = np.hstack(parts)
    fit_rows = np.union1d(ctx.train, ctx.pool)
    mu = S[fit_rows].mean(0)
    sd = S[fit_rows].std(0)
    sd[sd < 1e-9] = 1.0
    return (S - mu) / sd

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/synth.py


## part1/experiments/anchored_graph_components_v1/vendor/p1lib/viz.py

In [23]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/viz.py
"""Shared matplotlib style for report figures (static PNG/PDF).

Categorical hues are assigned in a FIXED order (validated reference palette, light mode);
sequential = one blue ramp; thin marks, solid hairline grid, no dual axes.
"""
from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
TEXT2 = "#52514e"
GRID = "#e4e3df"
MUTED = "#a3a29c"

# fixed identity colors for recurring entities so a method keeps its hue across figures
PROTOCOL_COLORS = {"A": SERIES[0], "B": SERIES[1], "C": SERIES[2]}
LABEL_COLORS = dict(zip(["Safety", "Funding", "Efficacy", "Recruitment", "Other"], SERIES[:5]))


def setup():
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.grid": True, "grid.color": GRID,
        "grid.linewidth": 0.6, "grid.linestyle": "-", "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2, "text.color": TEXT,
        "axes.titlesize": 11, "axes.titleweight": "semibold", "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.5, "legend.frameon": False,
        "lines.linewidth": 2.0, "lines.markersize": 6, "font.family": "DejaVu Sans",
        "axes.prop_cycle": mpl.cycler(color=SERIES), "figure.dpi": 110, "savefig.dpi": 200,
        "savefig.bbox": "tight",
    })


def save(fig, path):
    fig.savefig(path)
    if path.endswith(".png"):
        fig.savefig(path[:-4] + ".pdf")
    return path

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/p1lib/viz.py


## part1/experiments/anchored_graph_components_v1/vendor/strict_adapter.py

In [24]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/vendor/strict_adapter.py
"""Inductive, group-safe adapters for the four fixed Dylan v2 candidates.

Original p1lib is copied unchanged under source/. This file changes fitting
boundaries and nested calibration, not classifier settings or selection weights.
"""
from dataclasses import replace
import hashlib

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

from p1lib import cache, features, graph, labelers
from p1lib.classifiers import make_classifier, KNNClf
from p1lib.context import Ctx
from p1lib.head import apply_temperature, fit_temperature, safe_log
from p1lib.pipeline import _mask_absent

AUDIT = []


def groups(D):
    if not hasattr(D, '_embedding_groups'):
        D._embedding_groups = pd.util.hash_pandas_object(
            D.df[[f'emb_{i}' for i in range(150)]].round(10), index=False).to_numpy()
    return D._embedding_groups


def context(D, train, query, pool, seed=0, name='', depth=0):
    train, query, pool = [np.asarray(v, dtype=int) for v in [train, query, pool]]
    unsup = np.union1d(train, pool)
    assert not np.intersect1d(unsup, query).size
    assert not np.intersect1d(unsup, D.idx['test']).size
    # The only accessible labels belong to the fitting gold rows.
    y = np.full_like(D.y, -1); y[train] = D.y[train]
    Y = np.full_like(D.Y, -1); Y[train] = D.Y[train]
    df = D.df.copy(); df.loc[:, ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']] = Y
    hidden = replace(D, df=df, y=y, Y=Y)
    hidden._embedding_groups = groups(D)
    return Ctx(hidden, train, query, pool, unsup, False, seed, name, depth)


def subcontext(ctx, train, query):
    g = groups(ctx.D)
    banned = g[query]
    train = np.asarray(train)[~np.isin(g[train], banned)]
    pool = ctx.pool[~np.isin(g[ctx.pool], banned)]
    sub = context(ctx.D, train, query, pool, ctx.seed, ctx.name + '/inner', ctx.depth + 1)
    assert not np.intersect1d(g[sub.unsup], g[query]).size
    assert np.isin(sub.unsup, ctx.unsup).all()
    AUDIT.append(dict(depth=sub.depth, gold=len(train), masked=len(pool), query=len(query),
                      fitting_ids_sha256=hashlib.sha256(sub.D.df.iloc[sub.unsup].ID.to_numpy(str).tobytes()).hexdigest(),
                      gold_group_overlap=0, masked_group_overlap=0))
    return sub


def inner_folds(ctx, n=5):
    _, inv = np.unique(groups(ctx.D)[ctx.train], return_inverse=True)
    target = np.zeros((inv.max() + 1, 5), dtype=int)
    np.maximum.at(target, inv, ctx.D.CM[ctx.y_train])
    splitter = MultilabelStratifiedKFold(n_splits=n, shuffle=True, random_state=ctx.seed)
    for tr, te in splitter.split(np.zeros((len(target), 1)), target):
        yield np.flatnonzero(np.isin(inv, tr)), np.flatnonzero(np.isin(inv, te))


def base_oof(ctx, base_cfg, feature_cfg):
    def compute():
        K, CM = ctx.D.K, ctx.D.CM
        X = features.featurize(ctx, feature_cfg)
        P_oof = np.zeros((len(ctx.train), K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            Xi = features.featurize(sub, feature_cfg)
            m = make_classifier(base_cfg, K, CM).fit(Xi[sub.train], sub.y_train)
            P_oof[te] = m.predict_proba(Xi[sub.eval])
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], ctx.y_train)
        return P_oof, m.predict_proba(X)
    return cache.memo_ctx('strict_base_oof', [base_cfg, feature_cfg], ctx, compute)


def cs_probabilities(lcfg, ctx, rows):
    assert lcfg['type'] == 'cs'
    core = {k: v for k, v in lcfg.items() if k not in ('lam', 'tau', 'hard', 'pool')}
    def compute():
        oof, all_p = base_oof(ctx, lcfg.get('base', {'type': 'lr', 'C': .1}),
                            lcfg.get('base_features', {'scale': True}))
        return graph.correct_and_smooth(ctx, lcfg.get('graph'), float(lcfg.get('a1', .5)),
                float(lcfg.get('a2', .5)), ctx.train, ctx.y_train, oof, all_p, ctx.D.K, rows)
    return cache.memo_ctx('strict_cs', core, ctx, compute, extra=np.asarray(rows).tolist())


def knn_features(cfg, ctx):
    def compute():
        space = cfg.get('space', {'scale': True, 'l2': True})
        knn = dict(type='knn', k=int(cfg.get('k', 10)), weights=cfg.get('weights', 'uniform'),
                   beta=float(cfg.get('beta', 1)))
        Z = features.featurize(ctx, space)
        m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Z[ctx.train], ctx.y_train)
        out = m.predict_proba(Z)
        g = groups(ctx.D)
        # Preserve the source's leave-one-out idea, but leave out the entire
        # embedding group and refit scaling with that group's masked copies purged.
        for value in np.unique(g[ctx.train]):
            held = ctx.train[g[ctx.train] == value]
            train = ctx.train[g[ctx.train] != value]
            sub = subcontext(ctx, train, held)
            # Avoid caching one large transformed matrix per leave-group-out fit.
            Zi = features._embed(sub.D, features.norm_features_cfg(space), sub.unsup)
            m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Zi[sub.train], sub.y_train)
            out[held] = m.predict_proba(Zi[held])
        return safe_log(out)
    return cache.memo_ctx('strict_knn_feature', cfg, ctx, compute)


def kmeans_features(cfg, ctx):
    assert cfg['type'] == 'kmeans' and cfg.get('out') == 'soft'
    space = cfg.get('space', labelers.DEFAULT_CLUSTER_SPACE)
    Z = features.featurize(ctx, space)
    key = dict(algo='kmeans', k=int(cfg.get('k', 16)), space=space,
               **{k: cfg[k] for k in ('cov', 'n_init') if k in cfg})
    fit = cache.memo('cluster', key, ctx.unsup,
                     lambda: labelers._cluster_fit(ctx.D, 'kmeans', key['k'], Z, ctx.unsup, cfg))
    d2 = fit['dist'] ** 2
    # Source uses all global rows for this median, including eval/test. Restrict it.
    T = float(cfg.get('temp', 1)) * np.median(d2[ctx.unsup].min(1))
    logp = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
    result = np.exp(logp)
    return result / result.sum(1, keepdims=True)


def design(cfg, ctx):
    parts = []
    if cfg.get('features') is not None:
        assert not cfg['features'].get('blocks')
        parts.append(features.featurize(ctx, cfg['features']))
    synth = []
    for scfg in cfg.get('synth', []):
        if scfg['type'] == 'kmeans':
            synth.append(kmeans_features(scfg, ctx))
        elif scfg['type'] == 'knn_vote':
            synth.append(knn_features(scfg, ctx))
        else:
            raise ValueError(scfg)
    if synth:
        S = np.hstack(synth)
        mu, sd = S[ctx.unsup].mean(0), S[ctx.unsup].std(0)
        sd[sd < 1e-9] = 1
        parts.append((S - mu) / sd)
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def uncalibrated(cfg, ctx, rows):
    K, CM, y = ctx.D.K, ctx.D.CM, ctx.y_train
    if cfg['clf']['type'] == 'direct':
        return dict(P=cs_probabilities(cfg['clf']['labeler'], ctx, rows),
                    prior=np.bincount(y, minlength=K) / len(y), info={})
    X = design(cfg, ctx)
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    lcfg = cfg.get('labeler')
    if lcfg:
        assert lcfg.get('pool', 'unl') == 'unl'
        pr = ctx.pool
        Q = cs_probabilities(lcfg, ctx, pr)
        keep = Q.max(1) >= float(lcfg.get('tau', 0))
        if lcfg.get('hard', False):
            Q = np.eye(K)[Q.argmax(1)]
        pr, Q = pr[keep], Q[keep]
        w = np.full(len(pr), float(lcfg.get('lam', .1)))
    Xt, yt, wt = labelers.expand(X[ctx.train], np.eye(K)[y], np.ones(len(y)), X[pr], Q, w)
    m = make_classifier(cfg['clf'], K, CM).fit(Xt, yt, wt)
    return dict(P=m.predict_proba(X[rows]), prior=m.prior_, info={'n_pseudo_rows': len(pr)})


def run_single(cfg, ctx, rows):
    out = uncalibrated(cfg, ctx, rows)
    T = 1.
    if cfg.get('temperature', True):
        oof = np.zeros((len(ctx.train), ctx.D.K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            # Relearn features, pseudo labels and classifier inside each inner fold.
            oof[te] = uncalibrated(cfg, sub, sub.eval)['P']
        T = fit_temperature(oof, ctx.y_train)
        out['P'] = apply_temperature(out['P'], T)
    out['P'] = _mask_absent(out['P'], out['prior'])
    out['T'] = T
    return out


def run_model(cfg, ctx, rows):
    if 'ensemble' not in cfg:
        return run_single(cfg, ctx, rows)
    members = cfg['ensemble']
    weights = np.array(cfg.get('weights') or [1.] * len(members)); weights /= weights.sum()
    outs = [run_single(m, ctx, rows) for m in members]
    return dict(P=sum(w * o['P'] for w, o in zip(weights, outs)),
                prior=sum(w * o['prior'] for w, o in zip(weights, outs)),
                T=[o['T'] for o in outs], info={'members': [o['info'] for o in outs]})

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/vendor/strict_adapter.py


## part1/experiments/anchored_graph_components_v1/verify.py

In [25]:
%%writefile _source_export/part1/experiments/anchored_graph_components_v1/verify.py
"""Independently rescore temporary predictions, or verify retained compact evidence."""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

from run import HERE, LABELS, VARIANTS, check_frozen, cases, fitting_context, strict, write_json


def row_f1(truth, pred):
    return 2 * (truth * pred).sum(1) / (truth.sum(1) + pred.sum(1))


def label_metrics(truth, pred):
    tp = (truth * pred).sum(0)
    fp = ((1 - truth) * pred).sum(0)
    fn = (truth * (1 - pred)).sum(0)
    divide = lambda a, b: np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b > 0)
    return divide(tp, tp + fp), divide(tp, tp + fn), divide(2 * tp, 2 * tp + fp + fn), truth.sum(0)


def verify_compact(output):
    check_frozen()
    repeats = pd.read_csv(output / 'repeat_scores.csv')
    summary = pd.read_csv(output / 'summary.csv')
    labels = pd.read_csv(output / 'per_label_repeat_scores.csv')
    averaged = labels.groupby(['pool', 'recipe', 'label'])[['precision', 'recall', 'f1', 'support']].mean()
    saved = pd.read_csv(output / 'per_label_summary.csv').set_index(['pool', 'recipe', 'label'])
    np.testing.assert_allclose(averaged.to_numpy(), saved.loc[averaged.index, averaged.columns].to_numpy(), atol=1e-14, rtol=0)
    paired = []
    for _, row in summary.iterrows():
        sub = repeats[(repeats.pool == row.pool) & (repeats.recipe == row.recipe)].set_index('seed')
        parent = repeats[(repeats.pool == row.pool) & (repeats.recipe == 'anchored')].set_index('seed')
        delta = sub.sample_f1 - parent.loc[sub.index, 'sample_f1']
        np.testing.assert_allclose([sub.sample_f1.mean(), sub.sample_f1.std(ddof=0),
            sub.macro_f1.mean(), delta.mean()], [row.sample_f1_mean, row.sample_f1_population_sd,
            row.macro_f1_mean, row.paired_delta_mean], atol=1e-14, rtol=0)
        assert len(sub) == row.repeats
        assert ((delta > 1e-12).sum(), (abs(delta) <= 1e-12).sum(), (delta < -1e-12).sum()) == (row.wins, row.ties, row.losses)
        for seed, d in delta.items():
            paired.append(dict(pool=row.pool, recipe=row.recipe, seed=seed, delta_vs_anchored=float(d)))
    pd.DataFrame(paired).to_csv(output / 'paired_repeat_scores.csv', index=False)
    screen = repeats[repeats.seed.isin([20261126, 20261127, 20261128])]
    passers = []
    for variant in VARIANTS:
        passed = []
        for pool in ['train', 'pooled']:
            sub = screen[(screen.pool == pool) & (screen.recipe == variant)].set_index('seed')
            parent = screen[(screen.pool == pool) & (screen.recipe == 'anchored')].set_index('seed')
            delta = sub.sample_f1 - parent.sample_f1
            passed.append(delta.mean() > 0 and (delta > 1e-12).sum() >= 2)
        if all(passed): passers.append(variant)
    assert passers == json.loads((output / 'screen_gate.json').read_text())['passers']
    return dict(compact_metrics_and_gate_verified=True, source_and_protected_hashes_verified=True,
                screen_passers=passers)


def verify_predictions(output):
    counts = dict(outer_cases=0, inner_boundaries=0, prediction_rows=0)
    repeat_scores = pd.read_csv(output / 'repeat_scores.csv').set_index(['pool', 'seed', 'recipe'])
    label_scores = pd.read_csv(output / 'per_label_repeat_scores.csv').set_index(['pool', 'seed', 'recipe', 'label'])
    complementarity = []
    for stage in ['screen', 'confirm', 'supplementary']:
        folder = output / stage
        if not folder.exists(): continue
        grouped = {}
        for case in cases(stage):
            key = f'{case["pool"]}_{case["seed"]}_{case["fold"]}'
            p = folder / f'{key}.npz'
            assert p.exists(), p
            a = np.load(p)
            np.testing.assert_array_equal(a['ids'], case['query'].ID.to_numpy(str))
            np.testing.assert_array_equal(a['truth'], case['query'][LABELS].to_numpy(int))
            for name in a.files:
                if not name.startswith('scores_'): continue
                scores = a[name]
                assert np.isfinite(scores).all() and (scores >= 0).all() and (scores <= 1 + 1e-12).all()
                pred = (scores >= .3).astype(int)
                pred[np.arange(len(pred)), scores.argmax(1)] = 1
                np.testing.assert_array_equal(pred, a[name.replace('scores_', 'predictions_')])
                counts['prediction_rows'] += len(pred)
            ctx = fitting_context(case['fit'])
            g = strict.groups(ctx.D)
            coverage = np.zeros(len(ctx.train), int)
            for tr, te in strict.inner_folds(ctx):
                query = ctx.train[te]
                train = ctx.train[tr]
                pool = ctx.pool[~np.isin(g[ctx.pool], g[query])]
                assert not np.intersect1d(g[np.r_[train, pool]], g[query]).size
                assert not np.intersect1d(np.r_[train, pool], query).size
                assert (ctx.D.Y[train].sum(0) > 0).all()
                coverage[te] += 1
                counts['inner_boundaries'] += 1
            assert (coverage == 1).all()
            counts['outer_cases'] += 1
            grouped.setdefault((case['pool'], case['seed']), []).append(a)
        for (pool, seed), arrays in grouped.items():
            truth = np.concatenate([a['truth'] for a in arrays])
            anchor = np.concatenate([a['predictions_anchored'] for a in arrays])
            ids = np.concatenate([a['ids'] for a in arrays])
            assert len(ids) == len(np.unique(ids))
            anchor_f1 = row_f1(truth, anchor)
            for name in arrays[0].files:
                if not name.startswith('predictions_'): continue
                recipe = name.removeprefix('predictions_')
                pred = np.concatenate([a[name] for a in arrays])
                rf = row_f1(truth, pred)
                precision, recall, lf, support = label_metrics(truth, pred)
                record = repeat_scores.loc[(pool, seed, recipe)]
                np.testing.assert_allclose([rf.mean(), lf.mean()], [record.sample_f1, record.macro_f1], atol=1e-14, rtol=0)
                for j, label in enumerate(LABELS):
                    row = label_scores.loc[(pool, seed, recipe, label)]
                    np.testing.assert_allclose([precision[j], recall[j], lf[j], support[j]],
                        [row.precision, row.recall, row.f1, row.support], atol=1e-14, rtol=0)
                if recipe == 'anchored': continue
                exact = (truth == pred).all(1)
                anchor_exact = (truth == anchor).all(1)
                complementarity.append(dict(pool=pool, seed=seed, recipe=recipe,
                    changed_rows=int((pred != anchor).any(1).sum()),
                    higher_row_f1=int((rf > anchor_f1).sum()), lower_row_f1=int((rf < anchor_f1).sum()),
                    exact_fixes=int((exact & ~anchor_exact).sum()),
                    exact_breaks=int((~exact & anchor_exact).sum()),
                    total_row_f1_gain=float((rf - anchor_f1).clip(0).sum()),
                    total_row_f1_loss=float((anchor_f1 - rf).clip(0).sum())))
    pd.DataFrame(complementarity).to_csv(output / 'complementarity.csv', index=False)
    audits = json.loads((output / 'fit_audits.json').read_text())
    assert all(a['gold_unchanged'] for a in audits)
    checks = [a for a in audits if a.get('no_teacher_clone_exact')]
    assert len(checks) == 1 and checks[0]['zero_residual_identity'] and checks[0]['query_batch_independence']
    return dict(**counts, independent_row_and_label_rescoring=True, decoder_verified=True,
                ids_truth_and_fold_coverage_verified=True, nested_group_boundaries_verified=True,
                baseline_reference_max_abs=max(a['reference_score_max_abs'] or 0 for a in audits),
                preflight_no_teacher_clone_exact=True, preflight_zero_residual_identity=True,
                preflight_query_batch_independence=True, gold_unchanged=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--output', type=Path, default=HERE / 'results')
    parser.add_argument('--predictions', action='store_true')
    args = parser.parse_args()
    result = verify_compact(args.output)
    if args.predictions:
        result.update(verify_predictions(args.output))
        write_json(args.output / 'verification.json', result)
    else:
        assert json.loads((args.output / 'verification.json').read_text())['independent_row_and_label_rescoring']
        result['historical_full_rescoring_record_present'] = True
    print(json.dumps(result, indent=2))


if __name__ == '__main__':
    main()

Overwriting _source_export/part1/experiments/anchored_graph_components_v1/verify.py


## part1/experiments/blend_refinement_v1/run.py

In [26]:
%%writefile _source_export/part1/experiments/blend_refinement_v1/run.py
"""Five predeclared refinements of the user-reported .89935 blend; no new fitting."""
from datetime import datetime,timezone
import hashlib,json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import expit,logit
from sklearn.metrics import f1_score,precision_recall_fscore_support

HERE=Path(__file__).resolve().parent;P1=HERE.parents[1]
LABELS=['Safety','Funding','Efficacy','Recruitment','Other']
CANDIDATES=['cand2_10','cand2_30','cand2_40','parent_lda10','parent_lda25']
RECIPES=['anchored','parent','lda_control']+CANDIDATES

def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save(p,v):Path(p).write_text(json.dumps(v,indent=2)+'\n')
def read(p):return json.loads(Path(p).read_text())

def freeze():
    files=[Path(__file__),HERE/'evaluation_inputs.npz',HERE/'final_inputs.npz',HERE/'parent_scored.csv',HERE/'parent_reported_result.json',HERE/'provenance.json']
    hashes={f.name:sha(f) for f in files}
    p=HERE/'protocol.json'
    if p.exists():assert read(p)['hashes']==hashes
    else:save(p,dict(created_utc=datetime.now(timezone.utc).isoformat(),parent='80% anchored /20% Cand2, user-reported .89935',
        secondary_control='Original anchored .89636 and previously submitted LDA ensemble .89401',
        candidates=CANDIDATES,recipes=dict(cand2_10='90% anchored +10% Cand2, native .3 decoder',cand2_30='70% anchored +30% Cand2, native .3 decoder',cand2_40='60% anchored +40% Cand2, native .3 decoder',parent_lda10='90% centered current80/20 blend +10% existing centered LDA70/15/15 ensemble, .5 decoder',parent_lda25='75% centered current80/20 blend +25% existing centered LDA70/15/15 ensemble, .5 decoder'),
        centering='expit(logit(clip(score,1e-9,1-1e-9))-logit(.3)); nonempty argmax fallback for every decoder',
        evaluation='All saved grouped eight-repeat five-fold plans in both pools, original validation and all three chronology windows; reuse exact component predictions. No new classifier fitting.',
        supplementary_exception='All five fixed candidates receive supplementary checks as an explicitly declared diagnostic.',
        promotion_rule='Positive means and >=5/8 paired wins in BOTH pools against .89935 parent; keep supplementary regressions visible.',
        exploratory_selection='Exclude candidates losing >.005 sample F1 on original validation or pooled chronology if any survive. Rank remaining by maximum minimum(train_delta,pooled_delta), then sum of those two deltas, then registration order. If none survive, rank all and disclose regression. Skip a candidate only if final label predictions exactly duplicate the scored parent. Export one; no automatic promotion.',
        access='All component fits inductive on eligible training inputs; final components trained on 235 gold +1014 masked rows. Test inputs prediction only. IDs only alignment.',
        limitation='Selected after the parent Kaggle outcome; small fixed search on heavily reused local development folds. Not an untouched estimate. No J tuning or hidden-label inference.',hashes=hashes))

def scores(name,a,c,l):
    parent=.8*a+.2*c
    if name=='anchored':return a,.3
    if name=='parent':return parent,.3
    if name=='lda_control':return l,.5
    if name.startswith('cand2_'):
        w=int(name.split('_')[1])/100;return (1-w)*a+w*c,.3
    w=int(name.split('lda')[1])/100
    centered=expit(logit(np.clip(parent,1e-9,1-1e-9))-logit(.3))
    return (1-w)*centered+w*l,.5

def decide(s,t):
    p=(s>=t).astype(int);p[np.arange(len(p)),s.argmax(1)]=1;return p

def frame(ids,p):return pd.DataFrame(dict(ID=ids,Predicted=[' '.join(l for l,v in zip(LABELS,row) if v) for row in p]))

def main():
    freeze();z=np.load(HERE/'evaluation_inputs.npz');rows=[];labels=[];windows=[];preds={}
    for name in RECIPES:
        s,t=scores(name,z['anchored'],z['cand2'],z['lda']);p=decide(s,t);preds[name]=p
        assert np.isfinite(s).all() and (p.sum(1)>0).all()
        for pool in np.unique(z['pool']):
            for seed in np.unique(z['seed'][z['pool']==pool]):
                ix=(z['pool']==pool)&(z['seed']==seed);y=z['truth'][ix];pr=p[ix]
                assert len(set(z['ids'][ix]))==len(y)
                assert len(y)==dict(train=157,pooled=235,validation=78,chronology=98)[pool]
                value=f1_score(y,pr,average='samples');manual=float((2*(y*pr).sum(1)/(y.sum(1)+pr.sum(1))).mean());assert abs(value-manual)<1e-14
                rows.append(dict(pool=pool,seed=int(seed),recipe=name,n=len(y),sample_f1=value,macro_f1=f1_score(y,pr,average='macro',zero_division=0)))
                precision,recall,f,support=precision_recall_fscore_support(y,pr,zero_division=0)
                for j,label in enumerate(LABELS):labels.append(dict(pool=pool,seed=int(seed),recipe=name,label=label,precision=precision[j],recall=recall[j],f1=f[j],support=int(support[j])))
        for case in np.unique(z['case'][z['pool']=='chronology']):
            ix=z['case']==case;windows.append(dict(case=case,recipe=name,n=int(ix.sum()),sample_f1=f1_score(z['truth'][ix],p[ix],average='samples')))
    repeats=pd.DataFrame(rows);summary=[];paired=[]
    for (pool,name),g in repeats.groupby(['pool','recipe']):
        parent=repeats[(repeats.pool==pool)&(repeats.recipe=='parent')].set_index('seed')
        d=g.sample_f1.to_numpy()-parent.loc[g.seed,'sample_f1'].to_numpy()
        summary.append(dict(pool=pool,recipe=name,repeats=len(g),mean=g.sample_f1.mean(),population_sd=g.sample_f1.std(ddof=0),macro_f1=g.macro_f1.mean(),delta=float(d.mean()),wins=int((d>1e-12).sum()),ties=int((abs(d)<=1e-12).sum()),losses=int((d< -1e-12).sum())))
        paired.extend(dict(pool=pool,recipe=name,seed=int(seed),delta=float(v)) for seed,v in zip(g.seed,d))
    summary=pd.DataFrame(summary);r=HERE/'results';r.mkdir(exist_ok=True)
    repeats.to_csv(r/'repeat_scores.csv',index=False);summary.to_csv(r/'summary.csv',index=False);pd.DataFrame(labels).to_csv(r/'per_label_scores.csv',index=False);pd.DataFrame(windows).to_csv(r/'chronology_windows.csv',index=False);pd.DataFrame(paired).to_csv(r/'paired_repeat_deltas.csv',index=False)
    expected=dict(anchored=[.9320594479830149,.9213652482269503,.8910256410256411,.8843537414965985],parent=[.935244161358811,.9216312056737589,.8910256410256411,.8877551020408163],lda_control=[.9317940552016984,.9216312056737589,.9059829059829061,.8775510204081632])
    for name,values in expected.items():
        got=summary[summary.recipe==name].set_index('pool').loc[['train','pooled','validation','chronology'],'mean'].to_numpy()
        np.testing.assert_allclose(got,values,atol=1e-12,rtol=0)
    ranking=[]
    for order,name in enumerate(CANDIDATES):
        q=summary[summary.recipe==name].set_index('pool');d=q.delta
        eligible=d['validation']>=-.005-1e-12 and d['chronology']>=-.005-1e-12
        qualifies=bool(((q.loc[['train','pooled'],'delta']>1e-12)&(q.loc[['train','pooled'],'wins']>=5)).all())
        ranking.append(dict(recipe=name,guardrail_passed=bool(eligible),cv_min_delta=float(min(d['train'],d['pooled'])),cv_total_delta=float(d['train']+d['pooled']),order=order,full_promotion_rule=qualifies))
    anyeligible=any(v['guardrail_passed'] for v in ranking)
    ranking.sort(key=lambda v:(-(int(v['guardrail_passed']) if anyeligible else 0),-v['cv_min_delta'],-v['cv_total_delta'],v['order']))
    final=np.load(HERE/'final_inputs.npz');parent=frame(final['ids'],decide(*scores('parent',final['anchored'],final['cand2'],final['lda'])))
    pd.testing.assert_frame_equal(parent,pd.read_csv(HERE/'parent_scored.csv'))
    assert sha(HERE/'parent_scored.csv')==read(HERE/'parent_reported_result.json')['submission_sha256']
    sample=pd.read_csv(P1/'data/part1_sample_submission.csv');assert parent.ID.tolist()==sample.ID.tolist()
    selected=None
    for candidate in ranking:
        name=candidate['recipe'];s,t=scores(name,final['anchored'],final['cand2'],final['lda']);p=decide(s,t);out=frame(final['ids'],p)
        changed=int((out.Predicted!=parent.Predicted).sum());candidate['final_changed_rows_vs_parent']=changed
        if changed and selected is None:
            selected=dict(candidate,changed_vs_parent=changed);chosen_s=s;chosen_p=p;chosen_frame=out
    assert selected is not None
    dest=P1/'submissions/candidates'/('refinement_'+selected['recipe']+'.csv');dest.parent.mkdir(parents=True,exist_ok=True)
    assert not dest.exists();chosen_frame.to_csv(dest,index=False,lineterminator='\n')
    np.savez_compressed(HERE/'selected_final_predictions.npz',ids=final['ids'],scores=chosen_s,predictions=chosen_p)
    np.savez_compressed(r/'selected_evaluation_predictions.npz',ids=z['ids'],truth=z['truth'],pool=z['pool'],seed=z['seed'],case=z['case'],predictions=preds[selected['recipe']],parent_predictions=preds['parent'])
    save(r/'selection.json',dict(ranking=ranking,selected=selected))
    save(HERE/'prepared_submission.json',dict(status='prepared; no upload or Kaggle score',selected=selected,csv=str(dest.relative_to(P1)),sha256=sha(dest),rows=312,gold=235,masked=1014,test_inputs_used_for_fitting=False,no_new_component_fits=True,parent_scored_csv_reproduced=True,protocol_sha256=sha(HERE/'protocol.json')))
    freeze();print(summary.pivot(index='recipe',columns='pool',values='mean').to_string());print(json.dumps(selected,indent=2));print(dest)

if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/blend_refinement_v1/run.py


## part1/experiments/blend_refinement_v2/run.py

In [27]:
%%writefile _source_export/part1/experiments/blend_refinement_v2/run.py
"""Second fixed batch of four probability-combination refinements of the user-reported .89935 blend; no new fitting."""
from datetime import datetime,timezone
import hashlib,json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import expit,logit
from sklearn.metrics import f1_score,precision_recall_fscore_support

HERE=Path(__file__).resolve().parent;P1=HERE.parents[1]
LABELS=['Safety','Funding','Efficacy','Recruitment','Other']
CANDIDATES=['native_lda10','native_lda20','log_pool','uncertainty']
RECIPES=['anchored','parent','lda_control']+CANDIDATES

def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save(p,v):Path(p).write_text(json.dumps(v,indent=2)+'\n')
def read(p):return json.loads(Path(p).read_text())

def freeze():
    files=[Path(__file__),HERE/'evaluation_inputs.npz',HERE/'final_inputs.npz',HERE/'parent_scored.csv',HERE/'parent_reported_result.json',HERE/'provenance.json']
    hashes={f.name:sha(f) for f in files}
    p=HERE/'protocol.json'
    if p.exists():assert read(p)['hashes']==hashes
    else:save(p,dict(created_utc=datetime.now(timezone.utc).isoformat(),parent='80% anchored /20% Cand2, user-reported .89935',
        secondary_control='Original anchored .89636 and previously submitted LDA ensemble .89401',
        candidates=CANDIDATES,recipes=dict(native_lda10='70% anchored +20% Cand2 +10% LDA after inverse centering to native .3 cutoff',native_lda20='60% anchored +20% Cand2 +20% inverse-centered LDA',log_pool='expit(.8*logit(anchored)+.2*logit(Cand2)); .3 cutoff',uncertainty='Per-label Cand2 weight .4*(1-2*abs(centered_anchor-.5)); convex native score combination, .3 cutoff'),
        centering='expit(logit(clip(score,1e-9,1-1e-9))-logit(.3)); nonempty argmax fallback for every decoder',
        evaluation='All saved grouped eight-repeat five-fold plans in both pools, original validation and all three chronology windows; reuse exact component predictions. No new classifier fitting.',
        supplementary_exception='All four fixed candidates receive supplementary checks as an explicitly declared diagnostic.',
        promotion_rule='Positive means and >=5/8 paired wins in BOTH pools against .89935 parent; keep supplementary regressions visible.',
        exploratory_selection='Exclude candidates losing >.005 sample F1 on original validation or pooled chronology if any survive. Rank remaining by maximum minimum(train_delta,pooled_delta), then sum of those two deltas, then registration order. If none survive, rank all and disclose regression. Skip a candidate only if final label predictions exactly duplicate the scored parent. Export one; no automatic promotion.',
        access='All component fits inductive on eligible training inputs; final components trained on 235 gold +1014 masked rows. Test inputs prediction only. IDs only alignment.',
        batch_history='V1 nearby weights and centered LDA additions all failed to improve both pools. This second batch tests four fixed functional forms; no further batch. Deliver the best v2 candidate only if its registered CV ranking exceeds the v1 selected candidate; otherwise retain v1.',
        limitation='Selected after the parent Kaggle outcome; small fixed search on heavily reused local development folds. Not an untouched estimate. No J tuning or hidden-label inference.',hashes=hashes))

def scores(name,a,c,l):
    parent=.8*a+.2*c
    if name=='anchored':return a,.3
    if name=='parent':return parent,.3
    if name=='lda_control':return l,.5
    eps=1e-9
    if name.startswith('native_lda'):
        w=int(name.split('lda')[1])/100
        native=expit(logit(np.clip(l,eps,1-eps))+logit(.3))
        return (.8-w)*a+.2*c+w*native,.3
    if name=='log_pool':
        return expit(.8*logit(np.clip(a,eps,1-eps))+.2*logit(np.clip(c,eps,1-eps))),.3
    assert name=='uncertainty'
    center=expit(logit(np.clip(a,eps,1-eps))-logit(.3))
    w=.4*(1-2*abs(center-.5))
    return (1-w)*a+w*c,.3


def decide(s,t):
    p=(s>=t).astype(int);p[np.arange(len(p)),s.argmax(1)]=1;return p

def frame(ids,p):return pd.DataFrame(dict(ID=ids,Predicted=[' '.join(l for l,v in zip(LABELS,row) if v) for row in p]))

def main():
    freeze();z=np.load(HERE/'evaluation_inputs.npz');rows=[];labels=[];windows=[];preds={}
    for name in RECIPES:
        s,t=scores(name,z['anchored'],z['cand2'],z['lda']);p=decide(s,t);preds[name]=p
        assert np.isfinite(s).all() and (p.sum(1)>0).all()
        for pool in np.unique(z['pool']):
            for seed in np.unique(z['seed'][z['pool']==pool]):
                ix=(z['pool']==pool)&(z['seed']==seed);y=z['truth'][ix];pr=p[ix]
                assert len(set(z['ids'][ix]))==len(y)
                assert len(y)==dict(train=157,pooled=235,validation=78,chronology=98)[pool]
                value=f1_score(y,pr,average='samples');manual=float((2*(y*pr).sum(1)/(y.sum(1)+pr.sum(1))).mean());assert abs(value-manual)<1e-14
                rows.append(dict(pool=pool,seed=int(seed),recipe=name,n=len(y),sample_f1=value,macro_f1=f1_score(y,pr,average='macro',zero_division=0)))
                precision,recall,f,support=precision_recall_fscore_support(y,pr,zero_division=0)
                for j,label in enumerate(LABELS):labels.append(dict(pool=pool,seed=int(seed),recipe=name,label=label,precision=precision[j],recall=recall[j],f1=f[j],support=int(support[j])))
        for case in np.unique(z['case'][z['pool']=='chronology']):
            ix=z['case']==case;windows.append(dict(case=case,recipe=name,n=int(ix.sum()),sample_f1=f1_score(z['truth'][ix],p[ix],average='samples')))
    repeats=pd.DataFrame(rows);summary=[];paired=[]
    for (pool,name),g in repeats.groupby(['pool','recipe']):
        parent=repeats[(repeats.pool==pool)&(repeats.recipe=='parent')].set_index('seed')
        d=g.sample_f1.to_numpy()-parent.loc[g.seed,'sample_f1'].to_numpy()
        summary.append(dict(pool=pool,recipe=name,repeats=len(g),mean=g.sample_f1.mean(),population_sd=g.sample_f1.std(ddof=0),macro_f1=g.macro_f1.mean(),delta=float(d.mean()),wins=int((d>1e-12).sum()),ties=int((abs(d)<=1e-12).sum()),losses=int((d< -1e-12).sum())))
        paired.extend(dict(pool=pool,recipe=name,seed=int(seed),delta=float(v)) for seed,v in zip(g.seed,d))
    summary=pd.DataFrame(summary);r=HERE/'results';r.mkdir(exist_ok=True)
    repeats.to_csv(r/'repeat_scores.csv',index=False);summary.to_csv(r/'summary.csv',index=False);pd.DataFrame(labels).to_csv(r/'per_label_scores.csv',index=False);pd.DataFrame(windows).to_csv(r/'chronology_windows.csv',index=False);pd.DataFrame(paired).to_csv(r/'paired_repeat_deltas.csv',index=False)
    expected=dict(anchored=[.9320594479830149,.9213652482269503,.8910256410256411,.8843537414965985],parent=[.935244161358811,.9216312056737589,.8910256410256411,.8877551020408163],lda_control=[.9317940552016984,.9216312056737589,.9059829059829061,.8775510204081632])
    for name,values in expected.items():
        got=summary[summary.recipe==name].set_index('pool').loc[['train','pooled','validation','chronology'],'mean'].to_numpy()
        np.testing.assert_allclose(got,values,atol=1e-12,rtol=0)
    ranking=[]
    for order,name in enumerate(CANDIDATES):
        q=summary[summary.recipe==name].set_index('pool');d=q.delta
        eligible=d['validation']>=-.005-1e-12 and d['chronology']>=-.005-1e-12
        qualifies=bool(((q.loc[['train','pooled'],'delta']>1e-12)&(q.loc[['train','pooled'],'wins']>=5)).all())
        ranking.append(dict(recipe=name,guardrail_passed=bool(eligible),cv_min_delta=float(min(d['train'],d['pooled'])),cv_total_delta=float(d['train']+d['pooled']),order=order,full_promotion_rule=qualifies))
    anyeligible=any(v['guardrail_passed'] for v in ranking)
    ranking.sort(key=lambda v:(-(int(v['guardrail_passed']) if anyeligible else 0),-v['cv_min_delta'],-v['cv_total_delta'],v['order']))
    final=np.load(HERE/'final_inputs.npz');parent=frame(final['ids'],decide(*scores('parent',final['anchored'],final['cand2'],final['lda'])))
    pd.testing.assert_frame_equal(parent,pd.read_csv(HERE/'parent_scored.csv'))
    assert sha(HERE/'parent_scored.csv')==read(HERE/'parent_reported_result.json')['submission_sha256']
    sample=pd.read_csv(P1/'data/part1_sample_submission.csv');assert parent.ID.tolist()==sample.ID.tolist()
    selected=None
    for candidate in ranking:
        name=candidate['recipe'];s,t=scores(name,final['anchored'],final['cand2'],final['lda']);p=decide(s,t);out=frame(final['ids'],p)
        changed=int((out.Predicted!=parent.Predicted).sum());candidate['final_changed_rows_vs_parent']=changed
        if changed and selected is None:
            selected=dict(candidate,changed_vs_parent=changed);chosen_s=s;chosen_p=p;chosen_frame=out
    assert selected is not None
    dest=P1/'submissions/candidates'/('refinement_'+selected['recipe']+'.csv');dest.parent.mkdir(parents=True,exist_ok=True)
    assert not dest.exists();chosen_frame.to_csv(dest,index=False,lineterminator='\n')
    np.savez_compressed(HERE/'selected_final_predictions.npz',ids=final['ids'],scores=chosen_s,predictions=chosen_p)
    np.savez_compressed(r/'selected_evaluation_predictions.npz',ids=z['ids'],truth=z['truth'],pool=z['pool'],seed=z['seed'],case=z['case'],predictions=preds[selected['recipe']],parent_predictions=preds['parent'])
    save(r/'selection.json',dict(ranking=ranking,selected=selected))
    save(HERE/'prepared_submission.json',dict(status='prepared; no upload or Kaggle score',selected=selected,csv=str(dest.relative_to(P1)),sha256=sha(dest),rows=312,gold=235,masked=1014,test_inputs_used_for_fitting=False,no_new_component_fits=True,parent_scored_csv_reproduced=True,protocol_sha256=sha(HERE/'protocol.json')))
    freeze();print(summary.pivot(index='recipe',columns='pool',values='mean').to_string());print(json.dumps(selected,indent=2));print(dest)

if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/blend_refinement_v2/run.py


## part1/experiments/blend_refinement_v2/verify.py

In [28]:
%%writefile _source_export/part1/experiments/blend_refinement_v2/verify.py
"""Verify frozen sources, retained scores and the selected export without fitting."""
from pathlib import Path
import importlib.util,json,hashlib
import numpy as np
import pandas as pd

ROOT=Path(__file__).resolve().parent.parent
for version in ['blend_refinement_v1','blend_refinement_v2']:
    p=ROOT/version
    spec=importlib.util.spec_from_file_location(version,p/'run.py');m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m);m.freeze()
    z=np.load(p/'evaluation_inputs.npz');repeats=pd.read_csv(p/'results/repeat_scores.csv');labels=pd.read_csv(p/'results/per_label_scores.csv')
    for name in m.RECIPES:
        s,t=m.scores(name,z['anchored'],z['cand2'],z['lda']);pred=m.decide(s,t)
        for row in repeats[repeats.recipe==name].itertuples():
            ix=(z['pool']==row.pool)&(z['seed']==row.seed);y=z['truth'][ix];yp=pred[ix]
            value=np.mean(2*np.sum(y*yp,axis=1)/(y.sum(1)+yp.sum(1)));assert abs(value-row.sample_f1)<1e-14
            tp=(y*yp).sum(0);fp=((1-y)*yp).sum(0);fn=(y*(1-yp)).sum(0)
            div=lambda a,b:np.divide(a,b,out=np.zeros_like(a,dtype=float),where=b>0)
            pr,re,f=div(tp,tp+fp),div(tp,tp+fn),div(2*tp,2*tp+fp+fn)
            assert abs(f.mean()-row.macro_f1)<1e-14
            got=labels[(labels.recipe==name)&(labels.pool==row.pool)&(labels.seed==row.seed)].set_index('label').loc[m.LABELS]
            np.testing.assert_allclose(got[['precision','recall','f1','support']],np.column_stack([pr,re,f,y.sum(0)]),atol=1e-14,rtol=0)
    if version.endswith('v2'):
        final=np.load(p/'final_inputs.npz');r=m.read(p/'prepared_submission.json');f=m.P1/r['csv'];assert m.sha(f)==r['sha256']
        a=np.clip(final['anchored'],1e-9,1-1e-9);c=np.clip(final['cand2'],1e-9,1-1e-9)
        odds=(a/(1-a))**.8*(c/(1-c))**.2;score=odds/(1+odds);pred=m.decide(score,.3)
        out=pd.read_csv(f);expected=m.frame(final['ids'],pred);pd.testing.assert_frame_equal(out,expected)
        parent=m.frame(final['ids'],m.decide(.8*final['anchored']+.2*final['cand2'],.3));pd.testing.assert_frame_equal(parent,pd.read_csv(p/'parent_scored.csv'))
        assert (out.Predicted!=parent.Predicted).sum()==5 and out.ID.is_unique and len(out)==312
print('Both fixed batches independently rescored; selected log-odds formula, exact parent CSV, export and hashes verified.')

Overwriting _source_export/part1/experiments/blend_refinement_v2/verify.py


## part1/experiments/cand2_followup_v1/analyze.py

In [29]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/analyze.py
"""Independent metric arithmetic, fixed-blend evaluation and descriptive rank audit."""
import argparse
from itertools import combinations
import json
from pathlib import Path
import numpy as np
import pandas as pd
from run import HERE,P1,LABELS,read,save,sha,freeze


def measure(y,p):
    row=2*(y*p).sum(1)/(y.sum(1)+p.sum(1))
    tp=(y*p).sum(0);fp=((1-y)*p).sum(0);fn=(y*(1-p)).sum(0)
    div=lambda a,b:np.divide(a,b,out=np.zeros_like(a,dtype=float),where=b>0)
    pr,re,f=div(tp,tp+fp),div(tp,tp+fn),div(2*tp,2*tp+fp+fn)
    return dict(sample_f1=float(row.mean()),macro_f1=float(f.mean())),pr,re,f,y.sum(0)


def independent(output,current_only=False):
    freeze();groups={};predicted_rows=0;case_count=0
    for c in read(HERE/'case_plan.json'):
        current=c['access']=='strict_inductive'
        if current_only and not current:continue
        path=(HERE/'inputs' if current else output)/(c['key']+'.npz')
        if not current:assert sha(path)==read(path.with_suffix('.json'))['sha256']
        with np.load(path) as z:a={k:z[k].copy() for k in z.files}
        assert a['ids'].tolist()==c['query_ids']
        if current:
            s=(4*a['anchored_scores']+a['cand2_scores'])/5
            pred=(s>=.3).astype(int);pred[np.arange(len(s)),s.argmax(1)]=1
            a['blend_predictions']=pred
        else:
            np.testing.assert_allclose(a['blend_scores'],(4*a['anchored_scores']+a['cand2_scores'])/5,atol=1e-15,rtol=0)
            s=a['blend_scores'];p=(s>=.3).astype(int);p[np.arange(len(p)),s.argmax(1)]=1
            np.testing.assert_array_equal(p,a['blend_predictions'])
        kind=c['kind'];rep=c.get('repeat')
        if rep is None:rep=c['key']
        track='current' if current else 'J'
        groups.setdefault((track,kind,str(rep)),[]).append(a)
        case_count+=1
    metrics=[];labels=[];complementarity=[]
    for (track,kind,rep),parts in groups.items():
        y=np.concatenate([p['truth'] for p in parts]);ids=np.concatenate([p['ids'] for p in parts])
        assert len(ids)==len(set(ids))
        if kind in ['A','train','pooled']:assert len(parts)==5
        anchor=np.concatenate([p['anchored_predictions'] for p in parts])
        af=2*(y*anchor).sum(1)/(y.sum(1)+anchor.sum(1))
        for name in ['anchored','cand2','blend']:
            p=np.concatenate([a[name+'_predictions'] for a in parts]);assert np.isin(p,[0,1]).all() and (p.sum(1)>0).all()
            m,pr,re,f,support=measure(y,p)
            metrics.append(dict(track=track,kind=kind,repeat=rep,recipe=name,n=len(y),**m));predicted_rows+=len(y)
            for j,label in enumerate(LABELS):labels.append(dict(track=track,kind=kind,repeat=rep,recipe=name,label=label,precision=pr[j],recall=re[j],f1=f[j],support=int(support[j])))
            rf=2*(y*p).sum(1)/(y.sum(1)+p.sum(1))
            if name!='anchored':complementarity.append(dict(track=track,kind=kind,repeat=rep,recipe=name,
                changed=int(np.any(p!=anchor,axis=1).sum()),improve=int((rf>af+1e-12).sum()),worsen=int((rf<af-1e-12).sum()),
                exact_fixes=int(((p==y).all(1)&~(anchor==y).all(1)).sum()),exact_breaks=int((~(p==y).all(1)&(anchor==y).all(1)).sum())))
    metrics=pd.DataFrame(metrics);labels=pd.DataFrame(labels)
    summary=[]
    for (track,kind,name),g in metrics.groupby(['track','kind','recipe']):
        parent=metrics[(metrics.track==track)&(metrics.kind==kind)&(metrics.recipe=='anchored')].set_index('repeat')
        delta=g.sample_f1.to_numpy()-parent.loc[g['repeat'],'sample_f1'].to_numpy()
        # Chronology pools held-out rows across non-overlapping windows, unlike J's equal-window B.
        mean=np.average(g.sample_f1,weights=g.n) if kind=='chronology' else g.sample_f1.mean()
        summary.append(dict(track=track,kind=kind,recipe=name,repeats=len(g),sample_f1_mean=float(mean),
            population_sd=float(g.sample_f1.std(ddof=0)),paired_delta_mean=float(np.average(delta,weights=g.n) if kind=='chronology' else delta.mean()),
            wins=int((delta>1e-12).sum()),ties=int((abs(delta)<=1e-12).sum()),losses=int((delta< -1e-12).sum())))
    summary=pd.DataFrame(summary)
    dest=output/('current_only' if current_only else 'analysis');dest.mkdir(parents=True,exist_ok=True)
    metrics.to_csv(dest/'repeat_metrics.csv',index=False);labels.to_csv(dest/'label_metrics.csv',index=False)
    summary.to_csv(dest/'summary.csv',index=False);pd.DataFrame(complementarity).to_csv(dest/'complementarity.csv',index=False)
    old=pd.read_csv(HERE/'prior_strict_comparison.csv').set_index('model')
    for name in ['anchored','cand2']:
        for kind in ['train','pooled','validation','chronology']:
            value=summary[(summary.track=='current')&(summary.recipe==name)&(summary.kind==kind)].sample_f1_mean.iloc[0]
            np.testing.assert_allclose(value,old.loc[name,kind],atol=1e-14,rtol=0)
    check=dict(independent_metric_arithmetic=True,prior_current_metrics_reproduced=True,cases=case_count,prediction_rows=predicted_rows)
    if not current_only:
        fresh=pd.read_csv(output/'repeat_metrics.csv')
        for row in metrics.itertuples():
            # Runner uses year keys for B; compare those in sorted order below instead.
            if row.kind in ['B','chronology']:continue
            got=fresh[(fresh.track==row.track)&(fresh.kind==row.kind)&(fresh.recipe==row.recipe)&(fresh['repeat'].astype(str)==row.repeat)]
            assert len(got)==1,(row,got)
            np.testing.assert_allclose([row.sample_f1,row.macro_f1],got[['sample_f1','macro_f1']].to_numpy()[0],atol=1e-14,rtol=0)
        for track,kind in [('J','B'),('current','chronology')]:
            for name in ['anchored','cand2','blend']:
                g=metrics[(metrics.track==track)&(metrics.kind==kind)&(metrics.recipe==name)]
                got=fresh[(fresh.track==track)&(fresh.kind==kind)&(fresh.recipe==name)]
                np.testing.assert_allclose(np.sort(g.sample_f1),np.sort(got.sample_f1),atol=1e-14,rtol=0)
        fresh_labels=pd.read_csv(output/'label_metrics.csv')
        for (track,kind,name,label),g in labels.groupby(['track','kind','recipe','label']):
            other=fresh_labels[(fresh_labels.track==track)&(fresh_labels.kind==kind)&(fresh_labels.recipe==name)&(fresh_labels.label==label)]
            assert len(g)==len(other)
            for metric in ['precision','recall','f1','support']:
                np.testing.assert_allclose(np.sort(g[metric]),np.sort(other[metric]),atol=1e-14,rtol=0)
        js=[]
        for name in ['anchored','cand2','blend']:
            s=summary[(summary.track=='J')&(summary.recipe==name)].set_index('kind').sample_f1_mean
            js.append(dict(recipe=name,A=s['A'],B=s['B'],J=(s['A']+s['B'])/2,validation=s['validation']))
        pd.DataFrame(js).to_csv(dest/'matched_J.csv',index=False)
        historical=pd.read_csv(HERE/'historical_comparison.csv')
        np.testing.assert_allclose(js[0]['J'],historical.set_index('model').loc['anchored','dylan_J'],atol=1e-14,rtol=0)
        cand2=dict(model='cand2_inductive5',kaggle=read(HERE/'cand2_reported_result.json')['reported_kaggle_score'],
            dylan_J=js[1]['J'],grouped_train=old.loc['cand2','train'],grouped_pooled=old.loc['cand2','pooled'])
        panel=pd.concat([historical,pd.DataFrame([cand2])],ignore_index=True)
        panel.to_csv(dest/'expanded_panel.csv',index=False)
        pairs=[];ranks=[]
        for cleaner in [False,True]:
            df=panel[panel.model!='original_ani'] if cleaner else panel
            for metric in ['dylan_J','grouped_train','grouped_pooled']:
                correct=wrong=tie=0
                for (_,a),(_,b) in combinations(df.iterrows(),2):
                    if abs(a.kaggle-b.kaggle)<1e-10:continue
                    sg=np.sign(a[metric]-b[metric]);truth=np.sign(a.kaggle-b.kaggle)
                    status='tie' if sg==0 else ('correct' if sg==truth else 'wrong')
                    correct+=status=='correct';wrong+=status=='wrong';tie+=status=='tie'
                    pairs.append(dict(exclude_original_ani=cleaner,metric=metric,model_a=a.model,model_b=b.model,status=status))
                ranks.append(dict(exclude_original_ani=cleaner,metric=metric,correct=correct,wrong=wrong,tied=tie,total=correct+wrong+tie))
        pd.DataFrame(ranks).to_csv(dest/'rank_agreement.csv',index=False);pd.DataFrame(pairs).to_csv(dest/'pairwise_orderings.csv',index=False)
        check['fresh_J_parent_reproduced']=True
        check['independent_vs_runner_metrics']=True
        print(pd.DataFrame(js).to_string(index=False));print(pd.DataFrame(ranks).to_string(index=False))
    save(dest/'verification.json',check)
    print(summary.to_string(index=False))


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--output',type=Path,required=True);p.add_argument('--current-only',action='store_true');args=p.parse_args()
    independent(args.output,args.current_only)

Overwriting _source_export/part1/experiments/cand2_followup_v1/analyze.py


## part1/experiments/cand2_followup_v1/export_blend.py

In [30]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/export_blend.py
"""Prepare the fixed 80/20 exploratory blend; never tune on test predictions."""
import io
from datetime import datetime,timezone
import hashlib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from run import HERE,P1,LABELS,EMB,AnchoredReasonClassifier,select_reasons,freeze,read,save,sha


def csv_frame(ids,pred):
    return pd.DataFrame(dict(ID=ids,Predicted=[' '.join(l for l,yes in zip(LABELS,row) if yes) for row in pred]))


def main():
    freeze()
    csv=P1/'submissions/candidates/anchored80_cand2_inductive20.csv'
    assert not csv.exists(),'Do not overwrite an exported artifact.'
    original=read(HERE/'cand2_final_provenance.json')
    assert sha(HERE/'cand2_final_predictions.npz')==original['source_prediction_sha256']
    save(HERE/'blend_export_protocol.json',dict(created_utc=datetime.now(timezone.utc).isoformat(),
        recipe='80% anchored raw decision scores + 20% strict-inductive five-seed Cand2 label marginals; .3+argmax',
        purpose='Concrete next exploratory direction after reported Cand2 result and fixed blend diagnostic; not a promoted leader.',
        status='Fails full both-pool repeat-win rule; no Kaggle outcome or upload.',
        settings_unchanged_from_scored_protocol=sha(HERE/'protocol.json'),
        builder_sha256=sha(__file__),cand2_prediction_sha256=original['source_prediction_sha256'],
        parent_source_sha256=sha(P1/'anchored_solution.py'),
        access='Inductive final235; original 1014 masked training rows; test inputs prediction only. No new Cand2 fitting.',
        source_prediction_record='cand2_final_provenance.json'))
    frame=pd.concat([pd.read_csv(P1/'data/part1_train.csv'),pd.read_csv(P1/'data/part1_val.csv')],ignore_index=True)
    test=pd.read_csv(P1/'data/part1_test.csv')
    with threadpool_limits(limits=1):
        parent=AnchoredReasonClassifier().fit(frame[EMB].to_numpy(float),frame[LABELS].to_numpy(int))
        a=parent.predict_proba(test[EMB].to_numpy(float))
    anchor=csv_frame(test.ID,select_reasons(a))
    expected=pd.read_csv(P1/'submissions/archive/submitted/anchored_reason_trainval235__kaggle_0.89636.csv')
    pd.testing.assert_frame_equal(anchor,expected)
    with np.load(HERE/'cand2_final_predictions.npz') as z:
        assert test.ID.tolist()==z['ids'].tolist()
        cand2=csv_frame(z['ids'],z['predictions'])
        assert hashlib.sha256(cand2.to_csv(index=False,lineterminator='\n').encode()).hexdigest()==original['source_submission_sha256']
        c=z['Q']@z['CM']
    s=.8*a+.2*c;pred=select_reasons(s);out=csv_frame(test.ID,pred)
    sample=pd.read_csv(P1/'data/part1_sample_submission.csv')
    assert out.ID.tolist()==sample.ID.tolist() and out.ID.is_unique and len(out)==312
    assert (pred.sum(1)>0).all() and np.isfinite(s).all()
    csv.parent.mkdir(parents=True,exist_ok=True);out.to_csv(csv,index=False,lineterminator='\n')
    np.savez_compressed(HERE/'blend_final_predictions.npz',ids=test.ID.to_numpy(str),anchored_scores=a,cand2_scores=c,scores=s,predictions=pred)
    record=dict(status='prepared exploratory candidate; not uploaded; Kaggle score unknown',
        csv=str(csv.relative_to(P1)),sha256=sha(csv),rows=312,gold=235,masked=1014,test_features_used_for_fit=False,
        changed_vs_anchored=int((out.Predicted!=anchor.Predicted).sum()),changed_vs_cand2=int((out.Predicted.to_numpy()!=cand2.Predicted.to_numpy()).sum()),
        parent_scored_predictions_reproduced=True,cand2_exact_scored_csv_reconstructed=True,
        final_prediction_sha256=sha(HERE/'blend_final_predictions.npz'),export_protocol_sha256=sha(HERE/'blend_export_protocol.json'))
    save(HERE/'blend_prepared_submission.json',record);freeze();print(record)


if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/cand2_followup_v1/export_blend.py


## part1/experiments/cand2_followup_v1/run.py

In [31]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/run.py
"""Matched-recipe J diagnostic and one fixed complementary blend after Cand2's score."""
import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[key]='1'
import argparse
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import replace
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import sys
import time

HERE=Path(__file__).resolve().parent
P1=HERE.parents[1]
sys.path[:0]=[str(HERE/'source'),str(P1)]
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support
from threadpoolctl import threadpool_limits
from p1lib import cache
from p1lib.data import load, LABELS, EMB
from p1lib.head import apply_head
from anchored_solution import AnchoredReasonClassifier,select_reasons
import strict_adapter as strict


def read(p):return json.loads(Path(p).read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save(p,v):Path(p).write_text(json.dumps(v,indent=2)+'\n')


def freeze():
    files=[HERE/'run.py',HERE/'strict_adapter.py',HERE/'cand2_manifest.json',HERE/'case_plan.json',
        HERE/'historical_comparison.csv',HERE/'prior_strict_comparison.csv',HERE/'cand2_reported_result.json',P1/'anchored_solution.py']
    files+=sorted((HERE/'source/p1lib').glob('*.py'))+sorted((P1/'data').glob('*.csv'))+sorted((HERE/'inputs').glob('*.npz'))
    hashes={str(p.relative_to(P1)):sha(p) for p in files}
    p=HERE/'protocol.json'
    if p.exists():assert read(p)['hashes']==hashes
    else:
        save(p,dict(version='cand2-followup-v1',created_utc=datetime.now(timezone.utc).isoformat(),
            base_main='31f3fa9',parent='anchored 0.89636',known_cand2_kaggle=.89102,
            question='Does fixed J rank the exact submitted inductive five-seed Cand2 correctly? Does one small fixed blend help?',
            J='Original saved A: 10 repeats x 5 ungrouped folds; B: mean cuts 2004/2005/2006; J=(A+B)/2. Original validation separate.',
            J_access='Identical eligible gold/masked IDs to native J and anchored controls. Query/test rows excluded from all model fitting. Outer J folds retain duplicate-group overlap and B retains all masked training years; this is deliberately NOT the current grouped or strict-chronology benchmark. Inner Cand2 calibration remains group-safe.',
            recipe='Exact submitted strict adapter, five inner seeds 0..4, unchanged config/head. Average float64 combo probabilities before expected-F1 decoding.',
            J_controls='Fresh anchored fit on each saved case, requiring exact score/prediction agreement with prior audit.',
            blend='0.8 * anchored native scores + 0.2 * Cand2 label marginals, .3 cutoff plus argmax; only one weight; no threshold/weight search.',
            blend_evaluation='Reuse verified predictions on all 80 grouped folds, original validation and three chronology windows. Also report J diagnostic. No new blend component fits for current-policy comparison.',
            advancement='Positive paired sample-F1 mean and >=5/8 wins in BOTH grouped pools. Supplementary checks predeclared for this fixed diagnostic even if CV fails; no automatic submission or promotion.',
            limitations='Kaggle result was known before this fixed diagnostic; no metric weights/splits/model settings selected after results. One new model is not multiple independent observations. Native J result must not be attached to the inductive submitted artifact.',hashes=hashes))


def context(D,c):
    pos={v:i for i,v in enumerate(D.df.ID)}
    tr,qu,po,fit=[np.array([pos[v] for v in c[k]]) for k in ['train_gold','query_ids','masked_ids','fit_ids']]
    keep=np.union1d(fit,qu);remap={v:i for i,v in enumerate(keep)}
    tr,qu,po=[np.array([remap[v] for v in x]) for x in [tr,qu,po]]
    idx=dict(train=np.array([i for i,v in enumerate(keep) if v in set(D.idx['train'])]),
        lab=tr,unl=po,val=np.array([],int),test=np.array([],int),labval=np.r_[tr,qu])
    sub=replace(D,df=D.df.iloc[keep].reset_index(drop=True),emb=D.emb[keep],Y=D.Y[keep],y=D.y[keep],idx=idx,year=D.year[keep])
    ctx=strict.context(sub,tr,qu,po,seed=0)
    assert ctx.D.df.iloc[ctx.unsup].ID.tolist()==c['fit_ids']
    assert ctx.D.df.iloc[qu].ID.tolist()==c['query_ids']
    assert (ctx.D.Y[qu]==-1).all() and not np.intersect1d(ctx.unsup,qu).size
    return ctx


def blend(anchor,candidate):return .8*anchor+.2*candidate


def fit_j(c,output):
    start=time.monotonic()
    D=load(str(P1/'data'));ctx=context(D,c)
    cfg=read(HERE/'cand2_manifest.json')['config']
    with np.load(HERE/'inputs'/(c['key']+'.npz')) as z:ref={k:z[k].copy() for k in z.files}
    assert ref['ids'].tolist()==c['query_ids']
    Y=ctx.D.Y[ctx.unsup].copy();Y[Y<0]=-100
    with threadpool_limits(limits=1):
        parent=AnchoredReasonClassifier().fit(ctx.D.emb[ctx.unsup],Y)
        a=parent.predict_proba(ctx.D.emb[ctx.eval])
        error=float(np.abs(a-ref['anchored_scores']).max())
        assert error<1e-10,error
        np.testing.assert_array_equal(select_reasons(a),ref['anchored_predictions'])
        probabilities=[];priors=[];temperatures=[];audits=[]
        for seed in range(5):
            cache.clear();strict.AUDIT.clear()
            cs=replace(ctx,seed=seed)
            out=strict.run_model(cfg['model'],cs,np.r_[cs.eval,cs.pool])
            probabilities.append(out['P'][:len(cs.eval)]);priors.append(out['prior']);temperatures.append(float(out['T']))
            assert not any(v['gold_group_overlap'] or v['masked_group_overlap'] for v in strict.AUDIT)
            audits.extend(strict.AUDIT)
        yp,Q,_=apply_head(np.mean(probabilities,axis=0),np.mean(priors,axis=0),ctx.D.CM,cfg['head'])
    candidate=Q@ctx.D.CM;b=blend(a,candidate)
    arrays=dict(ids=ref['ids'],truth=ref['truth'],anchored_scores=a,anchored_predictions=select_reasons(a),
        cand2_scores=candidate,cand2_predictions=yp,blend_scores=b,blend_predictions=select_reasons(b))
    for name in ['anchored','cand2','blend']:
        assert np.isfinite(arrays[name+'_scores']).all() and (arrays[name+'_predictions'].sum(1)>0).all()
    dest=Path(output)/(c['key']+'.npz');np.savez_compressed(dest,**arrays)
    meta=dict(key=c['key'],kind=c['kind'],repeat=c.get('repeat'),cut=c.get('cut'),
        fresh_parent_max_error=error,gold=len(ctx.train),masked=len(ctx.pool),query=len(ctx.eval),
        inner_contexts=len(audits),inner_group_overlaps=0,temperatures=temperatures,
        outer_gold_group_overlap=c['gold_group_overlap'],outer_masked_group_overlap=c['masked_group_overlap'],
        outer_future_masked=c['future_masked'],sha256=sha(dest),seconds=time.monotonic()-start)
    save(dest.with_suffix('.json'),meta)
    return c['key'],round(meta['seconds'],2)


def collect(output):
    grouped={};metrics=[];labels=[];audits=[]
    for c in read(HERE/'case_plan.json'):
        if c['access']=='strict_inductive':
            with np.load(HERE/'inputs'/(c['key']+'.npz')) as z:a={k:z[k].copy() for k in z.files}
            a['blend_scores']=blend(a['anchored_scores'],a['cand2_scores']);a['blend_predictions']=select_reasons(a['blend_scores'])
            track='current'
        else:
            path=Path(output)/(c['key']+'.npz');meta=read(path.with_suffix('.json'));assert sha(path)==meta['sha256'];audits.append(meta)
            with np.load(path) as z:a={k:z[k].copy() for k in z.files}
            track='J'
        kind=c['kind'];rep=c.get('repeat')
        if kind in ['train','pooled']:
            # Saved canonical plan names its repeat by seed.
            rep=c['seed'] if 'seed' in c else rep
        if kind in ['B','chronology']:rep=c.get('cut',c['key'])
        if rep is None:rep=c['key']
        grouped.setdefault((track,kind,str(rep)),[]).append(a)
    for (track,kind,rep),parts in grouped.items():
        truth=np.concatenate([a['truth'] for a in parts]);ids=np.concatenate([a['ids'] for a in parts])
        assert len(set(ids))==len(ids)
        for name in ['anchored','cand2','blend']:
            pred=np.concatenate([a[name+'_predictions'] for a in parts])
            row=dict(track=track,kind=kind,repeat=rep,recipe=name,n=len(truth),sample_f1=f1_score(truth,pred,average='samples'),macro_f1=f1_score(truth,pred,average='macro',zero_division=0))
            metrics.append(row)
            precision,recall,f,support=precision_recall_fscore_support(truth,pred,zero_division=0)
            for j,label in enumerate(LABELS):labels.append(dict(track=track,kind=kind,repeat=rep,recipe=name,label=label,precision=precision[j],recall=recall[j],f1=f[j],support=int(support[j])))
    return pd.DataFrame(metrics),pd.DataFrame(labels),audits,grouped


def main():
    p=argparse.ArgumentParser();p.add_argument('--output',type=Path,required=True);p.add_argument('--workers',type=int,default=4)
    args=p.parse_args();freeze();args.output.mkdir(parents=True,exist_ok=True)
    cases=[c for c in read(HERE/'case_plan.json') if c['access']=='native_dylan']
    with ProcessPoolExecutor(max_workers=args.workers) as pool:
        jobs=[]
        for c in cases:
            file=args.output/(c['key']+'.npz')
            if file.exists() and file.with_suffix('.json').exists():continue
            jobs.append(pool.submit(fit_j,c,args.output))
        for i,f in enumerate(as_completed(jobs),1):print(i,len(jobs),f.result(),flush=True)
    metrics,labels,audits,_=collect(args.output)
    metrics.to_csv(args.output/'repeat_metrics.csv',index=False);labels.to_csv(args.output/'label_metrics.csv',index=False)
    save(args.output/'fit_audits.json',audits)
    print(metrics.groupby(['track','kind','recipe']).sample_f1.mean().to_string(),flush=True)
    freeze()


if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/cand2_followup_v1/run.py


## part1/experiments/cand2_followup_v1/source/p1lib/__init__.py

In [32]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/__init__.py
"""p1lib: composable semi-supervised experimentation framework for CS5785 midterm Part 1."""
from .data import LABELS, load, metrics, row_f1, write_submission  # noqa: F401
from .evaluate import evaluate, make_folds, score, fold_outputs, summary_row, paired_se  # noqa: F401
from .pipeline import run_model, validate  # noqa: F401

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/__init__.py


## part1/experiments/cand2_followup_v1/source/p1lib/blocks.py

In [33]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/blocks.py
"""Optional non-embedding feature blocks (each a toggle in the features config).

Every block is label-free: vocabularies / IDF / SVD are fit only on the rows the fold allows
(`fit_rows`), then applied to all rows.  Output is a dense (n_all, d) array, standardized on
`fit_rows`.

Blocks
  criteria  : TF-IDF of eligibility criteria  -> TruncatedSVD(d)
  diseases  : TF-IDF of diseases_cleaned      -> TruncatedSVD(d)
  drugs     : multi-hot matched_drugs         -> TruncatedSVD(d)
  pathways  : multi-hot drug_pathways (optionally only Reactome level <= max_level) -> SVD(d)
  morgan    : RDKit Morgan fingerprint (radius 2, 1024 bits, max over drugs) -> SVD(d)
  drugmeta  : [has_matched_drug, log1p(#matched drugs)]
  year      : standardized start_year  (sampling artifact; ablation only)
"""
from __future__ import annotations

import numpy as np
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from .cache import memo

_MORGAN = None


def _split_semicolon(s):
    if not isinstance(s, str) or not s.strip():
        return []
    return [t.strip().lower() for t in s.split(";") if t.strip()]


def _morgan_matrix(D):
    global _MORGAN
    if _MORGAN is not None:
        return _MORGAN
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog("rdApp.*")
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    out = np.zeros((D.n, 1024), dtype=np.float32)
    for i, s in enumerate(D.df["drug_smiles"].to_numpy()):
        if not isinstance(s, str):
            continue
        fps = [gen.GetFingerprintAsNumPy(m) for t in s.split(";") if (m := Chem.MolFromSmiles(t)) is not None]
        if fps:
            out[i] = np.max(fps, axis=0)
    _MORGAN = out
    return out


def _svd(M, fit_rows, d, seed=0):
    d = int(min(d, M.shape[1] - 1, len(fit_rows) - 1))
    svd = TruncatedSVD(n_components=max(d, 1), random_state=seed).fit(M[fit_rows])
    return svd.transform(M)


def _standardize(Z, fit_rows):
    mu = Z[fit_rows].mean(0)
    sd = Z[fit_rows].std(0)
    sd[sd < 1e-12] = 1.0
    return (Z - mu) / sd


def build_block(D, name: str, cfg: dict, fit_rows: np.ndarray) -> np.ndarray:
    cfg = dict(cfg or {})
    key = {"block": name, **cfg}
    return memo("block", key, fit_rows, lambda: _build(D, name, cfg, fit_rows))


def _build(D, name, cfg, fit_rows):
    d = int(cfg.get("d", 32))
    df = D.df
    if name == "criteria" or name == "diseases":
        col = "criteria" if name == "criteria" else "diseases_cleaned"
        text = df[col].fillna("").astype(str).to_numpy()
        vec = TfidfVectorizer(max_features=int(cfg.get("max_features", 5000)), min_df=3,
                              sublinear_tf=True, ngram_range=(1, int(cfg.get("ngram", 1))),
                              stop_words="english")
        vec.fit(text[fit_rows])
        M = vec.transform(text)
        Z = _svd(M, fit_rows, d)
    elif name in ("drugs", "pathways"):
        col = "matched_drugs" if name == "drugs" else "drug_pathways"
        toks = [_split_semicolon(s) for s in df[col].to_numpy()]
        if name == "pathways" and cfg.get("max_level") is not None:
            lvl = dict(zip(D.catalog["reactome_id"].str.lower(), D.catalog["pathway_level"]))
            ml = int(cfg["max_level"])
            toks = [[t for t in ts if lvl.get(t, 99) <= ml] for ts in toks]
        docs = [" ".join(t.replace(" ", "_") for t in ts) for ts in toks]
        vec = CountVectorizer(binary=True, min_df=3, token_pattern=r"[^ ]+")
        try:
            vec.fit([docs[i] for i in fit_rows])
            M = vec.transform(docs).astype(float)
        except ValueError:            # empty vocabulary
            M = sparse.csr_matrix((D.n, 1))
        Z = _svd(M, fit_rows, d) if M.shape[1] > 2 else M.toarray()
    elif name == "morgan":
        M = _morgan_matrix(D)
        Z = _svd(sparse.csr_matrix(M), fit_rows, d)
    elif name == "drugmeta":
        toks = [_split_semicolon(s) for s in df["matched_drugs"].to_numpy()]
        Z = np.c_[[len(t) > 0 for t in toks], np.log1p([len(t) for t in toks])].astype(float)
    elif name == "year":
        Z = df["start_year"].to_numpy(float)[:, None]
    else:
        raise ValueError(f"unknown block {name}")
    return _standardize(np.asarray(Z, float), fit_rows) * float(cfg.get("weight", 1.0))

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/blocks.py


## part1/experiments/cand2_followup_v1/source/p1lib/cache.py

In [34]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/cache.py
"""Tiny in-process memo for label-free fits (scalers, PCA, clusterers, graphs).

A label-free object depends only on (its config, the set of rows whose features it was fit on).
In transductive mode that row set is the same (all 1,561 rows) for every fold, so one fit is
reused across folds and trials; in inductive mode the row set changes per fold and the cache
simply misses.  Nothing that touches labels is ever cached here.
"""
from __future__ import annotations

import hashlib
import json

import numpy as np

_CACHE: dict = {}
_MAX = 4000


def cfg_key(obj) -> str:
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]


def rows_key(rows: np.ndarray) -> str:
    return hashlib.md5(np.sort(np.asarray(rows)).astype(np.int64).tobytes()).hexdigest()[:16]


def memo(kind: str, cfg, rows, fn):
    key = (kind, cfg_key(cfg), rows_key(rows))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val


def clear():
    _CACHE.clear()


def memo_ctx(kind: str, cfg, ctx, fn, extra=None):
    """Memo for LABEL-DEPENDENT results: keyed by the exact training rows (so a result is only
    reused for the identical fold), the feature rows, the pool and the requested rows."""
    key = (kind, cfg_key(cfg), rows_key(ctx.train), rows_key(ctx.unsup), rows_key(ctx.pool),
           ctx.seed, None if extra is None else cfg_key(extra))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/cache.py


## part1/experiments/cand2_followup_v1/source/p1lib/classifiers.py

In [35]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/classifiers.py
"""Classifiers over the K label combinations.  Common interface:

    clf = make_classifier(cfg, K, CM)
    clf.fit(X, y, w)            # y: int combo ids, w: sample weights (soft labels are
                                #    passed as replicated rows, one per combo, weight lam*q)
    P = clf.predict_proba(X)    # (n, K), columns aligned to all K combos
    clf.prior_                  # effective source prior the model was trained under
                                # (needed by SLD; uniform when class_weight='balanced')

cfg["type"] in {"lr", "ovr", "lda", "knn", "hgb", "nb"}.
"""
from __future__ import annotations

import warnings

import numpy as np
from sklearn.covariance import ledoit_wolf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


def _weighted_prior(y, w, K):
    p = np.bincount(y, weights=w, minlength=K).astype(float)
    return p / p.sum()


def _balance(y, w, K):
    """class_weight='balanced' computed from the weighted label mass."""
    mass = np.bincount(y, weights=w, minlength=K)
    present = mass > 0
    cw = np.zeros(K)
    cw[present] = w.sum() / (present.sum() * mass[present])
    return w * cw[y], present


class _Base:
    def __init__(self, cfg, K, CM):
        self.cfg = cfg
        self.K = K
        self.CM = CM

    def _prep(self, y, w):
        w = np.ones(len(y)) if w is None else np.asarray(w, float)
        if self.cfg.get("class_weight") == "balanced":
            w, present = _balance(y, w, self.K)
            self.prior_ = present / present.sum()
        else:
            self.prior_ = _weighted_prior(y, w, self.K)
        return w

    def _full(self, Pc, classes):
        P = np.zeros((Pc.shape[0], self.K))
        P[:, classes] = Pc
        P = P + 1e-9
        return P / P.sum(1, keepdims=True)


class LRPowerset(_Base):
    """Multinomial logistic regression on the label powerset (T1)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.m = LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                    tol=1e-5).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class OvRLR(_Base):
    """One-vs-rest LR on the 5 labels; combo probabilities = product of marginals renormalized
    over the K allowed combinations (so the output is always a valid label set)."""

    def fit(self, X, y, w=None):
        w0 = np.ones(len(y)) if w is None else np.asarray(w, float)
        self.prior_ = _weighted_prior(y, w0, self.K)
        B = self.CM[y]
        self.ms = []
        for j in range(5):
            yj = B[:, j]
            wj = w0.copy()
            if self.cfg.get("class_weight") == "balanced":
                pos = wj[yj == 1].sum()
                neg = wj[yj == 0].sum()
                if pos > 0 and neg > 0:
                    wj = np.where(yj == 1, wj * (pos + neg) / (2 * pos), wj * (pos + neg) / (2 * neg))
            if yj.min() == yj.max():
                self.ms.append(float(yj[0]))
                continue
            self.ms.append(LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                              tol=1e-5).fit(X, yj, sample_weight=wj))
        if self.cfg.get("class_weight") == "balanced":
            self.prior_ = np.ones(self.K) / self.K
        return self

    def marginals(self, X):
        M = np.zeros((len(X), 5))
        for j, m in enumerate(self.ms):
            M[:, j] = m if isinstance(m, float) else m.predict_proba(X)[:, 1]
        return np.clip(M, 1e-6, 1 - 1e-6)

    def predict_proba(self, X):
        M = self.marginals(X)
        L = np.log(M) @ self.CM.T + np.log(1 - M) @ (1 - self.CM).T
        L -= L.max(1, keepdims=True)
        P = np.exp(L)
        return P / P.sum(1, keepdims=True)


class ShrinkLDA(_Base):
    """Shrinkage LDA (tied Gaussian classes) with sample-weight support.  Follows sklearn's
    solver='lsqr', shrinkage='auto': per-class Ledoit-Wolf covariances (on standardized
    features) averaged with the class priors."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        K, d = self.K, X.shape[1]
        self.classes_ = np.unique(y)
        means = np.zeros((K, d))
        S = np.zeros((d, d))
        prior = self.prior_
        for k in self.classes_:
            m = y == k
            wk = w[m]
            Xk = X[m]
            mu = (wk[:, None] * Xk).sum(0) / wk.sum()
            means[k] = mu
            Xc = (Xk - mu) * np.sqrt(wk / wk.mean())[:, None]
            if len(Xk) < 2:
                continue
            sd = Xc.std(0)
            sd[sd < 1e-12] = 1.0
            C = ledoit_wolf(Xc / sd, assume_centered=True)[0]
            S += prior[k] * (sd[:, None] * C * sd[None, :])
        S += 1e-8 * np.eye(d)
        self.means_ = means
        self.coef_ = np.linalg.solve(S, means[self.classes_].T).T
        self.intercept_ = -0.5 * (self.coef_ * means[self.classes_]).sum(1) + np.log(prior[self.classes_])
        return self

    def predict_proba(self, X):
        Lg = X @ self.coef_.T + self.intercept_
        Lg -= Lg.max(1, keepdims=True)
        P = np.exp(Lg)
        P /= P.sum(1, keepdims=True)
        return self._full(P, self.classes_)


class KNNClf(_Base):
    """Cosine kNN over combos with distance weighting and soft targets; smoothed toward the
    training prior with beta pseudo-counts so probabilities are never exactly 0."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        self.Yw = np.eye(self.K)[y] * w[:, None]
        self.w = w
        return self

    def predict_proba(self, X, exclude_self=False):
        k = int(self.cfg.get("k", 10))
        Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        S = Xn @ self.Xn.T
        if exclude_self:
            np.fill_diagonal(S, -np.inf)
        k = min(k, S.shape[1] - (1 if exclude_self else 0))
        nn = np.argpartition(-S, k - 1, axis=1)[:, :k]
        sims = np.take_along_axis(S, nn, 1)
        if self.cfg.get("weights", "distance") == "distance":
            wt = np.exp(sims / float(self.cfg.get("tau", 0.1)))
        else:
            wt = np.ones_like(sims)
        votes = np.einsum("nk,nkc->nc", wt, self.Yw[nn])
        mass = np.einsum("nk,nk->n", wt, self.w[nn])
        beta = float(self.cfg.get("beta", 1.0))
        P = (votes + beta * self.prior_) / (mass + beta)[:, None]
        return P / P.sum(1, keepdims=True)


class HGB(_Base):
    """HistGradientBoosting over combos (mainly as a stacker on meta-features)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        c = self.cfg
        self.m = HistGradientBoostingClassifier(
            learning_rate=float(c.get("learning_rate", 0.05)), max_iter=int(c.get("max_iter", 100)),
            max_leaf_nodes=int(c.get("max_leaf_nodes", 6)), max_depth=c.get("max_depth", 3),
            min_samples_leaf=int(c.get("min_samples_leaf", 20)),
            l2_regularization=float(c.get("l2_regularization", 1.0)),
            max_features=float(c.get("max_features", 1.0)), early_stopping=False,
            random_state=int(c.get("seed", 0))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class GaussNB(_Base):
    """Gaussian naive Bayes with variance smoothing (cheap, high-bias baseline)."""

    def fit(self, X, y, w=None):
        from sklearn.naive_bayes import GaussianNB
        w = self._prep(y, w)
        self.m = GaussianNB(var_smoothing=float(self.cfg.get("var_smoothing", 1e-2))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class PriorClf(_Base):
    """Predicts the training prior for every row (MAP -> always the majority combo)."""

    def fit(self, X, y, w=None):
        self._prep(y, w)
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))


REGISTRY = {"prior": PriorClf, "lr": LRPowerset, "ovr": OvRLR, "lda": ShrinkLDA, "knn": KNNClf, "hgb": HGB, "nb": GaussNB}


def make_classifier(cfg, K, CM):
    return REGISTRY[cfg["type"]](cfg, K, CM)

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/classifiers.py


## part1/experiments/cand2_followup_v1/source/p1lib/configs.py

In [36]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/configs.py
"""Named reference configurations (the report's five techniques, baselines and ensembles).

Every entry is a full experiment config: {"model": ..., "head": ..., "transductive": ...}.
Build new ones by copying and editing; `exp(model, head)` is a small helper.
"""
from __future__ import annotations

import copy

STD = {"scale": True}
PCA64 = {"scale": True, "pca": 64}
GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}

NO_HEAD = {"prior": "none", "decode": "map"}
SLD_HEAD = {"prior": "sld", "sld_a": 100, "decode": "map"}


def exp(model, head=None, transductive=True):
    return {"model": copy.deepcopy(model), "head": copy.deepcopy(head or NO_HEAD), "transductive": transductive}


def single(clf, features=STD, synth=None, labeler=None, temperature=False):
    return {"features": copy.deepcopy(features), "synth": copy.deepcopy(synth or []),
            "labeler": copy.deepcopy(labeler), "clf": copy.deepcopy(clf), "temperature": temperature}


LR = {"type": "lr", "C": 0.1}
LDA = {"type": "lda"}
OVR = {"type": "ovr", "C": 0.1}
KNN = {"type": "knn", "k": 10, "weights": "distance", "tau": 0.1, "beta": 1.0}

M_LR = single(LR)
M_LDA = single(LDA)
M_OVR = single(OVR)
M_KNN = single(KNN, features={"scale": True, "pca": 64, "l2": True})
M_CS = single({"type": "direct", "labeler": {"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5}})
M_SPREAD = single({"type": "direct", "labeler": {"type": "spread", "graph": GRAPH, "alpha": 0.5}})
M_CLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "kmeans", "k": 24, "m": 4}})
M_GMMCLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "gmm", "k": 16, "cov": "diag", "m": 4}})

# T1 with SLD-filled pseudo-labels for the unlabeled pool
M_LR_SLDPL = single(LR, labeler={"type": "sld", "base": LR, "a": 100, "lam": 0.1})
# T2 C&S pseudo-labels feeding LR
M_LR_CSPL = single(LR, labeler={"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5, "lam": 0.1})
# T4 cluster-then-label pseudo-labels feeding LR
M_LR_CLPL = single(LR, labeler={"type": "cluster", "algo": "kmeans", "k": 24, "m": 4, "lam": 0.1})
# T3 guarded self-training
M_LR_ST = single(LR, labeler={"type": "selftrain", "members": ["lr", "lda", "cs"], "rounds": 2, "tau": 0.75,
                              "lam": 0.3, "ramp": True, "align": "sld", "weighting": "softmatch",
                              "cluster_filter": True})
M_LR_ST_NAIVE = single(LR, labeler={"type": "selftrain", "members": ["lr"], "rounds": 3, "tau": 0.75,
                                    "lam": 0.5, "align": "none", "hard": True})
# synthetic-feature variants
M_LR_SYN = single(LR, synth=[{"type": "gmm", "k": 16, "cov": "diag"}, {"type": "knn_vote", "k": 10},
                             {"type": "graph", "mode": "spread", "graph": GRAPH, "alpha": 0.5},
                             {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}])
# T5 HGB stacker on nested-OOF meta-features
META = [{"type": "oof_model", "model": M_LR}, {"type": "oof_model", "model": M_LDA},
        {"type": "graph", "mode": "cs", "graph": GRAPH, "alpha": 0.5},
        {"type": "knn_vote", "k": 10}, {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}]
HGB = {"type": "hgb", "learning_rate": 0.05, "max_iter": 100, "max_leaf_nodes": 4, "max_depth": 2,
       "min_samples_leaf": 20, "l2_regularization": 1.0}
M_STACK_HGB = single(HGB, features=None, synth=META)
M_STACK_LR = single({"type": "lr", "C": 0.1}, features=None, synth=META)
M_HGB_RAW = single(HGB, features=STD)

# report's recommended ensemble: T1-LR + T1-LDA + T2-C&S, equal weights
M_ENS3 = {"ensemble": [M_LR, M_LDA, M_CS], "weights": None}

M_PRIOR = single({"type": "prior"})
M_NB = single({"type": "nb"}, features=PCA64)
SLD_EF1 = {"prior": "sld", "sld_a": 25, "decode": "expf1"}

REFERENCE = {
    "Majority": exp(M_PRIOR),
    "T1_LR": exp(M_LR),
    "T1_LR+SLD": exp(M_LR, SLD_HEAD),
    "T1_LDA": exp(M_LDA),
    "T1_LDA+SLD": exp(M_LDA, SLD_HEAD),
    "OvR_LR": exp(M_OVR),
    "kNN": exp(M_KNN),
    "T2_CS": exp(M_CS),
    "T2_CS+SLD": exp(M_CS, SLD_HEAD),
    "T2_spread": exp(M_SPREAD),
    "T4_kmeans_direct": exp(M_CLUSTER),
    "T4_gmm_direct": exp(M_GMMCLUSTER),
    "T1_LR+SLDpseudo": exp(M_LR_SLDPL),
    "T1_LR+SLDpseudo+SLD": exp(M_LR_SLDPL, SLD_HEAD),
    "T2_LR+CSpseudo": exp(M_LR_CSPL),
    "T4_LR+clusterpseudo": exp(M_LR_CLPL),
    "T3_selftrain_guarded": exp(M_LR_ST),
    "T3_selftrain_naive": exp(M_LR_ST_NAIVE),
    "LR+synth": exp(M_LR_SYN),
    "T5_stack_HGB": exp(M_STACK_HGB),
    "T5_stack_HGB+SLD": exp(M_STACK_HGB, SLD_HEAD),
    "stack_LR": exp(M_STACK_LR),
    "HGB_raw": exp(M_HGB_RAW),
    "ENS3": exp(M_ENS3),
    "ENS3+SLD": exp(M_ENS3, SLD_HEAD),
}

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/configs.py


## part1/experiments/cand2_followup_v1/source/p1lib/context.py

In [37]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/context.py
"""Fold context: which rows are labeled-for-training, which are predicted, which are unlabeled.

Every component receives a `Ctx` and may only read labels of `ctx.train`.  Features of
`ctx.unsup` rows may be used by label-free steps (scaler, PCA, clustering, graph nodes).

transductive=True  -> unsup = all 1,561 rows (train + val + test features, no labels)
transductive=False -> unsup = all rows except the eval rows and the test rows
"""
from __future__ import annotations

from dataclasses import dataclass, replace

import numpy as np


@dataclass
class Ctx:
    D: object
    train: np.ndarray            # global idx of labeled training rows (labels visible)
    eval: np.ndarray             # global idx of rows to predict (labels hidden)
    pool: np.ndarray             # global idx of the pseudo-label pool (default: 1,014 unlabeled)
    unsup: np.ndarray            # global idx of rows whose features label-free steps may use
    transductive: bool = True
    seed: int = 0
    name: str = ""
    depth: int = 0               # nesting depth (0 = outer fold)

    @property
    def y_train(self):
        return self.D.y[self.train]

    @property
    def sld_target(self):
        """Rows SLD estimates the target prior on: the eval batch when transductive, else the
        unlabeled pool (inductive: we may not look at the eval/test features)."""
        return self.eval if self.transductive else self.pool

    def sub(self, train, eval_, name=None):
        """Inner context for nested OOF: same unlabeled pool / feature rows, fewer labels."""
        return replace(self, train=np.asarray(train), eval=np.asarray(eval_),
                       name=name or self.name + "/inner", depth=self.depth + 1)


def make_ctx(D, train, eval_, transductive=True, pool=None, seed=0, name=""):
    train = np.asarray(train)
    eval_ = np.asarray(eval_)
    pool = D.idx["unl"] if pool is None else np.asarray(pool)
    if transductive:
        unsup = np.arange(D.n)
    else:
        banned = np.zeros(D.n, bool)
        banned[eval_] = True
        banned[D.idx["test"]] = True
        unsup = np.where(~banned)[0]
    assert len(np.intersect1d(train, eval_)) == 0, "train/eval overlap"
    assert (D.y[train] >= 0).all(), "training rows must be labeled"
    return Ctx(D=D, train=train, eval=eval_, pool=pool, unsup=unsup,
               transductive=transductive, seed=seed, name=name)

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/context.py


## part1/experiments/cand2_followup_v1/source/p1lib/data.py

In [38]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/data.py
"""Data loading, label-powerset encoding and metrics for Part 1.

Global row order used everywhere in this package:
    [ part1_train (1171 rows, file order) | part1_val (78) | part1_test (312) ]  -> 1561 rows.
`Data.y` holds the label-combo id (0..7) for labeled rows and -1 for masked/test rows.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field

import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

LABELS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMB = [f"emb_{i}" for i in range(150)]
BITS = np.array([1, 2, 4, 8, 16])

DEFAULT_DATA_DIR = os.environ.get(
    "P1_DATA_DIR",
    os.path.join(os.path.dirname(__file__), "..", "..", "..", "part1", "data"),
)


@dataclass
class Data:
    df: pd.DataFrame                 # all 1561 rows, global order, with a 'split' column
    emb: np.ndarray                  # (1561, 150) raw embeddings
    Y: np.ndarray                    # (1561, 5) labels, -1 where unknown
    y: np.ndarray                    # (1561,) combo id, -1 where unknown
    combos: np.ndarray               # (K,) bitmask of each allowed combo
    CM: np.ndarray                   # (K, 5) binary label matrix of each combo
    idx: dict = field(default_factory=dict)   # named index groups
    year: np.ndarray = None
    catalog: pd.DataFrame = None

    @property
    def K(self) -> int:
        return len(self.combos)

    @property
    def n(self) -> int:
        return len(self.df)

    def combo_names(self):
        return ["+".join(LABELS[j][0] for j in range(5) if self.CM[k, j]) for k in range(self.K)]


def load(data_dir: str | None = None) -> Data:
    d = os.path.abspath(data_dir or DEFAULT_DATA_DIR)
    tr = pd.read_csv(os.path.join(d, "part1_train.csv"))
    va = pd.read_csv(os.path.join(d, "part1_val.csv"))
    te = pd.read_csv(os.path.join(d, "part1_test.csv"))
    cat = pd.read_csv(os.path.join(d, "pathway_catalog.csv"))
    te = pd.concat([te, pd.DataFrame(-1, index=te.index, columns=LABELS)], axis=1)
    tr = tr.assign(split="train")
    va = va.assign(split="val")
    te = te.assign(split="test")
    df = pd.concat([tr, va, te], ignore_index=True).copy()
    Y = df[LABELS].to_numpy().astype(int)
    Y[Y == -100] = -1
    known = (Y >= 0).all(1)
    mask = (Y.clip(0) * BITS).sum(1)
    combos = np.unique(mask[known])
    y = np.full(len(df), -1)
    y[known] = np.searchsorted(combos, mask[known])
    CM = np.array([[(c >> j) & 1 for j in range(5)] for c in combos])
    split = df["split"].to_numpy()
    idx = {
        "train": np.where(split == "train")[0],
        "lab": np.where((split == "train") & known)[0],
        "unl": np.where((split == "train") & ~known)[0],
        "val": np.where(split == "val")[0],
        "test": np.where(split == "test")[0],
    }
    idx["labval"] = np.r_[idx["lab"], idx["val"]]
    D = Data(df=df, emb=df[EMB].to_numpy(float), Y=Y, y=y, combos=combos, CM=CM, idx=idx,
             year=df["start_year"].to_numpy(int), catalog=cat)
    assert len(idx["lab"]) == 157 and len(idx["unl"]) == 1014 and len(idx["val"]) == 78 and len(idx["test"]) == 312
    assert (Y[idx["lab"]].sum(1) >= 1).all(), "labeled rows are never all-zero"
    return D


# ----------------------------------------------------------------------------- metrics
def row_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-row F1 between binary label sets (both non-empty here)."""
    inter = (Ytrue * Ypred).sum(1)
    denom = Ytrue.sum(1) + Ypred.sum(1)
    out = np.where(denom > 0, 2 * inter / np.maximum(denom, 1), 1.0)
    return out


def label_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-label (column) binary F1; F1 = 0 when a label has no positives and none predicted."""
    tp = (Ytrue * Ypred).sum(0)
    fp = ((1 - Ytrue) * Ypred).sum(0)
    fn = (Ytrue * (1 - Ypred)).sum(0)
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)


def metrics(Ytrue: np.ndarray, Ypred: np.ndarray) -> dict:
    rf = row_f1(Ytrue, Ypred)
    lf = label_f1(Ytrue, Ypred)
    return {
        "sample_f1": float(rf.mean()),
        "sample_f1_se": float(rf.std(ddof=1) / np.sqrt(len(rf))),
        "macro_f1": float(lf.mean()),
        **{f"f1_{LABELS[j]}": float(lf[j]) for j in range(5)},
        "n": int(len(rf)),
    }


def bootstrap_se(Ytrue: np.ndarray, Ypreds: list[np.ndarray], B: int = 300, seed: int = 0) -> dict:
    """Bootstrap SE (over rows) of macro and per-label F1, averaged over a list of prediction
    matrices for the same rows (e.g. CV repeats) using shared resamples."""
    rng = np.random.default_rng(seed)
    n = len(Ytrue)
    macro, per = [], []
    for _ in range(B):
        b = rng.integers(0, n, n)
        lfs = np.mean([label_f1(Ytrue[b], P[b]) for P in Ypreds], axis=0)
        per.append(lfs)
        macro.append(lfs.mean())
    per = np.array(per)
    return {"macro_f1_se": float(np.std(macro, ddof=1)),
            **{f"f1_{LABELS[j]}_se": float(per[:, j].std(ddof=1)) for j in range(5)}}


def combos_to_labels(D: Data, kidx: np.ndarray) -> np.ndarray:
    return D.CM[kidx]


def write_submission(D: Data, Ypred: np.ndarray, path: str, ids=None) -> pd.DataFrame:
    """Kaggle format: ID,Predicted with space-separated reason names (always >=1)."""
    assert (Ypred.sum(1) >= 1).all(), "every test row needs at least one predicted reason"
    ids = D.df["ID"].to_numpy()[D.idx["test"]] if ids is None else ids
    pred = [" ".join(LABELS[j] for j in range(5) if row[j]) for row in Ypred]
    sub = pd.DataFrame({"ID": ids, "Predicted": pred})
    sub.to_csv(path, index=False, lineterminator="\n")
    return sub

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/data.py


## part1/experiments/cand2_followup_v1/source/p1lib/evaluate.py

In [39]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/evaluate.py
"""Leak-free evaluation harness.

Protocols (all over the 235 labeled rows = 157 labeled-train + 78 val):
  A : RepeatedStratifiedKFold(5 x R) stratified on the label-combo id         (no shift)
  B : chronological holdout: train start_year < cut, evaluate >= cut         (shift, cut=2005)
  C : official split: train on the 157 labeled-train rows, evaluate on val   (tie-breaker)
Every fold builds a fresh Ctx; everything label-dependent is fit inside `run_model` from
ctx.train labels only.  Label-free fits are reused across folds only when their feature-row
set is identical (transductive mode).

Fold outputs (calibrated combo probabilities BEFORE the head) are cached on disk keyed by the
model part of the config, so SLD strength / decoding are post-hoc and free to vary.
"""
from __future__ import annotations

import hashlib
import json
import os
import time

import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold

from .context import make_ctx
from .data import LABELS, bootstrap_se, label_f1, metrics, row_f1
from .head import apply_head
from .pipeline import model_part, run_model, validate

# fold-output cache lives OUTSIDE the midterm/ folder (regenerable, ~0.5 GB); override with P1_CACHE
CACHE_DIR = os.environ.get("P1_CACHE", os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "..", ".p1_cache", "folds")))
CHRONO_CUT = 2005
CHRONO_CUTS = (2004, 2005, 2006)   # objective uses the mean over these cuts


def cfg_hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:20]


# ----------------------------------------------------------------------------- folds
def make_folds(D, protocols=("A", "B", "C"), R=4, seed=0, rows=None, chrono_cuts=CHRONO_CUTS):
    """rows: restrict to a subset of labeled rows (used by the nested search-overfitting check)."""
    labval = D.idx["labval"] if rows is None else np.asarray(rows)
    y = D.y[labval]
    folds = []
    if "A" in protocols:
        rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=R, random_state=seed)
        for i, (tr, te) in enumerate(rskf.split(np.zeros(len(y)), y)):
            folds.append({"name": f"A{i // 5}_{i % 5}", "protocol": "A", "repeat": i // 5,
                          "train": labval[tr], "eval": labval[te]})
    if "B" in protocols:
        for cut in chrono_cuts:
            yr = D.year[labval]
            folds.append({"name": f"B{cut}", "protocol": "B", "cut": cut,
                          "train": labval[yr < cut], "eval": labval[yr >= cut]})
    if "C" in protocols:
        tr = np.intersect1d(labval, D.idx["lab"])
        te = np.intersect1d(labval, D.idx["val"])
        if len(tr) and len(te):
            folds.append({"name": "C", "protocol": "C", "train": tr, "eval": te})
    return folds


def fold_tag(fold):
    h = hashlib.md5(np.r_[fold["train"], [-1], fold["eval"]].astype(np.int64).tobytes()).hexdigest()[:10]
    return f"{fold['name']}_{h}"


# ----------------------------------------------------------------------------- one fold
def run_fold(cfg, D, fold, use_cache=True):
    mp = model_part(cfg)
    key = cfg_hash(mp)
    path = os.path.join(CACHE_DIR, key, fold_tag(fold) + ".npz")
    if use_cache and os.path.exists(path):
        try:
            z = np.load(path, allow_pickle=False)
            out = {k: z[k] for k in z.files}
            out["T"] = out["T"].tolist()
            return out
        except Exception:
            pass
    tr = cfg.get("transductive", True)
    ctx = make_ctx(D, fold["train"], fold["eval"], transductive=tr, seed=0, name=fold["name"])
    t0 = time.time()
    rows = fold["eval"] if tr else np.r_[fold["eval"], ctx.pool]
    o = run_model(cfg["model"], ctx, rows)
    ne = len(fold["eval"])
    out = {"P_eval": o["P"][:ne].astype(np.float32), "prior": np.asarray(o["prior"], float),
           "T": np.atleast_1d(np.asarray(o["T"], float)), "secs": np.float32(time.time() - t0)}
    if not tr:
        out["P_target"] = o["P"][ne:].astype(np.float32)
    if use_cache:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + f".{os.getpid()}.tmp.npz"
        np.savez_compressed(tmp, **out)
        os.replace(tmp, path)
        mpath = os.path.join(CACHE_DIR, key, "config.json")
        if not os.path.exists(mpath):
            with open(mpath, "w") as f:
                json.dump(mp, f, indent=1, default=str)
    out["T"] = out["T"].tolist()
    return out


def fold_outputs(cfg, D, folds, n_jobs=1, use_cache=True):
    if n_jobs == 1:
        return [run_fold(cfg, D, f, use_cache) for f in folds]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=n_jobs, backend="loky")(delayed(run_fold)(cfg, D, f, use_cache) for f in folds)


# ----------------------------------------------------------------------------- scoring
def score(cfg, D, folds, outs, bootstrap=True):
    """Apply the head to cached fold outputs and compute A/B/C metrics with SEs."""
    head = cfg.get("head", {})
    res = {}
    per = {}
    for f, o in zip(folds, outs):
        P = o["P_eval"].astype(float)
        P = P / P.sum(1, keepdims=True)
        Pt = o.get("P_target")
        Pt = None if Pt is None else Pt.astype(float) / Pt.astype(float).sum(1, keepdims=True)
        Yp, Q, pi = apply_head(P, o["prior"], D.CM, head, Pt)
        per[f["name"]] = (f, Yp, pi)
    # ---- A ----
    A = [v for v in per.values() if v[0]["protocol"] == "A"]
    if A:
        rows = np.unique(np.concatenate([f["eval"] for f, _, _ in A]))
        pos = {r: i for i, r in enumerate(rows)}
        reps = sorted({f["repeat"] for f, _, _ in A})
        Ytrue = D.Y[rows]
        Ypreds = []
        for r in reps:
            Yp = np.zeros((len(rows), 5), int)
            for f, yp, _ in A:
                if f["repeat"] == r:
                    Yp[[pos[i] for i in f["eval"]]] = yp
            Ypreds.append(Yp)
        rep_metrics = [metrics(Ytrue, Yp) for Yp in Ypreds]
        rf = np.mean([row_f1(Ytrue, Yp) for Yp in Ypreds], axis=0)
        a = {k: float(np.mean([m[k] for m in rep_metrics])) for k in rep_metrics[0] if k not in ("n",)}
        a["sample_f1_se"] = float(rf.std(ddof=1) / np.sqrt(len(rf)))
        a["sample_f1_rep_sd"] = float(np.std([m["sample_f1"] for m in rep_metrics], ddof=1)) if len(reps) > 1 else 0.0
        a["n"] = int(len(rows))
        a["R"] = len(reps)
        if bootstrap:
            a.update(bootstrap_se(Ytrue, Ypreds, B=200))
        res["A"] = a
        res["rowf1_A"] = rf.tolist()
        res["rows_A"] = rows.tolist()
    # ---- B ----  (one or several chronological cuts; B = mean over cuts, B_se = mean per-cut SE
    #               because the cuts share most rows and are highly correlated)
    Bs = [v for v in per.values() if v[0]["protocol"] == "B"]
    if Bs:
        bm = []
        for f, yp, pi in Bs:
            Yt = D.Y[f["eval"]]
            m = metrics(Yt, yp)
            if bootstrap:
                m.update(bootstrap_se(Yt, [yp], B=200))
            m["cut"] = f["cut"]
            if pi is not None:
                m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
            m["true_prior_labels"] = Yt.mean(0).round(3).tolist()
            bm.append(m)
            res.setdefault("rowf1_B", []).extend(row_f1(Yt, yp).tolist())
        res["B_ncuts"] = len(bm)
        if len(bm) == 1:
            res["B"] = bm[0]
        else:
            agg = {k: float(np.mean([m[k] for m in bm])) for k in bm[0]
                   if isinstance(bm[0][k], (int, float)) and k != "cut"}
            agg["cuts"] = bm
            res["B"] = agg
        for m in bm:
            if m["cut"] == CHRONO_CUT:
                res["B2005"] = {k: v for k, v in m.items()}
    # ---- C ----
    Cs = [v for v in per.values() if v[0]["protocol"] == "C"]
    if Cs:
        f, yp, pi = Cs[0]
        Yt = D.Y[f["eval"]]
        m = metrics(Yt, yp)
        if bootstrap:
            m.update(bootstrap_se(Yt, [yp], B=200))
        if pi is not None:
            m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
        res["C"] = m
        res["rowf1_C"] = row_f1(Yt, yp).tolist()
    # ---- robust objective ----
    if "A" in res and "B" in res:
        res["J"] = 0.5 * res["A"]["sample_f1"] + 0.5 * res["B"]["sample_f1"]
        res["J_se"] = 0.5 * float(np.hypot(res["A"]["sample_f1_se"], res["B"]["sample_f1_se"]))
    res["secs"] = float(sum(float(o["secs"]) for o in outs))
    res["T_mean"] = float(np.mean([np.mean(o["T"]) for o in outs]))
    return res


def evaluate(cfg, D, protocols=("A", "B", "C"), R=4, n_jobs=1, seed=0, use_cache=True, rows=None,
             bootstrap=True, chrono_cuts=CHRONO_CUTS):
    validate(cfg)
    folds = make_folds(D, protocols, R, seed, rows=rows, chrono_cuts=chrono_cuts)
    outs = fold_outputs(cfg, D, folds, n_jobs, use_cache)
    return score(cfg, D, folds, outs, bootstrap=bootstrap)


def summary_row(name, res):
    r = {"name": name}
    for p in ("A", "B", "B2005", "C"):
        if p in res:
            r[f"{p}"] = res[p]["sample_f1"]
            r[f"{p}_se"] = res[p]["sample_f1_se"]
            r[f"{p}_macro"] = res[p]["macro_f1"]
    if "J" in res:
        r["J"] = res["J"]
        r["J_se"] = res["J_se"]
    r["secs"] = res.get("secs")
    return r


def paired_se(res1, res2, w=(0.5, 0.5)):
    """SE of the difference in J between two configs using per-row paired differences."""
    dA = np.asarray(res1["rowf1_A"]) - np.asarray(res2["rowf1_A"])
    dB = np.asarray(res1["rowf1_B"]) - np.asarray(res2["rowf1_B"])
    nB = len(dB) / res1.get("B_ncuts", 1)          # cuts overlap: effective n ~ one cut
    return float(np.sqrt((w[0] ** 2) * dA.var(ddof=1) / len(dA) + (w[1] ** 2) * dB.var(ddof=1) / nB))

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/evaluate.py


## part1/experiments/cand2_followup_v1/source/p1lib/features.py

In [40]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/features.py
"""Preprocessors: standardize -> [PCA] -> [L2-normalize] on the embeddings, plus optional
non-embedding blocks.  All fitting uses only `ctx.unsup` rows (label-free), so the same
object is refit per fold when the feature-row set changes (inductive) and reused otherwise.

features config (all keys optional):
    {"scale": True, "pca": None|int, "whiten": False, "l2": False,
     "blocks": {"criteria": {"d": 32, "weight": 1.0}, "pathways": {...}, ...}}
"""
from __future__ import annotations

import numpy as np
from sklearn.decomposition import PCA

from .blocks import build_block
from .cache import memo

DEFAULT_FEATURES = {"scale": True, "pca": None, "l2": False, "blocks": {}}


def norm_features_cfg(cfg):
    c = dict(DEFAULT_FEATURES)
    c.update(cfg or {})
    c["blocks"] = dict(c.get("blocks") or {})
    return c


def _embed(D, cfg, rows):
    X = D.emb.copy()
    if cfg["scale"]:
        mu = X[rows].mean(0)
        sd = X[rows].std(0)
        X = (X - mu) / sd
    if cfg["pca"]:
        p = PCA(n_components=int(cfg["pca"]), whiten=bool(cfg.get("whiten", False)), random_state=0).fit(X[rows])
        X = p.transform(X)
    if cfg["l2"]:
        X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    return X


def featurize(ctx, cfg) -> np.ndarray:
    """Return the (n_all, d) representation for every global row."""
    cfg = norm_features_cfg(cfg)
    emb_cfg = {k: cfg[k] for k in ("scale", "pca", "l2", "whiten") if k in cfg}
    X = memo("embed", emb_cfg, ctx.unsup, lambda: _embed(ctx.D, cfg, ctx.unsup))
    if not cfg["blocks"]:
        return X
    parts = [X] if cfg.get("use_emb", True) else []
    for name in sorted(cfg["blocks"]):
        parts.append(build_block(ctx.D, name, cfg["blocks"][name], ctx.unsup))
    return np.hstack(parts)

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/features.py


## part1/experiments/cand2_followup_v1/source/p1lib/graph.py

In [41]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/graph.py
"""Symmetric RBF-weighted kNN graph over the feature rows a fold may use, and
normalized-Laplacian diffusion (Zhou et al. 2004 label spreading; Huang et al. 2021 C&S).

graph config: {"space": features-cfg, "k": 10, "s": 1.0, "sym": "max"|"min"}
Weights w_ij = exp(-d_ij^2 / (s * median kNN distance)^2) on cosine-normalized vectors,
symmetrized with max (union) or min (mutual kNN), S = D^-1/2 W D^-1/2.
Diffusion solves (I - alpha S) F = (1 - alpha) F0 exactly (LU factor cached per alpha).
"""
from __future__ import annotations

import numpy as np
import scipy.sparse as sp
from scipy.linalg import lu_factor, lu_solve
from sklearn.neighbors import NearestNeighbors

from .cache import memo
from .features import featurize

DEFAULT_GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}


class Graph:
    def __init__(self, Z_nodes, nodes, n_all, k, s, sym):
        self.nodes = nodes
        self.pos = np.full(n_all, -1)
        self.pos[nodes] = np.arange(len(nodes))
        self.Zn = Z_nodes / np.maximum(np.linalg.norm(Z_nodes, axis=1, keepdims=True), 1e-12)
        self.nn = NearestNeighbors(n_neighbors=k + 1).fit(self.Zn)
        dist, ind = self.nn.kneighbors(self.Zn)
        dist, ind = dist[:, 1:], ind[:, 1:]
        self.sigma = s * np.median(dist)
        n = len(nodes)
        rows = np.repeat(np.arange(n), k)
        W = sp.csr_matrix((np.exp(-(dist.ravel() ** 2) / self.sigma ** 2), (rows, ind.ravel())), shape=(n, n))
        W = W.maximum(W.T) if sym == "max" else W.minimum(W.T)
        deg = np.asarray(W.sum(1)).ravel()
        iso = deg <= 0
        if iso.any():                      # isolated nodes (possible with mutual kNN): self-loop
            W = W + sp.diags(iso.astype(float))
            deg = np.asarray(W.sum(1)).ravel()
        Dm = sp.diags(1 / np.sqrt(deg))
        self.S = (Dm @ W @ Dm).toarray()
        self.k = k
        self._lu = {}

    def propagate(self, F0, alpha):
        if alpha <= 0:
            return F0.copy()
        if alpha not in self._lu:
            self._lu[alpha] = lu_factor(np.eye(len(self.S)) - alpha * self.S)
        return (1 - alpha) * lu_solve(self._lu[alpha], F0)

    def extend(self, Z_rows, F_nodes):
        """Inductive extension for rows that are not graph nodes: RBF-weighted kNN average."""
        Zn = Z_rows / np.maximum(np.linalg.norm(Z_rows, axis=1, keepdims=True), 1e-12)
        dist, ind = self.nn.kneighbors(Zn, n_neighbors=self.k)
        w = np.exp(-dist ** 2 / self.sigma ** 2) + 1e-12
        return np.einsum("nk,nkc->nc", w, F_nodes[ind]) / w.sum(1, keepdims=True)


def get_graph(ctx, gcfg) -> Graph:
    g = dict(DEFAULT_GRAPH)
    g.update(gcfg or {})
    Z = featurize(ctx, g["space"])

    def build():
        return Graph(Z[ctx.unsup], ctx.unsup, ctx.D.n, int(g["k"]), float(g["s"]), g["sym"])

    return memo("graph", {k: g[k] for k in ("space", "k", "s", "sym")}, ctx.unsup, build), Z


def node_values(graph, Z, F_nodes, rows):
    """Read propagated values for arbitrary global rows (extend for non-nodes)."""
    out = np.zeros((len(rows), F_nodes.shape[1]))
    p = graph.pos[rows]
    m = p >= 0
    out[m] = F_nodes[p[m]]
    if (~m).any():
        out[~m] = graph.extend(Z[rows[~m]], F_nodes)
    return out


def label_spread(ctx, gcfg, alpha, train_rows, y_train, K, out_rows, base_all=None):
    """Pure label spreading (base_all=None) or the Smooth step of C&S (base_all = (n_all,K)
    base probabilities).  Training nodes are clamped to their one-hot labels in F0."""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    F0 = np.zeros((n, K))
    if base_all is not None:
        F0 = base_all[graph.nodes].copy()
    p = graph.pos[train_rows]
    keep = p >= 0
    F0[p[keep]] = np.eye(K)[y_train[keep]]
    F = graph.propagate(F0, alpha)
    return node_values(graph, Z, F, out_rows)


def correct_and_smooth(ctx, gcfg, a1, a2, train_rows, y_train, P_oof_train, P_all, K, out_rows,
                       scale_mode="autoscale"):
    """Correct & Smooth (Huang et al. 2021).
    P_oof_train : base probabilities of the training rows, OUT-OF-FOLD (in-sample residuals ~0)
    P_all       : base probabilities for every global row (fit on all training rows)"""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    p = graph.pos[train_rows]
    keep = p >= 0
    Ytr = np.eye(K)[y_train]
    base = P_all[graph.nodes].copy()
    if a1 > 0:
        E0 = np.zeros((n, K))
        E0[p[keep]] = (Ytr - P_oof_train)[keep]
        E = graph.propagate(E0, a1)
        if scale_mode == "autoscale":
            sigma = np.abs(E0[p[keep]]).sum(1).mean()
            scale = sigma / np.maximum(np.abs(E).sum(1, keepdims=True), 1e-12)
            base = base + scale * E
        else:
            base = base + float(scale_mode) * E
        base = np.clip(base, 0, None)
        base = base / np.maximum(base.sum(1, keepdims=True), 1e-12)
    F0 = base.copy()
    F0[p[keep]] = Ytr[keep]
    F = graph.propagate(F0, a2)
    out = node_values(graph, Z, F, out_rows)
    out = np.clip(out, 0, None) + 1e-9
    return out / out.sum(1, keepdims=True)

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/graph.py


## part1/experiments/cand2_followup_v1/source/p1lib/head.py

In [42]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/head.py
"""Shared head: temperature calibration, ONE prior correction (SLD-EM, MAP-stabilized),
and threshold-free decoding to an allowed label set with >= 1 label.

All probabilities are over the K allowed label combinations (label powerset).
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax

EPS = 1e-6


def safe_log(P):
    return np.log(np.clip(P, EPS, None))


def renorm(P):
    P = np.clip(P, 0, None) + 1e-12
    return P / P.sum(1, keepdims=True)


# ----------------------------------------------------------------------------- temperature
def fit_temperature(P_oof: np.ndarray, y: np.ndarray, w: np.ndarray | None = None,
                    bounds=(0.05, 20.0)) -> float:
    """One-parameter temperature T minimizing NLL of softmax(log P / T) on OOF probabilities."""
    L = safe_log(P_oof)
    w = np.ones(len(y)) if w is None else w

    def nll(logT):
        T = np.exp(logT)
        lp = log_softmax(L / T, axis=1)
        return -(w * lp[np.arange(len(y)), y]).sum() / w.sum()

    r = minimize_scalar(nll, bounds=(np.log(bounds[0]), np.log(bounds[1])), method="bounded",
                        options={"xatol": 1e-4})
    return float(np.exp(r.x))


def apply_temperature(P: np.ndarray, T: float) -> np.ndarray:
    if T is None or T == 1.0:
        return P
    return np.exp(log_softmax(safe_log(P) / T, axis=1))


# ----------------------------------------------------------------------------- SLD-EM
def sld_em(P: np.ndarray, prior_src: np.ndarray, a: float | None = 0.0, floor: float = 0.25,
           tol: float = 1e-8, max_iter: int = 1000, cap: float | None = None):
    """Saerens-Latinne-Decaestecker EM for the target label prior, with a Dirichlet(a*pi_s)
    pseudo-count toward the source prior (MAP) and a floor pi >= floor*pi_s.

    cap: optional ceiling pi_k <= cap * pi_s,k (guards the degenerate MLE in which a rare
    'bridge' combo such as Safety+Efficacy absorbs the mass of its parents).
    Returns (Q, pi_hat): prior-corrected posteriors and the estimated target prior.
    a=None or a=inf means 'off' (returns P unchanged)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6            # combos absent from training stay at exactly 0
    ps = np.where(present, prior_src, 0.0)
    ps = ps / ps.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ratio_den = np.where(present, ps, 1.0)
    pi = ps.copy()
    n = len(P)
    for _ in range(max_iter):
        Q = P * (pi / ratio_den)
        Q /= Q.sum(1, keepdims=True)
        new = (Q.sum(0) + a * ps) / (n + a)
        new = np.maximum(new, floor * ps)
        if cap:
            new = np.minimum(new, cap * ps)
        new /= new.sum()
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * (pi / ratio_den)
    Q /= Q.sum(1, keepdims=True)
    return Q, pi


def sld_em_factorized(P: np.ndarray, prior_src: np.ndarray, CM: np.ndarray, a: float | None = 0.0,
                      floor: float = 0.25, tol: float = 1e-8, max_iter: int = 1000):
    """Factorized SLD: the target/source combo-prior ratio is constrained to a product of
    per-label ratios, w_k = prod_j (pi_j/ps_j)^{CM_kj} ((1-pi_j)/(1-ps_j))^{1-CM_kj}.
    Only 5 free parameters (one per reason) instead of K-1, so a rare 'bridge' combo
    (Safety+Efficacy) cannot absorb the mass of its parents.  EM-style fixed point: E-step
    reweights combos by w, M-step sets each label prior to the mean marginal of Q (with the same
    Dirichlet pseudo-count a toward the source marginal and floor)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6
    ps_k = np.where(present, prior_src, 0.0)
    ps_k = ps_k / ps_k.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ps = np.clip(ps_k @ CM, 1e-6, 1 - 1e-6)            # source label marginals
    pi = ps.copy()
    n = len(P)

    def weights(pi):
        lw = np.log(pi / ps) @ CM.T + np.log((1 - pi) / (1 - ps)) @ (1 - CM).T
        return np.exp(lw - lw.max())

    for _ in range(max_iter):
        Q = P * weights(pi)
        Q /= Q.sum(1, keepdims=True)
        m = Q @ CM                                          # (n, 5) label marginals
        new = (m.sum(0) + a * ps) / (n + a)
        new = np.clip(np.maximum(new, floor * ps), 1e-6, 1 - 1e-6)
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * weights(pi)
    Q /= Q.sum(1, keepdims=True)
    pi_k = ps_k * weights(pi)
    return Q, pi_k / pi_k.sum()


# ----------------------------------------------------------------------------- decoding
_SUBSETS = np.array([[(s >> j) & 1 for j in range(5)] for s in range(1, 32)])


def _f1_matrix(CM):
    inter = _SUBSETS @ CM.T                                   # (31, K)
    return 2 * inter / (_SUBSETS.sum(1)[:, None] + CM.sum(1)[None, :])


def decode(P: np.ndarray, CM: np.ndarray, method: str = "map") -> np.ndarray:
    """Map combo posteriors to a binary label matrix (n, 5); never empty.
    map   : the most probable allowed combination
    expf1 : Bayes-optimal subset for expected sample-F1 over the 31 non-empty subsets"""
    if method == "map":
        return CM[P.argmax(1)]
    if method == "expf1":
        F = _f1_matrix(CM)
        return _SUBSETS[(P @ F.T).argmax(1)]
    raise ValueError(method)


def apply_head(P_eval, prior_src, CM, head: dict, P_target=None):
    """SLD (optional) + decoding.  P_target: posteriors of the SLD target rows (defaults to the
    eval rows, i.e. transductive).  Returns (Ypred, Q_eval, pi_hat)."""
    head = head or {}
    pi_hat = None
    Q = P_eval
    if head.get("prior", "none") == "sld":
        a = head.get("sld_a", 0.0)
        floor = head.get("sld_floor", 0.25)
        cap = head.get("sld_cap")
        if head.get("sld_mode", "combo") == "factorized":
            em = lambda P_: sld_em_factorized(P_, prior_src, CM, a=a, floor=floor)
        else:
            em = lambda P_: sld_em(P_, prior_src, a=a, floor=floor, cap=cap)
        if P_target is None or P_target is P_eval:
            Q, pi_hat = em(P_eval)
        else:
            _, pi_hat = em(P_target)
            present = prior_src > 1e-6
            Q = renorm(np.where(present, P_eval * (pi_hat / np.where(present, prior_src, 1.0)), 0.0))
    return decode(Q, CM, head.get("decode", "map")), Q, pi_hat

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/head.py


## part1/experiments/cand2_followup_v1/source/p1lib/labelers.py

In [43]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/labelers.py
"""Labelers: fill the missing labels of the unlabeled pool with soft combo distributions.

Every labeler type exposes
    soft_labels(lcfg, ctx, rows) -> Q (len(rows), K)      for rows NOT in ctx.train
and, where cheap and exact, a leave-one-out version for training rows (used when the labeler's
output is a synthetic feature or a direct predictor that needs OOF scores):
    soft_labels_train_oof(lcfg, ctx) -> Q (len(ctx.train), K)

`pseudo_labels(lcfg, ctx, clf_cfg)` turns soft labels into weighted pseudo-rows
(rows, Q, w) for the pipeline's classifier (soft-label replication, weight lam * q_ik).

Types
  cluster   : cluster-then-label (kmeans | gmm | seeded_kmeans | hdbscan), m-smoothed map   [T4]
  spread    : label spreading on the kNN graph (Zhou et al. 2004)                            [T2]
  cs        : Correct & Smooth on the kNN graph (Huang et al. 2021)                          [T2]
  sld       : calibrated base classifier + SLD-EM on the pool (Saerens et al. 2002)          [T1]
  selftrain : guarded self-training (agreement, alignment, soft weights, ramp)               [T3]
"""
from __future__ import annotations

import numpy as np
from sklearn.cluster import HDBSCAN, KMeans, kmeans_plusplus
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold

from .cache import memo, memo_ctx, rows_key
from .classifiers import make_classifier
from .features import featurize
from .graph import correct_and_smooth, label_spread
from .head import apply_temperature, fit_temperature, sld_em, sld_em_factorized

DEFAULT_CLUSTER_SPACE = {"scale": True, "pca": 32, "l2": True}


# ============================================================================ clustering
def _cluster_fit(D, algo, k, Z, rows, cfg, seed=0):
    info = {}
    if algo == "kmeans":
        km = KMeans(n_clusters=k, n_init=int(cfg.get("n_init", 10)), max_iter=300, tol=1e-4,
                    random_state=seed).fit(Z[rows])
        dist = km.transform(Z)
        info.update(inertia=float(km.inertia_), n_iter=int(km.n_iter_))
        return {"dist": dist, "labels": km.predict(Z), "info": info}
    if algo == "gmm":
        g = GaussianMixture(n_components=k, covariance_type=cfg.get("cov", "diag"), reg_covar=1e-3,
                            n_init=int(cfg.get("n_init", 3)), max_iter=500, tol=1e-4,
                            init_params="k-means++", random_state=seed).fit(Z[rows])
        info.update(converged=bool(g.converged_), n_iter=int(g.n_iter_), bic=float(g.bic(Z[rows])),
                    lower_bound=float(g.lower_bound_))
        return {"resp": g.predict_proba(Z), "labels": g.predict(Z), "info": info}
    if algo == "hdbscan":
        h = HDBSCAN(min_cluster_size=int(cfg.get("min_cluster_size", 10)),
                    min_samples=cfg.get("min_samples")).fit(Z[rows])
        lab = np.full(D.n, -1)
        lab[rows] = h.labels_
        others = np.setdiff1d(np.arange(D.n), rows)
        if len(others):                     # inductive: attach to the nearest fitted point
            Zr = Z[rows]
            nn = np.argmax(Z[others] @ Zr.T / (np.linalg.norm(Z[others], axis=1)[:, None] *
                                                np.linalg.norm(Zr, axis=1)[None] + 1e-12), axis=1)
            lab[others] = h.labels_[nn]
        info.update(n_clusters=int(h.labels_.max() + 1), noise_frac=float((h.labels_ < 0).mean()))
        return {"labels": lab, "info": info}
    raise ValueError(algo)


def cluster_resp(lcfg, ctx):
    """(n_all, C) cluster membership (hard one-hot, kmeans-soft, or GMM responsibilities)."""
    algo = lcfg.get("algo", "kmeans")
    k = int(lcfg.get("k", 24))
    space = lcfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    if algo == "seeded_kmeans":                 # label-dependent init -> not shared across folds
        y = ctx.y_train
        cls = np.unique(y)
        seeds = np.array([Z[ctx.train][y == c].mean(0) for c in cls])
        extra = max(k - len(cls), 0)
        if extra:
            more, _ = kmeans_plusplus(Z[ctx.unsup], n_clusters=extra, random_state=ctx.seed)
            seeds = np.vstack([seeds, more])
        km = KMeans(n_clusters=len(seeds), init=seeds, n_init=1, max_iter=300).fit(Z[ctx.unsup])
        lab = km.predict(Z)
        return np.eye(len(seeds))[lab]
    cfg_key = {"algo": algo, "k": k, "space": space, **{kk: lcfg[kk] for kk in ("cov", "n_init", "min_cluster_size", "min_samples") if kk in lcfg}}
    fit = memo("cluster", cfg_key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, lcfg))
    if algo == "gmm":
        return fit["resp"]
    if algo == "kmeans" and lcfg.get("soft_temp"):
        d2 = fit["dist"] ** 2
        T = float(lcfg["soft_temp"]) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    lab = fit["labels"]
    C = int(max(lab.max() + 1, 1))
    R = np.zeros((ctx.D.n, C))
    R[lab >= 0, lab[lab >= 0]] = 1.0          # hdbscan noise rows keep all-zero membership
    return R


def _cluster_map(R_train, y, K, m, prior):
    N = R_train.T @ np.eye(K)[y]                    # (C, K) label mass per cluster
    return N, (N + m * prior) / (N.sum(1, keepdims=True) + m)


def cluster_soft(lcfg, ctx, rows):
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)
    y = ctx.y_train
    prior = np.bincount(y, minlength=K) / len(y)
    _, pi = _cluster_map(R[ctx.train], y, K, float(lcfg.get("m", 4)), prior)
    Q = R[rows] @ pi
    mass = R[rows].sum(1, keepdims=True)
    Q = Q + (1 - np.clip(mass, 0, 1)) * prior        # noise / empty -> global prior
    return Q / Q.sum(1, keepdims=True)


def cluster_soft_train_oof(lcfg, ctx):
    """Exact leave-one-out cluster map for training rows."""
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)[ctx.train]
    y = ctx.y_train
    m = float(lcfg.get("m", 4))
    prior = np.bincount(y, minlength=K) / len(y)
    N, _ = _cluster_map(R, y, K, m, prior)
    nc = N.sum(1)                                    # (C,)
    out = np.zeros((len(y), K))
    for i in range(len(y)):
        Ni = N - np.outer(R[i], np.eye(K)[y[i]])
        pi = (Ni + m * prior) / (nc - R[i] + m)[:, None]
        q = R[i] @ pi
        q = q + (1 - min(R[i].sum(), 1)) * prior
        out[i] = q / q.sum()
    return out


# ============================================================================ graph labelers
def spread_soft(lcfg, ctx, rows):
    K = ctx.D.K
    F = label_spread(ctx, lcfg.get("graph"), float(lcfg.get("alpha", 0.5)), ctx.train, ctx.y_train, K, rows)
    prior = np.bincount(ctx.y_train, minlength=K) / len(ctx.train)
    F = np.clip(F, 0, None)
    s = F.sum(1, keepdims=True)
    F = np.where(s > 1e-12, F / np.maximum(s, 1e-12), prior)   # unreachable rows -> prior
    beta = float(lcfg.get("beta", 0.01))
    F = (F + beta * prior) / (1 + beta)
    return F


def _base_oof_and_all(base_cfg, ctx, X, n_inner=5):
    """Base classifier probabilities: OOF for training rows, full-fit for every other row."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    P_oof = np.zeros((len(y), K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
        P_oof[te] = m.predict_proba(X[ctx.train[te]])
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P_all = m.predict_proba(X)
    return P_oof, P_all


def cs_soft(lcfg, ctx, rows):
    K = ctx.D.K
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    P_oof, P_all = _base_oof_and_all(base_cfg, ctx, X)
    return correct_and_smooth(ctx, lcfg.get("graph"), float(lcfg.get("a1", 0.5)), float(lcfg.get("a2", 0.5)),
                              ctx.train, ctx.y_train, P_oof, P_all, K, rows)


# ============================================================================ SLD labeler
def calibrated_base(base_cfg, ctx, X, rows, temperature=True):
    """Fit base classifier on ctx.train, temperature-scale with inner OOF, predict rows."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P = m.predict_proba(X[rows])
    T = 1.0
    if temperature:
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            mm = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
            P_oof[te] = mm.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    return P, m.prior_, T


def sld_soft(lcfg, ctx, rows):
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    P, prior, _ = calibrated_base(base_cfg, ctx, X, rows, lcfg.get("temperature", False))
    if lcfg.get("mode", "combo") == "factorized":
        Q, pi = sld_em_factorized(P, prior, ctx.D.CM, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    else:
        Q, pi = sld_em(P, prior, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    return Q


# ============================================================================ self-training
def _sinkhorn(P, target, iters=200):
    Q = P.copy()
    n = len(Q)
    for _ in range(iters):
        col = Q.sum(0)
        Q = Q * (target * n / np.maximum(col, 1e-12))
        Q = Q / Q.sum(1, keepdims=True)
    return Q


def _softmatch_weight(conf):
    mu, sd = conf.mean(), conf.std() + 1e-6
    return np.where(conf >= mu, 1.0, np.exp(-((conf - mu) ** 2) / (2 * sd ** 2)))


def selftrain_pseudo(lcfg, ctx, clf_cfg, X, pool_rows):
    """Guarded self-training.  Returns (rows, Q, w, info) for the final pseudo-labeled set."""
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    Ytr = np.eye(K)[y]
    R = int(lcfg.get("rounds", 2))
    lam_max = float(lcfg.get("lam", 0.3))
    members = lcfg.get("members", ["lr"])
    align = lcfg.get("align", "none")
    tau = lcfg.get("tau", 0.75)
    top_p = lcfg.get("top_p")
    hard = bool(lcfg.get("hard", False))
    weighting = lcfg.get("weighting", "hard")
    # members that do not use pseudo-labels are computed once
    static = []
    if "lda" in members:
        static.append(make_classifier({"type": "lda"}, K, CM).fit(X[ctx.train], y).predict_proba(X[pool_rows]))
    if "cs" in members:
        static.append(cs_soft({"a1": 0.5, "a2": 0.5}, ctx, pool_rows))
    cl_major = None
    if lcfg.get("cluster_filter"):
        cl_major = cluster_soft({"algo": "kmeans", "k": int(lcfg.get("cluster_k", 24)), "m": 1}, ctx, pool_rows).argmax(1)
    sel_rows, Q, w = pool_rows[:0], np.zeros((0, K)), np.zeros(0)
    prev_lab = None
    info = {}
    for r in range(1, R + 1):
        lam_r = lam_max * (r / R if lcfg.get("ramp", False) else 1.0)
        Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[sel_rows], Q, w)
        main = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
        Pm_list = [main.predict_proba(X[pool_rows])] + static
        Pm = np.mean(Pm_list, axis=0)
        prior_src = np.bincount(y, minlength=K) / len(y)
        if align == "sld":
            Pm, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
        elif align == "sinkhorn":
            _, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
            Pm = _sinkhorn(Pm, pi)
        conf = Pm.max(1)
        lab = Pm.argmax(1)
        if top_p:
            keep = np.zeros(len(lab), bool)
            for c in range(K):
                mc = np.where(lab == c)[0]
                if len(mc):
                    thr = np.quantile(conf[mc], 1 - float(top_p))
                    keep[mc[conf[mc] >= thr]] = True
        else:
            keep = conf >= float(tau)
        if len(Pm_list) > 1:
            agree = np.sum([P.argmax(1) == lab for P in Pm_list], axis=0)
            keep &= agree >= 2
        if cl_major is not None:
            keep &= cl_major == lab
        g = _softmatch_weight(conf) if weighting == "softmatch" else np.ones(len(conf))
        sel_rows = pool_rows[keep]
        Q = np.eye(K)[lab[keep]] if hard else Pm[keep]
        w = lam_r * g[keep]
        info = {"round": r, "n_selected": int(keep.sum()), "lam": lam_r}
        if prev_lab is not None and np.mean(prev_lab != lab) < 0.01:
            break
        prev_lab = lab
    return sel_rows, Q, w, info


# ============================================================================ dispatch
SOFT = {"cluster": cluster_soft, "spread": spread_soft, "cs": cs_soft, "sld": sld_soft}


_POST = ("lam", "tau", "hard", "pool")      # applied after soft labels -> not part of the memo key


def soft_labels(lcfg, ctx, rows):
    rows = np.asarray(rows)
    core = {k: v for k, v in lcfg.items() if k not in _POST}
    return memo_ctx("soft", core, ctx, lambda: SOFT[lcfg["type"]](lcfg, ctx, rows), extra=rows.tolist())


def soft_labels_train_oof(lcfg, ctx, n_inner=5):
    """OOF soft labels for the training rows (exact LOO for cluster; inner K-fold otherwise)."""
    if lcfg["type"] == "cluster":
        core = {k: v for k, v in lcfg.items() if k not in _POST}
        return memo_ctx("soft_oof", core, ctx, lambda: cluster_soft_train_oof(lcfg, ctx))
    from .pipeline import inner_folds
    y = ctx.y_train
    out = np.zeros((len(y), ctx.D.K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        sub = ctx.sub(ctx.train[tr], ctx.train[te])
        out[te] = soft_labels(lcfg, sub, ctx.train[te])
    return out


def pool_rows_for(lcfg, ctx):
    if lcfg.get("pool", "unl") == "all" and ctx.transductive:
        rows = np.union1d(np.union1d(ctx.pool, ctx.eval), ctx.D.idx["test"])
        return np.setdiff1d(rows, ctx.train)
    return np.setdiff1d(ctx.pool, ctx.train)


def pseudo_labels(lcfg, ctx, clf_cfg, X):
    """Weighted pseudo-rows (rows, Q, w, info) for the pipeline classifier."""
    rows = pool_rows_for(lcfg, ctx)
    if lcfg["type"] == "selftrain":
        return selftrain_pseudo(lcfg, ctx, clf_cfg, X, rows)
    Q = soft_labels(lcfg, ctx, rows)
    conf = Q.max(1)
    keep = conf >= float(lcfg.get("tau", 0.0))
    if lcfg.get("hard", False):
        Q = np.eye(ctx.D.K)[Q.argmax(1)]
    lam = float(lcfg.get("lam", 0.1))
    return rows[keep], Q[keep], np.full(keep.sum(), lam), {"n_selected": int(keep.sum())}


def expand(X_l, Y_l, w_l, X_u, Q_u, w_u, min_q=0.05):
    """Soft-label replication: one copy of each row per combo with weight w * q (exact
    expected cross-entropy for LR).  Y_l may be one-hot or soft."""
    Xs, ys, ws = [], [], []
    for Xb, Yb, wb in ((X_l, Y_l, w_l), (X_u, Q_u, w_u)):
        if len(Xb) == 0:
            continue
        i, k = np.where(Yb >= min_q)
        Xs.append(Xb[i])
        ys.append(k)
        ws.append(wb[i] * Yb[i, k])
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ws)

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/labelers.py


## part1/experiments/cand2_followup_v1/source/p1lib/pipeline.py

In [44]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/pipeline.py
"""Pipeline composition from config dicts.

A *single* pipeline config:
    {
      "features":   {"scale": True, "pca": 64, "l2": False, "blocks": {...}} | None,
      "synth":      [ {"type": "gmm", "k": 16}, {"type": "knn_vote", "k": 10}, ... ],
      "labeler":    {"type": "cluster"|"spread"|"cs"|"sld"|"selftrain", "lam": 0.1, ...} | None,
      "clf":        {"type": "lr"|"ovr"|"lda"|"knn"|"hgb"|"nb"|"direct", ...},
      "temperature": True,
    }
An *ensemble*:  {"ensemble": [single, single, ...], "weights": None | [w1, w2, ...]}
A *stack* is a single pipeline whose synth list holds {"type": "oof_model", "model": single}
entries (nested OOF meta-features) and whose features are None (or a small base block).

A full experiment config adds the head and the transductive toggle:
    {"model": <single|ensemble>, "head": {"prior": "none"|"sld", "sld_a": 100,
     "decode": "map"|"expf1"}, "transductive": True}

run_model(model_cfg, ctx, rows) -> {"P": calibrated combo probs for rows, "prior": pi_s, ...}
Only labels of ctx.train are ever read.
"""
from __future__ import annotations

import copy
import warnings

import numpy as np
from sklearn.model_selection import StratifiedKFold

from .classifiers import make_classifier
from .features import featurize
from .head import apply_temperature, fit_temperature


def _mask_absent(P, prior):
    """Combos with zero training mass keep probability exactly 0 (temperature/log clipping
    would otherwise resurrect them and SLD would then amplify them)."""
    present = np.asarray(prior) > 1e-6
    if present.all():
        return P
    P = np.where(present[None, :], P, 0.0)
    return P / np.maximum(P.sum(1, keepdims=True), 1e-300)
from .labelers import expand, pseudo_labels, soft_labels, soft_labels_train_oof
from .synth import build_synth_block

warnings.filterwarnings("ignore", message="The least populated class")


def inner_folds(y, n_splits=5, seed=0):
    n_splits = int(min(n_splits, max(2, np.bincount(y).max())))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(y)), y))


def _design(cfg, ctx):
    """Base features + synthetic features for every global row."""
    parts = []
    if cfg.get("features") is not None:
        parts.append(featurize(ctx, cfg["features"]))
    S = build_synth_block(cfg.get("synth"), ctx)
    if S is not None:
        parts.append(S)
    if not parts:
        raise ValueError("pipeline has neither base features nor synthetic features")
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def run_single(cfg, ctx, rows):
    rows = np.asarray(rows)
    D = ctx.D
    K, CM = D.K, D.CM
    y = ctx.y_train
    clf_cfg = cfg["clf"]
    info = {}

    # ---- direct labeler-as-predictor (cluster / spread / cs / sld) ----
    if clf_cfg["type"] == "direct":
        lcfg = clf_cfg["labeler"]
        P = soft_labels(lcfg, ctx, rows)
        prior = np.bincount(y, minlength=K) / len(y)
        T = 1.0
        if cfg.get("temperature", True):
            P_oof = soft_labels_train_oof(lcfg, ctx)
            T = fit_temperature(P_oof, y)
            P = apply_temperature(P, T)
        return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}

    X = _design(cfg, ctx)
    Ytr = np.eye(K)[y]
    # ---- labeler -> weighted pseudo rows from the unlabeled pool ----
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    if cfg.get("labeler"):
        pr, Q, w, linfo = pseudo_labels(cfg["labeler"], ctx, clf_cfg, X)
        info["labeler"] = linfo
    Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[pr], Q, w)
    clf = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
    P = clf.predict_proba(X[rows])
    prior = clf.prior_
    T = 1.0
    if cfg.get("temperature", True):
        # classifier-stage OOF: pseudo rows fixed, inner folds over the labeled rows
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            Xi, yi, wi = expand(X[ctx.train[tr]], Ytr[tr], np.ones(len(tr)), X[pr], Q, w)
            m = make_classifier(clf_cfg, K, CM).fit(Xi, yi, wi)
            P_oof[te] = m.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    info["n_pseudo_rows"] = int(len(pr))
    return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}


def run_model(model_cfg, ctx, rows):
    if "ensemble" in model_cfg:
        members = model_cfg["ensemble"]
        wts = np.asarray(model_cfg.get("weights") or [1.0] * len(members), float)
        wts = wts / wts.sum()
        outs = [run_single(m, ctx, rows) for m in members]
        P = sum(wi * o["P"] for wi, o in zip(wts, outs))
        prior = sum(wi * o["prior"] for wi, o in zip(wts, outs))
        return {"P": P, "prior": prior, "T": [o["T"] for o in outs], "info": {"members": [o["info"] for o in outs]},
                "member_P": [o["P"] for o in outs], "member_prior": [o["prior"] for o in outs]}
    return run_single(model_cfg, ctx, rows)


# ----------------------------------------------------------------------------- validation
def final_stage_singles(model_cfg):
    return model_cfg["ensemble"] if "ensemble" in model_cfg else [model_cfg]


def validate(cfg):
    """Enforce the 'only ONE prior correction' rule and basic sanity."""
    model = cfg["model"]
    head = cfg.get("head", {})
    corrections = set()
    for s in final_stage_singles(model):
        clf = s["clf"]
        if clf.get("class_weight") == "balanced":
            corrections.add("class_weight=balanced")
        lab = s.get("labeler") or {}
        if lab.get("type") == "selftrain" and lab.get("align", "none") != "none" and clf.get("class_weight") == "balanced":
            raise ValueError("self-training alignment + balanced class weights = double prior correction")
    if head.get("prior", "none") == "sld":
        corrections.add("sld")
    if len(corrections) > 1:
        raise ValueError(f"more than one prior correction: {sorted(corrections)}")
    return sorted(corrections)


def model_part(cfg):
    """Config without the post-hoc head (SLD / decoding) -> cache key for fold outputs."""
    return {"model": cfg["model"], "transductive": cfg.get("transductive", True)}


def deep(cfg):
    return copy.deepcopy(cfg)

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/pipeline.py


## part1/experiments/cand2_followup_v1/source/p1lib/search.py

In [45]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/search.py
"""Optuna TPE search over the composed config space, robust objective, 1-SE selection and the
nested search-overfitting check.

Objective  J = 0.5 * A + 0.5 * B   (A = repeated stratified CV sample-F1, B = mean chronological
holdout sample-F1 over cuts 2004/2005/2006).  C (official val) is logged, never optimized.
Selection  = one-standard-error rule on J, then the SIMPLEST config (fewest active components),
then a guard: B must not be worse than the plain-LR baseline's B by more than one SE.
"""
from __future__ import annotations

import copy
import fcntl
import json
import os
import time
import traceback

import numpy as np

from .evaluate import evaluate, make_folds, fold_outputs, score, CHRONO_CUTS
from .pipeline import validate

GRAPH_SPACE = {"scale": True, "pca": 64, "l2": True}


# ============================================================================ search space
def suggest_config(t, allow_blocks=True, allow_stack=False, require_unsup=False):
    """Define-by-run space.  Returns a full experiment config."""
    pca = t.suggest_categorical("pca", [0, 32, 64, 96])
    l2 = t.suggest_categorical("l2", [False, True])
    feats = {"scale": True, "pca": pca or None, "l2": l2, "blocks": {}}
    if allow_blocks:
        if t.suggest_categorical("blk_criteria", [False, True]):
            feats["blocks"]["criteria"] = {"d": 16}
        if t.suggest_categorical("blk_drug", [False, True]):
            feats["blocks"]["pathways"] = {"d": 16, "max_level": 2}
            feats["blocks"]["drugmeta"] = {}

    prior = t.suggest_categorical("prior", ["none", "sld"])
    head = {"prior": prior, "decode": t.suggest_categorical("decode", ["map", "expf1"])}
    if prior == "sld":
        head["sld_a"] = t.suggest_categorical("sld_a", [0, 25, 100, 400])
        head["sld_floor"] = 0.25

    clf_t = t.suggest_categorical("clf", ["lr", "ovr", "lda", "knn"] if require_unsup else ["lr", "ovr", "lda", "knn", "cs"])
    temperature = t.suggest_categorical("temperature", [False, True])
    if clf_t == "cs":       # Correct & Smooth as the predictor (T2); base = powerset LR
        gk = t.suggest_categorical("cs_graph_k", [5, 10, 20])
        lcfg = {"type": "cs", "graph": {"space": GRAPH_SPACE, "k": gk, "s": 1.0, "sym": "max"},
                "a1": t.suggest_categorical("cs_a1", [0.0, 0.5, 0.8]),
                "a2": t.suggest_float("cs_a2", 0.1, 0.9),
                "base": {"type": "lr", "C": t.suggest_float("C", 0.01, 3.0, log=True)},
                "base_features": {"scale": True, "pca": pca or None}}
        model = {"features": None, "synth": [], "labeler": None,
                 "clf": {"type": "direct", "labeler": lcfg}, "temperature": temperature}
        return {"model": model, "head": head, "transductive": True}

    clf = {"type": clf_t}
    if clf_t in ("lr", "ovr"):
        clf["C"] = t.suggest_float("C", 0.01, 3.0, log=True)
    if clf_t == "knn":
        clf.update(k=t.suggest_categorical("knn_k", [3, 5, 10, 20]), weights="distance",
                   tau=t.suggest_categorical("knn_tau", [0.05, 0.1, 0.3]), beta=1.0)
    if prior != "sld" and clf_t in ("lr", "ovr", "lda"):
        if t.suggest_categorical("class_weight", ["none", "balanced"]) == "balanced":
            clf["class_weight"] = "balanced"

    synth = []
    if t.suggest_categorical("syn_gmm", [False, True]):
        synth.append({"type": "gmm", "k": t.suggest_categorical("syn_gmm_k", [8, 16, 32]), "cov": "diag",
                      "space": {"scale": True, "pca": 32, "l2": True}})
    if t.suggest_categorical("syn_kmeans", [False, True]):
        synth.append({"type": "kmeans", "k": t.suggest_categorical("syn_kmeans_k", [8, 16, 32]),
                      "out": t.suggest_categorical("syn_kmeans_out", ["dist", "soft"])})
    if t.suggest_categorical("syn_knn", [False, True]):
        synth.append({"type": "knn_vote", "k": t.suggest_categorical("syn_knn_k", [5, 10, 20])})
    if t.suggest_categorical("syn_graph", [False, True]):
        synth.append({"type": "graph", "mode": t.suggest_categorical("syn_graph_mode", ["spread", "cs"]),
                      "graph": {"space": GRAPH_SPACE, "k": 10, "s": 1.0, "sym": "max"},
                      "alpha": t.suggest_float("syn_graph_alpha", 0.2, 0.9)})
    if t.suggest_categorical("syn_clmap", [False, True]):
        synth.append({"type": "cluster_map", "algo": "kmeans", "k": t.suggest_categorical("syn_clmap_k", [8, 16, 32]), "m": 4})
    if t.suggest_categorical("syn_oof", [False, True]):
        synth.append({"type": "oof_model", "model": {"features": {"scale": True}, "clf": {"type": "lda"}, "temperature": True}})

    lab_t = t.suggest_categorical("labeler", ["cluster", "spread", "cs", "selftrain"] if require_unsup else
                                  ["none", "cluster", "spread", "cs", "sld", "selftrain"])
    labeler = None
    if lab_t != "none":
        lam = t.suggest_float("lam", 0.03, 0.5, log=True)
        if lab_t == "cluster":
            labeler = {"type": "cluster", "algo": t.suggest_categorical("cl_algo", ["kmeans", "gmm", "seeded_kmeans"]),
                       "k": t.suggest_categorical("cl_k", [8, 16, 24, 32, 48]), "m": t.suggest_categorical("cl_m", [1, 4]),
                       "lam": lam}
        elif lab_t in ("spread", "cs"):
            g = {"space": GRAPH_SPACE, "k": t.suggest_categorical("lab_graph_k", [5, 10, 20]), "s": 1.0, "sym": "max"}
            labeler = {"type": lab_t, "graph": g, "lam": lam}
            if lab_t == "spread":
                labeler["alpha"] = t.suggest_float("lab_alpha", 0.2, 0.9)
            else:
                labeler.update(a1=t.suggest_categorical("lab_a1", [0.0, 0.5, 0.8]), a2=t.suggest_float("lab_a2", 0.2, 0.9))
        elif lab_t == "sld":
            labeler = {"type": "sld", "base": {"type": "lr", "C": 0.1}, "a": t.suggest_categorical("lab_sld_a", [25, 100, 400]),
                       "lam": lam}
        elif lab_t == "selftrain":
            labeler = {"type": "selftrain", "lam": lam,
                       "members": t.suggest_categorical("st_members", ["lr", "lr+lda+cs"]).split("+"),
                       "rounds": t.suggest_categorical("st_rounds", [1, 2, 3]),
                       "tau": t.suggest_categorical("st_tau", [0.6, 0.75, 0.9]),
                       "align": t.suggest_categorical("st_align", ["none", "sld"]) if clf.get("class_weight") != "balanced" else "none",
                       "weighting": t.suggest_categorical("st_weighting", ["hard", "softmatch"]),
                       "ramp": t.suggest_categorical("st_ramp", [False, True]),
                       "cluster_filter": t.suggest_categorical("st_cluster_filter", [False, True])}
            if require_unsup and "cs" not in labeler["members"]:
                labeler["cluster_filter"] = True
        if lab_t != "selftrain" and t.suggest_categorical("lab_hard", [False, True]):
            labeler["hard"] = True
            labeler["tau"] = t.suggest_categorical("lab_tau", [0.5, 0.7, 0.9])
    model = {"features": feats, "synth": synth, "labeler": labeler, "clf": clf, "temperature": temperature}
    return {"model": model, "head": head, "transductive": True}


# ============================================================================ complexity
CLF_COST = {"lr": 0.0, "lda": 0.0, "ovr": 0.25, "nb": 0.25, "knn": 0.5, "direct": 1.0, "hgb": 2.0}


def complexity(cfg) -> float:
    """Number of active components (+ small costs for extra knobs).  Lower = simpler."""
    m = cfg["model"]
    singles = m["ensemble"] if "ensemble" in m else [m]
    c = 0.0
    for s in singles:
        c += len(s.get("synth") or [])
        for sy in s.get("synth") or []:
            if sy["type"] == "oof_model":
                c += complexity({"model": sy["model"], "head": {}})
        lab = s.get("labeler")
        if lab:
            c += 1.0 + (0.5 * (len(lab.get("members", [])) > 1) + 0.25 * (lab.get("rounds", 1) - 1)
                        if lab["type"] == "selftrain" else 0.0)
        f = s.get("features") or {}
        c += len(f.get("blocks") or {})
        clf = s["clf"]
        c += CLF_COST.get(clf["type"], 1.0)
        if clf.get("class_weight") == "balanced":
            c += 0.5
        if s.get("temperature"):
            c += 0.25
    if "ensemble" in m:
        c += len(singles) - 1
    if cfg.get("head", {}).get("prior") == "sld":
        c += 0.5
    return c


def unsup_labeling(cfg) -> bool:
    """Assignment rule: an UNSUPERVISED method must take part in filling the missing labels.
    True if the label-filling path uses clustering or a similarity graph: a cluster / label-
    spreading / C&S labeler, or guarded self-training with the K-means consistency filter or a
    C&S member, or a direct cluster/graph predictor."""
    m = cfg["model"]
    for s in (m["ensemble"] if "ensemble" in m else [m]):
        lab = s.get("labeler") or {}
        if lab.get("type") in ("cluster", "spread", "cs"):
            return True
        if lab.get("type") == "selftrain" and (lab.get("cluster_filter") or "cs" in lab.get("members", [])):
            return True
        if s["clf"]["type"] == "direct" and s["clf"]["labeler"]["type"] in ("cluster", "spread", "cs"):
            return True
    return False


# ============================================================================ objective
def append_jsonl(path, rec):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    line = json.dumps(rec, default=float) + "\n"
    with open(path, "a") as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        f.write(line)
        f.flush()
        fcntl.flock(f, fcntl.LOCK_UN)


def make_objective(D, log_path, R=4, rows=None, chrono_cuts=CHRONO_CUTS, prune_margin=0.05,
                   allow_blocks=True, cv_seed=0, tag="main", require_unsup=False):
    import optuna

    def objective(trial):
        cfg = suggest_config(trial, allow_blocks=allow_blocks, require_unsup=require_unsup)
        validate(cfg)
        t0 = time.time()
        # stage 1: chrono cuts + val + first CV repeat; prune clearly bad configs
        folds1 = make_folds(D, ("A", "B", "C"), R=R, seed=cv_seed, rows=rows, chrono_cuts=chrono_cuts)
        first = [f for f in folds1 if f["protocol"] != "A" or f["repeat"] == 0]
        rest = [f for f in folds1 if f["protocol"] == "A" and f["repeat"] > 0]
        try:
            outs1 = fold_outputs(cfg, D, first, n_jobs=1)
            r1 = score(cfg, D, first, outs1, bootstrap=False)
            j1 = r1.get("J", np.nan)
            trial.set_user_attr("J_stage1", float(j1))
            try:
                best = trial.study.best_value
            except ValueError:
                best = -np.inf
            if np.isfinite(best) and j1 < best - prune_margin:
                append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "pruned", "J_stage1": j1,
                                        "params": trial.params, "config": cfg, "secs": time.time() - t0})
                raise optuna.TrialPruned()
            outs2 = fold_outputs(cfg, D, rest, n_jobs=1)
            res = score(cfg, D, first + rest, outs1 + outs2, bootstrap=False)
        except optuna.TrialPruned:
            raise
        except Exception as e:
            append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "fail", "error": repr(e),
                                    "tb": traceback.format_exc()[-2000:], "params": trial.params, "config": cfg})
            raise
        cx = complexity(cfg)
        attrs = {"A": res["A"]["sample_f1"], "A_se": res["A"]["sample_f1_se"], "A_macro": res["A"]["macro_f1"],
                 "B": res["B"]["sample_f1"], "B_se": res["B"]["sample_f1_se"], "B_macro": res["B"]["macro_f1"],
                 "J": res["J"], "J_se": res["J_se"], "complexity": cx, "secs": time.time() - t0}
        if "B2005" in res:
            attrs["B2005"] = res["B2005"]["sample_f1"]
        if "C" in res:
            attrs.update(C=res["C"]["sample_f1"], C_se=res["C"]["sample_f1_se"], C_macro=res["C"]["macro_f1"])
        for k, v in attrs.items():
            trial.set_user_attr(k, float(v))
        trial.set_user_attr("config", json.dumps(cfg))
        append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "complete", **attrs, "params": trial.params,
                                "config": cfg, "rowf1_A": res["rowf1_A"], "rowf1_B": res["rowf1_B"],
                                "rowf1_C": res.get("rowf1_C"), "B_ncuts": res.get("B_ncuts", 1)})
        return res["J"]

    return objective


def run_worker(storage_path, study_name, n_trials, log_path, seed, R=4, rows=None, chrono_cuts=CHRONO_CUTS,
               allow_blocks=True, tag="main", timeout=None, sampler_name="tpe", cv_seed=0, require_unsup=False):
    """One search worker process (shares the study through a journal file)."""
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    from .data import load
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    D = load()
    storage = JournalStorage(JournalFileBackend(storage_path))
    if sampler_name == "random":
        sampler = optuna.samplers.RandomSampler(seed=seed)
    else:
        sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True, group=True, constant_liar=True,
                                             n_startup_trials=40)
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize",
                                sampler=sampler, load_if_exists=True)
    obj = make_objective(D, log_path, R=R, rows=rows, chrono_cuts=chrono_cuts, allow_blocks=allow_blocks, tag=tag,
                         cv_seed=cv_seed, prune_margin=(np.inf if sampler_name == "random" else 0.05),
                         require_unsup=require_unsup)
    study.optimize(obj, n_trials=n_trials, timeout=timeout, catch=(Exception,))


def launch(storage_path, study_name, n_workers, trials_per_worker, log_path, R=4, rows=None,
           chrono_cuts=CHRONO_CUTS, allow_blocks=True, tag="main", timeout=None, base_seed=0, enqueue=None,
           sampler_name="tpe", cv_seed=0, require_unsup=False):
    """Run n_workers processes in parallel (spawn) against one journal-backed study."""
    import multiprocessing as mp
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    os.makedirs(os.path.dirname(storage_path), exist_ok=True)
    storage = JournalStorage(JournalFileBackend(storage_path))
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize", load_if_exists=True)
    for params in enqueue or []:
        study.enqueue_trial(params, skip_if_exists=True)
    ctx = mp.get_context("spawn")
    procs = []
    for w in range(n_workers):
        p = ctx.Process(target=run_worker, args=(storage_path, study_name, trials_per_worker, log_path,
                                                 base_seed + w, R, None if rows is None else list(map(int, rows)),
                                                 chrono_cuts, allow_blocks, tag, timeout, sampler_name, cv_seed,
                                                 require_unsup))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
    return study


# ============================================================================ selection
def load_log(log_path, tag=None, state="complete"):
    recs = []
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            if (tag is None or r.get("tag") == tag) and (state is None or r.get("state") == state):
                recs.append(r)
    return recs


def select_one_se(recs, baseline=None, key="J", se_key="J_se"):
    """1-SE rule: candidates with J >= J_best - SE(J_best); choose minimal complexity, ties by J.
    Guard: if a baseline record is given, the choice must have B >= B_base - B_se_base."""
    if not recs:
        return None, []
    best = max(recs, key=lambda r: r[key])
    thr = best[key] - best[se_key]
    cands = [r for r in recs if r[key] >= thr]
    cands.sort(key=lambda r: (r["complexity"], -r[key]))
    for r in cands:
        if baseline is None or r["B"] >= baseline["B"] - baseline["B_se"]:
            return r, cands
    return cands[0], cands


def dedupe(recs):
    """Keep one record per distinct config (TPE re-samples identical categorical configs)."""
    seen = {}
    for r in recs:
        k = json.dumps(r["config"], sort_keys=True)
        if k not in seen or r[ "J"] > seen[k]["J"]:
            seen[k] = r
    return list(seen.values())

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/search.py


## part1/experiments/cand2_followup_v1/source/p1lib/submit.py

In [46]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/submit.py
"""Final refit on all 235 labeled rows, test predictions, Kaggle CSV + manifest, filled train labels."""
from __future__ import annotations

import datetime
import hashlib
import json
import os

import numpy as np
import pandas as pd

from .context import make_ctx
from .data import LABELS, write_submission
from .head import apply_head, decode, sld_em
from .pipeline import run_model, validate


def final_fit(cfg, D, n_seeds=1):
    """Refit on labeled-train + val (235 rows); return calibrated probs for test and the pool."""
    validate(cfg)
    test, unl = D.idx["test"], D.idx["unl"]
    rows = np.r_[test, unl]
    Ps, priors, Ts = [], [], []
    for s in range(n_seeds):
        ctx = make_ctx(D, D.idx["labval"], test, transductive=cfg.get("transductive", True), seed=s, name="final")
        o = run_model(cfg["model"], ctx, rows)
        Ps.append(o["P"]); priors.append(np.asarray(o["prior"])); Ts.append(o["T"])
    P = np.mean(Ps, axis=0)
    prior = np.mean(priors, axis=0)
    return {"P_test": P[: len(test)], "P_pool": P[len(test):], "prior": prior, "T": Ts}


def predict_test(cfg, D, fit):
    head = cfg.get("head", {})
    tr = cfg.get("transductive", True)
    target = None if tr else fit["P_pool"]          # inductive: SLD prior estimated on the unlabeled pool
    Y, Q, pi = apply_head(fit["P_test"], fit["prior"], D.CM, head, target)
    return Y, Q, pi


def fill_pool_labels(cfg, D, fit):
    """Hard labels for the 1,014 masked train rows from the refit model (SLD toward the pool's own
    prior when the head uses SLD)."""
    head = cfg.get("head", {})
    P = fit["P_pool"]
    if head.get("prior") == "sld":
        P, _ = sld_em(P, fit["prior"], a=head.get("sld_a", 0), floor=head.get("sld_floor", 0.25))
    return decode(P, D.CM, head.get("decode", "map")), P


def write_candidate(cfg, D, out_dir, name, scores=None, n_seeds=1, sample_path=None):
    os.makedirs(out_dir, exist_ok=True)
    fit = final_fit(cfg, D, n_seeds=n_seeds)
    Y, Q, pi = predict_test(cfg, D, fit)
    path = os.path.join(out_dir, f"{name}.csv")
    sub = write_submission(D, Y, path)
    if sample_path:                                  # exact format / order check vs the sample file
        sample = pd.read_csv(sample_path)
        assert list(sub.columns) == list(sample.columns) and (sub["ID"].values == sample["ID"].values).all()
    sha = hashlib.sha1(open(path, "rb").read()).hexdigest()
    manifest = {
        "name": name, "csv": os.path.abspath(path), "sha1": sha,
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "config": cfg, "transductive_test_features_used": bool(cfg.get("transductive", True)),
        "trained_on": "157 labeled train rows + 78 val rows (235); 1,014 unlabeled train rows as pool",
        "n_seeds": n_seeds,
        "scores": scores or {},
        "test_pred_label_rate": dict(zip(LABELS, Y.mean(0).round(3).tolist())),
        "test_pred_cardinality": float(Y.sum(1).mean()),
        "test_pred_combo_counts": {"+".join(LABELS[j] for j in range(5) if r[j]): int(c)
                                   for r, c in zip(*np.unique(Y, axis=0, return_counts=True))},
        "sld_test_prior_labels": None if pi is None else dict(zip(LABELS, (pi @ D.CM).round(3).tolist())),
        "train_prior_labels": dict(zip(LABELS, (fit["prior"] @ D.CM).round(3).tolist())),
        "temperature": fit["T"],
    }
    with open(os.path.join(out_dir, f"{name}.manifest.json"), "w") as f:
        json.dump(manifest, f, indent=1, default=float)
    return manifest, fit, Y


def write_filled_train(cfg, D, fit, path):
    Yp, P = fill_pool_labels(cfg, D, fit)
    tr = D.df.iloc[D.idx["train"]].copy()
    tr = tr.drop(columns=["split"])
    pos = {g: i for i, g in enumerate(D.idx["unl"])}
    lab = tr[LABELS].to_numpy().copy()
    src = np.array(["given"] * len(tr), dtype=object)
    for i, g in enumerate(D.idx["train"]):
        if g in pos:
            lab[i] = Yp[pos[g]]
            src[i] = "filled"
    tr[LABELS] = lab
    tr["label_source"] = src
    tr["fill_confidence"] = np.where(src == "filled", [P[pos[g]].max() if g in pos else 1.0 for g in D.idx["train"]], 1.0)
    tr.to_csv(path, index=False)
    return tr

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/submit.py


## part1/experiments/cand2_followup_v1/source/p1lib/synth.py

In [47]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/synth.py
"""Synthetic features: unsupervised / semi-supervised model outputs turned into new columns
that downstream classifiers (or a stacker) consume.

    synth_features(scfg, ctx) -> (n_all, m) array for every global row

Label-free generators (fit on ctx.unsup features only, cached):
  gmm        : GMM posterior responsibilities (or log-responsibilities)
  kmeans     : distances to K-means centroids, or soft assignments softmax(-d^2/T)
Label-dependent generators (training rows get OUT-OF-FOLD values, so no row sees its own label):
  knn_vote   : cosine kNN label-vote fractions over the K combos (exact leave-one-out)
  graph      : label-spreading or C&S scores (inner K-fold for training rows)
  cluster_map: cluster -> label-combo mapped probabilities (exact leave-one-out)
  oof_model  : out-of-fold probabilities of any single pipeline config (inner K-fold = nested OOF)
Label-dependent outputs are returned as log-probabilities (K columns).
"""
from __future__ import annotations

import numpy as np

from .cache import memo, memo_ctx
from .classifiers import KNNClf
from .features import featurize
from .head import safe_log
from .labelers import DEFAULT_CLUSTER_SPACE, _cluster_fit, soft_labels, soft_labels_train_oof

LABEL_FREE = {"gmm", "kmeans"}


def _label_free(scfg, ctx):
    algo = scfg["type"]
    k = int(scfg.get("k", 16))
    space = scfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    key = {"algo": algo, "k": k, "space": space, **{kk: scfg[kk] for kk in ("cov", "n_init") if kk in scfg}}
    fit = memo("cluster", key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, scfg))
    if algo == "gmm":
        R = fit["resp"]
        return np.log(np.clip(R, 1e-6, None)) if scfg.get("out", "resp") == "logresp" else R
    d = fit["dist"]
    if scfg.get("out", "dist") == "soft":
        d2 = d ** 2
        T = float(scfg.get("temp", 1.0)) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    return d


def _knn_vote(scfg, ctx):
    K = ctx.D.K
    Z = featurize(ctx, scfg.get("space", {"scale": True, "l2": True}))
    y = ctx.y_train
    m = KNNClf({"type": "knn", "k": int(scfg.get("k", 10)), "weights": scfg.get("weights", "uniform"),
                "beta": float(scfg.get("beta", 1.0))}, K, ctx.D.CM).fit(Z[ctx.train], y)
    out = m.predict_proba(Z)
    out[ctx.train] = m.predict_proba(Z[ctx.train], exclude_self=True)
    return safe_log(out)


def _labeler_feature(lcfg, ctx):
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out = np.zeros((ctx.D.n, ctx.D.K))
    out[rows] = soft_labels(lcfg, ctx, rows)
    out[ctx.train] = soft_labels_train_oof(lcfg, ctx)
    return safe_log(out)


def _oof_model(scfg, ctx):
    from .pipeline import inner_folds, run_single
    sub = scfg["model"]
    y = ctx.y_train
    out = np.zeros((ctx.D.n, ctx.D.K))
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out[rows] = run_single(sub, ctx, rows)["P"]
    for tr, te in inner_folds(y, int(scfg.get("n_inner", 5)), ctx.seed + 17):
        inner = ctx.sub(ctx.train[tr], ctx.train[te])
        out[ctx.train[te]] = run_single(sub, inner, ctx.train[te])["P"]
    return safe_log(out)


def synth_features(scfg, ctx) -> np.ndarray:
    if scfg["type"] in LABEL_FREE:
        return _synth(scfg, ctx)
    return memo_ctx("synth", scfg, ctx, lambda: _synth(scfg, ctx))


def _synth(scfg, ctx) -> np.ndarray:
    t = scfg["type"]
    if t in LABEL_FREE:
        return _label_free(scfg, ctx)
    if t == "knn_vote":
        return _knn_vote(scfg, ctx)
    if t == "graph":
        mode = scfg.get("mode", "spread")
        lcfg = {"type": "spread" if mode == "spread" else "cs", "graph": scfg.get("graph"),
                "alpha": scfg.get("alpha", 0.5), "a1": scfg.get("a1", 0.5), "a2": scfg.get("alpha", 0.5)}
        return _labeler_feature(lcfg, ctx)
    if t == "cluster_map":
        lcfg = {"type": "cluster", **{k: v for k, v in scfg.items() if k != "type"}}
        return _labeler_feature(lcfg, ctx)
    if t == "oof_model":
        return _oof_model(scfg, ctx)
    raise ValueError(t)


def build_synth_block(synth_list, ctx):
    """Concatenate all synthetic features and standardize them on train+pool rows."""
    if not synth_list:
        return None
    parts = [synth_features(s, ctx) for s in synth_list]
    S = np.hstack(parts)
    fit_rows = np.union1d(ctx.train, ctx.pool)
    mu = S[fit_rows].mean(0)
    sd = S[fit_rows].std(0)
    sd[sd < 1e-9] = 1.0
    return (S - mu) / sd

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/synth.py


## part1/experiments/cand2_followup_v1/source/p1lib/viz.py

In [48]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/source/p1lib/viz.py
"""Shared matplotlib style for report figures (static PNG/PDF).

Categorical hues are assigned in a FIXED order (validated reference palette, light mode);
sequential = one blue ramp; thin marks, solid hairline grid, no dual axes.
"""
from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
TEXT2 = "#52514e"
GRID = "#e4e3df"
MUTED = "#a3a29c"

# fixed identity colors for recurring entities so a method keeps its hue across figures
PROTOCOL_COLORS = {"A": SERIES[0], "B": SERIES[1], "C": SERIES[2]}
LABEL_COLORS = dict(zip(["Safety", "Funding", "Efficacy", "Recruitment", "Other"], SERIES[:5]))


def setup():
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.grid": True, "grid.color": GRID,
        "grid.linewidth": 0.6, "grid.linestyle": "-", "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2, "text.color": TEXT,
        "axes.titlesize": 11, "axes.titleweight": "semibold", "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.5, "legend.frameon": False,
        "lines.linewidth": 2.0, "lines.markersize": 6, "font.family": "DejaVu Sans",
        "axes.prop_cycle": mpl.cycler(color=SERIES), "figure.dpi": 110, "savefig.dpi": 200,
        "savefig.bbox": "tight",
    })


def save(fig, path):
    fig.savefig(path)
    if path.endswith(".png"):
        fig.savefig(path[:-4] + ".pdf")
    return path

Overwriting _source_export/part1/experiments/cand2_followup_v1/source/p1lib/viz.py


## part1/experiments/cand2_followup_v1/strict_adapter.py

In [49]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/strict_adapter.py
"""Inductive, group-safe adapters for the four fixed Dylan v2 candidates.

Original p1lib is copied unchanged under source/. This file changes fitting
boundaries and nested calibration, not classifier settings or selection weights.
"""
from dataclasses import replace
import hashlib

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

from p1lib import cache, features, graph, labelers
from p1lib.classifiers import make_classifier, KNNClf
from p1lib.context import Ctx
from p1lib.head import apply_temperature, fit_temperature, safe_log
from p1lib.pipeline import _mask_absent

AUDIT = []


def groups(D):
    if not hasattr(D, '_embedding_groups'):
        D._embedding_groups = pd.util.hash_pandas_object(
            D.df[[f'emb_{i}' for i in range(150)]].round(10), index=False).to_numpy()
    return D._embedding_groups


def context(D, train, query, pool, seed=0, name='', depth=0):
    train, query, pool = [np.asarray(v, dtype=int) for v in [train, query, pool]]
    unsup = np.union1d(train, pool)
    assert not np.intersect1d(unsup, query).size
    assert not np.intersect1d(unsup, D.idx['test']).size
    # The only accessible labels belong to the fitting gold rows.
    y = np.full_like(D.y, -1); y[train] = D.y[train]
    Y = np.full_like(D.Y, -1); Y[train] = D.Y[train]
    df = D.df.copy(); df.loc[:, ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']] = Y
    hidden = replace(D, df=df, y=y, Y=Y)
    hidden._embedding_groups = groups(D)
    return Ctx(hidden, train, query, pool, unsup, False, seed, name, depth)


def subcontext(ctx, train, query):
    g = groups(ctx.D)
    banned = g[query]
    train = np.asarray(train)[~np.isin(g[train], banned)]
    pool = ctx.pool[~np.isin(g[ctx.pool], banned)]
    sub = context(ctx.D, train, query, pool, ctx.seed, ctx.name + '/inner', ctx.depth + 1)
    assert not np.intersect1d(g[sub.unsup], g[query]).size
    assert np.isin(sub.unsup, ctx.unsup).all()
    AUDIT.append(dict(depth=sub.depth, gold=len(train), masked=len(pool), query=len(query),
                      fitting_ids_sha256=hashlib.sha256(sub.D.df.iloc[sub.unsup].ID.to_numpy(str).tobytes()).hexdigest(),
                      gold_group_overlap=0, masked_group_overlap=0))
    return sub


def inner_folds(ctx, n=5):
    _, inv = np.unique(groups(ctx.D)[ctx.train], return_inverse=True)
    target = np.zeros((inv.max() + 1, 5), dtype=int)
    np.maximum.at(target, inv, ctx.D.CM[ctx.y_train])
    splitter = MultilabelStratifiedKFold(n_splits=n, shuffle=True, random_state=ctx.seed)
    for tr, te in splitter.split(np.zeros((len(target), 1)), target):
        yield np.flatnonzero(np.isin(inv, tr)), np.flatnonzero(np.isin(inv, te))


def base_oof(ctx, base_cfg, feature_cfg):
    def compute():
        K, CM = ctx.D.K, ctx.D.CM
        X = features.featurize(ctx, feature_cfg)
        P_oof = np.zeros((len(ctx.train), K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            Xi = features.featurize(sub, feature_cfg)
            m = make_classifier(base_cfg, K, CM).fit(Xi[sub.train], sub.y_train)
            P_oof[te] = m.predict_proba(Xi[sub.eval])
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], ctx.y_train)
        return P_oof, m.predict_proba(X)
    return cache.memo_ctx('strict_base_oof', [base_cfg, feature_cfg], ctx, compute)


def cs_probabilities(lcfg, ctx, rows):
    assert lcfg['type'] == 'cs'
    core = {k: v for k, v in lcfg.items() if k not in ('lam', 'tau', 'hard', 'pool')}
    def compute():
        oof, all_p = base_oof(ctx, lcfg.get('base', {'type': 'lr', 'C': .1}),
                            lcfg.get('base_features', {'scale': True}))
        return graph.correct_and_smooth(ctx, lcfg.get('graph'), float(lcfg.get('a1', .5)),
                float(lcfg.get('a2', .5)), ctx.train, ctx.y_train, oof, all_p, ctx.D.K, rows)
    return cache.memo_ctx('strict_cs', core, ctx, compute, extra=np.asarray(rows).tolist())


def knn_features(cfg, ctx):
    def compute():
        space = cfg.get('space', {'scale': True, 'l2': True})
        knn = dict(type='knn', k=int(cfg.get('k', 10)), weights=cfg.get('weights', 'uniform'),
                   beta=float(cfg.get('beta', 1)))
        Z = features.featurize(ctx, space)
        m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Z[ctx.train], ctx.y_train)
        out = m.predict_proba(Z)
        g = groups(ctx.D)
        # Preserve the source's leave-one-out idea, but leave out the entire
        # embedding group and refit scaling with that group's masked copies purged.
        for value in np.unique(g[ctx.train]):
            held = ctx.train[g[ctx.train] == value]
            train = ctx.train[g[ctx.train] != value]
            sub = subcontext(ctx, train, held)
            # Avoid caching one large transformed matrix per leave-group-out fit.
            Zi = features._embed(sub.D, features.norm_features_cfg(space), sub.unsup)
            m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Zi[sub.train], sub.y_train)
            out[held] = m.predict_proba(Zi[held])
        return safe_log(out)
    return cache.memo_ctx('strict_knn_feature', cfg, ctx, compute)


def kmeans_features(cfg, ctx):
    assert cfg['type'] == 'kmeans' and cfg.get('out') == 'soft'
    space = cfg.get('space', labelers.DEFAULT_CLUSTER_SPACE)
    Z = features.featurize(ctx, space)
    key = dict(algo='kmeans', k=int(cfg.get('k', 16)), space=space,
               **{k: cfg[k] for k in ('cov', 'n_init') if k in cfg})
    fit = cache.memo('cluster', key, ctx.unsup,
                     lambda: labelers._cluster_fit(ctx.D, 'kmeans', key['k'], Z, ctx.unsup, cfg))
    d2 = fit['dist'] ** 2
    # Source uses all global rows for this median, including eval/test. Restrict it.
    T = float(cfg.get('temp', 1)) * np.median(d2[ctx.unsup].min(1))
    logp = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
    result = np.exp(logp)
    return result / result.sum(1, keepdims=True)


def design(cfg, ctx):
    parts = []
    if cfg.get('features') is not None:
        assert not cfg['features'].get('blocks')
        parts.append(features.featurize(ctx, cfg['features']))
    synth = []
    for scfg in cfg.get('synth', []):
        if scfg['type'] == 'kmeans':
            synth.append(kmeans_features(scfg, ctx))
        elif scfg['type'] == 'knn_vote':
            synth.append(knn_features(scfg, ctx))
        else:
            raise ValueError(scfg)
    if synth:
        S = np.hstack(synth)
        mu, sd = S[ctx.unsup].mean(0), S[ctx.unsup].std(0)
        sd[sd < 1e-9] = 1
        parts.append((S - mu) / sd)
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def uncalibrated(cfg, ctx, rows):
    K, CM, y = ctx.D.K, ctx.D.CM, ctx.y_train
    if cfg['clf']['type'] == 'direct':
        return dict(P=cs_probabilities(cfg['clf']['labeler'], ctx, rows),
                    prior=np.bincount(y, minlength=K) / len(y), info={})
    X = design(cfg, ctx)
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    lcfg = cfg.get('labeler')
    if lcfg:
        assert lcfg.get('pool', 'unl') == 'unl'
        pr = ctx.pool
        Q = cs_probabilities(lcfg, ctx, pr)
        keep = Q.max(1) >= float(lcfg.get('tau', 0))
        if lcfg.get('hard', False):
            Q = np.eye(K)[Q.argmax(1)]
        pr, Q = pr[keep], Q[keep]
        w = np.full(len(pr), float(lcfg.get('lam', .1)))
    Xt, yt, wt = labelers.expand(X[ctx.train], np.eye(K)[y], np.ones(len(y)), X[pr], Q, w)
    m = make_classifier(cfg['clf'], K, CM).fit(Xt, yt, wt)
    return dict(P=m.predict_proba(X[rows]), prior=m.prior_, info={'n_pseudo_rows': len(pr)})


def run_single(cfg, ctx, rows):
    out = uncalibrated(cfg, ctx, rows)
    T = 1.
    if cfg.get('temperature', True):
        oof = np.zeros((len(ctx.train), ctx.D.K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            # Relearn features, pseudo labels and classifier inside each inner fold.
            oof[te] = uncalibrated(cfg, sub, sub.eval)['P']
        T = fit_temperature(oof, ctx.y_train)
        out['P'] = apply_temperature(out['P'], T)
    out['P'] = _mask_absent(out['P'], out['prior'])
    out['T'] = T
    return out


def run_model(cfg, ctx, rows):
    if 'ensemble' not in cfg:
        return run_single(cfg, ctx, rows)
    members = cfg['ensemble']
    weights = np.array(cfg.get('weights') or [1.] * len(members)); weights /= weights.sum()
    outs = [run_single(m, ctx, rows) for m in members]
    return dict(P=sum(w * o['P'] for w, o in zip(weights, outs)),
                prior=sum(w * o['prior'] for w, o in zip(weights, outs)),
                T=[o['T'] for o in outs], info={'members': [o['info'] for o in outs]})

Overwriting _source_export/part1/experiments/cand2_followup_v1/strict_adapter.py


## part1/experiments/cand2_followup_v1/verify_retained.py

In [50]:
%%writefile _source_export/part1/experiments/cand2_followup_v1/verify_retained.py
"""Check retained matched comparisons and independently rescore them; no model fits."""
import json
import csv
import tempfile
from pathlib import Path
import shutil
import numpy as np
from run import HERE,P1,LABELS,read,sha,freeze,collect,save
from analyze import independent


def main():
    freeze()
    retained=read(HERE/'retention.json')
    for name,digest in retained['sha256'].items():assert sha(HERE/name)==digest,name
    for name,meta in read(HERE/'input_provenance.json').items():assert sha(HERE/'inputs'/name)==meta['subset_sha256'],name
    for name,digest in read(HERE/'provenance.json')['copied_sha256'].items():assert sha(HERE/name)==digest,name
    export=read(HERE/'blend_prepared_submission.json');protocol=read(HERE/'blend_export_protocol.json')
    assert sha(P1/export['csv'])==export['sha256']
    assert sha(HERE/'export_blend.py')==protocol['builder_sha256']
    with np.load(HERE/'blend_final_predictions.npz') as z:
        scores=(4*z['anchored_scores']+z['cand2_scores'])/5
        np.testing.assert_allclose(scores,z['scores'],atol=1e-15,rtol=0)
        pred=(scores>=.3).astype(int);pred[np.arange(len(pred)),scores.argmax(1)]=1
        np.testing.assert_array_equal(pred,z['predictions'])
        with (P1/export['csv']).open(newline='') as f:rows=list(csv.DictReader(f))
        assert [r['ID'] for r in rows]==z['ids'].tolist()
        assert [r['Predicted'] for r in rows]==[' '.join(l for l,v in zip(LABELS,p) if v) for p in pred]
    output=Path(tempfile.mkdtemp(prefix='cand2-followup-verify-'))
    try:
        for p in (HERE/'results/J_packets').iterdir():
            if p.is_file():shutil.copyfile(p,output/p.name)
        metrics,labels,audits,_=collect(output)
        metrics.to_csv(output/'repeat_metrics.csv',index=False);labels.to_csv(output/'label_metrics.csv',index=False)
        independent(output)
        for name in ['summary.csv','matched_J.csv','rank_agreement.csv','expanded_panel.csv','pairwise_orderings.csv','repeat_metrics.csv','label_metrics.csv','complementarity.csv']:
            assert (output/'analysis'/name).read_bytes()==(HERE/'results'/name).read_bytes(),name
        with np.load(output/'native_dylan__validation.npz') as new,np.load(HERE/'inputs/strict_inductive__validation.npz') as old:
            np.testing.assert_array_equal(new['cand2_predictions'],old['cand2_predictions'])
            np.testing.assert_allclose(new['cand2_scores'],old['cand2_scores'],atol=1e-12,rtol=0)
        print('Retained hashes, repeated aggregation, independent rescoring and exact-recipe validation control passed.')
    finally:shutil.rmtree(output)


if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/cand2_followup_v1/verify_retained.py


## part1/experiments/cand2_submission_v1/build.py

In [51]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/build.py
"""Export the previously evaluated strict-inductive, five-seed Cand2 recipe."""
import os
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    os.environ[key] = '1'
from concurrent.futures import ProcessPoolExecutor
from dataclasses import replace
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import sys

HERE = Path(__file__).resolve().parent
P1 = HERE.parents[1]
sys.path.insert(0, str(HERE / 'source'))
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from p1lib import cache
from p1lib.data import load, LABELS, write_submission
from p1lib.head import apply_head
import strict_adapter as strict

CSV = P1 / 'submissions/candidates/cand2_cs_features_inductive5.csv'


def read(p): return json.loads(Path(p).read_text())
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save(p, obj): Path(p).write_text(json.dumps(obj, indent=2) + '\n')


def freeze():
    files = [HERE / 'build.py', HERE / 'strict_adapter.py', HERE / 'original_manifest.json',
             HERE / 'validation_reference.npz', HERE / 'validation_case.json', HERE / 'prior_comparison.csv']
    files += sorted((HERE / 'source/p1lib').glob('*.py')) + sorted((P1 / 'data').glob('*.csv'))
    hashes = {str(p.relative_to(P1)): sha(p) for p in files}
    path = HERE / 'protocol.json'
    if path.exists():
        assert read(path)['source_and_data_sha256'] == hashes
    else:
        save(path, dict(created_utc=datetime.now(timezone.utc).isoformat(),
            base_main='256fa8dabb2cd243f1162a0356e2c55b7a369a59', source_study_commit='2bf7e4f',
            recipe='cand2_compliant_argmax, strict-inductive adapter, five-inner-seed average',
            parent='Anchored 0.89636', status='Exploratory final fit authorized by user; failed prior CV advancement rule.',
            settings='Identical model/head to original manifest; transductive outer flag replaced by explicit strict context.',
            seeds=[0,1,2,3,4], aggregation='Average float64 combination posteriors and priors before expected-F1 decoder.',
            fitting='235 verified train/validation labels + 1014 originally masked training rows. No test inputs in fitting.',
            nested='Grouped inner calibration; complete inner refit; masked duplicate group purge; group-safe synthetic KNN features.',
            checks='Fresh five-seed validation replay must match retained benchmark before exporting final235 predictions.',
            selection='Selected for a distinct graph/feature/decision-rule direction, not a claimed local benchmark improvement.',
            no_upload=True, source_and_data_sha256=hashes))


def prepare(D, validation):
    if not validation:
        return strict.context(D, D.idx['labval'], D.idx['test'], D.idx['unl'], seed=0), D.idx['test']
    case = read(HERE / 'validation_case.json')
    pos = {v: i for i, v in enumerate(D.df.ID)}
    train, query, pool, fit = [np.array([pos[v] for v in case[key]]) for key in ['train_gold','query_ids','masked_ids','fit_ids']]
    keep = np.union1d(fit, query)
    remap = {v: i for i, v in enumerate(keep)}
    tr, qu, po = [np.array([remap[v] for v in rows]) for rows in [train, query, pool]]
    idx = dict(train=np.array([i for i,v in enumerate(keep) if v in set(D.idx['train'])]),
        lab=np.array([i for i,v in enumerate(keep) if v in set(D.idx['lab'])]),
        val=np.array([i for i,v in enumerate(keep) if v in set(D.idx['val'])]),
        unl=po, test=np.array([],int), labval=np.r_[tr,qu])
    sub = replace(D, df=D.df.iloc[keep].reset_index(drop=True), emb=D.emb[keep],
                  Y=D.Y[keep], y=D.y[keep], idx=idx, year=D.year[keep])
    return strict.context(sub,tr,qu,po,seed=0), qu


def one_seed(arg):
    validation, seed = arg
    D = load(str(P1 / 'data'))
    ctx, query = prepare(D, validation)
    ctx = replace(ctx, seed=seed)
    assert not np.intersect1d(ctx.unsup, query).size
    assert (ctx.D.Y[query] == -1).all()
    cfg = read(HERE / 'original_manifest.json')['config']
    cache.clear(); strict.AUDIT.clear()
    with threadpool_limits(limits=1):
        out = strict.run_model(cfg['model'],ctx,np.r_[query,ctx.pool])
    assert not any(a['gold_group_overlap'] or a['masked_group_overlap'] for a in strict.AUDIT)
    print(('Validation' if validation else 'Final235'), 'seed', seed, 'complete', flush=True)
    return dict(P=out['P'][:len(query)], prior=out['prior'], temperature=out['T'],
                nested_contexts=len(strict.AUDIT), fitting_gold=len(ctx.train), fitting_masked=len(ctx.pool))


def average(outputs, CM):
    P = np.mean([v['P'] for v in outputs],axis=0)
    prior = np.mean([v['prior'] for v in outputs],axis=0)
    assert np.isfinite(P).all() and np.allclose(P.sum(1),1)
    yp,Q,_ = apply_head(P,prior,CM,read(HERE/'original_manifest.json')['config']['head'])
    return yp,Q,P


def main():
    freeze()
    assert not CSV.exists(), 'Preserve exact exported CSV; use a new output version for another export.'
    D=load(str(P1/'data'))
    with ProcessPoolExecutor(max_workers=5) as pool:
        val=list(pool.map(one_seed,[(True,s) for s in range(5)]))
        vp,vq,vp0=average(val,D.CM)
        with np.load(HERE/'validation_reference.npz') as ref:
            np.testing.assert_allclose(vp0,ref['cand2_P'],atol=1e-12,rtol=0)
            np.testing.assert_allclose(vq,ref['cand2_Q'],atol=1e-12,rtol=0)
            np.testing.assert_array_equal(vp,ref['cand2_predictions'])
            replay_error=float(abs(vq-ref['cand2_Q']).max())
        print('Five-seed validation benchmark exactly reproduced.',flush=True)
        final=list(pool.map(one_seed,[(False,s) for s in range(5)]))
    yp,Q,P=average(final,D.CM)
    assert yp.shape==(312,5) and (yp.sum(1)>0).all()
    CSV.parent.mkdir(parents=True,exist_ok=True)
    sub=write_submission(D,yp,str(CSV))
    sample=pd.read_csv(P1/'data/part1_sample_submission.csv')
    assert list(sub.columns)==list(sample.columns) and sub.ID.tolist()==sample.ID.tolist()
    assert sub.ID.is_unique and len(sub)==312 and sub.Predicted.str.len().gt(0).all()
    allowed=set(LABELS)
    assert all(set(v.split()).issubset(allowed) for v in sub.Predicted)
    np.savez_compressed(HERE/'final_predictions.npz',ids=sub.ID.to_numpy(str),P=P,Q=Q,CM=D.CM,predictions=yp)
    anchor=pd.read_csv(P1/'submissions/archive/submitted/anchored_reason_trainval235__kaggle_0.89636.csv').set_index('ID')
    diff=sub.Predicted.to_numpy()!=anchor.loc[sub.ID,'Predicted'].to_numpy()
    changed=sub.loc[diff].copy();changed['anchored']=anchor.loc[changed.ID,'Predicted'].to_numpy()
    changed.to_csv(HERE/'changed_predictions.csv',index=False)
    meta=dict(status='prepared; not uploaded; Kaggle score unknown',csv=str(CSV.relative_to(P1)),
        sha256=sha(CSV),rows=312,verified_gold=235,originally_masked=1014,
        test_features_used_for_fit=False,prior_cv_passed=False,
        changed_vs_anchored=int(diff.sum()),label_counts=dict(zip(LABELS,map(int,yp.sum(0)))),
        mean_label_count=float(yp.sum(1).mean()),validation_replay_max_error=replay_error,
        validation_predictions_identical=True,seeds=[0,1,2,3,4],
        temperatures=[float(v['temperature']) for v in final],
        nested_contexts=sum(v['nested_contexts'] for v in final),
        prediction_sha256=sha(HERE/'final_predictions.npz'),protocol_sha256=sha(HERE/'protocol.json'))
    save(HERE/'prepared_submission.json',meta)
    freeze()
    print(json.dumps(meta,indent=2),flush=True)


if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/cand2_submission_v1/build.py


## part1/experiments/cand2_submission_v1/check_export.py

In [52]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/check_export.py
"""Independent saved-prediction decoding and export verification; no fitting."""
import json
import numpy as np
import pandas as pd
from build import HERE, P1, CSV, LABELS, freeze, sha, read, save


def main():
    freeze()
    record=read(HERE/'prepared_submission.json')
    assert sha(CSV)==record['sha256']
    assert sha(HERE/'final_predictions.npz')==record['prediction_sha256']
    with np.load(HERE/'final_predictions.npz') as z:
        q,cm=z['Q'],z['CM']
        possible=np.array([[(bits>>j)&1 for j in range(5)] for bits in range(1,32)])
        utility=np.array([[2*np.sum(a*b)/(a.sum()+b.sum()) for b in cm] for a in possible])
        decoded=possible[np.argmax(q@utility.T,axis=1)]
        np.testing.assert_array_equal(decoded,z['predictions'])
        assert np.isfinite(q).all() and np.allclose(q.sum(1),1) and (q>=0).all()
        output=pd.read_csv(CSV)
        sample=pd.read_csv(P1/'data/part1_sample_submission.csv')
        assert output.ID.tolist()==sample.ID.tolist()==z['ids'].tolist()
        assert output.ID.is_unique and len(output)==312
        expected=[' '.join(label for label,yes in zip(LABELS,row) if yes) for row in decoded]
        assert output.Predicted.tolist()==expected
    anchor=pd.read_csv(P1/'submissions/archive/submitted/anchored_reason_trainval235__kaggle_0.89636.csv').set_index('ID')
    diff=sum(set(a.split())!=set(b.split()) for a,b in zip(output.Predicted,anchor.loc[output.ID,'Predicted']))
    assert diff==record['changed_vs_anchored']==49
    for name,expected_hash in read(HERE/'provenance.json')['copied_sha256'].items():
        assert sha(HERE/name)==expected_hash,name
    result=dict(status='passed',source_data_and_artifact_hashes=True,
        independent_expected_f1_decode=True,ids_order_and_nonempty_labels=True,
        rows=312,changed_label_sets_vs_anchored=diff,validation_replay_max_error=record['validation_replay_max_error'],
        kaggle_score=None,uploaded=False)
    save(HERE/'verification.json',result)
    print(json.dumps(result,indent=2))


if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/cand2_submission_v1/check_export.py


## part1/experiments/cand2_submission_v1/source/p1lib/__init__.py

In [53]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/__init__.py
"""p1lib: composable semi-supervised experimentation framework for CS5785 midterm Part 1."""
from .data import LABELS, load, metrics, row_f1, write_submission  # noqa: F401
from .evaluate import evaluate, make_folds, score, fold_outputs, summary_row, paired_se  # noqa: F401
from .pipeline import run_model, validate  # noqa: F401

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/__init__.py


## part1/experiments/cand2_submission_v1/source/p1lib/blocks.py

In [54]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/blocks.py
"""Optional non-embedding feature blocks (each a toggle in the features config).

Every block is label-free: vocabularies / IDF / SVD are fit only on the rows the fold allows
(`fit_rows`), then applied to all rows.  Output is a dense (n_all, d) array, standardized on
`fit_rows`.

Blocks
  criteria  : TF-IDF of eligibility criteria  -> TruncatedSVD(d)
  diseases  : TF-IDF of diseases_cleaned      -> TruncatedSVD(d)
  drugs     : multi-hot matched_drugs         -> TruncatedSVD(d)
  pathways  : multi-hot drug_pathways (optionally only Reactome level <= max_level) -> SVD(d)
  morgan    : RDKit Morgan fingerprint (radius 2, 1024 bits, max over drugs) -> SVD(d)
  drugmeta  : [has_matched_drug, log1p(#matched drugs)]
  year      : standardized start_year  (sampling artifact; ablation only)
"""
from __future__ import annotations

import numpy as np
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from .cache import memo

_MORGAN = None


def _split_semicolon(s):
    if not isinstance(s, str) or not s.strip():
        return []
    return [t.strip().lower() for t in s.split(";") if t.strip()]


def _morgan_matrix(D):
    global _MORGAN
    if _MORGAN is not None:
        return _MORGAN
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog("rdApp.*")
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    out = np.zeros((D.n, 1024), dtype=np.float32)
    for i, s in enumerate(D.df["drug_smiles"].to_numpy()):
        if not isinstance(s, str):
            continue
        fps = [gen.GetFingerprintAsNumPy(m) for t in s.split(";") if (m := Chem.MolFromSmiles(t)) is not None]
        if fps:
            out[i] = np.max(fps, axis=0)
    _MORGAN = out
    return out


def _svd(M, fit_rows, d, seed=0):
    d = int(min(d, M.shape[1] - 1, len(fit_rows) - 1))
    svd = TruncatedSVD(n_components=max(d, 1), random_state=seed).fit(M[fit_rows])
    return svd.transform(M)


def _standardize(Z, fit_rows):
    mu = Z[fit_rows].mean(0)
    sd = Z[fit_rows].std(0)
    sd[sd < 1e-12] = 1.0
    return (Z - mu) / sd


def build_block(D, name: str, cfg: dict, fit_rows: np.ndarray) -> np.ndarray:
    cfg = dict(cfg or {})
    key = {"block": name, **cfg}
    return memo("block", key, fit_rows, lambda: _build(D, name, cfg, fit_rows))


def _build(D, name, cfg, fit_rows):
    d = int(cfg.get("d", 32))
    df = D.df
    if name == "criteria" or name == "diseases":
        col = "criteria" if name == "criteria" else "diseases_cleaned"
        text = df[col].fillna("").astype(str).to_numpy()
        vec = TfidfVectorizer(max_features=int(cfg.get("max_features", 5000)), min_df=3,
                              sublinear_tf=True, ngram_range=(1, int(cfg.get("ngram", 1))),
                              stop_words="english")
        vec.fit(text[fit_rows])
        M = vec.transform(text)
        Z = _svd(M, fit_rows, d)
    elif name in ("drugs", "pathways"):
        col = "matched_drugs" if name == "drugs" else "drug_pathways"
        toks = [_split_semicolon(s) for s in df[col].to_numpy()]
        if name == "pathways" and cfg.get("max_level") is not None:
            lvl = dict(zip(D.catalog["reactome_id"].str.lower(), D.catalog["pathway_level"]))
            ml = int(cfg["max_level"])
            toks = [[t for t in ts if lvl.get(t, 99) <= ml] for ts in toks]
        docs = [" ".join(t.replace(" ", "_") for t in ts) for ts in toks]
        vec = CountVectorizer(binary=True, min_df=3, token_pattern=r"[^ ]+")
        try:
            vec.fit([docs[i] for i in fit_rows])
            M = vec.transform(docs).astype(float)
        except ValueError:            # empty vocabulary
            M = sparse.csr_matrix((D.n, 1))
        Z = _svd(M, fit_rows, d) if M.shape[1] > 2 else M.toarray()
    elif name == "morgan":
        M = _morgan_matrix(D)
        Z = _svd(sparse.csr_matrix(M), fit_rows, d)
    elif name == "drugmeta":
        toks = [_split_semicolon(s) for s in df["matched_drugs"].to_numpy()]
        Z = np.c_[[len(t) > 0 for t in toks], np.log1p([len(t) for t in toks])].astype(float)
    elif name == "year":
        Z = df["start_year"].to_numpy(float)[:, None]
    else:
        raise ValueError(f"unknown block {name}")
    return _standardize(np.asarray(Z, float), fit_rows) * float(cfg.get("weight", 1.0))

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/blocks.py


## part1/experiments/cand2_submission_v1/source/p1lib/cache.py

In [55]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/cache.py
"""Tiny in-process memo for label-free fits (scalers, PCA, clusterers, graphs).

A label-free object depends only on (its config, the set of rows whose features it was fit on).
In transductive mode that row set is the same (all 1,561 rows) for every fold, so one fit is
reused across folds and trials; in inductive mode the row set changes per fold and the cache
simply misses.  Nothing that touches labels is ever cached here.
"""
from __future__ import annotations

import hashlib
import json

import numpy as np

_CACHE: dict = {}
_MAX = 4000


def cfg_key(obj) -> str:
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]


def rows_key(rows: np.ndarray) -> str:
    return hashlib.md5(np.sort(np.asarray(rows)).astype(np.int64).tobytes()).hexdigest()[:16]


def memo(kind: str, cfg, rows, fn):
    key = (kind, cfg_key(cfg), rows_key(rows))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val


def clear():
    _CACHE.clear()


def memo_ctx(kind: str, cfg, ctx, fn, extra=None):
    """Memo for LABEL-DEPENDENT results: keyed by the exact training rows (so a result is only
    reused for the identical fold), the feature rows, the pool and the requested rows."""
    key = (kind, cfg_key(cfg), rows_key(ctx.train), rows_key(ctx.unsup), rows_key(ctx.pool),
           ctx.seed, None if extra is None else cfg_key(extra))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/cache.py


## part1/experiments/cand2_submission_v1/source/p1lib/classifiers.py

In [56]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/classifiers.py
"""Classifiers over the K label combinations.  Common interface:

    clf = make_classifier(cfg, K, CM)
    clf.fit(X, y, w)            # y: int combo ids, w: sample weights (soft labels are
                                #    passed as replicated rows, one per combo, weight lam*q)
    P = clf.predict_proba(X)    # (n, K), columns aligned to all K combos
    clf.prior_                  # effective source prior the model was trained under
                                # (needed by SLD; uniform when class_weight='balanced')

cfg["type"] in {"lr", "ovr", "lda", "knn", "hgb", "nb"}.
"""
from __future__ import annotations

import warnings

import numpy as np
from sklearn.covariance import ledoit_wolf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


def _weighted_prior(y, w, K):
    p = np.bincount(y, weights=w, minlength=K).astype(float)
    return p / p.sum()


def _balance(y, w, K):
    """class_weight='balanced' computed from the weighted label mass."""
    mass = np.bincount(y, weights=w, minlength=K)
    present = mass > 0
    cw = np.zeros(K)
    cw[present] = w.sum() / (present.sum() * mass[present])
    return w * cw[y], present


class _Base:
    def __init__(self, cfg, K, CM):
        self.cfg = cfg
        self.K = K
        self.CM = CM

    def _prep(self, y, w):
        w = np.ones(len(y)) if w is None else np.asarray(w, float)
        if self.cfg.get("class_weight") == "balanced":
            w, present = _balance(y, w, self.K)
            self.prior_ = present / present.sum()
        else:
            self.prior_ = _weighted_prior(y, w, self.K)
        return w

    def _full(self, Pc, classes):
        P = np.zeros((Pc.shape[0], self.K))
        P[:, classes] = Pc
        P = P + 1e-9
        return P / P.sum(1, keepdims=True)


class LRPowerset(_Base):
    """Multinomial logistic regression on the label powerset (T1)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.m = LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                    tol=1e-5).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class OvRLR(_Base):
    """One-vs-rest LR on the 5 labels; combo probabilities = product of marginals renormalized
    over the K allowed combinations (so the output is always a valid label set)."""

    def fit(self, X, y, w=None):
        w0 = np.ones(len(y)) if w is None else np.asarray(w, float)
        self.prior_ = _weighted_prior(y, w0, self.K)
        B = self.CM[y]
        self.ms = []
        for j in range(5):
            yj = B[:, j]
            wj = w0.copy()
            if self.cfg.get("class_weight") == "balanced":
                pos = wj[yj == 1].sum()
                neg = wj[yj == 0].sum()
                if pos > 0 and neg > 0:
                    wj = np.where(yj == 1, wj * (pos + neg) / (2 * pos), wj * (pos + neg) / (2 * neg))
            if yj.min() == yj.max():
                self.ms.append(float(yj[0]))
                continue
            self.ms.append(LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                              tol=1e-5).fit(X, yj, sample_weight=wj))
        if self.cfg.get("class_weight") == "balanced":
            self.prior_ = np.ones(self.K) / self.K
        return self

    def marginals(self, X):
        M = np.zeros((len(X), 5))
        for j, m in enumerate(self.ms):
            M[:, j] = m if isinstance(m, float) else m.predict_proba(X)[:, 1]
        return np.clip(M, 1e-6, 1 - 1e-6)

    def predict_proba(self, X):
        M = self.marginals(X)
        L = np.log(M) @ self.CM.T + np.log(1 - M) @ (1 - self.CM).T
        L -= L.max(1, keepdims=True)
        P = np.exp(L)
        return P / P.sum(1, keepdims=True)


class ShrinkLDA(_Base):
    """Shrinkage LDA (tied Gaussian classes) with sample-weight support.  Follows sklearn's
    solver='lsqr', shrinkage='auto': per-class Ledoit-Wolf covariances (on standardized
    features) averaged with the class priors."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        K, d = self.K, X.shape[1]
        self.classes_ = np.unique(y)
        means = np.zeros((K, d))
        S = np.zeros((d, d))
        prior = self.prior_
        for k in self.classes_:
            m = y == k
            wk = w[m]
            Xk = X[m]
            mu = (wk[:, None] * Xk).sum(0) / wk.sum()
            means[k] = mu
            Xc = (Xk - mu) * np.sqrt(wk / wk.mean())[:, None]
            if len(Xk) < 2:
                continue
            sd = Xc.std(0)
            sd[sd < 1e-12] = 1.0
            C = ledoit_wolf(Xc / sd, assume_centered=True)[0]
            S += prior[k] * (sd[:, None] * C * sd[None, :])
        S += 1e-8 * np.eye(d)
        self.means_ = means
        self.coef_ = np.linalg.solve(S, means[self.classes_].T).T
        self.intercept_ = -0.5 * (self.coef_ * means[self.classes_]).sum(1) + np.log(prior[self.classes_])
        return self

    def predict_proba(self, X):
        Lg = X @ self.coef_.T + self.intercept_
        Lg -= Lg.max(1, keepdims=True)
        P = np.exp(Lg)
        P /= P.sum(1, keepdims=True)
        return self._full(P, self.classes_)


class KNNClf(_Base):
    """Cosine kNN over combos with distance weighting and soft targets; smoothed toward the
    training prior with beta pseudo-counts so probabilities are never exactly 0."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        self.Yw = np.eye(self.K)[y] * w[:, None]
        self.w = w
        return self

    def predict_proba(self, X, exclude_self=False):
        k = int(self.cfg.get("k", 10))
        Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        S = Xn @ self.Xn.T
        if exclude_self:
            np.fill_diagonal(S, -np.inf)
        k = min(k, S.shape[1] - (1 if exclude_self else 0))
        nn = np.argpartition(-S, k - 1, axis=1)[:, :k]
        sims = np.take_along_axis(S, nn, 1)
        if self.cfg.get("weights", "distance") == "distance":
            wt = np.exp(sims / float(self.cfg.get("tau", 0.1)))
        else:
            wt = np.ones_like(sims)
        votes = np.einsum("nk,nkc->nc", wt, self.Yw[nn])
        mass = np.einsum("nk,nk->n", wt, self.w[nn])
        beta = float(self.cfg.get("beta", 1.0))
        P = (votes + beta * self.prior_) / (mass + beta)[:, None]
        return P / P.sum(1, keepdims=True)


class HGB(_Base):
    """HistGradientBoosting over combos (mainly as a stacker on meta-features)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        c = self.cfg
        self.m = HistGradientBoostingClassifier(
            learning_rate=float(c.get("learning_rate", 0.05)), max_iter=int(c.get("max_iter", 100)),
            max_leaf_nodes=int(c.get("max_leaf_nodes", 6)), max_depth=c.get("max_depth", 3),
            min_samples_leaf=int(c.get("min_samples_leaf", 20)),
            l2_regularization=float(c.get("l2_regularization", 1.0)),
            max_features=float(c.get("max_features", 1.0)), early_stopping=False,
            random_state=int(c.get("seed", 0))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class GaussNB(_Base):
    """Gaussian naive Bayes with variance smoothing (cheap, high-bias baseline)."""

    def fit(self, X, y, w=None):
        from sklearn.naive_bayes import GaussianNB
        w = self._prep(y, w)
        self.m = GaussianNB(var_smoothing=float(self.cfg.get("var_smoothing", 1e-2))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class PriorClf(_Base):
    """Predicts the training prior for every row (MAP -> always the majority combo)."""

    def fit(self, X, y, w=None):
        self._prep(y, w)
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))


REGISTRY = {"prior": PriorClf, "lr": LRPowerset, "ovr": OvRLR, "lda": ShrinkLDA, "knn": KNNClf, "hgb": HGB, "nb": GaussNB}


def make_classifier(cfg, K, CM):
    return REGISTRY[cfg["type"]](cfg, K, CM)

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/classifiers.py


## part1/experiments/cand2_submission_v1/source/p1lib/configs.py

In [57]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/configs.py
"""Named reference configurations (the report's five techniques, baselines and ensembles).

Every entry is a full experiment config: {"model": ..., "head": ..., "transductive": ...}.
Build new ones by copying and editing; `exp(model, head)` is a small helper.
"""
from __future__ import annotations

import copy

STD = {"scale": True}
PCA64 = {"scale": True, "pca": 64}
GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}

NO_HEAD = {"prior": "none", "decode": "map"}
SLD_HEAD = {"prior": "sld", "sld_a": 100, "decode": "map"}


def exp(model, head=None, transductive=True):
    return {"model": copy.deepcopy(model), "head": copy.deepcopy(head or NO_HEAD), "transductive": transductive}


def single(clf, features=STD, synth=None, labeler=None, temperature=False):
    return {"features": copy.deepcopy(features), "synth": copy.deepcopy(synth or []),
            "labeler": copy.deepcopy(labeler), "clf": copy.deepcopy(clf), "temperature": temperature}


LR = {"type": "lr", "C": 0.1}
LDA = {"type": "lda"}
OVR = {"type": "ovr", "C": 0.1}
KNN = {"type": "knn", "k": 10, "weights": "distance", "tau": 0.1, "beta": 1.0}

M_LR = single(LR)
M_LDA = single(LDA)
M_OVR = single(OVR)
M_KNN = single(KNN, features={"scale": True, "pca": 64, "l2": True})
M_CS = single({"type": "direct", "labeler": {"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5}})
M_SPREAD = single({"type": "direct", "labeler": {"type": "spread", "graph": GRAPH, "alpha": 0.5}})
M_CLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "kmeans", "k": 24, "m": 4}})
M_GMMCLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "gmm", "k": 16, "cov": "diag", "m": 4}})

# T1 with SLD-filled pseudo-labels for the unlabeled pool
M_LR_SLDPL = single(LR, labeler={"type": "sld", "base": LR, "a": 100, "lam": 0.1})
# T2 C&S pseudo-labels feeding LR
M_LR_CSPL = single(LR, labeler={"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5, "lam": 0.1})
# T4 cluster-then-label pseudo-labels feeding LR
M_LR_CLPL = single(LR, labeler={"type": "cluster", "algo": "kmeans", "k": 24, "m": 4, "lam": 0.1})
# T3 guarded self-training
M_LR_ST = single(LR, labeler={"type": "selftrain", "members": ["lr", "lda", "cs"], "rounds": 2, "tau": 0.75,
                              "lam": 0.3, "ramp": True, "align": "sld", "weighting": "softmatch",
                              "cluster_filter": True})
M_LR_ST_NAIVE = single(LR, labeler={"type": "selftrain", "members": ["lr"], "rounds": 3, "tau": 0.75,
                                    "lam": 0.5, "align": "none", "hard": True})
# synthetic-feature variants
M_LR_SYN = single(LR, synth=[{"type": "gmm", "k": 16, "cov": "diag"}, {"type": "knn_vote", "k": 10},
                             {"type": "graph", "mode": "spread", "graph": GRAPH, "alpha": 0.5},
                             {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}])
# T5 HGB stacker on nested-OOF meta-features
META = [{"type": "oof_model", "model": M_LR}, {"type": "oof_model", "model": M_LDA},
        {"type": "graph", "mode": "cs", "graph": GRAPH, "alpha": 0.5},
        {"type": "knn_vote", "k": 10}, {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}]
HGB = {"type": "hgb", "learning_rate": 0.05, "max_iter": 100, "max_leaf_nodes": 4, "max_depth": 2,
       "min_samples_leaf": 20, "l2_regularization": 1.0}
M_STACK_HGB = single(HGB, features=None, synth=META)
M_STACK_LR = single({"type": "lr", "C": 0.1}, features=None, synth=META)
M_HGB_RAW = single(HGB, features=STD)

# report's recommended ensemble: T1-LR + T1-LDA + T2-C&S, equal weights
M_ENS3 = {"ensemble": [M_LR, M_LDA, M_CS], "weights": None}

M_PRIOR = single({"type": "prior"})
M_NB = single({"type": "nb"}, features=PCA64)
SLD_EF1 = {"prior": "sld", "sld_a": 25, "decode": "expf1"}

REFERENCE = {
    "Majority": exp(M_PRIOR),
    "T1_LR": exp(M_LR),
    "T1_LR+SLD": exp(M_LR, SLD_HEAD),
    "T1_LDA": exp(M_LDA),
    "T1_LDA+SLD": exp(M_LDA, SLD_HEAD),
    "OvR_LR": exp(M_OVR),
    "kNN": exp(M_KNN),
    "T2_CS": exp(M_CS),
    "T2_CS+SLD": exp(M_CS, SLD_HEAD),
    "T2_spread": exp(M_SPREAD),
    "T4_kmeans_direct": exp(M_CLUSTER),
    "T4_gmm_direct": exp(M_GMMCLUSTER),
    "T1_LR+SLDpseudo": exp(M_LR_SLDPL),
    "T1_LR+SLDpseudo+SLD": exp(M_LR_SLDPL, SLD_HEAD),
    "T2_LR+CSpseudo": exp(M_LR_CSPL),
    "T4_LR+clusterpseudo": exp(M_LR_CLPL),
    "T3_selftrain_guarded": exp(M_LR_ST),
    "T3_selftrain_naive": exp(M_LR_ST_NAIVE),
    "LR+synth": exp(M_LR_SYN),
    "T5_stack_HGB": exp(M_STACK_HGB),
    "T5_stack_HGB+SLD": exp(M_STACK_HGB, SLD_HEAD),
    "stack_LR": exp(M_STACK_LR),
    "HGB_raw": exp(M_HGB_RAW),
    "ENS3": exp(M_ENS3),
    "ENS3+SLD": exp(M_ENS3, SLD_HEAD),
}

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/configs.py


## part1/experiments/cand2_submission_v1/source/p1lib/context.py

In [58]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/context.py
"""Fold context: which rows are labeled-for-training, which are predicted, which are unlabeled.

Every component receives a `Ctx` and may only read labels of `ctx.train`.  Features of
`ctx.unsup` rows may be used by label-free steps (scaler, PCA, clustering, graph nodes).

transductive=True  -> unsup = all 1,561 rows (train + val + test features, no labels)
transductive=False -> unsup = all rows except the eval rows and the test rows
"""
from __future__ import annotations

from dataclasses import dataclass, replace

import numpy as np


@dataclass
class Ctx:
    D: object
    train: np.ndarray            # global idx of labeled training rows (labels visible)
    eval: np.ndarray             # global idx of rows to predict (labels hidden)
    pool: np.ndarray             # global idx of the pseudo-label pool (default: 1,014 unlabeled)
    unsup: np.ndarray            # global idx of rows whose features label-free steps may use
    transductive: bool = True
    seed: int = 0
    name: str = ""
    depth: int = 0               # nesting depth (0 = outer fold)

    @property
    def y_train(self):
        return self.D.y[self.train]

    @property
    def sld_target(self):
        """Rows SLD estimates the target prior on: the eval batch when transductive, else the
        unlabeled pool (inductive: we may not look at the eval/test features)."""
        return self.eval if self.transductive else self.pool

    def sub(self, train, eval_, name=None):
        """Inner context for nested OOF: same unlabeled pool / feature rows, fewer labels."""
        return replace(self, train=np.asarray(train), eval=np.asarray(eval_),
                       name=name or self.name + "/inner", depth=self.depth + 1)


def make_ctx(D, train, eval_, transductive=True, pool=None, seed=0, name=""):
    train = np.asarray(train)
    eval_ = np.asarray(eval_)
    pool = D.idx["unl"] if pool is None else np.asarray(pool)
    if transductive:
        unsup = np.arange(D.n)
    else:
        banned = np.zeros(D.n, bool)
        banned[eval_] = True
        banned[D.idx["test"]] = True
        unsup = np.where(~banned)[0]
    assert len(np.intersect1d(train, eval_)) == 0, "train/eval overlap"
    assert (D.y[train] >= 0).all(), "training rows must be labeled"
    return Ctx(D=D, train=train, eval=eval_, pool=pool, unsup=unsup,
               transductive=transductive, seed=seed, name=name)

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/context.py


## part1/experiments/cand2_submission_v1/source/p1lib/data.py

In [59]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/data.py
"""Data loading, label-powerset encoding and metrics for Part 1.

Global row order used everywhere in this package:
    [ part1_train (1171 rows, file order) | part1_val (78) | part1_test (312) ]  -> 1561 rows.
`Data.y` holds the label-combo id (0..7) for labeled rows and -1 for masked/test rows.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field

import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

LABELS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMB = [f"emb_{i}" for i in range(150)]
BITS = np.array([1, 2, 4, 8, 16])

DEFAULT_DATA_DIR = os.environ.get(
    "P1_DATA_DIR",
    os.path.join(os.path.dirname(__file__), "..", "..", "..", "part1", "data"),
)


@dataclass
class Data:
    df: pd.DataFrame                 # all 1561 rows, global order, with a 'split' column
    emb: np.ndarray                  # (1561, 150) raw embeddings
    Y: np.ndarray                    # (1561, 5) labels, -1 where unknown
    y: np.ndarray                    # (1561,) combo id, -1 where unknown
    combos: np.ndarray               # (K,) bitmask of each allowed combo
    CM: np.ndarray                   # (K, 5) binary label matrix of each combo
    idx: dict = field(default_factory=dict)   # named index groups
    year: np.ndarray = None
    catalog: pd.DataFrame = None

    @property
    def K(self) -> int:
        return len(self.combos)

    @property
    def n(self) -> int:
        return len(self.df)

    def combo_names(self):
        return ["+".join(LABELS[j][0] for j in range(5) if self.CM[k, j]) for k in range(self.K)]


def load(data_dir: str | None = None) -> Data:
    d = os.path.abspath(data_dir or DEFAULT_DATA_DIR)
    tr = pd.read_csv(os.path.join(d, "part1_train.csv"))
    va = pd.read_csv(os.path.join(d, "part1_val.csv"))
    te = pd.read_csv(os.path.join(d, "part1_test.csv"))
    cat = pd.read_csv(os.path.join(d, "pathway_catalog.csv"))
    te = pd.concat([te, pd.DataFrame(-1, index=te.index, columns=LABELS)], axis=1)
    tr = tr.assign(split="train")
    va = va.assign(split="val")
    te = te.assign(split="test")
    df = pd.concat([tr, va, te], ignore_index=True).copy()
    Y = df[LABELS].to_numpy().astype(int)
    Y[Y == -100] = -1
    known = (Y >= 0).all(1)
    mask = (Y.clip(0) * BITS).sum(1)
    combos = np.unique(mask[known])
    y = np.full(len(df), -1)
    y[known] = np.searchsorted(combos, mask[known])
    CM = np.array([[(c >> j) & 1 for j in range(5)] for c in combos])
    split = df["split"].to_numpy()
    idx = {
        "train": np.where(split == "train")[0],
        "lab": np.where((split == "train") & known)[0],
        "unl": np.where((split == "train") & ~known)[0],
        "val": np.where(split == "val")[0],
        "test": np.where(split == "test")[0],
    }
    idx["labval"] = np.r_[idx["lab"], idx["val"]]
    D = Data(df=df, emb=df[EMB].to_numpy(float), Y=Y, y=y, combos=combos, CM=CM, idx=idx,
             year=df["start_year"].to_numpy(int), catalog=cat)
    assert len(idx["lab"]) == 157 and len(idx["unl"]) == 1014 and len(idx["val"]) == 78 and len(idx["test"]) == 312
    assert (Y[idx["lab"]].sum(1) >= 1).all(), "labeled rows are never all-zero"
    return D


# ----------------------------------------------------------------------------- metrics
def row_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-row F1 between binary label sets (both non-empty here)."""
    inter = (Ytrue * Ypred).sum(1)
    denom = Ytrue.sum(1) + Ypred.sum(1)
    out = np.where(denom > 0, 2 * inter / np.maximum(denom, 1), 1.0)
    return out


def label_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-label (column) binary F1; F1 = 0 when a label has no positives and none predicted."""
    tp = (Ytrue * Ypred).sum(0)
    fp = ((1 - Ytrue) * Ypred).sum(0)
    fn = (Ytrue * (1 - Ypred)).sum(0)
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)


def metrics(Ytrue: np.ndarray, Ypred: np.ndarray) -> dict:
    rf = row_f1(Ytrue, Ypred)
    lf = label_f1(Ytrue, Ypred)
    return {
        "sample_f1": float(rf.mean()),
        "sample_f1_se": float(rf.std(ddof=1) / np.sqrt(len(rf))),
        "macro_f1": float(lf.mean()),
        **{f"f1_{LABELS[j]}": float(lf[j]) for j in range(5)},
        "n": int(len(rf)),
    }


def bootstrap_se(Ytrue: np.ndarray, Ypreds: list[np.ndarray], B: int = 300, seed: int = 0) -> dict:
    """Bootstrap SE (over rows) of macro and per-label F1, averaged over a list of prediction
    matrices for the same rows (e.g. CV repeats) using shared resamples."""
    rng = np.random.default_rng(seed)
    n = len(Ytrue)
    macro, per = [], []
    for _ in range(B):
        b = rng.integers(0, n, n)
        lfs = np.mean([label_f1(Ytrue[b], P[b]) for P in Ypreds], axis=0)
        per.append(lfs)
        macro.append(lfs.mean())
    per = np.array(per)
    return {"macro_f1_se": float(np.std(macro, ddof=1)),
            **{f"f1_{LABELS[j]}_se": float(per[:, j].std(ddof=1)) for j in range(5)}}


def combos_to_labels(D: Data, kidx: np.ndarray) -> np.ndarray:
    return D.CM[kidx]


def write_submission(D: Data, Ypred: np.ndarray, path: str, ids=None) -> pd.DataFrame:
    """Kaggle format: ID,Predicted with space-separated reason names (always >=1)."""
    assert (Ypred.sum(1) >= 1).all(), "every test row needs at least one predicted reason"
    ids = D.df["ID"].to_numpy()[D.idx["test"]] if ids is None else ids
    pred = [" ".join(LABELS[j] for j in range(5) if row[j]) for row in Ypred]
    sub = pd.DataFrame({"ID": ids, "Predicted": pred})
    sub.to_csv(path, index=False, lineterminator="\n")
    return sub

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/data.py


## part1/experiments/cand2_submission_v1/source/p1lib/evaluate.py

In [60]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/evaluate.py
"""Leak-free evaluation harness.

Protocols (all over the 235 labeled rows = 157 labeled-train + 78 val):
  A : RepeatedStratifiedKFold(5 x R) stratified on the label-combo id         (no shift)
  B : chronological holdout: train start_year < cut, evaluate >= cut         (shift, cut=2005)
  C : official split: train on the 157 labeled-train rows, evaluate on val   (tie-breaker)
Every fold builds a fresh Ctx; everything label-dependent is fit inside `run_model` from
ctx.train labels only.  Label-free fits are reused across folds only when their feature-row
set is identical (transductive mode).

Fold outputs (calibrated combo probabilities BEFORE the head) are cached on disk keyed by the
model part of the config, so SLD strength / decoding are post-hoc and free to vary.
"""
from __future__ import annotations

import hashlib
import json
import os
import time

import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold

from .context import make_ctx
from .data import LABELS, bootstrap_se, label_f1, metrics, row_f1
from .head import apply_head
from .pipeline import model_part, run_model, validate

# fold-output cache lives OUTSIDE the midterm/ folder (regenerable, ~0.5 GB); override with P1_CACHE
CACHE_DIR = os.environ.get("P1_CACHE", os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "..", ".p1_cache", "folds")))
CHRONO_CUT = 2005
CHRONO_CUTS = (2004, 2005, 2006)   # objective uses the mean over these cuts


def cfg_hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:20]


# ----------------------------------------------------------------------------- folds
def make_folds(D, protocols=("A", "B", "C"), R=4, seed=0, rows=None, chrono_cuts=CHRONO_CUTS):
    """rows: restrict to a subset of labeled rows (used by the nested search-overfitting check)."""
    labval = D.idx["labval"] if rows is None else np.asarray(rows)
    y = D.y[labval]
    folds = []
    if "A" in protocols:
        rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=R, random_state=seed)
        for i, (tr, te) in enumerate(rskf.split(np.zeros(len(y)), y)):
            folds.append({"name": f"A{i // 5}_{i % 5}", "protocol": "A", "repeat": i // 5,
                          "train": labval[tr], "eval": labval[te]})
    if "B" in protocols:
        for cut in chrono_cuts:
            yr = D.year[labval]
            folds.append({"name": f"B{cut}", "protocol": "B", "cut": cut,
                          "train": labval[yr < cut], "eval": labval[yr >= cut]})
    if "C" in protocols:
        tr = np.intersect1d(labval, D.idx["lab"])
        te = np.intersect1d(labval, D.idx["val"])
        if len(tr) and len(te):
            folds.append({"name": "C", "protocol": "C", "train": tr, "eval": te})
    return folds


def fold_tag(fold):
    h = hashlib.md5(np.r_[fold["train"], [-1], fold["eval"]].astype(np.int64).tobytes()).hexdigest()[:10]
    return f"{fold['name']}_{h}"


# ----------------------------------------------------------------------------- one fold
def run_fold(cfg, D, fold, use_cache=True):
    mp = model_part(cfg)
    key = cfg_hash(mp)
    path = os.path.join(CACHE_DIR, key, fold_tag(fold) + ".npz")
    if use_cache and os.path.exists(path):
        try:
            z = np.load(path, allow_pickle=False)
            out = {k: z[k] for k in z.files}
            out["T"] = out["T"].tolist()
            return out
        except Exception:
            pass
    tr = cfg.get("transductive", True)
    ctx = make_ctx(D, fold["train"], fold["eval"], transductive=tr, seed=0, name=fold["name"])
    t0 = time.time()
    rows = fold["eval"] if tr else np.r_[fold["eval"], ctx.pool]
    o = run_model(cfg["model"], ctx, rows)
    ne = len(fold["eval"])
    out = {"P_eval": o["P"][:ne].astype(np.float32), "prior": np.asarray(o["prior"], float),
           "T": np.atleast_1d(np.asarray(o["T"], float)), "secs": np.float32(time.time() - t0)}
    if not tr:
        out["P_target"] = o["P"][ne:].astype(np.float32)
    if use_cache:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + f".{os.getpid()}.tmp.npz"
        np.savez_compressed(tmp, **out)
        os.replace(tmp, path)
        mpath = os.path.join(CACHE_DIR, key, "config.json")
        if not os.path.exists(mpath):
            with open(mpath, "w") as f:
                json.dump(mp, f, indent=1, default=str)
    out["T"] = out["T"].tolist()
    return out


def fold_outputs(cfg, D, folds, n_jobs=1, use_cache=True):
    if n_jobs == 1:
        return [run_fold(cfg, D, f, use_cache) for f in folds]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=n_jobs, backend="loky")(delayed(run_fold)(cfg, D, f, use_cache) for f in folds)


# ----------------------------------------------------------------------------- scoring
def score(cfg, D, folds, outs, bootstrap=True):
    """Apply the head to cached fold outputs and compute A/B/C metrics with SEs."""
    head = cfg.get("head", {})
    res = {}
    per = {}
    for f, o in zip(folds, outs):
        P = o["P_eval"].astype(float)
        P = P / P.sum(1, keepdims=True)
        Pt = o.get("P_target")
        Pt = None if Pt is None else Pt.astype(float) / Pt.astype(float).sum(1, keepdims=True)
        Yp, Q, pi = apply_head(P, o["prior"], D.CM, head, Pt)
        per[f["name"]] = (f, Yp, pi)
    # ---- A ----
    A = [v for v in per.values() if v[0]["protocol"] == "A"]
    if A:
        rows = np.unique(np.concatenate([f["eval"] for f, _, _ in A]))
        pos = {r: i for i, r in enumerate(rows)}
        reps = sorted({f["repeat"] for f, _, _ in A})
        Ytrue = D.Y[rows]
        Ypreds = []
        for r in reps:
            Yp = np.zeros((len(rows), 5), int)
            for f, yp, _ in A:
                if f["repeat"] == r:
                    Yp[[pos[i] for i in f["eval"]]] = yp
            Ypreds.append(Yp)
        rep_metrics = [metrics(Ytrue, Yp) for Yp in Ypreds]
        rf = np.mean([row_f1(Ytrue, Yp) for Yp in Ypreds], axis=0)
        a = {k: float(np.mean([m[k] for m in rep_metrics])) for k in rep_metrics[0] if k not in ("n",)}
        a["sample_f1_se"] = float(rf.std(ddof=1) / np.sqrt(len(rf)))
        a["sample_f1_rep_sd"] = float(np.std([m["sample_f1"] for m in rep_metrics], ddof=1)) if len(reps) > 1 else 0.0
        a["n"] = int(len(rows))
        a["R"] = len(reps)
        if bootstrap:
            a.update(bootstrap_se(Ytrue, Ypreds, B=200))
        res["A"] = a
        res["rowf1_A"] = rf.tolist()
        res["rows_A"] = rows.tolist()
    # ---- B ----  (one or several chronological cuts; B = mean over cuts, B_se = mean per-cut SE
    #               because the cuts share most rows and are highly correlated)
    Bs = [v for v in per.values() if v[0]["protocol"] == "B"]
    if Bs:
        bm = []
        for f, yp, pi in Bs:
            Yt = D.Y[f["eval"]]
            m = metrics(Yt, yp)
            if bootstrap:
                m.update(bootstrap_se(Yt, [yp], B=200))
            m["cut"] = f["cut"]
            if pi is not None:
                m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
            m["true_prior_labels"] = Yt.mean(0).round(3).tolist()
            bm.append(m)
            res.setdefault("rowf1_B", []).extend(row_f1(Yt, yp).tolist())
        res["B_ncuts"] = len(bm)
        if len(bm) == 1:
            res["B"] = bm[0]
        else:
            agg = {k: float(np.mean([m[k] for m in bm])) for k in bm[0]
                   if isinstance(bm[0][k], (int, float)) and k != "cut"}
            agg["cuts"] = bm
            res["B"] = agg
        for m in bm:
            if m["cut"] == CHRONO_CUT:
                res["B2005"] = {k: v for k, v in m.items()}
    # ---- C ----
    Cs = [v for v in per.values() if v[0]["protocol"] == "C"]
    if Cs:
        f, yp, pi = Cs[0]
        Yt = D.Y[f["eval"]]
        m = metrics(Yt, yp)
        if bootstrap:
            m.update(bootstrap_se(Yt, [yp], B=200))
        if pi is not None:
            m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
        res["C"] = m
        res["rowf1_C"] = row_f1(Yt, yp).tolist()
    # ---- robust objective ----
    if "A" in res and "B" in res:
        res["J"] = 0.5 * res["A"]["sample_f1"] + 0.5 * res["B"]["sample_f1"]
        res["J_se"] = 0.5 * float(np.hypot(res["A"]["sample_f1_se"], res["B"]["sample_f1_se"]))
    res["secs"] = float(sum(float(o["secs"]) for o in outs))
    res["T_mean"] = float(np.mean([np.mean(o["T"]) for o in outs]))
    return res


def evaluate(cfg, D, protocols=("A", "B", "C"), R=4, n_jobs=1, seed=0, use_cache=True, rows=None,
             bootstrap=True, chrono_cuts=CHRONO_CUTS):
    validate(cfg)
    folds = make_folds(D, protocols, R, seed, rows=rows, chrono_cuts=chrono_cuts)
    outs = fold_outputs(cfg, D, folds, n_jobs, use_cache)
    return score(cfg, D, folds, outs, bootstrap=bootstrap)


def summary_row(name, res):
    r = {"name": name}
    for p in ("A", "B", "B2005", "C"):
        if p in res:
            r[f"{p}"] = res[p]["sample_f1"]
            r[f"{p}_se"] = res[p]["sample_f1_se"]
            r[f"{p}_macro"] = res[p]["macro_f1"]
    if "J" in res:
        r["J"] = res["J"]
        r["J_se"] = res["J_se"]
    r["secs"] = res.get("secs")
    return r


def paired_se(res1, res2, w=(0.5, 0.5)):
    """SE of the difference in J between two configs using per-row paired differences."""
    dA = np.asarray(res1["rowf1_A"]) - np.asarray(res2["rowf1_A"])
    dB = np.asarray(res1["rowf1_B"]) - np.asarray(res2["rowf1_B"])
    nB = len(dB) / res1.get("B_ncuts", 1)          # cuts overlap: effective n ~ one cut
    return float(np.sqrt((w[0] ** 2) * dA.var(ddof=1) / len(dA) + (w[1] ** 2) * dB.var(ddof=1) / nB))

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/evaluate.py


## part1/experiments/cand2_submission_v1/source/p1lib/features.py

In [61]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/features.py
"""Preprocessors: standardize -> [PCA] -> [L2-normalize] on the embeddings, plus optional
non-embedding blocks.  All fitting uses only `ctx.unsup` rows (label-free), so the same
object is refit per fold when the feature-row set changes (inductive) and reused otherwise.

features config (all keys optional):
    {"scale": True, "pca": None|int, "whiten": False, "l2": False,
     "blocks": {"criteria": {"d": 32, "weight": 1.0}, "pathways": {...}, ...}}
"""
from __future__ import annotations

import numpy as np
from sklearn.decomposition import PCA

from .blocks import build_block
from .cache import memo

DEFAULT_FEATURES = {"scale": True, "pca": None, "l2": False, "blocks": {}}


def norm_features_cfg(cfg):
    c = dict(DEFAULT_FEATURES)
    c.update(cfg or {})
    c["blocks"] = dict(c.get("blocks") or {})
    return c


def _embed(D, cfg, rows):
    X = D.emb.copy()
    if cfg["scale"]:
        mu = X[rows].mean(0)
        sd = X[rows].std(0)
        X = (X - mu) / sd
    if cfg["pca"]:
        p = PCA(n_components=int(cfg["pca"]), whiten=bool(cfg.get("whiten", False)), random_state=0).fit(X[rows])
        X = p.transform(X)
    if cfg["l2"]:
        X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    return X


def featurize(ctx, cfg) -> np.ndarray:
    """Return the (n_all, d) representation for every global row."""
    cfg = norm_features_cfg(cfg)
    emb_cfg = {k: cfg[k] for k in ("scale", "pca", "l2", "whiten") if k in cfg}
    X = memo("embed", emb_cfg, ctx.unsup, lambda: _embed(ctx.D, cfg, ctx.unsup))
    if not cfg["blocks"]:
        return X
    parts = [X] if cfg.get("use_emb", True) else []
    for name in sorted(cfg["blocks"]):
        parts.append(build_block(ctx.D, name, cfg["blocks"][name], ctx.unsup))
    return np.hstack(parts)

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/features.py


## part1/experiments/cand2_submission_v1/source/p1lib/graph.py

In [62]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/graph.py
"""Symmetric RBF-weighted kNN graph over the feature rows a fold may use, and
normalized-Laplacian diffusion (Zhou et al. 2004 label spreading; Huang et al. 2021 C&S).

graph config: {"space": features-cfg, "k": 10, "s": 1.0, "sym": "max"|"min"}
Weights w_ij = exp(-d_ij^2 / (s * median kNN distance)^2) on cosine-normalized vectors,
symmetrized with max (union) or min (mutual kNN), S = D^-1/2 W D^-1/2.
Diffusion solves (I - alpha S) F = (1 - alpha) F0 exactly (LU factor cached per alpha).
"""
from __future__ import annotations

import numpy as np
import scipy.sparse as sp
from scipy.linalg import lu_factor, lu_solve
from sklearn.neighbors import NearestNeighbors

from .cache import memo
from .features import featurize

DEFAULT_GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}


class Graph:
    def __init__(self, Z_nodes, nodes, n_all, k, s, sym):
        self.nodes = nodes
        self.pos = np.full(n_all, -1)
        self.pos[nodes] = np.arange(len(nodes))
        self.Zn = Z_nodes / np.maximum(np.linalg.norm(Z_nodes, axis=1, keepdims=True), 1e-12)
        self.nn = NearestNeighbors(n_neighbors=k + 1).fit(self.Zn)
        dist, ind = self.nn.kneighbors(self.Zn)
        dist, ind = dist[:, 1:], ind[:, 1:]
        self.sigma = s * np.median(dist)
        n = len(nodes)
        rows = np.repeat(np.arange(n), k)
        W = sp.csr_matrix((np.exp(-(dist.ravel() ** 2) / self.sigma ** 2), (rows, ind.ravel())), shape=(n, n))
        W = W.maximum(W.T) if sym == "max" else W.minimum(W.T)
        deg = np.asarray(W.sum(1)).ravel()
        iso = deg <= 0
        if iso.any():                      # isolated nodes (possible with mutual kNN): self-loop
            W = W + sp.diags(iso.astype(float))
            deg = np.asarray(W.sum(1)).ravel()
        Dm = sp.diags(1 / np.sqrt(deg))
        self.S = (Dm @ W @ Dm).toarray()
        self.k = k
        self._lu = {}

    def propagate(self, F0, alpha):
        if alpha <= 0:
            return F0.copy()
        if alpha not in self._lu:
            self._lu[alpha] = lu_factor(np.eye(len(self.S)) - alpha * self.S)
        return (1 - alpha) * lu_solve(self._lu[alpha], F0)

    def extend(self, Z_rows, F_nodes):
        """Inductive extension for rows that are not graph nodes: RBF-weighted kNN average."""
        Zn = Z_rows / np.maximum(np.linalg.norm(Z_rows, axis=1, keepdims=True), 1e-12)
        dist, ind = self.nn.kneighbors(Zn, n_neighbors=self.k)
        w = np.exp(-dist ** 2 / self.sigma ** 2) + 1e-12
        return np.einsum("nk,nkc->nc", w, F_nodes[ind]) / w.sum(1, keepdims=True)


def get_graph(ctx, gcfg) -> Graph:
    g = dict(DEFAULT_GRAPH)
    g.update(gcfg or {})
    Z = featurize(ctx, g["space"])

    def build():
        return Graph(Z[ctx.unsup], ctx.unsup, ctx.D.n, int(g["k"]), float(g["s"]), g["sym"])

    return memo("graph", {k: g[k] for k in ("space", "k", "s", "sym")}, ctx.unsup, build), Z


def node_values(graph, Z, F_nodes, rows):
    """Read propagated values for arbitrary global rows (extend for non-nodes)."""
    out = np.zeros((len(rows), F_nodes.shape[1]))
    p = graph.pos[rows]
    m = p >= 0
    out[m] = F_nodes[p[m]]
    if (~m).any():
        out[~m] = graph.extend(Z[rows[~m]], F_nodes)
    return out


def label_spread(ctx, gcfg, alpha, train_rows, y_train, K, out_rows, base_all=None):
    """Pure label spreading (base_all=None) or the Smooth step of C&S (base_all = (n_all,K)
    base probabilities).  Training nodes are clamped to their one-hot labels in F0."""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    F0 = np.zeros((n, K))
    if base_all is not None:
        F0 = base_all[graph.nodes].copy()
    p = graph.pos[train_rows]
    keep = p >= 0
    F0[p[keep]] = np.eye(K)[y_train[keep]]
    F = graph.propagate(F0, alpha)
    return node_values(graph, Z, F, out_rows)


def correct_and_smooth(ctx, gcfg, a1, a2, train_rows, y_train, P_oof_train, P_all, K, out_rows,
                       scale_mode="autoscale"):
    """Correct & Smooth (Huang et al. 2021).
    P_oof_train : base probabilities of the training rows, OUT-OF-FOLD (in-sample residuals ~0)
    P_all       : base probabilities for every global row (fit on all training rows)"""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    p = graph.pos[train_rows]
    keep = p >= 0
    Ytr = np.eye(K)[y_train]
    base = P_all[graph.nodes].copy()
    if a1 > 0:
        E0 = np.zeros((n, K))
        E0[p[keep]] = (Ytr - P_oof_train)[keep]
        E = graph.propagate(E0, a1)
        if scale_mode == "autoscale":
            sigma = np.abs(E0[p[keep]]).sum(1).mean()
            scale = sigma / np.maximum(np.abs(E).sum(1, keepdims=True), 1e-12)
            base = base + scale * E
        else:
            base = base + float(scale_mode) * E
        base = np.clip(base, 0, None)
        base = base / np.maximum(base.sum(1, keepdims=True), 1e-12)
    F0 = base.copy()
    F0[p[keep]] = Ytr[keep]
    F = graph.propagate(F0, a2)
    out = node_values(graph, Z, F, out_rows)
    out = np.clip(out, 0, None) + 1e-9
    return out / out.sum(1, keepdims=True)

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/graph.py


## part1/experiments/cand2_submission_v1/source/p1lib/head.py

In [63]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/head.py
"""Shared head: temperature calibration, ONE prior correction (SLD-EM, MAP-stabilized),
and threshold-free decoding to an allowed label set with >= 1 label.

All probabilities are over the K allowed label combinations (label powerset).
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax

EPS = 1e-6


def safe_log(P):
    return np.log(np.clip(P, EPS, None))


def renorm(P):
    P = np.clip(P, 0, None) + 1e-12
    return P / P.sum(1, keepdims=True)


# ----------------------------------------------------------------------------- temperature
def fit_temperature(P_oof: np.ndarray, y: np.ndarray, w: np.ndarray | None = None,
                    bounds=(0.05, 20.0)) -> float:
    """One-parameter temperature T minimizing NLL of softmax(log P / T) on OOF probabilities."""
    L = safe_log(P_oof)
    w = np.ones(len(y)) if w is None else w

    def nll(logT):
        T = np.exp(logT)
        lp = log_softmax(L / T, axis=1)
        return -(w * lp[np.arange(len(y)), y]).sum() / w.sum()

    r = minimize_scalar(nll, bounds=(np.log(bounds[0]), np.log(bounds[1])), method="bounded",
                        options={"xatol": 1e-4})
    return float(np.exp(r.x))


def apply_temperature(P: np.ndarray, T: float) -> np.ndarray:
    if T is None or T == 1.0:
        return P
    return np.exp(log_softmax(safe_log(P) / T, axis=1))


# ----------------------------------------------------------------------------- SLD-EM
def sld_em(P: np.ndarray, prior_src: np.ndarray, a: float | None = 0.0, floor: float = 0.25,
           tol: float = 1e-8, max_iter: int = 1000, cap: float | None = None):
    """Saerens-Latinne-Decaestecker EM for the target label prior, with a Dirichlet(a*pi_s)
    pseudo-count toward the source prior (MAP) and a floor pi >= floor*pi_s.

    cap: optional ceiling pi_k <= cap * pi_s,k (guards the degenerate MLE in which a rare
    'bridge' combo such as Safety+Efficacy absorbs the mass of its parents).
    Returns (Q, pi_hat): prior-corrected posteriors and the estimated target prior.
    a=None or a=inf means 'off' (returns P unchanged)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6            # combos absent from training stay at exactly 0
    ps = np.where(present, prior_src, 0.0)
    ps = ps / ps.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ratio_den = np.where(present, ps, 1.0)
    pi = ps.copy()
    n = len(P)
    for _ in range(max_iter):
        Q = P * (pi / ratio_den)
        Q /= Q.sum(1, keepdims=True)
        new = (Q.sum(0) + a * ps) / (n + a)
        new = np.maximum(new, floor * ps)
        if cap:
            new = np.minimum(new, cap * ps)
        new /= new.sum()
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * (pi / ratio_den)
    Q /= Q.sum(1, keepdims=True)
    return Q, pi


def sld_em_factorized(P: np.ndarray, prior_src: np.ndarray, CM: np.ndarray, a: float | None = 0.0,
                      floor: float = 0.25, tol: float = 1e-8, max_iter: int = 1000):
    """Factorized SLD: the target/source combo-prior ratio is constrained to a product of
    per-label ratios, w_k = prod_j (pi_j/ps_j)^{CM_kj} ((1-pi_j)/(1-ps_j))^{1-CM_kj}.
    Only 5 free parameters (one per reason) instead of K-1, so a rare 'bridge' combo
    (Safety+Efficacy) cannot absorb the mass of its parents.  EM-style fixed point: E-step
    reweights combos by w, M-step sets each label prior to the mean marginal of Q (with the same
    Dirichlet pseudo-count a toward the source marginal and floor)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6
    ps_k = np.where(present, prior_src, 0.0)
    ps_k = ps_k / ps_k.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ps = np.clip(ps_k @ CM, 1e-6, 1 - 1e-6)            # source label marginals
    pi = ps.copy()
    n = len(P)

    def weights(pi):
        lw = np.log(pi / ps) @ CM.T + np.log((1 - pi) / (1 - ps)) @ (1 - CM).T
        return np.exp(lw - lw.max())

    for _ in range(max_iter):
        Q = P * weights(pi)
        Q /= Q.sum(1, keepdims=True)
        m = Q @ CM                                          # (n, 5) label marginals
        new = (m.sum(0) + a * ps) / (n + a)
        new = np.clip(np.maximum(new, floor * ps), 1e-6, 1 - 1e-6)
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * weights(pi)
    Q /= Q.sum(1, keepdims=True)
    pi_k = ps_k * weights(pi)
    return Q, pi_k / pi_k.sum()


# ----------------------------------------------------------------------------- decoding
_SUBSETS = np.array([[(s >> j) & 1 for j in range(5)] for s in range(1, 32)])


def _f1_matrix(CM):
    inter = _SUBSETS @ CM.T                                   # (31, K)
    return 2 * inter / (_SUBSETS.sum(1)[:, None] + CM.sum(1)[None, :])


def decode(P: np.ndarray, CM: np.ndarray, method: str = "map") -> np.ndarray:
    """Map combo posteriors to a binary label matrix (n, 5); never empty.
    map   : the most probable allowed combination
    expf1 : Bayes-optimal subset for expected sample-F1 over the 31 non-empty subsets"""
    if method == "map":
        return CM[P.argmax(1)]
    if method == "expf1":
        F = _f1_matrix(CM)
        return _SUBSETS[(P @ F.T).argmax(1)]
    raise ValueError(method)


def apply_head(P_eval, prior_src, CM, head: dict, P_target=None):
    """SLD (optional) + decoding.  P_target: posteriors of the SLD target rows (defaults to the
    eval rows, i.e. transductive).  Returns (Ypred, Q_eval, pi_hat)."""
    head = head or {}
    pi_hat = None
    Q = P_eval
    if head.get("prior", "none") == "sld":
        a = head.get("sld_a", 0.0)
        floor = head.get("sld_floor", 0.25)
        cap = head.get("sld_cap")
        if head.get("sld_mode", "combo") == "factorized":
            em = lambda P_: sld_em_factorized(P_, prior_src, CM, a=a, floor=floor)
        else:
            em = lambda P_: sld_em(P_, prior_src, a=a, floor=floor, cap=cap)
        if P_target is None or P_target is P_eval:
            Q, pi_hat = em(P_eval)
        else:
            _, pi_hat = em(P_target)
            present = prior_src > 1e-6
            Q = renorm(np.where(present, P_eval * (pi_hat / np.where(present, prior_src, 1.0)), 0.0))
    return decode(Q, CM, head.get("decode", "map")), Q, pi_hat

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/head.py


## part1/experiments/cand2_submission_v1/source/p1lib/labelers.py

In [64]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/labelers.py
"""Labelers: fill the missing labels of the unlabeled pool with soft combo distributions.

Every labeler type exposes
    soft_labels(lcfg, ctx, rows) -> Q (len(rows), K)      for rows NOT in ctx.train
and, where cheap and exact, a leave-one-out version for training rows (used when the labeler's
output is a synthetic feature or a direct predictor that needs OOF scores):
    soft_labels_train_oof(lcfg, ctx) -> Q (len(ctx.train), K)

`pseudo_labels(lcfg, ctx, clf_cfg)` turns soft labels into weighted pseudo-rows
(rows, Q, w) for the pipeline's classifier (soft-label replication, weight lam * q_ik).

Types
  cluster   : cluster-then-label (kmeans | gmm | seeded_kmeans | hdbscan), m-smoothed map   [T4]
  spread    : label spreading on the kNN graph (Zhou et al. 2004)                            [T2]
  cs        : Correct & Smooth on the kNN graph (Huang et al. 2021)                          [T2]
  sld       : calibrated base classifier + SLD-EM on the pool (Saerens et al. 2002)          [T1]
  selftrain : guarded self-training (agreement, alignment, soft weights, ramp)               [T3]
"""
from __future__ import annotations

import numpy as np
from sklearn.cluster import HDBSCAN, KMeans, kmeans_plusplus
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold

from .cache import memo, memo_ctx, rows_key
from .classifiers import make_classifier
from .features import featurize
from .graph import correct_and_smooth, label_spread
from .head import apply_temperature, fit_temperature, sld_em, sld_em_factorized

DEFAULT_CLUSTER_SPACE = {"scale": True, "pca": 32, "l2": True}


# ============================================================================ clustering
def _cluster_fit(D, algo, k, Z, rows, cfg, seed=0):
    info = {}
    if algo == "kmeans":
        km = KMeans(n_clusters=k, n_init=int(cfg.get("n_init", 10)), max_iter=300, tol=1e-4,
                    random_state=seed).fit(Z[rows])
        dist = km.transform(Z)
        info.update(inertia=float(km.inertia_), n_iter=int(km.n_iter_))
        return {"dist": dist, "labels": km.predict(Z), "info": info}
    if algo == "gmm":
        g = GaussianMixture(n_components=k, covariance_type=cfg.get("cov", "diag"), reg_covar=1e-3,
                            n_init=int(cfg.get("n_init", 3)), max_iter=500, tol=1e-4,
                            init_params="k-means++", random_state=seed).fit(Z[rows])
        info.update(converged=bool(g.converged_), n_iter=int(g.n_iter_), bic=float(g.bic(Z[rows])),
                    lower_bound=float(g.lower_bound_))
        return {"resp": g.predict_proba(Z), "labels": g.predict(Z), "info": info}
    if algo == "hdbscan":
        h = HDBSCAN(min_cluster_size=int(cfg.get("min_cluster_size", 10)),
                    min_samples=cfg.get("min_samples")).fit(Z[rows])
        lab = np.full(D.n, -1)
        lab[rows] = h.labels_
        others = np.setdiff1d(np.arange(D.n), rows)
        if len(others):                     # inductive: attach to the nearest fitted point
            Zr = Z[rows]
            nn = np.argmax(Z[others] @ Zr.T / (np.linalg.norm(Z[others], axis=1)[:, None] *
                                                np.linalg.norm(Zr, axis=1)[None] + 1e-12), axis=1)
            lab[others] = h.labels_[nn]
        info.update(n_clusters=int(h.labels_.max() + 1), noise_frac=float((h.labels_ < 0).mean()))
        return {"labels": lab, "info": info}
    raise ValueError(algo)


def cluster_resp(lcfg, ctx):
    """(n_all, C) cluster membership (hard one-hot, kmeans-soft, or GMM responsibilities)."""
    algo = lcfg.get("algo", "kmeans")
    k = int(lcfg.get("k", 24))
    space = lcfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    if algo == "seeded_kmeans":                 # label-dependent init -> not shared across folds
        y = ctx.y_train
        cls = np.unique(y)
        seeds = np.array([Z[ctx.train][y == c].mean(0) for c in cls])
        extra = max(k - len(cls), 0)
        if extra:
            more, _ = kmeans_plusplus(Z[ctx.unsup], n_clusters=extra, random_state=ctx.seed)
            seeds = np.vstack([seeds, more])
        km = KMeans(n_clusters=len(seeds), init=seeds, n_init=1, max_iter=300).fit(Z[ctx.unsup])
        lab = km.predict(Z)
        return np.eye(len(seeds))[lab]
    cfg_key = {"algo": algo, "k": k, "space": space, **{kk: lcfg[kk] for kk in ("cov", "n_init", "min_cluster_size", "min_samples") if kk in lcfg}}
    fit = memo("cluster", cfg_key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, lcfg))
    if algo == "gmm":
        return fit["resp"]
    if algo == "kmeans" and lcfg.get("soft_temp"):
        d2 = fit["dist"] ** 2
        T = float(lcfg["soft_temp"]) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    lab = fit["labels"]
    C = int(max(lab.max() + 1, 1))
    R = np.zeros((ctx.D.n, C))
    R[lab >= 0, lab[lab >= 0]] = 1.0          # hdbscan noise rows keep all-zero membership
    return R


def _cluster_map(R_train, y, K, m, prior):
    N = R_train.T @ np.eye(K)[y]                    # (C, K) label mass per cluster
    return N, (N + m * prior) / (N.sum(1, keepdims=True) + m)


def cluster_soft(lcfg, ctx, rows):
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)
    y = ctx.y_train
    prior = np.bincount(y, minlength=K) / len(y)
    _, pi = _cluster_map(R[ctx.train], y, K, float(lcfg.get("m", 4)), prior)
    Q = R[rows] @ pi
    mass = R[rows].sum(1, keepdims=True)
    Q = Q + (1 - np.clip(mass, 0, 1)) * prior        # noise / empty -> global prior
    return Q / Q.sum(1, keepdims=True)


def cluster_soft_train_oof(lcfg, ctx):
    """Exact leave-one-out cluster map for training rows."""
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)[ctx.train]
    y = ctx.y_train
    m = float(lcfg.get("m", 4))
    prior = np.bincount(y, minlength=K) / len(y)
    N, _ = _cluster_map(R, y, K, m, prior)
    nc = N.sum(1)                                    # (C,)
    out = np.zeros((len(y), K))
    for i in range(len(y)):
        Ni = N - np.outer(R[i], np.eye(K)[y[i]])
        pi = (Ni + m * prior) / (nc - R[i] + m)[:, None]
        q = R[i] @ pi
        q = q + (1 - min(R[i].sum(), 1)) * prior
        out[i] = q / q.sum()
    return out


# ============================================================================ graph labelers
def spread_soft(lcfg, ctx, rows):
    K = ctx.D.K
    F = label_spread(ctx, lcfg.get("graph"), float(lcfg.get("alpha", 0.5)), ctx.train, ctx.y_train, K, rows)
    prior = np.bincount(ctx.y_train, minlength=K) / len(ctx.train)
    F = np.clip(F, 0, None)
    s = F.sum(1, keepdims=True)
    F = np.where(s > 1e-12, F / np.maximum(s, 1e-12), prior)   # unreachable rows -> prior
    beta = float(lcfg.get("beta", 0.01))
    F = (F + beta * prior) / (1 + beta)
    return F


def _base_oof_and_all(base_cfg, ctx, X, n_inner=5):
    """Base classifier probabilities: OOF for training rows, full-fit for every other row."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    P_oof = np.zeros((len(y), K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
        P_oof[te] = m.predict_proba(X[ctx.train[te]])
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P_all = m.predict_proba(X)
    return P_oof, P_all


def cs_soft(lcfg, ctx, rows):
    K = ctx.D.K
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    P_oof, P_all = _base_oof_and_all(base_cfg, ctx, X)
    return correct_and_smooth(ctx, lcfg.get("graph"), float(lcfg.get("a1", 0.5)), float(lcfg.get("a2", 0.5)),
                              ctx.train, ctx.y_train, P_oof, P_all, K, rows)


# ============================================================================ SLD labeler
def calibrated_base(base_cfg, ctx, X, rows, temperature=True):
    """Fit base classifier on ctx.train, temperature-scale with inner OOF, predict rows."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P = m.predict_proba(X[rows])
    T = 1.0
    if temperature:
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            mm = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
            P_oof[te] = mm.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    return P, m.prior_, T


def sld_soft(lcfg, ctx, rows):
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    P, prior, _ = calibrated_base(base_cfg, ctx, X, rows, lcfg.get("temperature", False))
    if lcfg.get("mode", "combo") == "factorized":
        Q, pi = sld_em_factorized(P, prior, ctx.D.CM, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    else:
        Q, pi = sld_em(P, prior, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    return Q


# ============================================================================ self-training
def _sinkhorn(P, target, iters=200):
    Q = P.copy()
    n = len(Q)
    for _ in range(iters):
        col = Q.sum(0)
        Q = Q * (target * n / np.maximum(col, 1e-12))
        Q = Q / Q.sum(1, keepdims=True)
    return Q


def _softmatch_weight(conf):
    mu, sd = conf.mean(), conf.std() + 1e-6
    return np.where(conf >= mu, 1.0, np.exp(-((conf - mu) ** 2) / (2 * sd ** 2)))


def selftrain_pseudo(lcfg, ctx, clf_cfg, X, pool_rows):
    """Guarded self-training.  Returns (rows, Q, w, info) for the final pseudo-labeled set."""
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    Ytr = np.eye(K)[y]
    R = int(lcfg.get("rounds", 2))
    lam_max = float(lcfg.get("lam", 0.3))
    members = lcfg.get("members", ["lr"])
    align = lcfg.get("align", "none")
    tau = lcfg.get("tau", 0.75)
    top_p = lcfg.get("top_p")
    hard = bool(lcfg.get("hard", False))
    weighting = lcfg.get("weighting", "hard")
    # members that do not use pseudo-labels are computed once
    static = []
    if "lda" in members:
        static.append(make_classifier({"type": "lda"}, K, CM).fit(X[ctx.train], y).predict_proba(X[pool_rows]))
    if "cs" in members:
        static.append(cs_soft({"a1": 0.5, "a2": 0.5}, ctx, pool_rows))
    cl_major = None
    if lcfg.get("cluster_filter"):
        cl_major = cluster_soft({"algo": "kmeans", "k": int(lcfg.get("cluster_k", 24)), "m": 1}, ctx, pool_rows).argmax(1)
    sel_rows, Q, w = pool_rows[:0], np.zeros((0, K)), np.zeros(0)
    prev_lab = None
    info = {}
    for r in range(1, R + 1):
        lam_r = lam_max * (r / R if lcfg.get("ramp", False) else 1.0)
        Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[sel_rows], Q, w)
        main = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
        Pm_list = [main.predict_proba(X[pool_rows])] + static
        Pm = np.mean(Pm_list, axis=0)
        prior_src = np.bincount(y, minlength=K) / len(y)
        if align == "sld":
            Pm, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
        elif align == "sinkhorn":
            _, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
            Pm = _sinkhorn(Pm, pi)
        conf = Pm.max(1)
        lab = Pm.argmax(1)
        if top_p:
            keep = np.zeros(len(lab), bool)
            for c in range(K):
                mc = np.where(lab == c)[0]
                if len(mc):
                    thr = np.quantile(conf[mc], 1 - float(top_p))
                    keep[mc[conf[mc] >= thr]] = True
        else:
            keep = conf >= float(tau)
        if len(Pm_list) > 1:
            agree = np.sum([P.argmax(1) == lab for P in Pm_list], axis=0)
            keep &= agree >= 2
        if cl_major is not None:
            keep &= cl_major == lab
        g = _softmatch_weight(conf) if weighting == "softmatch" else np.ones(len(conf))
        sel_rows = pool_rows[keep]
        Q = np.eye(K)[lab[keep]] if hard else Pm[keep]
        w = lam_r * g[keep]
        info = {"round": r, "n_selected": int(keep.sum()), "lam": lam_r}
        if prev_lab is not None and np.mean(prev_lab != lab) < 0.01:
            break
        prev_lab = lab
    return sel_rows, Q, w, info


# ============================================================================ dispatch
SOFT = {"cluster": cluster_soft, "spread": spread_soft, "cs": cs_soft, "sld": sld_soft}


_POST = ("lam", "tau", "hard", "pool")      # applied after soft labels -> not part of the memo key


def soft_labels(lcfg, ctx, rows):
    rows = np.asarray(rows)
    core = {k: v for k, v in lcfg.items() if k not in _POST}
    return memo_ctx("soft", core, ctx, lambda: SOFT[lcfg["type"]](lcfg, ctx, rows), extra=rows.tolist())


def soft_labels_train_oof(lcfg, ctx, n_inner=5):
    """OOF soft labels for the training rows (exact LOO for cluster; inner K-fold otherwise)."""
    if lcfg["type"] == "cluster":
        core = {k: v for k, v in lcfg.items() if k not in _POST}
        return memo_ctx("soft_oof", core, ctx, lambda: cluster_soft_train_oof(lcfg, ctx))
    from .pipeline import inner_folds
    y = ctx.y_train
    out = np.zeros((len(y), ctx.D.K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        sub = ctx.sub(ctx.train[tr], ctx.train[te])
        out[te] = soft_labels(lcfg, sub, ctx.train[te])
    return out


def pool_rows_for(lcfg, ctx):
    if lcfg.get("pool", "unl") == "all" and ctx.transductive:
        rows = np.union1d(np.union1d(ctx.pool, ctx.eval), ctx.D.idx["test"])
        return np.setdiff1d(rows, ctx.train)
    return np.setdiff1d(ctx.pool, ctx.train)


def pseudo_labels(lcfg, ctx, clf_cfg, X):
    """Weighted pseudo-rows (rows, Q, w, info) for the pipeline classifier."""
    rows = pool_rows_for(lcfg, ctx)
    if lcfg["type"] == "selftrain":
        return selftrain_pseudo(lcfg, ctx, clf_cfg, X, rows)
    Q = soft_labels(lcfg, ctx, rows)
    conf = Q.max(1)
    keep = conf >= float(lcfg.get("tau", 0.0))
    if lcfg.get("hard", False):
        Q = np.eye(ctx.D.K)[Q.argmax(1)]
    lam = float(lcfg.get("lam", 0.1))
    return rows[keep], Q[keep], np.full(keep.sum(), lam), {"n_selected": int(keep.sum())}


def expand(X_l, Y_l, w_l, X_u, Q_u, w_u, min_q=0.05):
    """Soft-label replication: one copy of each row per combo with weight w * q (exact
    expected cross-entropy for LR).  Y_l may be one-hot or soft."""
    Xs, ys, ws = [], [], []
    for Xb, Yb, wb in ((X_l, Y_l, w_l), (X_u, Q_u, w_u)):
        if len(Xb) == 0:
            continue
        i, k = np.where(Yb >= min_q)
        Xs.append(Xb[i])
        ys.append(k)
        ws.append(wb[i] * Yb[i, k])
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ws)

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/labelers.py


## part1/experiments/cand2_submission_v1/source/p1lib/pipeline.py

In [65]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/pipeline.py
"""Pipeline composition from config dicts.

A *single* pipeline config:
    {
      "features":   {"scale": True, "pca": 64, "l2": False, "blocks": {...}} | None,
      "synth":      [ {"type": "gmm", "k": 16}, {"type": "knn_vote", "k": 10}, ... ],
      "labeler":    {"type": "cluster"|"spread"|"cs"|"sld"|"selftrain", "lam": 0.1, ...} | None,
      "clf":        {"type": "lr"|"ovr"|"lda"|"knn"|"hgb"|"nb"|"direct", ...},
      "temperature": True,
    }
An *ensemble*:  {"ensemble": [single, single, ...], "weights": None | [w1, w2, ...]}
A *stack* is a single pipeline whose synth list holds {"type": "oof_model", "model": single}
entries (nested OOF meta-features) and whose features are None (or a small base block).

A full experiment config adds the head and the transductive toggle:
    {"model": <single|ensemble>, "head": {"prior": "none"|"sld", "sld_a": 100,
     "decode": "map"|"expf1"}, "transductive": True}

run_model(model_cfg, ctx, rows) -> {"P": calibrated combo probs for rows, "prior": pi_s, ...}
Only labels of ctx.train are ever read.
"""
from __future__ import annotations

import copy
import warnings

import numpy as np
from sklearn.model_selection import StratifiedKFold

from .classifiers import make_classifier
from .features import featurize
from .head import apply_temperature, fit_temperature


def _mask_absent(P, prior):
    """Combos with zero training mass keep probability exactly 0 (temperature/log clipping
    would otherwise resurrect them and SLD would then amplify them)."""
    present = np.asarray(prior) > 1e-6
    if present.all():
        return P
    P = np.where(present[None, :], P, 0.0)
    return P / np.maximum(P.sum(1, keepdims=True), 1e-300)
from .labelers import expand, pseudo_labels, soft_labels, soft_labels_train_oof
from .synth import build_synth_block

warnings.filterwarnings("ignore", message="The least populated class")


def inner_folds(y, n_splits=5, seed=0):
    n_splits = int(min(n_splits, max(2, np.bincount(y).max())))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(y)), y))


def _design(cfg, ctx):
    """Base features + synthetic features for every global row."""
    parts = []
    if cfg.get("features") is not None:
        parts.append(featurize(ctx, cfg["features"]))
    S = build_synth_block(cfg.get("synth"), ctx)
    if S is not None:
        parts.append(S)
    if not parts:
        raise ValueError("pipeline has neither base features nor synthetic features")
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def run_single(cfg, ctx, rows):
    rows = np.asarray(rows)
    D = ctx.D
    K, CM = D.K, D.CM
    y = ctx.y_train
    clf_cfg = cfg["clf"]
    info = {}

    # ---- direct labeler-as-predictor (cluster / spread / cs / sld) ----
    if clf_cfg["type"] == "direct":
        lcfg = clf_cfg["labeler"]
        P = soft_labels(lcfg, ctx, rows)
        prior = np.bincount(y, minlength=K) / len(y)
        T = 1.0
        if cfg.get("temperature", True):
            P_oof = soft_labels_train_oof(lcfg, ctx)
            T = fit_temperature(P_oof, y)
            P = apply_temperature(P, T)
        return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}

    X = _design(cfg, ctx)
    Ytr = np.eye(K)[y]
    # ---- labeler -> weighted pseudo rows from the unlabeled pool ----
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    if cfg.get("labeler"):
        pr, Q, w, linfo = pseudo_labels(cfg["labeler"], ctx, clf_cfg, X)
        info["labeler"] = linfo
    Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[pr], Q, w)
    clf = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
    P = clf.predict_proba(X[rows])
    prior = clf.prior_
    T = 1.0
    if cfg.get("temperature", True):
        # classifier-stage OOF: pseudo rows fixed, inner folds over the labeled rows
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            Xi, yi, wi = expand(X[ctx.train[tr]], Ytr[tr], np.ones(len(tr)), X[pr], Q, w)
            m = make_classifier(clf_cfg, K, CM).fit(Xi, yi, wi)
            P_oof[te] = m.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    info["n_pseudo_rows"] = int(len(pr))
    return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}


def run_model(model_cfg, ctx, rows):
    if "ensemble" in model_cfg:
        members = model_cfg["ensemble"]
        wts = np.asarray(model_cfg.get("weights") or [1.0] * len(members), float)
        wts = wts / wts.sum()
        outs = [run_single(m, ctx, rows) for m in members]
        P = sum(wi * o["P"] for wi, o in zip(wts, outs))
        prior = sum(wi * o["prior"] for wi, o in zip(wts, outs))
        return {"P": P, "prior": prior, "T": [o["T"] for o in outs], "info": {"members": [o["info"] for o in outs]},
                "member_P": [o["P"] for o in outs], "member_prior": [o["prior"] for o in outs]}
    return run_single(model_cfg, ctx, rows)


# ----------------------------------------------------------------------------- validation
def final_stage_singles(model_cfg):
    return model_cfg["ensemble"] if "ensemble" in model_cfg else [model_cfg]


def validate(cfg):
    """Enforce the 'only ONE prior correction' rule and basic sanity."""
    model = cfg["model"]
    head = cfg.get("head", {})
    corrections = set()
    for s in final_stage_singles(model):
        clf = s["clf"]
        if clf.get("class_weight") == "balanced":
            corrections.add("class_weight=balanced")
        lab = s.get("labeler") or {}
        if lab.get("type") == "selftrain" and lab.get("align", "none") != "none" and clf.get("class_weight") == "balanced":
            raise ValueError("self-training alignment + balanced class weights = double prior correction")
    if head.get("prior", "none") == "sld":
        corrections.add("sld")
    if len(corrections) > 1:
        raise ValueError(f"more than one prior correction: {sorted(corrections)}")
    return sorted(corrections)


def model_part(cfg):
    """Config without the post-hoc head (SLD / decoding) -> cache key for fold outputs."""
    return {"model": cfg["model"], "transductive": cfg.get("transductive", True)}


def deep(cfg):
    return copy.deepcopy(cfg)

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/pipeline.py


## part1/experiments/cand2_submission_v1/source/p1lib/search.py

In [66]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/search.py
"""Optuna TPE search over the composed config space, robust objective, 1-SE selection and the
nested search-overfitting check.

Objective  J = 0.5 * A + 0.5 * B   (A = repeated stratified CV sample-F1, B = mean chronological
holdout sample-F1 over cuts 2004/2005/2006).  C (official val) is logged, never optimized.
Selection  = one-standard-error rule on J, then the SIMPLEST config (fewest active components),
then a guard: B must not be worse than the plain-LR baseline's B by more than one SE.
"""
from __future__ import annotations

import copy
import fcntl
import json
import os
import time
import traceback

import numpy as np

from .evaluate import evaluate, make_folds, fold_outputs, score, CHRONO_CUTS
from .pipeline import validate

GRAPH_SPACE = {"scale": True, "pca": 64, "l2": True}


# ============================================================================ search space
def suggest_config(t, allow_blocks=True, allow_stack=False, require_unsup=False):
    """Define-by-run space.  Returns a full experiment config."""
    pca = t.suggest_categorical("pca", [0, 32, 64, 96])
    l2 = t.suggest_categorical("l2", [False, True])
    feats = {"scale": True, "pca": pca or None, "l2": l2, "blocks": {}}
    if allow_blocks:
        if t.suggest_categorical("blk_criteria", [False, True]):
            feats["blocks"]["criteria"] = {"d": 16}
        if t.suggest_categorical("blk_drug", [False, True]):
            feats["blocks"]["pathways"] = {"d": 16, "max_level": 2}
            feats["blocks"]["drugmeta"] = {}

    prior = t.suggest_categorical("prior", ["none", "sld"])
    head = {"prior": prior, "decode": t.suggest_categorical("decode", ["map", "expf1"])}
    if prior == "sld":
        head["sld_a"] = t.suggest_categorical("sld_a", [0, 25, 100, 400])
        head["sld_floor"] = 0.25

    clf_t = t.suggest_categorical("clf", ["lr", "ovr", "lda", "knn"] if require_unsup else ["lr", "ovr", "lda", "knn", "cs"])
    temperature = t.suggest_categorical("temperature", [False, True])
    if clf_t == "cs":       # Correct & Smooth as the predictor (T2); base = powerset LR
        gk = t.suggest_categorical("cs_graph_k", [5, 10, 20])
        lcfg = {"type": "cs", "graph": {"space": GRAPH_SPACE, "k": gk, "s": 1.0, "sym": "max"},
                "a1": t.suggest_categorical("cs_a1", [0.0, 0.5, 0.8]),
                "a2": t.suggest_float("cs_a2", 0.1, 0.9),
                "base": {"type": "lr", "C": t.suggest_float("C", 0.01, 3.0, log=True)},
                "base_features": {"scale": True, "pca": pca or None}}
        model = {"features": None, "synth": [], "labeler": None,
                 "clf": {"type": "direct", "labeler": lcfg}, "temperature": temperature}
        return {"model": model, "head": head, "transductive": True}

    clf = {"type": clf_t}
    if clf_t in ("lr", "ovr"):
        clf["C"] = t.suggest_float("C", 0.01, 3.0, log=True)
    if clf_t == "knn":
        clf.update(k=t.suggest_categorical("knn_k", [3, 5, 10, 20]), weights="distance",
                   tau=t.suggest_categorical("knn_tau", [0.05, 0.1, 0.3]), beta=1.0)
    if prior != "sld" and clf_t in ("lr", "ovr", "lda"):
        if t.suggest_categorical("class_weight", ["none", "balanced"]) == "balanced":
            clf["class_weight"] = "balanced"

    synth = []
    if t.suggest_categorical("syn_gmm", [False, True]):
        synth.append({"type": "gmm", "k": t.suggest_categorical("syn_gmm_k", [8, 16, 32]), "cov": "diag",
                      "space": {"scale": True, "pca": 32, "l2": True}})
    if t.suggest_categorical("syn_kmeans", [False, True]):
        synth.append({"type": "kmeans", "k": t.suggest_categorical("syn_kmeans_k", [8, 16, 32]),
                      "out": t.suggest_categorical("syn_kmeans_out", ["dist", "soft"])})
    if t.suggest_categorical("syn_knn", [False, True]):
        synth.append({"type": "knn_vote", "k": t.suggest_categorical("syn_knn_k", [5, 10, 20])})
    if t.suggest_categorical("syn_graph", [False, True]):
        synth.append({"type": "graph", "mode": t.suggest_categorical("syn_graph_mode", ["spread", "cs"]),
                      "graph": {"space": GRAPH_SPACE, "k": 10, "s": 1.0, "sym": "max"},
                      "alpha": t.suggest_float("syn_graph_alpha", 0.2, 0.9)})
    if t.suggest_categorical("syn_clmap", [False, True]):
        synth.append({"type": "cluster_map", "algo": "kmeans", "k": t.suggest_categorical("syn_clmap_k", [8, 16, 32]), "m": 4})
    if t.suggest_categorical("syn_oof", [False, True]):
        synth.append({"type": "oof_model", "model": {"features": {"scale": True}, "clf": {"type": "lda"}, "temperature": True}})

    lab_t = t.suggest_categorical("labeler", ["cluster", "spread", "cs", "selftrain"] if require_unsup else
                                  ["none", "cluster", "spread", "cs", "sld", "selftrain"])
    labeler = None
    if lab_t != "none":
        lam = t.suggest_float("lam", 0.03, 0.5, log=True)
        if lab_t == "cluster":
            labeler = {"type": "cluster", "algo": t.suggest_categorical("cl_algo", ["kmeans", "gmm", "seeded_kmeans"]),
                       "k": t.suggest_categorical("cl_k", [8, 16, 24, 32, 48]), "m": t.suggest_categorical("cl_m", [1, 4]),
                       "lam": lam}
        elif lab_t in ("spread", "cs"):
            g = {"space": GRAPH_SPACE, "k": t.suggest_categorical("lab_graph_k", [5, 10, 20]), "s": 1.0, "sym": "max"}
            labeler = {"type": lab_t, "graph": g, "lam": lam}
            if lab_t == "spread":
                labeler["alpha"] = t.suggest_float("lab_alpha", 0.2, 0.9)
            else:
                labeler.update(a1=t.suggest_categorical("lab_a1", [0.0, 0.5, 0.8]), a2=t.suggest_float("lab_a2", 0.2, 0.9))
        elif lab_t == "sld":
            labeler = {"type": "sld", "base": {"type": "lr", "C": 0.1}, "a": t.suggest_categorical("lab_sld_a", [25, 100, 400]),
                       "lam": lam}
        elif lab_t == "selftrain":
            labeler = {"type": "selftrain", "lam": lam,
                       "members": t.suggest_categorical("st_members", ["lr", "lr+lda+cs"]).split("+"),
                       "rounds": t.suggest_categorical("st_rounds", [1, 2, 3]),
                       "tau": t.suggest_categorical("st_tau", [0.6, 0.75, 0.9]),
                       "align": t.suggest_categorical("st_align", ["none", "sld"]) if clf.get("class_weight") != "balanced" else "none",
                       "weighting": t.suggest_categorical("st_weighting", ["hard", "softmatch"]),
                       "ramp": t.suggest_categorical("st_ramp", [False, True]),
                       "cluster_filter": t.suggest_categorical("st_cluster_filter", [False, True])}
            if require_unsup and "cs" not in labeler["members"]:
                labeler["cluster_filter"] = True
        if lab_t != "selftrain" and t.suggest_categorical("lab_hard", [False, True]):
            labeler["hard"] = True
            labeler["tau"] = t.suggest_categorical("lab_tau", [0.5, 0.7, 0.9])
    model = {"features": feats, "synth": synth, "labeler": labeler, "clf": clf, "temperature": temperature}
    return {"model": model, "head": head, "transductive": True}


# ============================================================================ complexity
CLF_COST = {"lr": 0.0, "lda": 0.0, "ovr": 0.25, "nb": 0.25, "knn": 0.5, "direct": 1.0, "hgb": 2.0}


def complexity(cfg) -> float:
    """Number of active components (+ small costs for extra knobs).  Lower = simpler."""
    m = cfg["model"]
    singles = m["ensemble"] if "ensemble" in m else [m]
    c = 0.0
    for s in singles:
        c += len(s.get("synth") or [])
        for sy in s.get("synth") or []:
            if sy["type"] == "oof_model":
                c += complexity({"model": sy["model"], "head": {}})
        lab = s.get("labeler")
        if lab:
            c += 1.0 + (0.5 * (len(lab.get("members", [])) > 1) + 0.25 * (lab.get("rounds", 1) - 1)
                        if lab["type"] == "selftrain" else 0.0)
        f = s.get("features") or {}
        c += len(f.get("blocks") or {})
        clf = s["clf"]
        c += CLF_COST.get(clf["type"], 1.0)
        if clf.get("class_weight") == "balanced":
            c += 0.5
        if s.get("temperature"):
            c += 0.25
    if "ensemble" in m:
        c += len(singles) - 1
    if cfg.get("head", {}).get("prior") == "sld":
        c += 0.5
    return c


def unsup_labeling(cfg) -> bool:
    """Assignment rule: an UNSUPERVISED method must take part in filling the missing labels.
    True if the label-filling path uses clustering or a similarity graph: a cluster / label-
    spreading / C&S labeler, or guarded self-training with the K-means consistency filter or a
    C&S member, or a direct cluster/graph predictor."""
    m = cfg["model"]
    for s in (m["ensemble"] if "ensemble" in m else [m]):
        lab = s.get("labeler") or {}
        if lab.get("type") in ("cluster", "spread", "cs"):
            return True
        if lab.get("type") == "selftrain" and (lab.get("cluster_filter") or "cs" in lab.get("members", [])):
            return True
        if s["clf"]["type"] == "direct" and s["clf"]["labeler"]["type"] in ("cluster", "spread", "cs"):
            return True
    return False


# ============================================================================ objective
def append_jsonl(path, rec):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    line = json.dumps(rec, default=float) + "\n"
    with open(path, "a") as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        f.write(line)
        f.flush()
        fcntl.flock(f, fcntl.LOCK_UN)


def make_objective(D, log_path, R=4, rows=None, chrono_cuts=CHRONO_CUTS, prune_margin=0.05,
                   allow_blocks=True, cv_seed=0, tag="main", require_unsup=False):
    import optuna

    def objective(trial):
        cfg = suggest_config(trial, allow_blocks=allow_blocks, require_unsup=require_unsup)
        validate(cfg)
        t0 = time.time()
        # stage 1: chrono cuts + val + first CV repeat; prune clearly bad configs
        folds1 = make_folds(D, ("A", "B", "C"), R=R, seed=cv_seed, rows=rows, chrono_cuts=chrono_cuts)
        first = [f for f in folds1 if f["protocol"] != "A" or f["repeat"] == 0]
        rest = [f for f in folds1 if f["protocol"] == "A" and f["repeat"] > 0]
        try:
            outs1 = fold_outputs(cfg, D, first, n_jobs=1)
            r1 = score(cfg, D, first, outs1, bootstrap=False)
            j1 = r1.get("J", np.nan)
            trial.set_user_attr("J_stage1", float(j1))
            try:
                best = trial.study.best_value
            except ValueError:
                best = -np.inf
            if np.isfinite(best) and j1 < best - prune_margin:
                append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "pruned", "J_stage1": j1,
                                        "params": trial.params, "config": cfg, "secs": time.time() - t0})
                raise optuna.TrialPruned()
            outs2 = fold_outputs(cfg, D, rest, n_jobs=1)
            res = score(cfg, D, first + rest, outs1 + outs2, bootstrap=False)
        except optuna.TrialPruned:
            raise
        except Exception as e:
            append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "fail", "error": repr(e),
                                    "tb": traceback.format_exc()[-2000:], "params": trial.params, "config": cfg})
            raise
        cx = complexity(cfg)
        attrs = {"A": res["A"]["sample_f1"], "A_se": res["A"]["sample_f1_se"], "A_macro": res["A"]["macro_f1"],
                 "B": res["B"]["sample_f1"], "B_se": res["B"]["sample_f1_se"], "B_macro": res["B"]["macro_f1"],
                 "J": res["J"], "J_se": res["J_se"], "complexity": cx, "secs": time.time() - t0}
        if "B2005" in res:
            attrs["B2005"] = res["B2005"]["sample_f1"]
        if "C" in res:
            attrs.update(C=res["C"]["sample_f1"], C_se=res["C"]["sample_f1_se"], C_macro=res["C"]["macro_f1"])
        for k, v in attrs.items():
            trial.set_user_attr(k, float(v))
        trial.set_user_attr("config", json.dumps(cfg))
        append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "complete", **attrs, "params": trial.params,
                                "config": cfg, "rowf1_A": res["rowf1_A"], "rowf1_B": res["rowf1_B"],
                                "rowf1_C": res.get("rowf1_C"), "B_ncuts": res.get("B_ncuts", 1)})
        return res["J"]

    return objective


def run_worker(storage_path, study_name, n_trials, log_path, seed, R=4, rows=None, chrono_cuts=CHRONO_CUTS,
               allow_blocks=True, tag="main", timeout=None, sampler_name="tpe", cv_seed=0, require_unsup=False):
    """One search worker process (shares the study through a journal file)."""
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    from .data import load
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    D = load()
    storage = JournalStorage(JournalFileBackend(storage_path))
    if sampler_name == "random":
        sampler = optuna.samplers.RandomSampler(seed=seed)
    else:
        sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True, group=True, constant_liar=True,
                                             n_startup_trials=40)
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize",
                                sampler=sampler, load_if_exists=True)
    obj = make_objective(D, log_path, R=R, rows=rows, chrono_cuts=chrono_cuts, allow_blocks=allow_blocks, tag=tag,
                         cv_seed=cv_seed, prune_margin=(np.inf if sampler_name == "random" else 0.05),
                         require_unsup=require_unsup)
    study.optimize(obj, n_trials=n_trials, timeout=timeout, catch=(Exception,))


def launch(storage_path, study_name, n_workers, trials_per_worker, log_path, R=4, rows=None,
           chrono_cuts=CHRONO_CUTS, allow_blocks=True, tag="main", timeout=None, base_seed=0, enqueue=None,
           sampler_name="tpe", cv_seed=0, require_unsup=False):
    """Run n_workers processes in parallel (spawn) against one journal-backed study."""
    import multiprocessing as mp
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    os.makedirs(os.path.dirname(storage_path), exist_ok=True)
    storage = JournalStorage(JournalFileBackend(storage_path))
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize", load_if_exists=True)
    for params in enqueue or []:
        study.enqueue_trial(params, skip_if_exists=True)
    ctx = mp.get_context("spawn")
    procs = []
    for w in range(n_workers):
        p = ctx.Process(target=run_worker, args=(storage_path, study_name, trials_per_worker, log_path,
                                                 base_seed + w, R, None if rows is None else list(map(int, rows)),
                                                 chrono_cuts, allow_blocks, tag, timeout, sampler_name, cv_seed,
                                                 require_unsup))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
    return study


# ============================================================================ selection
def load_log(log_path, tag=None, state="complete"):
    recs = []
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            if (tag is None or r.get("tag") == tag) and (state is None or r.get("state") == state):
                recs.append(r)
    return recs


def select_one_se(recs, baseline=None, key="J", se_key="J_se"):
    """1-SE rule: candidates with J >= J_best - SE(J_best); choose minimal complexity, ties by J.
    Guard: if a baseline record is given, the choice must have B >= B_base - B_se_base."""
    if not recs:
        return None, []
    best = max(recs, key=lambda r: r[key])
    thr = best[key] - best[se_key]
    cands = [r for r in recs if r[key] >= thr]
    cands.sort(key=lambda r: (r["complexity"], -r[key]))
    for r in cands:
        if baseline is None or r["B"] >= baseline["B"] - baseline["B_se"]:
            return r, cands
    return cands[0], cands


def dedupe(recs):
    """Keep one record per distinct config (TPE re-samples identical categorical configs)."""
    seen = {}
    for r in recs:
        k = json.dumps(r["config"], sort_keys=True)
        if k not in seen or r[ "J"] > seen[k]["J"]:
            seen[k] = r
    return list(seen.values())

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/search.py


## part1/experiments/cand2_submission_v1/source/p1lib/submit.py

In [67]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/submit.py
"""Final refit on all 235 labeled rows, test predictions, Kaggle CSV + manifest, filled train labels."""
from __future__ import annotations

import datetime
import hashlib
import json
import os

import numpy as np
import pandas as pd

from .context import make_ctx
from .data import LABELS, write_submission
from .head import apply_head, decode, sld_em
from .pipeline import run_model, validate


def final_fit(cfg, D, n_seeds=1):
    """Refit on labeled-train + val (235 rows); return calibrated probs for test and the pool."""
    validate(cfg)
    test, unl = D.idx["test"], D.idx["unl"]
    rows = np.r_[test, unl]
    Ps, priors, Ts = [], [], []
    for s in range(n_seeds):
        ctx = make_ctx(D, D.idx["labval"], test, transductive=cfg.get("transductive", True), seed=s, name="final")
        o = run_model(cfg["model"], ctx, rows)
        Ps.append(o["P"]); priors.append(np.asarray(o["prior"])); Ts.append(o["T"])
    P = np.mean(Ps, axis=0)
    prior = np.mean(priors, axis=0)
    return {"P_test": P[: len(test)], "P_pool": P[len(test):], "prior": prior, "T": Ts}


def predict_test(cfg, D, fit):
    head = cfg.get("head", {})
    tr = cfg.get("transductive", True)
    target = None if tr else fit["P_pool"]          # inductive: SLD prior estimated on the unlabeled pool
    Y, Q, pi = apply_head(fit["P_test"], fit["prior"], D.CM, head, target)
    return Y, Q, pi


def fill_pool_labels(cfg, D, fit):
    """Hard labels for the 1,014 masked train rows from the refit model (SLD toward the pool's own
    prior when the head uses SLD)."""
    head = cfg.get("head", {})
    P = fit["P_pool"]
    if head.get("prior") == "sld":
        P, _ = sld_em(P, fit["prior"], a=head.get("sld_a", 0), floor=head.get("sld_floor", 0.25))
    return decode(P, D.CM, head.get("decode", "map")), P


def write_candidate(cfg, D, out_dir, name, scores=None, n_seeds=1, sample_path=None):
    os.makedirs(out_dir, exist_ok=True)
    fit = final_fit(cfg, D, n_seeds=n_seeds)
    Y, Q, pi = predict_test(cfg, D, fit)
    path = os.path.join(out_dir, f"{name}.csv")
    sub = write_submission(D, Y, path)
    if sample_path:                                  # exact format / order check vs the sample file
        sample = pd.read_csv(sample_path)
        assert list(sub.columns) == list(sample.columns) and (sub["ID"].values == sample["ID"].values).all()
    sha = hashlib.sha1(open(path, "rb").read()).hexdigest()
    manifest = {
        "name": name, "csv": os.path.abspath(path), "sha1": sha,
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "config": cfg, "transductive_test_features_used": bool(cfg.get("transductive", True)),
        "trained_on": "157 labeled train rows + 78 val rows (235); 1,014 unlabeled train rows as pool",
        "n_seeds": n_seeds,
        "scores": scores or {},
        "test_pred_label_rate": dict(zip(LABELS, Y.mean(0).round(3).tolist())),
        "test_pred_cardinality": float(Y.sum(1).mean()),
        "test_pred_combo_counts": {"+".join(LABELS[j] for j in range(5) if r[j]): int(c)
                                   for r, c in zip(*np.unique(Y, axis=0, return_counts=True))},
        "sld_test_prior_labels": None if pi is None else dict(zip(LABELS, (pi @ D.CM).round(3).tolist())),
        "train_prior_labels": dict(zip(LABELS, (fit["prior"] @ D.CM).round(3).tolist())),
        "temperature": fit["T"],
    }
    with open(os.path.join(out_dir, f"{name}.manifest.json"), "w") as f:
        json.dump(manifest, f, indent=1, default=float)
    return manifest, fit, Y


def write_filled_train(cfg, D, fit, path):
    Yp, P = fill_pool_labels(cfg, D, fit)
    tr = D.df.iloc[D.idx["train"]].copy()
    tr = tr.drop(columns=["split"])
    pos = {g: i for i, g in enumerate(D.idx["unl"])}
    lab = tr[LABELS].to_numpy().copy()
    src = np.array(["given"] * len(tr), dtype=object)
    for i, g in enumerate(D.idx["train"]):
        if g in pos:
            lab[i] = Yp[pos[g]]
            src[i] = "filled"
    tr[LABELS] = lab
    tr["label_source"] = src
    tr["fill_confidence"] = np.where(src == "filled", [P[pos[g]].max() if g in pos else 1.0 for g in D.idx["train"]], 1.0)
    tr.to_csv(path, index=False)
    return tr

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/submit.py


## part1/experiments/cand2_submission_v1/source/p1lib/synth.py

In [68]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/synth.py
"""Synthetic features: unsupervised / semi-supervised model outputs turned into new columns
that downstream classifiers (or a stacker) consume.

    synth_features(scfg, ctx) -> (n_all, m) array for every global row

Label-free generators (fit on ctx.unsup features only, cached):
  gmm        : GMM posterior responsibilities (or log-responsibilities)
  kmeans     : distances to K-means centroids, or soft assignments softmax(-d^2/T)
Label-dependent generators (training rows get OUT-OF-FOLD values, so no row sees its own label):
  knn_vote   : cosine kNN label-vote fractions over the K combos (exact leave-one-out)
  graph      : label-spreading or C&S scores (inner K-fold for training rows)
  cluster_map: cluster -> label-combo mapped probabilities (exact leave-one-out)
  oof_model  : out-of-fold probabilities of any single pipeline config (inner K-fold = nested OOF)
Label-dependent outputs are returned as log-probabilities (K columns).
"""
from __future__ import annotations

import numpy as np

from .cache import memo, memo_ctx
from .classifiers import KNNClf
from .features import featurize
from .head import safe_log
from .labelers import DEFAULT_CLUSTER_SPACE, _cluster_fit, soft_labels, soft_labels_train_oof

LABEL_FREE = {"gmm", "kmeans"}


def _label_free(scfg, ctx):
    algo = scfg["type"]
    k = int(scfg.get("k", 16))
    space = scfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    key = {"algo": algo, "k": k, "space": space, **{kk: scfg[kk] for kk in ("cov", "n_init") if kk in scfg}}
    fit = memo("cluster", key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, scfg))
    if algo == "gmm":
        R = fit["resp"]
        return np.log(np.clip(R, 1e-6, None)) if scfg.get("out", "resp") == "logresp" else R
    d = fit["dist"]
    if scfg.get("out", "dist") == "soft":
        d2 = d ** 2
        T = float(scfg.get("temp", 1.0)) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    return d


def _knn_vote(scfg, ctx):
    K = ctx.D.K
    Z = featurize(ctx, scfg.get("space", {"scale": True, "l2": True}))
    y = ctx.y_train
    m = KNNClf({"type": "knn", "k": int(scfg.get("k", 10)), "weights": scfg.get("weights", "uniform"),
                "beta": float(scfg.get("beta", 1.0))}, K, ctx.D.CM).fit(Z[ctx.train], y)
    out = m.predict_proba(Z)
    out[ctx.train] = m.predict_proba(Z[ctx.train], exclude_self=True)
    return safe_log(out)


def _labeler_feature(lcfg, ctx):
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out = np.zeros((ctx.D.n, ctx.D.K))
    out[rows] = soft_labels(lcfg, ctx, rows)
    out[ctx.train] = soft_labels_train_oof(lcfg, ctx)
    return safe_log(out)


def _oof_model(scfg, ctx):
    from .pipeline import inner_folds, run_single
    sub = scfg["model"]
    y = ctx.y_train
    out = np.zeros((ctx.D.n, ctx.D.K))
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out[rows] = run_single(sub, ctx, rows)["P"]
    for tr, te in inner_folds(y, int(scfg.get("n_inner", 5)), ctx.seed + 17):
        inner = ctx.sub(ctx.train[tr], ctx.train[te])
        out[ctx.train[te]] = run_single(sub, inner, ctx.train[te])["P"]
    return safe_log(out)


def synth_features(scfg, ctx) -> np.ndarray:
    if scfg["type"] in LABEL_FREE:
        return _synth(scfg, ctx)
    return memo_ctx("synth", scfg, ctx, lambda: _synth(scfg, ctx))


def _synth(scfg, ctx) -> np.ndarray:
    t = scfg["type"]
    if t in LABEL_FREE:
        return _label_free(scfg, ctx)
    if t == "knn_vote":
        return _knn_vote(scfg, ctx)
    if t == "graph":
        mode = scfg.get("mode", "spread")
        lcfg = {"type": "spread" if mode == "spread" else "cs", "graph": scfg.get("graph"),
                "alpha": scfg.get("alpha", 0.5), "a1": scfg.get("a1", 0.5), "a2": scfg.get("alpha", 0.5)}
        return _labeler_feature(lcfg, ctx)
    if t == "cluster_map":
        lcfg = {"type": "cluster", **{k: v for k, v in scfg.items() if k != "type"}}
        return _labeler_feature(lcfg, ctx)
    if t == "oof_model":
        return _oof_model(scfg, ctx)
    raise ValueError(t)


def build_synth_block(synth_list, ctx):
    """Concatenate all synthetic features and standardize them on train+pool rows."""
    if not synth_list:
        return None
    parts = [synth_features(s, ctx) for s in synth_list]
    S = np.hstack(parts)
    fit_rows = np.union1d(ctx.train, ctx.pool)
    mu = S[fit_rows].mean(0)
    sd = S[fit_rows].std(0)
    sd[sd < 1e-9] = 1.0
    return (S - mu) / sd

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/synth.py


## part1/experiments/cand2_submission_v1/source/p1lib/viz.py

In [69]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/source/p1lib/viz.py
"""Shared matplotlib style for report figures (static PNG/PDF).

Categorical hues are assigned in a FIXED order (validated reference palette, light mode);
sequential = one blue ramp; thin marks, solid hairline grid, no dual axes.
"""
from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
TEXT2 = "#52514e"
GRID = "#e4e3df"
MUTED = "#a3a29c"

# fixed identity colors for recurring entities so a method keeps its hue across figures
PROTOCOL_COLORS = {"A": SERIES[0], "B": SERIES[1], "C": SERIES[2]}
LABEL_COLORS = dict(zip(["Safety", "Funding", "Efficacy", "Recruitment", "Other"], SERIES[:5]))


def setup():
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.grid": True, "grid.color": GRID,
        "grid.linewidth": 0.6, "grid.linestyle": "-", "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2, "text.color": TEXT,
        "axes.titlesize": 11, "axes.titleweight": "semibold", "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.5, "legend.frameon": False,
        "lines.linewidth": 2.0, "lines.markersize": 6, "font.family": "DejaVu Sans",
        "axes.prop_cycle": mpl.cycler(color=SERIES), "figure.dpi": 110, "savefig.dpi": 200,
        "savefig.bbox": "tight",
    })


def save(fig, path):
    fig.savefig(path)
    if path.endswith(".png"):
        fig.savefig(path[:-4] + ".pdf")
    return path

Overwriting _source_export/part1/experiments/cand2_submission_v1/source/p1lib/viz.py


## part1/experiments/cand2_submission_v1/strict_adapter.py

In [70]:
%%writefile _source_export/part1/experiments/cand2_submission_v1/strict_adapter.py
"""Inductive, group-safe adapters for the four fixed Dylan v2 candidates.

Original p1lib is copied unchanged under source/. This file changes fitting
boundaries and nested calibration, not classifier settings or selection weights.
"""
from dataclasses import replace
import hashlib

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

from p1lib import cache, features, graph, labelers
from p1lib.classifiers import make_classifier, KNNClf
from p1lib.context import Ctx
from p1lib.head import apply_temperature, fit_temperature, safe_log
from p1lib.pipeline import _mask_absent

AUDIT = []


def groups(D):
    if not hasattr(D, '_embedding_groups'):
        D._embedding_groups = pd.util.hash_pandas_object(
            D.df[[f'emb_{i}' for i in range(150)]].round(10), index=False).to_numpy()
    return D._embedding_groups


def context(D, train, query, pool, seed=0, name='', depth=0):
    train, query, pool = [np.asarray(v, dtype=int) for v in [train, query, pool]]
    unsup = np.union1d(train, pool)
    assert not np.intersect1d(unsup, query).size
    assert not np.intersect1d(unsup, D.idx['test']).size
    # The only accessible labels belong to the fitting gold rows.
    y = np.full_like(D.y, -1); y[train] = D.y[train]
    Y = np.full_like(D.Y, -1); Y[train] = D.Y[train]
    df = D.df.copy(); df.loc[:, ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']] = Y
    hidden = replace(D, df=df, y=y, Y=Y)
    hidden._embedding_groups = groups(D)
    return Ctx(hidden, train, query, pool, unsup, False, seed, name, depth)


def subcontext(ctx, train, query):
    g = groups(ctx.D)
    banned = g[query]
    train = np.asarray(train)[~np.isin(g[train], banned)]
    pool = ctx.pool[~np.isin(g[ctx.pool], banned)]
    sub = context(ctx.D, train, query, pool, ctx.seed, ctx.name + '/inner', ctx.depth + 1)
    assert not np.intersect1d(g[sub.unsup], g[query]).size
    assert np.isin(sub.unsup, ctx.unsup).all()
    AUDIT.append(dict(depth=sub.depth, gold=len(train), masked=len(pool), query=len(query),
                      fitting_ids_sha256=hashlib.sha256(sub.D.df.iloc[sub.unsup].ID.to_numpy(str).tobytes()).hexdigest(),
                      gold_group_overlap=0, masked_group_overlap=0))
    return sub


def inner_folds(ctx, n=5):
    _, inv = np.unique(groups(ctx.D)[ctx.train], return_inverse=True)
    target = np.zeros((inv.max() + 1, 5), dtype=int)
    np.maximum.at(target, inv, ctx.D.CM[ctx.y_train])
    splitter = MultilabelStratifiedKFold(n_splits=n, shuffle=True, random_state=ctx.seed)
    for tr, te in splitter.split(np.zeros((len(target), 1)), target):
        yield np.flatnonzero(np.isin(inv, tr)), np.flatnonzero(np.isin(inv, te))


def base_oof(ctx, base_cfg, feature_cfg):
    def compute():
        K, CM = ctx.D.K, ctx.D.CM
        X = features.featurize(ctx, feature_cfg)
        P_oof = np.zeros((len(ctx.train), K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            Xi = features.featurize(sub, feature_cfg)
            m = make_classifier(base_cfg, K, CM).fit(Xi[sub.train], sub.y_train)
            P_oof[te] = m.predict_proba(Xi[sub.eval])
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], ctx.y_train)
        return P_oof, m.predict_proba(X)
    return cache.memo_ctx('strict_base_oof', [base_cfg, feature_cfg], ctx, compute)


def cs_probabilities(lcfg, ctx, rows):
    assert lcfg['type'] == 'cs'
    core = {k: v for k, v in lcfg.items() if k not in ('lam', 'tau', 'hard', 'pool')}
    def compute():
        oof, all_p = base_oof(ctx, lcfg.get('base', {'type': 'lr', 'C': .1}),
                            lcfg.get('base_features', {'scale': True}))
        return graph.correct_and_smooth(ctx, lcfg.get('graph'), float(lcfg.get('a1', .5)),
                float(lcfg.get('a2', .5)), ctx.train, ctx.y_train, oof, all_p, ctx.D.K, rows)
    return cache.memo_ctx('strict_cs', core, ctx, compute, extra=np.asarray(rows).tolist())


def knn_features(cfg, ctx):
    def compute():
        space = cfg.get('space', {'scale': True, 'l2': True})
        knn = dict(type='knn', k=int(cfg.get('k', 10)), weights=cfg.get('weights', 'uniform'),
                   beta=float(cfg.get('beta', 1)))
        Z = features.featurize(ctx, space)
        m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Z[ctx.train], ctx.y_train)
        out = m.predict_proba(Z)
        g = groups(ctx.D)
        # Preserve the source's leave-one-out idea, but leave out the entire
        # embedding group and refit scaling with that group's masked copies purged.
        for value in np.unique(g[ctx.train]):
            held = ctx.train[g[ctx.train] == value]
            train = ctx.train[g[ctx.train] != value]
            sub = subcontext(ctx, train, held)
            # Avoid caching one large transformed matrix per leave-group-out fit.
            Zi = features._embed(sub.D, features.norm_features_cfg(space), sub.unsup)
            m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Zi[sub.train], sub.y_train)
            out[held] = m.predict_proba(Zi[held])
        return safe_log(out)
    return cache.memo_ctx('strict_knn_feature', cfg, ctx, compute)


def kmeans_features(cfg, ctx):
    assert cfg['type'] == 'kmeans' and cfg.get('out') == 'soft'
    space = cfg.get('space', labelers.DEFAULT_CLUSTER_SPACE)
    Z = features.featurize(ctx, space)
    key = dict(algo='kmeans', k=int(cfg.get('k', 16)), space=space,
               **{k: cfg[k] for k in ('cov', 'n_init') if k in cfg})
    fit = cache.memo('cluster', key, ctx.unsup,
                     lambda: labelers._cluster_fit(ctx.D, 'kmeans', key['k'], Z, ctx.unsup, cfg))
    d2 = fit['dist'] ** 2
    # Source uses all global rows for this median, including eval/test. Restrict it.
    T = float(cfg.get('temp', 1)) * np.median(d2[ctx.unsup].min(1))
    logp = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
    result = np.exp(logp)
    return result / result.sum(1, keepdims=True)


def design(cfg, ctx):
    parts = []
    if cfg.get('features') is not None:
        assert not cfg['features'].get('blocks')
        parts.append(features.featurize(ctx, cfg['features']))
    synth = []
    for scfg in cfg.get('synth', []):
        if scfg['type'] == 'kmeans':
            synth.append(kmeans_features(scfg, ctx))
        elif scfg['type'] == 'knn_vote':
            synth.append(knn_features(scfg, ctx))
        else:
            raise ValueError(scfg)
    if synth:
        S = np.hstack(synth)
        mu, sd = S[ctx.unsup].mean(0), S[ctx.unsup].std(0)
        sd[sd < 1e-9] = 1
        parts.append((S - mu) / sd)
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def uncalibrated(cfg, ctx, rows):
    K, CM, y = ctx.D.K, ctx.D.CM, ctx.y_train
    if cfg['clf']['type'] == 'direct':
        return dict(P=cs_probabilities(cfg['clf']['labeler'], ctx, rows),
                    prior=np.bincount(y, minlength=K) / len(y), info={})
    X = design(cfg, ctx)
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    lcfg = cfg.get('labeler')
    if lcfg:
        assert lcfg.get('pool', 'unl') == 'unl'
        pr = ctx.pool
        Q = cs_probabilities(lcfg, ctx, pr)
        keep = Q.max(1) >= float(lcfg.get('tau', 0))
        if lcfg.get('hard', False):
            Q = np.eye(K)[Q.argmax(1)]
        pr, Q = pr[keep], Q[keep]
        w = np.full(len(pr), float(lcfg.get('lam', .1)))
    Xt, yt, wt = labelers.expand(X[ctx.train], np.eye(K)[y], np.ones(len(y)), X[pr], Q, w)
    m = make_classifier(cfg['clf'], K, CM).fit(Xt, yt, wt)
    return dict(P=m.predict_proba(X[rows]), prior=m.prior_, info={'n_pseudo_rows': len(pr)})


def run_single(cfg, ctx, rows):
    out = uncalibrated(cfg, ctx, rows)
    T = 1.
    if cfg.get('temperature', True):
        oof = np.zeros((len(ctx.train), ctx.D.K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            # Relearn features, pseudo labels and classifier inside each inner fold.
            oof[te] = uncalibrated(cfg, sub, sub.eval)['P']
        T = fit_temperature(oof, ctx.y_train)
        out['P'] = apply_temperature(out['P'], T)
    out['P'] = _mask_absent(out['P'], out['prior'])
    out['T'] = T
    return out


def run_model(cfg, ctx, rows):
    if 'ensemble' not in cfg:
        return run_single(cfg, ctx, rows)
    members = cfg['ensemble']
    weights = np.array(cfg.get('weights') or [1.] * len(members)); weights /= weights.sum()
    outs = [run_single(m, ctx, rows) for m in members]
    return dict(P=sum(w * o['P'] for w, o in zip(weights, outs)),
                prior=sum(w * o['prior'] for w, o in zip(weights, outs)),
                T=[o['T'] for o in outs], info={'members': [o['info'] for o in outs]})

Overwriting _source_export/part1/experiments/cand2_submission_v1/strict_adapter.py


## part1/experiments/current_leaders_grouped_v1/snapshots/anchored_solution.py

In [71]:
%%writefile _source_export/part1/experiments/current_leaders_grouped_v1/snapshots/anchored_solution.py
"""Current Ani with three small changes: anchored weights, binary votes, weaker GMM.

Fit receives only permitted rows, in original CSV order. Fully missing labels
are -100; query/test rows must never be passed to fit. Prediction uses only the
150 stopping-reason embedding columns, ordered emb_0 through emb_149.
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, normalize


def select_reasons(scores):
    result=(scores>=.3).astype(int)
    result[np.arange(len(scores)),scores.argmax(axis=1)]=1
    return result


class AnchoredReasonClassifier:
    def fit(self, embeddings, labels):
        X,Y=np.asarray(embeddings,float),np.asarray(labels,int)
        if X.ndim!=2 or X.shape[1]!=150 or Y.shape!=(len(X),5):
            raise ValueError('Expected 150 embeddings and five labels per row.')
        known=np.isin(Y,[0,1]).all(axis=1)
        unknown=(Y==-100).all(axis=1)
        if not (known|unknown).all() or (Y[known].sum(axis=1)==0).any() or (Y[known].sum(axis=0)==0).any():
            raise ValueError('Use complete nonempty known labels or five -100 values; all five reasons must be represented.')
        self.scaler_=StandardScaler().fit(X)
        Z=self.scaler_.transform(X)
        self.pca_=PCA(30,random_state=0).fit(Z)
        self.gmm_=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(self.pca_.transform(Z))
        assert self.gmm_.converged_
        K,V=Z[known],Z[unknown];targets=Y[known]
        Rk=self.gmm_.predict_proba(self.pca_.transform(K))
        Ru=self.gmm_.predict_proba(self.pca_.transform(V))
        counts=(targets/targets.sum(axis=1,keepdims=True)).sum(axis=0)
        self.class_weights_=counts.mean()/counts
        Za,Ya,Ra=K,targets,Rk
        self.round_audit_=[]
        for step in range(3):
            row,reason=np.nonzero(Ya)
            weights=self.class_weights_[reason]/Ya.sum(axis=1)[row]
            weights*=np.where(row<len(K),1.,.25)
            self.logistic_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(Za[row],reason,sample_weight=weights)
            self.lda_=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(5)/5).fit(Za[row],reason)
            self.neighbor_X_=normalize(Za)
            self.neighbor_Y_=Ya.copy()
            self.round_audit_.append(dict(step=step,known=len(K),generated=len(Za)-len(K),
                                          known_weight=float(weights[row<len(K)].sum()),
                                          generated_weight=float(weights[row>=len(K)].sum())))
            if step==2:break
            T=Ya/Ya.sum(axis=1,keepdims=True)
            rates=(Ra.T@T+T.mean(axis=0))/(Ra.sum(axis=0)[:,None]+1)
            scores=.95*self._scores(V)+.05*(Ru@rates)
            self.generated_mask_=scores.max(axis=1)>=.8
            self.generated_labels_=select_reasons(scores[self.generated_mask_])
            Za=np.vstack([K,V[self.generated_mask_]])
            Ya=np.vstack([targets,self.generated_labels_])
            Ra=np.vstack([Rk,Ru[self.generated_mask_]])
        return self

    def _scores(self,Z):
        logistic=np.zeros((len(Z),5))
        logistic[:,self.logistic_.classes_]=self.logistic_.predict_proba(Z)
        similarity=normalize(Z)@self.neighbor_X_.T
        neighbors=np.argsort(-similarity,axis=1)[:,:10]
        weights=np.take_along_axis(similarity,neighbors,axis=1).clip(0)**2
        local=(weights[:,:,None]*self.neighbor_Y_[neighbors]).sum(axis=1)
        local/=weights.sum(axis=1,keepdims=True)+1e-9
        lda=np.zeros_like(logistic)
        lda[:,self.lda_.classes_]=self.lda_.predict_proba(Z)
        return (1-.3-.2)*logistic+.3*local+.2*lda

    def predict_proba(self, embeddings):
        """Selection scores, not calibrated marginal probabilities."""
        return self._scores(self.scaler_.transform(np.asarray(embeddings,float)))

    def predict(self, embeddings):
        return select_reasons(self.predict_proba(embeddings))

Overwriting _source_export/part1/experiments/current_leaders_grouped_v1/snapshots/anchored_solution.py


## part1/experiments/current_leaders_grouped_v1/snapshots/compare_current_leaders.py

In [72]:
%%writefile _source_export/part1/experiments/current_leaders_grouped_v1/snapshots/compare_current_leaders.py
"""Frozen cross-agent comparison on the shared grouped 157/235 protocols.

No tuning, final-data refitting, new test predictions, or uploads. Existing
competitors are independently rescored; our candidate and Ani control refit.
"""
from datetime import datetime,timezone
import hashlib
import json
from pathlib import Path
import shutil
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit,logit
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score,precision_recall_fscore_support
from threadpoolctl import threadpool_limits

from anchored_solution import AnchoredReasonClassifier
from paired_models import PairedModel,prepare,ANI

ROOT=Path(__file__).resolve().parent
OUT=ROOT/'experiments/current_leaders_grouped_v1'
OTHER=ROOT.parent.parent/'AML-Part1-NewDirections'
INTEGRATION=ROOT.parent.parent/'AML-Midterm-integration/part1'
EXP=INTEGRATION/'experiments'
REASONS=['Safety','Funding','Efficacy','Recruitment','Other']
FEATURES=[f'emb_{i}' for i in range(150)]
SOURCES={}
MODELS=['original_ani','current_ani_lda','previous_ensemble','lda_ensemble',
        'recent_cutoff_ensemble','xrfm20_blend','kernel15_blend','anchored_candidate']

def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save(p,value):Path(p).write_text(json.dumps(value,indent=2,default=lambda v:v.item() if isinstance(v,np.generic) else str(v))+'\n')
def source(p):SOURCES[str(p)]=sha(p);return Path(p)
def read(p):return json.loads(source(p).read_text())
def load(p):
    with np.load(source(p),allow_pickle=True) as z:return {k:z[k] for k in z.files}
def groups(df):return pd.util.hash_pandas_object(df[FEATURES].round(10),index=False).to_numpy()
def metric(y,p):
    precision,recall,f1,support=precision_recall_fscore_support(y,p,zero_division=0)
    return dict(sample_f1=float(f1_score(y,p,average='samples',zero_division=0)),macro_f1=float(f1.mean()),
                **{f'{kind}_{r}':float(values[j]) for kind,values in [('precision',precision),('recall',recall),('f1',f1),('support',support)] for j,r in enumerate(REASONS)})
def join(parts):
    for p in parts[1:]:
        np.testing.assert_array_equal(parts[0]['ids'],p['ids'])
        np.testing.assert_array_equal(parts[0]['truth'],p['truth'])
    return dict(ids=parts[0]['ids'].astype(str),truth=parts[0]['truth'],
                scores=np.concatenate([p['scores'] for p in parts]),predictions=np.concatenate([p['predictions'] for p in parts]))
def references(pool,seeds):
    models={}
    for name,prefix in [('original_ani','ani'),('previous_ensemble','threeway')]:
        models[name]=join([load(OTHER/'references'/f'{prefix}_{pool}_{suffix}.npz') for suffix in ['first','extra']])
    for name,stage,recipe in [('current_ani_lda','ani_lda_revision_cv_v1','ani__revision_full'),
                              ('lda_ensemble','ani_lda_revision_cv_v1','ensemble__revision_full'),
                              ('recent_cutoff_ensemble','nested_decision_lda_cv_v1','ensemble__global_recent')]:
        models[name]=join([load(EXP/stage/f'{pool}{n}'/f'{recipe}.npz') for n in [3,5]])
    for name,version,recipe in [('xrfm20_blend','v10','hybrid_leaf_one_raw'),('kernel15_blend','v5','rfm_complement_all15')]:
        parts=[load(OTHER/'studies'/version/'results'/recipe/pool/f'{seed}.npz') for seed in seeds]
        for p in parts:p['scores']=p['scores'][None];p['predictions']=p['predictions'][None]
        models[name]=join(parts)
    return models


def main():
    OUT.mkdir(exist_ok=True);(OUT/'snapshots').mkdir(exist_ok=True)
    train=pd.read_csv(source(ROOT/'data/part1_train.csv'));valid=pd.read_csv(source(ROOT/'data/part1_val.csv'))
    for p in [OTHER/'data',INTEGRATION/'data']:
        for filename in ['part1_train.csv','part1_val.csv']:assert sha(p/filename)==sha(ROOT/'data'/filename)
    for p in [ROOT/'anchored_solution.py',ROOT/'paired_models.py',Path(__file__)]:
        source(p);dest=OUT/'snapshots'/p.name
        if dest.exists():assert sha(dest)==sha(p)
        else:shutil.copyfile(p,dest)
    plans={};refs={}
    for pool in ['train','pooled']:
        parts=[read(OTHER/'protocols'/f'{pool}{n}.json') for n in [3,5]]
        assert parts[0]['gold_ids']==parts[1]['gold_ids']
        plans[pool]=dict(gold_ids=parts[0]['gold_ids'],folds=parts[0]['folds']+parts[1]['folds'],seeds=parts[0]['seeds']+parts[1]['seeds'])
        for n in [3,5]:shutil.copyfile(OTHER/'protocols'/f'{pool}{n}.json',OUT/'snapshots'/f'{pool}{n}.json')
        refs[pool]=references(pool,plans[pool]['seeds'])
    chrono=read(OTHER/'protocols/chronology.json')
    shutil.copyfile(OTHER/'protocols/chronology.json',OUT/'snapshots/chronology.json')
    protocol=dict(version='current-leaders-grouped-v1',created_at=datetime.now(timezone.utc).isoformat(),models=MODELS,
                  purpose='Compare the unchanged, now submitted anchored candidate with the latest other-agent leaders.',
                  schedule='All 8 saved repeats x 5 folds in BOTH pools, then original validation and 3 saved chronological windows; no screening or tuning.',
                  access='Inductive; purge masked duplicates of held-out embedding groups; no test-input fitting.',
                  scoring='Complete OOF sample F1 per repeat; mean and population SD; paired repeat wins. Chronology pools 98 held-out rows.',
                  evidence='Refit anchored candidate and current Ani LDA control on every fold. Rescore frozen competitor predictions after exact ID/truth checks.',
                  selection='No recipe selection, threshold change, promotion, new final fit or upload.',sources_before_fit=SOURCES.copy())
    if (OUT/'protocol.json').exists():
        old=read(OUT/'protocol.json');assert old['sources_before_fit']==protocol['sources_before_fit']
    else:save(OUT/'protocol.json',protocol)
    rows=[];audits=[];all_predictions=[];control_checks=[]
    full=pd.concat([train,valid],ignore_index=True)
    with threadpool_limits(limits=1),warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        for pool in ['train','pooled']:
            frame=train if pool=='train' else full
            gold=frame[frame[REASONS].ge(0).all(axis=1)].reset_index(drop=True)
            masked=train[train[REASONS].eq(-100).all(axis=1)].reset_index(drop=True)
            plan=plans[pool];assert gold.ID.tolist()==plan['gold_ids']
            y=gold[REASONS].to_numpy(int);gg,ug=groups(gold),groups(masked)
            models=refs[pool];scores=np.full((8,len(gold),5),np.nan);predictions=np.full((8,len(gold),5),-1,int)
            for name,z in models.items():
                np.testing.assert_array_equal(z['ids'],gold.ID)
                np.testing.assert_array_equal(z['truth'],y)
                assert z['predictions'].shape==(8,len(gold),5)
                np.savez_compressed(OUT/'snapshots'/f'{name}_{pool}.npz',**z)
            order={id:i for i,id in enumerate(frame.ID)}
            for ri,seed in enumerate(plan['seeds']):
                coverage=np.zeros(len(gold),int)
                for fold in [f for f in plan['folds'] if f['seed']==seed]:
                    a=np.array(fold['train_gold_positions']);b=np.array(fold['heldout_gold_positions'])
                    assert set(a).isdisjoint(b) and len(a)+len(b)==len(y)
                    assert set(gg[a]).isdisjoint(gg[b]) and gold.iloc[b].ID.tolist()==fold['heldout_ids']
                    keep=~np.isin(ug,gg[b]);assert (~keep).sum()==fold['masked_duplicates_excluded']
                    fit=pd.concat([gold.iloc[a],masked[keep]],ignore_index=True)
                    fit=fit.iloc[np.argsort([order[id] for id in fit.ID])].reset_index(drop=True)
                    assert set(groups(fit)).isdisjoint(gg[b]) and set(fit.ID).isdisjoint(gold.iloc[b].ID)
                    if pool=='train':assert set(fit.ID).isdisjoint(valid.ID)
                    model=AnchoredReasonClassifier().fit(fit[FEATURES],fit[REASONS])
                    scores[ri,b]=model.predict_proba(gold.iloc[b][FEATURES]);predictions[ri,b]=model.predict(gold.iloc[b][FEATURES]);coverage[b]+=1
                    fit_known=fit[REASONS].ge(0).all(axis=1).to_numpy()
                    x=fit[FEATURES].to_numpy();labels=fit[REASONS].to_numpy(int)
                    ctrl=PairedModel(ANI).fit(x[fit_known],labels[fit_known],x[~fit_known],prepare(x))
                    raw=ctrl.predict_proba(gold.iloc[b][FEATURES].to_numpy())
                    centered=expit(logit(np.clip(raw,1e-9,1-1e-9))-logit(.3))
                    old=models['current_ani_lda']['scores'][ri,b]
                    np.testing.assert_allclose(centered,old,atol=1e-7,rtol=0)
                    np.testing.assert_array_equal(ctrl.predict(gold.iloc[b][FEATURES].to_numpy()),models['current_ani_lda']['predictions'][ri,b])
                    control_checks.append(dict(pool=pool,seed=seed,fold=fold['fold'],max_score_error=float(abs(centered-old).max()),identical_predictions=True))
                    audits.append(dict(pool=pool,seed=seed,fold=fold['fold'],gold_ids=gold.iloc[a].ID.tolist(),masked_ids=masked[keep].ID.tolist(),
                                       held_ids=gold.iloc[b].ID.tolist(),masked_purged=int((~keep).sum()),fit_audit=model.round_audit_))
                assert (coverage==1).all()
                print(pool,seed,metric(y,predictions[ri])['sample_f1'],flush=True)
            models['anchored_candidate']=dict(scores=scores,predictions=predictions,ids=gold.ID.to_numpy(str),truth=y)
            np.savez_compressed(OUT/f'anchored_{pool}.npz',**models['anchored_candidate'])
            for name,z in models.items():
                for ri,seed in enumerate(plan['seeds']):
                    rows.append(dict(pool=pool,name=name,seed=seed,**metric(y,z['predictions'][ri])))
                    out=pd.DataFrame(dict(ID=gold.ID,name=name,pool=pool,seed=seed))
                    for j,r in enumerate(REASONS):out[f'actual_{r}']=y[:,j];out[f'predicted_{r}']=z['predictions'][ri,:,j]
                    all_predictions.append(out)
        diagnostic_rows=[];diagnostic_predictions=[]
        cases=[('validation',train,valid)]
        indexed=full.set_index('ID',drop=False)
        for w in chrono['chronological_splits']:
            fit=indexed.loc[w['training_ids']];q=indexed.loc[w['heldout_ids']]
            assert fit[REASONS].ge(0).all(axis=1).sum()==w['gold_rows']
            assert fit[REASONS].eq(-100).all(axis=1).sum()==w['masked_rows']
            assert len(q)==w['heldout_rows'] and set(groups(fit)).isdisjoint(groups(q))
            cases.append((w['name'],fit,q))
        for wi,(case,fit,q) in enumerate(cases):
            model=AnchoredReasonClassifier().fit(fit[FEATURES],fit[REASONS]);p=model.predict(q[FEATURES])
            models={'anchored_candidate':dict(ids=q.ID.to_numpy(str),truth=q[REASONS].to_numpy(int),predictions=p)}
            for name,stage,recipe in [('original_ani','ani_lda_revision_cv_v1','ani'),('current_ani_lda','ani_lda_revision_cv_v1','ani__revision_full'),
                                      ('previous_ensemble','ani_lda_revision_cv_v1','ensemble'),('lda_ensemble','ani_lda_revision_cv_v1','ensemble__revision_full'),
                                      ('recent_cutoff_ensemble','nested_decision_lda_cv_v1','ensemble__global_recent')]:
                models[name]=load(EXP/stage/'supplementary'/f'window{wi}_{recipe}.npz')
            for name,version,recipe in [('xrfm20_blend','v10','hybrid_leaf_one_raw'),('kernel15_blend','v5','rfm_complement_all15')]:
                models[name]=load(OTHER/'studies'/version/'supplementary'/recipe/f'{case}.npz')
            for name,z in models.items():
                lookup={id:i for i,id in enumerate(z['ids'])};idx=[lookup[id] for id in q.ID]
                np.testing.assert_array_equal(z['truth'][idx],q[REASONS])
                pred=z['predictions'][idx]
                diagnostic_rows.append(dict(name=name,case=case,**metric(q[REASONS].to_numpy(int),pred)))
                out=pd.DataFrame(dict(ID=q.ID.to_numpy(),name=name,case=case))
                for j,r in enumerate(REASONS):out[f'actual_{r}']=q[r].to_numpy();out[f'predicted_{r}']=pred[:,j]
                diagnostic_predictions.append(out)
            print('diagnostic',case,metric(q[REASONS].to_numpy(int),p)['sample_f1'],flush=True)
    repeats=pd.DataFrame(rows);repeats.to_csv(OUT/'repeat_scores.csv',index=False)
    summary=repeats.groupby(['pool','name']).agg(mean_f1=('sample_f1','mean'),sd=('sample_f1',lambda x:np.std(x,ddof=0)),macro_f1=('macro_f1','mean'))
    pairs=[]
    for pool in ['train','pooled']:
        w=repeats[repeats.pool==pool].pivot(index='seed',columns='name',values='sample_f1')
        for name in w.columns:
            d=w.anchored_candidate-w[name]
            pairs.append(dict(pool=pool,reference=name,anchored_delta=d.mean(),wins=int((d>1e-12).sum()),ties=int((abs(d)<1e-12).sum()),losses=int((d< -1e-12).sum()),repeat_deltas=d.tolist()))
    summary.to_csv(OUT/'cv_summary.csv');pd.DataFrame(pairs).to_csv(OUT/'paired_vs_anchored.csv',index=False)
    pd.concat(all_predictions).to_csv(OUT/'cv_predictions.csv',index=False)
    dp=pd.concat(diagnostic_predictions,ignore_index=True);dp.to_csv(OUT/'diagnostic_predictions.csv',index=False)
    for name,g in dp[dp.case!='validation'].groupby('name'):
        assert len(g)==98
        diagnostic_rows.append(dict(name=name,case='chronology98',**metric(g[[f'actual_{r}' for r in REASONS]].to_numpy(),g[[f'predicted_{r}' for r in REASONS]].to_numpy())))
    ds=pd.DataFrame(diagnostic_rows);ds.to_csv(OUT/'diagnostic_scores.csv',index=False)
    comparison=summary.reset_index().pivot(index='name',columns='pool',values='mean_f1')
    for case in ['validation','chronology98']:comparison[case]=ds[ds.case==case].set_index('name').sample_f1
    comparison['reported_kaggle']=pd.Series({'anchored_candidate':.89636,'original_ani':.89423,'previous_ensemble':.89401,'lda_ensemble':.89401})
    comparison.to_csv(OUT/'comparison.csv')
    save(OUT/'fit_audits.json',audits);save(OUT/'control_checks.json',control_checks)
    for p,h in SOURCES.items():assert sha(p)==h,p
    save(OUT/'provenance.json',dict(sources=SOURCES,anchored_cv_fits=80,current_ani_control_fits=80,additional_anchored_fits=4,
                                  no_recipe_changes=True,no_new_test_predictions=True,all_source_hashes_unchanged=True,
                                  original_ani_score_access='Reported .89423 belongs to the original transductive CSV; local rows use its matched inductive adaptation.'))
    print(comparison.to_string(),flush=True)
    print(pd.DataFrame(pairs).query("reference=='lda_ensemble'").to_string(index=False),flush=True)


if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/current_leaders_grouped_v1/snapshots/compare_current_leaders.py


## part1/experiments/current_leaders_grouped_v1/snapshots/paired_models.py

In [73]:
%%writefile _source_export/part1/experiments/current_leaders_grouped_v1/snapshots/paired_models.py
"""Small controlled changes to current Ani and the trusted two-pass recipe."""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, normalize

from simple_models import decide

ANI = dict(C=.1, pseudo_weight=1., balance_source='augmented', binary_neighbors=False,
           lda_weight=.2, lda_source='augmented', gmm_weight=.2, rounds=2)
TRUSTED = dict(C=.3, pseudo_weight=.25, balance_source='verified', binary_neighbors=True,
               lda_weight=0., gmm_weight=0., rounds=2)


def prepare(features, use_gmm=True):
    """Only features explicitly permitted by the outer/inner fitting split enter."""
    scaler=StandardScaler().fit(features)
    prep=dict(scaler=scaler)
    if use_gmm:
        Z=scaler.transform(features)
        pca=PCA(30,random_state=0).fit(Z)
        gmm=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(pca.transform(Z))
        assert gmm.converged_
        prep.update(pca=pca,gmm=gmm)
    return prep


def expand(Y):
    row, reason=np.nonzero(Y)
    weight=1./Y.sum(axis=1)[row]
    return row,reason,weight


def lda_fit(Z,Y):
    row,reason,_=expand(Y)
    k=len(np.unique(reason))
    return LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(k)/k).fit(Z[row],reason)


class PairedModel:
    def __init__(self,spec):
        self.spec=dict(spec)

    def fit(self,X,Y,U,preprocessor=None):
        s=self.spec
        # Preserve each reference's scaler summation order exactly. Both contain
        # the same permitted rows; the GMM reference uses original CSV order.
        self.preprocessor_=(preprocessor or prepare(np.vstack([X,U]),True)) if s.get('gmm_weight',0)>0 else prepare(np.vstack([X,U]),False)
        self.scaler_=self.preprocessor_['scaler']
        Z,V=self.scaler_.transform(X),self.scaler_.transform(U)
        row,reason,weights=expand(Y)
        verified_counts=(Y/Y.sum(axis=1,keepdims=True)).sum(axis=0)
        if s.get('gmm_weight',0):
            pca,gmm=self.preprocessor_['pca'],self.preprocessor_['gmm']
            Rk=gmm.predict_proba(pca.transform(Z));Ru=gmm.predict_proba(pca.transform(V))
            Ra=Rk
        self.base_projection_=None
        if s.get('base_components'):
            self.base_projection_=PCA(s['base_components'],random_state=0).fit(np.vstack([Z,V]))
            Z,V=self.base_projection_.transform(Z),self.base_projection_.transform(V)
        self.lda_projection_=None
        if s.get('lda_components'):
            self.lda_projection_=PCA(s['lda_components'],random_state=0).fit(np.vstack([Z,V]))
        self.known_lda_=lda_fit(self._lda_features(Z),Y) if s.get('lda_weight',0)>0 and s.get('lda_source')=='verified' else None
        Za,Ya=Z,Y
        self.audit_=[]
        self.generated_mask_=np.zeros(len(U),bool)
        self.generated_labels_=np.empty((0,5),int)
        for step in range(s.get('rounds',2)+1):
            row,reason,weights=expand(Ya)
            counts=(np.bincount(reason,weights=weights,minlength=5)+1e-9) if s.get('balance_source','augmented')=='augmented' else verified_counts
            weights=weights*counts.mean()/counts[reason] if s.get('balance_source','augmented')=='augmented' else (counts.mean()/counts[reason])/Ya.sum(axis=1)[row]
            weights*=np.where(row<len(X),1.,s.get('pseudo_weight',1.))
            self.lr_=LogisticRegression(C=s.get('C',.1),max_iter=3000,tol=s.get('tol',1e-4)).fit(Za[row],reason,sample_weight=weights)
            self.lda_=None
            if s.get('lda_weight',0)>0:
                self.lda_=self.known_lda_ if self.known_lda_ is not None else lda_fit(self._lda_features(Za),Ya)
            self.neighbor_X_=normalize(Za)
            self.neighbor_Y_=Ya.copy() if s.get('binary_neighbors',False) else Ya/Ya.sum(axis=1,keepdims=True)
            self.audit_.append(dict(step=step,known=len(X),generated=len(Za)-len(X),known_weight=float(weights[row<len(X)].sum()),generated_weight=float(weights[row>=len(X)].sum())))
            if step==s.get('rounds',2):break
            P=self._base(V)
            if s.get('gmm_weight',0):
                T=Ya/Ya.sum(axis=1,keepdims=True)
                rates=(Ra.T@T+T.mean(axis=0))/(Ra.sum(axis=0)[:,None]+1)
                P=(1-s['gmm_weight'])*P+s['gmm_weight']*(Ru@rates)
            keep=P.max(axis=1)>=s.get('confidence',.8)
            self.generated_mask_=keep
            self.generated_labels_=decide(P[keep],.3)
            Za=np.vstack([Z,V[keep]]);Ya=np.vstack([Y,self.generated_labels_])
            if s.get('gmm_weight',0):Ra=np.vstack([Rk,Ru[keep]])
        return self

    def _base(self,Z):
        lr=np.zeros((len(Z),5));lr[:,self.lr_.classes_]=self.lr_.predict_proba(Z)
        similarity=normalize(Z)@self.neighbor_X_.T
        neighbors=np.argsort(-similarity,axis=1)[:,:self.spec.get('k',10)]
        weight=np.take_along_axis(similarity,neighbors,axis=1).clip(0)**2
        local=(weight[:,:,None]*self.neighbor_Y_[neighbors]).sum(axis=1)/(weight.sum(axis=1,keepdims=True)+1e-9)
        wknn=self.spec.get('knn_weight',.3);wlda=self.spec.get('lda_weight',0.)
        P=(1-wknn-wlda)*lr+wknn*local
        if self.lda_ is not None:
            ld=np.zeros_like(P);ld[:,self.lda_.classes_]=self.lda_.predict_proba(self._lda_features(Z))
            P+=wlda*ld
        return P

    def _lda_features(self,Z):
        return self.lda_projection_.transform(Z) if self.lda_projection_ is not None else Z

    def predict_proba(self,X):
        Z=self.scaler_.transform(X)
        if self.base_projection_ is not None:Z=self.base_projection_.transform(Z)
        return self._base(Z)
    def predict(self,X):return decide(self.predict_proba(X),self.spec.get('cutoff',.3))


def first_specs():
    return [dict(ANI,name='ani_exact_control'),dict(TRUSTED,name='trusted_exact_control'),
            dict(TRUSTED,name='trusted_lda_verified',lda_weight=.2,lda_source='verified'),
            dict(TRUSTED,name='trusted_lda_augmented',lda_weight=.2,lda_source='augmented'),
            dict(TRUSTED,name='trusted_lda_verified_C0.1',lda_weight=.2,lda_source='verified',C=.1),
            dict(TRUSTED,name='trusted_lda_augmented_C0.1',lda_weight=.2,lda_source='augmented',C=.1),
            dict(TRUSTED,name='trusted_lda_verified_w0.1',lda_weight=.1,lda_source='verified'),
            dict(ANI,name='ani_without_gmm',gmm_weight=0.),
            dict(ANI,name='ani_trust_and_binary',pseudo_weight=.25,balance_source='verified',binary_neighbors=True)]

Overwriting _source_export/part1/experiments/current_leaders_grouped_v1/snapshots/paired_models.py


## part1/experiments/derived_benchmark_audit_v1/analyze.py

In [74]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/analyze.py
"""Rescore the fixed retrospective panel; no model fitting or metric selection."""
from itertools import combinations
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import precision_recall_fscore_support, f1_score

HERE = Path(__file__).resolve().parent
PART = HERE.parents[1]
LABELS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']
NBOOT, SEED = 5000, 20261107


def save(name, obj):
    (HERE / name).write_text(json.dumps(obj, indent=2, default=lambda v: v.item()) + '\n')


def sign(v):
    return 0 if abs(v) < 1e-12 else int(np.sign(v))


def ranking(panel, metric):
    correct = wrong = tied = kaggle_ties = 0
    for a, b in combinations(panel.index, 2):
        actual = sign(panel.loc[a, 'kaggle'] - panel.loc[b, 'kaggle'])
        predicted = sign(panel.loc[a, metric] - panel.loc[b, metric])
        if not actual:
            kaggle_ties += 1
        elif not predicted:
            tied += 1
        elif actual == predicted:
            correct += 1
        else:
            wrong += 1
    return dict(n=len(panel), spearman=float(spearmanr(panel.kaggle, panel[metric]).statistic),
                kendall_tau_b=float(kendalltau(panel.kaggle, panel[metric]).statistic),
                correctly_ordered=correct, reversed=wrong, benchmark_ties=tied,
                kaggle_ties_excluded=kaggle_ties, comparable_pairs=correct + wrong + tied,
                top_by_benchmark='|'.join(panel.index[panel[metric] >= panel[metric].max() - 1e-12]))


def main():
    comp = pd.read_csv(HERE / 'comparison.csv').set_index('model')
    models = list(comp.index)
    metrics = list(comp.columns.drop('kaggle'))
    rf = pd.read_csv(HERE / 'per_trial_f1.csv.gz')
    cases = json.loads((HERE / 'splits.json').read_text())
    full = pd.concat([pd.read_csv(PART / 'data/part1_train.csv'),
                      pd.read_csv(PART / 'data/part1_val.csv')], ignore_index=True).set_index('ID')
    gold = full[full[LABELS].ge(0).all(axis=1)]
    group_hash = pd.util.hash_pandas_object(gold[[f'emb_{i}' for i in range(150)]].round(10), index=False)
    group_pos = pd.Series(pd.factorize(group_hash)[0], index=gold.index)
    ngroup = group_pos.nunique()
    rng = np.random.default_rng(SEED)
    weights = rng.multinomial(ngroup, np.full(ngroup, 1 / ngroup), size=NBOOT)

    panels = {'all_six': comp, 'exclude_transductive_ani': comp.drop('original_ani'),
              'top_four': comp.loc[models[2:]]}
    panels.update({f'leave_out_{name}': comp.drop(name) for name in models})
    rankings = [dict(panel=name, metric=m, **ranking(panel, m))
                for name, panel in panels.items() for m in metrics]
    pd.DataFrame(rankings).to_csv(HERE / 'rank_agreement.csv', index=False)
    pairs = []
    for a, b in combinations(models, 2):
        for metric in metrics:
            pairs.append(dict(first=a, second=b, metric=metric,
                              kaggle_delta=comp.loc[b, 'kaggle'] - comp.loc[a, 'kaggle'],
                              benchmark_delta=comp.loc[b, metric] - comp.loc[a, metric],
                              kaggle_order=sign(comp.loc[b, 'kaggle'] - comp.loc[a, 'kaggle']),
                              benchmark_order=sign(comp.loc[b, metric] - comp.loc[a, metric])))
    pd.DataFrame(pairs).to_csv(HERE / 'pairwise_orderings.csv', index=False)

    # Each observed embedding group gets one shared bootstrap multiplicity across
    # every appearance in A/B, repeated CV, all metrics and all model recipes.
    # Fits stay fixed: these intervals do not include retraining/search uncertainty.
    def boot_component(frame):
        perrow = frame.groupby(['ID', 'model']).f1.mean().unstack('model')[models]
        counts = np.bincount(group_pos.loc[perrow.index], minlength=ngroup)
        sums = np.zeros((ngroup, len(models)))
        np.add.at(sums, group_pos.loc[perrow.index].to_numpy(), perrow.to_numpy())
        denominator = weights @ counts
        assert (denominator > 0).all()
        return (weights @ sums) / denominator[:, None]

    b = {}
    for metric, track, kind in [('dylan_A', 'dylan', 'A'), ('grouped_train', 'canonical', 'train'),
                               ('grouped_pooled', 'canonical', 'pooled'), ('validation', 'canonical', 'validation'),
                               ('chronology98', 'canonical', 'chronology')]:
        b[metric] = boot_component(rf[(rf.track == track) & (rf.kind == kind)])
    for metric, track in [('dylan_B', 'dylan'), ('strict_B', 'strict_B')]:
        bycut = [boot_component(rf[(rf.track == track) & (rf.cut == cut)]) for cut in (2004, 2005, 2006)]
        b[metric] = sum(bycut) / 3
        if metric == 'dylan_B':
            b['dylan_B2005'] = bycut[1]
    b['dylan_J'] = (b['dylan_A'] + b['dylan_B']) / 2
    b['aligned_J'] = (b['grouped_pooled'] + b['strict_B']) / 2
    b['current_J_diagnostic'] = (b['grouped_pooled'] + b['chronology98']) / 2
    intervals, deltas = [], []
    for metric in metrics:
        for i, model in enumerate(models):
            lo, hi = np.quantile(b[metric][:, i], [.025, .975])
            intervals.append(dict(model=model, metric=metric, estimate=comp.loc[model, metric],
                                  bootstrap_se=b[metric][:, i].std(ddof=1), ci_low=lo, ci_high=hi))
        for i, j in combinations(range(len(models)), 2):
            d = b[metric][:, j] - b[metric][:, i]
            lo, hi = np.quantile(d, [.025, .975])
            deltas.append(dict(first=models[i], second=models[j], metric=metric,
                               delta=comp.loc[models[j], metric] - comp.loc[models[i], metric],
                               bootstrap_se=d.std(ddof=1), ci_low=lo, ci_high=hi,
                               bootstrap_positive_fraction=(d > 1e-12).mean()))
    pd.DataFrame(intervals).to_csv(HERE / 'conditional_intervals.csv', index=False)
    pd.DataFrame(deltas).to_csv(HERE / 'paired_differences.csv', index=False)
    save('bootstrap_protocol.json', dict(replicates=NBOOT, seed=SEED, groups=int(ngroup), rows=len(gold),
         unit='Embedding group, shared multiplicities across models, repeats and overlapping cuts',
         estimates='Per-row average across repeated fits; B is equal mean of cut-specific weighted means',
         limitations='Conditional on fitted predictions. Not a test of Kaggle gaps; excludes retraining, search and unseen-distribution uncertainty. No bootstrap draws retained.'))

    # Preserve the native SE approximation for comparison, without treating A/B as independent.
    native_se = []
    for model in models:
        f = rf[rf.model == model]
        a = f[(f.track == 'dylan') & (f.kind == 'A')].groupby('ID').f1.mean()
        a_se = a.std(ddof=1) / np.sqrt(len(a))
        bs = [g.f1.std(ddof=1) / np.sqrt(len(g))
              for _, g in f[(f.track == 'dylan') & (f.kind == 'B')].groupby('cut')]
        native_se.append(dict(model=model, A_se=a_se, B_se=np.mean(bs),
                              J_se_independence_approx=.5 * np.hypot(a_se, np.mean(bs)),
                              J_se_paired_group_bootstrap=b['dylan_J'][:, models.index(model)].std(ddof=1)))
    pd.DataFrame(native_se).to_csv(HERE / 'native_se_comparison.csv', index=False)

    # Aggregate precision/recall/F1 within each complete OOF repeat (not folds).
    aggregate = []
    def model_metrics(chosen, model):
        ys, ps = [], []
        for case in chosen:
            with np.load(HERE / 'results' / f'{case["key"]}.npz') as z:
                ys.append(z['truth']); ps.append(z[model + '_predictions'])
        y, p = np.concatenate(ys), np.concatenate(ps)
        pr, re, f1, support = precision_recall_fscore_support(y, p, zero_division=0)
        return dict(sample_f1=f1_score(y, p, average='samples', zero_division=0), macro_f1=float(f1.mean()),
                    **{f'{key}_{label}': float(v[j]) for key, v in [('precision', pr), ('recall', re), ('f1', f1), ('support', support)] for j, label in enumerate(LABELS)})
    for model in models:
        for track, kind in [('dylan', 'A'), ('dylan', 'B'), ('strict_B', 'B'), ('canonical', 'train'),
                            ('canonical', 'pooled'), ('canonical', 'validation'), ('canonical', 'chronology')]:
            chosen = [c for c in cases if c['track'] == track and c['kind'] == kind]
            split_key = 'repeat' if kind in ['A', 'train', 'pooled'] else 'cut' if kind == 'B' else None
            chunks = [[c for c in chosen if c[split_key] == r] for r in sorted({c[split_key] for c in chosen})] if split_key else [chosen]
            summaries = pd.DataFrame([model_metrics(chunk, model) for chunk in chunks])
            aggregate.append(dict(model=model, track=track, kind=kind, blocks=len(chunks),
                                  repeat_sd_population=summaries.sample_f1.std(ddof=0) if split_key == 'repeat' else None,
                                  **summaries.mean().to_dict()))
    pd.DataFrame(aggregate).to_csv(HERE / 'aggregate_label_metrics.csv', index=False)
    repeat = pd.read_csv(HERE / 'repeat_scores.csv')
    repeat_comparisons = []
    for kind in ['A', 'train', 'pooled']:
        tab = repeat[repeat.kind == kind].pivot(index='repeat', columns='model', values='sample_f1')
        for model in models:
            d = tab[model] - tab.anchored
            repeat_comparisons.append(dict(model=model, reference='anchored', kind=kind, repeats=len(d),
                                           mean_delta=d.mean(), delta_sd_population=d.std(ddof=0),
                                           wins=(d > 1e-12).sum(), ties=(abs(d) <= 1e-12).sum(), losses=(d < -1e-12).sum()))
    pd.DataFrame(repeat_comparisons).to_csv(HERE / 'paired_repeat_summary.csv', index=False)

    audit = []
    for c in cases:
        audit.append(dict(case=c['key'], track=c['track'], kind=c['kind'], cut=c.get('cut'),
                          gold=len(c['train_gold']), masked=len(c['masked_ids']), query=len(c['query_ids']),
                          gold_group_overlap=c['gold_group_overlap'], masked_group_overlap=c['masked_group_overlap'],
                          masked_at_or_after_query_start=c['future_masked']))
    pd.DataFrame(audit).to_csv(HERE / 'boundary_audit.csv', index=False)
    A = [c for c in cases if c['track'] == 'dylan' and c['kind'] == 'A']
    B = [c for c in cases if c['track'] == 'dylan' and c['kind'] == 'B']
    save('boundary_summary.json', dict(
        dylan_A_folds=len(A), A_folds_with_gold_group_overlap=sum(c['gold_group_overlap'] > 0 for c in A),
        A_folds_with_masked_group_overlap=sum(c['masked_group_overlap'] > 0 for c in A),
        B_query_occurrences=sum(len(c['query_ids']) for c in B),
        B_unique_queries=len(set(v for c in B for v in c['query_ids'])),
        B_cuts=[row for row in audit if row['kind'] == 'B']))
    print(pd.DataFrame(rankings).query("panel == 'all_six'").to_string(index=False))


if __name__ == '__main__':
    main()

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/analyze.py


## part1/experiments/derived_benchmark_audit_v1/build_report.py

In [75]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/build_report.py
"""Build the portable HTML report from the checked audit tables."""
from html import escape
import json
from pathlib import Path

import pandas as pd

HERE = Path(__file__).resolve().parent
NAMES = {'dbscan': 'DBSCAN', 'tri_training': 'Tri-training', 'original_ani': 'Original Ani*',
         'previous_ensemble': 'Previous ensemble', 'lda_ensemble': 'LDA ensemble', 'anchored': 'Anchored'}
METRICS = {'dylan_A': 'Dylan A: random CV', 'dylan_B': 'Dylan B: chronology', 'dylan_J': 'Dylan J: ½A + ½B',
           'validation': 'Official validation', 'grouped_train': 'Current grouped Train157',
           'grouped_pooled': 'Current grouped Pooled235', 'chronology98': 'Current chronology98',
           'strict_B': 'Strict Dylan B', 'aligned_J': '½ grouped Pooled235 + ½ strict B',
           'current_J_diagnostic': '½ grouped Pooled235 + ½ chronology98'}


def table(df, places=4):
    return '<div class="scroll">' + df.to_html(index=False, border=0, classes='data',
                                               float_format=lambda x: f'{x:.{places}f}', na_rep='—') + '</div>'


def main():
    comp = pd.read_csv(HERE / 'comparison.csv')
    ranks = pd.read_csv(HERE / 'rank_agreement.csv')
    paired = pd.read_csv(HERE / 'paired_differences.csv')
    bound = pd.read_csv(HERE / 'boundary_audit.csv')
    protocol = json.loads((HERE / 'protocol.json').read_text())
    verified = json.loads((HERE / 'verification.json').read_text())
    main_table = comp[['model', 'kaggle', 'dylan_A', 'dylan_B', 'dylan_J', 'grouped_train', 'grouped_pooled']].copy()
    main_table.model = main_table.model.map(NAMES)
    main_table.columns = ['Recipe', 'Reported Kaggle', 'Dylan A', 'Dylan B', 'Dylan J', 'Grouped Train157', 'Grouped Pooled235']
    ranktable = ranks[(ranks.panel == 'all_six') & ranks.metric.isin(METRICS)].copy()
    ranktable['Correct pairs'] = ranktable.correctly_ordered.astype(str) + '/' + ranktable.comparable_pairs.astype(str)
    ranktable['Metric'] = ranktable.metric.map(METRICS)
    ranktable['Top recipe'] = ranktable.top_by_benchmark.apply(lambda s: ', '.join(NAMES[v] for v in s.split('|')))
    ranktable = ranktable[['Metric', 'spearman', 'Correct pairs', 'benchmark_ties', 'Top recipe']].rename(columns={'spearman': 'Rank correlation', 'benchmark_ties': 'Metric ties'})
    sensitivity = ranks[ranks.panel.isin(['all_six', 'exclude_transductive_ani', 'top_four'])].pivot(index='metric', columns='panel', values='spearman')
    sensitivity = sensitivity.loc[list(METRICS)].reset_index()
    sensitivity.metric = sensitivity.metric.map(METRICS)
    sensitivity.columns = ['Metric', 'All six', 'Exclude Ani: five', 'Top four only']
    delta = paired[(paired['first'] == 'lda_ensemble') & (paired['second'] == 'anchored') & paired.metric.isin(METRICS)].copy()
    delta.metric = delta.metric.map(METRICS)
    delta = delta[['metric', 'delta', 'ci_low', 'ci_high']]
    delta.columns = ['Metric', 'Anchored − LDA', 'Conditional 95% lower', 'Conditional 95% upper']
    bounds = bound[bound.kind == 'B'][['case', 'gold', 'masked', 'query', 'gold_group_overlap', 'masked_group_overlap', 'masked_at_or_after_query_start']].copy()
    bounds.columns = ['Case', 'Gold fit', 'Masked fit', 'Query', 'Gold group overlap', 'Masked group overlap', 'Masked later than fit cutoff']
    supplied = json.loads((HERE / 'source_snapshot/benchmark_results.json').read_text())
    candidate_rows = []
    for name, key in [('Anchored (freshly reproduced)', 'michael_anchored'), ('Dylan cand1 (supplied results)', 'cand1_selected_1se'), ('Dylan cand5 (supplied results)', 'cand5_lr_cs_sld_robust')]:
        r = supplied[key]
        candidate_rows.append({'Recipe / evidence': name, 'A': r['A']['sample_f1'], 'B': r['B']['sample_f1'], 'C': r['C']['sample_f1'], 'J': r['J']})
    artifacts = ''.join(f'<li>{escape(NAMES[r["name"]])}: <a href="../../{escape(r["csv"].removeprefix("part1/"))}">{escape(Path(r["csv"]).name)}</a>; SHA-256 <code>{r["sha256"]}</code>.</li>' for r in protocol['scored_artifacts'])
    html = f'''<!doctype html>
<html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Does Dylan v2's benchmark explain the Kaggle gains?</title>
<style>
:root {{color-scheme:light;--ink:#172e38;--muted:#526873;--line:#d6e0e4;--accent:#066c68}}
body {{font:16px/1.6 system-ui,sans-serif;color:var(--ink);background:#f3f6f7;margin:0}}
main {{max-width:1180px;margin:auto;padding:44px 32px 80px;background:white}}
h1 {{font-size:36px;line-height:1.2;max-width:900px}} h2 {{margin-top:42px;font-size:24px}}
h3 {{font-size:18px;margin-top:28px}} p {{max-width:980px}} a {{color:var(--accent)}}
.meta,.note {{color:var(--muted);font-size:14px}} .verdict {{border-left:5px solid var(--accent);padding:14px 24px;background:#edf7f5}}
.scroll {{overflow-x:auto}} table {{border-collapse:collapse;width:100%;font-size:14px;margin:16px 0}}
th,td {{padding:10px 12px;border-bottom:1px solid var(--line);text-align:right;vertical-align:top}}
th:first-child,td:first-child {{text-align:left}} th {{background:#edf3f5}} tr:hover td {{background:#f5faf9}}
code {{font-size:12px;overflow-wrap:anywhere}} li {{margin:8px 0}} details {{margin-top:22px}} @media print {{body,main{{background:white;padding:10px}}}}
</style><main>
<p class="meta">Part 1 · fixed retrospective audit · six previously scored recipes · {verified['case_prediction_rescores']} recipe evaluations</p>
<h1>Does Dylan v2's benchmark explain the Kaggle gains?</h1>
<div class="verdict"><p><strong>It adds a useful distribution-shift diagnostic, but this audit does not support replacing our current comparison protocol with Dylan's combined J score.</strong>
On these six recipes, J orders 10 of 14 unequal Kaggle pairs correctly; grouped Train157 orders 12.
J also reverses the latest LDA-to-anchored improvement. With the mismatched original Ani artifact excluded, J gets 8 of 9 pairs and Train157 gets 9 of 9.
These are retrospective checks on a small, selected panel—not proof that any metric will predict the next submission.</p></div>

<h2>What changed, and what was tested</h2>
<p>The updated main branch consolidates the historical LDA and New Directions research under <code>part1/research/</code>, with a current testing policy and submission ledger.
Anchored remains the highest recorded user-reported Kaggle result at <strong>0.89636</strong>.
This study starts at main <code>{protocol['base_commit'][:12]}</code> in the separate worktree <code>AML-Midterm-derived-benchmarks</code>, branch <code>codex/part1-derived-benchmark-audit</code>.</p>
<p>Dylan v2 is substantially more complete than v1: it has executable graph pseudo-labeling, calibration, class-prior adjustment, decoding, candidate selection and benchmark scripts.
The question here is whether its evaluation rules explain our historical Kaggle changes. We freshly evaluated six fixed historical recipes on 140 fitting/query contexts each, preserving all final 235-label submission artifacts.
The 140 contexts comprise 50 Dylan random folds, three Dylan chronological cuts, three strict versions of those cuts, and our 84 canonical checks. Official validation is shared between the two protocols.
Components shared by the historical ensembles are fitted once per context; “840 evaluations” is not a count of individual estimator <code>.fit()</code> calls.</p>
<p>No model settings, split assignments, chronology cutoffs or combination weights were tuned to these Kaggle scores.
The existing <a href="../../TESTING_POLICY.md">testing policy</a> remains unchanged. Exact recipes, source/data hashes, versions and exceptions are recorded in <a href="protocol.json">the frozen protocol</a>.</p>

<h2>What the derived scores actually mean</h2>
<ul>
<li><strong>A:</strong> five-fold CV repeated ten times on all 235 labeled rows, stratified by label combination, seed 1. This is ungrouped.</li>
<li><strong>B:</strong> average sample F1 across three tests: fit gold labels before 2004, 2005 or 2006 and evaluate all later gold labels. All 1,014 originally masked training rows remain available.</li>
<li><strong>C:</strong> the original 157-label training to 78-label validation check. Dylan logs C as a supplementary check; it does not enter J.</li>
<li><strong>J = ½A + ½B:</strong> an equal-weight preference for random-split and chronological performance. It is not a Kaggle score, probability of generalizing, or calibrated estimate of test F1.</li>
<li><strong>One-standard-error selection:</strong> favors a simpler candidate close to the best J. It is a selection rule, not an additional benchmark. Its complexity ordering was built for Dylan's own search space.</li>
</ul>
<p>B's three cuts contain 293 evaluation occurrences but only <strong>109 unique labeled rows</strong>. They are overlapping tests, not three independent replications.
All 50 A folds contain at least one gold embedding group crossing the fit/query boundary and at least one masked group matching a query group.
Our existing grouped CV prevents those overlaps. Matching the splits therefore does not make the access assumptions identical.</p>

<h2>Fresh results for the scored recipes</h2>
{table(main_table, 5)}
<p class="note">All scores are sample-wise F1. Kaggle numbers are recorded user reports, not new uploads. Original Ani* is a known mismatch: its .89423 scored CSV is transductive, while the historical local adapter is inductive. Its exclusion is therefore the cleaner sensitivity panel. The two .89401 scores are tied at their recorded precision.</p>
<p>The latest contrast is decisive for interpreting the claim: LDA ensemble → anchored improved the recorded Kaggle score by <strong>+0.00235</strong>, while Dylan J fell by <strong>−0.00145</strong>.
J correctly recognizes the large DBSCAN → tri-training improvement, but reverses tri-training → original Ani; that second contrast also has the access mismatch noted above.
The earlier ensemble → LDA ensemble change raises J by 0.00274 despite equal recorded Kaggle scores.</p>

<h2>How well do the rankings agree?</h2>
{table(ranktable, 3)}
<p>Rank correlation is Spearman's rho; higher means more agreement with the reported Kaggle ordering. Pair counts exclude the one Kaggle tie and give no credit to an unresolved benchmark tie.
Official validation is a poor ranking signal in this panel: its best recipe includes DBSCAN, the lowest Kaggle scorer. That supports keeping multiple checks visible; it does not establish a new universally correct combined metric.</p>
<h3>Sensitivity to which recipes are included</h3>
{table(sensitivity, 3)}
<p>Excluding original Ani removes the artifact/access mismatch. Dylan J then has exactly the same ranking as A and grouped Pooled235, so it adds no ordering benefit on those five recipes.
Among only the four close leaders, no metric gives strong agreement. The tiny recorded gaps should not be treated as a reliably known population ordering.
Across every leave-one-recipe-out panel, grouped Train157 has a higher rank correlation than J; J ranges from 0.103 to 0.821.
Full Kendall tau, all pairwise differences and leave-one-out results are saved in <a href="rank_agreement.csv">rank_agreement.csv</a> and <a href="pairwise_orderings.csv">pairwise_orderings.csv</a>.</p>

<h2>The chronology conclusion depends on the fitting boundary</h2>
<p>We also held Dylan's three query populations fixed while purging duplicate embedding groups from fitting gold/masked rows and requiring every fitting masked row to predate the cutoff.
This jointly changes group and time access; it is not an isolated ablation of either factor.</p>
{table(bounds, 0)}
<p>Under native B, tri-training is best and anchored trails LDA. Under strict B, anchored is best: <strong>0.85332 vs 0.84817</strong> for LDA.
Combining strict B with grouped Pooled235 at the same fixed half/half weight also favors anchored (0.88734 vs 0.88490).
That diagnostic has better retrospective agreement than native J, but still does not outperform grouped Train157 in the six-recipe panel.</p>
<p>The native 2005 cut accounts for much of anchored's disadvantage: it loses to LDA by 0.02041 there while winning the 2004 and 2006 cuts.
On that 98-row cut, anchored has better row F1 on 5 rows, worse on 8, and equal on 85. Under the strict 2005 boundary it wins 6, loses 2, and ties 90.
Chronology is a useful stress test; these results do not justify treating one particular masked-data access rule as the definition of generalization.</p>

<h2>Small leader differences remain uncertain</h2>
{table(delta, 5)}
<p>Intervals use 5,000 paired bootstrap resamples of the 222 embedding groups, with the same weights for all models and every appearance of each group across repeats and overlapping cuts.
They are conditional on the fitted predictions; they do not include model-search uncertainty, refitting variation or an unseen target distribution.</p>
<p>Anchored minus LDA has J = −0.00145 with a conditional interval of approximately <strong>[−0.01370, +0.01217]</strong>.
The benchmark does not clearly separate the two leaders. Anchored's J standard error is 0.02029 under this group bootstrap, versus Dylan's 0.01751 approximation, which omits A/B covariance.
A different random seed on the same 235 labels is a split-stability check, not an untouched confirmation set.</p>

<h2>What this says about Dylan's new candidates</h2>
{table(pd.DataFrame(candidate_rows), 5)}
<p>These reproduce the supplied claim: anchored is stronger on A and C, while cand1/cand5 are stronger on B and J.
Only anchored was freshly rerun in this candidate table; cand1/cand5 values come from <a href="source_snapshot/benchmark_results.json">Dylan's supplied results</a>.
Neither has a recorded Kaggle score in our ledger, so neither enters the rank-correlation analysis.</p>
<p>The updated package also proposes equal-weight anchored/cand1 blends, with supplied J values around .896–.897.
Those blends likewise have no recorded Kaggle outcome here and were not freshly evaluated in this historical-recipe audit.</p>
<p>Dylan's candidates use held-out and actual test features for label-free preprocessing by default, while the supplied anchored run excludes those rows from fitting.
That is an explicit transductive comparison, not our default inductive test. It can be explored separately but does not establish that a new model beats the submitted one under matched access.</p>
<p>The claim that anchored loses chronology <em>because</em> it lacks prior correction is not isolated by that comparison: the candidates also change pseudo-labeling, classifier, calibration, decoding and feature access.
Cand1 uses balanced class priors, not the SLD head used by cand5. The supplied methodology itself reports that its search procedure can fail under a chronological outer split and that B may exaggerate target shift.
Its broad claim that public score gaps below about 0.05 are noise is not established by this audit; we lack the public evaluation membership and labels needed to quantify those paired gaps.</p>

<h2>Recommendation</h2>
<ol>
<li><strong>Keep anchored as the scored leader and retain both grouped CV pools as the primary comparison.</strong> Do not switch to Train157 alone because it won this retrospective ranking exercise.</li>
<li><strong>Keep chronology as a visible secondary diagnostic.</strong> Strict B adds an earlier 2004 stress cut and an alternate fixed-origin view. Keep each cut's result visible; do not count the overlapping cuts as independent confirmations.</li>
<li><strong>Do not promote using J alone.</strong> Its half/half weighting encodes a preference for these particular shift tests; this panel does not show that it improves Kaggle selection.</li>
<li><strong>For a future candidate comparison, freeze candidates and access first.</strong> Compare anchored and Dylan v2 under the same grouped protocol, report any transductive experiment separately, and record the next scored artifact before revising the evaluation rule.</li>
</ol>
<p>There is also a hard limitation: the same DBSCAN recipe scored .87500 after a 157-label final fit and .86452 after its 235-label final fit.
A fixed-recipe CV score cannot distinguish those two final artifacts without an explicit training-size experiment.
This audit uses only the 235-label version in its primary panel to avoid counting one recipe twice; it does not explain every historical Kaggle movement.</p>

<h2>Verification and retained evidence</h2>
<p>All 840 stored predictions were independently rescored. All 320 saved grouped fold controls reproduce their historical label predictions and aligned scores.
The four retained leaders match all 16 canonical aggregate scores; DBSCAN and tri-training match all eight archived aggregate checks.
Anchored reproduces Dylan's supplied A/B/C/J exactly within floating-point tolerance. All {verified['protected_source_hashes']} frozen source/data/artifact hashes passed, and no Kaggle test inputs were used for model fitting or prediction in this audit.</p>
<p>A verifier-only correction handled Ani's archived raw scores (cutoff .3) versus its centered ensemble-adapter scores (cutoff .5). Label predictions matched before the fix; no fitted output or metric changed.
The original and corrected runner hashes and failure details are retained in <a href="failure_notes.json">failure_notes.json</a> and the protocol amendment.</p>
<p>The approximately 9 MB study keeps compact matched-control predictions, exact splits, summaries and source snapshots. It retains no fitted estimators or duplicated datasets.
See <a href="README.md">replay instructions</a>, <a href="verification.json">verification.json</a>, <a href="comparison.csv">comparison.csv</a>,
<a href="aggregate_label_metrics.csv">per-label metrics</a>, <a href="paired_repeat_summary.csv">paired repeat results</a>,
<a href="conditional_intervals.csv">conditional intervals</a>, and <a href="bootstrap_protocol.json">bootstrap protocol</a>.</p>
<details><summary>Exact scored artifacts and source evidence</summary><ul>{artifacts}</ul>
<p>Supplied definitions: <a href="source_snapshot/evaluate.py">evaluation code</a>, <a href="source_snapshot/context.py">feature access</a>,
<a href="source_snapshot/methodology.md">methodology</a>, <a href="source_snapshot/benchmark_README.md">benchmark claims</a>.
The copied source files are unchanged evidence from Dylan v2, not newly endorsed policy.</p></details>
</main></html>'''
    (HERE / 'report.html').write_text(html)
    print(HERE / 'report.html')


if __name__ == '__main__':
    main()

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/build_report.py


## part1/experiments/derived_benchmark_audit_v1/source_snapshot/benchmark_run.py

In [76]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/benchmark_run.py
"""Benchmark Michael's AnchoredReasonClassifier (part1/anchored_solution.py) on the framework's
exact A/B/C folds (R=10, seed=1, same as the v2 leaderboard), paired against cand1 and cand5."""
import json, sys, time, warnings
import numpy as np
sys.path.insert(0, "../midterm/part1")
from joblib import Parallel, delayed
from p1lib.data import load, metrics, row_f1, LABELS
from p1lib.evaluate import make_folds, evaluate, paired_se
from anchored_solution import AnchoredReasonClassifier

D = load()
EMB = [f"emb_{i}" for i in range(150)]
X = D.df[EMB].to_numpy(float)
folds = make_folds(D, ("A", "B", "C"), R=10, seed=1)
unl = D.idx["unl"]

def run(f):
    warnings.simplefilter("ignore")
    fit_rows = np.sort(np.r_[f["train"], unl])            # permitted rows only, original CSV order
    Y = np.full((len(fit_rows), 5), -100)
    is_lab = np.isin(fit_rows, f["train"])
    Y[is_lab] = D.Y[fit_rows[is_lab]]
    t = time.time()
    try:
        m = AnchoredReasonClassifier().fit(X[fit_rows], Y)
        return f["name"], m.predict(X[f["eval"]]), time.time() - t, None
    except Exception as e:
        return f["name"], None, time.time() - t, repr(e)

outs = {n: (yp, s, err) for n, yp, s, err in Parallel(n_jobs=16)(delayed(run)(f) for f in folds)}
errs = {n: e for n, (_, _, e) in outs.items() if e}
print("fold errors:", errs if errs else "none")

res = {}
A = [f for f in folds if f["protocol"] == "A"]
rows = np.unique(np.concatenate([f["eval"] for f in A])); pos = {r: i for i, r in enumerate(rows)}
Yt = D.Y[rows]; preds = []
for r in sorted({f["repeat"] for f in A}):
    Yp = np.zeros((len(rows), 5), int)
    for f in A:
        if f["repeat"] == r: Yp[[pos[i] for i in f["eval"]]] = outs[f["name"]][0]
    preds.append(Yp)
rm = [metrics(Yt, p) for p in preds]
rf = np.mean([row_f1(Yt, p) for p in preds], axis=0)
res["A"] = {k: float(np.mean([m[k] for m in rm])) for k in rm[0] if k != "n"}
res["A"]["sample_f1_se"] = float(rf.std(ddof=1) / np.sqrt(len(rf))); res["rowf1_A"] = rf.tolist()
bm, rB = [], []
for f in [f for f in folds if f["protocol"] == "B"]:
    m = metrics(D.Y[f["eval"]], outs[f["name"]][0]); m["cut"] = f["cut"]; bm.append(m)
    rB.extend(row_f1(D.Y[f["eval"]], outs[f["name"]][0]).tolist())
res["B"] = {k: float(np.mean([m[k] for m in bm])) for k in bm[0] if k not in ("n", "cut")}
res["B_cuts"] = {m["cut"]: m["sample_f1"] for m in bm}; res["rowf1_B"] = rB; res["B_ncuts"] = len(bm)
fC = [f for f in folds if f["protocol"] == "C"][0]
res["C"] = metrics(D.Y[fC["eval"]], outs["C"][0])
res["J"] = 0.5 * res["A"]["sample_f1"] + 0.5 * res["B"]["sample_f1"]
res["J_se"] = 0.5 * float(np.hypot(res["A"]["sample_f1_se"], res["B"]["sample_f1_se"]))
res["fit_secs_mean"] = float(np.mean([s for _, s, _ in outs.values()]))

cands = {}
for c in ["cand1_selected_1se", "cand5_lr_cs_sld_robust"]:
    cfg = json.load(open(f"../midterm/part1/submissions/{c}.manifest.json"))["config"]
    cands[c] = evaluate(cfg, D, R=10, seed=1, n_jobs=16, bootstrap=False)

def line(name, r):
    return (f"{name:28s} A={r['A']['sample_f1']:.3f}±{r['A']['sample_f1_se']:.3f}  B={r['B']['sample_f1']:.3f}  "
            f"C={r['C']['sample_f1']:.3f}  J={r['J']:.3f}  macroA={r['A']['macro_f1']:.3f}  macroC={r['C']['macro_f1']:.3f}")
print(line("michael_anchored", res)); print("  B by cut:", {k: round(v, 3) for k, v in res["B_cuts"].items()})
print("  C per-label:", {l: round(res['C'][f'f1_{l}'], 3) for l in LABELS})
for c, r in cands.items():
    print(line(c, r)); print(f"  J diff vs michael: {r['J'] - res['J']:+.3f} (paired SE {paired_se(r, res):.3f})")
    print(f"  A diff: {r['A']['sample_f1'] - res['A']['sample_f1']:+.3f}  B diff: {r['B']['sample_f1'] - res['B']['sample_f1']:+.3f}  C diff: {r['C']['sample_f1'] - res['C']['sample_f1']:+.3f}")
json.dump({"michael_anchored": {k: v for k, v in res.items() if not k.startswith("rowf1")},
           **{c: {k: r[k] for k in ("A", "B", "C", "J", "J_se")} for c, r in cands.items()}},
          open("results.json", "w"), indent=2, default=float)

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/benchmark_run.py


## part1/experiments/derived_benchmark_audit_v1/source_snapshot/context.py

In [77]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/context.py
"""Fold context: which rows are labeled-for-training, which are predicted, which are unlabeled.

Every component receives a `Ctx` and may only read labels of `ctx.train`.  Features of
`ctx.unsup` rows may be used by label-free steps (scaler, PCA, clustering, graph nodes).

transductive=True  -> unsup = all 1,561 rows (train + val + test features, no labels)
transductive=False -> unsup = all rows except the eval rows and the test rows
"""
from __future__ import annotations

from dataclasses import dataclass, replace

import numpy as np


@dataclass
class Ctx:
    D: object
    train: np.ndarray            # global idx of labeled training rows (labels visible)
    eval: np.ndarray             # global idx of rows to predict (labels hidden)
    pool: np.ndarray             # global idx of the pseudo-label pool (default: 1,014 unlabeled)
    unsup: np.ndarray            # global idx of rows whose features label-free steps may use
    transductive: bool = True
    seed: int = 0
    name: str = ""
    depth: int = 0               # nesting depth (0 = outer fold)

    @property
    def y_train(self):
        return self.D.y[self.train]

    @property
    def sld_target(self):
        """Rows SLD estimates the target prior on: the eval batch when transductive, else the
        unlabeled pool (inductive: we may not look at the eval/test features)."""
        return self.eval if self.transductive else self.pool

    def sub(self, train, eval_, name=None):
        """Inner context for nested OOF: same unlabeled pool / feature rows, fewer labels."""
        return replace(self, train=np.asarray(train), eval=np.asarray(eval_),
                       name=name or self.name + "/inner", depth=self.depth + 1)


def make_ctx(D, train, eval_, transductive=True, pool=None, seed=0, name=""):
    train = np.asarray(train)
    eval_ = np.asarray(eval_)
    pool = D.idx["unl"] if pool is None else np.asarray(pool)
    if transductive:
        unsup = np.arange(D.n)
    else:
        banned = np.zeros(D.n, bool)
        banned[eval_] = True
        banned[D.idx["test"]] = True
        unsup = np.where(~banned)[0]
    assert len(np.intersect1d(train, eval_)) == 0, "train/eval overlap"
    assert (D.y[train] >= 0).all(), "training rows must be labeled"
    return Ctx(D=D, train=train, eval=eval_, pool=pool, unsup=unsup,
               transductive=transductive, seed=seed, name=name)

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/context.py


## part1/experiments/derived_benchmark_audit_v1/source_snapshot/data.py

In [78]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/data.py
"""Data loading, label-powerset encoding and metrics for Part 1.

Global row order used everywhere in this package:
    [ part1_train (1171 rows, file order) | part1_val (78) | part1_test (312) ]  -> 1561 rows.
`Data.y` holds the label-combo id (0..7) for labeled rows and -1 for masked/test rows.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field

import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

LABELS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMB = [f"emb_{i}" for i in range(150)]
BITS = np.array([1, 2, 4, 8, 16])

DEFAULT_DATA_DIR = os.environ.get(
    "P1_DATA_DIR",
    os.path.join(os.path.dirname(__file__), "..", "..", "..", "part1", "data"),
)


@dataclass
class Data:
    df: pd.DataFrame                 # all 1561 rows, global order, with a 'split' column
    emb: np.ndarray                  # (1561, 150) raw embeddings
    Y: np.ndarray                    # (1561, 5) labels, -1 where unknown
    y: np.ndarray                    # (1561,) combo id, -1 where unknown
    combos: np.ndarray               # (K,) bitmask of each allowed combo
    CM: np.ndarray                   # (K, 5) binary label matrix of each combo
    idx: dict = field(default_factory=dict)   # named index groups
    year: np.ndarray = None
    catalog: pd.DataFrame = None

    @property
    def K(self) -> int:
        return len(self.combos)

    @property
    def n(self) -> int:
        return len(self.df)

    def combo_names(self):
        return ["+".join(LABELS[j][0] for j in range(5) if self.CM[k, j]) for k in range(self.K)]


def load(data_dir: str | None = None) -> Data:
    d = os.path.abspath(data_dir or DEFAULT_DATA_DIR)
    tr = pd.read_csv(os.path.join(d, "part1_train.csv"))
    va = pd.read_csv(os.path.join(d, "part1_val.csv"))
    te = pd.read_csv(os.path.join(d, "part1_test.csv"))
    cat = pd.read_csv(os.path.join(d, "pathway_catalog.csv"))
    te = pd.concat([te, pd.DataFrame(-1, index=te.index, columns=LABELS)], axis=1)
    tr = tr.assign(split="train")
    va = va.assign(split="val")
    te = te.assign(split="test")
    df = pd.concat([tr, va, te], ignore_index=True).copy()
    Y = df[LABELS].to_numpy().astype(int)
    Y[Y == -100] = -1
    known = (Y >= 0).all(1)
    mask = (Y.clip(0) * BITS).sum(1)
    combos = np.unique(mask[known])
    y = np.full(len(df), -1)
    y[known] = np.searchsorted(combos, mask[known])
    CM = np.array([[(c >> j) & 1 for j in range(5)] for c in combos])
    split = df["split"].to_numpy()
    idx = {
        "train": np.where(split == "train")[0],
        "lab": np.where((split == "train") & known)[0],
        "unl": np.where((split == "train") & ~known)[0],
        "val": np.where(split == "val")[0],
        "test": np.where(split == "test")[0],
    }
    idx["labval"] = np.r_[idx["lab"], idx["val"]]
    D = Data(df=df, emb=df[EMB].to_numpy(float), Y=Y, y=y, combos=combos, CM=CM, idx=idx,
             year=df["start_year"].to_numpy(int), catalog=cat)
    assert len(idx["lab"]) == 157 and len(idx["unl"]) == 1014 and len(idx["val"]) == 78 and len(idx["test"]) == 312
    assert (Y[idx["lab"]].sum(1) >= 1).all(), "labeled rows are never all-zero"
    return D


# ----------------------------------------------------------------------------- metrics
def row_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-row F1 between binary label sets (both non-empty here)."""
    inter = (Ytrue * Ypred).sum(1)
    denom = Ytrue.sum(1) + Ypred.sum(1)
    out = np.where(denom > 0, 2 * inter / np.maximum(denom, 1), 1.0)
    return out


def label_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-label (column) binary F1; F1 = 0 when a label has no positives and none predicted."""
    tp = (Ytrue * Ypred).sum(0)
    fp = ((1 - Ytrue) * Ypred).sum(0)
    fn = (Ytrue * (1 - Ypred)).sum(0)
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)


def metrics(Ytrue: np.ndarray, Ypred: np.ndarray) -> dict:
    rf = row_f1(Ytrue, Ypred)
    lf = label_f1(Ytrue, Ypred)
    return {
        "sample_f1": float(rf.mean()),
        "sample_f1_se": float(rf.std(ddof=1) / np.sqrt(len(rf))),
        "macro_f1": float(lf.mean()),
        **{f"f1_{LABELS[j]}": float(lf[j]) for j in range(5)},
        "n": int(len(rf)),
    }


def bootstrap_se(Ytrue: np.ndarray, Ypreds: list[np.ndarray], B: int = 300, seed: int = 0) -> dict:
    """Bootstrap SE (over rows) of macro and per-label F1, averaged over a list of prediction
    matrices for the same rows (e.g. CV repeats) using shared resamples."""
    rng = np.random.default_rng(seed)
    n = len(Ytrue)
    macro, per = [], []
    for _ in range(B):
        b = rng.integers(0, n, n)
        lfs = np.mean([label_f1(Ytrue[b], P[b]) for P in Ypreds], axis=0)
        per.append(lfs)
        macro.append(lfs.mean())
    per = np.array(per)
    return {"macro_f1_se": float(np.std(macro, ddof=1)),
            **{f"f1_{LABELS[j]}_se": float(per[:, j].std(ddof=1)) for j in range(5)}}


def combos_to_labels(D: Data, kidx: np.ndarray) -> np.ndarray:
    return D.CM[kidx]


def write_submission(D: Data, Ypred: np.ndarray, path: str, ids=None) -> pd.DataFrame:
    """Kaggle format: ID,Predicted with space-separated reason names (always >=1)."""
    assert (Ypred.sum(1) >= 1).all(), "every test row needs at least one predicted reason"
    ids = D.df["ID"].to_numpy()[D.idx["test"]] if ids is None else ids
    pred = [" ".join(LABELS[j] for j in range(5) if row[j]) for row in Ypred]
    sub = pd.DataFrame({"ID": ids, "Predicted": pred})
    sub.to_csv(path, index=False, lineterminator="\n")
    return sub

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/data.py


## part1/experiments/derived_benchmark_audit_v1/source_snapshot/evaluate.py

In [79]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/evaluate.py
"""Leak-free evaluation harness.

Protocols (all over the 235 labeled rows = 157 labeled-train + 78 val):
  A : RepeatedStratifiedKFold(5 x R) stratified on the label-combo id         (no shift)
  B : chronological holdout: train start_year < cut, evaluate >= cut         (shift, cut=2005)
  C : official split: train on the 157 labeled-train rows, evaluate on val   (tie-breaker)
Every fold builds a fresh Ctx; everything label-dependent is fit inside `run_model` from
ctx.train labels only.  Label-free fits are reused across folds only when their feature-row
set is identical (transductive mode).

Fold outputs (calibrated combo probabilities BEFORE the head) are cached on disk keyed by the
model part of the config, so SLD strength / decoding are post-hoc and free to vary.
"""
from __future__ import annotations

import hashlib
import json
import os
import time

import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold

from .context import make_ctx
from .data import LABELS, bootstrap_se, label_f1, metrics, row_f1
from .head import apply_head
from .pipeline import model_part, run_model, validate

# fold-output cache lives OUTSIDE the midterm/ folder (regenerable, ~0.5 GB); override with P1_CACHE
CACHE_DIR = os.environ.get("P1_CACHE", os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "..", ".p1_cache", "folds")))
CHRONO_CUT = 2005
CHRONO_CUTS = (2004, 2005, 2006)   # objective uses the mean over these cuts


def cfg_hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:20]


# ----------------------------------------------------------------------------- folds
def make_folds(D, protocols=("A", "B", "C"), R=4, seed=0, rows=None, chrono_cuts=CHRONO_CUTS):
    """rows: restrict to a subset of labeled rows (used by the nested search-overfitting check)."""
    labval = D.idx["labval"] if rows is None else np.asarray(rows)
    y = D.y[labval]
    folds = []
    if "A" in protocols:
        rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=R, random_state=seed)
        for i, (tr, te) in enumerate(rskf.split(np.zeros(len(y)), y)):
            folds.append({"name": f"A{i // 5}_{i % 5}", "protocol": "A", "repeat": i // 5,
                          "train": labval[tr], "eval": labval[te]})
    if "B" in protocols:
        for cut in chrono_cuts:
            yr = D.year[labval]
            folds.append({"name": f"B{cut}", "protocol": "B", "cut": cut,
                          "train": labval[yr < cut], "eval": labval[yr >= cut]})
    if "C" in protocols:
        tr = np.intersect1d(labval, D.idx["lab"])
        te = np.intersect1d(labval, D.idx["val"])
        if len(tr) and len(te):
            folds.append({"name": "C", "protocol": "C", "train": tr, "eval": te})
    return folds


def fold_tag(fold):
    h = hashlib.md5(np.r_[fold["train"], [-1], fold["eval"]].astype(np.int64).tobytes()).hexdigest()[:10]
    return f"{fold['name']}_{h}"


# ----------------------------------------------------------------------------- one fold
def run_fold(cfg, D, fold, use_cache=True):
    mp = model_part(cfg)
    key = cfg_hash(mp)
    path = os.path.join(CACHE_DIR, key, fold_tag(fold) + ".npz")
    if use_cache and os.path.exists(path):
        try:
            z = np.load(path, allow_pickle=False)
            out = {k: z[k] for k in z.files}
            out["T"] = out["T"].tolist()
            return out
        except Exception:
            pass
    tr = cfg.get("transductive", True)
    ctx = make_ctx(D, fold["train"], fold["eval"], transductive=tr, seed=0, name=fold["name"])
    t0 = time.time()
    rows = fold["eval"] if tr else np.r_[fold["eval"], ctx.pool]
    o = run_model(cfg["model"], ctx, rows)
    ne = len(fold["eval"])
    out = {"P_eval": o["P"][:ne].astype(np.float32), "prior": np.asarray(o["prior"], float),
           "T": np.atleast_1d(np.asarray(o["T"], float)), "secs": np.float32(time.time() - t0)}
    if not tr:
        out["P_target"] = o["P"][ne:].astype(np.float32)
    if use_cache:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + f".{os.getpid()}.tmp.npz"
        np.savez_compressed(tmp, **out)
        os.replace(tmp, path)
        mpath = os.path.join(CACHE_DIR, key, "config.json")
        if not os.path.exists(mpath):
            with open(mpath, "w") as f:
                json.dump(mp, f, indent=1, default=str)
    out["T"] = out["T"].tolist()
    return out


def fold_outputs(cfg, D, folds, n_jobs=1, use_cache=True):
    if n_jobs == 1:
        return [run_fold(cfg, D, f, use_cache) for f in folds]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=n_jobs, backend="loky")(delayed(run_fold)(cfg, D, f, use_cache) for f in folds)


# ----------------------------------------------------------------------------- scoring
def score(cfg, D, folds, outs, bootstrap=True):
    """Apply the head to cached fold outputs and compute A/B/C metrics with SEs."""
    head = cfg.get("head", {})
    res = {}
    per = {}
    for f, o in zip(folds, outs):
        P = o["P_eval"].astype(float)
        P = P / P.sum(1, keepdims=True)
        Pt = o.get("P_target")
        Pt = None if Pt is None else Pt.astype(float) / Pt.astype(float).sum(1, keepdims=True)
        Yp, Q, pi = apply_head(P, o["prior"], D.CM, head, Pt)
        per[f["name"]] = (f, Yp, pi)
    # ---- A ----
    A = [v for v in per.values() if v[0]["protocol"] == "A"]
    if A:
        rows = np.unique(np.concatenate([f["eval"] for f, _, _ in A]))
        pos = {r: i for i, r in enumerate(rows)}
        reps = sorted({f["repeat"] for f, _, _ in A})
        Ytrue = D.Y[rows]
        Ypreds = []
        for r in reps:
            Yp = np.zeros((len(rows), 5), int)
            for f, yp, _ in A:
                if f["repeat"] == r:
                    Yp[[pos[i] for i in f["eval"]]] = yp
            Ypreds.append(Yp)
        rep_metrics = [metrics(Ytrue, Yp) for Yp in Ypreds]
        rf = np.mean([row_f1(Ytrue, Yp) for Yp in Ypreds], axis=0)
        a = {k: float(np.mean([m[k] for m in rep_metrics])) for k in rep_metrics[0] if k not in ("n",)}
        a["sample_f1_se"] = float(rf.std(ddof=1) / np.sqrt(len(rf)))
        a["sample_f1_rep_sd"] = float(np.std([m["sample_f1"] for m in rep_metrics], ddof=1)) if len(reps) > 1 else 0.0
        a["n"] = int(len(rows))
        a["R"] = len(reps)
        if bootstrap:
            a.update(bootstrap_se(Ytrue, Ypreds, B=200))
        res["A"] = a
        res["rowf1_A"] = rf.tolist()
        res["rows_A"] = rows.tolist()
    # ---- B ----  (one or several chronological cuts; B = mean over cuts, B_se = mean per-cut SE
    #               because the cuts share most rows and are highly correlated)
    Bs = [v for v in per.values() if v[0]["protocol"] == "B"]
    if Bs:
        bm = []
        for f, yp, pi in Bs:
            Yt = D.Y[f["eval"]]
            m = metrics(Yt, yp)
            if bootstrap:
                m.update(bootstrap_se(Yt, [yp], B=200))
            m["cut"] = f["cut"]
            if pi is not None:
                m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
            m["true_prior_labels"] = Yt.mean(0).round(3).tolist()
            bm.append(m)
            res.setdefault("rowf1_B", []).extend(row_f1(Yt, yp).tolist())
        res["B_ncuts"] = len(bm)
        if len(bm) == 1:
            res["B"] = bm[0]
        else:
            agg = {k: float(np.mean([m[k] for m in bm])) for k in bm[0]
                   if isinstance(bm[0][k], (int, float)) and k != "cut"}
            agg["cuts"] = bm
            res["B"] = agg
        for m in bm:
            if m["cut"] == CHRONO_CUT:
                res["B2005"] = {k: v for k, v in m.items()}
    # ---- C ----
    Cs = [v for v in per.values() if v[0]["protocol"] == "C"]
    if Cs:
        f, yp, pi = Cs[0]
        Yt = D.Y[f["eval"]]
        m = metrics(Yt, yp)
        if bootstrap:
            m.update(bootstrap_se(Yt, [yp], B=200))
        if pi is not None:
            m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
        res["C"] = m
        res["rowf1_C"] = row_f1(Yt, yp).tolist()
    # ---- robust objective ----
    if "A" in res and "B" in res:
        res["J"] = 0.5 * res["A"]["sample_f1"] + 0.5 * res["B"]["sample_f1"]
        res["J_se"] = 0.5 * float(np.hypot(res["A"]["sample_f1_se"], res["B"]["sample_f1_se"]))
    res["secs"] = float(sum(float(o["secs"]) for o in outs))
    res["T_mean"] = float(np.mean([np.mean(o["T"]) for o in outs]))
    return res


def evaluate(cfg, D, protocols=("A", "B", "C"), R=4, n_jobs=1, seed=0, use_cache=True, rows=None,
             bootstrap=True, chrono_cuts=CHRONO_CUTS):
    validate(cfg)
    folds = make_folds(D, protocols, R, seed, rows=rows, chrono_cuts=chrono_cuts)
    outs = fold_outputs(cfg, D, folds, n_jobs, use_cache)
    return score(cfg, D, folds, outs, bootstrap=bootstrap)


def summary_row(name, res):
    r = {"name": name}
    for p in ("A", "B", "B2005", "C"):
        if p in res:
            r[f"{p}"] = res[p]["sample_f1"]
            r[f"{p}_se"] = res[p]["sample_f1_se"]
            r[f"{p}_macro"] = res[p]["macro_f1"]
    if "J" in res:
        r["J"] = res["J"]
        r["J_se"] = res["J_se"]
    r["secs"] = res.get("secs")
    return r


def paired_se(res1, res2, w=(0.5, 0.5)):
    """SE of the difference in J between two configs using per-row paired differences."""
    dA = np.asarray(res1["rowf1_A"]) - np.asarray(res2["rowf1_A"])
    dB = np.asarray(res1["rowf1_B"]) - np.asarray(res2["rowf1_B"])
    nB = len(dB) / res1.get("B_ncuts", 1)          # cuts overlap: effective n ~ one cut
    return float(np.sqrt((w[0] ** 2) * dA.var(ddof=1) / len(dA) + (w[1] ** 2) * dB.var(ddof=1) / nB))

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/source_snapshot/evaluate.py


## part1/experiments/derived_benchmark_audit_v1/study.py

In [80]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/study.py
"""Retrospective audit of Dylan v2's fixed derived benchmark against scored leaders.

No model/metric tuning. All original sources and scored CSVs are read-only.
Run from repository root: .venv/bin/python part1/experiments/derived_benchmark_audit_v1/study.py
"""
import os
for var in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[var]='1'
import argparse
import ast
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import importlib.util
import json
from pathlib import Path
import shutil
import subprocess
import sys
import time
import traceback
from types import SimpleNamespace
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import precision_recall_fscore_support
from threadpoolctl import threadpool_limits

HERE=Path(__file__).resolve().parent
PART=HERE.parents[1]
REPO=PART.parent
ARCHIVE=PART/'research/lda_ensemble'
HISTORY=PART/'experiments/current_leaders_grouped_v1'
DYLAN=REPO.parent/'dylan/dylan-model-v2'
LABELS=['Safety','Funding','Efficacy','Recruitment','Other']
EMB=[f'emb_{i}' for i in range(150)]
MODELS=['dbscan','tri_training','original_ani','previous_ensemble','lda_ensemble','anchored']
SOURCE_MODULE=None


def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def read(path):return json.loads(Path(path).read_text())
def save(path,value):
    Path(path).write_text(json.dumps(value,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x))+'\n')
def groups(frame):return pd.util.hash_pandas_object(frame[EMB].round(10),index=False).to_numpy()
def row_f1(y,p):return 2*(y*p).sum(axis=1)/(y.sum(axis=1)+p.sum(axis=1))
def decide(p):
    pred=(p>=.5).astype(int);empty=pred.sum(1)==0
    pred[np.flatnonzero(empty),p[empty].argmax(1)]=1
    return pred
def metrics(y,p):
    pr,re,f,s=precision_recall_fscore_support(y,p,zero_division=0)
    return dict(sample_f1=float(row_f1(y,p).mean()),macro_f1=float(f.mean()),
                **{f'{k}_{l}':float(v[j]) for k,v in [('precision',pr),('recall',re),('f1',f),('support',s)] for j,l in enumerate(LABELS)})


def import_file(name,path):
    spec=importlib.util.spec_from_file_location(name,path);m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m);return m


def source_module():
    global SOURCE_MODULE
    if SOURCE_MODULE is None:
        sys.path.insert(0,str(PART))
        direct=import_file('historical_direct_models',ARCHIVE/'experiments/ani_direct_comparison_v1/models.py')
        anchor=import_file('submitted_anchored',PART/'anchored_solution.py')
        SOURCE_MODULE=(direct,anchor)
    return SOURCE_MODULE


def predict(fit,query):
    """Query labels are removed before any source model is invoked."""
    m,anchor=source_module();q=query.drop(columns=LABELS).copy()
    packet=m.f.train_native(fit)
    ani=m.f.native_prediction(packet,q)
    lda,_,_=m.revision.fit_addition(packet,fit,q,'revision_full')
    tri,svm=m.partners.partners_in_frame(fit,q)
    known=fit[LABELS].ge(0).all(axis=1).to_numpy()
    dbscan=m.f.native.predict(fit,np.flatnonzero(known),np.flatnonzero(~known),q,'repo_dbscan',EMB,LABELS)
    model=anchor.AnchoredReasonClassifier().fit(fit[EMB],fit[LABELS])
    scores=dict(dbscan=dbscan,tri_training=tri,original_ani=ani,
                previous_ensemble=.7*ani+.15*tri+.15*svm,lda_ensemble=.7*lda+.15*tri+.15*svm)
    predictions={name:decide(p) for name,p in scores.items()}
    scores['anchored']=model.predict_proba(q[EMB]);predictions['anchored']=model.predict(q[EMB])
    for name in MODELS:
        assert predictions[name].shape==(len(q),5) and (predictions[name].sum(1)>0).all()
        assert np.isfinite(scores[name]).all()
    return scores,predictions


def ledger():
    records=[]
    mapping={'tri_training':'tri-training-trainval235','original_ani':'ani-original',
             'previous_ensemble':'three-way-trainval235','lda_ensemble':'lda-ensemble-trainval235','anchored':'anchored-trainval235'}
    for name,stem in mapping.items():
        p=PART/'submissions/results'/f'{stem}.json';r=read(p)
        csv=REPO/r['submission_file'];assert sha(csv)==r['submission_sha256']
        records.append(dict(name=name,kaggle=r['reported_kaggle_score'],labels=235,
                            csv=str(csv.relative_to(REPO)),sha256=sha(csv),record=str(p.relative_to(REPO)),
                            fitting_access=r['fitting_access'],score_source=r['score_provenance']))
    csv=PART/'submissions/archive/submitted/dbscan_logistic_trainval235__kaggle_0.86452.csv'
    r=read(PART/'experiments/final_dbscan_trainval235/submission_manifest.json')
    assert sha(csv)=='19f29c7bcbda974f8ae9646d3ced83bad2953ead45412e88731e8c12e55d2661'
    records.append(dict(name='dbscan',kaggle=.86452,labels=235,csv=str(csv.relative_to(REPO)),sha256=sha(csv),
                        record='part1/submissions/index.csv',fitting_access='inductive',score_source='Recorded user-reported score'))
    return sorted(records,key=lambda x:MODELS.index(x['name']))


def initialize():
    (HERE/'results').mkdir(exist_ok=True);(HERE/'source_snapshot').mkdir(exist_ok=True)
    # Small source/context snapshots make the supplied benchmark reproducible after relocation.
    supplied={'evaluate.py':'midterm/part1/p1lib/evaluate.py','data.py':'midterm/part1/p1lib/data.py',
              'context.py':'midterm/part1/p1lib/context.py','benchmark_run.py':'benchmark_michael/run.py',
              'benchmark_README.md':'benchmark_michael/README.md','benchmark_results.json':'benchmark_michael/results.json',
              'methodology.md':'docs/02_methodology.md','metrics_explainer.md':'docs/explainers/02_metrics_and_what_they_prove.md'}
    for dest,src in supplied.items():
        p=HERE/'source_snapshot'/dest
        if p.exists():
            if (DYLAN/src).exists():assert sha(p)==sha(DYLAN/src)
        else:shutil.copyfile(DYLAN/src,p)
    files=[Path(__file__),PART/'anchored_solution.py',PART/'TESTING_POLICY.md',PART/'RETENTION.md',PART/'submissions/index.csv']
    files+=list((PART/'submissions/results').glob('*.json'))+list((PART/'data').glob('*.csv'))
    files+=list((ARCHIVE/'research/independent').glob('*.py'))
    files+=list((ARCHIVE/'research/independent/experiments').glob('phase*-baseline.json'))
    files+=[ARCHIVE/'research/independent/experiments/cross_repo_benchmark_v1/registry.json',ARCHIVE/'research/independent/experiments/ani_improvement_v1/registry.json']
    for stage in ['feature_gap_cv_v1','ani_lda_revision_cv_v1','ani_direct_comparison_v1']:
        files+=list((ARCHIVE/'experiments'/stage).glob('*.py'))
    files+=list((HISTORY/'snapshots').glob('*.json'))+list((HISTORY/'snapshots').glob('*.npz'))
    files+=[HISTORY/'anchored_train.npz',HISTORY/'anchored_pooled.npz',HISTORY/'diagnostic_predictions.csv',HISTORY/'comparison.csv']
    files+=list((HERE/'source_snapshot').glob('*'))
    submitted=ledger();files += [REPO/x['csv'] for x in submitted]
    hashes={str(p.relative_to(REPO)):sha(p) for p in sorted(set(files))}
    if (HERE/'protocol.json').exists():
        assert read(HERE/'protocol.json')['source_hashes']==hashes
        return
    save(HERE/'protocol.json',dict(version='derived-benchmark-audit-v1',created_utc=datetime.now(timezone.utc).isoformat(),
         base_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(),models=MODELS,scored_artifacts=submitted,
         hypothesis='Does Dylan v2 J rank previously scored Part 1 recipes more consistently with their reported Kaggle scores than A, B, official validation and our current grouped metrics?',
         parent='Anchored 0.89636; this is a retrospective metric audit, not model development or promotion.',
         dylan_protocol='Exact supplied make_folds: A=50 ungrouped stratified folds R10 seed1 over235; B=mean of cuts2004/2005/2006 with all1014 masked rows; C=original157-to78. J=.5*A+.5*B. Models remain inductive.',
         canonical_protocol='All80 saved grouped folds plus original validation and three saved chronological windows. Freshly refit all six recipes; compare four leaders against retained exact predictions and DBSCAN/tri against retained aggregate results.',
         strict_B_diagnostic='Use the same three Dylan query cuts; purge fitting gold and masked duplicate embedding groups; allow only masked rows earlier than the cut. Report grouped pooled A plus strict B with the same .5/.5 weight.',
         derived_scores=['Dylan_J=(Dylan_A+Dylan_B)/2','aligned_J=(grouped_pooled_CV+strict_Dylan_B)/2','current_J_diagnostic=(grouped_pooled_CV+canonical_chronology98)/2'],
         retrospective_analysis='Spearman, Kendall tau-b, pairwise ordering with ties, named historical contrasts; leave-one-model-out, exclude-transductive-Ani and top-four sensitivities. Never tune J weights, chronology cutoffs or recipes to Kaggle.',
         uncertainty='Paired bootstrap of embedding groups with shared weights across all occurrences of a row/repeat/cut; accounts for A/B overlap conditional on fitted predictions. Report original SE approximation separately. No independent holdout or exact Kaggle uncertainty claim.',
         access_caveat='Original Ani Kaggle CSV was transductive. Primary local comparison uses its established inductive adapter and explicitly reports exclusion sensitivity. Dylan candidates have no recorded Kaggle score and are excluded from correlation analysis.',
         dbscan_refit_caveat='157-label DBSCAN scored .87500 while235 scored .86452. One fixed recipe evaluated on fixed folds cannot distinguish these final fits; use only235 in same-label-count primary panel and report this limitation separately.',
         scope='All fixed cases are diagnostic, including supplementary checks. User explicitly requested evaluation of the derived score; the repository default selection policy is unchanged.',
         retention='Keep compact per-fold scores/predictions and fitting IDs because they are the evidence for this matched metric audit; no fitted model caches or duplicated datasets.',
         versions={n:importlib.metadata.version(n) for n in ['numpy','pandas','scipy','scikit-learn','iterative-stratification','joblib']},
         source_hashes=hashes,no_new_submission=True))


def make_cases():
    tr=pd.read_csv(PART/'data/part1_train.csv');va=pd.read_csv(PART/'data/part1_val.csv')
    full=pd.concat([tr,va],ignore_index=True);gg=groups(full)
    lab=np.flatnonzero(tr[LABELS].ge(0).all(axis=1));unl=np.flatnonzero(tr[LABELS].eq(-100).all(axis=1))
    val=np.arange(len(tr),len(full));labval=np.r_[lab,val]
    masks=full[LABELS].clip(lower=0).to_numpy(int)@np.array([1,2,4,8,16]);keys=np.unique(masks[labval])
    combo=np.full(len(full),-1);combo[labval]=np.searchsorted(keys,masks[labval])
    D=SimpleNamespace(idx=dict(lab=lab,val=val,labval=labval,unl=unl),y=combo,year=full.start_year.to_numpy())
    tree=ast.parse((HERE/'source_snapshot/evaluate.py').read_text())
    fn=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name=='make_folds')
    env=dict(np=np,RepeatedStratifiedKFold=RepeatedStratifiedKFold,CHRONO_CUTS=(2004,2005,2006))
    exec(compile(ast.Module(body=[fn],type_ignores=[]),'supplied_make_folds','exec'),env)
    cases=[]
    def add(key,track,kind,a,b,u,**kw):
        fitrows=np.sort(np.r_[a,u]);assert len(set(fitrows)&set(b))==0
        cases.append(dict(key=key,track=track,kind=kind,train_gold=full.iloc[a].ID.tolist(),
                          fit_ids=full.iloc[fitrows].ID.tolist(),query_ids=full.iloc[b].ID.tolist(),
                          masked_ids=full.iloc[u].ID.tolist(),gold_group_overlap=len(set(gg[a])&set(gg[b])),
                          masked_group_overlap=len(set(gg[u])&set(gg[b])),future_masked=int((full.iloc[u].start_year>=full.iloc[b].start_year.min()).sum()),**kw))
    for f in env['make_folds'](D,R=10,seed=1):
        if f['protocol']=='C':continue
        a,b=f['train'],f['eval']
        add('dylan_'+f['name'],'dylan',f['protocol'],a,b,unl,repeat=f.get('repeat'),cut=f.get('cut'))
        if f['protocol']=='B':
            keepg=~np.isin(gg[a],gg[b]);keepu=(full.iloc[unl].start_year.to_numpy()<f['cut'])&~np.isin(gg[unl],gg[b])
            add('strict_'+f['name'],'strict_B','B',a[keepg],b,unl[keepu],cut=f['cut'])
    for pool,gold in [('train',lab),('pooled',labval)]:
        for n in [3,5]:
            plan=read(HISTORY/'snapshots'/f'{pool}{n}.json');assert full.iloc[gold].ID.tolist()==plan['gold_ids']
            for f in plan['folds']:
                a,b=gold[np.array(f['train_gold_positions'])],gold[np.array(f['heldout_gold_positions'])]
                assert not set(gg[a])&set(gg[b]);u=unl[~np.isin(gg[unl],gg[b])]
                assert len(unl)-len(u)==f['masked_duplicates_excluded']
                add(f'grouped_{pool}_{f["seed"]}_{f["fold"]}','canonical',pool,a,b,u,repeat=f['seed'],fold=f['fold'])
    add('validation','canonical','validation',lab,val,unl)
    ix={v:i for i,v in enumerate(full.ID)}
    for w in read(HISTORY/'snapshots/chronology.json')['chronological_splits']:
        fit=np.array([ix[v] for v in w['training_ids']]);b=np.array([ix[v] for v in w['heldout_ids']])
        known=full.iloc[fit][LABELS].ge(0).all(axis=1).to_numpy()
        add(w['name'],'canonical','chronology',fit[known],b,fit[~known])
    if (HERE/'splits.json').exists():assert read(HERE/'splits.json')==cases
    else:save(HERE/'splits.json',cases)
    return full,cases


def worker(case):
    dest=HERE/'results'/case['key']
    if dest.with_suffix('.json').exists():
        r=read(dest.with_suffix('.json'));assert sha(dest.with_suffix('.npz'))==r['sha256'];return {'key':case['key'],'cached':True}
    start=time.time()
    try:
        full=pd.concat([pd.read_csv(PART/'data/part1_train.csv'),pd.read_csv(PART/'data/part1_val.csv')],ignore_index=True).set_index('ID',drop=False)
        fit=full.loc[case['fit_ids']].reset_index(drop=True);q=full.loc[case['query_ids']].reset_index(drop=True)
        with threadpool_limits(limits=1):scores,preds=predict(fit,q)
        arrays=dict(ids=q.ID.to_numpy(str),truth=q[LABELS].to_numpy(int))
        for name in MODELS:arrays[name+'_scores']=scores[name];arrays[name+'_predictions']=preds[name]
        np.savez_compressed(dest.with_suffix('.npz'),**arrays)
        result=dict(key=case['key'],seconds=time.time()-start,sha256=sha(dest.with_suffix('.npz')),
                    metrics={name:metrics(arrays['truth'],preds[name]) for name in MODELS})
        save(dest.with_suffix('.json'),result)
        return dict(key=case['key'],seconds=round(result['seconds'],2),anchored=result['metrics']['anchored']['sample_f1'])
    except Exception:
        save(dest.with_suffix('.failure.json'),dict(traceback=traceback.format_exc()));raise


def parity(cases):
    checks=[]
    for pool in ['train','pooled']:
        files={'anchored':HISTORY/f'anchored_{pool}.npz',**{n:HISTORY/'snapshots'/f'{n}_{pool}.npz' for n in ['original_ani','previous_ensemble','lda_ensemble']}}
        for name,path in files.items():
            with np.load(path,allow_pickle=True) as z:
                lookup={v:i for i,v in enumerate(z['ids'])};seeds=sorted({c['repeat'] for c in cases if c['kind']==pool})
                for c in [c for c in cases if c['kind']==pool]:
                    if not (HERE/'results'/f'{c["key"]}.npz').exists():continue
                    with np.load(HERE/'results'/f'{c["key"]}.npz') as out:
                        ii=[lookup[v] for v in out['ids']];ri=seeds.index(c['repeat'])
                        np.testing.assert_array_equal(out['truth'],z['truth'][ii])
                        np.testing.assert_array_equal(out[name+'_predictions'],z['predictions'][ri,ii])
                        expected=z['scores'][ri,ii]
                        # The Ani snapshot is raw p at cutoff .3; its ensemble adapter
                        # centers those same scores to cutoff .5. Predictions are identical.
                        if name=='original_ani':expected=expit(logit(np.clip(expected,1e-9,1-1e-9))-logit(.3))
                        err=float(abs(out[name+'_scores']-expected).max())
                        np.testing.assert_allclose(out[name+'_scores'],expected,atol=1e-7,rtol=0)
                        checks.append(dict(case=c['key'],model=name,identical_predictions=True,max_score_error=err))
    save(HERE/'control_replay_checks.json',checks)
    return checks


def summarize(cases):
    rows=[];perrows=[]
    for c in cases:
        with np.load(HERE/'results'/f'{c["key"]}.npz') as z:
            for name in MODELS:
                y,p=z['truth'],z[name+'_predictions']
                rows.append(dict(model=name,case=c['key'],track=c['track'],kind=c['kind'],repeat=c.get('repeat'),cut=c.get('cut'),**metrics(y,p)))
                for ident,v in zip(z['ids'],row_f1(y,p)):
                    perrows.append(dict(ID=ident,model=name,case=c['key'],track=c['track'],kind=c['kind'],repeat=c.get('repeat'),cut=c.get('cut'),f1=float(v)))
    pd.DataFrame(rows).to_csv(HERE/'case_scores.csv',index=False)
    rf=pd.DataFrame(perrows);rf.to_csv(HERE/'per_trial_f1.csv.gz',index=False)
    rep=[]
    for (name,track,kind,r),g in rf[rf.kind.isin(['A','train','pooled'])].groupby(['model','track','kind','repeat']):
        assert g.ID.is_unique
        rep.append(dict(model=name,track=track,kind=kind,repeat=int(r),sample_f1=g.f1.mean()))
    pd.DataFrame(rep).to_csv(HERE/'repeat_scores.csv',index=False)
    out=[]
    for name in MODELS:
        g=rf[rf.model==name]
        mean=lambda track,kind:float(g[(g.track==track)&(g.kind==kind)].f1.mean())
        A=mean('dylan','A');B=g[(g.track=='dylan')&(g.kind=='B')].groupby('cut').f1.mean().mean()
        strict=g[(g.track=='strict_B')&(g.kind=='B')].groupby('cut').f1.mean().mean()
        train,pooled,chrono,C=[mean('canonical',k) for k in ['train','pooled','chronology','validation']]
        B2005=mean('dylan','B') # replaced with the exact single cut below
        B2005=g[(g.track=='dylan')&(g.cut==2005)].f1.mean()
        out.append(dict(model=name,kaggle=next(x['kaggle'] for x in ledger() if x['name']==name),
                        dylan_A=A,dylan_B=B,dylan_B2005=B2005,dylan_J=.5*(A+B),validation=C,
                        grouped_train=train,grouped_pooled=pooled,chronology98=chrono,
                        strict_B=strict,aligned_J=.5*(pooled+strict),current_J_diagnostic=.5*(pooled+chrono)))
    comparison=pd.DataFrame(out);comparison.to_csv(HERE/'comparison.csv',index=False)
    checks=parity(cases);assert len(checks)==320
    supplied=read(HERE/'source_snapshot/benchmark_results.json')['michael_anchored']
    our=comparison.set_index('model').loc['anchored']
    for p in ['A','B','J']:
        expected=supplied[p] if p=='J' else supplied[p]['sample_f1']
        assert abs(our['dylan_'+p]-expected)<1e-12,(p,our['dylan_'+p],expected)
    for p,h in read(HERE/'protocol.json')['source_hashes'].items():assert sha(REPO/p)==h,p
    save(HERE/'completion.json',dict(cases=len(cases),recipe_evaluations=len(cases)*len(MODELS),
         anchored_supplied_A_B_J_reproduced=True,canonical_reference_fold_prediction_checks=len(checks),protected_hashes_unchanged=True))
    print(comparison.to_string(index=False),flush=True)


def main():
    ap=argparse.ArgumentParser();ap.add_argument('--workers',type=int,default=4);ap.add_argument('--smoke',action='store_true');args=ap.parse_args()
    initialize();_,cases=make_cases()
    if args.smoke:
        chosen=[next(c for c in cases if c['kind']==k) for k in ['train','pooled','validation']]
        for c in chosen:print(worker(c),flush=True)
        print('Parity',len(parity(chosen)),flush=True);return
    with ProcessPoolExecutor(max_workers=args.workers) as pool:
        pending=[pool.submit(worker,c) for c in cases]
        for i,f in enumerate(as_completed(pending),1):print(i,len(pending),json.dumps(f.result()),flush=True)
    summarize(cases)


if __name__=='__main__':main()

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/study.py


## part1/experiments/derived_benchmark_audit_v1/verify.py

In [81]:
%%writefile _source_export/part1/experiments/derived_benchmark_audit_v1/verify.py
"""Independent rescoring, coverage, provenance and historical control checks."""
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support

HERE = Path(__file__).resolve().parent
PART = HERE.parents[1]
REPO = PART.parent
LABELS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']


def read(path):
    return json.loads(path.read_text())


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main():
    protocol = read(HERE / 'protocol.json')
    for path, digest in protocol['source_hashes'].items():
        assert sha(REPO / path) == digest, path
    full = pd.concat([pd.read_csv(PART / 'data/part1_train.csv'), pd.read_csv(PART / 'data/part1_val.csv')]).set_index('ID')
    test_ids = set(pd.read_csv(PART / 'data/part1_test.csv').ID)
    hashes = pd.util.hash_pandas_object(full[[f'emb_{i}' for i in range(150)]].round(10), index=False)
    gold_ids = set(full.index[full[LABELS].ge(0).all(axis=1)])
    train_ids = set(pd.read_csv(PART / 'data/part1_train.csv').ID)
    train_gold = gold_ids & train_ids
    masked_ids = set(full.index[full[LABELS].eq(-100).all(axis=1)])
    cases = read(HERE / 'splits.json')
    comp = pd.read_csv(HERE / 'comparison.csv').set_index('model')
    models = protocol['models']
    checks = 0
    occurrences = {}
    per_model = {name: [] for name in models}
    for c in cases:
        fit, query, gold, masked = (set(c[k]) for k in ['fit_ids', 'query_ids', 'train_gold', 'masked_ids'])
        assert len(fit) == len(c['fit_ids']) and len(query) == len(c['query_ids'])
        assert fit == gold | masked and not gold & masked
        assert gold <= gold_ids and query <= gold_ids and masked <= masked_ids
        assert not fit & query and not (fit | query) & test_ids
        if c['kind'] == 'train':
            assert (fit | query) <= train_ids
        require_groups = c['track'] == 'strict_B' or (c['track'] == 'canonical' and c['kind'] != 'validation')
        if require_groups:
            assert not set(hashes.loc[c['fit_ids']]) & set(hashes.loc[c['query_ids']])
        if c['track'] == 'strict_B':
            assert full.loc[c['fit_ids']].start_year.max() < c['cut']
            assert full.loc[c['query_ids']].start_year.min() >= c['cut']
        if c['kind'] == 'chronology':
            assert full.loc[c['fit_ids']].start_year.max() < full.loc[c['query_ids']].start_year.min()
        if c['kind'] in ['A', 'train', 'pooled']:
            occurrences.setdefault((c['kind'], c['repeat']), []).extend(c['query_ids'])
        path = HERE / 'results' / f'{c["key"]}.npz'
        sidecar = read(path.with_suffix('.json'))
        assert sha(path) == sidecar['sha256']
        with np.load(path) as z:
            assert list(z['ids']) == c['query_ids']
            np.testing.assert_array_equal(z['truth'], full.loc[c['query_ids'], LABELS].to_numpy())
            for name in models:
                y, p, s = z['truth'], z[name + '_predictions'], z[name + '_scores']
                assert p.shape == y.shape == s.shape and np.isfinite(s).all()
                assert set(np.unique(p)) <= {0, 1} and (p.sum(1) > 0).all()
                cutoff = .3 if name == 'anchored' else .5
                expected = (s >= cutoff).astype(int)
                empty = np.flatnonzero(expected.sum(1) == 0)
                expected[empty, s[empty].argmax(1)] = 1
                np.testing.assert_array_equal(expected, p)
                actual = f1_score(y, p, average='samples', zero_division=0)
                assert abs(actual - sidecar['metrics'][name]['sample_f1']) < 1e-12
                pr, re, f1, support = precision_recall_fscore_support(y, p, zero_division=0)
                assert abs(f1.mean() - sidecar['metrics'][name]['macro_f1']) < 1e-12
                for key, values in [('precision', pr), ('recall', re), ('f1', f1), ('support', support)]:
                    for j, label in enumerate(LABELS):
                        assert abs(values[j] - sidecar['metrics'][name][key + '_' + label]) < 1e-12
                per_model[name].append(dict(case=c, ids=z['ids'].copy(), y=y.copy(), p=p.copy(), score=actual))
                checks += 1
    for (kind, repeat), ids in occurrences.items():
        assert len(ids) == len(set(ids))
        assert set(ids) == (train_gold if kind == 'train' else gold_ids)
    assert len(occurrences) == 26
    assert len(cases) == 140 and checks == 840

    def pooled(records):
        return f1_score(np.concatenate([r['y'] for r in records]), np.concatenate([r['p'] for r in records]), average='samples', zero_division=0)

    for name in models:
        rec = per_model[name]
        def select(track, kind):
            return [r for r in rec if r['case']['track'] == track and r['case']['kind'] == kind]
        # Independently rebuild all main metric expressions from label predictions.
        expected = dict(dylan_A=pooled(select('dylan', 'A')),
                        dylan_B=np.mean([r['score'] for r in select('dylan', 'B')]),
                        strict_B=np.mean([r['score'] for r in select('strict_B', 'B')]),
                        validation=pooled(select('canonical', 'validation')),
                        grouped_train=pooled(select('canonical', 'train')),
                        grouped_pooled=pooled(select('canonical', 'pooled')),
                        chronology98=pooled(select('canonical', 'chronology')),
                        dylan_B2005=next(r['score'] for r in select('dylan', 'B') if r['case']['cut'] == 2005))
        expected['dylan_J'] = (expected['dylan_A'] + expected['dylan_B']) / 2
        expected['aligned_J'] = (expected['grouped_pooled'] + expected['strict_B']) / 2
        expected['current_J_diagnostic'] = (expected['grouped_pooled'] + expected['chronology98']) / 2
        for key, value in expected.items():
            assert abs(value - comp.loc[name, key]) < 1e-12, (name, key)

    archive = PART / 'research/lda_ensemble/research/independent/experiments/cross_repo_benchmark_v1/confirmation8/comparison.csv'
    historical = pd.read_csv(archive).set_index('candidate')
    old_checks = []
    for name, ref in [('dbscan', 'DBSCAN + logistic C1000'), ('tri_training', 'Classical tri-training')]:
        for metric, oldmetric in [('grouped_train', 'train_cv_8'), ('grouped_pooled', 'pooled_cv_8'),
                                  ('validation', 'validation'), ('chronology98', 'chronological')]:
            delta = float(comp.loc[name, metric] - historical.loc[ref, oldmetric])
            assert abs(delta) < 1e-12
            old_checks.append(dict(model=name, metric=metric, delta=delta))
    current = pd.read_csv(PART / 'experiments/current_leaders_grouped_v1/comparison.csv')
    current = current.set_index('name')
    current_checks = 0
    for name in ['original_ani', 'previous_ensemble', 'lda_ensemble', 'anchored']:
        ref = 'anchored_candidate' if name == 'anchored' else name
        for metric, oldmetric in [('grouped_train', 'train'), ('grouped_pooled', 'pooled'),
                                  ('validation', 'validation'), ('chronology98', 'chronology98')]:
            assert abs(comp.loc[name, metric] - current.loc[ref, oldmetric]) < 1e-12
            current_checks += 1
    # The runner additionally verifies exact predictions and aligned probability
    # scores for all four retained leaders on 80 canonical folds (320 checks).
    replay = read(HERE / 'control_replay_checks.json')
    assert len(replay) == 320 and all(r['identical_predictions'] and r['max_score_error'] <= 1e-7 for r in replay)
    supplied = read(HERE / 'source_snapshot/benchmark_results.json')['michael_anchored']
    for key in ['A', 'B', 'C', 'J']:
        expected = supplied[key] if key == 'J' else supplied[key]['sample_f1']
        assert abs(comp.loc['anchored', 'validation' if key == 'C' else 'dylan_' + key] - expected) < 1e-12
    for artifact in protocol['scored_artifacts']:
        assert sha(REPO / artifact['csv']) == artifact['sha256']
    result = dict(status='passed', case_prediction_rescores=checks, protected_source_hashes=len(protocol['source_hashes']),
                  exact_historical_fold_controls=len(replay), DBSCAN_tri_aggregate_controls=old_checks,
                  current_leader_aggregate_controls=current_checks,
                  complete_OOF_repeats=len(occurrences), supplied_anchored_A_B_C_J_reproduced=True,
                  zero_test_input_access=True, strict_and_canonical_group_boundaries=True,
                  study_outputs='Predictions are matched benchmark controls, not new submission artifacts.')
    (HERE / 'verification.json').write_text(json.dumps(result, indent=2) + '\n')
    print(json.dumps(result, indent=2))


if __name__ == '__main__':
    main()

Overwriting _source_export/part1/experiments/derived_benchmark_audit_v1/verify.py


## part1/research/anchored_neighbor_blend_v1/check_neighbor_math.py

In [82]:
%%writefile _source_export/part1/research/anchored_neighbor_blend_v1/check_neighbor_math.py
"""Small mathematical checks of the new voting rules; no model fitting."""
from pathlib import Path
import sys

import numpy as np
from sklearn.preprocessing import normalize

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from neighbor_models import NeighborAdjustedAnchored, embedding_groups


class ZeroClassifier:
    classes_ = np.arange(5)

    def predict_proba(self, x):
        return np.zeros((len(x), 5))


def fixture(x, y, mode, gold=None):
    m = NeighborAdjustedAnchored(mode)
    m.n_gold_ = len(x) if gold is None else gold
    m.neighbor_X_, m.neighbor_Y_ = normalize(x), y.copy()
    groups = embedding_groups(x)
    m.gold_groups_, m.masked_groups_ = groups[:m.n_gold_], groups[m.n_gold_:]
    m.generated_mask_ = np.ones(len(x)-m.n_gold_, bool)
    m.logistic_ = m.lda_ = ZeroClassifier()
    return m


def main():
    x = np.eye(150)[:3]
    y = np.eye(5, dtype=int)[:3]
    q = (x[0]+.4*x[1]+.2*x[2])[None]
    repeated_x = np.vstack([x, np.repeat(x[1:2], 71, axis=0)])
    repeated_y = np.vstack([y, np.repeat(y[1:2], 71, axis=0)])
    grouped = fixture(x, y, "grouped_neighbors")._scores(q)
    repeated = fixture(repeated_x, repeated_y, "grouped_neighbors")
    np.testing.assert_allclose(repeated._scores(q), grouped, atol=1e-14, rtol=0)
    assert repeated.neighbor_audit_["voting_bank_rows"] == 3
    assert not np.allclose(fixture(repeated_x, repeated_y, "row_control")._scores(q), grouped)
    # Unique groups preserve the original row-vote computation.
    np.testing.assert_allclose(fixture(x, y, "row_control")._scores(q), grouped, atol=1e-14, rtol=0)
    # A shared vector with conflicting labels contributes the member average.
    mixed = fixture(np.vstack([x, x[1]]), np.vstack([y, y[2]]), "grouped_neighbors")
    mean_labels = y.astype(float)
    mean_labels[1] = (y[1]+y[2])/2
    expected = fixture(x, mean_labels, "row_control")._scores(q)
    np.testing.assert_allclose(mixed._scores(q), expected, atol=1e-14, rtol=0)
    # Equal similarity: one gold and one quarter-weight generated vote.
    weighted = fixture(x[:2], y[:2], "pseudo_neighbor_quarter", gold=1)
    actual = weighted._scores((x[0]+x[1])[None])/.3
    expected = np.array([[.5, .125, 0, 0, 0]])/(.625+1e-9)
    np.testing.assert_allclose(actual, expected, atol=1e-14, rtol=0)
    print("Passed: duplicate invariance, unique-group parity, conflicting-label averaging, quarter-weight normalization; fits=0.")


if __name__ == "__main__":
    main()

Overwriting _source_export/part1/research/anchored_neighbor_blend_v1/check_neighbor_math.py


## part1/research/anchored_neighbor_blend_v1/evidence.py

In [83]:
%%writefile _source_export/part1/research/anchored_neighbor_blend_v1/evidence.py
"""Independently rescore, retain useful evidence, or verify it without fitting.

The metric implementation uses TP/FP/FN arithmetic, not the fitting runner's
sklearn metrics. Retention never deletes its input directory.
"""
import argparse
import json
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

import study

STUDY = Path(__file__).resolve().parent
LABELS = study.LABELS
KEEP = ["anchored", "anchored75_lda25", "lda_control"]
TABLES = ["repeat_scores.csv", "cv_summary.csv", "paired_summary.csv",
          "paired_repeat_deltas.csv", "supplementary_scores.csv", "verification.json"]


def ratio(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def row_f1(y, p):
    return ratio(2*(y*p).sum(axis=1), y.sum(axis=1)+p.sum(axis=1))


def metrics(y, p):
    assert np.isin(y, [0, 1]).all() and np.isin(p, [0, 1]).all()
    tp, fp, fn = (y*p).sum(axis=0), ((1-y)*p).sum(axis=0), (y*(1-p)).sum(axis=0)
    values = dict(precision=ratio(tp, tp+fp), recall=ratio(tp, tp+fn),
                  f1=ratio(2*tp, 2*tp+fp+fn), support=y.sum(axis=0))
    out = dict(sample_f1=float(row_f1(y, p).mean()), macro_f1=float(values["f1"].mean()))
    out.update({kind+"_"+label: float(values[kind][i])
                for kind in values for i, label in enumerate(LABELS)})
    return out


def read_raw(raw):
    arrays = {k: [] for k in ["case", "pool", "seed", "ids", "truth", "scores", "predictions", "parent_raw_scores"]}
    for case in study.benchmark.cases():
        with np.load(raw/"predictions"/(case["key"]+".npz"), allow_pickle=False) as z:
            np.testing.assert_array_equal(z["ids"], case["query"].ID)
            np.testing.assert_array_equal(z["truth"], case["query"][LABELS])
            for key in ["ids", "truth", "scores", "predictions", "parent_raw_scores"]:
                arrays[key].append(z[key])
            for key, value in [("case", case["key"]), ("pool", case["pool"]), ("seed", case["seed"] or 0)]:
                arrays[key].append(np.repeat(value, len(z["ids"])))
    return {k: np.concatenate(v, axis=1 if k in ["scores", "predictions"] else 0) for k, v in arrays.items()}


def rescore(data, modes, results):
    records, errors = 0, []
    repeats = pd.read_csv(results/"repeat_scores.csv")
    supplementary = pd.read_csv(results/"supplementary_scores.csv")
    for i, mode in enumerate(modes):
        np.testing.assert_array_equal(study.decide(data["scores"][i]), data["predictions"][i])
        for table in [repeats, supplementary]:
            for row in table[table["mode"] == mode].to_dict("records"):
                if "pool" in row:
                    mask = (data["pool"] == row["pool"]) & (data["seed"] == row["seed"])
                elif row["case"] == "chronology98":
                    mask = (data["pool"] == "supplementary") & (data["case"] != "validation")
                else:
                    mask = data["case"] == row["case"]
                assert mask.any() and len(np.unique(data["ids"][mask])) == int(mask.sum())
                actual = metrics(data["truth"][mask], data["predictions"][i, mask])
                for key, value in actual.items():
                    error = abs(value-row[key])
                    assert error < 1e-12, (mode, row, key, value)
                    errors.append(error)
                records += 1
    return dict(passed=True, method="Direct TP/FP/FN arithmetic; no sklearn metric calls",
                records_checked=records, metric_values_checked=len(errors),
                max_absolute_error=max(errors), modes=list(modes), models_fitted=0)


def changes(data, modes):
    rows = []
    for key in np.unique(data["case"]):
        mask = data["case"] == key
        y, p = data["truth"][mask], data["predictions"][:, mask]
        base = row_f1(y, p[0])
        for i, mode in enumerate(modes[1:], 1):
            d = row_f1(y, p[i])-base
            changed = (p[i] != p[0]).any(axis=1)
            rows.append(dict(case=key, pool=data["pool"][mask][0], mode=mode,
                             changed_labelsets=int(changed.sum()), improved=int((d > 1e-12).sum()),
                             worsened=int((d < -1e-12).sum()),
                             same_score_changed=int((changed & (abs(d) <= 1e-12)).sum()),
                             sum_row_f1_delta=float(d.sum())))
    return pd.DataFrame(rows)


def summarize_access(audits):
    assert len(audits) == 84
    assert all(not a["query_inputs_used_for_fit"] and not a["query_labels_used_for_fit"] for a in audits)
    records = []
    for a in audits:
        base = {k: a[k] for k in ["key", "pool", "seed", "input_rows", "gold_rows", "target_rows", "fit_groups", "fit_ids_sha256", "purged_masked"]}
        base["masked_rows"] = a["input_rows"]-a["gold_rows"]
        for model in a["models"]:
            row = dict(base, mode=model["mode"], generated_rows=model["rounds"][-1]["generated"])
            if "neighbors" in model:
                row.update({k: v for k, v in model["neighbors"].items() if k not in ["mode", "gold_rows", "generated_rows"]})
            records.append(row)
    return pd.DataFrame(records)


def retain(raw, destination):
    assert not (destination/"results").exists() and not (destination/"retention.json").exists()
    assert study.sources() == json.loads((STUDY/"protocol.json").read_text())["sources"]
    assert json.loads((raw/"verification.json").read_text())["passed"]
    data = read_raw(raw)
    # Verify all arms before retaining only the useful blend and its controls.
    independent = rescore(data, study.MODES, raw)
    assert independent["records_checked"] == 105 and independent["metric_values_checked"] == 2310
    results = destination/"results"
    results.mkdir(parents=True)
    for name in TABLES:
        shutil.copyfile(raw/name, results/name)
    study.save(results/"independent_rescore.json", independent)
    changes(data, study.MODES).to_csv(results/"prediction_change_summary.csv", index=False)
    audits = json.loads((raw/"fit_audits.json").read_text())
    summarize_access(audits).to_csv(results/"fit_access_summary.csv", index=False)
    indices = [study.MODES.index(m) for m in KEEP]
    for key in ["scores", "predictions"]:
        data[key] = data[key][indices]
    np.savez_compressed(results/"useful_comparison.npz", modes=np.array(KEEP), **data)
    manifest = dict(
        policy="No candidate passed both-pool advancement. Retain full blend/anchored/LDA predictions as a useful matched comparison: the fixed blend won all eight Pooled235 repeats without a sample-F1 supplementary regression. Keep compact results and fit-access counts for both unsuccessful neighbor arms.",
        registration_commit="924ae04",
        source_packets_verified=84,
        raw_prediction_bytes=sum(p.stat().st_size for p in (raw/"predictions").glob("*.npz")),
        raw_fit_audit_bytes=(raw/"fit_audits.json").stat().st_size,
        omitted="Unsuccessful neighbor predictions and detailed fit-round audits; no fitted models were saved. Raw input is disposable after retained verification passes.",
        files={str(p.relative_to(destination)): dict(sha256=study.sha(p), bytes=p.stat().st_size)
               for p in sorted(results.iterdir())})
    study.save(destination/"retention.json", manifest)
    print(json.dumps(independent, indent=2))
    verify(destination)


def verify(destination):
    manifest = json.loads((destination/"retention.json").read_text())
    protocol = json.loads((STUDY/"protocol.json").read_text())
    assert study.sources() == protocol["sources"]
    for relative, record in manifest["files"].items():
        path = destination/relative
        assert path.stat().st_size == record["bytes"] and study.sha(path) == record["sha256"], relative
    results = destination/"results"
    check = json.loads((results/"verification.json").read_text())
    assert check["passed"] and check["protocol_sha256"] == study.sha(STUDY/"protocol.json")
    assert (check["new_cv_fits"], check["new_supplementary_fits"], check["implementation_control_fits"]) == (240, 12, 1)
    with np.load(results/"useful_comparison.npz", allow_pickle=False) as z:
        data = {k: z[k] for k in z.files}
    assert list(data["modes"]) == KEEP
    np.testing.assert_allclose(data["scores"][0], study.centered(data["parent_raw_scores"]), atol=0, rtol=0)
    np.testing.assert_allclose(data["scores"][1], .75*data["scores"][0]+.25*data["scores"][2], atol=0, rtol=0)
    cases = list(study.benchmark.cases())
    assert len(cases) == len(np.unique(data["case"])) == 84
    access = pd.read_csv(results/"fit_access_summary.csv")
    assert len(access) == 252
    for c in cases:
        mask = data["case"] == c["key"]
        np.testing.assert_array_equal(data["ids"][mask], c["query"].ID)
        np.testing.assert_array_equal(data["truth"][mask], c["query"][LABELS])
        assert (data["pool"][mask] == c["pool"]).all() and (data["seed"][mask] == (c["seed"] or 0)).all()
        np.testing.assert_allclose(data["scores"][2, mask], study.lda_for(c), atol=0, rtol=0)
        audit = access[access.key == c["key"]]
        assert set(audit["mode"]) == set(study.MODES[:3]) and len(audit) == 3
        gold = int(c["fit"][LABELS].ge(0).all(axis=1).sum())
        assert (audit.input_rows == len(c["fit"])).all() and (audit.gold_rows == gold).all()
        assert (audit.target_rows == len(c["query"])).all()
        assert (audit.masked_rows == len(c["fit"])-gold).all()
        assert (audit.purged_masked == c["purged_masked"]).all()
        neighbor = audit.set_index("mode").loc["grouped_neighbors"]
        assert neighbor.voting_bank_rows == neighbor.distinct_embedding_groups
        neighbor = audit.set_index("mode").loc["pseudo_neighbor_quarter"]
        assert neighbor.generated_vote_weight == .25 and neighbor.voting_bank_rows == neighbor.original_bank_rows
        if c["pool"] != "supplementary":
            with np.load(study.REFERENCE/f"anchored_{c['pool']}.npz", allow_pickle=True) as z:
                np.testing.assert_allclose(data["parent_raw_scores"][mask], z["scores"][c["repeat_index"], c["positions"]], atol=0, rtol=0)
    independent = rescore(data, KEEP, results)
    assert independent["records_checked"] == 63
    old = json.loads((results/"independent_rescore.json").read_text())
    assert old["passed"] and old["records_checked"] == 105 and old["metric_values_checked"] == 2310
    repeats, summary, pairs, deltas, supplementary = [pd.read_csv(results/(n+".csv")) for n in
        ["repeat_scores", "cv_summary", "paired_summary", "paired_repeat_deltas", "supplementary_scores"]]
    assert (len(repeats), len(summary), len(pairs), len(deltas), len(supplementary)) == (80, 10, 6, 48, 25)
    for frame in [repeats, supplementary]:
        np.testing.assert_allclose(frame[["f1_"+label for label in LABELS]].mean(axis=1), frame.macro_f1, atol=1e-12, rtol=0)
    for row in summary.itertuples():
        group = repeats[(repeats.pool == row.pool) & (repeats["mode"] == row.mode)]
        assert len(group) == 8 and group.seed.is_unique
        np.testing.assert_allclose([group.sample_f1.mean(), group.sample_f1.std(ddof=0), group.macro_f1.mean()],
                                   [row.mean_sample_f1, row.population_sd, row.mean_macro_f1], atol=1e-12, rtol=0)
    qualified = {m: True for m in study.CANDIDATES}
    for row in pairs.itertuples():
        pivot = repeats[repeats.pool == row.pool].pivot(index="seed", columns="mode", values="sample_f1")
        d = pivot[row.mode]-pivot.anchored
        saved = deltas[(deltas.pool == row.pool) & (deltas["mode"] == row.mode)].set_index("seed").loc[d.index]
        np.testing.assert_allclose(d, saved.sample_f1_delta, atol=1e-12, rtol=0)
        wins, ties, losses = int((d > 1e-12).sum()), int((abs(d) <= 1e-12).sum()), int((d < -1e-12).sum())
        assert (wins, ties, losses) == (row.wins, row.ties, row.losses)
        np.testing.assert_allclose(d.mean(), row.mean_delta, atol=1e-12, rtol=0)
        passed = bool(d.mean() > 0 and wins >= 5)
        assert passed == row.meets_pool_rule
        qualified[row.mode] &= passed
    assert qualified == check["qualified_cv"]
    windows = [c for c in cases if c["pool"] == "supplementary" and c["key"] != "validation"]
    for mode in study.MODES:
        scores = supplementary[supplementary["mode"] == mode].set_index("case").sample_f1
        pooled = sum(scores[c["key"]]*len(c["query"]) for c in windows)/98
        np.testing.assert_allclose(pooled, scores["chronology98"], atol=1e-12, rtol=0)
    observed = changes(data, KEEP).sort_values(["case", "mode"]).reset_index(drop=True)
    saved = pd.read_csv(results/"prediction_change_summary.csv")
    saved = saved[saved["mode"].isin(KEEP)].sort_values(["case", "mode"]).reset_index(drop=True)
    pd.testing.assert_frame_equal(observed, saved, check_dtype=False, atol=1e-12, rtol=0)
    print(json.dumps(dict(passed=True, retained_files_checked=len(manifest["files"]),
                         protected_sources_checked=len(protocol["sources"]),
                         retained_prediction_metric_values_rescored=independent["metric_values_checked"],
                         qualified_cv=qualified, models_fitted=0), indent=2))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("command", choices=["retain", "verify"])
    parser.add_argument("--raw", type=Path)
    parser.add_argument("--destination", type=Path, default=STUDY)
    args = parser.parse_args()
    if args.command == "retain":
        assert args.raw is not None
        retain(args.raw.resolve(), args.destination.resolve())
    else:
        verify(args.destination.resolve())

Overwriting _source_export/part1/research/anchored_neighbor_blend_v1/evidence.py


## part1/research/anchored_neighbor_blend_v1/neighbor_models.py

In [84]:
%%writefile _source_export/part1/research/anchored_neighbor_blend_v1/neighbor_models.py
"""Two isolated changes to the frozen anchored model's neighbor votes.

The inherited fit method preserves preprocessing, logistic regression, LDA,
cluster assistance, confidence, iterations and decoder. Modified neighbor votes
are used both while generating labels and at final prediction.
"""
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.preprocessing import normalize

from anchored_solution import AnchoredReasonClassifier


def embedding_groups(x):
    # Same rounded-coordinate hash as the frozen grouped benchmark.
    return pd.util.hash_pandas_object(pd.DataFrame(np.asarray(x)).round(10), index=False).to_numpy()


def centered(scores):
    """Map native .3 decision cutoff to .5; these are not calibrated probabilities."""
    return expit(logit(np.clip(scores, 1e-9, 1-1e-9))-logit(.3))


def decide(scores):
    result = (scores >= .5).astype(int)
    result[np.arange(len(scores)), scores.argmax(axis=1)] = 1
    return result


class NeighborAdjustedAnchored(AnchoredReasonClassifier):
    def __init__(self, mode):
        if mode not in ["row_control", "grouped_neighbors", "pseudo_neighbor_quarter"]:
            raise ValueError(mode)
        self.mode = mode

    def fit(self, embeddings, labels):
        x, y = np.asarray(embeddings, float), np.asarray(labels, int)
        known = np.isin(y, [0, 1]).all(axis=1)
        group = embedding_groups(x)
        self.gold_groups_ = group[known]
        self.masked_groups_ = group[~known]
        self.n_gold_ = int(known.sum())
        return super().fit(x, y)

    def _scores(self, z):
        logistic = np.zeros((len(z), 5))
        logistic[:, self.logistic_.classes_] = self.logistic_.predict_proba(z)
        bank_x, bank_y = self.neighbor_X_, self.neighbor_Y_
        generated = len(bank_x)-self.n_gold_
        groups = self.gold_groups_
        if generated:
            assert int(self.generated_mask_.sum()) == generated
            groups = np.r_[groups, self.masked_groups_[self.generated_mask_]]
        assert len(groups) == len(bank_x)
        distinct, first, inverse, counts = np.unique(
            groups, return_index=True, return_inverse=True, return_counts=True)
        if self.mode == "grouped_neighbors":
            # One vote per group; average conflicting/member label vectors.
            # Preserve first-occurrence order to make similarity ties deterministic.
            order = np.argsort(first)
            totals = np.zeros((len(distinct), 5))
            np.add.at(totals, inverse, bank_y)
            bank_x = bank_x[first[order]]
            bank_y = (totals/counts[:, None])[order]
        similarity = normalize(z) @ bank_x.T
        neighbors = np.argsort(-similarity, axis=1)[:, :10]
        weights = np.take_along_axis(similarity, neighbors, axis=1).clip(0)**2
        if self.mode == "pseudo_neighbor_quarter":
            # Keep top-ten retrieval unchanged; modify votes and denominator.
            weights *= np.where(neighbors < self.n_gold_, 1., .25)
        local = (weights[:, :, None]*bank_y[neighbors]).sum(axis=1)
        local /= weights.sum(axis=1, keepdims=True)+1e-9
        lda = np.zeros_like(logistic)
        lda[:, self.lda_.classes_] = self.lda_.predict_proba(z)
        self.neighbor_audit_ = dict(mode=self.mode, gold_rows=self.n_gold_,
                                    generated_rows=generated, original_bank_rows=len(groups),
                                    distinct_embedding_groups=len(distinct), voting_bank_rows=len(bank_x),
                                    largest_embedding_group=int(counts.max()),
                                    generated_vote_weight=.25 if self.mode == "pseudo_neighbor_quarter" else 1.)
        return (1-.3-.2)*logistic + .3*local + .2*lda

Overwriting _source_export/part1/research/anchored_neighbor_blend_v1/neighbor_models.py


## part1/research/anchored_neighbor_blend_v1/study.py

In [85]:
%%writefile _source_export/part1/research/anchored_neighbor_blend_v1/study.py
"""Fixed neighbor-vote and 75/25 ensemble experiments on all saved Part 1 splits."""
import argparse
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
from pathlib import Path
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

STUDY = Path(__file__).resolve().parent
PART1, ROOT = STUDY.parents[1], STUDY.parents[2]
FROZEN_HELPER = STUDY.parent/"anchored_transductive_v1"
sys.path[:0] = [str(PART1), str(FROZEN_HELPER)]
spec = importlib.util.spec_from_file_location("frozen_anchored_benchmark", FROZEN_HELPER/"run.py")
benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark)
from anchored_solution import AnchoredReasonClassifier
from neighbor_models import NeighborAdjustedAnchored, centered, decide, embedding_groups

REFERENCE = PART1/"experiments/current_leaders_grouped_v1"
LDA_ROOT = PART1/"research/lda_ensemble/experiments/ani_lda_revision_cv_v1"
LABELS, FEATURES = benchmark.LABELS, benchmark.FEATURES
MODES = ["anchored", "grouped_neighbors", "pseudo_neighbor_quarter", "anchored75_lda25", "lda_control"]
CANDIDATES = MODES[1:4]


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save(path, data):
    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False)+"\n")


def sources():
    paths = [Path(__file__), STUDY/"neighbor_models.py", STUDY/"check_neighbor_math.py", FROZEN_HELPER/"run.py", FROZEN_HELPER/"models.py",
             PART1/"anchored_solution.py", PART1/"TESTING_POLICY.md",
             PART1/"data/part1_train.csv", PART1/"data/part1_val.csv", PART1/"data/part1_test.csv",
             REFERENCE/"diagnostic_predictions.csv", REFERENCE/"comparison.csv",
             PART1/"submissions/results/anchored-trainval235.json",
             PART1/"submissions/results/lda-ensemble-trainval235.json",
             PART1/"submissions/archive/submitted/anchored_reason_trainval235__kaggle_0.89636.csv",
             PART1/"research/lda_ensemble/experiments/final_lda_ensemble_trainval235_v1/frozen_recipe.json"]
    for pool in ["train", "pooled"]:
        paths += [REFERENCE/f"anchored_{pool}.npz", REFERENCE/"snapshots"/f"lda_ensemble_{pool}.npz"]
    paths += [REFERENCE/"snapshots"/(s+".json") for s in ["train3", "train5", "pooled3", "pooled5", "chronology"]]
    paths += [LDA_ROOT/"supplementary"/f"window{i}_ensemble__revision_full.npz" for i in range(4)]
    return {str(p.relative_to(ROOT)): sha(p) for p in paths}


def register():
    p = STUDY/"protocol.json"
    assert not p.exists()
    save(p, dict(
        version="anchored-neighbor-blend-v1", registered_at=datetime.now(timezone.utc).isoformat(),
        parent="Frozen anchored 0.89636 recipe and predictions, unchanged inductive fitting.",
        authorization="User requested all three proposed changes: duplicate neighbor influence, generated-label vote weighting, and a fixed anchored/LDA blend.",
        candidates={
            "grouped_neighbors": "At each existing generated-label pass and final prediction, collapse the current neighbor bank by the saved rounded raw-embedding group. Use the first member's vector, average member binary label vectors, preserve first-occurrence order, and retrieve ten distinct groups with one cosine-squared weight each. LR, LDA and preprocessing still fit the unchanged row population.",
            "pseudo_neighbor_quarter": "Keep ordinary ten-row neighbor retrieval. Multiply each generated-row neighbor's cosine-squared weight by 0.25, gold by 1, then normalize by their summed weights. Apply in teaching and final prediction. All other settings fixed.",
            "anchored75_lda25": "0.75 * expit(logit(clip(anchored_score,1e-9,1-1e-9))-logit(0.3)) + 0.25 * frozen LDA ensemble score. Cutoff 0.5 plus nonempty argmax. Scores share a decision center; they are not asserted to be calibrated probabilities."
        },
        hypotheses=["Repeated embeddings may crowd distinct evidence out of a ten-row neighborhood.",
                    "Reduced generated-label influence may help neighbors avoid reinforcing mistakes, as anchored LR already does.",
                    "The two competitive submitted recipes may correct complementary errors."],
        controls="Fresh unchanged anchored fit on every case. Frozen LDA scores are reused only after exact saved-plan, target-ID, truth, shape and decoder checks; standalone LDA metrics must match the common comparison.",
        schedule="All eight saved five-fold repeats in BOTH pools and validation plus three chronological windows for all three candidates, without screening or tuning. 252 new scored fits (84 each for parent and two neighbor variants); blend and standalone LDA use 84 matched saved LDA evaluations.",
        implementation_checks="One first-fold row-control subclass refit must match the parent; synthetic checks ensure duplicate collapse, conflicting-label averaging and quarter-weight normalization implement the stated changes.",
        supplementary_exception="Evaluate every candidate on all four supplementary cases regardless of CV outcome, declared before scoring.",
        advancement="Positive mean paired sample-F1 difference and at least 5/8 wins against anchored in BOTH pools. Report validation and every chronological window without pooling unlike checks. Qualification only warrants review; no automatic leader replacement, final235 test fit, upload, merge or push.",
        access="Strict inductive: original saved fitting IDs/order and masked-group purges; Train157 excludes validation-source rows. No query/test inputs or labels enter fitting; actual test file is hashed only.",
        scoring="Complete OOF sample F1 for each repeat, population SD, paired repeat wins/ties/losses, macro F1 and all label precision/recall/F1/support. Chronology pooled across the same 98 held-out predictions.",
        no_search="Exactly three proposed arms. No combined neighbor variant, other vote weights, other blending weights, decoder search or selected seeds.",
        limitations="Reused development folds and small later-year samples; local improvement need not improve Kaggle. The registered 75/25 blend is evaluated on existing cross-fitted components, not fitted to outer labels.",
        retention="Keep complete predictions for qualifying candidates/useful controls; otherwise verify and retain compact recipes, all metrics, fit-access summaries and failure notes, then discard temporary trial-level outputs.",
        versions={"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
        sources=sources()))
    print("Registered", p)


def lda_for(case):
    if case["pool"] != "supplementary":
        path = REFERENCE/"snapshots"/f"lda_ensemble_{case['pool']}.npz"
        with np.load(path, allow_pickle=True) as z:
            b, ri = case["positions"], case["repeat_index"]
            data = dict(ids=z["ids"][b], truth=z["truth"][b], scores=z["scores"][ri, b], predictions=z["predictions"][ri, b])
    else:
        windows = json.loads((REFERENCE/"snapshots/chronology.json").read_text())["chronological_splits"]
        names = ["validation"]+[w["name"] for w in windows]
        path = LDA_ROOT/"supplementary"/f"window{names.index(case['key'])}_ensemble__revision_full.npz"
        with np.load(path, allow_pickle=True) as z:
            positions = {v: i for i, v in enumerate(z["ids"])}
            b = [positions[v] for v in case["query"].ID]
            data = {k: z[k][b] for k in ["ids", "truth", "scores", "predictions"]}
    np.testing.assert_array_equal(data["ids"], case["query"].ID)
    np.testing.assert_array_equal(data["truth"], case["query"][LABELS])
    np.testing.assert_array_equal(decide(data["scores"]), data["predictions"])
    return data["scores"]


def evaluate(case):
    start = time.monotonic()
    fit, query = case["fit"], case["query"]
    x, y, qx = fit[FEATURES].to_numpy(float), fit[LABELS].to_numpy(int), query[FEATURES].to_numpy(float)
    lda = lda_for(case)
    audits, scores = [], []
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        parent = AnchoredReasonClassifier().fit(x, y)
        raw = parent.predict_proba(qx)
        scores.append(centered(raw))
        audits.append(dict(mode="anchored", rounds=parent.round_audit_))
        np.testing.assert_array_equal(decide(scores[0]), parent.predict(qx))
        for mode in CANDIDATES[:2]:
            model = NeighborAdjustedAnchored(mode).fit(x, y)
            scores.append(centered(model.predict_proba(qx)))
            audits.append(dict(mode=mode, rounds=model.round_audit_, neighbors=model.neighbor_audit_))
        error = None
        if case["key"] == "train_20261126_0":
            control = NeighborAdjustedAnchored("row_control").fit(x, y).predict_proba(qx)
            np.testing.assert_allclose(control, raw, atol=1e-10, rtol=0)
            error = float(np.max(np.abs(control-raw)))
    scores += [.75*scores[0]+.25*lda, lda]
    scores = np.stack(scores)
    predictions = np.stack([decide(s) for s in scores])
    packet = dict(ids=query.ID.to_numpy(str), truth=query[LABELS].to_numpy(int), scores=scores,
                  predictions=predictions, parent_raw_scores=raw)
    assert np.isfinite(scores).all()
    audit = {k: v for k, v in case.items() if k not in ["fit", "query"]}
    audit.update(models=audits, input_rows=len(fit), gold_rows=int(fit[LABELS].ge(0).all(axis=1).sum()),
                 fit_groups=len(np.unique(embedding_groups(x))), target_rows=len(query),
                 fit_ids_sha256=hashlib.sha256("\n".join(fit.ID).encode()).hexdigest(),
                 query_inputs_used_for_fit=False, query_labels_used_for_fit=False,
                 row_control_max_error=error, seconds=time.monotonic()-start)
    return case["key"], packet, audit


def run(output, workers):
    protocol = json.loads((STUDY/"protocol.json").read_text())
    assert sources() == protocol["sources"]
    output = Path(output).resolve()
    assert not output.exists(), "Choose a new output directory."
    (output/"predictions").mkdir(parents=True)
    audits = []
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(evaluate, c) for c in benchmark.cases()]
        for future in as_completed(futures):
            key, packet, audit = future.result()
            np.savez_compressed(output/"predictions"/(key+".npz"), **packet)
            audits.append(audit)
            print(json.dumps(dict(completed=len(audits), total=84, case=key)), flush=True)
    save(output/"fit_audits.json", sorted(audits, key=lambda a: a["key"]))
    verify(output)


def verify(output):
    output = Path(output).resolve()
    assert sources() == json.loads((STUDY/"protocol.json").read_text())["sources"]
    refs = {}
    for pool in ["train", "pooled"]:
        with np.load(REFERENCE/f"anchored_{pool}.npz", allow_pickle=True) as z:
            refs[pool] = {k: z[k] for k in z.files}
    old_diagnostic = pd.read_csv(REFERENCE/"diagnostic_predictions.csv")
    old_diagnostic = old_diagnostic[old_diagnostic.name == "anchored_candidate"]
    entries, supplementary, errors = [], [], []
    for c in benchmark.cases():
        with np.load(output/"predictions"/(c["key"]+".npz"), allow_pickle=False) as z:
            np.testing.assert_array_equal(z["ids"], c["query"].ID)
            np.testing.assert_array_equal(z["truth"], c["query"][LABELS])
            assert z["scores"].shape == (len(MODES), len(c["query"]), 5)
            np.testing.assert_allclose(z["scores"][4], lda_for(c), atol=0, rtol=0)
            np.testing.assert_allclose(z["scores"][3], .75*centered(z["parent_raw_scores"])+.25*z["scores"][4], atol=0, rtol=0)
            if c["pool"] != "supplementary":
                b, ri, ref = c["positions"], c["repeat_index"], refs[c["pool"]]
                np.testing.assert_array_equal(z["ids"], ref["ids"][b])
                np.testing.assert_allclose(z["parent_raw_scores"], ref["scores"][ri, b], atol=1e-7, rtol=0)
                np.testing.assert_array_equal(z["predictions"][0], ref["predictions"][ri, b])
                errors.append(float(np.max(np.abs(z["parent_raw_scores"]-ref["scores"][ri, b]))))
            else:
                old = old_diagnostic[old_diagnostic["case"] == c["key"]].set_index("ID").loc[z["ids"]]
                np.testing.assert_array_equal(z["predictions"][0], old[["predicted_"+r for r in LABELS]])
            for i, mode in enumerate(MODES):
                np.testing.assert_array_equal(z["predictions"][i], decide(z["scores"][i]))
                frame = pd.DataFrame(dict(ID=z["ids"], pool=c["pool"], seed=c["seed"], case=c["key"], mode=mode))
                for j, label in enumerate(LABELS):
                    frame["actual_"+label] = z["truth"][:, j]
                    frame["predicted_"+label] = z["predictions"][i, :, j]
                entries.append(frame)
                if c["pool"] == "supplementary":
                    supplementary.append(dict(case=c["key"], mode=mode, **benchmark.metrics(z["truth"], z["predictions"][i])))
    combined = pd.concat(entries, ignore_index=True)
    truth, prediction = [[prefix+r for r in LABELS] for prefix in ["actual_", "predicted_"]]
    records = []
    for (pool, seed, mode), g in combined[combined.pool != "supplementary"].groupby(["pool", "seed", "mode"]):
        assert g.ID.is_unique and set(g.ID) == set(refs[pool]["ids"])
        records.append(dict(pool=pool, seed=int(seed), mode=mode, **benchmark.metrics(g[truth], g[prediction])))
    for mode, g in combined[(combined.pool == "supplementary") & (combined["case"] != "validation")].groupby("mode"):
        assert len(g) == 98 and g.ID.is_unique
        supplementary.append(dict(case="chronology98", mode=mode, **benchmark.metrics(g[truth], g[prediction])))
    repeats = pd.DataFrame(records)
    assert len(repeats) == 80
    summary = repeats.groupby(["pool", "mode"]).agg(mean_sample_f1=("sample_f1", "mean"),
             population_sd=("sample_f1", lambda s: s.std(ddof=0)), mean_macro_f1=("macro_f1", "mean")).reset_index()
    pairs, differences, qualifies = [], [], {m: True for m in CANDIDATES}
    for pool in ["train", "pooled"]:
        pivot = repeats[repeats.pool == pool].pivot(index="seed", columns="mode", values="sample_f1")
        for mode in CANDIDATES:
            d = pivot[mode]-pivot.anchored
            wins, ties, losses = int((d > 1e-12).sum()), int((d.abs() <= 1e-12).sum()), int((d < -1e-12).sum())
            passes = bool(d.mean() > 0 and wins >= 5)
            qualifies[mode] &= passes
            pairs.append(dict(pool=pool, mode=mode, mean_delta=float(d.mean()), wins=wins, ties=ties,
                              losses=losses, meets_pool_rule=passes))
            differences += [dict(pool=pool, mode=mode, seed=int(s), sample_f1_delta=float(v)) for s, v in d.items()]
    supp = pd.DataFrame(supplementary)
    # Independently establish that the reused LDA records are the published comparator.
    common = pd.read_csv(REFERENCE/"comparison.csv").set_index("name")
    for pool in ["train", "pooled"]:
        actual = summary[(summary.pool == pool) & (summary["mode"] == "lda_control")].mean_sample_f1.item()
        np.testing.assert_allclose(actual, common.loc["lda_ensemble", pool], atol=1e-12, rtol=0)
    for case in ["validation", "chronology98"]:
        actual = supp[(supp["case"] == case) & (supp["mode"] == "lda_control")].sample_f1.item()
        np.testing.assert_allclose(actual, common.loc["lda_ensemble", case], atol=1e-12, rtol=0)
    repeats.to_csv(output/"repeat_scores.csv", index=False)
    summary.to_csv(output/"cv_summary.csv", index=False)
    pd.DataFrame(pairs).to_csv(output/"paired_summary.csv", index=False)
    pd.DataFrame(differences).to_csv(output/"paired_repeat_deltas.csv", index=False)
    supp.to_csv(output/"supplementary_scores.csv", index=False)
    audits = json.loads((output/"fit_audits.json").read_text())
    assert len(audits) == 84 and all(not a["query_inputs_used_for_fit"] and not a["query_labels_used_for_fit"] for a in audits)
    check = dict(passed=True, new_cv_fits=240, new_supplementary_fits=12, implementation_control_fits=1,
                 reused_lda_cv_evaluations=80, reused_lda_supplementary_evaluations=4,
                 blend_evaluated_cases=84, parent_frozen_cv_checks=80, parent_supplementary_checks=4,
                 parent_max_score_error=max(errors), row_control_max_error=next(a["row_control_max_error"] for a in audits if a["row_control_max_error"] is not None),
                 repeats_scored=80, qualified_cv=qualifies, all_source_hashes_unchanged=True,
                 lda_standalone_matches_common_comparison=True, actual_test_inputs_used=False,
                 protocol_sha256=sha(STUDY/"protocol.json"))
    save(output/"verification.json", check)
    print(summary.to_string(index=False), flush=True)
    print(pd.DataFrame(pairs).to_string(index=False), flush=True)
    print(supp[supp["case"].isin(["validation", "chronology98"])][["case", "mode", "sample_f1", "macro_f1"]].to_string(index=False), flush=True)
    print(json.dumps(check, indent=2), flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("command", choices=["register", "run", "verify"])
    parser.add_argument("--output")
    parser.add_argument("--workers", type=int, default=4)
    args = parser.parse_args()
    if args.command == "register":
        register()
    elif args.command == "run":
        assert args.output and args.workers > 0
        run(args.output, args.workers)
    else:
        assert args.output
        verify(args.output)

Overwriting _source_export/part1/research/anchored_neighbor_blend_v1/study.py

## part1/research/anchored_transductive_v1/models.py

In [86]:
%%writefile _source_export/part1/research/anchored_transductive_v1/models.py
"""Two fixed target-input ablations of the frozen anchored classifier.

Only embedding arrays, never target labels, enter this interface. The original
classifier remains unchanged. Target self-training uses its exact fit method;
geometry-only copies that method with one change to the preprocessing pool.
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, normalize

from anchored_solution import AnchoredReasonClassifier, select_reasons


class GeometryOnlyAnchored(AnchoredReasonClassifier):
    def fit(self, embeddings, labels, target_embeddings):
        X, Y = np.asarray(embeddings, float), np.asarray(labels, int)
        Q = np.asarray(target_embeddings, float)
        if X.ndim != 2 or X.shape[1] != 150 or Y.shape != (len(X), 5):
            raise ValueError("Expected 150 embeddings and five labels per fitting row.")
        if Q.ndim != 2 or Q.shape[1] != 150:
            raise ValueError("Target inputs contain only 150 embeddings per row.")
        known = np.isin(Y, [0, 1]).all(axis=1)
        unknown = (Y == -100).all(axis=1)
        if (not (known | unknown).all() or (Y[known].sum(axis=1) == 0).any()
                or (Y[known].sum(axis=0) == 0).any()):
            raise ValueError("Complete nonempty known labels or five -100 values required.")
        # Sole algorithmic difference: fit geometry on eligible rows + targets.
        geometry = np.vstack([X, Q])
        self.scaler_ = StandardScaler().fit(geometry)
        Z = self.scaler_.transform(X)
        geometry_z = self.scaler_.transform(geometry)
        self.pca_ = PCA(30, random_state=0).fit(geometry_z)
        self.gmm_ = GaussianMixture(30, covariance_type="diag", n_init=3,
                                   random_state=0).fit(self.pca_.transform(geometry_z))
        assert self.gmm_.converged_
        K, V, targets = Z[known], Z[unknown], Y[known]
        Rk = self.gmm_.predict_proba(self.pca_.transform(K))
        Ru = self.gmm_.predict_proba(self.pca_.transform(V))
        counts = (targets / targets.sum(axis=1, keepdims=True)).sum(axis=0)
        self.class_weights_ = counts.mean() / counts
        Za, Ya, Ra = K, targets, Rk
        self.round_audit_ = []
        for step in range(3):
            row, reason = np.nonzero(Ya)
            weights = self.class_weights_[reason] / Ya.sum(axis=1)[row]
            weights *= np.where(row < len(K), 1., .25)
            self.logistic_ = LogisticRegression(C=.1, max_iter=3000, tol=1e-4).fit(
                Za[row], reason, sample_weight=weights)
            self.lda_ = LinearDiscriminantAnalysis(
                solver="lsqr", shrinkage="auto", priors=np.ones(5)/5).fit(Za[row], reason)
            self.neighbor_X_ = normalize(Za)
            self.neighbor_Y_ = Ya.copy()
            self.round_audit_.append(dict(
                step=step, known=len(K), generated=len(Za)-len(K),
                known_weight=float(weights[row < len(K)].sum()),
                generated_weight=float(weights[row >= len(K)].sum())))
            if step == 2:
                break
            T = Ya / Ya.sum(axis=1, keepdims=True)
            rates = (Ra.T @ T + T.mean(axis=0)) / (Ra.sum(axis=0)[:, None] + 1)
            scores = .95*self._scores(V) + .05*(Ru @ rates)
            self.generated_mask_ = scores.max(axis=1) >= .8
            self.generated_labels_ = select_reasons(scores[self.generated_mask_])
            Za = np.vstack([K, V[self.generated_mask_]])
            Ya = np.vstack([targets, self.generated_labels_])
            Ra = np.vstack([Rk, Ru[self.generated_mask_]])
        return self


def fit_predict(x, y, query_x, mode):
    """Use the same fixed model; change only which inputs fitting may see."""
    x, y, query_x = np.asarray(x, float), np.asarray(y, int), np.asarray(query_x, float)
    if mode == "inductive":
        model = AnchoredReasonClassifier().fit(x, y)
    elif mode == "geometry_only":
        model = GeometryOnlyAnchored().fit(x, y, query_x)
    elif mode == "target_self_training":
        target_y = np.full((len(query_x), 5), -100, dtype=int)
        model = AnchoredReasonClassifier().fit(np.vstack([x, query_x]), np.vstack([y, target_y]))
    else:
        raise ValueError(mode)
    known = np.isin(y, [0, 1]).all(axis=1)
    extra = len(query_x) if mode != "inductive" else 0
    assert model.scaler_.n_samples_seen_ == len(x) + extra
    accepted_targets = (int(model.generated_mask_[-len(query_x):].sum())
                        if mode == "target_self_training" else 0)
    audit = dict(mode=mode, gold_rows=int(known.sum()), original_masked_rows=int((~known).sum()),
                 target_rows=len(query_x), geometry_rows=len(x)+extra,
                 target_labels_supplied=False, target_generated_rows_in_final_fit=accepted_targets,
                 target_pseudo_labeling=mode == "target_self_training", rounds=model.round_audit_)
    return model.predict_proba(query_x), audit

Overwriting _source_export/part1/research/anchored_transductive_v1/models.py


## part1/research/anchored_transductive_v1/run.py

In [87]:
%%writefile _source_export/part1/research/anchored_transductive_v1/run.py
"""Register, run and independently verify the fixed transductive comparison."""
import argparse
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score, precision_recall_fscore_support
from threadpoolctl import threadpool_limits

STUDY = Path(__file__).resolve().parent
PART1 = STUDY.parents[1]
ROOT = PART1.parent
sys.path.insert(0, str(PART1))
from anchored_solution import AnchoredReasonClassifier, select_reasons
from models import GeometryOnlyAnchored, fit_predict

REFERENCE = PART1 / "experiments/current_leaders_grouped_v1"
LABELS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
MODES = ["inductive", "geometry_only", "target_self_training"]


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def groups(frame):
    return pd.util.hash_pandas_object(frame[FEATURES].round(10), index=False).to_numpy()


def metrics(y, prediction):
    precision, recall, f1, support = precision_recall_fscore_support(y, prediction, zero_division=0)
    return dict(sample_f1=float(f1_score(y, prediction, average="samples", zero_division=0)),
                macro_f1=float(f1.mean()), **{
                    f"{kind}_{label}": float(values[j])
                    for kind, values in [("precision", precision), ("recall", recall),
                                         ("f1", f1), ("support", support)]
                    for j, label in enumerate(LABELS)})


def sources():
    paths = [STUDY/"models.py", Path(__file__), PART1/"anchored_solution.py",
             PART1/"data/part1_train.csv", PART1/"data/part1_val.csv",
             PART1/"data/part1_test.csv", PART1/"TESTING_POLICY.md", ROOT/"Midterm_Project.pdf",
             REFERENCE/"anchored_train.npz", REFERENCE/"anchored_pooled.npz",
             REFERENCE/"diagnostic_predictions.csv",
             PART1/"submissions/archive/submitted/anchored_reason_trainval235__kaggle_0.89636.csv",
             PART1/"experiments/paired_improvement/candidate/manifest.json",
             PART1/"research/lda_ensemble/research/ani_snapshots/a548721/part1_best_model.ipynb"]
    paths += [REFERENCE/"snapshots"/(name+".json")
              for name in ["train3", "train5", "pooled3", "pooled5", "chronology"]]
    return {str(p.relative_to(ROOT)): sha(p) for p in paths}


def register():
    path = STUDY/"protocol.json"
    assert not path.exists(), "Never overwrite a registered study."
    save(path, dict(
        version="anchored-transductive-v1", registered_at=datetime.now(timezone.utc).isoformat(),
        parent="Unchanged anchored classifier; user-reported Kaggle 0.89636; inductive final235 fit.",
        hypothesis="Target embeddings may improve geometry; confident target-generated labels may add support or reinforce errors.",
        modes={
            "inductive": "Frozen parent fit on eligible gold and originally masked training inputs only.",
            "geometry_only": "Append target embeddings only when fitting StandardScaler, PCA30 and diagonal GMM30; target rows cannot receive generated training labels.",
            "target_self_training": "Append target embeddings with five -100 labels to the exact frozen parent fit; geometry and confidence-filtered target-generated labels may influence fitting."
        },
        settings="All parent features, random seeds, C, priors, weights, thresholds, two generated-label passes and decoder fixed; no tuning.",
        schedule="Predeclared full 8 repeats x 5 folds on both saved grouped pools plus validation78 and three chronological windows for EVERY mode: 252 scored fits.",
        supplementary_exception="Run all four supplementary fits even if CV fails, to answer this focused access question without post-result selection.",
        target_access="Only each evaluated batch's embeddings enter transductive fits. Target truth never enters fit_predict. Actual Kaggle test inputs are hashed for protection but not loaded or used in evaluation.",
        group_policy="Same gold/masked fitting IDs, original order and group purges as the frozen parent. Only the explicit target batch is added. Train157 excludes every validation-source row.",
        chronology="Same earlier fitting IDs and later query IDs; transductive variants deliberately see later query embeddings. These are transductive chronological diagnostics, not strict future-input-excluded forecasts.",
        original_ani_difference="Ani's final fit used test inputs for geometry and generated labels. Its old CV used globally fitted geometry and actual Kaggle inputs; this study instead refits each fold and uses its held-out batch as the target analogue.",
        advancement="Positive paired mean sample-F1 delta and at least 5/8 repeat wins in BOTH pools. Supplementary regressions remain visible; no automatic replacement, final test fit, upload or merge.",
        metrics="Complete OOF sample-F1 per repeat, mean, population SD, paired repeat deltas/wins/ties/losses, macro-F1 and every label precision/recall/F1/support; chronology pooled over 98 rows.",
        controls="Refit parent on all 84 cases; require every CV score array within 1e-7 and all CV/supplementary predicted labels identical to frozen records. Empty-target geometry implementation must match parent on the first fold.",
        limitations=["Reused development splits are not an untouched test.",
                     "CV/chronology target batches have fewer rows than the 312-row final test; a local effect may not transfer.",
                     "The PDF describes supplied unlabeled data, unsupervised augmentation and no test-label reverse engineering; it does not explicitly settle transductive submission permission."],
        retention="Keep code, protocol, aggregate/per-repeat/per-label results and verification. Full predictions only for a qualifying candidate or useful matched control; no fitted models or per-fold datasets.",
        versions={"python": platform.python_version(), "numpy": np.__version__,
                  "pandas": pd.__version__, "scikit_learn": sklearn.__version__},
        sources=sources()))
    print("Registered", path)


def cases():
    train = pd.read_csv(PART1/"data/part1_train.csv")
    val = pd.read_csv(PART1/"data/part1_val.csv")
    full = pd.concat([train, val], ignore_index=True)
    masked = train[train[LABELS].eq(-100).all(axis=1)].reset_index(drop=True)
    result = []
    for pool, frame in [("train", train), ("pooled", full)]:
        gold = frame[frame[LABELS].ge(0).all(axis=1)].reset_index(drop=True)
        plans = [json.loads((REFERENCE/"snapshots"/f"{pool}{n}.json").read_text()) for n in [3, 5]]
        assert plans[0]["gold_ids"] == plans[1]["gold_ids"] == gold.ID.tolist()
        seeds = plans[0]["seeds"] + plans[1]["seeds"]
        order = {v: i for i, v in enumerate(frame.ID)}
        gg, ug = groups(gold), groups(masked)
        for f in plans[0]["folds"] + plans[1]["folds"]:
            a, b = np.array(f["train_gold_positions"]), np.array(f["heldout_gold_positions"])
            assert set(a).isdisjoint(b) and len(a)+len(b) == len(gold)
            assert set(gg[a]).isdisjoint(gg[b]) and gold.iloc[b].ID.tolist() == f["heldout_ids"]
            keep = ~np.isin(ug, gg[b])
            assert int((~keep).sum()) == f["masked_duplicates_excluded"]
            fit = pd.concat([gold.iloc[a], masked[keep]], ignore_index=True)
            fit = fit.iloc[np.argsort([order[v] for v in fit.ID])].reset_index(drop=True)
            query = gold.iloc[b].reset_index(drop=True)
            assert set(groups(fit)).isdisjoint(groups(query)) and set(fit.ID).isdisjoint(query.ID)
            if pool == "train":
                assert set(fit.ID).isdisjoint(val.ID) and set(query.ID).isdisjoint(val.ID)
            result.append(dict(key=f"{pool}_{f['seed']}_{f['fold']}", pool=pool, seed=f["seed"],
                               fold=f["fold"], repeat_index=seeds.index(f["seed"]), positions=b.tolist(),
                               purged_masked=int((~keep).sum()), fit=fit, query=query))
    result.append(dict(key="validation", pool="supplementary", seed=-1, fold=-1,
                       repeat_index=-1, positions=[], purged_masked=0, fit=train, query=val))
    chrono = json.loads((REFERENCE/"snapshots/chronology.json").read_text())
    indexed = full.set_index("ID", drop=False)
    for w in chrono["chronological_splits"]:
        fit, query = indexed.loc[w["training_ids"]], indexed.loc[w["heldout_ids"]]
        assert fit[LABELS].ge(0).all(axis=1).sum() == w["gold_rows"]
        assert fit[LABELS].eq(-100).all(axis=1).sum() == w["masked_rows"]
        assert len(query) == w["heldout_rows"] and set(groups(fit)).isdisjoint(groups(query))
        assert set(fit.ID).isdisjoint(query.ID)
        result.append(dict(key=w["name"], pool="supplementary", seed=-1, fold=-1,
                           repeat_index=-1, positions=[], purged_masked=w["purged_earlier_masked"],
                           fit=fit.reset_index(drop=True), query=query.reset_index(drop=True)))
    assert len(result) == 84
    return result


def evaluate_case(case):
    start = time.monotonic()
    fit, query = case["fit"], case["query"]
    x, y = fit[FEATURES].to_numpy(float), fit[LABELS].to_numpy(int)
    qx = query[FEATURES].to_numpy(float)  # Label-free interface for every mode.
    scores, audits = [], []
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        for mode in MODES:
            score, audit = fit_predict(x, y, qx, mode)
            assert np.isfinite(score).all() and score.shape == (len(qx), 5)
            scores.append(score)
            audits.append(audit)
        equivalence = None
        if case["key"] == "train_20261126_0":
            empty = GeometryOnlyAnchored().fit(x, y, np.empty((0, 150))).predict_proba(qx)
            np.testing.assert_allclose(empty, scores[0], atol=1e-7, rtol=0)
            np.testing.assert_array_equal(select_reasons(empty), select_reasons(scores[0]))
            equivalence = float(np.max(np.abs(empty-scores[0])))
    data = dict(ids=query.ID.to_numpy(str), truth=query[LABELS].to_numpy(int), scores=np.stack(scores))
    data["predictions"] = np.stack([select_reasons(s) for s in scores])
    audit = {k: v for k, v in case.items() if k not in ["fit", "query"]}
    audit.update(fit_ids_sha256=hashlib.sha256("\n".join(fit.ID).encode()).hexdigest(),
                 query_ids_sha256=hashlib.sha256("\n".join(query.ID).encode()).hexdigest(),
                 modes=audits, seconds=time.monotonic()-start, empty_target_max_error=equivalence)
    return case["key"], data, audit


def run(output, workers):
    protocol = json.loads((STUDY/"protocol.json").read_text())
    assert protocol["sources"] == sources(), "Registered sources changed. Version the study instead."
    output = Path(output).resolve()
    assert not output.exists(), "Choose a new output directory."
    output.mkdir(parents=True)
    (output/"predictions").mkdir()
    job_cases = cases()
    audits = []
    with ProcessPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(evaluate_case, c) for c in job_cases]
        for future in as_completed(futures):
            key, data, audit = future.result()
            np.savez_compressed(output/"predictions"/(key+".npz"), **data)
            audits.append(audit)
            print(json.dumps(dict(completed=len(audits), total=84, case=key,
                                  seconds=round(audit["seconds"], 2))), flush=True)
    save(output/"fit_audits.json", sorted(audits, key=lambda a: a["key"]))
    assert sources() == protocol["sources"]
    verify(output)


def verify(output):
    output = Path(output).resolve()
    protocol = json.loads((STUDY/"protocol.json").read_text())
    assert sources() == protocol["sources"]
    job_cases = cases()
    refs = {}
    for pool in ["train", "pooled"]:
        with np.load(REFERENCE/f"anchored_{pool}.npz", allow_pickle=True) as z:
            refs[pool] = {k: z[k] for k in z.files}
    diagnostic = pd.read_csv(REFERENCE/"diagnostic_predictions.csv")
    diagnostic = diagnostic[diagnostic.name == "anchored_candidate"]
    entries, score_rows, check_errors = [], [], []
    for c in job_cases:
        with np.load(output/"predictions"/(c["key"]+".npz"), allow_pickle=False) as z:
            np.testing.assert_array_equal(z["ids"], c["query"].ID)
            np.testing.assert_array_equal(z["truth"], c["query"][LABELS])
            assert z["scores"].shape == (3, len(c["query"]), 5)
            for i in range(3):
                np.testing.assert_array_equal(select_reasons(z["scores"][i]), z["predictions"][i])
            if c["pool"] != "supplementary":
                ref = refs[c["pool"]]
                b = c["positions"]
                np.testing.assert_array_equal(z["ids"], ref["ids"][b])
                np.testing.assert_array_equal(z["truth"], ref["truth"][b])
                old = ref["scores"][c["repeat_index"], b]
                np.testing.assert_allclose(z["scores"][0], old, atol=1e-7, rtol=0)
                np.testing.assert_array_equal(z["predictions"][0], ref["predictions"][c["repeat_index"], b])
                check_errors.append(float(np.max(np.abs(z["scores"][0]-old))))
            else:
                old = diagnostic[diagnostic["case"] == c["key"]].set_index("ID").loc[z["ids"]]
                np.testing.assert_array_equal(z["truth"], old[["actual_"+s for s in LABELS]])
                np.testing.assert_array_equal(z["predictions"][0], old[["predicted_"+s for s in LABELS]])
            for i, mode in enumerate(MODES):
                entry = pd.DataFrame(dict(ID=z["ids"], pool=c["pool"], seed=c["seed"],
                                          case=c["key"], mode=mode))
                for j, label in enumerate(LABELS):
                    entry["actual_"+label] = z["truth"][:, j]
                    entry["predicted_"+label] = z["predictions"][i, :, j]
                entries.append(entry)
                if c["pool"] == "supplementary":
                    score_rows.append(dict(case=c["key"], mode=mode, **metrics(z["truth"], z["predictions"][i])))
    frame = pd.concat(entries, ignore_index=True)
    truth_cols, pred_cols = [[prefix+s for s in LABELS] for prefix in ["actual_", "predicted_"]]
    repeats = []
    for (pool, seed, mode), g in frame[frame.pool != "supplementary"].groupby(["pool", "seed", "mode"]):
        assert g.ID.is_unique and len(g) == (157 if pool == "train" else 235)
        assert set(g.ID) == set(refs[pool]["ids"])
        repeats.append(dict(pool=pool, seed=int(seed), mode=mode, **metrics(g[truth_cols], g[pred_cols])))
    for mode, g in frame[(frame.pool == "supplementary") & (frame["case"] != "validation")].groupby("mode"):
        assert len(g) == 98 and g.ID.is_unique
        score_rows.append(dict(case="chronology98", mode=mode, **metrics(g[truth_cols], g[pred_cols])))
    repeats = pd.DataFrame(repeats)
    assert len(repeats) == 48
    repeats.to_csv(output/"repeat_scores.csv", index=False)
    summary = repeats.groupby(["pool", "mode"]).agg(
        mean_sample_f1=("sample_f1", "mean"), population_sd=("sample_f1", lambda s: s.std(ddof=0)),
        mean_macro_f1=("macro_f1", "mean")).reset_index()
    summary.to_csv(output/"cv_summary.csv", index=False)
    pairs, deltas, qualified = [], [], {m: True for m in MODES[1:]}
    for pool in ["train", "pooled"]:
        pivot = repeats[repeats.pool == pool].pivot(index="seed", columns="mode", values="sample_f1")
        for mode in MODES[1:]:
            delta = pivot[mode] - pivot.inductive
            wins, ties, losses = int((delta > 1e-12).sum()), int((delta.abs() <= 1e-12).sum()), int((delta < -1e-12).sum())
            passed = bool(delta.mean() > 0 and wins >= 5)
            qualified[mode] &= passed
            pairs.append(dict(pool=pool, mode=mode, mean_delta=float(delta.mean()), wins=wins,
                              ties=ties, losses=losses, meets_pool_rule=passed))
            deltas += [dict(pool=pool, mode=mode, seed=int(seed), sample_f1_delta=float(d)) for seed, d in delta.items()]
    pd.DataFrame(pairs).to_csv(output/"paired_summary.csv", index=False)
    pd.DataFrame(deltas).to_csv(output/"paired_repeat_deltas.csv", index=False)
    supplementary = pd.DataFrame(score_rows)
    supplementary.to_csv(output/"supplementary_scores.csv", index=False)
    audits = json.loads((output/"fit_audits.json").read_text())
    assert len(audits) == 84 and {a["key"] for a in audits} == {c["key"] for c in job_cases}
    for a in audits:
        assert len(a["modes"]) == 3
        for m in a["modes"]:
            assert m["target_labels_supplied"] is False
            assert m["gold_rows"] == m["rounds"][0]["known"]
            if m["mode"] != "target_self_training":
                assert m["target_generated_rows_in_final_fit"] == 0
    check = dict(passed=True, full_cv_fits=240, supplementary_fits=12,
                 implementation_equivalence_fits=1, frozen_parent_cv_checks=80,
                 frozen_parent_supplementary_checks=4, parent_max_score_error=max(check_errors),
                 empty_target_max_error=next(a["empty_target_max_error"] for a in audits if a["empty_target_max_error"] is not None),
                 repeats_scored=48, qualified_cv=qualified, protected_hashes_unchanged=True,
                 query_truth_passed_to_model=False, actual_kaggle_test_inputs_used=False,
                 new_test_predictions=False, protocol_sha256=sha(STUDY/"protocol.json"))
    save(output/"verification.json", check)
    print(summary.to_string(index=False), flush=True)
    print(pd.DataFrame(pairs).to_string(index=False), flush=True)
    print(supplementary[supplementary["case"].isin(["validation", "chronology98"])][["case", "mode", "sample_f1", "macro_f1"]].to_string(index=False), flush=True)
    print(json.dumps(check, indent=2), flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("command", choices=["register", "run", "verify"])
    parser.add_argument("--output")
    parser.add_argument("--workers", type=int, default=4)
    args = parser.parse_args()
    if args.command == "register":
        register()
    elif args.command == "run":
        assert args.output and args.workers > 0
        run(args.output, args.workers)
    else:
        assert args.output
        verify(args.output)

Overwriting _source_export/part1/research/anchored_transductive_v1/run.py


## part1/research/anchored_transductive_v1/verify_retained.py

In [88]:
%%writefile _source_export/part1/research/anchored_transductive_v1/verify_retained.py
"""Read-only verification of compact records; no fitting or discarded-cache rescoring."""
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

STUDY = Path(__file__).resolve().parent
ROOT = STUDY.parents[2]
RESULTS = STUDY/"results"


def main():
    manifest = json.loads((STUDY/"retention.json").read_text())
    protocol = json.loads((STUDY/"protocol.json").read_text())
    for relative, record in manifest["files"].items():
        data = (STUDY/relative).read_bytes()
        assert len(data) == record["bytes"]
        assert hashlib.sha256(data).hexdigest() == record["sha256"], relative
    for relative, digest in protocol["sources"].items():
        assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest() == digest, relative
    check = json.loads((RESULTS/"verification.json").read_text())
    assert check["protocol_sha256"] == hashlib.sha256((STUDY/"protocol.json").read_bytes()).hexdigest()
    assert check["passed"] and check["full_cv_fits"] == 240 and check["supplementary_fits"] == 12
    repeats = pd.read_csv(RESULTS/"repeat_scores.csv")
    summary = pd.read_csv(RESULTS/"cv_summary.csv")
    pairs = pd.read_csv(RESULTS/"paired_summary.csv")
    deltas = pd.read_csv(RESULTS/"paired_repeat_deltas.csv")
    assert len(repeats) == 48 and len(summary) == 6 and len(pairs) == 4 and len(deltas) == 32
    label_f1 = ["f1_"+s for s in ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]]
    np.testing.assert_allclose(repeats[label_f1].mean(axis=1), repeats.macro_f1, atol=1e-12, rtol=0)
    qualified = {m: True for m in ["geometry_only", "target_self_training"]}
    for row in summary.itertuples():
        group = repeats[(repeats.pool == row.pool) & (repeats["mode"] == row.mode)]
        assert len(group) == 8 and group.seed.is_unique
        np.testing.assert_allclose([group.sample_f1.mean(), group.sample_f1.std(ddof=0), group.macro_f1.mean()],
                                   [row.mean_sample_f1, row.population_sd, row.mean_macro_f1], atol=1e-12, rtol=0)
    for row in pairs.itertuples():
        p = repeats[repeats.pool == row.pool].pivot(index="seed", columns="mode", values="sample_f1")
        d = p[row.mode]-p.inductive
        saved = deltas[(deltas.pool == row.pool) & (deltas["mode"] == row.mode)].set_index("seed").loc[d.index]
        np.testing.assert_allclose(saved.sample_f1_delta, d, atol=1e-12, rtol=0)
        wins, ties, losses = (d > 1e-12).sum(), (d.abs() <= 1e-12).sum(), (d < -1e-12).sum()
        assert (wins, ties, losses) == (row.wins, row.ties, row.losses)
        np.testing.assert_allclose(d.mean(), row.mean_delta, atol=1e-12, rtol=0)
        passed = bool(d.mean() > 0 and wins >= 5)
        assert passed == row.meets_pool_rule
        qualified[row.mode] &= passed
    assert qualified == check["qualified_cv"] and not any(qualified.values())
    supplementary = pd.read_csv(RESULTS/"supplementary_scores.csv")
    assert len(supplementary) == 15
    np.testing.assert_allclose(supplementary[label_f1].mean(axis=1), supplementary.macro_f1, atol=1e-12, rtol=0)
    chronology = json.loads((ROOT/"part1/experiments/current_leaders_grouped_v1/snapshots/chronology.json").read_text())
    for mode in qualified.keys() | {"inductive"}:
        scores = supplementary[supplementary["mode"] == mode].set_index("case").sample_f1
        total = sum(w["heldout_rows"] for w in chronology["chronological_splits"])
        assert total == 98
        pooled = sum(scores[w["name"]]*w["heldout_rows"] for w in chronology["chronological_splits"])/total
        np.testing.assert_allclose(pooled, scores["chronology98"], atol=1e-12, rtol=0)
    result = dict(passed=True, retained_files_checked=len(manifest["files"]),
                  protected_sources_checked=len(protocol["sources"]), repeat_records=48,
                  supplementary_records=15, qualified_cv=qualified, models_fitted=0,
                  discarded_trial_predictions_rescored=False)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()

Overwriting _source_export/part1/research/anchored_transductive_v1/verify_retained.py


## part1/research/complementary_heads_v1/check_complement.py

In [89]:
%%writefile _source_export/part1/research/complementary_heads_v1/check_complement.py
"""Finite-difference the new nonconvex loss and verify nested group boundaries."""
import numpy as np
from scipy.optimize import approx_fprime
from scipy.special import expit
from threadpoolctl import threadpool_limits
import head_study as study
from complement import loss_gradient, ComplementaryKernel, centered_logit, decide, fit_fusion


def check():
    rng=np.random.default_rng(1127)
    f=np.c_[np.ones(19),rng.normal(size=(19,6))]
    y=rng.integers(0,2,size=(19,5))
    error=rng.normal(size=(19,5));error-=error.mean(0)
    w=rng.normal(0,.2,size=35)
    errors=[]
    for strength in [0.,.1,.3]:
        _,g=loss_gradient(w,f,y,error,strength)
        numeric=approx_fprime(w,lambda b:loss_gradient(b,f,y,error,strength)[0],1e-7)
        delta=float(np.abs(g-numeric).max());assert delta<2e-7,delta;errors.append(delta)
    # Calibration zero-correction boundary matches anchored's exact decoder.
    scores=rng.uniform(.001,.999,size=(80,5))
    np.testing.assert_array_equal(study.select_reasons(scores),decide(expit(centered_logit(scores))))
    checked=0;retry_seeds=set()
    for case in study.cases():
        study.reference(case)
        assert set(case['fit'].ID).isdisjoint(case['query'].ID)
        if case['key']!='validation':
            assert not set(study.benchmark.groups(case['fit']))&set(study.benchmark.groups(case['query']))
        for _,_,part,held,seed in study.inner_inputs(case['fit'],270301):
            retry_seeds.add(seed);checked+=1
            for _,_,sub,subheld,subseed in study.inner_inputs(part,270302):
                assert set(sub.ID).isdisjoint(held.ID)
                assert set(subheld.ID).isdisjoint(held.ID)
                retry_seeds.add(subseed);checked+=1
    # The full head optimizer and readout can run without any query-label access.
    X=rng.normal(size=(180,150));Y=np.full((180,5),-100)
    Y[:90]=rng.integers(0,2,size=(90,5));anchor=rng.uniform(.01,.99,size=(90,5))
    with threadpool_limits(limits=1):
        model=ComplementaryKernel().fit(X,Y,anchor)
        logits=model.predict_logits(X[:90])
        for name,v in logits.items():
            assert v.shape==(90,5) and np.isfinite(v).all()
            p,_=fit_fusion(anchor,v,Y[:90],anchor[:7],v[:7]);assert p.shape==(7,5)
    result=dict(passed=True,max_gradient_absolute_errors=errors,
        checked_nested_partitions=checked,chosen_inner_seeds=sorted(retry_seeds),
        synthetic_optimizer_and_fusion=True,decoder_equivalence=True)
    print(result);return result


if __name__=='__main__':study.save(study.HERE/'math_and_boundaries.json',check())

Overwriting _source_export/part1/research/complementary_heads_v1/check_complement.py


## part1/research/complementary_heads_v1/complement.py

In [90]:
%%writefile _source_export/part1/research/complementary_heads_v1/complement.py
"""Classical nonlinear head with an explicit, supervised error-diversity loss.

Anchored is frozen. Its held-out residuals guide a Nyström kernel logistic head;
the supervised loss is never reversed. This is not joint neural-head training.
"""
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.kernel_approximation import Nystroem
from sklearn.preprocessing import StandardScaler

STRENGTHS = {'ordinary': 0., 'diverse10': .1, 'diverse30': .3}
RIDGE = .005


def centered_logit(scores):
    return logit(np.clip(scores, 1e-6, 1-1e-6))-logit(.3)


def decide(scores):
    result = (scores >= .5).astype(int)
    result[np.arange(len(result)), scores.argmax(1)] = 1
    return result


def loss_gradient(flat, features, labels, anchor_error, strength, ridge=RIDGE):
    """Mean BCE per class + L2 + strength/2 * summed squared error correlations."""
    n, d = features.shape
    W = flat.reshape(d, 5)
    logits = features @ W
    probabilities = expit(logits)
    error = probabilities-labels
    loss = np.logaddexp(0., logits).sum()/n-(labels*logits).sum()/n
    loss += ridge/2*np.square(W[1:]).sum()+1e-6/2*np.square(W[0]).sum()
    derivative = error/n
    residual = error-error.mean(0)
    va = np.square(anchor_error).mean(0)+1e-8
    vb = np.square(residual).mean(0)+1e-8
    covariance = (anchor_error*residual).mean(0)
    corr2 = np.square(covariance)/(va*vb)
    if strength:
        loss += strength/2*corr2.sum()
        dprob = strength/n*(covariance/(va*vb)*anchor_error
                           -np.square(covariance)/(va*vb**2)*residual)
        derivative += dprob*probabilities*(1-probabilities)
    gradient = features.T @ derivative
    gradient[1:] += ridge*W[1:]
    gradient[0] += 1e-6*W[0]
    return float(loss), gradient.ravel()


class ComplementaryKernel:
    def fit(self, X, Y, anchor_oof):
        gold = (Y >= 0).all(1)
        y = Y[gold]
        assert anchor_oof.shape == y.shape
        self.scaler = StandardScaler().fit(X)
        self.map = Nystroem(kernel='rbf', gamma=1/150, n_components=128,
                           random_state=270303, n_jobs=1)
        transformed = self.map.fit_transform(self.scaler.transform(X))
        features = np.c_[np.ones(len(y)), transformed[gold]]
        error = expit(centered_logit(anchor_oof))-y
        error -= error.mean(0)
        initial = np.zeros((features.shape[1], 5))
        initial[0] = logit(np.clip(y.mean(0), .02, .98))
        self.weights = {}
        self.audit = {}
        for name, strength in STRENGTHS.items():
            start = initial.ravel() if name == 'ordinary' else self.weights['ordinary'].ravel()
            fit = minimize(loss_gradient, start, args=(features, y, error, strength),
                           method='L-BFGS-B', jac=True,
                           options={'maxiter': 1000, 'ftol': 1e-11, 'gtol': 1e-7})
            assert fit.success, fit.message
            self.weights[name] = fit.x.reshape(features.shape[1], 5)
            residual = expit(features @ self.weights[name])-y
            residual -= residual.mean(0)
            correlation = (error*residual).mean(0)/np.sqrt(
                (np.square(error).mean(0)+1e-8)*(np.square(residual).mean(0)+1e-8))
            self.audit[name] = dict(iterations=int(fit.nit), objective=float(fit.fun),
                max_gradient=float(np.abs(fit.jac).max()),
                training_error_correlation=correlation.tolist(),
                training_mean_squared_error_correlation=float(np.square(correlation).mean()))
        return self

    def predict_logits(self, X):
        features = np.c_[np.ones(len(X)), self.map.transform(self.scaler.transform(X))]
        return {name: features @ W for name, W in self.weights.items()}


def fusion_features(anchor, second_logits=None):
    # Comparable label-indexed pre-decision scores across independently fitted heads.
    # No fold-specific kernel landmarks are incorrectly treated as aligned features.
    a = np.clip(centered_logit(anchor), -6, 6)/3
    if second_logits is None:
        return np.c_[np.ones(len(a)), a]
    return np.c_[np.ones(len(a)), a, np.clip(second_logits, -6, 6)/3]


def fit_fusion(anchor_oof, second_oof, labels, anchor_query, second_query):
    """Five ridge-logistic readouts of all ten head logits, offset at anchored."""
    F = fusion_features(anchor_oof, second_oof)
    Q = fusion_features(anchor_query, second_query)
    offset = centered_logit(anchor_oof)
    ridge = 20.

    def objective(flat):
        W = flat.reshape(F.shape[1], 5)
        logits = offset+F @ W
        value = np.logaddexp(0., logits).sum()-(labels*logits).sum()+ridge/2*np.square(W).sum()
        gradient = F.T @ (expit(logits)-labels)+ridge*W
        return float(value), gradient.ravel()

    fit = minimize(objective, np.zeros(F.shape[1]*5), method='L-BFGS-B', jac=True,
                   options={'maxiter': 500, 'ftol': 1e-12, 'gtol': 1e-7})
    assert fit.success, fit.message
    weights = fit.x.reshape(F.shape[1], 5)
    return expit(centered_logit(anchor_query)+Q @ weights), weights

Overwriting _source_export/part1/research/complementary_heads_v1/complement.py


## part1/research/complementary_heads_v1/head_study.py

In [91]:
%%writefile _source_export/part1/research/complementary_heads_v1/head_study.py
"""Frozen, nested evaluation of supervised error-diverse kernel heads."""
import os
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    os.environ[variable] = '1'
import argparse
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
from pathlib import Path
import platform
import sys
import time
import numpy as np
import pandas as pd
import sklearn
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from scipy.special import expit
from sklearn.metrics import f1_score, precision_recall_fscore_support
from threadpoolctl import threadpool_limits

HERE = Path(__file__).resolve().parent
P1 = HERE.parents[1]
ROOT = P1.parent
HELPER = P1/'research/anchored_transductive_v1'
sys.path[:0] = [str(HERE), str(P1), str(HELPER)]
from anchored_solution import AnchoredReasonClassifier, select_reasons
from complement import ComplementaryKernel, STRENGTHS, centered_logit, decide, fit_fusion
spec = importlib.util.spec_from_file_location('_complement_benchmark', HELPER/'run.py')
benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark)
LABELS, EMB = benchmark.LABELS, benchmark.FEATURES
CANDIDATES = [prefix+name for prefix in ['blend_', 'fusion_'] for name in STRENGTHS]
NAMES = ['parent', 'anchored', 'calibration'] + ['head_'+n for n in STRENGTHS] + CANDIDATES


def read(path): return json.loads(Path(path).read_text())
def save(path, value): Path(path).write_text(json.dumps(value, indent=2, allow_nan=False)+'\n')
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def idhash(ids): return hashlib.sha256('\n'.join(ids).encode()).hexdigest()
def loadnp(path):
    with np.load(path, allow_pickle=False) as z: return {k: z[k].copy() for k in z.files}


def cases():
    result = benchmark.cases()
    for case in result:
        for key in ['fit', 'query']:
            case[key] = case[key][['ID', 'start_year']+EMB+LABELS].reset_index(drop=True)
        case['reference_key'] = 'strict_inductive__'+('grouped_' if case['pool'] != 'supplementary' else '')+case['key']
    return result


def reference(case):
    z = loadnp(HERE/'parent_inputs.npz')
    keep = z['case'] == case['reference_key']
    np.testing.assert_array_equal(z['ids'][keep], case['query'].ID)
    np.testing.assert_array_equal(z['truth'][keep], case['query'][LABELS])
    return z['anchored'][keep], z['cand2'][keep]


def inner_inputs(frame, seed):
    gold = frame[frame[LABELS].ge(0).all(axis=1)].reset_index(drop=True)
    gg, fg = benchmark.groups(gold), benchmark.groups(frame)
    _, inv = np.unique(gg, return_inverse=True)
    target = np.zeros((inv.max()+1, 5), int)
    np.maximum.at(target, inv, gold[LABELS].to_numpy(int))
    # Predeclared deterministic retry only for class coverage, never for scores.
    for chosen_seed in range(seed, seed+20):
        splitter = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=chosen_seed)
        proposed = list(splitter.split(np.zeros((len(target), 1)), target))
        if all((target[tr].sum(0) > 0).all() for tr, _ in proposed): break
    else: raise ValueError('No class-complete three-fold inner split')
    for fold, (_, te) in enumerate(proposed):
        positions = np.flatnonzero(np.isin(inv, te))
        held = gold.iloc[positions].reset_index(drop=True)
        part = frame[~np.isin(fg, gg[positions])].reset_index(drop=True)
        assert not set(benchmark.groups(part)) & set(benchmark.groups(held))
        assert set(part.ID).isdisjoint(held.ID)
        yield fold, positions, part, held, chosen_seed


def anchor_predict(fit, query_x):
    return AnchoredReasonClassifier().fit(fit[EMB].to_numpy(float), fit[LABELS].to_numpy(int)).predict_proba(query_x)


def split_audit(part, held, seed, fold):
    return dict(seed=seed, fold=fold, fitting_ids_sha256=idhash(part.ID),
                query_ids_sha256=idhash(held.ID), gold=int(part[LABELS].ge(0).all(axis=1).sum()),
                masked=int(part[LABELS].eq(-100).all(axis=1).sum()), query=len(held), group_overlap=0)


def crossfit_anchor(frame):
    n = int(frame[LABELS].ge(0).all(axis=1).sum())
    scores = np.zeros((n, 5)); coverage = np.zeros(n, int); audit = []
    for fold, positions, part, held, seed in inner_inputs(frame, 270302):
        scores[positions] = anchor_predict(part, held[EMB].to_numpy(float))
        coverage[positions] += 1
        audit.append(split_audit(part, held, seed, fold))
    assert (coverage == 1).all()
    return scores, audit


def fit_pair(fit, query_x):
    """No query IDs or labels are accepted by the model interface."""
    gold = fit[fit[LABELS].ge(0).all(axis=1)].reset_index(drop=True)
    y = gold[LABELS].to_numpy(int)
    oa = np.zeros_like(y, dtype=float)
    ob = {name: np.zeros_like(oa) for name in STRENGTHS}
    coverage = np.zeros(len(gold), int); audit = []
    for fold, positions, part, held, seed in inner_inputs(fit, 270301):
        hx = held[EMB].to_numpy(float)
        oa[positions] = anchor_predict(part, hx)
        training_anchor, subaudit = crossfit_anchor(part)
        kernel = ComplementaryKernel().fit(part[EMB].to_numpy(float), part[LABELS].to_numpy(int), training_anchor)
        for name, score in kernel.predict_logits(hx).items(): ob[name][positions] = score
        coverage[positions] += 1
        item = split_audit(part, held, seed, fold)
        item.update(error_training_subfolds=subaudit, optimization=kernel.audit)
        audit.append(item)
    assert (coverage == 1).all()
    a = anchor_predict(fit, query_x)
    kernel = ComplementaryKernel().fit(fit[EMB].to_numpy(float), fit[LABELS].to_numpy(int), oa)
    b = kernel.predict_logits(query_x)
    scores = {'anchored': expit(centered_logit(a))}
    scores['calibration'], cw = fit_fusion(oa, None, y, a, None)
    weights = {'calibration': cw.tolist()}
    for name in STRENGTHS:
        scores['head_'+name] = expit(b[name])
        scores['blend_'+name] = .8*scores['anchored']+.2*expit(b[name])
        scores['fusion_'+name], w = fit_fusion(oa, ob[name], y, a, b[name])
        weights[name] = w.tolist()
    return scores, dict(inner=audit, optimization=kernel.audit, fusion_weights=weights), a


def sources():
    paths = [HERE/name for name in ['head_study.py', 'complement.py', 'check_complement.py',
             'parent_inputs.npz', 'parent_reported_result.json', 'provenance.json']]
    paths += [P1/'anchored_solution.py', P1/'TESTING_POLICY.md', HELPER/'run.py', HELPER/'models.py']
    paths += list((P1/'data').glob('*.csv'))
    paths += [P1/'experiments/current_leaders_grouped_v1/snapshots'/f'{n}.json'
              for n in ['train3', 'train5', 'pooled3', 'pooled5', 'chronology']]
    return {str(p.relative_to(ROOT)): sha(p) for p in sorted(paths)}


def register():
    path = HERE/'protocol.json'; assert not path.exists()
    assert len(cases()) == 84
    save(path, dict(version='complementary-heads-v1', registered_at=datetime.now(timezone.utc).isoformat(),
        base_main='766d16d3bc74248cb7d8f4c1e7240094ebd5a55e',
        parent='Anchored .89636 is the architecture parent. Scored .89935 anchored80/Cand2_20 is the advancement control.',
        hypothesis='A supervised nonlinear head explicitly discouraged from sharing anchored residuals may learn useful complementary evidence; a regularized readout may use both heads better than a fixed blend.',
        adaptation='Course forbids deep learning. Frozen anchored is internally LR/KNN/LDA, not a neural network. Train classical Nyström RBF logistic head by gradients; no jointly updated anchor, gradient-reversal network, hidden-layer removal, or learned deep representation. Replace the output combination with learned readouts of all ten label-indexed pre-decision logits.',
        features='Only150 supplied reason embeddings. Anchor unchanged including its pseudo-label generation. Kernel scaler and128 Nyström RBF landmarks fit eligible gold+masked inputs, gamma1/150, seed270303. Kernel label loss uses supplied gold only; no imitation of anchor pseudo-labels.',
        head_loss='Sum across5 labels of mean binary cross-entropy +.005/2*sum(nonintercept_weights^2)+1e-6/2*sum(intercept^2) +lambda/2*sum(squared centered Pearson error correlations). Errors use centered-cutoff anchored scores sigmoid(logit(a)-logit(.3))-gold and kernel sigmoid(logit)-gold. Each variance adds1e-8. Lambda0 ordinary control, .1 and .3 diverse variants. No gradient reversal of supervised loss. Diverse fits start from ordinary solution. L-BFGS-B maxiter1000 ftol1e-11 gtol1e-7.',
        head_caveat='Anchored decision-aligned scores are not calibrated probabilities. Penalized objective is nonconvex; success here would concern this fixed adaptation only. Lower error correlation can sacrifice accuracy.',
        candidates=CANDIDATES,
        fixed_blends='80% decision-aligned anchored probabilities +20% kernel probabilities; threshold.5+argmax. This is a registered new score-space blend, not identical to the prior raw-score80/20 Cand2 blend.',
        learned_fusion='Five independent offset logistic readouts: base=logit(anchor)-logit(.3); intercept+all5 anchor centered logits+all5 kernel logits, each clipped[-6,6]/3. Sum BCE +20/2*sum(coefficients^2), including intercept; zero correction recovers anchor. Threshold.5+argmax. Calibration-only control removes kernel features. No fitted threshold or cross-validated regularization search.',
        nested_access='Outer saved84 cases. Three group-stratified inner folds seed270301 create genuine held-out scores for BOTH heads. For each inner kernel training, another three-fold anchor crossfit seed270302 supplies training residuals. Every level refits scaler/PCA/GMM/pseudo-labels/anchor, purges masked copies of held groups and excludes held gold. Deterministically retry up to20 seeds only if a fitting fold lacks a class. Full outer kernel uses first-level crossfitted anchor residuals. Final fusion fits inner scores only.13 fresh anchor fits and12 kernel fits per outer case.',
        schedule='Focused predeclared complete80 grouped cases+validation78+3chronology cases for every arm, including standalone heads and calibration as diagnostics. Supplementary exception is declared regardless of CV success.',
        scoring='Complete OOF sample-F1 per saved repeat,8means/populationSD, paired deltas/wins/ties/losses versus both controls and matched lambda0, macro/per-label F1 precision recall support, Other subgroups, heldout residual correlation and fixes/breaks.',
        advancement='Positive mean and >=5/8 repeat wins in BOTH pools versus scored .89935 control. Also report versus anchored and lambda0. Supplementary regressions visible. Prepare final235 only if qualifies and neither supplementary headline declines>.005; choose maximum minimum pool delta, then sum. No automatic upload or merge.',
        limitations='All labels/splits are reused development evidence. No fresh independent holdout, no inference of Kaggle labels. Standalone supervised head is diagnostic; combined candidate retains anchor unsupervised missing-label generation.',
        access='Strict inductive: query embeddings only at prediction; no actual Kaggle inputs loaded during benchmarking. Original validation preserves no outer group purge, matching historical protocol. Supplied input files hashed for protection.',
        reference='Fresh outer anchor must match retained scores within1e-7 and decisions exactly. Cached strict five-seed Cand2 only scores .89935 benchmark control; never used in training or new candidates.',
        retention='Keep source/frozen settings, every summary and verification. Keep useful matched ordinary/diverse fused predictions and controls; discard unsuccessful remaining caches after verification. No fitted models.',
        versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__),
        sources=sources()))
    print('Registered six combined candidates, three standalone diagnostics and calibration across84cases.', flush=True)


def guard(): assert read(HERE/'protocol.json')['sources'] == sources(), 'Frozen source changed'


def evaluate(case, output):
    started = time.monotonic(); fit, query = case['fit'], case['query']
    a_ref, c_ref = reference(case)
    with threadpool_limits(limits=1):
        scores, audit, a = fit_pair(fit, query[EMB].to_numpy(float))
    np.testing.assert_allclose(a, a_ref, atol=1e-7, rtol=0)
    np.testing.assert_array_equal(select_reasons(a), select_reasons(a_ref))
    scores['parent'] = expit(centered_logit(.8*a_ref+.2*c_ref))
    S = np.stack([scores[name] for name in NAMES])
    P = np.stack([decide(score) for score in S])
    assert np.isfinite(S).all() and (P.sum(2)>0).all()
    dest = Path(output)/'predictions'/(case['key']+'.npz')
    np.savez_compressed(dest, names=np.array(NAMES), ids=query.ID.to_numpy(str),
                        truth=query[LABELS].to_numpy(int), scores=S, predictions=P)
    audit.update(key=case['key'], pool=case['pool'], seed=case['seed'],
        fit_ids_sha256=idhash(fit.ID), query_ids_sha256=idhash(query.ID),
        anchor_max_error=float(np.abs(a-a_ref).max()), sha256=sha(dest), seconds=time.monotonic()-started)
    save(dest.with_suffix('.json'), audit)
    return dict(key=case['key'], seconds=round(audit['seconds'], 1))


def metric_rows(y, p):
    pr, re, f, support = precision_recall_fscore_support(y, p, zero_division=0)
    return dict(sample_f1=float(f1_score(y, p, average='samples')), macro_f1=float(f.mean())), [
        dict(label=name, precision=float(pr[j]), recall=float(re[j]), f1=float(f[j]), support=int(support[j]))
        for j, name in enumerate(LABELS)]


def collect(output):
    guard(); output=Path(output); grouped={}; audits=[]
    for case in cases():
        path=output/'predictions'/(case['key']+'.npz'); z=loadnp(path); audit=read(path.with_suffix('.json'))
        assert audit['sha256']==sha(path); audits.append(audit)
        np.testing.assert_array_equal(z['ids'],case['query'].ID)
        np.testing.assert_array_equal(z['truth'],case['query'][LABELS])
        key=(case['pool'],case['seed']) if case['pool']!='supplementary' else (case['key'],-1)
        grouped.setdefault(key,[]).append(z)
        if case['key'].startswith('through_'):grouped.setdefault(('chronology98',-1),[]).append(z)
    rows=[]; labels=[]; subgroups=[]; diversity=[]
    for (pool,seed),packets in grouped.items():
        y=np.concatenate([z['truth'] for z in packets]); ids=np.concatenate([z['ids'] for z in packets])
        assert len(ids)==len(set(ids))
        if pool in ['train','pooled']:assert len(packets)==5
        all_scores=np.concatenate([z['scores'] for z in packets],axis=1)
        all_predictions=np.concatenate([z['predictions'] for z in packets],axis=1)
        a=all_predictions[NAMES.index('anchored')]; ae=all_scores[NAMES.index('anchored')]-y
        ae-=ae.mean(0)
        for j,name in enumerate(NAMES):
            p=all_predictions[j]; m, ls=metric_rows(y,p)
            base=dict(pool=pool,seed=seed,recipe=name,n=len(y))
            rows.append(dict(**base,**m)); labels.extend(dict(**base,**label) for label in ls)
            for group,mask in [('other_alone',(y[:,4]==1)&(y.sum(1)==1)),
                ('other_with_recruitment',(y[:,4]==1)&(y[:,3]==1)),('not_other',y[:,4]==0)]:
                subgroups.append(dict(**base,subgroup=group,support=int(mask.sum()),other_predicted=int(p[mask,4].sum())))
            e=all_scores[j]-y; e-=e.mean(0)
            corr=(ae*e).mean(0)/np.sqrt((np.square(ae).mean(0)+1e-8)*(np.square(e).mean(0)+1e-8))
            for k,label in enumerate(LABELS):
                diversity.append(dict(**base,label=label,error_correlation=float(corr[k]),
                    fixes=int(((a[:,k]!=y[:,k])&(p[:,k]==y[:,k])).sum()),
                    breaks=int(((a[:,k]==y[:,k])&(p[:,k]!=y[:,k])).sum()),
                    jointly_wrong=int(((a[:,k]!=y[:,k])&(p[:,k]!=y[:,k])).sum())))
    out=output/'summary';out.mkdir(exist_ok=True);d=pd.DataFrame(rows)
    d.to_csv(out/'repeat_scores.csv',index=False)
    pd.DataFrame(labels).to_csv(out/'per_label_scores.csv',index=False)
    pd.DataFrame(subgroups).to_csv(out/'other_subgroups.csv',index=False)
    pd.DataFrame(diversity).to_csv(out/'diversity.csv',index=False)
    summary=d.groupby(['pool','recipe']).agg(mean_sample_f1=('sample_f1','mean'),
        population_sd=('sample_f1',lambda x:x.std(ddof=0)),mean_macro_f1=('macro_f1','mean')).reset_index()
    summary.to_csv(out/'overview.csv',index=False)
    pairs=[]; deltas=[]; qualifies={name:True for name in CANDIDATES}
    comparisons=[(name,control) for name in CANDIDATES for control in ['parent','anchored']]
    comparisons += [(prefix+name,prefix+'ordinary') for prefix in ['blend_','fusion_','head_'] for name in ['diverse10','diverse30']]
    comparisons += [('fusion_'+name,'calibration') for name in STRENGTHS]
    for pool in ['train','pooled']:
        pivot=d[d.pool==pool].pivot(index='seed',columns='recipe',values='sample_f1')
        for name,control in comparisons:
            v=pivot[name]-pivot[control]; wins=int((v>1e-12).sum());ties=int((v.abs()<=1e-12).sum())
            passed=bool(v.mean()>1e-12 and wins>=5)
            if control=='parent': qualifies[name]&=passed
            pairs.append(dict(pool=pool,recipe=name,control=control,mean_delta=float(v.mean()),wins=wins,ties=ties,losses=8-wins-ties,passes=passed))
            deltas.extend(dict(pool=pool,recipe=name,control=control,seed=int(k),delta=float(vv)) for k,vv in v.items())
    pd.DataFrame(pairs).to_csv(out/'paired_summary.csv',index=False)
    pd.DataFrame(deltas).to_csv(out/'paired_repeat_deltas.csv',index=False)
    save(out/'qualification.json',qualifies)
    save(out/'fit_summary.json',dict(cases=len(audits),fresh_anchored_fits=84*13,kernel_fits=84*12,
        fusion_fits=84*4,cached_cand2_control_cases=84,total_fit_seconds=sum(a['seconds'] for a in audits),
        max_anchor_error=max(a['anchor_max_error'] for a in audits)))
    opt=[]
    for audit in audits:
        for stage,entries in [('outer',audit['optimization'])]+[(f"inner{v['fold']}",v['optimization']) for v in audit['inner']]:
            for name,values in entries.items():
                opt.append(dict(case=audit['key'],stage=stage,recipe=name,iterations=values['iterations'],
                    max_gradient=values['max_gradient'],objective=values['objective'],
                    training_mean_squared_error_correlation=values['training_mean_squared_error_correlation']))
    pd.DataFrame(opt).to_csv(out/'optimization.csv',index=False)
    print(summary[summary.pool.isin(['train','pooled','validation','chronology98'])].to_string(index=False),flush=True)
    print('QUALIFIED',qualifies,flush=True)


def run(output,workers):
    guard();output=Path(output);assert not output.exists(),'Choose a fresh output directory.'
    (output/'predictions').mkdir(parents=True)
    with ProcessPoolExecutor(max_workers=workers) as executor:
        futures=[executor.submit(evaluate,case,str(output)) for case in cases()]
        for i,future in enumerate(as_completed(futures),1):print(json.dumps(dict(done=i,total=84,**future.result())),flush=True)
    collect(output)


if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('command',choices=['register','run','collect'])
    parser.add_argument('--output',type=Path,default=HERE/'results');parser.add_argument('--workers',type=int,default=4)
    args=parser.parse_args()
    if args.command=='register':register()
    elif args.command=='run':run(args.output,args.workers)
    else:collect(args.output)

Overwriting _source_export/part1/research/complementary_heads_v1/head_study.py


## part1/research/complementary_heads_v1/retained_evidence.py

In [92]:
%%writefile _source_export/part1/research/complementary_heads_v1/retained_evidence.py
"""Keep useful matched fusion/head comparisons after independent full verification."""
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import head_study as study
from verify_heads import metrics


def verify(directory):
    directory=Path(directory);study.guard()
    manifest=study.read(directory/'retention.json')
    for name,digest in manifest['files'].items():assert study.sha(directory/name)==digest
    z=study.loadnp(directory/'matched_predictions.npz')
    table=pd.read_csv(directory/'summary/repeat_scores.csv')
    labels=pd.read_csv(directory/'summary/per_label_scores.csv')
    grouped={};checked=0
    for case in study.cases():
        keep=z['case']==case['key']
        np.testing.assert_array_equal(z['ids'][keep],case['query'].ID)
        np.testing.assert_array_equal(z['truth'][keep],case['query'][study.LABELS])
        key=(case['pool'],case['seed']) if case['pool']!='supplementary' else (case['key'],-1)
        grouped.setdefault(key,[]).extend(np.flatnonzero(keep).tolist())
        if case['key'].startswith('through_'):grouped.setdefault(('chronology98',-1),[]).extend(np.flatnonzero(keep).tolist())
    for (pool,seed),positions in grouped.items():
        y=z['truth'][positions]
        for j,name in enumerate(z['names']):
            score=z['scores'][j,positions];p=(score>=.5).astype(int)
            p[np.arange(len(p)),score.argmax(1)]=1
            np.testing.assert_array_equal(p,z['predictions'][j,positions])
            s,m,l=metrics(y,p)
            row=table[(table.pool==pool)&(table.seed==seed)&(table.recipe==name)].iloc[0]
            np.testing.assert_allclose([s,m],[row.sample_f1,row.macro_f1],atol=1e-12,rtol=0)
            lab=labels[(labels.pool==pool)&(labels.seed==seed)&(labels.recipe==name)].set_index('label').loc[study.LABELS]
            np.testing.assert_allclose(l,lab[['precision','recall','f1','support']],atol=1e-12,rtol=0)
            checked+=1
    receipt=dict(passed=True,retained_metric_rows_rescored=checked,retained_recipes=z['names'].tolist(),
        prediction_rows=len(z['ids']),outer_cases=84,protected_hashes_unchanged=True)
    study.save(directory/'retained_verification.json',receipt);print(receipt)


def compact(directory, kept=None, rationale=None):
    directory=Path(directory)
    assert study.read(directory/'summary/verification.json')['passed']
    keep=list(kept) if kept is not None else ['parent','anchored','calibration']+['head_'+n for n in study.STRENGTHS]+['fusion_'+n for n in study.STRENGTHS]
    qualifying=study.read(directory/'summary/qualification.json')
    keep += [name for name,yes in qualifying.items() if yes and name not in keep]
    indices=[study.NAMES.index(name) for name in keep]
    packets=[];case_keys=[]
    for case in study.cases():
        z=study.loadnp(directory/'predictions'/(case['key']+'.npz'))
        packets.append(z);case_keys.extend([case['key']]*len(z['ids']))
    np.savez_compressed(directory/'matched_predictions.npz',names=np.array(keep),case=np.array(case_keys),
        ids=np.concatenate([z['ids'] for z in packets]),truth=np.concatenate([z['truth'] for z in packets]),
        scores=np.concatenate([z['scores'][indices] for z in packets],axis=1),
        predictions=np.concatenate([z['predictions'][indices] for z in packets],axis=1))
    paths=[directory/'matched_predictions.npz']+sorted((directory/'summary').glob('*'))
    manifest=dict(kept=keep,discarded=[n for n in study.NAMES if n not in keep],
        rationale=rationale or 'Matched lambda0/.1/.3 heads and learned fusion diagnose diversity versus accuracy; calibration and two established controls support interpretation. Unsuccessful fixed-blend fold predictions and all optimizer/nested fold caches removed after full verification.',
        full_verification='summary/verification.json',
        files={str(p.relative_to(directory)):study.sha(p) for p in paths})
    study.save(directory/'retention.json',manifest)
    verify(directory)
    paths=list((directory/'predictions').iterdir())
    assert len(paths)==168 and all(p.suffix in ['.npz','.json'] for p in paths)
    for path in paths:path.unlink()
    (directory/'predictions').rmdir()


if __name__=='__main__':
    command=sys.argv[1] if len(sys.argv)>1 else 'verify'
    directory=Path(sys.argv[2]) if len(sys.argv)>2 else study.HERE/'results'
    if command=='compact':compact(directory)
    else:verify(directory)

Overwriting _source_export/part1/research/complementary_heads_v1/retained_evidence.py


## part1/research/complementary_heads_v1/verify_heads.py

In [93]:
%%writefile _source_export/part1/research/complementary_heads_v1/verify_heads.py
"""Independent metric arithmetic and reconstruction of every nested boundary."""
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.special import expit, logit
import head_study as study


def metrics(y,p):
    tp=(y*p).sum(0);fp=((1-y)*p).sum(0);fn=(y*(1-p)).sum(0)
    pr=np.divide(tp,tp+fp,out=np.zeros(5),where=tp+fp>0)
    re=np.divide(tp,tp+fn,out=np.zeros(5),where=tp+fn>0)
    f=np.divide(2*tp,2*tp+fp+fn,out=np.zeros(5),where=2*tp+fp+fn>0)
    sample=np.mean(2*(y*p).sum(1)/(y.sum(1)+p.sum(1)))
    return sample,f.mean(),np.c_[pr,re,f,y.sum(0)]


def verify(output):
    study.guard();output=Path(output)
    table=pd.read_csv(output/'summary/repeat_scores.csv')
    label_table=pd.read_csv(output/'summary/per_label_scores.csv')
    pairs=pd.read_csv(output/'summary/paired_summary.csv')
    grouped={};boundaries=0;cases=study.cases()
    for case in cases:
        path=output/'predictions'/(case['key']+'.npz')
        z=study.loadnp(path);audit=study.read(path.with_suffix('.json'))
        assert audit['sha256']==study.sha(path)
        np.testing.assert_array_equal(z['ids'],case['query'].ID)
        np.testing.assert_array_equal(z['truth'],case['query'][study.LABELS])
        np.testing.assert_array_equal(z['names'],study.NAMES)
        assert study.idhash(case['fit'].ID)==audit['fit_ids_sha256']
        assert study.idhash(case['query'].ID)==audit['query_ids_sha256']
        assert set(case['fit'].ID).isdisjoint(case['query'].ID)
        if case['key']!='validation':
            assert not set(study.benchmark.groups(case['fit']))&set(study.benchmark.groups(case['query']))
        for step,(_,_,part,held,seed) in enumerate(study.inner_inputs(case['fit'],270301)):
            ai=audit['inner'][step]
            assert ai['fitting_ids_sha256']==study.idhash(part.ID)
            assert ai['query_ids_sha256']==study.idhash(held.ID) and ai['seed']==seed
            boundaries+=1
            for k,(_,_,sub,sh,ss) in enumerate(study.inner_inputs(part,270302)):
                aj=ai['error_training_subfolds'][k]
                assert aj['fitting_ids_sha256']==study.idhash(sub.ID)
                assert aj['query_ids_sha256']==study.idhash(sh.ID) and aj['seed']==ss
                assert set(sub.ID).isdisjoint(held.ID) and set(sh.ID).isdisjoint(held.ID)
                boundaries+=1
        a,c=study.reference(case)
        expected=expit(logit(np.clip(.8*a+.2*c,1e-6,1-1e-6))-logit(.3))
        np.testing.assert_allclose(z['scores'][0],expected,atol=1e-12,rtol=0)
        np.testing.assert_array_equal(z['predictions'][1],study.select_reasons(a))
        for name in study.STRENGTHS:
            np.testing.assert_allclose(z['scores'][study.NAMES.index('blend_'+name)],
                .8*z['scores'][1]+.2*z['scores'][study.NAMES.index('head_'+name)],atol=1e-12,rtol=0)
        for j in range(len(study.NAMES)):
            p=(z['scores'][j]>=.5).astype(int)
            p[np.arange(len(p)),z['scores'][j].argmax(1)]=1
            np.testing.assert_array_equal(p,z['predictions'][j])
        key=(case['pool'],case['seed']) if case['pool']!='supplementary' else (case['key'],-1)
        grouped.setdefault(key,[]).append(z)
        if case['key'].startswith('through_'):grouped.setdefault(('chronology98',-1),[]).append(z)
    checked=0
    for (pool,seed),packets in grouped.items():
        y=np.concatenate([z['truth'] for z in packets]);ids=np.concatenate([z['ids'] for z in packets])
        assert len(ids)==len(set(ids))
        if pool in ['train','pooled']:assert len(packets)==5 and len(y)==(157 if pool=='train' else 235)
        if pool=='chronology98':assert len(y)==98
        for j,name in enumerate(study.NAMES):
            p=np.concatenate([z['predictions'][j] for z in packets]);s,m,l=metrics(y,p)
            recorded=table[(table.pool==pool)&(table.seed==seed)&(table.recipe==name)].iloc[0]
            np.testing.assert_allclose([s,m],[recorded.sample_f1,recorded.macro_f1],atol=1e-12,rtol=0)
            labels=label_table[(label_table.pool==pool)&(label_table.seed==seed)&(label_table.recipe==name)].set_index('label').loc[study.LABELS]
            np.testing.assert_allclose(l,labels[['precision','recall','f1','support']],atol=1e-12,rtol=0)
            checked+=1
    for _,row in pairs.iterrows():
        pivot=table[table.pool==row.pool].pivot(index='seed',columns='recipe',values='sample_f1')
        delta=pivot[row.recipe]-pivot[row.control]
        np.testing.assert_allclose(delta.mean(),row.mean_delta,atol=1e-12,rtol=0)
        assert int((delta>1e-12).sum())==row.wins
        assert int((delta.abs()<=1e-12).sum())==row.ties
    expected={'train':.935244161358811,'pooled':.9216312056737589,'validation':.8910256410256411,'chronology98':.8877551020408163}
    for pool,value in expected.items():
        np.testing.assert_allclose(table[(table.pool==pool)&(table.recipe=='parent')].sample_f1.mean(),value,atol=1e-12,rtol=0)
    receipt=dict(passed=True,outer_cases=len(cases),nested_boundaries_checked=boundaries,
        independent_metric_rows=checked,independent_label_metric_rows=checked*5,
        all_saved_ids_and_groups_checked=True,protected_source_hashes_unchanged=True,
        all_four_parent_headlines_reproduced=True,protocol_sha256=study.sha(study.HERE/'protocol.json'))
    study.save(output/'summary/verification.json',receipt);print(receipt)


if __name__=='__main__':verify(Path(sys.argv[1]) if len(sys.argv)>1 else study.HERE/'results')

Overwriting _source_export/part1/research/complementary_heads_v1/verify_heads.py


## part1/research/complementary_heads_v2/check_strong.py

In [94]:
%%writefile _source_export/part1/research/complementary_heads_v2/check_strong.py
"""Check v2 gradient, threshold-space conversion, and synthetic optimizer."""
import strong_study as wrapper
import numpy as np
from scipy.special import expit,logit
from scipy.optimize import approx_fprime
from threadpoolctl import threadpool_limits
from complement import loss_gradient,decide,centered_logit
from strong_kernel import StrongKernel


def check():
    rng=np.random.default_rng(1128)
    F=np.c_[np.ones(23),rng.normal(size=(23,4))];Y=rng.integers(0,2,size=(23,5))
    e=rng.normal(size=(23,5));e-=e.mean(0);w=rng.normal(0,.2,size=25)
    errors=[]
    for strength in [0.,.1,.3]:
        _,gradient=loss_gradient(w,F,Y,e,strength,.00005)
        numeric=approx_fprime(w,lambda b:loss_gradient(b,F,Y,e,strength,.00005)[0],1e-7)
        delta=float(np.abs(gradient-numeric).max());assert delta<2e-7;errors.append(delta)
    a=rng.uniform(.001,.999,size=(71,5));b=rng.uniform(.001,.999,size=(71,5))
    centered=expit(logit(b)-logit(.3))
    back=expit(logit(centered)+logit(.3))
    np.testing.assert_allclose(back,b,atol=1e-12,rtol=0)
    np.testing.assert_array_equal(decide(expit(centered_logit(.8*a+.2*back))),wrapper.study.select_reasons(.8*a+.2*b))
    X=rng.normal(size=(180,150));Y=np.full((180,5),-100);Y[:90]=rng.integers(0,2,size=(90,5))
    with threadpool_limits(limits=1):
        model=StrongKernel().fit(X,Y,rng.uniform(.01,.99,size=(90,5)))
        for name,z in model.predict_logits(X[:7]).items():assert z.shape==(7,5) and np.isfinite(z).all()
    result=dict(passed=True,max_gradient_absolute_errors=errors,raw_blend_decoder_equivalence=True,synthetic_optimizer=True)
    wrapper.study.save(wrapper.HERE/'math_checks.json',result);print(result)


if __name__=='__main__':check()

Overwriting _source_export/part1/research/complementary_heads_v2/check_strong.py


## part1/research/complementary_heads_v2/evidence.py

In [95]:
%%writefile _source_export/part1/research/complementary_heads_v2/evidence.py
"""Use the retained-evidence verifier with the v2 model configuration."""
import sys
from pathlib import Path
import strong_study as wrapper
import retained_evidence

if __name__=="__main__":
    command=sys.argv[1] if len(sys.argv)>1 else "verify"
    directory=Path(sys.argv[2]) if len(sys.argv)>2 else wrapper.HERE/"results"
    if command=="compact":retained_evidence.compact(directory,
        kept=['parent','anchored','calibration','head_ordinary','head_diverse30',
              'blend_ordinary','blend_diverse30','fusion_ordinary','fusion_diverse30'],
        rationale='Retain the matched lambda0/.3 heads, fixed blends and learned fusion, plus calibration and established controls. This preserves the small fixed-blend diversity benefit alongside its standalone accuracy cost. Intermediate .1 predictions and all fold caches were independently verified before removal; every summary is kept.')
    else:retained_evidence.verify(directory)

Overwriting _source_export/part1/research/complementary_heads_v2/evidence.py


## part1/research/complementary_heads_v2/strong_kernel.py

In [96]:
%%writefile _source_export/part1/research/complementary_heads_v2/strong_kernel.py
"""Stronger matched kernel heads; same diversity loss as v1, distinct protocol."""
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.kernel_approximation import Nystroem
from sklearn.preprocessing import StandardScaler
from complement import ComplementaryKernel, STRENGTHS, centered_logit, loss_gradient


class StrongKernel(ComplementaryKernel):
    def fit(self, X, Y, anchor_oof):
        gold=(Y>=0).all(1);y=Y[gold]
        assert anchor_oof.shape==y.shape
        self.scaler=StandardScaler().fit(X)
        self.map=Nystroem(kernel='rbf',gamma=1/150,n_components=128,random_state=270303,n_jobs=1)
        transformed=self.map.fit_transform(self.scaler.transform(X))
        features=np.c_[np.ones(len(y)),transformed[gold]]
        error=expit(centered_logit(anchor_oof))-y;error-=error.mean(0)
        initial=np.zeros((features.shape[1],5));initial[0]=logit(np.clip(y.mean(0),.02,.98))
        self.weights={};self.audit={}
        for name,strength in STRENGTHS.items():
            start=initial.ravel() if name=='ordinary' else self.weights['ordinary'].ravel()
            result=minimize(loss_gradient,start,args=(features,y,error,strength,.00005),
                method='L-BFGS-B',jac=True,options={'maxiter':1000,'ftol':1e-11,'gtol':1e-7})
            assert result.success,result.message
            self.weights[name]=result.x.reshape(features.shape[1],5)
            residual=expit(features@self.weights[name])-y;residual-=residual.mean(0)
            correlation=(error*residual).mean(0)/np.sqrt((np.square(error).mean(0)+1e-8)*(np.square(residual).mean(0)+1e-8))
            self.audit[name]=dict(iterations=int(result.nit),objective=float(result.fun),
                max_gradient=float(np.abs(result.jac).max()),training_error_correlation=correlation.tolist(),
                training_mean_squared_error_correlation=float(np.square(correlation).mean()))
        return self

    def predict_logits(self,X):
        # Head cutoff .3 is represented at .5 for the unchanged fusion interface.
        return {name:z-logit(.3) for name,z in super().predict_logits(X).items()}

Overwriting _source_export/part1/research/complementary_heads_v2/strong_kernel.py


## part1/research/complementary_heads_v2/strong_study.py

In [97]:
%%writefile _source_export/part1/research/complementary_heads_v2/strong_study.py
"""Reuse v1's frozen nested runner; version the head and combination changes."""
import os
for variable in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[variable]='1'
import argparse
from datetime import datetime,timezone
from pathlib import Path
import sys
import numpy as np
from scipy.special import expit,logit

HERE=Path(__file__).resolve().parent
V1=HERE.parent/'complementary_heads_v1'
sys.path.insert(0,str(V1))
import head_study as study
from complement import centered_logit
from strong_kernel import StrongKernel

original_fit_pair=study.fit_pair
study.HERE=HERE
study.ComplementaryKernel=StrongKernel


def fit_pair(fit,query_x):
    scores,audit,anchor=original_fit_pair(fit,query_x)
    for name in study.STRENGTHS:
        raw_second=expit(logit(np.clip(scores['head_'+name],1e-12,1-1e-12))+logit(.3))
        scores['blend_'+name]=expit(centered_logit(.8*anchor+.2*raw_second))
    return scores,audit,anchor


def reference(case):
    z=study.loadnp(V1/'parent_inputs.npz');keep=z['case']==case['reference_key']
    np.testing.assert_array_equal(z['ids'][keep],case['query'].ID)
    np.testing.assert_array_equal(z['truth'][keep],case['query'][study.LABELS])
    return z['anchored'][keep],z['cand2'][keep]


def sources():
    old=study.read(V1/'protocol.json')['sources']
    # The original implementation and evidence remain protected, byte for byte.
    result={name:study.sha(study.ROOT/name) for name in old}
    assert result==old
    extra=[HERE/'strong_kernel.py',HERE/'strong_study.py',HERE/'check_strong.py',V1/'protocol.json']
    result.update({str(p.relative_to(study.ROOT)):study.sha(p) for p in extra})
    return result


def register():
    path=HERE/'protocol.json';assert not path.exists()
    protocol=study.read(V1/'protocol.json')
    protocol.update(version='complementary-heads-v2',registered_at=datetime.now(timezone.utc).isoformat(),
        preceding_result='v1 completed all84cases. Ordinary standalone Train.712049/Pooled.719858/Chronology.408163. No combined candidate passed. This adaptive followup strengthens the weak control before using it to judge the diversity hypothesis; it is not independent confirmation.',
        hypothesis='The v1 second head was too weak before adding diversity. Lower kernel weight regularization by100x and align its decision threshold to.3, then repeat matched ordinary and two diversity strengths.',
        head_changes='Identical supervised BCE and error-correlation objective, map/scaler/landmarks and seed. Nonintercept L2 coefficient .00005 instead of .005, intercept remains1e-6. Still lambda0/.1/.3, same optimizer and ordinary warm start. Raw head probabilities decoded at.3+argmax. Corresponding centered logits replace the old zero-centered second-head logits in fusion.',
        head_loss='Same exact v1 loss_gradient with explicit ridge=.00005. Correlation uses sigmoid(centered anchored logits)-y versus raw kernel sigmoid(logits)-y; raw kernel probabilities are used in the training loss, regardless of its .3 final decision cutoff. No gradient reversal of label loss.',
        fixed_blends='Raw .8*anchored_score+.2*kernel_probability, decoded at.3+argmax. Internally mapped monotonically to .5 for the shared evaluator. Unlike v1, this uses the same raw arithmetic score space as the current Cand2 blend.',
        learned_fusion='Unchanged nested cross-label offset logistic, except kernel logits now subtract logit(.3) before clipping/scaling. Calibration-only control must reproduce v1 exactly.',
        followup_scope='One stronger fixed kernel recipe, same three diversity strengths, six combined candidates. No further parameter grid or outcome-selected cutoff. Full84cases again; all negatives retained.',
        reference='Same unchanged anchor and scored .89935 controls. Fresh anchor must match retained outputs within1e-7. Every nested fit is fresh; only current Cand2 comparison is cached.',
        sources=sources())
    study.save(path,protocol);print('Registered stronger head v2; six combined candidates, all84cases.',flush=True)


study.fit_pair=fit_pair
study.reference=reference
study.sources=sources


if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('command',choices=['register','run','collect'])
    parser.add_argument('--output',type=Path,default=HERE/'results');parser.add_argument('--workers',type=int,default=4)
    args=parser.parse_args()
    if args.command=='register':register()
    elif args.command=='run':study.run(args.output,args.workers)
    else:study.collect(args.output)

Overwriting _source_export/part1/research/complementary_heads_v2/strong_study.py


## part1/research/complementary_heads_v2/verify_strong.py

In [98]:
%%writefile _source_export/part1/research/complementary_heads_v2/verify_strong.py
"""Independent metric arithmetic and reconstruction of every nested boundary."""
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.special import expit, logit
import strong_study as wrapper
study=wrapper.study


def metrics(y,p):
    tp=(y*p).sum(0);fp=((1-y)*p).sum(0);fn=(y*(1-p)).sum(0)
    pr=np.divide(tp,tp+fp,out=np.zeros(5),where=tp+fp>0)
    re=np.divide(tp,tp+fn,out=np.zeros(5),where=tp+fn>0)
    f=np.divide(2*tp,2*tp+fp+fn,out=np.zeros(5),where=2*tp+fp+fn>0)
    sample=np.mean(2*(y*p).sum(1)/(y.sum(1)+p.sum(1)))
    return sample,f.mean(),np.c_[pr,re,f,y.sum(0)]


def verify(output):
    study.guard();output=Path(output)
    table=pd.read_csv(output/'summary/repeat_scores.csv')
    label_table=pd.read_csv(output/'summary/per_label_scores.csv')
    pairs=pd.read_csv(output/'summary/paired_summary.csv')
    grouped={};boundaries=0;cases=study.cases()
    earlier=study.loadnp(wrapper.V1/'results/matched_predictions.npz')
    for case in cases:
        path=output/'predictions'/(case['key']+'.npz')
        z=study.loadnp(path);audit=study.read(path.with_suffix('.json'))
        assert audit['sha256']==study.sha(path)
        np.testing.assert_array_equal(z['ids'],case['query'].ID)
        np.testing.assert_array_equal(z['truth'],case['query'][study.LABELS])
        np.testing.assert_array_equal(z['names'],study.NAMES)
        old=earlier['case']==case['key']
        np.testing.assert_array_equal(earlier['ids'][old],z['ids'])
        np.testing.assert_allclose(z['scores'][study.NAMES.index('calibration')],
            earlier['scores'][earlier['names'].tolist().index('calibration'),old],atol=1e-12,rtol=0)
        assert study.idhash(case['fit'].ID)==audit['fit_ids_sha256']
        assert study.idhash(case['query'].ID)==audit['query_ids_sha256']
        assert set(case['fit'].ID).isdisjoint(case['query'].ID)
        if case['key']!='validation':
            assert not set(study.benchmark.groups(case['fit']))&set(study.benchmark.groups(case['query']))
        for step,(_,_,part,held,seed) in enumerate(study.inner_inputs(case['fit'],270301)):
            ai=audit['inner'][step]
            assert ai['fitting_ids_sha256']==study.idhash(part.ID)
            assert ai['query_ids_sha256']==study.idhash(held.ID) and ai['seed']==seed
            boundaries+=1
            for k,(_,_,sub,sh,ss) in enumerate(study.inner_inputs(part,270302)):
                aj=ai['error_training_subfolds'][k]
                assert aj['fitting_ids_sha256']==study.idhash(sub.ID)
                assert aj['query_ids_sha256']==study.idhash(sh.ID) and aj['seed']==ss
                assert set(sub.ID).isdisjoint(held.ID) and set(sh.ID).isdisjoint(held.ID)
                boundaries+=1
        a,c=study.reference(case)
        expected=expit(logit(np.clip(.8*a+.2*c,1e-6,1-1e-6))-logit(.3))
        np.testing.assert_allclose(z['scores'][0],expected,atol=1e-12,rtol=0)
        np.testing.assert_array_equal(z['predictions'][1],study.select_reasons(a))
        for name in study.STRENGTHS:
            np.testing.assert_allclose(z['scores'][study.NAMES.index('blend_'+name)],
                expit(logit(np.clip(.8*a+.2*expit(logit(z['scores'][study.NAMES.index('head_'+name)])+logit(.3)),1e-6,1-1e-6))-logit(.3)),atol=1e-12,rtol=0)
        for j in range(len(study.NAMES)):
            p=(z['scores'][j]>=.5).astype(int)
            p[np.arange(len(p)),z['scores'][j].argmax(1)]=1
            np.testing.assert_array_equal(p,z['predictions'][j])
        key=(case['pool'],case['seed']) if case['pool']!='supplementary' else (case['key'],-1)
        grouped.setdefault(key,[]).append(z)
        if case['key'].startswith('through_'):grouped.setdefault(('chronology98',-1),[]).append(z)
    checked=0
    for (pool,seed),packets in grouped.items():
        y=np.concatenate([z['truth'] for z in packets]);ids=np.concatenate([z['ids'] for z in packets])
        assert len(ids)==len(set(ids))
        if pool in ['train','pooled']:assert len(packets)==5 and len(y)==(157 if pool=='train' else 235)
        if pool=='chronology98':assert len(y)==98
        for j,name in enumerate(study.NAMES):
            p=np.concatenate([z['predictions'][j] for z in packets]);s,m,l=metrics(y,p)
            recorded=table[(table.pool==pool)&(table.seed==seed)&(table.recipe==name)].iloc[0]
            np.testing.assert_allclose([s,m],[recorded.sample_f1,recorded.macro_f1],atol=1e-12,rtol=0)
            labels=label_table[(label_table.pool==pool)&(label_table.seed==seed)&(label_table.recipe==name)].set_index('label').loc[study.LABELS]
            np.testing.assert_allclose(l,labels[['precision','recall','f1','support']],atol=1e-12,rtol=0)
            checked+=1
    for _,row in pairs.iterrows():
        pivot=table[table.pool==row.pool].pivot(index='seed',columns='recipe',values='sample_f1')
        delta=pivot[row.recipe]-pivot[row.control]
        np.testing.assert_allclose(delta.mean(),row.mean_delta,atol=1e-12,rtol=0)
        assert int((delta>1e-12).sum())==row.wins
        assert int((delta.abs()<=1e-12).sum())==row.ties
    expected={'train':.935244161358811,'pooled':.9216312056737589,'validation':.8910256410256411,'chronology98':.8877551020408163}
    for pool,value in expected.items():
        np.testing.assert_allclose(table[(table.pool==pool)&(table.recipe=='parent')].sample_f1.mean(),value,atol=1e-12,rtol=0)
    receipt=dict(passed=True,outer_cases=len(cases),nested_boundaries_checked=boundaries,
        independent_metric_rows=checked,independent_label_metric_rows=checked*5,
        all_saved_ids_and_groups_checked=True,protected_source_hashes_unchanged=True,
        all_four_parent_headlines_reproduced=True,v1_calibration_reproduced=True,
        protocol_sha256=study.sha(study.HERE/'protocol.json'))
    study.save(output/'summary/verification.json',receipt);print(receipt)


if __name__=='__main__':verify(Path(sys.argv[1]) if len(sys.argv)>1 else study.HERE/'results')

Overwriting _source_export/part1/research/complementary_heads_v2/verify_strong.py


## part1/research/other_combiner_v1/run.py

In [99]:
%%writefile _source_export/part1/research/other_combiner_v1/run.py
"""Frozen full comparison of three research directions against the .89935 blend."""
import os
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[name]='1'
import argparse
from concurrent.futures import ProcessPoolExecutor,as_completed
from dataclasses import replace
from datetime import datetime,timezone
import hashlib
import importlib.util
import json
from pathlib import Path
import platform
import sys
import time

HERE=Path(__file__).resolve().parent
P1=HERE.parents[1];ROOT=P1.parent
HELPER=P1/'research/anchored_transductive_v1'
sys.path[:0]=[str(HERE/'source'),str(P1),str(HELPER),str(HERE)]
import numpy as np
import pandas as pd
import sklearn
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import f1_score,precision_recall_fscore_support
from threadpoolctl import threadpool_limits
from anchored_solution import AnchoredReasonClassifier,select_reasons
from p1lib.data import Data,LABELS,EMB
from p1lib import cache
from p1lib.head import apply_head
import strict_adapter as strict
from study_models import (OtherSpecialists,PseudoWeightedAnchored,WeightedReasonLDA,
                          learned_heads,centered_logit,decide,offset_fit)
spec=importlib.util.spec_from_file_location('benchmark_helper',HELPER/'run.py')
benchmark=importlib.util.module_from_spec(spec);spec.loader.exec_module(benchmark)
CANDIDATES=['other_rbf','other_conditional_rbf','pseudo_lda_quarter','pseudo_lda_confidence',
            'combiner_per_label','combiner_other_only','calibration_control']
NAMES=['parent','anchored']+CANDIDATES
INNER_SEED=270201


def read(p):return json.loads(Path(p).read_text())
def save(p,x):Path(p).write_text(json.dumps(x,indent=2,allow_nan=False)+'\n')
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def idhash(v):return hashlib.sha256('\n'.join(v).encode()).hexdigest()
def loadnp(p):
    with np.load(p,allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}


def cases():
    source=read(HERE/'source_cases.json')
    result=benchmark.cases()
    cols=['ID','start_year']+EMB+LABELS
    for c in result:
        ref=next(x for x in source if x['query_ids']==c['query'].ID.tolist()) if c['pool']=='supplementary' else next(
            x for x in source if x['kind']==c['pool'] and x['repeat']==c['seed'] and x['fold']==c['fold'])
        assert ref['fit_ids']==c['fit'].ID.tolist()
        assert ref['query_ids']==c['query'].ID.tolist()
        c['source_key']=ref['key']
        c['fit']=c['fit'][cols].reset_index(drop=True)
        c['query']=c['query'][cols].reset_index(drop=True)
    return result


def sources():
    paths=[HERE/n for n in ['run.py','study_models.py','strict_adapter.py','cand2_manifest.json',
        'parent_inputs.npz','parent_reported_result.json','source_cases.json','source_provenance.json']]
    paths+=list((HERE/'source/p1lib').glob('*.py'))
    paths+=[HELPER/'run.py',HELPER/'models.py',P1/'anchored_solution.py',P1/'TESTING_POLICY.md']
    paths+=list((P1/'data').glob('*.csv'))
    paths+=[P1/'experiments/current_leaders_grouped_v1/snapshots'/f'{name}.json'
            for name in ['train3','train5','pooled3','pooled5','chronology']]
    return {str(p.relative_to(ROOT)):sha(p) for p in sorted(paths)}


def register():
    p=HERE/'protocol.json';assert not p.exists()
    schedule=cases();assert len(schedule)==84
    save(p,dict(version='other-combiner-v1',registered_utc=datetime.now(timezone.utc).isoformat(),
        base_main='766d16d3bc74248cb7d8f4c1e7240094ebd5a55e',
        parent='80% anchored /20% strict five-seed Cand2, user-reported Kaggle .89935; original anchored .89636 secondary control.',
        authorization='User requested all three proposed directions, prioritizing Other and learned combination.',
        observed_parent_diagnostic='On reused pooled OOF predictions: 26 Other-only positives per repeat, mean recall .7259615; nine Recruitment+Other positives, mean recall .0277778. This diagnostic informed the two fixed specialist hypotheses before scoring new recipes.',
        candidates={
            'other_rbf':'Gold-only Other vs rest SVC: standardized embeddings, RBF gamma1/150, C1, balanced classes. Its OOF margin plus intercept corrects parent Other logit only.',
            'other_conditional_rbf':'Two identical SVCs trained separately on fitting gold with/without Recruitment. Standardized inner-OOF margins gated by parent decision-centered Recruitment score; intercept and two margins correct Other only.',
            'pseudo_lda_quarter':'Anchored keeps LR, neighbor voting, cluster assistance and gold labels unchanged. At rounds1/2 use uniform-prior weighted shrinkage LDA, gold positive rows weight1, pseudo positive rows .25. Relearn pseudo labels each round, then blend altered anchored80/Cand2_unchanged20.',
            'pseudo_lda_confidence':'Same weighted LDA but each pseudo positive weight additionally multiplied by (teacher score for that reason / maximum teacher score for row)^2. Reduces uncertain secondary-label influence. Neighbors and LR still use original hard labels.',
            'combiner_per_label':'Separate regularized offset logistic correction per reason. Offset logit(clip(.8a+.2c))-logit(.3); features intercept, clipped centered anchored logit/3, clipped centered Cand2 logit/3. Zero correction exactly preserves parent.',
            'combiner_other_only':'Same fitted Other combiner, preserving the four other scores. Isolates targeted benefit from changes to stronger labels.',
            'calibration_control':'Offset logistic using intercept and clipped centered parent logit/3 for all reasons. Distinguishes recalibration from added component information.'},
        fixed_head='All corrections minimize summed binary log loss +20/2*sum(beta^2), penalizing intercept; no class weighting or threshold tuning. Predict centered score>=.5 plus argmax. Specialist margin scale learned only from inner OOF training rows, SD floor .1, clipping +/-4. Component logit features clip +/-6. No further settings search.',
        inner='Three iterative multilabel-stratified embedding-group folds, seed270201, inside each outer fitting partition. Refit preprocessing, anchored pseudo-labeling, five-seed Cand2 (including its own nested temperature fitting), and SVCs from scratch. Purge masked copies of inner-held groups. No cached outer predictions used to train the heads.',
        cand2='Exact strict adapter/source/config from cde3d500. Seeds0..4; average combo probabilities; convert to label marginals. Original manifest native transductive flags are overridden. Fixed existing eight-combination alphabet, no query label frequency used.',
        schedule='All seven fixed candidates on all 80 saved grouped folds plus validation and three chronological windows. No post-screen candidate selection; full eight-repeat comparison for every arm.',
        supplementary_exception='All candidates receive validation/chronology even if CV fails, to answer the three requested hypotheses.',
        access='Inductive, same saved gold/masked fitting IDs/order and outer purges. Train157 excludes validation-source rows. Query truth is removed from every model context. Original validation preserves historical no-outer-purge behavior. No test inputs loaded during benchmarking.',
        advancement='Positive mean sample-F1 delta and >=5/8 wins in BOTH pools against .89935 parent. Also report against anchored. Keep validation and each chronology window visible. No automatic replacement or upload. If any qualify, rank by minimum of two CV deltas then their sum, inspect supplements and prepare a final235 CSV only for a candidate with no >.005 supplementary regression.',
        scoring='Complete OOF sample F1 per repeat; population SD; paired repeat deltas and wins/ties/losses; macro F1; per-label precision/recall/F1/support. Report Other-only vs Recruitment+Other recall and false positives as diagnostics, not selection thresholds.',
        controls='Fresh anchored84 must match retained scores (atol1e-7) and predictions exactly. First-case Cand2 refit must match retained scores (atol1e-7). Pseudo control subclass equals anchored; weighted LDA at unit weights agrees with ordinary shrinkage LDA. Independent metric arithmetic and ID/group/coverage checks.',
        limitation='All labels and benchmark splits are reused development data. Only nine supplied Recruitment+Other examples. A local gain is not an independent estimate or guaranteed Kaggle gain. No J-weight fitting and no hidden-label inference.',
        retention='Keep source/config, all summaries, failure notes and verification. Keep detailed leader/parent and useful diagnostic predictions; discard bulky unsuccessful inner caches after verification. No fitted models persisted.',
        versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__),
        sources=sources()))
    print('Registered seven fixed candidates and 84 outer cases.',flush=True)


def guard():assert read(HERE/'protocol.json')['sources']==sources(),'Frozen sources changed'


def cand2(fit,query):
    df=pd.concat([fit,query],ignore_index=True).copy()
    df.loc[len(fit):,LABELS]=-1
    Y=df[LABELS].to_numpy(int);Y[Y<0]=-1
    combos=np.array([1,2,4,5,8,10,16,24])
    CM=((combos[:,None]>>np.arange(5))&1).astype(int)
    known=(Y>=0).all(1);y=np.full(len(df),-1)
    bit=(Y[known]*(1<<np.arange(5))).sum(1)
    assert np.isin(bit,combos).all();y[known]=np.searchsorted(combos,bit)
    tr=np.flatnonzero(known);po=np.flatnonzero((Y[:len(fit)]<0).all(1));qu=np.arange(len(fit),len(df))
    D=Data(df=df,emb=df[EMB].to_numpy(float),Y=Y,y=y,combos=combos,CM=CM,
        idx=dict(train=np.arange(len(fit)),lab=tr,unl=po,val=np.array([],int),
                 test=np.array([],int),labval=tr),year=df.start_year.to_numpy(int))
    cfg=read(HERE/'cand2_manifest.json')['config']
    ps=[];priors=[];temps=[];n=0
    for seed in range(5):
        cache.clear();strict.AUDIT.clear()
        ctx=strict.context(D,tr,qu,po,seed=seed)
        assert (ctx.D.Y[qu]<0).all()
        result=strict.run_model(cfg['model'],ctx,qu)
        ps.append(result['P']);priors.append(result['prior']);temps.append(float(result['T']))
        assert all(a['gold_group_overlap']==a['masked_group_overlap']==0 for a in strict.AUDIT)
        n+=len(strict.AUDIT)
    _,Q,_=apply_head(np.mean(ps,axis=0),np.mean(priors,axis=0),CM,cfg['head'])
    cache.clear();strict.AUDIT.clear()
    return Q@CM,dict(temperatures=temps,internal_group_safe_contexts=n)


def inner_inputs(fit):
    gold=fit[fit[LABELS].ge(0).all(axis=1)].reset_index(drop=True)
    gg=benchmark.groups(gold);fg=benchmark.groups(fit)
    _,inv=np.unique(gg,return_inverse=True)
    target=np.zeros((inv.max()+1,5),int);np.maximum.at(target,inv,gold[LABELS].to_numpy(int))
    splitter=MultilabelStratifiedKFold(n_splits=3,shuffle=True,random_state=INNER_SEED)
    for fold,(tr,te) in enumerate(splitter.split(np.zeros((len(target),1)),target)):
        b=np.flatnonzero(np.isin(inv,te));query=gold.iloc[b].reset_index(drop=True)
        part=fit[~np.isin(fg,gg[b])].reset_index(drop=True)
        assert not set(benchmark.groups(part))&set(benchmark.groups(query))
        assert set(part.ID).isdisjoint(query.ID)
        yield fold,b,part,query


def refs(c):
    z=loadnp(HERE/'parent_inputs.npz');keep=z['case']==c['source_key']
    ids=z['ids'][keep];assert ids.tolist()==c['query'].ID.tolist()
    np.testing.assert_array_equal(z['truth'][keep],c['query'][LABELS])
    return z['anchored'][keep],z['cand2'][keep]


def evaluate(c,output):
    start=time.monotonic();fit,query=c['fit'],c['query']
    # Model functions receive query inputs only; actual labels stay with evaluator.
    clean=query.copy();clean.loc[:,LABELS]=-100
    x,y,qx=fit[EMB].to_numpy(float),fit[LABELS].to_numpy(int),clean[EMB].to_numpy(float)
    a_ref,cc=refs(c);audit={}
    with threadpool_limits(limits=1):
        parent=AnchoredReasonClassifier().fit(x,y);a=parent.predict_proba(qx)
        np.testing.assert_allclose(a,a_ref,atol=1e-7,rtol=0)
        np.testing.assert_array_equal(select_reasons(a),select_reasons(a_ref))
        audit['parent_max_error']=float(np.max(np.abs(a-a_ref)))
        if c['key']=='train_20261126_0':
            fresh,ca=cand2(fit,clean)
            np.testing.assert_allclose(fresh,cc,atol=1e-7,rtol=0)
            audit['cand2_control_max_error']=float(np.max(np.abs(fresh-cc)))
            control=PseudoWeightedAnchored('control').fit(x,y).predict_proba(qx)
            np.testing.assert_allclose(control,a,atol=1e-10,rtol=0)
            audit['pseudo_subclass_control_max_error']=float(np.max(np.abs(control-a)))
        scores=dict(parent=expit_safe(.8*a_ref+.2*cc),anchored=expit_safe(a_ref))
        for name in ['pseudo_lda_quarter','pseudo_lda_confidence']:
            m=PseudoWeightedAnchored(name).fit(x,y)
            scores[name]=expit_safe(.8*m.predict_proba(qx)+.2*cc)
            audit[name]=m.round_audit_
        gold=fit[fit[LABELS].ge(0).all(axis=1)].reset_index(drop=True)
        oa=np.zeros((len(gold),5));oc=np.zeros_like(oa);oh=np.zeros((len(gold),3));coverage=np.zeros(len(gold),int)
        inner=[]
        for fold,b,part,held in inner_inputs(fit):
            heldclean=held.copy();heldclean.loc[:,LABELS]=-100
            xx,yy,hx=part[EMB].to_numpy(float),part[LABELS].to_numpy(int),heldclean[EMB].to_numpy(float)
            oa[b]=AnchoredReasonClassifier().fit(xx,yy).predict_proba(hx)
            oc[b],ca=cand2(part,heldclean)
            oh[b]=OtherSpecialists().fit(xx,yy).predict(hx);coverage[b]+=1
            inner.append(dict(fold=fold,train_ids=part.ID.tolist(),query_ids=held.ID.tolist(),
                              group_overlap=0,**ca))
        assert (coverage==1).all()
        qh=OtherSpecialists().fit(x,y).predict(qx)
        heads,coef=learned_heads(oa,oc,oh,a_ref,cc,qh,gold[LABELS].to_numpy(int))
        scores.update(heads)
        audit['inner']=inner;audit['coefficients']=coef
        audit['inner_gold_ids']=gold.ID.tolist()
    S=np.stack([scores[n] for n in NAMES]);P=np.stack([decide(s) for s in S])
    assert np.isfinite(S).all() and (P.sum(2)>0).all()
    np.testing.assert_array_equal(P[0],select_reasons(.8*a_ref+.2*cc))
    for name in ['other_rbf','other_conditional_rbf','combiner_other_only']:
        np.testing.assert_array_equal(S[NAMES.index(name),:,:4],S[0,:,:4])
    packet=dict(ids=query.ID.to_numpy(str),truth=query[LABELS].to_numpy(int),scores=S,predictions=P,
                names=np.array(NAMES),inner_gold_ids=gold.ID.to_numpy(str),inner_anchored=oa,inner_cand2=oc,
                inner_specialists=oh,outer_specialists=qh)
    dest=Path(output)/'predictions'/(c['key']+'.npz');np.savez_compressed(dest,**packet)
    audit.update(key=c['key'],pool=c['pool'],seed=c['seed'],fold=c['fold'],source_key=c['source_key'],
        fitting_ids_sha256=idhash(fit.ID),query_ids_sha256=idhash(query.ID),gold=len(gold),
        masked=len(fit)-len(gold),seconds=time.monotonic()-start,sha256=sha(dest))
    save(dest.with_suffix('.json'),audit)
    return dict(key=c['key'],seconds=round(audit['seconds'],1))


def expit_safe(s):
    from scipy.special import expit
    return expit(centered_logit(s))


def measure(y,p):
    pr,re,f,s=precision_recall_fscore_support(y,p,zero_division=0)
    return dict(sample_f1=float(f1_score(y,p,average='samples')),macro_f1=float(f.mean())),[
        dict(label=label,precision=float(pr[j]),recall=float(re[j]),f1=float(f[j]),support=int(s[j]))
        for j,label in enumerate(LABELS)]


def collect(output,partial=False):
    output=Path(output);packets={};audits=[]
    for c in cases():
        p=output/'predictions'/(c['key']+'.npz')
        if not p.exists():
            assert partial
            continue
        z=loadnp(p);a=read(p.with_suffix('.json'));assert sha(p)==a['sha256']
        np.testing.assert_array_equal(z['ids'],c['query'].ID)
        np.testing.assert_array_equal(z['truth'],c['query'][LABELS])
        for j in range(len(NAMES)):np.testing.assert_array_equal(z['predictions'][j],decide(z['scores'][j]))
        key=(c['pool'],str(c['seed'])) if c['pool']!='supplementary' else (c['key'],'-1')
        packets.setdefault(key,[]).append(z);audits.append(a)
    rows=[];labels=[];diagnostics=[]
    for (pool,seed),parts in packets.items():
        if pool in ['train','pooled'] and len(parts)!=5:continue
        y=np.concatenate([p['truth'] for p in parts]);ids=np.concatenate([p['ids'] for p in parts])
        assert len(set(ids))==len(ids)
        for j,name in enumerate(NAMES):
            pred=np.concatenate([p['predictions'][j] for p in parts]);m,l=measure(y,pred)
            base=dict(pool=pool,seed=int(seed),recipe=name,n=len(y))
            rows.append(dict(**base,**m));labels.extend(dict(**base,**v) for v in l)
            for label,keep in [('other_alone',(y[:,4]==1)&(y.sum(1)==1)),
                               ('other_with_recruitment',(y[:,4]==1)&(y[:,3]==1)),
                               ('not_other',y[:,4]==0)]:
                diagnostics.append(dict(**base,subgroup=label,support=int(keep.sum()),
                    other_predicted=int(pred[keep,4].sum())))
    chrono=[p for (pool,_),parts in packets.items() if pool.startswith('through_') for p in parts]
    if len(chrono)==3:
        y=np.concatenate([p['truth'] for p in chrono]);assert len(y)==98
        for j,name in enumerate(NAMES):
            pred=np.concatenate([p['predictions'][j] for p in chrono]);m,l=measure(y,pred)
            base=dict(pool='chronology98',seed=-1,recipe=name,n=len(y))
            rows.append(dict(**base,**m));labels.extend(dict(**base,**v) for v in l)
    results=output/'summary';results.mkdir(exist_ok=True)
    d=pd.DataFrame(rows);ld=pd.DataFrame(labels)
    d.to_csv(results/'repeat_scores.csv',index=False);ld.to_csv(results/'per_label_scores.csv',index=False)
    pd.DataFrame(diagnostics).to_csv(results/'other_subgroups.csv',index=False)
    s=d.groupby(['pool','recipe']).agg(mean_sample_f1=('sample_f1','mean'),
        population_sd=('sample_f1',lambda x:x.std(ddof=0)),mean_macro_f1=('macro_f1','mean'),repeats=('seed','count')).reset_index()
    s.to_csv(results/'overview.csv',index=False)
    pairs=[];deltas=[];qualification={name:True for name in CANDIDATES}
    for pool in ['train','pooled']:
        pivot=d[d.pool==pool].pivot(index='seed',columns='recipe',values='sample_f1')
        for control in ['parent','anchored']:
            for name in CANDIDATES:
                v=pivot[name]-pivot[control];wins=int((v>1e-12).sum());ties=int((v.abs()<=1e-12).sum())
                passing=bool(len(v)==8 and v.mean()>1e-12 and wins>=5)
                if control=='parent':qualification[name]&=passing
                pairs.append(dict(pool=pool,recipe=name,control=control,mean_delta=float(v.mean()),wins=wins,
                    ties=ties,losses=len(v)-wins-ties,repeats=len(v),passes=passing))
                deltas.extend(dict(pool=pool,recipe=name,control=control,seed=int(k),delta=float(x)) for k,x in v.items())
    pd.DataFrame(pairs).to_csv(results/'paired_summary.csv',index=False)
    pd.DataFrame(deltas).to_csv(results/'paired_repeat_deltas.csv',index=False)
    save(results/'qualification.json',qualification)
    save(results/'fit_summary.json',dict(cases=len(audits),seconds=sum(a['seconds'] for a in audits),
        parent_max_error=max(a['parent_max_error'] for a in audits),
        cand2_control=[a['cand2_control_max_error'] for a in audits if 'cand2_control_max_error' in a],
        nested_contexts=sum(v['internal_group_safe_contexts'] for a in audits for v in a['inner'])))
    print(s[s.pool.isin(['train','pooled','validation','chronology98'])].to_string(index=False),flush=True)
    return d,ld


def run(output,workers):
    guard();output=Path(output);(output/'predictions').mkdir(parents=True,exist_ok=True)
    plan=cases();todo=[c for c in plan if not (output/'predictions'/(c['key']+'.json')).exists()]
    with ProcessPoolExecutor(max_workers=workers) as executor:
        jobs=[executor.submit(evaluate,c,str(output)) for c in todo]
        for i,f in enumerate(as_completed(jobs),1):print(json.dumps(dict(done=i,total=len(todo),**f.result())),flush=True)
    guard();collect(output)


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('command',choices=['register','run','collect'])
    p.add_argument('--output',type=Path,default=HERE/'results');p.add_argument('--workers',type=int,default=4)
    p.add_argument('--partial',action='store_true');args=p.parse_args()
    if args.command=='register':register()
    elif args.command=='run':run(args.output,args.workers)
    else:guard();collect(args.output,args.partial)

Overwriting _source_export/part1/research/other_combiner_v1/run.py


## part1/research/other_combiner_v1/study_models.py

In [100]:
%%writefile _source_export/part1/research/other_combiner_v1/study_models.py
"""Fixed Other specialists, nested residual combiners and weighted anchored LDA."""
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.covariance import ledoit_wolf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.svm import SVC
from anchored_solution import AnchoredReasonClassifier, select_reasons


def centered_logit(scores):
    return logit(np.clip(scores, 1e-6, 1-1e-6))-logit(.3)


def decide(scores):
    pred=(scores>=.5).astype(int)
    pred[np.arange(len(scores)), scores.argmax(1)]=1
    return pred


class OtherSpecialists:
    """Gold-only RBF heads; geometry uses eligible gold and masked inputs."""
    def fit(self, X, Y):
        gold=(Y>=0).all(1)
        self.scaler=StandardScaler().fit(X)
        Z=self.scaler.transform(X[gold]); y=Y[gold]
        self.models=[]
        for keep in [np.ones(len(y), bool), y[:,3]==0, y[:,3]==1]:
            target=y[keep,4]
            if len(np.unique(target))<2:
                self.models.append(float(2*target.mean()-1) if len(target) else -1.)
            else:
                self.models.append(SVC(C=1., kernel='rbf', gamma=1/150,
                    class_weight='balanced', probability=False, tol=1e-5).fit(Z[keep],target))
        return self

    def predict(self, X):
        Z=self.scaler.transform(X)
        return np.column_stack([np.full(len(X),m) if isinstance(m,float)
                                else m.decision_function(Z) for m in self.models])


def offset_fit(F, y, offset, penalty=20.):
    """Sum binary log loss + lambda/2 * squared coefficients, including bias."""
    def objective(b):
        z=offset+F@b
        loss=np.logaddexp(0,z).sum()-y@z+.5*penalty*(b@b)
        grad=F.T@(expit(z)-y)+penalty*b
        return loss,grad
    result=minimize(objective,np.zeros(F.shape[1]),jac=True,method='L-BFGS-B',
                    options=dict(maxiter=1000,ftol=1e-12,gtol=1e-8))
    if not result.success and np.max(np.abs(result.jac))>1e-5:
        raise RuntimeError(result.message)
    return result.x


def learned_heads(a,c,h,qa,qc,qh,y):
    """Train only on inner-OOF inputs. Evaluate on separate outer query scores."""
    b=.8*a+.2*c; qb=.8*qa+.2*qc
    o=centered_logit(b); qo=centered_logit(qb)
    baseline=expit(qo)
    z=np.clip(np.stack([centered_logit(a),centered_logit(c)],axis=2),-6,6)/3
    qz=np.clip(np.stack([centered_logit(qa),centered_logit(qc)],axis=2),-6,6)/3
    scores={}; coefficients={}
    for name in ['calibration_control','combiner_per_label']:
        out=baseline.copy(); bs=[]
        for j in range(5):
            if name=='calibration_control':
                F=np.c_[np.ones(len(y)),np.clip(o[:,j],-6,6)/3]
                Q=np.c_[np.ones(len(qa)),np.clip(qo[:,j],-6,6)/3]
            else:
                F=np.c_[np.ones(len(y)),z[:,j]]
                Q=np.c_[np.ones(len(qa)),qz[:,j]]
            coef=offset_fit(F,y[:,j],o[:,j]);bs.append(coef.tolist())
            out[:,j]=expit(qo[:,j]+Q@coef)
        scores[name]=out;coefficients[name]=bs
    scores['combiner_other_only']=baseline.copy()
    scores['combiner_other_only'][:,4]=scores['combiner_per_label'][:,4]
    for name,idx in [('other_rbf',[0]),('other_conditional_rbf',[1,2])]:
        mean=h[:,idx].mean(0);sd=np.maximum(h[:,idx].std(0),.1)
        F=np.clip((h[:,idx]-mean)/sd,-4,4)
        Q=np.clip((qh[:,idx]-mean)/sd,-4,4)
        if len(idx)==2:
            r=expit(o[:,3]);qr=expit(qo[:,3])
            F*=np.c_[1-r,r];Q*=np.c_[1-qr,qr]
        F=np.c_[np.ones(len(y)),F];Q=np.c_[np.ones(len(qa)),Q]
        coef=offset_fit(F,y[:,4],o[:,4])
        out=baseline.copy();out[:,4]=expit(qo[:,4]+Q@coef)
        scores[name]=out
        coefficients[name]=dict(beta=coef.tolist(),mean=mean.tolist(),std=sd.tolist())
    return scores,coefficients


class WeightedReasonLDA:
    """Uniform-prior weighted class means and standardized shrinkage covariances."""
    def fit(self,X,y,w):
        self.classes_=np.arange(5);means=[];cov=np.zeros((X.shape[1],X.shape[1]))
        for k in self.classes_:
            Xk=X[y==k];wk=w[y==k]
            mu=np.average(Xk,axis=0,weights=wk);means.append(mu)
            R=(Xk-mu)*np.sqrt(wk/wk.mean())[:,None]
            # RMS, not variance around the weighted residual array's unweighted mean.
            sd=np.sqrt(np.mean(R*R,axis=0));sd[sd<1e-12]=1
            C=ledoit_wolf(R/sd,assume_centered=True)[0]
            cov+=.2*(sd[:,None]*C*sd[None,:])
        means=np.array(means)
        self.coef_=np.linalg.lstsq(cov,means.T,rcond=None)[0].T
        self.intercept_=-.5*np.sum(means*self.coef_,axis=1)+np.log(.2)
        return self

    def predict_proba(self,X):
        z=X@self.coef_.T+self.intercept_;z-=z.max(1,keepdims=True)
        p=np.exp(z);return p/p.sum(1,keepdims=True)


class PseudoWeightedAnchored(AnchoredReasonClassifier):
    def __init__(self,mode):self.mode=mode

    def fit(self,X,Y):
        # Original input geometry, class balancing, neighbor bank and LR remain fixed.
        from sklearn.decomposition import PCA
        from sklearn.mixture import GaussianMixture
        X,Y=np.asarray(X,float),np.asarray(Y,int);known=(Y>=0).all(1)
        self.scaler_=StandardScaler().fit(X);Z=self.scaler_.transform(X)
        self.pca_=PCA(30,random_state=0).fit(Z)
        self.gmm_=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(self.pca_.transform(Z))
        assert self.gmm_.converged_
        K,V=Z[known],Z[~known];targets=Y[known]
        Rk=self.gmm_.predict_proba(self.pca_.transform(K));Ru=self.gmm_.predict_proba(self.pca_.transform(V))
        counts=(targets/targets.sum(1,keepdims=True)).sum(0)
        self.class_weights_=counts.mean()/counts
        Za,Ya,Ra=K,targets,Rk;confidence=np.ones_like(Ya,float)
        self.round_audit_=[]
        for step in range(3):
            row,reason=np.nonzero(Ya)
            weights=self.class_weights_[reason]/Ya.sum(1)[row]
            weights*=np.where(row<len(K),1.,.25)
            self.logistic_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(Za[row],reason,sample_weight=weights)
            if step==0 or self.mode=='control':
                self.lda_=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(5)/5).fit(Za[row],reason)
            else:
                lw=np.where(row<len(K),1.,.25)
                if self.mode=='pseudo_lda_confidence':lw*=confidence[row,reason]
                self.lda_=WeightedReasonLDA().fit(Za[row],reason,lw)
            self.neighbor_X_=normalize(Za);self.neighbor_Y_=Ya.copy()
            self.round_audit_.append(dict(step=step,gold=len(K),pseudo=len(Za)-len(K)))
            if step==2:break
            T=Ya/Ya.sum(1,keepdims=True)
            rates=(Ra.T@T+T.mean(0))/(Ra.sum(0)[:,None]+1)
            s=.95*self._scores(V)+.05*(Ru@rates)
            keep=s.max(1)>=.8;generated=select_reasons(s[keep])
            self.generated_mask_=keep;self.generated_labels_=generated
            confidence=np.vstack([np.ones_like(targets,float),
                (s[keep]/s[keep].max(1,keepdims=True))**2])
            Za=np.vstack([K,V[keep]]);Ya=np.vstack([targets,generated]);Ra=np.vstack([Rk,Ru[keep]])
        return self

Overwriting _source_export/part1/research/other_combiner_v1/study_models.py


## part1/research/other_combiner_v2/diagnose.py

In [101]:
%%writefile _source_export/part1/research/other_combiner_v2/diagnose.py
"""Explanatory discrimination/error checks; never selects thresholds or fits models."""
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score,average_precision_score
sys.path.insert(0,str(Path(__file__).resolve().parent))
import run as study


def main():
    output=study.HERE/'results';rows=[];changes=[];groups={}
    for case in study.cases():
        if case['pool']=='supplementary':continue
        path=output/'predictions'/(case['key']+'.npz')
        if not path.exists():continue
        z=study.loadnp(path);a,c=study.refs(case)
        z['native_parent']=.8*a+.2*c
        groups.setdefault((case['pool'],case['seed']),[]).append(z)
    for (pool,seed),zs in groups.items():
        if len(zs)!=5:continue
        y=np.concatenate([z['truth'] for z in zs]);parent=np.concatenate([z['predictions'][0] for z in zs])
        values=dict(parent=np.concatenate([z['native_parent'][:,4] for z in zs]),
                    global_svc=np.concatenate([z['outer_specialists'][:,0] for z in zs]),
                    conditional_svc=np.concatenate([z['outer_specialists'][:,2] for z in zs]))
        for subgroup,keep in [('all',np.ones(len(y),bool)),('gold_recruitment',y[:,3]==1)]:
            for name,s in values.items():
                rows.append(dict(pool=pool,seed=seed,subgroup=subgroup,score=name,n=int(keep.sum()),
                    positives=int(y[keep,4].sum()),auc=float(roc_auc_score(y[keep,4],s[keep])),
                    average_precision=float(average_precision_score(y[keep,4],s[keep]))))
        base_f1=2*(y*parent).sum(1)/(y.sum(1)+parent.sum(1))
        for j,name in enumerate(study.NAMES):
            p=np.concatenate([z['predictions'][j] for z in zs]);new_f1=2*(y*p).sum(1)/(y.sum(1)+p.sum(1))
            changes.append(dict(pool=pool,seed=seed,recipe=name,
                improved_rows=int((new_f1>base_f1+1e-12).sum()),worsened_rows=int((new_f1<base_f1-1e-12).sum()),
                changed_label_sets=int((p!=parent).any(1).sum()),
                other_misses_fixed=int(((y[:,4]==1)&(parent[:,4]==0)&(p[:,4]==1)).sum()),
                other_false_positives_added=int(((y[:,4]==0)&(parent[:,4]==0)&(p[:,4]==1)).sum()),
                recruitment_other_recovered=int(((y[:,4]==1)&(y[:,3]==1)&(parent[:,4]==0)&(p[:,4]==1)).sum())))
    pd.DataFrame(rows).to_csv(output/'summary/discrimination_diagnostic.csv',index=False)
    pd.DataFrame(changes).to_csv(output/'summary/change_diagnostic.csv',index=False)
    print(pd.DataFrame(rows).groupby(['pool','subgroup','score'])[['auc','average_precision']].mean().round(5).to_string())


if __name__=='__main__':main()

Overwriting _source_export/part1/research/other_combiner_v2/diagnose.py


## part1/research/other_combiner_v2/retain.py

In [102]:
%%writefile _source_export/part1/research/other_combiner_v2/retain.py
"""Compact checked results; discard unsuccessful inner caches after rescoring."""
from pathlib import Path
import importlib.util
import shutil
import numpy as np
import run as study


def main():
    here=study.HERE;output=here/'results';summary=output/'summary'
    receipt=study.read(summary/'verification.json');assert receipt['passed'] and receipt['outer_cases']==84
    keep=['parent','pseudo_lda_quarter','combiner_other_only'];indices=[study.NAMES.index(n) for n in keep]
    packets=[];meta=[];coefficients={};equivalent=True
    for c in study.cases():
        p=output/'predictions'/(c['key']+'.npz');z=study.loadnp(p);a=study.read(p.with_suffix('.json'))
        assert study.sha(p)==a['sha256']
        equivalent&=np.array_equal(z['predictions'][study.NAMES.index('pseudo_lda_quarter')],
                                   z['predictions'][study.NAMES.index('pseudo_lda_confidence')])
        packets.append(z);pool=c['pool'] if c['pool']!='supplementary' else c['key']
        meta.append(dict(pool=np.full(len(z['ids']),pool),seed=np.full(len(z['ids']),c['seed']),case=np.full(len(z['ids']),c['key'])))
        for name,v in a['coefficients'].items():
            b=v['beta'] if isinstance(v,dict) else v
            coefficients.setdefault((pool,name),[]).append(np.asarray(b))
    arrays={k:np.concatenate([z[k] for z in packets]) for k in ['ids','truth']}
    arrays.update({k:np.concatenate([m[k] for m in meta]) for k in ['pool','seed','case']})
    arrays.update(names=np.array(keep),scores=np.concatenate([z['scores'][indices] for z in packets],axis=1),
                  predictions=np.concatenate([z['predictions'][indices] for z in packets],axis=1))
    np.savez_compressed(output/'retained_comparison.npz',**arrays)
    study.save(summary/'coefficient_summary.json',[dict(pool=k[0],recipe=k[1],outer_cases=len(v),
        mean=np.mean(v,axis=0).tolist(),population_sd=np.std(v,axis=0).tolist()) for k,v in coefficients.items()])
    study.save(summary/'pseudo_variants_equivalence.json',dict(all_84_hard_prediction_arrays_identical=bool(equivalent),
        note='Raw scores differ; no equality of fitted models is claimed.'))
    assert equivalent
    # Per-case trained head coefficients and inner datasets are unsuccessful fit caches.
    (summary/'coefficients.json').unlink()
    files=list(summary.glob('*'))+[output/'retained_comparison.npz']
    manifest=dict(kept_comparisons=keep,
        reasons={'parent':'Scored .89935 reference.',
                 'pseudo_lda_quarter':'Useful matched diagnosis: recovers more Recruitment+Other but regresses chronology.',
                 'combiner_other_only':'Useful matched diagnosis: tied CV averages with a small chronology improvement.'},
        full_checks_before_cleanup=receipt,
        discard='84 per-case npz and 84 fit-audit JSON files, containing unsuccessful inner predictions/features and per-case fitted coefficients; per-case coefficient aggregate removed. Source, compact metrics for all seven variants, negative results and reproduction code retained.',
        retained_sha256={str(p.relative_to(here)):study.sha(p) for p in files})
    study.save(here/'retention.json',manifest)
    spec=importlib.util.spec_from_file_location('this_study_retained_check',here/'verify_retained.py')
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    module.main()
    shutil.rmtree(output/'predictions')
    print('Removed checked unsuccessful fold caches; retained three useful matched comparisons.')


if __name__=='__main__':main()

Overwriting _source_export/part1/research/other_combiner_v2/retain.py


## part1/research/other_combiner_v2/run.py

In [103]:
%%writefile _source_export/part1/research/other_combiner_v2/run.py
"""Frozen full comparison of three research directions against the .89935 blend."""
import os
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[name]='1'
import argparse
from concurrent.futures import ProcessPoolExecutor,as_completed
from dataclasses import replace
from datetime import datetime,timezone
import hashlib
import importlib.util
import json
from pathlib import Path
import platform
import sys
import time

HERE=Path(__file__).resolve().parent
P1=HERE.parents[1];ROOT=P1.parent
HELPER=P1/'research/anchored_transductive_v1'
sys.path[:0]=[str(HERE/'source'),str(P1),str(HELPER),str(HERE)]
import numpy as np
import pandas as pd
import sklearn
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import f1_score,precision_recall_fscore_support
from threadpoolctl import threadpool_limits
from anchored_solution import AnchoredReasonClassifier,select_reasons
from p1lib.data import Data,LABELS,EMB
from p1lib import cache
from p1lib.head import apply_head
import strict_adapter as strict
from study_models import (OtherSpecialists,PseudoWeightedAnchored,WeightedReasonLDA,
                          learned_heads,centered_logit,decide,offset_fit)
spec=importlib.util.spec_from_file_location('benchmark_helper',HELPER/'run.py')
benchmark=importlib.util.module_from_spec(spec);spec.loader.exec_module(benchmark)
CANDIDATES=['other_rbf','other_conditional_rbf','pseudo_lda_quarter','pseudo_lda_confidence',
            'combiner_per_label','combiner_other_only','calibration_control']
NAMES=['parent','anchored']+CANDIDATES
INNER_SEED=270201


def read(p):return json.loads(Path(p).read_text())
def save(p,x):Path(p).write_text(json.dumps(x,indent=2,allow_nan=False)+'\n')
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def idhash(v):return hashlib.sha256('\n'.join(v).encode()).hexdigest()
def loadnp(p):
    with np.load(p,allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}


def cases():
    source=read(HERE/'source_cases.json')
    result=benchmark.cases()
    cols=['ID','start_year']+EMB+LABELS
    for c in result:
        ref=next(x for x in source if x['query_ids']==c['query'].ID.tolist()) if c['pool']=='supplementary' else next(
            x for x in source if x['kind']==c['pool'] and x['repeat']==c['seed'] and x['fold']==c['fold'])
        assert ref['fit_ids']==c['fit'].ID.tolist()
        assert ref['query_ids']==c['query'].ID.tolist()
        c['source_key']=ref['key']
        c['fit']=c['fit'][cols].reset_index(drop=True)
        c['query']=c['query'][cols].reset_index(drop=True)
    return result


def sources():
    paths=[HERE/n for n in ['run.py','study_models.py','strict_adapter.py','cand2_manifest.json',
        'parent_inputs.npz','parent_reported_result.json','source_cases.json','source_provenance.json']]
    paths+=list((HERE/'source/p1lib').glob('*.py'))
    paths+=[HELPER/'run.py',HELPER/'models.py',P1/'anchored_solution.py',P1/'TESTING_POLICY.md']
    paths+=list((P1/'data').glob('*.csv'))
    paths+=[P1/'experiments/current_leaders_grouped_v1/snapshots'/f'{name}.json'
            for name in ['train3','train5','pooled3','pooled5','chronology']]
    return {str(p.relative_to(ROOT)):sha(p) for p in sorted(paths)}


def register():
    p=HERE/'protocol.json';assert not p.exists()
    schedule=cases();assert len(schedule)==84
    save(p,dict(version='other-combiner-v2',registered_utc=datetime.now(timezone.utc).isoformat(),
        base_main='766d16d3bc74248cb7d8f4c1e7240094ebd5a55e',
        parent='80% anchored /20% strict five-seed Cand2, user-reported Kaggle .89935; original anchored .89636 secondary control.',
        authorization='User requested all three proposed directions, prioritizing Other and learned combination.',
        observed_parent_diagnostic='On reused pooled OOF predictions: 26 Other-only positives per repeat, mean recall .7259615; nine Recruitment+Other positives, mean recall .0277778. This diagnostic informed the two fixed specialist hypotheses before scoring new recipes.',
        candidates={
            'other_rbf':'Gold-only Other vs rest SVC: standardized embeddings, RBF gamma1/150, C1, balanced classes. Its OOF margin plus intercept corrects parent Other logit only.',
            'other_conditional_rbf':'Two identical SVCs trained separately on fitting gold with/without Recruitment. Standardized inner-OOF margins gated by parent decision-centered Recruitment score; intercept and two margins correct Other only.',
            'pseudo_lda_quarter':'Anchored keeps LR, neighbor voting, cluster assistance and gold labels unchanged. At rounds1/2 use uniform-prior weighted shrinkage LDA, gold positive rows weight1, pseudo positive rows .25. Relearn pseudo labels each round, then blend altered anchored80/Cand2_unchanged20.',
            'pseudo_lda_confidence':'Same weighted LDA but each pseudo positive weight additionally multiplied by (teacher score for that reason / maximum teacher score for row)^2. Reduces uncertain secondary-label influence. Neighbors and LR still use original hard labels.',
            'combiner_per_label':'Separate regularized offset logistic correction per reason. Offset logit(clip(.8a+.2c))-logit(.3); features intercept, clipped centered anchored logit/3, clipped centered Cand2 logit/3. Zero correction exactly preserves parent.',
            'combiner_other_only':'Same fitted Other combiner, preserving the four other scores. Isolates targeted benefit from changes to stronger labels.',
            'calibration_control':'Offset logistic using intercept and clipped centered parent logit/3 for all reasons. Distinguishes recalibration from added component information.'},
        fixed_head='All corrections minimize summed binary log loss +20/2*sum(beta^2), penalizing intercept; no class weighting or threshold tuning. Predict centered score>=.5 plus argmax. Specialist margin scale learned only from inner OOF training rows, SD floor .1, clipping +/-4. Component logit features clip +/-6. No further settings search.',
        inner='Three iterative multilabel-stratified embedding-group folds, seed270201, inside each outer fitting partition. Refit preprocessing, anchored pseudo-labeling, five-seed Cand2 (including its own nested temperature fitting), and SVCs from scratch. Purge masked copies of inner-held groups. No cached outer predictions used to train the heads.',
        cand2='Exact strict adapter/source/config from cde3d500. Seeds0..4; average combo probabilities; convert to label marginals. Original manifest native transductive flags are overridden. Fixed existing eight-combination alphabet, no query label frequency used.',
        schedule='All seven fixed candidates on all 80 saved grouped folds plus validation and three chronological windows. No post-screen candidate selection; full eight-repeat comparison for every arm.',
        supplementary_exception='All candidates receive validation/chronology even if CV fails, to answer the three requested hypotheses.',
        access='Inductive, same saved gold/masked fitting IDs/order and outer purges. Train157 excludes validation-source rows. Query truth is removed from every model context. Original validation preserves historical no-outer-purge behavior. No test inputs loaded during benchmarking.',
        advancement='Positive mean sample-F1 delta and >=5/8 wins in BOTH pools against .89935 parent. Also report against anchored. Keep validation and each chronology window visible. No automatic replacement or upload. If any qualify, rank by minimum of two CV deltas then their sum, inspect supplements and prepare a final235 CSV only for a candidate with no >.005 supplementary regression.',
        scoring='Complete OOF sample F1 per repeat; population SD; paired repeat deltas and wins/ties/losses; macro F1; per-label precision/recall/F1/support. Report Other-only vs Recruitment+Other recall and false positives as diagnostics, not selection thresholds.',
        controls='Fresh anchored84 must match retained scores (atol1e-7) and predictions exactly. First-case Cand2 refit must match retained scores (atol1e-7). Pseudo control subclass equals anchored; weighted LDA at unit weights agrees with ordinary shrinkage LDA. Independent metric arithmetic and ID/group/coverage checks.',
        limitation='All labels and benchmark splits are reused development data. Only nine supplied Recruitment+Other examples. A local gain is not an independent estimate or guaranteed Kaggle gain. No J-weight fitting and no hidden-label inference.',
        retention='Keep source/config, all summaries, failure notes and verification. Keep detailed leader/parent and useful diagnostic predictions; discard bulky unsuccessful inner caches after verification. No fitted models persisted.',
        versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__),
        sources=sources()))
    print('Registered seven fixed candidates and 84 outer cases.',flush=True)


def guard():assert read(HERE/'protocol.json')['sources']==sources(),'Frozen sources changed'


def cand2(fit,query):
    df=pd.concat([fit,query],ignore_index=True).copy()
    df.loc[len(fit):,LABELS]=-1
    Y=df[LABELS].to_numpy(int,copy=True);Y[Y<0]=-1
    combos=np.array([1,2,4,5,8,10,16,24])
    CM=((combos[:,None]>>np.arange(5))&1).astype(int)
    known=(Y>=0).all(1);y=np.full(len(df),-1)
    bit=(Y[known]*(1<<np.arange(5))).sum(1)
    assert np.isin(bit,combos).all();y[known]=np.searchsorted(combos,bit)
    tr=np.flatnonzero(known);po=np.flatnonzero((Y[:len(fit)]<0).all(1));qu=np.arange(len(fit),len(df))
    D=Data(df=df,emb=df[EMB].to_numpy(float),Y=Y,y=y,combos=combos,CM=CM,
        idx=dict(train=np.arange(len(fit)),lab=tr,unl=po,val=np.array([],int),
                 test=np.array([],int),labval=tr),year=df.start_year.to_numpy(int))
    cfg=read(HERE/'cand2_manifest.json')['config']
    ps=[];priors=[];temps=[];n=0
    for seed in range(5):
        cache.clear();strict.AUDIT.clear()
        ctx=strict.context(D,tr,qu,po,seed=seed)
        assert (ctx.D.Y[qu]<0).all()
        result=strict.run_model(cfg['model'],ctx,qu)
        ps.append(result['P']);priors.append(result['prior']);temps.append(float(result['T']))
        assert all(a['gold_group_overlap']==a['masked_group_overlap']==0 for a in strict.AUDIT)
        n+=len(strict.AUDIT)
    _,Q,_=apply_head(np.mean(ps,axis=0),np.mean(priors,axis=0),CM,cfg['head'])
    cache.clear();strict.AUDIT.clear()
    return Q@CM,dict(temperatures=temps,internal_group_safe_contexts=n)


def inner_inputs(fit):
    gold=fit[fit[LABELS].ge(0).all(axis=1)].reset_index(drop=True)
    gg=benchmark.groups(gold);fg=benchmark.groups(fit)
    _,inv=np.unique(gg,return_inverse=True)
    target=np.zeros((inv.max()+1,5),int);np.maximum.at(target,inv,gold[LABELS].to_numpy(int))
    splitter=MultilabelStratifiedKFold(n_splits=3,shuffle=True,random_state=INNER_SEED)
    for fold,(tr,te) in enumerate(splitter.split(np.zeros((len(target),1)),target)):
        b=np.flatnonzero(np.isin(inv,te));query=gold.iloc[b].reset_index(drop=True)
        part=fit[~np.isin(fg,gg[b])].reset_index(drop=True)
        assert not set(benchmark.groups(part))&set(benchmark.groups(query))
        assert set(part.ID).isdisjoint(query.ID)
        yield fold,b,part,query


def refs(c):
    z=loadnp(HERE/'parent_inputs.npz');keep=z['case']==c['source_key']
    ids=z['ids'][keep];assert ids.tolist()==c['query'].ID.tolist()
    np.testing.assert_array_equal(z['truth'][keep],c['query'][LABELS])
    return z['anchored'][keep],z['cand2'][keep]


def evaluate(c,output):
    start=time.monotonic();fit,query=c['fit'],c['query']
    # Model functions receive query inputs only; actual labels stay with evaluator.
    clean=query.copy();clean.loc[:,LABELS]=-100
    x,y,qx=fit[EMB].to_numpy(float),fit[LABELS].to_numpy(int),clean[EMB].to_numpy(float)
    a_ref,cc=refs(c);audit={}
    with threadpool_limits(limits=1):
        parent=AnchoredReasonClassifier().fit(x,y);a=parent.predict_proba(qx)
        np.testing.assert_allclose(a,a_ref,atol=1e-7,rtol=0)
        np.testing.assert_array_equal(select_reasons(a),select_reasons(a_ref))
        audit['parent_max_error']=float(np.max(np.abs(a-a_ref)))
        if c['key']=='train_20261126_0':
            fresh,ca=cand2(fit,clean)
            np.testing.assert_allclose(fresh,cc,atol=1e-7,rtol=0)
            audit['cand2_control_max_error']=float(np.max(np.abs(fresh-cc)))
            control=PseudoWeightedAnchored('control').fit(x,y).predict_proba(qx)
            np.testing.assert_allclose(control,a,atol=1e-10,rtol=0)
            audit['pseudo_subclass_control_max_error']=float(np.max(np.abs(control-a)))
        scores=dict(parent=expit_safe(.8*a_ref+.2*cc),anchored=expit_safe(a_ref))
        for name in ['pseudo_lda_quarter','pseudo_lda_confidence']:
            m=PseudoWeightedAnchored(name).fit(x,y)
            scores[name]=expit_safe(.8*m.predict_proba(qx)+.2*cc)
            audit[name]=m.round_audit_
        gold=fit[fit[LABELS].ge(0).all(axis=1)].reset_index(drop=True)
        oa=np.zeros((len(gold),5));oc=np.zeros_like(oa);oh=np.zeros((len(gold),3));coverage=np.zeros(len(gold),int)
        inner=[]
        for fold,b,part,held in inner_inputs(fit):
            heldclean=held.copy();heldclean.loc[:,LABELS]=-100
            xx,yy,hx=part[EMB].to_numpy(float),part[LABELS].to_numpy(int),heldclean[EMB].to_numpy(float)
            oa[b]=AnchoredReasonClassifier().fit(xx,yy).predict_proba(hx)
            oc[b],ca=cand2(part,heldclean)
            oh[b]=OtherSpecialists().fit(xx,yy).predict(hx);coverage[b]+=1
            inner.append(dict(fold=fold,train_ids=part.ID.tolist(),query_ids=held.ID.tolist(),
                              group_overlap=0,**ca))
        assert (coverage==1).all()
        qh=OtherSpecialists().fit(x,y).predict(qx)
        heads,coef=learned_heads(oa,oc,oh,a_ref,cc,qh,gold[LABELS].to_numpy(int))
        scores.update(heads)
        audit['inner']=inner;audit['coefficients']=coef
        audit['inner_gold_ids']=gold.ID.tolist()
    S=np.stack([scores[n] for n in NAMES]);P=np.stack([decide(s) for s in S])
    assert np.isfinite(S).all() and (P.sum(2)>0).all()
    np.testing.assert_array_equal(P[0],select_reasons(.8*a_ref+.2*cc))
    for name in ['other_rbf','other_conditional_rbf','combiner_other_only']:
        np.testing.assert_array_equal(S[NAMES.index(name),:,:4],S[0,:,:4])
    packet=dict(ids=query.ID.to_numpy(str),truth=query[LABELS].to_numpy(int),scores=S,predictions=P,
                names=np.array(NAMES),inner_gold_ids=gold.ID.to_numpy(str),inner_anchored=oa,inner_cand2=oc,
                inner_specialists=oh,outer_specialists=qh)
    dest=Path(output)/'predictions'/(c['key']+'.npz');np.savez_compressed(dest,**packet)
    audit.update(key=c['key'],pool=c['pool'],seed=c['seed'],fold=c['fold'],source_key=c['source_key'],
        fitting_ids_sha256=idhash(fit.ID),query_ids_sha256=idhash(query.ID),gold=len(gold),
        masked=len(fit)-len(gold),seconds=time.monotonic()-start,sha256=sha(dest))
    save(dest.with_suffix('.json'),audit)
    return dict(key=c['key'],seconds=round(audit['seconds'],1))


def expit_safe(s):
    from scipy.special import expit
    return expit(centered_logit(s))


def measure(y,p):
    pr,re,f,s=precision_recall_fscore_support(y,p,zero_division=0)
    return dict(sample_f1=float(f1_score(y,p,average='samples')),macro_f1=float(f.mean())),[
        dict(label=label,precision=float(pr[j]),recall=float(re[j]),f1=float(f[j]),support=int(s[j]))
        for j,label in enumerate(LABELS)]


def collect(output,partial=False):
    output=Path(output);packets={};audits=[]
    for c in cases():
        p=output/'predictions'/(c['key']+'.npz')
        if not p.exists():
            assert partial
            continue
        z=loadnp(p);a=read(p.with_suffix('.json'));assert sha(p)==a['sha256']
        np.testing.assert_array_equal(z['ids'],c['query'].ID)
        np.testing.assert_array_equal(z['truth'],c['query'][LABELS])
        for j in range(len(NAMES)):np.testing.assert_array_equal(z['predictions'][j],decide(z['scores'][j]))
        key=(c['pool'],str(c['seed'])) if c['pool']!='supplementary' else (c['key'],'-1')
        packets.setdefault(key,[]).append(z);audits.append(a)
    rows=[];labels=[];diagnostics=[]
    for (pool,seed),parts in packets.items():
        if pool in ['train','pooled'] and len(parts)!=5:continue
        y=np.concatenate([p['truth'] for p in parts]);ids=np.concatenate([p['ids'] for p in parts])
        assert len(set(ids))==len(ids)
        for j,name in enumerate(NAMES):
            pred=np.concatenate([p['predictions'][j] for p in parts]);m,l=measure(y,pred)
            base=dict(pool=pool,seed=int(seed),recipe=name,n=len(y))
            rows.append(dict(**base,**m));labels.extend(dict(**base,**v) for v in l)
            for label,keep in [('other_alone',(y[:,4]==1)&(y.sum(1)==1)),
                               ('other_with_recruitment',(y[:,4]==1)&(y[:,3]==1)),
                               ('not_other',y[:,4]==0)]:
                diagnostics.append(dict(**base,subgroup=label,support=int(keep.sum()),
                    other_predicted=int(pred[keep,4].sum())))
    chrono=[p for (pool,_),parts in packets.items() if pool.startswith('through_') for p in parts]
    if len(chrono)==3:
        y=np.concatenate([p['truth'] for p in chrono]);assert len(y)==98
        for j,name in enumerate(NAMES):
            pred=np.concatenate([p['predictions'][j] for p in chrono]);m,l=measure(y,pred)
            base=dict(pool='chronology98',seed=-1,recipe=name,n=len(y))
            rows.append(dict(**base,**m));labels.extend(dict(**base,**v) for v in l)
    results=output/'summary';results.mkdir(exist_ok=True)
    d=pd.DataFrame(rows);ld=pd.DataFrame(labels)
    d.to_csv(results/'repeat_scores.csv',index=False);ld.to_csv(results/'per_label_scores.csv',index=False)
    pd.DataFrame(diagnostics).to_csv(results/'other_subgroups.csv',index=False)
    s=d.groupby(['pool','recipe']).agg(mean_sample_f1=('sample_f1','mean'),
        population_sd=('sample_f1',lambda x:x.std(ddof=0)),mean_macro_f1=('macro_f1','mean'),repeats=('seed','count')).reset_index()
    s.to_csv(results/'overview.csv',index=False)
    pairs=[];deltas=[];qualification={name:True for name in CANDIDATES}
    for pool in ['train','pooled']:
        pivot=d[d.pool==pool].pivot(index='seed',columns='recipe',values='sample_f1')
        for control in ['parent','anchored']:
            for name in CANDIDATES:
                v=pivot[name]-pivot[control];wins=int((v>1e-12).sum());ties=int((v.abs()<=1e-12).sum())
                passing=bool(len(v)==8 and v.mean()>1e-12 and wins>=5)
                if control=='parent':qualification[name]&=passing
                pairs.append(dict(pool=pool,recipe=name,control=control,mean_delta=float(v.mean()),wins=wins,
                    ties=ties,losses=len(v)-wins-ties,repeats=len(v),passes=passing))
                deltas.extend(dict(pool=pool,recipe=name,control=control,seed=int(k),delta=float(x)) for k,x in v.items())
    pd.DataFrame(pairs).to_csv(results/'paired_summary.csv',index=False)
    pd.DataFrame(deltas).to_csv(results/'paired_repeat_deltas.csv',index=False)
    save(results/'qualification.json',qualification)
    save(results/'fit_summary.json',dict(cases=len(audits),seconds=sum(a['seconds'] for a in audits),
        parent_max_error=max(a['parent_max_error'] for a in audits),
        cand2_control=[a['cand2_control_max_error'] for a in audits if 'cand2_control_max_error' in a],
        nested_contexts=sum(v['internal_group_safe_contexts'] for a in audits for v in a['inner'])))
    print(s[s.pool.isin(['train','pooled','validation','chronology98'])].to_string(index=False),flush=True)
    return d,ld


def run(output,workers):
    guard();output=Path(output);(output/'predictions').mkdir(parents=True,exist_ok=True)
    plan=cases();todo=[c for c in plan if not (output/'predictions'/(c['key']+'.json')).exists()]
    with ProcessPoolExecutor(max_workers=workers) as executor:
        jobs=[executor.submit(evaluate,c,str(output)) for c in todo]
        for i,f in enumerate(as_completed(jobs),1):print(json.dumps(dict(done=i,total=len(todo),**f.result())),flush=True)
    guard();collect(output)


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('command',choices=['register','run','collect'])
    p.add_argument('--output',type=Path,default=HERE/'results');p.add_argument('--workers',type=int,default=4)
    p.add_argument('--partial',action='store_true');args=p.parse_args()
    if args.command=='register':register()
    elif args.command=='run':run(args.output,args.workers)
    else:guard();collect(args.output,args.partial)

Overwriting _source_export/part1/research/other_combiner_v2/run.py


## part1/research/other_combiner_v2/source/p1lib/__init__.py

In [104]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/__init__.py
"""p1lib: composable semi-supervised experimentation framework for CS5785 midterm Part 1."""
from .data import LABELS, load, metrics, row_f1, write_submission  # noqa: F401
from .evaluate import evaluate, make_folds, score, fold_outputs, summary_row, paired_se  # noqa: F401
from .pipeline import run_model, validate  # noqa: F401

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/__init__.py


## part1/research/other_combiner_v2/source/p1lib/blocks.py

In [105]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/blocks.py
"""Optional non-embedding feature blocks (each a toggle in the features config).

Every block is label-free: vocabularies / IDF / SVD are fit only on the rows the fold allows
(`fit_rows`), then applied to all rows.  Output is a dense (n_all, d) array, standardized on
`fit_rows`.

Blocks
  criteria  : TF-IDF of eligibility criteria  -> TruncatedSVD(d)
  diseases  : TF-IDF of diseases_cleaned      -> TruncatedSVD(d)
  drugs     : multi-hot matched_drugs         -> TruncatedSVD(d)
  pathways  : multi-hot drug_pathways (optionally only Reactome level <= max_level) -> SVD(d)
  morgan    : RDKit Morgan fingerprint (radius 2, 1024 bits, max over drugs) -> SVD(d)
  drugmeta  : [has_matched_drug, log1p(#matched drugs)]
  year      : standardized start_year  (sampling artifact; ablation only)
"""
from __future__ import annotations

import numpy as np
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from .cache import memo

_MORGAN = None


def _split_semicolon(s):
    if not isinstance(s, str) or not s.strip():
        return []
    return [t.strip().lower() for t in s.split(";") if t.strip()]


def _morgan_matrix(D):
    global _MORGAN
    if _MORGAN is not None:
        return _MORGAN
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog("rdApp.*")
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    out = np.zeros((D.n, 1024), dtype=np.float32)
    for i, s in enumerate(D.df["drug_smiles"].to_numpy()):
        if not isinstance(s, str):
            continue
        fps = [gen.GetFingerprintAsNumPy(m) for t in s.split(";") if (m := Chem.MolFromSmiles(t)) is not None]
        if fps:
            out[i] = np.max(fps, axis=0)
    _MORGAN = out
    return out


def _svd(M, fit_rows, d, seed=0):
    d = int(min(d, M.shape[1] - 1, len(fit_rows) - 1))
    svd = TruncatedSVD(n_components=max(d, 1), random_state=seed).fit(M[fit_rows])
    return svd.transform(M)


def _standardize(Z, fit_rows):
    mu = Z[fit_rows].mean(0)
    sd = Z[fit_rows].std(0)
    sd[sd < 1e-12] = 1.0
    return (Z - mu) / sd


def build_block(D, name: str, cfg: dict, fit_rows: np.ndarray) -> np.ndarray:
    cfg = dict(cfg or {})
    key = {"block": name, **cfg}
    return memo("block", key, fit_rows, lambda: _build(D, name, cfg, fit_rows))


def _build(D, name, cfg, fit_rows):
    d = int(cfg.get("d", 32))
    df = D.df
    if name == "criteria" or name == "diseases":
        col = "criteria" if name == "criteria" else "diseases_cleaned"
        text = df[col].fillna("").astype(str).to_numpy()
        vec = TfidfVectorizer(max_features=int(cfg.get("max_features", 5000)), min_df=3,
                              sublinear_tf=True, ngram_range=(1, int(cfg.get("ngram", 1))),
                              stop_words="english")
        vec.fit(text[fit_rows])
        M = vec.transform(text)
        Z = _svd(M, fit_rows, d)
    elif name in ("drugs", "pathways"):
        col = "matched_drugs" if name == "drugs" else "drug_pathways"
        toks = [_split_semicolon(s) for s in df[col].to_numpy()]
        if name == "pathways" and cfg.get("max_level") is not None:
            lvl = dict(zip(D.catalog["reactome_id"].str.lower(), D.catalog["pathway_level"]))
            ml = int(cfg["max_level"])
            toks = [[t for t in ts if lvl.get(t, 99) <= ml] for ts in toks]
        docs = [" ".join(t.replace(" ", "_") for t in ts) for ts in toks]
        vec = CountVectorizer(binary=True, min_df=3, token_pattern=r"[^ ]+")
        try:
            vec.fit([docs[i] for i in fit_rows])
            M = vec.transform(docs).astype(float)
        except ValueError:            # empty vocabulary
            M = sparse.csr_matrix((D.n, 1))
        Z = _svd(M, fit_rows, d) if M.shape[1] > 2 else M.toarray()
    elif name == "morgan":
        M = _morgan_matrix(D)
        Z = _svd(sparse.csr_matrix(M), fit_rows, d)
    elif name == "drugmeta":
        toks = [_split_semicolon(s) for s in df["matched_drugs"].to_numpy()]
        Z = np.c_[[len(t) > 0 for t in toks], np.log1p([len(t) for t in toks])].astype(float)
    elif name == "year":
        Z = df["start_year"].to_numpy(float)[:, None]
    else:
        raise ValueError(f"unknown block {name}")
    return _standardize(np.asarray(Z, float), fit_rows) * float(cfg.get("weight", 1.0))

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/blocks.py


## part1/research/other_combiner_v2/source/p1lib/cache.py

In [106]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/cache.py
"""Tiny in-process memo for label-free fits (scalers, PCA, clusterers, graphs).

A label-free object depends only on (its config, the set of rows whose features it was fit on).
In transductive mode that row set is the same (all 1,561 rows) for every fold, so one fit is
reused across folds and trials; in inductive mode the row set changes per fold and the cache
simply misses.  Nothing that touches labels is ever cached here.
"""
from __future__ import annotations

import hashlib
import json

import numpy as np

_CACHE: dict = {}
_MAX = 4000


def cfg_key(obj) -> str:
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]


def rows_key(rows: np.ndarray) -> str:
    return hashlib.md5(np.sort(np.asarray(rows)).astype(np.int64).tobytes()).hexdigest()[:16]


def memo(kind: str, cfg, rows, fn):
    key = (kind, cfg_key(cfg), rows_key(rows))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val


def clear():
    _CACHE.clear()


def memo_ctx(kind: str, cfg, ctx, fn, extra=None):
    """Memo for LABEL-DEPENDENT results: keyed by the exact training rows (so a result is only
    reused for the identical fold), the feature rows, the pool and the requested rows."""
    key = (kind, cfg_key(cfg), rows_key(ctx.train), rows_key(ctx.unsup), rows_key(ctx.pool),
           ctx.seed, None if extra is None else cfg_key(extra))
    if key in _CACHE:
        return _CACHE[key]
    val = fn()
    if len(_CACHE) > _MAX:
        _CACHE.clear()
    _CACHE[key] = val
    return val

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/cache.py


## part1/research/other_combiner_v2/source/p1lib/classifiers.py

In [107]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/classifiers.py
"""Classifiers over the K label combinations.  Common interface:

    clf = make_classifier(cfg, K, CM)
    clf.fit(X, y, w)            # y: int combo ids, w: sample weights (soft labels are
                                #    passed as replicated rows, one per combo, weight lam*q)
    P = clf.predict_proba(X)    # (n, K), columns aligned to all K combos
    clf.prior_                  # effective source prior the model was trained under
                                # (needed by SLD; uniform when class_weight='balanced')

cfg["type"] in {"lr", "ovr", "lda", "knn", "hgb", "nb"}.
"""
from __future__ import annotations

import warnings

import numpy as np
from sklearn.covariance import ledoit_wolf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


def _weighted_prior(y, w, K):
    p = np.bincount(y, weights=w, minlength=K).astype(float)
    return p / p.sum()


def _balance(y, w, K):
    """class_weight='balanced' computed from the weighted label mass."""
    mass = np.bincount(y, weights=w, minlength=K)
    present = mass > 0
    cw = np.zeros(K)
    cw[present] = w.sum() / (present.sum() * mass[present])
    return w * cw[y], present


class _Base:
    def __init__(self, cfg, K, CM):
        self.cfg = cfg
        self.K = K
        self.CM = CM

    def _prep(self, y, w):
        w = np.ones(len(y)) if w is None else np.asarray(w, float)
        if self.cfg.get("class_weight") == "balanced":
            w, present = _balance(y, w, self.K)
            self.prior_ = present / present.sum()
        else:
            self.prior_ = _weighted_prior(y, w, self.K)
        return w

    def _full(self, Pc, classes):
        P = np.zeros((Pc.shape[0], self.K))
        P[:, classes] = Pc
        P = P + 1e-9
        return P / P.sum(1, keepdims=True)


class LRPowerset(_Base):
    """Multinomial logistic regression on the label powerset (T1)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.m = LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                    tol=1e-5).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class OvRLR(_Base):
    """One-vs-rest LR on the 5 labels; combo probabilities = product of marginals renormalized
    over the K allowed combinations (so the output is always a valid label set)."""

    def fit(self, X, y, w=None):
        w0 = np.ones(len(y)) if w is None else np.asarray(w, float)
        self.prior_ = _weighted_prior(y, w0, self.K)
        B = self.CM[y]
        self.ms = []
        for j in range(5):
            yj = B[:, j]
            wj = w0.copy()
            if self.cfg.get("class_weight") == "balanced":
                pos = wj[yj == 1].sum()
                neg = wj[yj == 0].sum()
                if pos > 0 and neg > 0:
                    wj = np.where(yj == 1, wj * (pos + neg) / (2 * pos), wj * (pos + neg) / (2 * neg))
            if yj.min() == yj.max():
                self.ms.append(float(yj[0]))
                continue
            self.ms.append(LogisticRegression(C=float(self.cfg.get("C", 0.1)), l1_ratio=0, max_iter=5000,
                                              tol=1e-5).fit(X, yj, sample_weight=wj))
        if self.cfg.get("class_weight") == "balanced":
            self.prior_ = np.ones(self.K) / self.K
        return self

    def marginals(self, X):
        M = np.zeros((len(X), 5))
        for j, m in enumerate(self.ms):
            M[:, j] = m if isinstance(m, float) else m.predict_proba(X)[:, 1]
        return np.clip(M, 1e-6, 1 - 1e-6)

    def predict_proba(self, X):
        M = self.marginals(X)
        L = np.log(M) @ self.CM.T + np.log(1 - M) @ (1 - self.CM).T
        L -= L.max(1, keepdims=True)
        P = np.exp(L)
        return P / P.sum(1, keepdims=True)


class ShrinkLDA(_Base):
    """Shrinkage LDA (tied Gaussian classes) with sample-weight support.  Follows sklearn's
    solver='lsqr', shrinkage='auto': per-class Ledoit-Wolf covariances (on standardized
    features) averaged with the class priors."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        K, d = self.K, X.shape[1]
        self.classes_ = np.unique(y)
        means = np.zeros((K, d))
        S = np.zeros((d, d))
        prior = self.prior_
        for k in self.classes_:
            m = y == k
            wk = w[m]
            Xk = X[m]
            mu = (wk[:, None] * Xk).sum(0) / wk.sum()
            means[k] = mu
            Xc = (Xk - mu) * np.sqrt(wk / wk.mean())[:, None]
            if len(Xk) < 2:
                continue
            sd = Xc.std(0)
            sd[sd < 1e-12] = 1.0
            C = ledoit_wolf(Xc / sd, assume_centered=True)[0]
            S += prior[k] * (sd[:, None] * C * sd[None, :])
        S += 1e-8 * np.eye(d)
        self.means_ = means
        self.coef_ = np.linalg.solve(S, means[self.classes_].T).T
        self.intercept_ = -0.5 * (self.coef_ * means[self.classes_]).sum(1) + np.log(prior[self.classes_])
        return self

    def predict_proba(self, X):
        Lg = X @ self.coef_.T + self.intercept_
        Lg -= Lg.max(1, keepdims=True)
        P = np.exp(Lg)
        P /= P.sum(1, keepdims=True)
        return self._full(P, self.classes_)


class KNNClf(_Base):
    """Cosine kNN over combos with distance weighting and soft targets; smoothed toward the
    training prior with beta pseudo-counts so probabilities are never exactly 0."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        self.Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        self.Yw = np.eye(self.K)[y] * w[:, None]
        self.w = w
        return self

    def predict_proba(self, X, exclude_self=False):
        k = int(self.cfg.get("k", 10))
        Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
        S = Xn @ self.Xn.T
        if exclude_self:
            np.fill_diagonal(S, -np.inf)
        k = min(k, S.shape[1] - (1 if exclude_self else 0))
        nn = np.argpartition(-S, k - 1, axis=1)[:, :k]
        sims = np.take_along_axis(S, nn, 1)
        if self.cfg.get("weights", "distance") == "distance":
            wt = np.exp(sims / float(self.cfg.get("tau", 0.1)))
        else:
            wt = np.ones_like(sims)
        votes = np.einsum("nk,nkc->nc", wt, self.Yw[nn])
        mass = np.einsum("nk,nk->n", wt, self.w[nn])
        beta = float(self.cfg.get("beta", 1.0))
        P = (votes + beta * self.prior_) / (mass + beta)[:, None]
        return P / P.sum(1, keepdims=True)


class HGB(_Base):
    """HistGradientBoosting over combos (mainly as a stacker on meta-features)."""

    def fit(self, X, y, w=None):
        w = self._prep(y, w)
        c = self.cfg
        self.m = HistGradientBoostingClassifier(
            learning_rate=float(c.get("learning_rate", 0.05)), max_iter=int(c.get("max_iter", 100)),
            max_leaf_nodes=int(c.get("max_leaf_nodes", 6)), max_depth=c.get("max_depth", 3),
            min_samples_leaf=int(c.get("min_samples_leaf", 20)),
            l2_regularization=float(c.get("l2_regularization", 1.0)),
            max_features=float(c.get("max_features", 1.0)), early_stopping=False,
            random_state=int(c.get("seed", 0))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class GaussNB(_Base):
    """Gaussian naive Bayes with variance smoothing (cheap, high-bias baseline)."""

    def fit(self, X, y, w=None):
        from sklearn.naive_bayes import GaussianNB
        w = self._prep(y, w)
        self.m = GaussianNB(var_smoothing=float(self.cfg.get("var_smoothing", 1e-2))).fit(X, y, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self._full(self.m.predict_proba(X), self.m.classes_)


class PriorClf(_Base):
    """Predicts the training prior for every row (MAP -> always the majority combo)."""

    def fit(self, X, y, w=None):
        self._prep(y, w)
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))


REGISTRY = {"prior": PriorClf, "lr": LRPowerset, "ovr": OvRLR, "lda": ShrinkLDA, "knn": KNNClf, "hgb": HGB, "nb": GaussNB}


def make_classifier(cfg, K, CM):
    return REGISTRY[cfg["type"]](cfg, K, CM)

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/classifiers.py


## part1/research/other_combiner_v2/source/p1lib/configs.py

In [108]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/configs.py
"""Named reference configurations (the report's five techniques, baselines and ensembles).

Every entry is a full experiment config: {"model": ..., "head": ..., "transductive": ...}.
Build new ones by copying and editing; `exp(model, head)` is a small helper.
"""
from __future__ import annotations

import copy

STD = {"scale": True}
PCA64 = {"scale": True, "pca": 64}
GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}

NO_HEAD = {"prior": "none", "decode": "map"}
SLD_HEAD = {"prior": "sld", "sld_a": 100, "decode": "map"}


def exp(model, head=None, transductive=True):
    return {"model": copy.deepcopy(model), "head": copy.deepcopy(head or NO_HEAD), "transductive": transductive}


def single(clf, features=STD, synth=None, labeler=None, temperature=False):
    return {"features": copy.deepcopy(features), "synth": copy.deepcopy(synth or []),
            "labeler": copy.deepcopy(labeler), "clf": copy.deepcopy(clf), "temperature": temperature}


LR = {"type": "lr", "C": 0.1}
LDA = {"type": "lda"}
OVR = {"type": "ovr", "C": 0.1}
KNN = {"type": "knn", "k": 10, "weights": "distance", "tau": 0.1, "beta": 1.0}

M_LR = single(LR)
M_LDA = single(LDA)
M_OVR = single(OVR)
M_KNN = single(KNN, features={"scale": True, "pca": 64, "l2": True})
M_CS = single({"type": "direct", "labeler": {"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5}})
M_SPREAD = single({"type": "direct", "labeler": {"type": "spread", "graph": GRAPH, "alpha": 0.5}})
M_CLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "kmeans", "k": 24, "m": 4}})
M_GMMCLUSTER = single({"type": "direct", "labeler": {"type": "cluster", "algo": "gmm", "k": 16, "cov": "diag", "m": 4}})

# T1 with SLD-filled pseudo-labels for the unlabeled pool
M_LR_SLDPL = single(LR, labeler={"type": "sld", "base": LR, "a": 100, "lam": 0.1})
# T2 C&S pseudo-labels feeding LR
M_LR_CSPL = single(LR, labeler={"type": "cs", "graph": GRAPH, "a1": 0.5, "a2": 0.5, "lam": 0.1})
# T4 cluster-then-label pseudo-labels feeding LR
M_LR_CLPL = single(LR, labeler={"type": "cluster", "algo": "kmeans", "k": 24, "m": 4, "lam": 0.1})
# T3 guarded self-training
M_LR_ST = single(LR, labeler={"type": "selftrain", "members": ["lr", "lda", "cs"], "rounds": 2, "tau": 0.75,
                              "lam": 0.3, "ramp": True, "align": "sld", "weighting": "softmatch",
                              "cluster_filter": True})
M_LR_ST_NAIVE = single(LR, labeler={"type": "selftrain", "members": ["lr"], "rounds": 3, "tau": 0.75,
                                    "lam": 0.5, "align": "none", "hard": True})
# synthetic-feature variants
M_LR_SYN = single(LR, synth=[{"type": "gmm", "k": 16, "cov": "diag"}, {"type": "knn_vote", "k": 10},
                             {"type": "graph", "mode": "spread", "graph": GRAPH, "alpha": 0.5},
                             {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}])
# T5 HGB stacker on nested-OOF meta-features
META = [{"type": "oof_model", "model": M_LR}, {"type": "oof_model", "model": M_LDA},
        {"type": "graph", "mode": "cs", "graph": GRAPH, "alpha": 0.5},
        {"type": "knn_vote", "k": 10}, {"type": "cluster_map", "algo": "kmeans", "k": 24, "m": 4}]
HGB = {"type": "hgb", "learning_rate": 0.05, "max_iter": 100, "max_leaf_nodes": 4, "max_depth": 2,
       "min_samples_leaf": 20, "l2_regularization": 1.0}
M_STACK_HGB = single(HGB, features=None, synth=META)
M_STACK_LR = single({"type": "lr", "C": 0.1}, features=None, synth=META)
M_HGB_RAW = single(HGB, features=STD)

# report's recommended ensemble: T1-LR + T1-LDA + T2-C&S, equal weights
M_ENS3 = {"ensemble": [M_LR, M_LDA, M_CS], "weights": None}

M_PRIOR = single({"type": "prior"})
M_NB = single({"type": "nb"}, features=PCA64)
SLD_EF1 = {"prior": "sld", "sld_a": 25, "decode": "expf1"}

REFERENCE = {
    "Majority": exp(M_PRIOR),
    "T1_LR": exp(M_LR),
    "T1_LR+SLD": exp(M_LR, SLD_HEAD),
    "T1_LDA": exp(M_LDA),
    "T1_LDA+SLD": exp(M_LDA, SLD_HEAD),
    "OvR_LR": exp(M_OVR),
    "kNN": exp(M_KNN),
    "T2_CS": exp(M_CS),
    "T2_CS+SLD": exp(M_CS, SLD_HEAD),
    "T2_spread": exp(M_SPREAD),
    "T4_kmeans_direct": exp(M_CLUSTER),
    "T4_gmm_direct": exp(M_GMMCLUSTER),
    "T1_LR+SLDpseudo": exp(M_LR_SLDPL),
    "T1_LR+SLDpseudo+SLD": exp(M_LR_SLDPL, SLD_HEAD),
    "T2_LR+CSpseudo": exp(M_LR_CSPL),
    "T4_LR+clusterpseudo": exp(M_LR_CLPL),
    "T3_selftrain_guarded": exp(M_LR_ST),
    "T3_selftrain_naive": exp(M_LR_ST_NAIVE),
    "LR+synth": exp(M_LR_SYN),
    "T5_stack_HGB": exp(M_STACK_HGB),
    "T5_stack_HGB+SLD": exp(M_STACK_HGB, SLD_HEAD),
    "stack_LR": exp(M_STACK_LR),
    "HGB_raw": exp(M_HGB_RAW),
    "ENS3": exp(M_ENS3),
    "ENS3+SLD": exp(M_ENS3, SLD_HEAD),
}

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/configs.py


## part1/research/other_combiner_v2/source/p1lib/context.py

In [109]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/context.py
"""Fold context: which rows are labeled-for-training, which are predicted, which are unlabeled.

Every component receives a `Ctx` and may only read labels of `ctx.train`.  Features of
`ctx.unsup` rows may be used by label-free steps (scaler, PCA, clustering, graph nodes).

transductive=True  -> unsup = all 1,561 rows (train + val + test features, no labels)
transductive=False -> unsup = all rows except the eval rows and the test rows
"""
from __future__ import annotations

from dataclasses import dataclass, replace

import numpy as np


@dataclass
class Ctx:
    D: object
    train: np.ndarray            # global idx of labeled training rows (labels visible)
    eval: np.ndarray             # global idx of rows to predict (labels hidden)
    pool: np.ndarray             # global idx of the pseudo-label pool (default: 1,014 unlabeled)
    unsup: np.ndarray            # global idx of rows whose features label-free steps may use
    transductive: bool = True
    seed: int = 0
    name: str = ""
    depth: int = 0               # nesting depth (0 = outer fold)

    @property
    def y_train(self):
        return self.D.y[self.train]

    @property
    def sld_target(self):
        """Rows SLD estimates the target prior on: the eval batch when transductive, else the
        unlabeled pool (inductive: we may not look at the eval/test features)."""
        return self.eval if self.transductive else self.pool

    def sub(self, train, eval_, name=None):
        """Inner context for nested OOF: same unlabeled pool / feature rows, fewer labels."""
        return replace(self, train=np.asarray(train), eval=np.asarray(eval_),
                       name=name or self.name + "/inner", depth=self.depth + 1)


def make_ctx(D, train, eval_, transductive=True, pool=None, seed=0, name=""):
    train = np.asarray(train)
    eval_ = np.asarray(eval_)
    pool = D.idx["unl"] if pool is None else np.asarray(pool)
    if transductive:
        unsup = np.arange(D.n)
    else:
        banned = np.zeros(D.n, bool)
        banned[eval_] = True
        banned[D.idx["test"]] = True
        unsup = np.where(~banned)[0]
    assert len(np.intersect1d(train, eval_)) == 0, "train/eval overlap"
    assert (D.y[train] >= 0).all(), "training rows must be labeled"
    return Ctx(D=D, train=train, eval=eval_, pool=pool, unsup=unsup,
               transductive=transductive, seed=seed, name=name)

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/context.py


## part1/research/other_combiner_v2/source/p1lib/data.py

In [110]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/data.py
"""Data loading, label-powerset encoding and metrics for Part 1.

Global row order used everywhere in this package:
    [ part1_train (1171 rows, file order) | part1_val (78) | part1_test (312) ]  -> 1561 rows.
`Data.y` holds the label-combo id (0..7) for labeled rows and -1 for masked/test rows.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field

import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

LABELS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMB = [f"emb_{i}" for i in range(150)]
BITS = np.array([1, 2, 4, 8, 16])

DEFAULT_DATA_DIR = os.environ.get(
    "P1_DATA_DIR",
    os.path.join(os.path.dirname(__file__), "..", "..", "..", "part1", "data"),
)


@dataclass
class Data:
    df: pd.DataFrame                 # all 1561 rows, global order, with a 'split' column
    emb: np.ndarray                  # (1561, 150) raw embeddings
    Y: np.ndarray                    # (1561, 5) labels, -1 where unknown
    y: np.ndarray                    # (1561,) combo id, -1 where unknown
    combos: np.ndarray               # (K,) bitmask of each allowed combo
    CM: np.ndarray                   # (K, 5) binary label matrix of each combo
    idx: dict = field(default_factory=dict)   # named index groups
    year: np.ndarray = None
    catalog: pd.DataFrame = None

    @property
    def K(self) -> int:
        return len(self.combos)

    @property
    def n(self) -> int:
        return len(self.df)

    def combo_names(self):
        return ["+".join(LABELS[j][0] for j in range(5) if self.CM[k, j]) for k in range(self.K)]


def load(data_dir: str | None = None) -> Data:
    d = os.path.abspath(data_dir or DEFAULT_DATA_DIR)
    tr = pd.read_csv(os.path.join(d, "part1_train.csv"))
    va = pd.read_csv(os.path.join(d, "part1_val.csv"))
    te = pd.read_csv(os.path.join(d, "part1_test.csv"))
    cat = pd.read_csv(os.path.join(d, "pathway_catalog.csv"))
    te = pd.concat([te, pd.DataFrame(-1, index=te.index, columns=LABELS)], axis=1)
    tr = tr.assign(split="train")
    va = va.assign(split="val")
    te = te.assign(split="test")
    df = pd.concat([tr, va, te], ignore_index=True).copy()
    Y = df[LABELS].to_numpy().astype(int)
    Y[Y == -100] = -1
    known = (Y >= 0).all(1)
    mask = (Y.clip(0) * BITS).sum(1)
    combos = np.unique(mask[known])
    y = np.full(len(df), -1)
    y[known] = np.searchsorted(combos, mask[known])
    CM = np.array([[(c >> j) & 1 for j in range(5)] for c in combos])
    split = df["split"].to_numpy()
    idx = {
        "train": np.where(split == "train")[0],
        "lab": np.where((split == "train") & known)[0],
        "unl": np.where((split == "train") & ~known)[0],
        "val": np.where(split == "val")[0],
        "test": np.where(split == "test")[0],
    }
    idx["labval"] = np.r_[idx["lab"], idx["val"]]
    D = Data(df=df, emb=df[EMB].to_numpy(float), Y=Y, y=y, combos=combos, CM=CM, idx=idx,
             year=df["start_year"].to_numpy(int), catalog=cat)
    assert len(idx["lab"]) == 157 and len(idx["unl"]) == 1014 and len(idx["val"]) == 78 and len(idx["test"]) == 312
    assert (Y[idx["lab"]].sum(1) >= 1).all(), "labeled rows are never all-zero"
    return D


# ----------------------------------------------------------------------------- metrics
def row_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-row F1 between binary label sets (both non-empty here)."""
    inter = (Ytrue * Ypred).sum(1)
    denom = Ytrue.sum(1) + Ypred.sum(1)
    out = np.where(denom > 0, 2 * inter / np.maximum(denom, 1), 1.0)
    return out


def label_f1(Ytrue: np.ndarray, Ypred: np.ndarray) -> np.ndarray:
    """Per-label (column) binary F1; F1 = 0 when a label has no positives and none predicted."""
    tp = (Ytrue * Ypred).sum(0)
    fp = ((1 - Ytrue) * Ypred).sum(0)
    fn = (Ytrue * (1 - Ypred)).sum(0)
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)


def metrics(Ytrue: np.ndarray, Ypred: np.ndarray) -> dict:
    rf = row_f1(Ytrue, Ypred)
    lf = label_f1(Ytrue, Ypred)
    return {
        "sample_f1": float(rf.mean()),
        "sample_f1_se": float(rf.std(ddof=1) / np.sqrt(len(rf))),
        "macro_f1": float(lf.mean()),
        **{f"f1_{LABELS[j]}": float(lf[j]) for j in range(5)},
        "n": int(len(rf)),
    }


def bootstrap_se(Ytrue: np.ndarray, Ypreds: list[np.ndarray], B: int = 300, seed: int = 0) -> dict:
    """Bootstrap SE (over rows) of macro and per-label F1, averaged over a list of prediction
    matrices for the same rows (e.g. CV repeats) using shared resamples."""
    rng = np.random.default_rng(seed)
    n = len(Ytrue)
    macro, per = [], []
    for _ in range(B):
        b = rng.integers(0, n, n)
        lfs = np.mean([label_f1(Ytrue[b], P[b]) for P in Ypreds], axis=0)
        per.append(lfs)
        macro.append(lfs.mean())
    per = np.array(per)
    return {"macro_f1_se": float(np.std(macro, ddof=1)),
            **{f"f1_{LABELS[j]}_se": float(per[:, j].std(ddof=1)) for j in range(5)}}


def combos_to_labels(D: Data, kidx: np.ndarray) -> np.ndarray:
    return D.CM[kidx]


def write_submission(D: Data, Ypred: np.ndarray, path: str, ids=None) -> pd.DataFrame:
    """Kaggle format: ID,Predicted with space-separated reason names (always >=1)."""
    assert (Ypred.sum(1) >= 1).all(), "every test row needs at least one predicted reason"
    ids = D.df["ID"].to_numpy()[D.idx["test"]] if ids is None else ids
    pred = [" ".join(LABELS[j] for j in range(5) if row[j]) for row in Ypred]
    sub = pd.DataFrame({"ID": ids, "Predicted": pred})
    sub.to_csv(path, index=False, lineterminator="\n")
    return sub

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/data.py


## part1/research/other_combiner_v2/source/p1lib/evaluate.py

In [111]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/evaluate.py
"""Leak-free evaluation harness.

Protocols (all over the 235 labeled rows = 157 labeled-train + 78 val):
  A : RepeatedStratifiedKFold(5 x R) stratified on the label-combo id         (no shift)
  B : chronological holdout: train start_year < cut, evaluate >= cut         (shift, cut=2005)
  C : official split: train on the 157 labeled-train rows, evaluate on val   (tie-breaker)
Every fold builds a fresh Ctx; everything label-dependent is fit inside `run_model` from
ctx.train labels only.  Label-free fits are reused across folds only when their feature-row
set is identical (transductive mode).

Fold outputs (calibrated combo probabilities BEFORE the head) are cached on disk keyed by the
model part of the config, so SLD strength / decoding are post-hoc and free to vary.
"""
from __future__ import annotations

import hashlib
import json
import os
import time

import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold

from .context import make_ctx
from .data import LABELS, bootstrap_se, label_f1, metrics, row_f1
from .head import apply_head
from .pipeline import model_part, run_model, validate

# fold-output cache lives OUTSIDE the midterm/ folder (regenerable, ~0.5 GB); override with P1_CACHE
CACHE_DIR = os.environ.get("P1_CACHE", os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "..", ".p1_cache", "folds")))
CHRONO_CUT = 2005
CHRONO_CUTS = (2004, 2005, 2006)   # objective uses the mean over these cuts


def cfg_hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:20]


# ----------------------------------------------------------------------------- folds
def make_folds(D, protocols=("A", "B", "C"), R=4, seed=0, rows=None, chrono_cuts=CHRONO_CUTS):
    """rows: restrict to a subset of labeled rows (used by the nested search-overfitting check)."""
    labval = D.idx["labval"] if rows is None else np.asarray(rows)
    y = D.y[labval]
    folds = []
    if "A" in protocols:
        rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=R, random_state=seed)
        for i, (tr, te) in enumerate(rskf.split(np.zeros(len(y)), y)):
            folds.append({"name": f"A{i // 5}_{i % 5}", "protocol": "A", "repeat": i // 5,
                          "train": labval[tr], "eval": labval[te]})
    if "B" in protocols:
        for cut in chrono_cuts:
            yr = D.year[labval]
            folds.append({"name": f"B{cut}", "protocol": "B", "cut": cut,
                          "train": labval[yr < cut], "eval": labval[yr >= cut]})
    if "C" in protocols:
        tr = np.intersect1d(labval, D.idx["lab"])
        te = np.intersect1d(labval, D.idx["val"])
        if len(tr) and len(te):
            folds.append({"name": "C", "protocol": "C", "train": tr, "eval": te})
    return folds


def fold_tag(fold):
    h = hashlib.md5(np.r_[fold["train"], [-1], fold["eval"]].astype(np.int64).tobytes()).hexdigest()[:10]
    return f"{fold['name']}_{h}"


# ----------------------------------------------------------------------------- one fold
def run_fold(cfg, D, fold, use_cache=True):
    mp = model_part(cfg)
    key = cfg_hash(mp)
    path = os.path.join(CACHE_DIR, key, fold_tag(fold) + ".npz")
    if use_cache and os.path.exists(path):
        try:
            z = np.load(path, allow_pickle=False)
            out = {k: z[k] for k in z.files}
            out["T"] = out["T"].tolist()
            return out
        except Exception:
            pass
    tr = cfg.get("transductive", True)
    ctx = make_ctx(D, fold["train"], fold["eval"], transductive=tr, seed=0, name=fold["name"])
    t0 = time.time()
    rows = fold["eval"] if tr else np.r_[fold["eval"], ctx.pool]
    o = run_model(cfg["model"], ctx, rows)
    ne = len(fold["eval"])
    out = {"P_eval": o["P"][:ne].astype(np.float32), "prior": np.asarray(o["prior"], float),
           "T": np.atleast_1d(np.asarray(o["T"], float)), "secs": np.float32(time.time() - t0)}
    if not tr:
        out["P_target"] = o["P"][ne:].astype(np.float32)
    if use_cache:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + f".{os.getpid()}.tmp.npz"
        np.savez_compressed(tmp, **out)
        os.replace(tmp, path)
        mpath = os.path.join(CACHE_DIR, key, "config.json")
        if not os.path.exists(mpath):
            with open(mpath, "w") as f:
                json.dump(mp, f, indent=1, default=str)
    out["T"] = out["T"].tolist()
    return out


def fold_outputs(cfg, D, folds, n_jobs=1, use_cache=True):
    if n_jobs == 1:
        return [run_fold(cfg, D, f, use_cache) for f in folds]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=n_jobs, backend="loky")(delayed(run_fold)(cfg, D, f, use_cache) for f in folds)


# ----------------------------------------------------------------------------- scoring
def score(cfg, D, folds, outs, bootstrap=True):
    """Apply the head to cached fold outputs and compute A/B/C metrics with SEs."""
    head = cfg.get("head", {})
    res = {}
    per = {}
    for f, o in zip(folds, outs):
        P = o["P_eval"].astype(float)
        P = P / P.sum(1, keepdims=True)
        Pt = o.get("P_target")
        Pt = None if Pt is None else Pt.astype(float) / Pt.astype(float).sum(1, keepdims=True)
        Yp, Q, pi = apply_head(P, o["prior"], D.CM, head, Pt)
        per[f["name"]] = (f, Yp, pi)
    # ---- A ----
    A = [v for v in per.values() if v[0]["protocol"] == "A"]
    if A:
        rows = np.unique(np.concatenate([f["eval"] for f, _, _ in A]))
        pos = {r: i for i, r in enumerate(rows)}
        reps = sorted({f["repeat"] for f, _, _ in A})
        Ytrue = D.Y[rows]
        Ypreds = []
        for r in reps:
            Yp = np.zeros((len(rows), 5), int)
            for f, yp, _ in A:
                if f["repeat"] == r:
                    Yp[[pos[i] for i in f["eval"]]] = yp
            Ypreds.append(Yp)
        rep_metrics = [metrics(Ytrue, Yp) for Yp in Ypreds]
        rf = np.mean([row_f1(Ytrue, Yp) for Yp in Ypreds], axis=0)
        a = {k: float(np.mean([m[k] for m in rep_metrics])) for k in rep_metrics[0] if k not in ("n",)}
        a["sample_f1_se"] = float(rf.std(ddof=1) / np.sqrt(len(rf)))
        a["sample_f1_rep_sd"] = float(np.std([m["sample_f1"] for m in rep_metrics], ddof=1)) if len(reps) > 1 else 0.0
        a["n"] = int(len(rows))
        a["R"] = len(reps)
        if bootstrap:
            a.update(bootstrap_se(Ytrue, Ypreds, B=200))
        res["A"] = a
        res["rowf1_A"] = rf.tolist()
        res["rows_A"] = rows.tolist()
    # ---- B ----  (one or several chronological cuts; B = mean over cuts, B_se = mean per-cut SE
    #               because the cuts share most rows and are highly correlated)
    Bs = [v for v in per.values() if v[0]["protocol"] == "B"]
    if Bs:
        bm = []
        for f, yp, pi in Bs:
            Yt = D.Y[f["eval"]]
            m = metrics(Yt, yp)
            if bootstrap:
                m.update(bootstrap_se(Yt, [yp], B=200))
            m["cut"] = f["cut"]
            if pi is not None:
                m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
            m["true_prior_labels"] = Yt.mean(0).round(3).tolist()
            bm.append(m)
            res.setdefault("rowf1_B", []).extend(row_f1(Yt, yp).tolist())
        res["B_ncuts"] = len(bm)
        if len(bm) == 1:
            res["B"] = bm[0]
        else:
            agg = {k: float(np.mean([m[k] for m in bm])) for k in bm[0]
                   if isinstance(bm[0][k], (int, float)) and k != "cut"}
            agg["cuts"] = bm
            res["B"] = agg
        for m in bm:
            if m["cut"] == CHRONO_CUT:
                res["B2005"] = {k: v for k, v in m.items()}
    # ---- C ----
    Cs = [v for v in per.values() if v[0]["protocol"] == "C"]
    if Cs:
        f, yp, pi = Cs[0]
        Yt = D.Y[f["eval"]]
        m = metrics(Yt, yp)
        if bootstrap:
            m.update(bootstrap_se(Yt, [yp], B=200))
        if pi is not None:
            m["sld_prior_labels"] = (pi @ D.CM).round(3).tolist()
        res["C"] = m
        res["rowf1_C"] = row_f1(Yt, yp).tolist()
    # ---- robust objective ----
    if "A" in res and "B" in res:
        res["J"] = 0.5 * res["A"]["sample_f1"] + 0.5 * res["B"]["sample_f1"]
        res["J_se"] = 0.5 * float(np.hypot(res["A"]["sample_f1_se"], res["B"]["sample_f1_se"]))
    res["secs"] = float(sum(float(o["secs"]) for o in outs))
    res["T_mean"] = float(np.mean([np.mean(o["T"]) for o in outs]))
    return res


def evaluate(cfg, D, protocols=("A", "B", "C"), R=4, n_jobs=1, seed=0, use_cache=True, rows=None,
             bootstrap=True, chrono_cuts=CHRONO_CUTS):
    validate(cfg)
    folds = make_folds(D, protocols, R, seed, rows=rows, chrono_cuts=chrono_cuts)
    outs = fold_outputs(cfg, D, folds, n_jobs, use_cache)
    return score(cfg, D, folds, outs, bootstrap=bootstrap)


def summary_row(name, res):
    r = {"name": name}
    for p in ("A", "B", "B2005", "C"):
        if p in res:
            r[f"{p}"] = res[p]["sample_f1"]
            r[f"{p}_se"] = res[p]["sample_f1_se"]
            r[f"{p}_macro"] = res[p]["macro_f1"]
    if "J" in res:
        r["J"] = res["J"]
        r["J_se"] = res["J_se"]
    r["secs"] = res.get("secs")
    return r


def paired_se(res1, res2, w=(0.5, 0.5)):
    """SE of the difference in J between two configs using per-row paired differences."""
    dA = np.asarray(res1["rowf1_A"]) - np.asarray(res2["rowf1_A"])
    dB = np.asarray(res1["rowf1_B"]) - np.asarray(res2["rowf1_B"])
    nB = len(dB) / res1.get("B_ncuts", 1)          # cuts overlap: effective n ~ one cut
    return float(np.sqrt((w[0] ** 2) * dA.var(ddof=1) / len(dA) + (w[1] ** 2) * dB.var(ddof=1) / nB))

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/evaluate.py


## part1/research/other_combiner_v2/source/p1lib/features.py

In [112]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/features.py
"""Preprocessors: standardize -> [PCA] -> [L2-normalize] on the embeddings, plus optional
non-embedding blocks.  All fitting uses only `ctx.unsup` rows (label-free), so the same
object is refit per fold when the feature-row set changes (inductive) and reused otherwise.

features config (all keys optional):
    {"scale": True, "pca": None|int, "whiten": False, "l2": False,
     "blocks": {"criteria": {"d": 32, "weight": 1.0}, "pathways": {...}, ...}}
"""
from __future__ import annotations

import numpy as np
from sklearn.decomposition import PCA

from .blocks import build_block
from .cache import memo

DEFAULT_FEATURES = {"scale": True, "pca": None, "l2": False, "blocks": {}}


def norm_features_cfg(cfg):
    c = dict(DEFAULT_FEATURES)
    c.update(cfg or {})
    c["blocks"] = dict(c.get("blocks") or {})
    return c


def _embed(D, cfg, rows):
    X = D.emb.copy()
    if cfg["scale"]:
        mu = X[rows].mean(0)
        sd = X[rows].std(0)
        X = (X - mu) / sd
    if cfg["pca"]:
        p = PCA(n_components=int(cfg["pca"]), whiten=bool(cfg.get("whiten", False)), random_state=0).fit(X[rows])
        X = p.transform(X)
    if cfg["l2"]:
        X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    return X


def featurize(ctx, cfg) -> np.ndarray:
    """Return the (n_all, d) representation for every global row."""
    cfg = norm_features_cfg(cfg)
    emb_cfg = {k: cfg[k] for k in ("scale", "pca", "l2", "whiten") if k in cfg}
    X = memo("embed", emb_cfg, ctx.unsup, lambda: _embed(ctx.D, cfg, ctx.unsup))
    if not cfg["blocks"]:
        return X
    parts = [X] if cfg.get("use_emb", True) else []
    for name in sorted(cfg["blocks"]):
        parts.append(build_block(ctx.D, name, cfg["blocks"][name], ctx.unsup))
    return np.hstack(parts)

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/features.py


## part1/research/other_combiner_v2/source/p1lib/graph.py

In [113]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/graph.py
"""Symmetric RBF-weighted kNN graph over the feature rows a fold may use, and
normalized-Laplacian diffusion (Zhou et al. 2004 label spreading; Huang et al. 2021 C&S).

graph config: {"space": features-cfg, "k": 10, "s": 1.0, "sym": "max"|"min"}
Weights w_ij = exp(-d_ij^2 / (s * median kNN distance)^2) on cosine-normalized vectors,
symmetrized with max (union) or min (mutual kNN), S = D^-1/2 W D^-1/2.
Diffusion solves (I - alpha S) F = (1 - alpha) F0 exactly (LU factor cached per alpha).
"""
from __future__ import annotations

import numpy as np
import scipy.sparse as sp
from scipy.linalg import lu_factor, lu_solve
from sklearn.neighbors import NearestNeighbors

from .cache import memo
from .features import featurize

DEFAULT_GRAPH = {"space": {"scale": True, "pca": 64, "l2": True}, "k": 10, "s": 1.0, "sym": "max"}


class Graph:
    def __init__(self, Z_nodes, nodes, n_all, k, s, sym):
        self.nodes = nodes
        self.pos = np.full(n_all, -1)
        self.pos[nodes] = np.arange(len(nodes))
        self.Zn = Z_nodes / np.maximum(np.linalg.norm(Z_nodes, axis=1, keepdims=True), 1e-12)
        self.nn = NearestNeighbors(n_neighbors=k + 1).fit(self.Zn)
        dist, ind = self.nn.kneighbors(self.Zn)
        dist, ind = dist[:, 1:], ind[:, 1:]
        self.sigma = s * np.median(dist)
        n = len(nodes)
        rows = np.repeat(np.arange(n), k)
        W = sp.csr_matrix((np.exp(-(dist.ravel() ** 2) / self.sigma ** 2), (rows, ind.ravel())), shape=(n, n))
        W = W.maximum(W.T) if sym == "max" else W.minimum(W.T)
        deg = np.asarray(W.sum(1)).ravel()
        iso = deg <= 0
        if iso.any():                      # isolated nodes (possible with mutual kNN): self-loop
            W = W + sp.diags(iso.astype(float))
            deg = np.asarray(W.sum(1)).ravel()
        Dm = sp.diags(1 / np.sqrt(deg))
        self.S = (Dm @ W @ Dm).toarray()
        self.k = k
        self._lu = {}

    def propagate(self, F0, alpha):
        if alpha <= 0:
            return F0.copy()
        if alpha not in self._lu:
            self._lu[alpha] = lu_factor(np.eye(len(self.S)) - alpha * self.S)
        return (1 - alpha) * lu_solve(self._lu[alpha], F0)

    def extend(self, Z_rows, F_nodes):
        """Inductive extension for rows that are not graph nodes: RBF-weighted kNN average."""
        Zn = Z_rows / np.maximum(np.linalg.norm(Z_rows, axis=1, keepdims=True), 1e-12)
        dist, ind = self.nn.kneighbors(Zn, n_neighbors=self.k)
        w = np.exp(-dist ** 2 / self.sigma ** 2) + 1e-12
        return np.einsum("nk,nkc->nc", w, F_nodes[ind]) / w.sum(1, keepdims=True)


def get_graph(ctx, gcfg) -> Graph:
    g = dict(DEFAULT_GRAPH)
    g.update(gcfg or {})
    Z = featurize(ctx, g["space"])

    def build():
        return Graph(Z[ctx.unsup], ctx.unsup, ctx.D.n, int(g["k"]), float(g["s"]), g["sym"])

    return memo("graph", {k: g[k] for k in ("space", "k", "s", "sym")}, ctx.unsup, build), Z


def node_values(graph, Z, F_nodes, rows):
    """Read propagated values for arbitrary global rows (extend for non-nodes)."""
    out = np.zeros((len(rows), F_nodes.shape[1]))
    p = graph.pos[rows]
    m = p >= 0
    out[m] = F_nodes[p[m]]
    if (~m).any():
        out[~m] = graph.extend(Z[rows[~m]], F_nodes)
    return out


def label_spread(ctx, gcfg, alpha, train_rows, y_train, K, out_rows, base_all=None):
    """Pure label spreading (base_all=None) or the Smooth step of C&S (base_all = (n_all,K)
    base probabilities).  Training nodes are clamped to their one-hot labels in F0."""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    F0 = np.zeros((n, K))
    if base_all is not None:
        F0 = base_all[graph.nodes].copy()
    p = graph.pos[train_rows]
    keep = p >= 0
    F0[p[keep]] = np.eye(K)[y_train[keep]]
    F = graph.propagate(F0, alpha)
    return node_values(graph, Z, F, out_rows)


def correct_and_smooth(ctx, gcfg, a1, a2, train_rows, y_train, P_oof_train, P_all, K, out_rows,
                       scale_mode="autoscale"):
    """Correct & Smooth (Huang et al. 2021).
    P_oof_train : base probabilities of the training rows, OUT-OF-FOLD (in-sample residuals ~0)
    P_all       : base probabilities for every global row (fit on all training rows)"""
    graph, Z = get_graph(ctx, gcfg)
    n = len(graph.nodes)
    p = graph.pos[train_rows]
    keep = p >= 0
    Ytr = np.eye(K)[y_train]
    base = P_all[graph.nodes].copy()
    if a1 > 0:
        E0 = np.zeros((n, K))
        E0[p[keep]] = (Ytr - P_oof_train)[keep]
        E = graph.propagate(E0, a1)
        if scale_mode == "autoscale":
            sigma = np.abs(E0[p[keep]]).sum(1).mean()
            scale = sigma / np.maximum(np.abs(E).sum(1, keepdims=True), 1e-12)
            base = base + scale * E
        else:
            base = base + float(scale_mode) * E
        base = np.clip(base, 0, None)
        base = base / np.maximum(base.sum(1, keepdims=True), 1e-12)
    F0 = base.copy()
    F0[p[keep]] = Ytr[keep]
    F = graph.propagate(F0, a2)
    out = node_values(graph, Z, F, out_rows)
    out = np.clip(out, 0, None) + 1e-9
    return out / out.sum(1, keepdims=True)

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/graph.py


## part1/research/other_combiner_v2/source/p1lib/head.py

In [114]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/head.py
"""Shared head: temperature calibration, ONE prior correction (SLD-EM, MAP-stabilized),
and threshold-free decoding to an allowed label set with >= 1 label.

All probabilities are over the K allowed label combinations (label powerset).
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax

EPS = 1e-6


def safe_log(P):
    return np.log(np.clip(P, EPS, None))


def renorm(P):
    P = np.clip(P, 0, None) + 1e-12
    return P / P.sum(1, keepdims=True)


# ----------------------------------------------------------------------------- temperature
def fit_temperature(P_oof: np.ndarray, y: np.ndarray, w: np.ndarray | None = None,
                    bounds=(0.05, 20.0)) -> float:
    """One-parameter temperature T minimizing NLL of softmax(log P / T) on OOF probabilities."""
    L = safe_log(P_oof)
    w = np.ones(len(y)) if w is None else w

    def nll(logT):
        T = np.exp(logT)
        lp = log_softmax(L / T, axis=1)
        return -(w * lp[np.arange(len(y)), y]).sum() / w.sum()

    r = minimize_scalar(nll, bounds=(np.log(bounds[0]), np.log(bounds[1])), method="bounded",
                        options={"xatol": 1e-4})
    return float(np.exp(r.x))


def apply_temperature(P: np.ndarray, T: float) -> np.ndarray:
    if T is None or T == 1.0:
        return P
    return np.exp(log_softmax(safe_log(P) / T, axis=1))


# ----------------------------------------------------------------------------- SLD-EM
def sld_em(P: np.ndarray, prior_src: np.ndarray, a: float | None = 0.0, floor: float = 0.25,
           tol: float = 1e-8, max_iter: int = 1000, cap: float | None = None):
    """Saerens-Latinne-Decaestecker EM for the target label prior, with a Dirichlet(a*pi_s)
    pseudo-count toward the source prior (MAP) and a floor pi >= floor*pi_s.

    cap: optional ceiling pi_k <= cap * pi_s,k (guards the degenerate MLE in which a rare
    'bridge' combo such as Safety+Efficacy absorbs the mass of its parents).
    Returns (Q, pi_hat): prior-corrected posteriors and the estimated target prior.
    a=None or a=inf means 'off' (returns P unchanged)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6            # combos absent from training stay at exactly 0
    ps = np.where(present, prior_src, 0.0)
    ps = ps / ps.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ratio_den = np.where(present, ps, 1.0)
    pi = ps.copy()
    n = len(P)
    for _ in range(max_iter):
        Q = P * (pi / ratio_den)
        Q /= Q.sum(1, keepdims=True)
        new = (Q.sum(0) + a * ps) / (n + a)
        new = np.maximum(new, floor * ps)
        if cap:
            new = np.minimum(new, cap * ps)
        new /= new.sum()
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * (pi / ratio_den)
    Q /= Q.sum(1, keepdims=True)
    return Q, pi


def sld_em_factorized(P: np.ndarray, prior_src: np.ndarray, CM: np.ndarray, a: float | None = 0.0,
                      floor: float = 0.25, tol: float = 1e-8, max_iter: int = 1000):
    """Factorized SLD: the target/source combo-prior ratio is constrained to a product of
    per-label ratios, w_k = prod_j (pi_j/ps_j)^{CM_kj} ((1-pi_j)/(1-ps_j))^{1-CM_kj}.
    Only 5 free parameters (one per reason) instead of K-1, so a rare 'bridge' combo
    (Safety+Efficacy) cannot absorb the mass of its parents.  EM-style fixed point: E-step
    reweights combos by w, M-step sets each label prior to the mean marginal of Q (with the same
    Dirichlet pseudo-count a toward the source marginal and floor)."""
    if a is None or not np.isfinite(a):
        return P, prior_src.copy()
    present = prior_src > 1e-6
    ps_k = np.where(present, prior_src, 0.0)
    ps_k = ps_k / ps_k.sum()
    P = np.where(present[None, :], P, 0.0)
    P = P / np.maximum(P.sum(1, keepdims=True), 1e-300)
    ps = np.clip(ps_k @ CM, 1e-6, 1 - 1e-6)            # source label marginals
    pi = ps.copy()
    n = len(P)

    def weights(pi):
        lw = np.log(pi / ps) @ CM.T + np.log((1 - pi) / (1 - ps)) @ (1 - CM).T
        return np.exp(lw - lw.max())

    for _ in range(max_iter):
        Q = P * weights(pi)
        Q /= Q.sum(1, keepdims=True)
        m = Q @ CM                                          # (n, 5) label marginals
        new = (m.sum(0) + a * ps) / (n + a)
        new = np.clip(np.maximum(new, floor * ps), 1e-6, 1 - 1e-6)
        done = np.abs(new - pi).sum() < tol
        pi = new
        if done:
            break
    Q = P * weights(pi)
    Q /= Q.sum(1, keepdims=True)
    pi_k = ps_k * weights(pi)
    return Q, pi_k / pi_k.sum()


# ----------------------------------------------------------------------------- decoding
_SUBSETS = np.array([[(s >> j) & 1 for j in range(5)] for s in range(1, 32)])


def _f1_matrix(CM):
    inter = _SUBSETS @ CM.T                                   # (31, K)
    return 2 * inter / (_SUBSETS.sum(1)[:, None] + CM.sum(1)[None, :])


def decode(P: np.ndarray, CM: np.ndarray, method: str = "map") -> np.ndarray:
    """Map combo posteriors to a binary label matrix (n, 5); never empty.
    map   : the most probable allowed combination
    expf1 : Bayes-optimal subset for expected sample-F1 over the 31 non-empty subsets"""
    if method == "map":
        return CM[P.argmax(1)]
    if method == "expf1":
        F = _f1_matrix(CM)
        return _SUBSETS[(P @ F.T).argmax(1)]
    raise ValueError(method)


def apply_head(P_eval, prior_src, CM, head: dict, P_target=None):
    """SLD (optional) + decoding.  P_target: posteriors of the SLD target rows (defaults to the
    eval rows, i.e. transductive).  Returns (Ypred, Q_eval, pi_hat)."""
    head = head or {}
    pi_hat = None
    Q = P_eval
    if head.get("prior", "none") == "sld":
        a = head.get("sld_a", 0.0)
        floor = head.get("sld_floor", 0.25)
        cap = head.get("sld_cap")
        if head.get("sld_mode", "combo") == "factorized":
            em = lambda P_: sld_em_factorized(P_, prior_src, CM, a=a, floor=floor)
        else:
            em = lambda P_: sld_em(P_, prior_src, a=a, floor=floor, cap=cap)
        if P_target is None or P_target is P_eval:
            Q, pi_hat = em(P_eval)
        else:
            _, pi_hat = em(P_target)
            present = prior_src > 1e-6
            Q = renorm(np.where(present, P_eval * (pi_hat / np.where(present, prior_src, 1.0)), 0.0))
    return decode(Q, CM, head.get("decode", "map")), Q, pi_hat

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/head.py


## part1/research/other_combiner_v2/source/p1lib/labelers.py

In [115]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/labelers.py
"""Labelers: fill the missing labels of the unlabeled pool with soft combo distributions.

Every labeler type exposes
    soft_labels(lcfg, ctx, rows) -> Q (len(rows), K)      for rows NOT in ctx.train
and, where cheap and exact, a leave-one-out version for training rows (used when the labeler's
output is a synthetic feature or a direct predictor that needs OOF scores):
    soft_labels_train_oof(lcfg, ctx) -> Q (len(ctx.train), K)

`pseudo_labels(lcfg, ctx, clf_cfg)` turns soft labels into weighted pseudo-rows
(rows, Q, w) for the pipeline's classifier (soft-label replication, weight lam * q_ik).

Types
  cluster   : cluster-then-label (kmeans | gmm | seeded_kmeans | hdbscan), m-smoothed map   [T4]
  spread    : label spreading on the kNN graph (Zhou et al. 2004)                            [T2]
  cs        : Correct & Smooth on the kNN graph (Huang et al. 2021)                          [T2]
  sld       : calibrated base classifier + SLD-EM on the pool (Saerens et al. 2002)          [T1]
  selftrain : guarded self-training (agreement, alignment, soft weights, ramp)               [T3]
"""
from __future__ import annotations

import numpy as np
from sklearn.cluster import HDBSCAN, KMeans, kmeans_plusplus
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold

from .cache import memo, memo_ctx, rows_key
from .classifiers import make_classifier
from .features import featurize
from .graph import correct_and_smooth, label_spread
from .head import apply_temperature, fit_temperature, sld_em, sld_em_factorized

DEFAULT_CLUSTER_SPACE = {"scale": True, "pca": 32, "l2": True}


# ============================================================================ clustering
def _cluster_fit(D, algo, k, Z, rows, cfg, seed=0):
    info = {}
    if algo == "kmeans":
        km = KMeans(n_clusters=k, n_init=int(cfg.get("n_init", 10)), max_iter=300, tol=1e-4,
                    random_state=seed).fit(Z[rows])
        dist = km.transform(Z)
        info.update(inertia=float(km.inertia_), n_iter=int(km.n_iter_))
        return {"dist": dist, "labels": km.predict(Z), "info": info}
    if algo == "gmm":
        g = GaussianMixture(n_components=k, covariance_type=cfg.get("cov", "diag"), reg_covar=1e-3,
                            n_init=int(cfg.get("n_init", 3)), max_iter=500, tol=1e-4,
                            init_params="k-means++", random_state=seed).fit(Z[rows])
        info.update(converged=bool(g.converged_), n_iter=int(g.n_iter_), bic=float(g.bic(Z[rows])),
                    lower_bound=float(g.lower_bound_))
        return {"resp": g.predict_proba(Z), "labels": g.predict(Z), "info": info}
    if algo == "hdbscan":
        h = HDBSCAN(min_cluster_size=int(cfg.get("min_cluster_size", 10)),
                    min_samples=cfg.get("min_samples")).fit(Z[rows])
        lab = np.full(D.n, -1)
        lab[rows] = h.labels_
        others = np.setdiff1d(np.arange(D.n), rows)
        if len(others):                     # inductive: attach to the nearest fitted point
            Zr = Z[rows]
            nn = np.argmax(Z[others] @ Zr.T / (np.linalg.norm(Z[others], axis=1)[:, None] *
                                                np.linalg.norm(Zr, axis=1)[None] + 1e-12), axis=1)
            lab[others] = h.labels_[nn]
        info.update(n_clusters=int(h.labels_.max() + 1), noise_frac=float((h.labels_ < 0).mean()))
        return {"labels": lab, "info": info}
    raise ValueError(algo)


def cluster_resp(lcfg, ctx):
    """(n_all, C) cluster membership (hard one-hot, kmeans-soft, or GMM responsibilities)."""
    algo = lcfg.get("algo", "kmeans")
    k = int(lcfg.get("k", 24))
    space = lcfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    if algo == "seeded_kmeans":                 # label-dependent init -> not shared across folds
        y = ctx.y_train
        cls = np.unique(y)
        seeds = np.array([Z[ctx.train][y == c].mean(0) for c in cls])
        extra = max(k - len(cls), 0)
        if extra:
            more, _ = kmeans_plusplus(Z[ctx.unsup], n_clusters=extra, random_state=ctx.seed)
            seeds = np.vstack([seeds, more])
        km = KMeans(n_clusters=len(seeds), init=seeds, n_init=1, max_iter=300).fit(Z[ctx.unsup])
        lab = km.predict(Z)
        return np.eye(len(seeds))[lab]
    cfg_key = {"algo": algo, "k": k, "space": space, **{kk: lcfg[kk] for kk in ("cov", "n_init", "min_cluster_size", "min_samples") if kk in lcfg}}
    fit = memo("cluster", cfg_key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, lcfg))
    if algo == "gmm":
        return fit["resp"]
    if algo == "kmeans" and lcfg.get("soft_temp"):
        d2 = fit["dist"] ** 2
        T = float(lcfg["soft_temp"]) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    lab = fit["labels"]
    C = int(max(lab.max() + 1, 1))
    R = np.zeros((ctx.D.n, C))
    R[lab >= 0, lab[lab >= 0]] = 1.0          # hdbscan noise rows keep all-zero membership
    return R


def _cluster_map(R_train, y, K, m, prior):
    N = R_train.T @ np.eye(K)[y]                    # (C, K) label mass per cluster
    return N, (N + m * prior) / (N.sum(1, keepdims=True) + m)


def cluster_soft(lcfg, ctx, rows):
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)
    y = ctx.y_train
    prior = np.bincount(y, minlength=K) / len(y)
    _, pi = _cluster_map(R[ctx.train], y, K, float(lcfg.get("m", 4)), prior)
    Q = R[rows] @ pi
    mass = R[rows].sum(1, keepdims=True)
    Q = Q + (1 - np.clip(mass, 0, 1)) * prior        # noise / empty -> global prior
    return Q / Q.sum(1, keepdims=True)


def cluster_soft_train_oof(lcfg, ctx):
    """Exact leave-one-out cluster map for training rows."""
    K = ctx.D.K
    R = cluster_resp(lcfg, ctx)[ctx.train]
    y = ctx.y_train
    m = float(lcfg.get("m", 4))
    prior = np.bincount(y, minlength=K) / len(y)
    N, _ = _cluster_map(R, y, K, m, prior)
    nc = N.sum(1)                                    # (C,)
    out = np.zeros((len(y), K))
    for i in range(len(y)):
        Ni = N - np.outer(R[i], np.eye(K)[y[i]])
        pi = (Ni + m * prior) / (nc - R[i] + m)[:, None]
        q = R[i] @ pi
        q = q + (1 - min(R[i].sum(), 1)) * prior
        out[i] = q / q.sum()
    return out


# ============================================================================ graph labelers
def spread_soft(lcfg, ctx, rows):
    K = ctx.D.K
    F = label_spread(ctx, lcfg.get("graph"), float(lcfg.get("alpha", 0.5)), ctx.train, ctx.y_train, K, rows)
    prior = np.bincount(ctx.y_train, minlength=K) / len(ctx.train)
    F = np.clip(F, 0, None)
    s = F.sum(1, keepdims=True)
    F = np.where(s > 1e-12, F / np.maximum(s, 1e-12), prior)   # unreachable rows -> prior
    beta = float(lcfg.get("beta", 0.01))
    F = (F + beta * prior) / (1 + beta)
    return F


def _base_oof_and_all(base_cfg, ctx, X, n_inner=5):
    """Base classifier probabilities: OOF for training rows, full-fit for every other row."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    P_oof = np.zeros((len(y), K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
        P_oof[te] = m.predict_proba(X[ctx.train[te]])
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P_all = m.predict_proba(X)
    return P_oof, P_all


def cs_soft(lcfg, ctx, rows):
    K = ctx.D.K
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    P_oof, P_all = _base_oof_and_all(base_cfg, ctx, X)
    return correct_and_smooth(ctx, lcfg.get("graph"), float(lcfg.get("a1", 0.5)), float(lcfg.get("a2", 0.5)),
                              ctx.train, ctx.y_train, P_oof, P_all, K, rows)


# ============================================================================ SLD labeler
def calibrated_base(base_cfg, ctx, X, rows, temperature=True):
    """Fit base classifier on ctx.train, temperature-scale with inner OOF, predict rows."""
    from .pipeline import inner_folds
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], y)
    P = m.predict_proba(X[rows])
    T = 1.0
    if temperature:
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            mm = make_classifier(base_cfg, K, CM).fit(X[ctx.train[tr]], y[tr])
            P_oof[te] = mm.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    return P, m.prior_, T


def sld_soft(lcfg, ctx, rows):
    X = featurize(ctx, lcfg.get("base_features", {"scale": True}))
    base_cfg = lcfg.get("base", {"type": "lr", "C": 0.1})
    P, prior, _ = calibrated_base(base_cfg, ctx, X, rows, lcfg.get("temperature", False))
    if lcfg.get("mode", "combo") == "factorized":
        Q, pi = sld_em_factorized(P, prior, ctx.D.CM, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    else:
        Q, pi = sld_em(P, prior, a=float(lcfg.get("a", 0.0)), floor=float(lcfg.get("floor", 0.25)))
    return Q


# ============================================================================ self-training
def _sinkhorn(P, target, iters=200):
    Q = P.copy()
    n = len(Q)
    for _ in range(iters):
        col = Q.sum(0)
        Q = Q * (target * n / np.maximum(col, 1e-12))
        Q = Q / Q.sum(1, keepdims=True)
    return Q


def _softmatch_weight(conf):
    mu, sd = conf.mean(), conf.std() + 1e-6
    return np.where(conf >= mu, 1.0, np.exp(-((conf - mu) ** 2) / (2 * sd ** 2)))


def selftrain_pseudo(lcfg, ctx, clf_cfg, X, pool_rows):
    """Guarded self-training.  Returns (rows, Q, w, info) for the final pseudo-labeled set."""
    K, CM = ctx.D.K, ctx.D.CM
    y = ctx.y_train
    Ytr = np.eye(K)[y]
    R = int(lcfg.get("rounds", 2))
    lam_max = float(lcfg.get("lam", 0.3))
    members = lcfg.get("members", ["lr"])
    align = lcfg.get("align", "none")
    tau = lcfg.get("tau", 0.75)
    top_p = lcfg.get("top_p")
    hard = bool(lcfg.get("hard", False))
    weighting = lcfg.get("weighting", "hard")
    # members that do not use pseudo-labels are computed once
    static = []
    if "lda" in members:
        static.append(make_classifier({"type": "lda"}, K, CM).fit(X[ctx.train], y).predict_proba(X[pool_rows]))
    if "cs" in members:
        static.append(cs_soft({"a1": 0.5, "a2": 0.5}, ctx, pool_rows))
    cl_major = None
    if lcfg.get("cluster_filter"):
        cl_major = cluster_soft({"algo": "kmeans", "k": int(lcfg.get("cluster_k", 24)), "m": 1}, ctx, pool_rows).argmax(1)
    sel_rows, Q, w = pool_rows[:0], np.zeros((0, K)), np.zeros(0)
    prev_lab = None
    info = {}
    for r in range(1, R + 1):
        lam_r = lam_max * (r / R if lcfg.get("ramp", False) else 1.0)
        Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[sel_rows], Q, w)
        main = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
        Pm_list = [main.predict_proba(X[pool_rows])] + static
        Pm = np.mean(Pm_list, axis=0)
        prior_src = np.bincount(y, minlength=K) / len(y)
        if align == "sld":
            Pm, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
        elif align == "sinkhorn":
            _, pi = sld_em_factorized(Pm, prior_src, CM, a=float(lcfg.get("align_a", 100.0)))
            Pm = _sinkhorn(Pm, pi)
        conf = Pm.max(1)
        lab = Pm.argmax(1)
        if top_p:
            keep = np.zeros(len(lab), bool)
            for c in range(K):
                mc = np.where(lab == c)[0]
                if len(mc):
                    thr = np.quantile(conf[mc], 1 - float(top_p))
                    keep[mc[conf[mc] >= thr]] = True
        else:
            keep = conf >= float(tau)
        if len(Pm_list) > 1:
            agree = np.sum([P.argmax(1) == lab for P in Pm_list], axis=0)
            keep &= agree >= 2
        if cl_major is not None:
            keep &= cl_major == lab
        g = _softmatch_weight(conf) if weighting == "softmatch" else np.ones(len(conf))
        sel_rows = pool_rows[keep]
        Q = np.eye(K)[lab[keep]] if hard else Pm[keep]
        w = lam_r * g[keep]
        info = {"round": r, "n_selected": int(keep.sum()), "lam": lam_r}
        if prev_lab is not None and np.mean(prev_lab != lab) < 0.01:
            break
        prev_lab = lab
    return sel_rows, Q, w, info


# ============================================================================ dispatch
SOFT = {"cluster": cluster_soft, "spread": spread_soft, "cs": cs_soft, "sld": sld_soft}


_POST = ("lam", "tau", "hard", "pool")      # applied after soft labels -> not part of the memo key


def soft_labels(lcfg, ctx, rows):
    rows = np.asarray(rows)
    core = {k: v for k, v in lcfg.items() if k not in _POST}
    return memo_ctx("soft", core, ctx, lambda: SOFT[lcfg["type"]](lcfg, ctx, rows), extra=rows.tolist())


def soft_labels_train_oof(lcfg, ctx, n_inner=5):
    """OOF soft labels for the training rows (exact LOO for cluster; inner K-fold otherwise)."""
    if lcfg["type"] == "cluster":
        core = {k: v for k, v in lcfg.items() if k not in _POST}
        return memo_ctx("soft_oof", core, ctx, lambda: cluster_soft_train_oof(lcfg, ctx))
    from .pipeline import inner_folds
    y = ctx.y_train
    out = np.zeros((len(y), ctx.D.K))
    for tr, te in inner_folds(y, n_inner, ctx.seed):
        sub = ctx.sub(ctx.train[tr], ctx.train[te])
        out[te] = soft_labels(lcfg, sub, ctx.train[te])
    return out


def pool_rows_for(lcfg, ctx):
    if lcfg.get("pool", "unl") == "all" and ctx.transductive:
        rows = np.union1d(np.union1d(ctx.pool, ctx.eval), ctx.D.idx["test"])
        return np.setdiff1d(rows, ctx.train)
    return np.setdiff1d(ctx.pool, ctx.train)


def pseudo_labels(lcfg, ctx, clf_cfg, X):
    """Weighted pseudo-rows (rows, Q, w, info) for the pipeline classifier."""
    rows = pool_rows_for(lcfg, ctx)
    if lcfg["type"] == "selftrain":
        return selftrain_pseudo(lcfg, ctx, clf_cfg, X, rows)
    Q = soft_labels(lcfg, ctx, rows)
    conf = Q.max(1)
    keep = conf >= float(lcfg.get("tau", 0.0))
    if lcfg.get("hard", False):
        Q = np.eye(ctx.D.K)[Q.argmax(1)]
    lam = float(lcfg.get("lam", 0.1))
    return rows[keep], Q[keep], np.full(keep.sum(), lam), {"n_selected": int(keep.sum())}


def expand(X_l, Y_l, w_l, X_u, Q_u, w_u, min_q=0.05):
    """Soft-label replication: one copy of each row per combo with weight w * q (exact
    expected cross-entropy for LR).  Y_l may be one-hot or soft."""
    Xs, ys, ws = [], [], []
    for Xb, Yb, wb in ((X_l, Y_l, w_l), (X_u, Q_u, w_u)):
        if len(Xb) == 0:
            continue
        i, k = np.where(Yb >= min_q)
        Xs.append(Xb[i])
        ys.append(k)
        ws.append(wb[i] * Yb[i, k])
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ws)

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/labelers.py


## part1/research/other_combiner_v2/source/p1lib/pipeline.py

In [116]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/pipeline.py
"""Pipeline composition from config dicts.

A *single* pipeline config:
    {
      "features":   {"scale": True, "pca": 64, "l2": False, "blocks": {...}} | None,
      "synth":      [ {"type": "gmm", "k": 16}, {"type": "knn_vote", "k": 10}, ... ],
      "labeler":    {"type": "cluster"|"spread"|"cs"|"sld"|"selftrain", "lam": 0.1, ...} | None,
      "clf":        {"type": "lr"|"ovr"|"lda"|"knn"|"hgb"|"nb"|"direct", ...},
      "temperature": True,
    }
An *ensemble*:  {"ensemble": [single, single, ...], "weights": None | [w1, w2, ...]}
A *stack* is a single pipeline whose synth list holds {"type": "oof_model", "model": single}
entries (nested OOF meta-features) and whose features are None (or a small base block).

A full experiment config adds the head and the transductive toggle:
    {"model": <single|ensemble>, "head": {"prior": "none"|"sld", "sld_a": 100,
     "decode": "map"|"expf1"}, "transductive": True}

run_model(model_cfg, ctx, rows) -> {"P": calibrated combo probs for rows, "prior": pi_s, ...}
Only labels of ctx.train are ever read.
"""
from __future__ import annotations

import copy
import warnings

import numpy as np
from sklearn.model_selection import StratifiedKFold

from .classifiers import make_classifier
from .features import featurize
from .head import apply_temperature, fit_temperature


def _mask_absent(P, prior):
    """Combos with zero training mass keep probability exactly 0 (temperature/log clipping
    would otherwise resurrect them and SLD would then amplify them)."""
    present = np.asarray(prior) > 1e-6
    if present.all():
        return P
    P = np.where(present[None, :], P, 0.0)
    return P / np.maximum(P.sum(1, keepdims=True), 1e-300)
from .labelers import expand, pseudo_labels, soft_labels, soft_labels_train_oof
from .synth import build_synth_block

warnings.filterwarnings("ignore", message="The least populated class")


def inner_folds(y, n_splits=5, seed=0):
    n_splits = int(min(n_splits, max(2, np.bincount(y).max())))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(y)), y))


def _design(cfg, ctx):
    """Base features + synthetic features for every global row."""
    parts = []
    if cfg.get("features") is not None:
        parts.append(featurize(ctx, cfg["features"]))
    S = build_synth_block(cfg.get("synth"), ctx)
    if S is not None:
        parts.append(S)
    if not parts:
        raise ValueError("pipeline has neither base features nor synthetic features")
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def run_single(cfg, ctx, rows):
    rows = np.asarray(rows)
    D = ctx.D
    K, CM = D.K, D.CM
    y = ctx.y_train
    clf_cfg = cfg["clf"]
    info = {}

    # ---- direct labeler-as-predictor (cluster / spread / cs / sld) ----
    if clf_cfg["type"] == "direct":
        lcfg = clf_cfg["labeler"]
        P = soft_labels(lcfg, ctx, rows)
        prior = np.bincount(y, minlength=K) / len(y)
        T = 1.0
        if cfg.get("temperature", True):
            P_oof = soft_labels_train_oof(lcfg, ctx)
            T = fit_temperature(P_oof, y)
            P = apply_temperature(P, T)
        return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}

    X = _design(cfg, ctx)
    Ytr = np.eye(K)[y]
    # ---- labeler -> weighted pseudo rows from the unlabeled pool ----
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    if cfg.get("labeler"):
        pr, Q, w, linfo = pseudo_labels(cfg["labeler"], ctx, clf_cfg, X)
        info["labeler"] = linfo
    Xt, yt, wt = expand(X[ctx.train], Ytr, np.ones(len(y)), X[pr], Q, w)
    clf = make_classifier(clf_cfg, K, CM).fit(Xt, yt, wt)
    P = clf.predict_proba(X[rows])
    prior = clf.prior_
    T = 1.0
    if cfg.get("temperature", True):
        # classifier-stage OOF: pseudo rows fixed, inner folds over the labeled rows
        P_oof = np.zeros((len(y), K))
        for tr, te in inner_folds(y, 5, ctx.seed):
            Xi, yi, wi = expand(X[ctx.train[tr]], Ytr[tr], np.ones(len(tr)), X[pr], Q, w)
            m = make_classifier(clf_cfg, K, CM).fit(Xi, yi, wi)
            P_oof[te] = m.predict_proba(X[ctx.train[te]])
        T = fit_temperature(P_oof, y)
        P = apply_temperature(P, T)
    info["n_pseudo_rows"] = int(len(pr))
    return {"P": _mask_absent(P, prior), "prior": prior, "T": T, "info": info}


def run_model(model_cfg, ctx, rows):
    if "ensemble" in model_cfg:
        members = model_cfg["ensemble"]
        wts = np.asarray(model_cfg.get("weights") or [1.0] * len(members), float)
        wts = wts / wts.sum()
        outs = [run_single(m, ctx, rows) for m in members]
        P = sum(wi * o["P"] for wi, o in zip(wts, outs))
        prior = sum(wi * o["prior"] for wi, o in zip(wts, outs))
        return {"P": P, "prior": prior, "T": [o["T"] for o in outs], "info": {"members": [o["info"] for o in outs]},
                "member_P": [o["P"] for o in outs], "member_prior": [o["prior"] for o in outs]}
    return run_single(model_cfg, ctx, rows)


# ----------------------------------------------------------------------------- validation
def final_stage_singles(model_cfg):
    return model_cfg["ensemble"] if "ensemble" in model_cfg else [model_cfg]


def validate(cfg):
    """Enforce the 'only ONE prior correction' rule and basic sanity."""
    model = cfg["model"]
    head = cfg.get("head", {})
    corrections = set()
    for s in final_stage_singles(model):
        clf = s["clf"]
        if clf.get("class_weight") == "balanced":
            corrections.add("class_weight=balanced")
        lab = s.get("labeler") or {}
        if lab.get("type") == "selftrain" and lab.get("align", "none") != "none" and clf.get("class_weight") == "balanced":
            raise ValueError("self-training alignment + balanced class weights = double prior correction")
    if head.get("prior", "none") == "sld":
        corrections.add("sld")
    if len(corrections) > 1:
        raise ValueError(f"more than one prior correction: {sorted(corrections)}")
    return sorted(corrections)


def model_part(cfg):
    """Config without the post-hoc head (SLD / decoding) -> cache key for fold outputs."""
    return {"model": cfg["model"], "transductive": cfg.get("transductive", True)}


def deep(cfg):
    return copy.deepcopy(cfg)

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/pipeline.py


## part1/research/other_combiner_v2/source/p1lib/search.py

In [117]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/search.py
"""Optuna TPE search over the composed config space, robust objective, 1-SE selection and the
nested search-overfitting check.

Objective  J = 0.5 * A + 0.5 * B   (A = repeated stratified CV sample-F1, B = mean chronological
holdout sample-F1 over cuts 2004/2005/2006).  C (official val) is logged, never optimized.
Selection  = one-standard-error rule on J, then the SIMPLEST config (fewest active components),
then a guard: B must not be worse than the plain-LR baseline's B by more than one SE.
"""
from __future__ import annotations

import copy
import fcntl
import json
import os
import time
import traceback

import numpy as np

from .evaluate import evaluate, make_folds, fold_outputs, score, CHRONO_CUTS
from .pipeline import validate

GRAPH_SPACE = {"scale": True, "pca": 64, "l2": True}


# ============================================================================ search space
def suggest_config(t, allow_blocks=True, allow_stack=False, require_unsup=False):
    """Define-by-run space.  Returns a full experiment config."""
    pca = t.suggest_categorical("pca", [0, 32, 64, 96])
    l2 = t.suggest_categorical("l2", [False, True])
    feats = {"scale": True, "pca": pca or None, "l2": l2, "blocks": {}}
    if allow_blocks:
        if t.suggest_categorical("blk_criteria", [False, True]):
            feats["blocks"]["criteria"] = {"d": 16}
        if t.suggest_categorical("blk_drug", [False, True]):
            feats["blocks"]["pathways"] = {"d": 16, "max_level": 2}
            feats["blocks"]["drugmeta"] = {}

    prior = t.suggest_categorical("prior", ["none", "sld"])
    head = {"prior": prior, "decode": t.suggest_categorical("decode", ["map", "expf1"])}
    if prior == "sld":
        head["sld_a"] = t.suggest_categorical("sld_a", [0, 25, 100, 400])
        head["sld_floor"] = 0.25

    clf_t = t.suggest_categorical("clf", ["lr", "ovr", "lda", "knn"] if require_unsup else ["lr", "ovr", "lda", "knn", "cs"])
    temperature = t.suggest_categorical("temperature", [False, True])
    if clf_t == "cs":       # Correct & Smooth as the predictor (T2); base = powerset LR
        gk = t.suggest_categorical("cs_graph_k", [5, 10, 20])
        lcfg = {"type": "cs", "graph": {"space": GRAPH_SPACE, "k": gk, "s": 1.0, "sym": "max"},
                "a1": t.suggest_categorical("cs_a1", [0.0, 0.5, 0.8]),
                "a2": t.suggest_float("cs_a2", 0.1, 0.9),
                "base": {"type": "lr", "C": t.suggest_float("C", 0.01, 3.0, log=True)},
                "base_features": {"scale": True, "pca": pca or None}}
        model = {"features": None, "synth": [], "labeler": None,
                 "clf": {"type": "direct", "labeler": lcfg}, "temperature": temperature}
        return {"model": model, "head": head, "transductive": True}

    clf = {"type": clf_t}
    if clf_t in ("lr", "ovr"):
        clf["C"] = t.suggest_float("C", 0.01, 3.0, log=True)
    if clf_t == "knn":
        clf.update(k=t.suggest_categorical("knn_k", [3, 5, 10, 20]), weights="distance",
                   tau=t.suggest_categorical("knn_tau", [0.05, 0.1, 0.3]), beta=1.0)
    if prior != "sld" and clf_t in ("lr", "ovr", "lda"):
        if t.suggest_categorical("class_weight", ["none", "balanced"]) == "balanced":
            clf["class_weight"] = "balanced"

    synth = []
    if t.suggest_categorical("syn_gmm", [False, True]):
        synth.append({"type": "gmm", "k": t.suggest_categorical("syn_gmm_k", [8, 16, 32]), "cov": "diag",
                      "space": {"scale": True, "pca": 32, "l2": True}})
    if t.suggest_categorical("syn_kmeans", [False, True]):
        synth.append({"type": "kmeans", "k": t.suggest_categorical("syn_kmeans_k", [8, 16, 32]),
                      "out": t.suggest_categorical("syn_kmeans_out", ["dist", "soft"])})
    if t.suggest_categorical("syn_knn", [False, True]):
        synth.append({"type": "knn_vote", "k": t.suggest_categorical("syn_knn_k", [5, 10, 20])})
    if t.suggest_categorical("syn_graph", [False, True]):
        synth.append({"type": "graph", "mode": t.suggest_categorical("syn_graph_mode", ["spread", "cs"]),
                      "graph": {"space": GRAPH_SPACE, "k": 10, "s": 1.0, "sym": "max"},
                      "alpha": t.suggest_float("syn_graph_alpha", 0.2, 0.9)})
    if t.suggest_categorical("syn_clmap", [False, True]):
        synth.append({"type": "cluster_map", "algo": "kmeans", "k": t.suggest_categorical("syn_clmap_k", [8, 16, 32]), "m": 4})
    if t.suggest_categorical("syn_oof", [False, True]):
        synth.append({"type": "oof_model", "model": {"features": {"scale": True}, "clf": {"type": "lda"}, "temperature": True}})

    lab_t = t.suggest_categorical("labeler", ["cluster", "spread", "cs", "selftrain"] if require_unsup else
                                  ["none", "cluster", "spread", "cs", "sld", "selftrain"])
    labeler = None
    if lab_t != "none":
        lam = t.suggest_float("lam", 0.03, 0.5, log=True)
        if lab_t == "cluster":
            labeler = {"type": "cluster", "algo": t.suggest_categorical("cl_algo", ["kmeans", "gmm", "seeded_kmeans"]),
                       "k": t.suggest_categorical("cl_k", [8, 16, 24, 32, 48]), "m": t.suggest_categorical("cl_m", [1, 4]),
                       "lam": lam}
        elif lab_t in ("spread", "cs"):
            g = {"space": GRAPH_SPACE, "k": t.suggest_categorical("lab_graph_k", [5, 10, 20]), "s": 1.0, "sym": "max"}
            labeler = {"type": lab_t, "graph": g, "lam": lam}
            if lab_t == "spread":
                labeler["alpha"] = t.suggest_float("lab_alpha", 0.2, 0.9)
            else:
                labeler.update(a1=t.suggest_categorical("lab_a1", [0.0, 0.5, 0.8]), a2=t.suggest_float("lab_a2", 0.2, 0.9))
        elif lab_t == "sld":
            labeler = {"type": "sld", "base": {"type": "lr", "C": 0.1}, "a": t.suggest_categorical("lab_sld_a", [25, 100, 400]),
                       "lam": lam}
        elif lab_t == "selftrain":
            labeler = {"type": "selftrain", "lam": lam,
                       "members": t.suggest_categorical("st_members", ["lr", "lr+lda+cs"]).split("+"),
                       "rounds": t.suggest_categorical("st_rounds", [1, 2, 3]),
                       "tau": t.suggest_categorical("st_tau", [0.6, 0.75, 0.9]),
                       "align": t.suggest_categorical("st_align", ["none", "sld"]) if clf.get("class_weight") != "balanced" else "none",
                       "weighting": t.suggest_categorical("st_weighting", ["hard", "softmatch"]),
                       "ramp": t.suggest_categorical("st_ramp", [False, True]),
                       "cluster_filter": t.suggest_categorical("st_cluster_filter", [False, True])}
            if require_unsup and "cs" not in labeler["members"]:
                labeler["cluster_filter"] = True
        if lab_t != "selftrain" and t.suggest_categorical("lab_hard", [False, True]):
            labeler["hard"] = True
            labeler["tau"] = t.suggest_categorical("lab_tau", [0.5, 0.7, 0.9])
    model = {"features": feats, "synth": synth, "labeler": labeler, "clf": clf, "temperature": temperature}
    return {"model": model, "head": head, "transductive": True}


# ============================================================================ complexity
CLF_COST = {"lr": 0.0, "lda": 0.0, "ovr": 0.25, "nb": 0.25, "knn": 0.5, "direct": 1.0, "hgb": 2.0}


def complexity(cfg) -> float:
    """Number of active components (+ small costs for extra knobs).  Lower = simpler."""
    m = cfg["model"]
    singles = m["ensemble"] if "ensemble" in m else [m]
    c = 0.0
    for s in singles:
        c += len(s.get("synth") or [])
        for sy in s.get("synth") or []:
            if sy["type"] == "oof_model":
                c += complexity({"model": sy["model"], "head": {}})
        lab = s.get("labeler")
        if lab:
            c += 1.0 + (0.5 * (len(lab.get("members", [])) > 1) + 0.25 * (lab.get("rounds", 1) - 1)
                        if lab["type"] == "selftrain" else 0.0)
        f = s.get("features") or {}
        c += len(f.get("blocks") or {})
        clf = s["clf"]
        c += CLF_COST.get(clf["type"], 1.0)
        if clf.get("class_weight") == "balanced":
            c += 0.5
        if s.get("temperature"):
            c += 0.25
    if "ensemble" in m:
        c += len(singles) - 1
    if cfg.get("head", {}).get("prior") == "sld":
        c += 0.5
    return c


def unsup_labeling(cfg) -> bool:
    """Assignment rule: an UNSUPERVISED method must take part in filling the missing labels.
    True if the label-filling path uses clustering or a similarity graph: a cluster / label-
    spreading / C&S labeler, or guarded self-training with the K-means consistency filter or a
    C&S member, or a direct cluster/graph predictor."""
    m = cfg["model"]
    for s in (m["ensemble"] if "ensemble" in m else [m]):
        lab = s.get("labeler") or {}
        if lab.get("type") in ("cluster", "spread", "cs"):
            return True
        if lab.get("type") == "selftrain" and (lab.get("cluster_filter") or "cs" in lab.get("members", [])):
            return True
        if s["clf"]["type"] == "direct" and s["clf"]["labeler"]["type"] in ("cluster", "spread", "cs"):
            return True
    return False


# ============================================================================ objective
def append_jsonl(path, rec):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    line = json.dumps(rec, default=float) + "\n"
    with open(path, "a") as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        f.write(line)
        f.flush()
        fcntl.flock(f, fcntl.LOCK_UN)


def make_objective(D, log_path, R=4, rows=None, chrono_cuts=CHRONO_CUTS, prune_margin=0.05,
                   allow_blocks=True, cv_seed=0, tag="main", require_unsup=False):
    import optuna

    def objective(trial):
        cfg = suggest_config(trial, allow_blocks=allow_blocks, require_unsup=require_unsup)
        validate(cfg)
        t0 = time.time()
        # stage 1: chrono cuts + val + first CV repeat; prune clearly bad configs
        folds1 = make_folds(D, ("A", "B", "C"), R=R, seed=cv_seed, rows=rows, chrono_cuts=chrono_cuts)
        first = [f for f in folds1 if f["protocol"] != "A" or f["repeat"] == 0]
        rest = [f for f in folds1 if f["protocol"] == "A" and f["repeat"] > 0]
        try:
            outs1 = fold_outputs(cfg, D, first, n_jobs=1)
            r1 = score(cfg, D, first, outs1, bootstrap=False)
            j1 = r1.get("J", np.nan)
            trial.set_user_attr("J_stage1", float(j1))
            try:
                best = trial.study.best_value
            except ValueError:
                best = -np.inf
            if np.isfinite(best) and j1 < best - prune_margin:
                append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "pruned", "J_stage1": j1,
                                        "params": trial.params, "config": cfg, "secs": time.time() - t0})
                raise optuna.TrialPruned()
            outs2 = fold_outputs(cfg, D, rest, n_jobs=1)
            res = score(cfg, D, first + rest, outs1 + outs2, bootstrap=False)
        except optuna.TrialPruned:
            raise
        except Exception as e:
            append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "fail", "error": repr(e),
                                    "tb": traceback.format_exc()[-2000:], "params": trial.params, "config": cfg})
            raise
        cx = complexity(cfg)
        attrs = {"A": res["A"]["sample_f1"], "A_se": res["A"]["sample_f1_se"], "A_macro": res["A"]["macro_f1"],
                 "B": res["B"]["sample_f1"], "B_se": res["B"]["sample_f1_se"], "B_macro": res["B"]["macro_f1"],
                 "J": res["J"], "J_se": res["J_se"], "complexity": cx, "secs": time.time() - t0}
        if "B2005" in res:
            attrs["B2005"] = res["B2005"]["sample_f1"]
        if "C" in res:
            attrs.update(C=res["C"]["sample_f1"], C_se=res["C"]["sample_f1_se"], C_macro=res["C"]["macro_f1"])
        for k, v in attrs.items():
            trial.set_user_attr(k, float(v))
        trial.set_user_attr("config", json.dumps(cfg))
        append_jsonl(log_path, {"tag": tag, "trial": trial.number, "state": "complete", **attrs, "params": trial.params,
                                "config": cfg, "rowf1_A": res["rowf1_A"], "rowf1_B": res["rowf1_B"],
                                "rowf1_C": res.get("rowf1_C"), "B_ncuts": res.get("B_ncuts", 1)})
        return res["J"]

    return objective


def run_worker(storage_path, study_name, n_trials, log_path, seed, R=4, rows=None, chrono_cuts=CHRONO_CUTS,
               allow_blocks=True, tag="main", timeout=None, sampler_name="tpe", cv_seed=0, require_unsup=False):
    """One search worker process (shares the study through a journal file)."""
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    from .data import load
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    D = load()
    storage = JournalStorage(JournalFileBackend(storage_path))
    if sampler_name == "random":
        sampler = optuna.samplers.RandomSampler(seed=seed)
    else:
        sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True, group=True, constant_liar=True,
                                             n_startup_trials=40)
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize",
                                sampler=sampler, load_if_exists=True)
    obj = make_objective(D, log_path, R=R, rows=rows, chrono_cuts=chrono_cuts, allow_blocks=allow_blocks, tag=tag,
                         cv_seed=cv_seed, prune_margin=(np.inf if sampler_name == "random" else 0.05),
                         require_unsup=require_unsup)
    study.optimize(obj, n_trials=n_trials, timeout=timeout, catch=(Exception,))


def launch(storage_path, study_name, n_workers, trials_per_worker, log_path, R=4, rows=None,
           chrono_cuts=CHRONO_CUTS, allow_blocks=True, tag="main", timeout=None, base_seed=0, enqueue=None,
           sampler_name="tpe", cv_seed=0, require_unsup=False):
    """Run n_workers processes in parallel (spawn) against one journal-backed study."""
    import multiprocessing as mp
    import optuna
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend
    os.makedirs(os.path.dirname(storage_path), exist_ok=True)
    storage = JournalStorage(JournalFileBackend(storage_path))
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize", load_if_exists=True)
    for params in enqueue or []:
        study.enqueue_trial(params, skip_if_exists=True)
    ctx = mp.get_context("spawn")
    procs = []
    for w in range(n_workers):
        p = ctx.Process(target=run_worker, args=(storage_path, study_name, trials_per_worker, log_path,
                                                 base_seed + w, R, None if rows is None else list(map(int, rows)),
                                                 chrono_cuts, allow_blocks, tag, timeout, sampler_name, cv_seed,
                                                 require_unsup))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
    return study


# ============================================================================ selection
def load_log(log_path, tag=None, state="complete"):
    recs = []
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            if (tag is None or r.get("tag") == tag) and (state is None or r.get("state") == state):
                recs.append(r)
    return recs


def select_one_se(recs, baseline=None, key="J", se_key="J_se"):
    """1-SE rule: candidates with J >= J_best - SE(J_best); choose minimal complexity, ties by J.
    Guard: if a baseline record is given, the choice must have B >= B_base - B_se_base."""
    if not recs:
        return None, []
    best = max(recs, key=lambda r: r[key])
    thr = best[key] - best[se_key]
    cands = [r for r in recs if r[key] >= thr]
    cands.sort(key=lambda r: (r["complexity"], -r[key]))
    for r in cands:
        if baseline is None or r["B"] >= baseline["B"] - baseline["B_se"]:
            return r, cands
    return cands[0], cands


def dedupe(recs):
    """Keep one record per distinct config (TPE re-samples identical categorical configs)."""
    seen = {}
    for r in recs:
        k = json.dumps(r["config"], sort_keys=True)
        if k not in seen or r[ "J"] > seen[k]["J"]:
            seen[k] = r
    return list(seen.values())

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/search.py


## part1/research/other_combiner_v2/source/p1lib/submit.py

In [118]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/submit.py
"""Final refit on all 235 labeled rows, test predictions, Kaggle CSV + manifest, filled train labels."""
from __future__ import annotations

import datetime
import hashlib
import json
import os

import numpy as np
import pandas as pd

from .context import make_ctx
from .data import LABELS, write_submission
from .head import apply_head, decode, sld_em
from .pipeline import run_model, validate


def final_fit(cfg, D, n_seeds=1):
    """Refit on labeled-train + val (235 rows); return calibrated probs for test and the pool."""
    validate(cfg)
    test, unl = D.idx["test"], D.idx["unl"]
    rows = np.r_[test, unl]
    Ps, priors, Ts = [], [], []
    for s in range(n_seeds):
        ctx = make_ctx(D, D.idx["labval"], test, transductive=cfg.get("transductive", True), seed=s, name="final")
        o = run_model(cfg["model"], ctx, rows)
        Ps.append(o["P"]); priors.append(np.asarray(o["prior"])); Ts.append(o["T"])
    P = np.mean(Ps, axis=0)
    prior = np.mean(priors, axis=0)
    return {"P_test": P[: len(test)], "P_pool": P[len(test):], "prior": prior, "T": Ts}


def predict_test(cfg, D, fit):
    head = cfg.get("head", {})
    tr = cfg.get("transductive", True)
    target = None if tr else fit["P_pool"]          # inductive: SLD prior estimated on the unlabeled pool
    Y, Q, pi = apply_head(fit["P_test"], fit["prior"], D.CM, head, target)
    return Y, Q, pi


def fill_pool_labels(cfg, D, fit):
    """Hard labels for the 1,014 masked train rows from the refit model (SLD toward the pool's own
    prior when the head uses SLD)."""
    head = cfg.get("head", {})
    P = fit["P_pool"]
    if head.get("prior") == "sld":
        P, _ = sld_em(P, fit["prior"], a=head.get("sld_a", 0), floor=head.get("sld_floor", 0.25))
    return decode(P, D.CM, head.get("decode", "map")), P


def write_candidate(cfg, D, out_dir, name, scores=None, n_seeds=1, sample_path=None):
    os.makedirs(out_dir, exist_ok=True)
    fit = final_fit(cfg, D, n_seeds=n_seeds)
    Y, Q, pi = predict_test(cfg, D, fit)
    path = os.path.join(out_dir, f"{name}.csv")
    sub = write_submission(D, Y, path)
    if sample_path:                                  # exact format / order check vs the sample file
        sample = pd.read_csv(sample_path)
        assert list(sub.columns) == list(sample.columns) and (sub["ID"].values == sample["ID"].values).all()
    sha = hashlib.sha1(open(path, "rb").read()).hexdigest()
    manifest = {
        "name": name, "csv": os.path.abspath(path), "sha1": sha,
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "config": cfg, "transductive_test_features_used": bool(cfg.get("transductive", True)),
        "trained_on": "157 labeled train rows + 78 val rows (235); 1,014 unlabeled train rows as pool",
        "n_seeds": n_seeds,
        "scores": scores or {},
        "test_pred_label_rate": dict(zip(LABELS, Y.mean(0).round(3).tolist())),
        "test_pred_cardinality": float(Y.sum(1).mean()),
        "test_pred_combo_counts": {"+".join(LABELS[j] for j in range(5) if r[j]): int(c)
                                   for r, c in zip(*np.unique(Y, axis=0, return_counts=True))},
        "sld_test_prior_labels": None if pi is None else dict(zip(LABELS, (pi @ D.CM).round(3).tolist())),
        "train_prior_labels": dict(zip(LABELS, (fit["prior"] @ D.CM).round(3).tolist())),
        "temperature": fit["T"],
    }
    with open(os.path.join(out_dir, f"{name}.manifest.json"), "w") as f:
        json.dump(manifest, f, indent=1, default=float)
    return manifest, fit, Y


def write_filled_train(cfg, D, fit, path):
    Yp, P = fill_pool_labels(cfg, D, fit)
    tr = D.df.iloc[D.idx["train"]].copy()
    tr = tr.drop(columns=["split"])
    pos = {g: i for i, g in enumerate(D.idx["unl"])}
    lab = tr[LABELS].to_numpy().copy()
    src = np.array(["given"] * len(tr), dtype=object)
    for i, g in enumerate(D.idx["train"]):
        if g in pos:
            lab[i] = Yp[pos[g]]
            src[i] = "filled"
    tr[LABELS] = lab
    tr["label_source"] = src
    tr["fill_confidence"] = np.where(src == "filled", [P[pos[g]].max() if g in pos else 1.0 for g in D.idx["train"]], 1.0)
    tr.to_csv(path, index=False)
    return tr

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/submit.py


## part1/research/other_combiner_v2/source/p1lib/synth.py

In [119]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/synth.py
"""Synthetic features: unsupervised / semi-supervised model outputs turned into new columns
that downstream classifiers (or a stacker) consume.

    synth_features(scfg, ctx) -> (n_all, m) array for every global row

Label-free generators (fit on ctx.unsup features only, cached):
  gmm        : GMM posterior responsibilities (or log-responsibilities)
  kmeans     : distances to K-means centroids, or soft assignments softmax(-d^2/T)
Label-dependent generators (training rows get OUT-OF-FOLD values, so no row sees its own label):
  knn_vote   : cosine kNN label-vote fractions over the K combos (exact leave-one-out)
  graph      : label-spreading or C&S scores (inner K-fold for training rows)
  cluster_map: cluster -> label-combo mapped probabilities (exact leave-one-out)
  oof_model  : out-of-fold probabilities of any single pipeline config (inner K-fold = nested OOF)
Label-dependent outputs are returned as log-probabilities (K columns).
"""
from __future__ import annotations

import numpy as np

from .cache import memo, memo_ctx
from .classifiers import KNNClf
from .features import featurize
from .head import safe_log
from .labelers import DEFAULT_CLUSTER_SPACE, _cluster_fit, soft_labels, soft_labels_train_oof

LABEL_FREE = {"gmm", "kmeans"}


def _label_free(scfg, ctx):
    algo = scfg["type"]
    k = int(scfg.get("k", 16))
    space = scfg.get("space", DEFAULT_CLUSTER_SPACE)
    Z = featurize(ctx, space)
    key = {"algo": algo, "k": k, "space": space, **{kk: scfg[kk] for kk in ("cov", "n_init") if kk in scfg}}
    fit = memo("cluster", key, ctx.unsup, lambda: _cluster_fit(ctx.D, algo, k, Z, ctx.unsup, scfg))
    if algo == "gmm":
        R = fit["resp"]
        return np.log(np.clip(R, 1e-6, None)) if scfg.get("out", "resp") == "logresp" else R
    d = fit["dist"]
    if scfg.get("out", "dist") == "soft":
        d2 = d ** 2
        T = float(scfg.get("temp", 1.0)) * np.median(d2.min(1))
        L = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
        R = np.exp(L)
        return R / R.sum(1, keepdims=True)
    return d


def _knn_vote(scfg, ctx):
    K = ctx.D.K
    Z = featurize(ctx, scfg.get("space", {"scale": True, "l2": True}))
    y = ctx.y_train
    m = KNNClf({"type": "knn", "k": int(scfg.get("k", 10)), "weights": scfg.get("weights", "uniform"),
                "beta": float(scfg.get("beta", 1.0))}, K, ctx.D.CM).fit(Z[ctx.train], y)
    out = m.predict_proba(Z)
    out[ctx.train] = m.predict_proba(Z[ctx.train], exclude_self=True)
    return safe_log(out)


def _labeler_feature(lcfg, ctx):
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out = np.zeros((ctx.D.n, ctx.D.K))
    out[rows] = soft_labels(lcfg, ctx, rows)
    out[ctx.train] = soft_labels_train_oof(lcfg, ctx)
    return safe_log(out)


def _oof_model(scfg, ctx):
    from .pipeline import inner_folds, run_single
    sub = scfg["model"]
    y = ctx.y_train
    out = np.zeros((ctx.D.n, ctx.D.K))
    rows = np.setdiff1d(np.arange(ctx.D.n), ctx.train)
    out[rows] = run_single(sub, ctx, rows)["P"]
    for tr, te in inner_folds(y, int(scfg.get("n_inner", 5)), ctx.seed + 17):
        inner = ctx.sub(ctx.train[tr], ctx.train[te])
        out[ctx.train[te]] = run_single(sub, inner, ctx.train[te])["P"]
    return safe_log(out)


def synth_features(scfg, ctx) -> np.ndarray:
    if scfg["type"] in LABEL_FREE:
        return _synth(scfg, ctx)
    return memo_ctx("synth", scfg, ctx, lambda: _synth(scfg, ctx))


def _synth(scfg, ctx) -> np.ndarray:
    t = scfg["type"]
    if t in LABEL_FREE:
        return _label_free(scfg, ctx)
    if t == "knn_vote":
        return _knn_vote(scfg, ctx)
    if t == "graph":
        mode = scfg.get("mode", "spread")
        lcfg = {"type": "spread" if mode == "spread" else "cs", "graph": scfg.get("graph"),
                "alpha": scfg.get("alpha", 0.5), "a1": scfg.get("a1", 0.5), "a2": scfg.get("alpha", 0.5)}
        return _labeler_feature(lcfg, ctx)
    if t == "cluster_map":
        lcfg = {"type": "cluster", **{k: v for k, v in scfg.items() if k != "type"}}
        return _labeler_feature(lcfg, ctx)
    if t == "oof_model":
        return _oof_model(scfg, ctx)
    raise ValueError(t)


def build_synth_block(synth_list, ctx):
    """Concatenate all synthetic features and standardize them on train+pool rows."""
    if not synth_list:
        return None
    parts = [synth_features(s, ctx) for s in synth_list]
    S = np.hstack(parts)
    fit_rows = np.union1d(ctx.train, ctx.pool)
    mu = S[fit_rows].mean(0)
    sd = S[fit_rows].std(0)
    sd[sd < 1e-9] = 1.0
    return (S - mu) / sd

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/synth.py


## part1/research/other_combiner_v2/source/p1lib/viz.py

In [120]:
%%writefile _source_export/part1/research/other_combiner_v2/source/p1lib/viz.py
"""Shared matplotlib style for report figures (static PNG/PDF).

Categorical hues are assigned in a FIXED order (validated reference palette, light mode);
sequential = one blue ramp; thin marks, solid hairline grid, no dual axes.
"""
from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
TEXT2 = "#52514e"
GRID = "#e4e3df"
MUTED = "#a3a29c"

# fixed identity colors for recurring entities so a method keeps its hue across figures
PROTOCOL_COLORS = {"A": SERIES[0], "B": SERIES[1], "C": SERIES[2]}
LABEL_COLORS = dict(zip(["Safety", "Funding", "Efficacy", "Recruitment", "Other"], SERIES[:5]))


def setup():
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.grid": True, "grid.color": GRID,
        "grid.linewidth": 0.6, "grid.linestyle": "-", "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelcolor": TEXT2, "xtick.color": TEXT2, "ytick.color": TEXT2, "text.color": TEXT,
        "axes.titlesize": 11, "axes.titleweight": "semibold", "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.5, "legend.frameon": False,
        "lines.linewidth": 2.0, "lines.markersize": 6, "font.family": "DejaVu Sans",
        "axes.prop_cycle": mpl.cycler(color=SERIES), "figure.dpi": 110, "savefig.dpi": 200,
        "savefig.bbox": "tight",
    })


def save(fig, path):
    fig.savefig(path)
    if path.endswith(".png"):
        fig.savefig(path[:-4] + ".pdf")
    return path

Overwriting _source_export/part1/research/other_combiner_v2/source/p1lib/viz.py


## part1/research/other_combiner_v2/strict_adapter.py

In [121]:
%%writefile _source_export/part1/research/other_combiner_v2/strict_adapter.py
"""Inductive, group-safe adapters for the four fixed Dylan v2 candidates.

Original p1lib is copied unchanged under source/. This file changes fitting
boundaries and nested calibration, not classifier settings or selection weights.
"""
from dataclasses import replace
import hashlib

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

from p1lib import cache, features, graph, labelers
from p1lib.classifiers import make_classifier, KNNClf
from p1lib.context import Ctx
from p1lib.head import apply_temperature, fit_temperature, safe_log
from p1lib.pipeline import _mask_absent

AUDIT = []


def groups(D):
    if not hasattr(D, '_embedding_groups'):
        D._embedding_groups = pd.util.hash_pandas_object(
            D.df[[f'emb_{i}' for i in range(150)]].round(10), index=False).to_numpy()
    return D._embedding_groups


def context(D, train, query, pool, seed=0, name='', depth=0):
    train, query, pool = [np.asarray(v, dtype=int) for v in [train, query, pool]]
    unsup = np.union1d(train, pool)
    assert not np.intersect1d(unsup, query).size
    assert not np.intersect1d(unsup, D.idx['test']).size
    # The only accessible labels belong to the fitting gold rows.
    y = np.full_like(D.y, -1); y[train] = D.y[train]
    Y = np.full_like(D.Y, -1); Y[train] = D.Y[train]
    df = D.df.copy(); df.loc[:, ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']] = Y
    hidden = replace(D, df=df, y=y, Y=Y)
    hidden._embedding_groups = groups(D)
    return Ctx(hidden, train, query, pool, unsup, False, seed, name, depth)


def subcontext(ctx, train, query):
    g = groups(ctx.D)
    banned = g[query]
    train = np.asarray(train)[~np.isin(g[train], banned)]
    pool = ctx.pool[~np.isin(g[ctx.pool], banned)]
    sub = context(ctx.D, train, query, pool, ctx.seed, ctx.name + '/inner', ctx.depth + 1)
    assert not np.intersect1d(g[sub.unsup], g[query]).size
    assert np.isin(sub.unsup, ctx.unsup).all()
    AUDIT.append(dict(depth=sub.depth, gold=len(train), masked=len(pool), query=len(query),
                      fitting_ids_sha256=hashlib.sha256(sub.D.df.iloc[sub.unsup].ID.to_numpy(str).tobytes()).hexdigest(),
                      gold_group_overlap=0, masked_group_overlap=0))
    return sub


def inner_folds(ctx, n=5):
    _, inv = np.unique(groups(ctx.D)[ctx.train], return_inverse=True)
    target = np.zeros((inv.max() + 1, 5), dtype=int)
    np.maximum.at(target, inv, ctx.D.CM[ctx.y_train])
    splitter = MultilabelStratifiedKFold(n_splits=n, shuffle=True, random_state=ctx.seed)
    for tr, te in splitter.split(np.zeros((len(target), 1)), target):
        yield np.flatnonzero(np.isin(inv, tr)), np.flatnonzero(np.isin(inv, te))


def base_oof(ctx, base_cfg, feature_cfg):
    def compute():
        K, CM = ctx.D.K, ctx.D.CM
        X = features.featurize(ctx, feature_cfg)
        P_oof = np.zeros((len(ctx.train), K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            Xi = features.featurize(sub, feature_cfg)
            m = make_classifier(base_cfg, K, CM).fit(Xi[sub.train], sub.y_train)
            P_oof[te] = m.predict_proba(Xi[sub.eval])
        m = make_classifier(base_cfg, K, CM).fit(X[ctx.train], ctx.y_train)
        return P_oof, m.predict_proba(X)
    return cache.memo_ctx('strict_base_oof', [base_cfg, feature_cfg], ctx, compute)


def cs_probabilities(lcfg, ctx, rows):
    assert lcfg['type'] == 'cs'
    core = {k: v for k, v in lcfg.items() if k not in ('lam', 'tau', 'hard', 'pool')}
    def compute():
        oof, all_p = base_oof(ctx, lcfg.get('base', {'type': 'lr', 'C': .1}),
                            lcfg.get('base_features', {'scale': True}))
        return graph.correct_and_smooth(ctx, lcfg.get('graph'), float(lcfg.get('a1', .5)),
                float(lcfg.get('a2', .5)), ctx.train, ctx.y_train, oof, all_p, ctx.D.K, rows)
    return cache.memo_ctx('strict_cs', core, ctx, compute, extra=np.asarray(rows).tolist())


def knn_features(cfg, ctx):
    def compute():
        space = cfg.get('space', {'scale': True, 'l2': True})
        knn = dict(type='knn', k=int(cfg.get('k', 10)), weights=cfg.get('weights', 'uniform'),
                   beta=float(cfg.get('beta', 1)))
        Z = features.featurize(ctx, space)
        m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Z[ctx.train], ctx.y_train)
        out = m.predict_proba(Z)
        g = groups(ctx.D)
        # Preserve the source's leave-one-out idea, but leave out the entire
        # embedding group and refit scaling with that group's masked copies purged.
        for value in np.unique(g[ctx.train]):
            held = ctx.train[g[ctx.train] == value]
            train = ctx.train[g[ctx.train] != value]
            sub = subcontext(ctx, train, held)
            # Avoid caching one large transformed matrix per leave-group-out fit.
            Zi = features._embed(sub.D, features.norm_features_cfg(space), sub.unsup)
            m = KNNClf(knn, ctx.D.K, ctx.D.CM).fit(Zi[sub.train], sub.y_train)
            out[held] = m.predict_proba(Zi[held])
        return safe_log(out)
    return cache.memo_ctx('strict_knn_feature', cfg, ctx, compute)


def kmeans_features(cfg, ctx):
    assert cfg['type'] == 'kmeans' and cfg.get('out') == 'soft'
    space = cfg.get('space', labelers.DEFAULT_CLUSTER_SPACE)
    Z = features.featurize(ctx, space)
    key = dict(algo='kmeans', k=int(cfg.get('k', 16)), space=space,
               **{k: cfg[k] for k in ('cov', 'n_init') if k in cfg})
    fit = cache.memo('cluster', key, ctx.unsup,
                     lambda: labelers._cluster_fit(ctx.D, 'kmeans', key['k'], Z, ctx.unsup, cfg))
    d2 = fit['dist'] ** 2
    # Source uses all global rows for this median, including eval/test. Restrict it.
    T = float(cfg.get('temp', 1)) * np.median(d2[ctx.unsup].min(1))
    logp = -(d2 - d2.min(1, keepdims=True)) / max(T, 1e-12)
    result = np.exp(logp)
    return result / result.sum(1, keepdims=True)


def design(cfg, ctx):
    parts = []
    if cfg.get('features') is not None:
        assert not cfg['features'].get('blocks')
        parts.append(features.featurize(ctx, cfg['features']))
    synth = []
    for scfg in cfg.get('synth', []):
        if scfg['type'] == 'kmeans':
            synth.append(kmeans_features(scfg, ctx))
        elif scfg['type'] == 'knn_vote':
            synth.append(knn_features(scfg, ctx))
        else:
            raise ValueError(scfg)
    if synth:
        S = np.hstack(synth)
        mu, sd = S[ctx.unsup].mean(0), S[ctx.unsup].std(0)
        sd[sd < 1e-9] = 1
        parts.append((S - mu) / sd)
    return np.hstack(parts) if len(parts) > 1 else parts[0]


def uncalibrated(cfg, ctx, rows):
    K, CM, y = ctx.D.K, ctx.D.CM, ctx.y_train
    if cfg['clf']['type'] == 'direct':
        return dict(P=cs_probabilities(cfg['clf']['labeler'], ctx, rows),
                    prior=np.bincount(y, minlength=K) / len(y), info={})
    X = design(cfg, ctx)
    pr, Q, w = np.array([], int), np.zeros((0, K)), np.zeros(0)
    lcfg = cfg.get('labeler')
    if lcfg:
        assert lcfg.get('pool', 'unl') == 'unl'
        pr = ctx.pool
        Q = cs_probabilities(lcfg, ctx, pr)
        keep = Q.max(1) >= float(lcfg.get('tau', 0))
        if lcfg.get('hard', False):
            Q = np.eye(K)[Q.argmax(1)]
        pr, Q = pr[keep], Q[keep]
        w = np.full(len(pr), float(lcfg.get('lam', .1)))
    Xt, yt, wt = labelers.expand(X[ctx.train], np.eye(K)[y], np.ones(len(y)), X[pr], Q, w)
    m = make_classifier(cfg['clf'], K, CM).fit(Xt, yt, wt)
    return dict(P=m.predict_proba(X[rows]), prior=m.prior_, info={'n_pseudo_rows': len(pr)})


def run_single(cfg, ctx, rows):
    out = uncalibrated(cfg, ctx, rows)
    T = 1.
    if cfg.get('temperature', True):
        oof = np.zeros((len(ctx.train), ctx.D.K))
        for tr, te in inner_folds(ctx):
            sub = subcontext(ctx, ctx.train[tr], ctx.train[te])
            # Relearn features, pseudo labels and classifier inside each inner fold.
            oof[te] = uncalibrated(cfg, sub, sub.eval)['P']
        T = fit_temperature(oof, ctx.y_train)
        out['P'] = apply_temperature(out['P'], T)
    out['P'] = _mask_absent(out['P'], out['prior'])
    out['T'] = T
    return out


def run_model(cfg, ctx, rows):
    if 'ensemble' not in cfg:
        return run_single(cfg, ctx, rows)
    members = cfg['ensemble']
    weights = np.array(cfg.get('weights') or [1.] * len(members)); weights /= weights.sum()
    outs = [run_single(m, ctx, rows) for m in members]
    return dict(P=sum(w * o['P'] for w, o in zip(weights, outs)),
                prior=sum(w * o['prior'] for w, o in zip(weights, outs)),
                T=[o['T'] for o in outs], info={'members': [o['info'] for o in outs]})

Overwriting _source_export/part1/research/other_combiner_v2/strict_adapter.py


## part1/research/other_combiner_v2/study_models.py

In [122]:
%%writefile _source_export/part1/research/other_combiner_v2/study_models.py
"""Fixed Other specialists, nested residual combiners and weighted anchored LDA."""
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.covariance import ledoit_wolf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.svm import SVC
from anchored_solution import AnchoredReasonClassifier, select_reasons


def centered_logit(scores):
    return logit(np.clip(scores, 1e-6, 1-1e-6))-logit(.3)


def decide(scores):
    pred=(scores>=.5).astype(int)
    pred[np.arange(len(scores)), scores.argmax(1)]=1
    return pred


class OtherSpecialists:
    """Gold-only RBF heads; geometry uses eligible gold and masked inputs."""
    def fit(self, X, Y):
        gold=(Y>=0).all(1)
        self.scaler=StandardScaler().fit(X)
        Z=self.scaler.transform(X[gold]); y=Y[gold]
        self.models=[]
        for keep in [np.ones(len(y), bool), y[:,3]==0, y[:,3]==1]:
            target=y[keep,4]
            if len(np.unique(target))<2:
                self.models.append(float(2*target.mean()-1) if len(target) else -1.)
            else:
                self.models.append(SVC(C=1., kernel='rbf', gamma=1/150,
                    class_weight='balanced', probability=False, tol=1e-5).fit(Z[keep],target))
        return self

    def predict(self, X):
        Z=self.scaler.transform(X)
        return np.column_stack([np.full(len(X),m) if isinstance(m,float)
                                else m.decision_function(Z) for m in self.models])


def offset_fit(F, y, offset, penalty=20.):
    """Sum binary log loss + lambda/2 * squared coefficients, including bias."""
    def objective(b):
        z=offset+F@b
        loss=np.logaddexp(0,z).sum()-y@z+.5*penalty*(b@b)
        grad=F.T@(expit(z)-y)+penalty*b
        return loss,grad
    result=minimize(objective,np.zeros(F.shape[1]),jac=True,method='L-BFGS-B',
                    options=dict(maxiter=1000,ftol=1e-12,gtol=1e-8))
    if not result.success and np.max(np.abs(result.jac))>1e-5:
        raise RuntimeError(result.message)
    return result.x


def learned_heads(a,c,h,qa,qc,qh,y):
    """Train only on inner-OOF inputs. Evaluate on separate outer query scores."""
    b=.8*a+.2*c; qb=.8*qa+.2*qc
    o=centered_logit(b); qo=centered_logit(qb)
    baseline=expit(qo)
    z=np.clip(np.stack([centered_logit(a),centered_logit(c)],axis=2),-6,6)/3
    qz=np.clip(np.stack([centered_logit(qa),centered_logit(qc)],axis=2),-6,6)/3
    scores={}; coefficients={}
    for name in ['calibration_control','combiner_per_label']:
        out=baseline.copy(); bs=[]
        for j in range(5):
            if name=='calibration_control':
                F=np.c_[np.ones(len(y)),np.clip(o[:,j],-6,6)/3]
                Q=np.c_[np.ones(len(qa)),np.clip(qo[:,j],-6,6)/3]
            else:
                F=np.c_[np.ones(len(y)),z[:,j]]
                Q=np.c_[np.ones(len(qa)),qz[:,j]]
            coef=offset_fit(F,y[:,j],o[:,j]);bs.append(coef.tolist())
            out[:,j]=expit(qo[:,j]+Q@coef)
        scores[name]=out;coefficients[name]=bs
    scores['combiner_other_only']=baseline.copy()
    scores['combiner_other_only'][:,4]=scores['combiner_per_label'][:,4]
    for name,idx in [('other_rbf',[0]),('other_conditional_rbf',[1,2])]:
        mean=h[:,idx].mean(0);sd=np.maximum(h[:,idx].std(0),.1)
        F=np.clip((h[:,idx]-mean)/sd,-4,4)
        Q=np.clip((qh[:,idx]-mean)/sd,-4,4)
        if len(idx)==2:
            r=expit(o[:,3]);qr=expit(qo[:,3])
            F*=np.c_[1-r,r];Q*=np.c_[1-qr,qr]
        F=np.c_[np.ones(len(y)),F];Q=np.c_[np.ones(len(qa)),Q]
        coef=offset_fit(F,y[:,4],o[:,4])
        out=baseline.copy();out[:,4]=expit(qo[:,4]+Q@coef)
        scores[name]=out
        coefficients[name]=dict(beta=coef.tolist(),mean=mean.tolist(),std=sd.tolist())
    return scores,coefficients


class WeightedReasonLDA:
    """Uniform-prior weighted class means and standardized shrinkage covariances."""
    def fit(self,X,y,w):
        self.classes_=np.arange(5);means=[];cov=np.zeros((X.shape[1],X.shape[1]))
        for k in self.classes_:
            Xk=X[y==k];wk=w[y==k]
            mu=np.average(Xk,axis=0,weights=wk);means.append(mu)
            R=(Xk-mu)*np.sqrt(wk/wk.mean())[:,None]
            # RMS, not variance around the weighted residual array's unweighted mean.
            sd=np.sqrt(np.mean(R*R,axis=0));sd[sd<1e-12]=1
            C=ledoit_wolf(R/sd,assume_centered=True)[0]
            cov+=.2*(sd[:,None]*C*sd[None,:])
        means=np.array(means)
        self.coef_=np.linalg.lstsq(cov,means.T,rcond=None)[0].T
        self.intercept_=-.5*np.sum(means*self.coef_,axis=1)+np.log(.2)
        return self

    def predict_proba(self,X):
        z=X@self.coef_.T+self.intercept_;z-=z.max(1,keepdims=True)
        p=np.exp(z);return p/p.sum(1,keepdims=True)


class PseudoWeightedAnchored(AnchoredReasonClassifier):
    def __init__(self,mode):self.mode=mode

    def fit(self,X,Y):
        # Original input geometry, class balancing, neighbor bank and LR remain fixed.
        from sklearn.decomposition import PCA
        from sklearn.mixture import GaussianMixture
        X,Y=np.asarray(X,float),np.asarray(Y,int);known=(Y>=0).all(1)
        self.scaler_=StandardScaler().fit(X);Z=self.scaler_.transform(X)
        self.pca_=PCA(30,random_state=0).fit(Z)
        self.gmm_=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(self.pca_.transform(Z))
        assert self.gmm_.converged_
        K,V=Z[known],Z[~known];targets=Y[known]
        Rk=self.gmm_.predict_proba(self.pca_.transform(K));Ru=self.gmm_.predict_proba(self.pca_.transform(V))
        counts=(targets/targets.sum(1,keepdims=True)).sum(0)
        self.class_weights_=counts.mean()/counts
        Za,Ya,Ra=K,targets,Rk;confidence=np.ones_like(Ya,float)
        self.round_audit_=[]
        for step in range(3):
            row,reason=np.nonzero(Ya)
            weights=self.class_weights_[reason]/Ya.sum(1)[row]
            weights*=np.where(row<len(K),1.,.25)
            self.logistic_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(Za[row],reason,sample_weight=weights)
            if step==0 or self.mode=='control':
                self.lda_=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(5)/5).fit(Za[row],reason)
            else:
                lw=np.where(row<len(K),1.,.25)
                if self.mode=='pseudo_lda_confidence':lw*=confidence[row,reason]
                self.lda_=WeightedReasonLDA().fit(Za[row],reason,lw)
            self.neighbor_X_=normalize(Za);self.neighbor_Y_=Ya.copy()
            self.round_audit_.append(dict(step=step,gold=len(K),pseudo=len(Za)-len(K)))
            if step==2:break
            T=Ya/Ya.sum(1,keepdims=True)
            rates=(Ra.T@T+T.mean(0))/(Ra.sum(0)[:,None]+1)
            s=.95*self._scores(V)+.05*(Ru@rates)
            keep=s.max(1)>=.8;generated=select_reasons(s[keep])
            self.generated_mask_=keep;self.generated_labels_=generated
            confidence=np.vstack([np.ones_like(targets,float),
                (s[keep]/s[keep].max(1,keepdims=True))**2])
            Za=np.vstack([K,V[keep]]);Ya=np.vstack([targets,generated]);Ra=np.vstack([Rk,Ru[keep]])
        return self

Overwriting _source_export/part1/research/other_combiner_v2/study_models.py


## part1/research/other_combiner_v2/verify.py

In [123]:
%%writefile _source_export/part1/research/other_combiner_v2/verify.py
"""Independent rescore, fitting-boundary checks and frozen-parent comparison."""
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

sys.path.insert(0,str(Path(__file__).resolve().parent))
import run as study


def independent(y,p):
    tp=(y*p).sum(0);fp=((1-y)*p).sum(0);fn=(y*(1-p)).sum(0)
    precision=np.divide(tp,tp+fp,out=np.zeros(5),where=(tp+fp)>0)
    recall=np.divide(tp,tp+fn,out=np.zeros(5),where=(tp+fn)>0)
    f=np.divide(2*tp,2*tp+fp+fn,out=np.zeros(5),where=(2*tp+fp+fn)>0)
    sample=(2*(y*p).sum(1)/(y.sum(1)+p.sum(1))).mean()
    return sample,f.mean(),precision,recall,f,y.sum(0)


def verify(output):
    study.guard();output=Path(output)
    recorded=pd.read_csv(output/'summary/repeat_scores.csv')
    labels=pd.read_csv(output/'summary/per_label_scores.csv')
    pairs=pd.read_csv(output/'summary/paired_summary.csv')
    groups={};outer_count=0;inner_count=0;coefs=[]
    for c in study.cases():
        path=output/'predictions'/(c['key']+'.npz')
        z=study.loadnp(path);meta=study.read(path.with_suffix('.json'))
        assert study.sha(path)==meta['sha256']
        assert z['names'].tolist()==study.NAMES
        np.testing.assert_array_equal(z['ids'],c['query'].ID)
        np.testing.assert_array_equal(z['truth'],c['query'][study.LABELS])
        a,cc=study.refs(c);native=.8*a+.2*cc
        parent=(native>=.3).astype(int);parent[np.arange(len(native)),native.argmax(1)]=1
        np.testing.assert_array_equal(parent,z['predictions'][0])
        assert np.isfinite(z['scores']).all() and (z['predictions'].sum(2)>0).all()
        assert meta['fitting_ids_sha256']==study.idhash(c['fit'].ID)
        assert set(c['fit'].ID).isdisjoint(c['query'].ID)
        if c['key']!='validation':
            assert not set(study.benchmark.groups(c['fit']))&set(study.benchmark.groups(c['query']))
        gold=c['fit'][c['fit'][study.LABELS].ge(0).all(axis=1)]
        held=[]
        for inner in meta['inner']:
            fitted=c['fit'].set_index('ID',drop=False).loc[inner['train_ids']]
            query=gold.set_index('ID',drop=False).loc[inner['query_ids']]
            assert set(fitted.ID)<=set(c['fit'].ID)
            assert set(fitted.ID).isdisjoint(c['query'].ID)
            assert set(fitted.ID).isdisjoint(query.ID)
            assert not set(study.benchmark.groups(fitted))&set(study.benchmark.groups(query))
            assert set(fitted[study.LABELS].min(axis=1).unique())<=set([-100,0,1])
            held+=inner['query_ids'];inner_count+=1
        assert sorted(held)==sorted(gold.ID) and len(held)==len(set(held))
        np.testing.assert_array_equal(z['inner_gold_ids'],gold.ID)
        for n in ['other_rbf','other_conditional_rbf','combiner_other_only']:
            j=study.NAMES.index(n)
            np.testing.assert_array_equal(z['scores'][j,:,:4],z['scores'][0,:,:4])
        # Reconstruct every fitted meta-head from saved genuine inner-OOF predictions.
        heads,coefficients=study.learned_heads(z['inner_anchored'],z['inner_cand2'],z['inner_specialists'],
            a,cc,z['outer_specialists'],gold[study.LABELS].to_numpy(int))
        assert coefficients==meta['coefficients']
        for name,s in heads.items():np.testing.assert_allclose(s,z['scores'][study.NAMES.index(name)],atol=1e-12,rtol=0)
        for name,coef in coefficients.items():
            coefs.append(dict(case=c['key'],pool=c['pool'],recipe=name,coefficients=coef))
        key=(c['pool'],c['seed']) if c['pool']!='supplementary' else (c['key'],-1)
        groups.setdefault(key,[]).append(z)
        if c['key'].startswith('through_'):groups.setdefault(('chronology98',-1),[]).append(z)
        outer_count+=1
    checked=0
    for (pool,seed),zs in groups.items():
        y=np.concatenate([z['truth'] for z in zs]);ids=np.concatenate([z['ids'] for z in zs])
        assert len(ids)==len(set(ids))
        if pool in ['train','pooled']:assert len(zs)==5 and len(y)==(157 if pool=='train' else 235)
        for j,name in enumerate(study.NAMES):
            p=np.concatenate([z['predictions'][j] for z in zs])
            sample,macro,pr,re,f,support=independent(y,p)
            saved=recorded[(recorded.pool==pool)&(recorded.seed==seed)&(recorded.recipe==name)].iloc[0]
            np.testing.assert_allclose([sample,macro],[saved.sample_f1,saved.macro_f1],atol=1e-12,rtol=0)
            lab=labels[(labels.pool==pool)&(labels.seed==seed)&(labels.recipe==name)].set_index('label').loc[study.LABELS]
            np.testing.assert_allclose(np.c_[pr,re,f,support],lab[['precision','recall','f1','support']],atol=1e-12,rtol=0)
            checked+=1
    for _,r in pairs.iterrows():
        d=recorded[recorded.pool==r.pool].pivot(index='seed',columns='recipe',values='sample_f1')
        v=d[r.recipe]-d[r.control]
        np.testing.assert_allclose(v.mean(),r.mean_delta,atol=1e-12,rtol=0)
        assert int((v>1e-12).sum())==r.wins and int((v.abs()<=1e-12).sum())==r.ties
    expected={'train':.935244161358811,'pooled':.9216312056737589,'validation':.8910256410256411,'chronology98':.8877551020408163}
    for pool,value in expected.items():
        np.testing.assert_allclose(recorded[(recorded.pool==pool)&(recorded.recipe=='parent')].sample_f1.mean(),value,atol=1e-12,rtol=0)
    study.save(output/'summary/coefficients.json',coefs)
    result=dict(passed=True,outer_cases=outer_count,inner_folds=inner_count,metric_rows_checked=checked,
        independent_sample_and_per_label_arithmetic=True,parent_four_headlines_reproduced=True,
        ids_groups_and_nested_coverage_checked=True,all_learned_heads_reconstructed=True,
        all_source_hashes_unchanged=True,protocol_sha256=study.sha(study.HERE/'protocol.json'))
    study.save(output/'summary/verification.json',result)
    print(json.dumps(result,indent=2))


if __name__=='__main__':verify(Path(sys.argv[1]) if len(sys.argv)>1 else study.HERE/'results')

Overwriting _source_export/part1/research/other_combiner_v2/verify.py


## part1/research/other_combiner_v2/verify_retained.py

In [124]:
%%writefile _source_export/part1/research/other_combiner_v2/verify_retained.py
"""Rescore compact useful comparisons after unsuccessful fold-cache cleanup."""
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
import run as study
spec=importlib.util.spec_from_file_location('this_study_full_check',study.HERE/'verify.py')
checker=importlib.util.module_from_spec(spec);spec.loader.exec_module(checker)
independent=checker.independent


def main():
    study.guard();manifest=study.read(study.HERE/'retention.json')
    for relative,digest in manifest['retained_sha256'].items():
        assert study.sha(study.HERE/relative)==digest,relative
    z=study.loadnp(study.HERE/'results/retained_comparison.npz')
    scores=pd.read_csv(study.HERE/'results/summary/repeat_scores.csv')
    labels=pd.read_csv(study.HERE/'results/summary/per_label_scores.csv')
    groups=[(pool,int(seed),(z['pool']==pool)&(z['seed']==seed))
            for pool,seed in sorted(set(zip(z['pool'],z['seed'])))]
    groups.append(('chronology98',-1,np.char.startswith(z['pool'],'through_')))
    checks=0
    for pool,seed,keep in groups:
        y=z['truth'][keep]
        assert len(set(z['ids'][keep]))==len(y)
        for j,name in enumerate(z['names']):
            p=z['predictions'][j,keep]
            np.testing.assert_array_equal(p,study.decide(z['scores'][j,keep]))
            sample,macro,precision,recall,f,support=independent(y,p)
            r=scores[(scores.pool==pool)&(scores.seed==seed)&(scores.recipe==name)].iloc[0]
            np.testing.assert_allclose([sample,macro],[r.sample_f1,r.macro_f1],atol=1e-12,rtol=0)
            l=labels[(labels.pool==pool)&(labels.seed==seed)&(labels.recipe==name)].set_index('label').loc[study.LABELS]
            np.testing.assert_allclose(np.c_[precision,recall,f,support],l[['precision','recall','f1','support']],atol=1e-12,rtol=0)
            checks+=1
    assert checks==63
    assert not any(study.read(study.HERE/'results/summary/qualification.json').values())
    print(dict(passed=True,retained_metric_rows_independently_rescored=checks,
               all_seven_compact_result_hashes_verified=True,
               full_189_metric_checks_and_nested_audits='Recorded before cleanup in verification.json'))


if __name__=='__main__':main()

Overwriting _source_export/part1/research/other_combiner_v2/verify_retained.py


## part1/research/partial_secondary_v1/check_math.py

In [125]:
%%writefile _source_export/part1/research/partial_secondary_v1/check_math.py
"""Independent mathematical controls for weighted LDA and unknown neighbor labels."""
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from study_run import HERE,save
from partial_model import BinaryOtherLDA,observed_other,other_neighbor_vote


def main():
    rng=np.random.default_rng(81);X=rng.normal(size=(80,12));y=np.repeat([0,1],[50,30]);Q=rng.normal(size=(21,12))
    m=BinaryOtherLDA().fit(X,y,np.ones(80))
    ref=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.array([.8,.2])).fit(X,y)
    error=float(np.abs(m.predict_proba(Q)-ref.predict_proba(Q)).max())
    np.testing.assert_allclose(m.predict_proba(Q),ref.predict_proba(Q),atol=1e-9,rtol=0)
    bank=np.array([[.8,.6],[0,1],[1,0]],float);labels=np.array([[0,0,0,1,1],[0,0,0,1,0],[0,0,0,1,0]])
    obs=observed_other(labels,2,True);np.testing.assert_array_equal(obs,[True,True,False])
    partial=other_neighbor_vote(np.array([[1.,0]]),bank,labels,obs).item()
    complete=other_neighbor_vote(np.array([[1.,0]]),bank,labels,np.ones(3,bool)).item()
    np.testing.assert_allclose(partial,1.,atol=1e-8)
    np.testing.assert_allclose(complete,.64/(1+.64),atol=1e-8)
    # Marking a selected pseudo Other positive must restore it as observed.
    labels[2,4]=1;assert observed_other(labels,2,True).all()
    result=dict(passed=True,unit_weight_lda_max_error=error,unknown_neighbor_does_not_vote_zero=True,
        gold_negatives_preserved=True,selected_pseudo_positive_observed=True,partial_vote=partial,complete_vote=complete)
    save(HERE/'math_check.json',result);print(result)


if __name__=='__main__':main()

Overwriting _source_export/part1/research/partial_secondary_v1/check_math.py


## part1/research/partial_secondary_v1/evidence_partial.py

In [126]:
%%writefile _source_export/part1/research/partial_secondary_v1/evidence_partial.py
"""Retain useful matched comparisons, then independently rescore their evidence."""
import sys
import shutil
import numpy as np
import pandas as pd
import study_run as study
from verify_partial import metrics


def verify():
    study.guard();here=study.HERE;manifest=study.read(here/'retention.json')
    for name,digest in manifest['hashes'].items():assert study.sha(here/name)==digest,name
    z=study.loadnp(here/'results/retained_comparison.npz')
    scores=pd.read_csv(here/'results/summary/repeat_scores.csv')
    labels=pd.read_csv(here/'results/summary/per_label_scores.csv')
    groups=[(p,int(s),(z['pool']==p)&(z['seed']==s)) for p,s in sorted(set(zip(z['pool'],z['seed'])))]
    groups.append(('chronology98',-1,np.char.startswith(z['pool'],'through_')))
    n=0
    for pool,seed,keep in groups:
        y=z['truth'][keep];assert len(set(z['ids'][keep]))==len(y)
        for j,name in enumerate(z['names']):
            p=z['predictions'][j,keep]
            np.testing.assert_array_equal(p,study.select_reasons(z['scores'][j,keep]))
            s,m,pr,re,f,support=metrics(y,p)
            r=scores[(scores.pool==pool)&(scores.seed==seed)&(scores.recipe==name)].iloc[0]
            np.testing.assert_allclose([s,m],[r.sample_f1,r.macro_f1],atol=1e-12,rtol=0)
            l=labels[(labels.pool==pool)&(labels.seed==seed)&(labels.recipe==name)].set_index('label').loc[study.LABELS]
            np.testing.assert_allclose(np.c_[pr,re,f,support],l[['precision','recall','f1','support']],atol=1e-12,rtol=0)
            n+=1
    assert n==63 and not any(study.read(here/'results/summary/qualification.json').values())
    print(dict(passed=True,retained_metric_records_rescored=n,all_five_variants_summary_hashes_verified=True,
        full_checks_before_cleanup=manifest['verification_before_cleanup']['metric_rows_independently_checked']))


def retain():
    here=study.HERE;out=here/'results';receipt=study.read(out/'summary/verification.json')
    assert receipt['passed'] and receipt['outer_cases']==84
    names=['parent','binary_complete','binary_unknown'];index=[study.NAMES.index(n) for n in names]
    zs=[];meta=[]
    for c in study.cases():
        path=out/'predictions'/(c['key']+'.npz');z=study.loadnp(path);audit=study.read(path.with_suffix('.json'))
        assert study.sha(path)==audit['sha256'];zs.append(z)
        pool=c['pool'] if c['pool']!='supplementary' else c['key']
        meta.append(dict(pool=np.full(len(z['ids']),pool),seed=np.full(len(z['ids']),c['seed']),case=np.full(len(z['ids']),c['key'])))
    arrays={k:np.concatenate([z[k] for z in zs]) for k in ['ids','truth']}
    arrays.update({k:np.concatenate([v[k] for v in meta]) for k in ['pool','seed','case']})
    arrays.update(names=np.array(names),scores=np.concatenate([z['scores'][index] for z in zs],axis=1),
        predictions=np.concatenate([z['predictions'][index] for z in zs],axis=1))
    np.savez_compressed(out/'retained_comparison.npz',**arrays)
    files=list((out/'summary').glob('*'))+[out/'retained_comparison.npz',here/'math_check.json']
    study.save(here/'retention.json',dict(kept_recipes=names,
        reason='Useful matched comparison isolates unknown labels from binary architecture and documents opposite CV/chronology behavior.',
        removed='84 per-case prediction packets and84 JSON fit audits after checking all summaries. No fitted estimators or generated datasets were saved. Aggregate fit-count audits remain.',
        verification_before_cleanup=receipt,hashes={str(p.relative_to(here)):study.sha(p) for p in files}))
    verify()
    shutil.rmtree(out/'predictions')
    print('Removed verified unsuccessful fold caches.')


if __name__=='__main__':
    assert len(sys.argv)==2 and sys.argv[1] in ['retain','verify']
    retain() if sys.argv[1]=='retain' else verify()

Overwriting _source_export/part1/research/partial_secondary_v1/evidence_partial.py


## part1/research/partial_secondary_v1/partial_model.py

In [127]:
%%writefile _source_export/part1/research/partial_secondary_v1/partial_model.py
"""Partial Other labels: an unselected pseudo-label need not be a negative."""
import numpy as np
from sklearn.covariance import ledoit_wolf
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler,normalize
from anchored_solution import AnchoredReasonClassifier,select_reasons


def observed_other(labels,n_gold,partial):
    """Gold labels are complete; absent pseudo Other is unknown in partial mode."""
    return np.ones(len(labels),bool) if not partial else (np.arange(len(labels))<n_gold)|(labels[:,4]==1)


def other_neighbor_vote(Z,bank,labels,observed):
    # Retrieve ten *observed* Other labels; unknown entries neither vote zero nor
    # crowd an observed neighbor out. Gold negatives remain eligible.
    bank=bank[observed];target=labels[observed,4]
    similarity=normalize(Z)@bank.T
    nearest=np.argsort(-similarity,axis=1)[:,:10]
    w=np.take_along_axis(similarity,nearest,axis=1).clip(0)**2
    return (w*target[nearest]).sum(1)/(w.sum(1)+1e-9)


class BinaryOtherLDA:
    """Two Gaussian classes; fixed .8/.2 priors and weighted shrinkage covariance."""
    def fit(self,X,y,w):
        prior=np.array([.8,.2]);means=[];cov=np.zeros((X.shape[1],X.shape[1]))
        for k in [0,1]:
            x=X[y==k];wk=w[y==k];assert len(x)>1 and wk.sum()>0
            mu=np.average(x,axis=0,weights=wk);means.append(mu)
            residual=(x-mu)*np.sqrt(wk/wk.mean())[:,None]
            sd=np.sqrt(np.mean(residual**2,axis=0));sd[sd<1e-12]=1
            C=ledoit_wolf(residual/sd,assume_centered=True)[0]
            cov+=prior[k]*(sd[:,None]*C*sd[None,:])
        means=np.asarray(means)
        self.coef_=np.linalg.lstsq(cov,means.T,rcond=None)[0].T
        self.intercept_=-.5*(self.coef_*means).sum(1)+np.log(prior)
        return self

    def predict_proba(self,X):
        s=X@self.coef_.T+self.intercept_;s-=s.max(1,keepdims=True)
        p=np.exp(s);return p/p.sum(1,keepdims=True)


class PartialOtherAnchored(AnchoredReasonClassifier):
    def __init__(self,mode):
        assert mode in ['control','neighbor_unknown','binary_complete','binary_unknown']
        self.mode=mode;self.binary=mode.startswith('binary_')
        self.partial=mode in ['neighbor_unknown','binary_unknown']

    def fit(self,X,Y):
        X,Y=np.asarray(X,float),np.asarray(Y,int)
        known=np.isin(Y,[0,1]).all(1);unknown=(Y==-100).all(1)
        assert (known|unknown).all() and (Y[known].sum(0)>0).all()
        self.n_gold_=int(known.sum());self.gold_=Y[known].copy()
        self.scaler_=StandardScaler().fit(X);Z=self.scaler_.transform(X)
        self.pca_=PCA(30,random_state=0).fit(Z)
        self.gmm_=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(self.pca_.transform(Z))
        assert self.gmm_.converged_
        K,V=Z[known],Z[unknown];targets=Y[known]
        Rk=self.gmm_.predict_proba(self.pca_.transform(K));Ru=self.gmm_.predict_proba(self.pca_.transform(V))
        counts=(targets/targets.sum(1,keepdims=True)).sum(0);self.class_weights_=counts.mean()/counts
        Za,Ya,Ra=K,targets,Rk;self.round_audit_=[]
        for step in range(3):
            np.testing.assert_array_equal(Ya[:self.n_gold_],self.gold_)
            row,reason=np.nonzero(Ya)
            weights=self.class_weights_[reason]/Ya.sum(1)[row]
            weights*=np.where(row<self.n_gold_,1.,.25)
            self.logistic_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(Za[row],reason,sample_weight=weights)
            self.lda_=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(5)/5).fit(Za[row],reason)
            self.neighbor_X_=normalize(Za);self.neighbor_Y_=Ya.copy()
            self.other_observed_=observed_other(Ya,self.n_gold_,self.partial)
            audit=dict(step=step,gold=self.n_gold_,generated=len(Za)-self.n_gold_,
                generated_multilabel=int((Ya[self.n_gold_:].sum(1)>1).sum()),
                observed_other_positive=int(((Ya[:,4]==1)&self.other_observed_).sum()),
                observed_other_negative=int(((Ya[:,4]==0)&self.other_observed_).sum()),
                unknown_other=int((~self.other_observed_).sum()))
            if self.partial:
                assert audit['observed_other_negative']==int((targets[:,4]==0).sum())
                assert self.other_observed_[:self.n_gold_].all()
            if self.binary:
                obs=self.other_observed_;yy=Ya[obs,4];xx=Za[obs]
                raw=np.where(np.arange(len(Za))[obs]<self.n_gold_,1.,.25)
                mass=np.bincount(yy,weights=raw,minlength=2)
                # Hold class prior and total training weight fixed when dropping
                # unknown negatives; otherwise positive-only selection changes prior.
                w=raw*(self.n_gold_*np.array([.8,.2])/mass)[yy]
                np.testing.assert_allclose(np.bincount(yy,weights=w,minlength=2),self.n_gold_*np.array([.8,.2]),atol=1e-10)
                self.other_lr_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(xx,yy,sample_weight=w)
                self.other_lda_=BinaryOtherLDA().fit(xx,yy,raw)
                audit['binary_training_rows']=len(yy)
                audit['binary_weight_by_class']=np.bincount(yy,weights=w,minlength=2).tolist()
            self.round_audit_.append(audit)
            if step==2:break
            T=Ya/Ya.sum(1,keepdims=True)
            rates=(Ra.T@T+T.mean(0))/(Ra.sum(0)[:,None]+1)
            if self.mode=='binary_unknown':
                o=self.other_observed_
                rates[:,4]=(Ra[o].T@T[o,4]+T[o,4].mean())/(Ra[o].sum(0)+1)
            s=.95*self._scores(V)+.05*(Ru@rates)
            keep=s.max(1)>=.8;generated=select_reasons(s[keep])
            self.generated_mask_=keep;self.generated_labels_=generated
            Za=np.vstack([K,V[keep]]);Ya=np.vstack([targets,generated]);Ra=np.vstack([Rk,Ru[keep]])
        return self

    def _scores(self,Z):
        logistic=np.zeros((len(Z),5));logistic[:,self.logistic_.classes_]=self.logistic_.predict_proba(Z)
        similarity=normalize(Z)@self.neighbor_X_.T
        neighbors=np.argsort(-similarity,axis=1)[:,:10]
        weights=np.take_along_axis(similarity,neighbors,axis=1).clip(0)**2
        local=(weights[:,:,None]*self.neighbor_Y_[neighbors]).sum(1)
        local/=weights.sum(1,keepdims=True)+1e-9
        lda=np.zeros_like(logistic);lda[:,self.lda_.classes_]=self.lda_.predict_proba(Z)
        if self.partial:
            local[:,4]=other_neighbor_vote(Z,self.neighbor_X_,self.neighbor_Y_,self.other_observed_)
        if self.binary:
            logistic[:,4]=self.other_lr_.predict_proba(Z)[:,1]
            lda[:,4]=self.other_lda_.predict_proba(Z)[:,1]
        return .5*logistic+.3*local+.2*lda

Overwriting _source_export/part1/research/partial_secondary_v1/partial_model.py


## part1/research/partial_secondary_v1/study_run.py

In [128]:
%%writefile _source_export/part1/research/partial_secondary_v1/study_run.py
"""Register and evaluate partial secondary labels on all saved grouped cases."""
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[v]='1'
import argparse
from concurrent.futures import ProcessPoolExecutor,as_completed
from datetime import datetime,timezone
import hashlib
import importlib.util
import json
from pathlib import Path
import platform
import sys
import time
import warnings

HERE=Path(__file__).resolve().parent;P1=HERE.parents[1];ROOT=P1.parent
HELPER=P1/'research/anchored_transductive_v1'
sys.path[:0]=[str(HERE),str(P1),str(HELPER)]
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score,precision_recall_fscore_support
from threadpoolctl import threadpool_limits
from anchored_solution import AnchoredReasonClassifier,select_reasons
from partial_model import PartialOtherAnchored
spec=importlib.util.spec_from_file_location('_partial_saved_benchmark',HELPER/'run.py')
benchmark=importlib.util.module_from_spec(spec);spec.loader.exec_module(benchmark)
LABELS=benchmark.LABELS;EMB=benchmark.FEATURES
CANDIDATES=['neighbor_unknown','binary_complete','binary_unknown','binary_complete_half','binary_unknown_half']
NAMES=['parent','anchored']+CANDIDATES


def read(p):return json.loads(Path(p).read_text())
def save(p,x):Path(p).write_text(json.dumps(x,indent=2,allow_nan=False)+'\n')
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def idhash(v):return hashlib.sha256('\n'.join(v).encode()).hexdigest()
def loadnp(p):
    with np.load(p,allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}


def cases():
    result=benchmark.cases()
    for c in result:
        for key in ['fit','query']:c[key]=c[key][['ID','start_year']+EMB+LABELS].copy()
        c['reference_key']='strict_inductive__'+('grouped_' if c['pool']!='supplementary' else '')+c['key']
    return result


def reference(c):
    z=loadnp(HERE/'parent_inputs.npz');keep=z['case']==c['reference_key']
    np.testing.assert_array_equal(z['ids'][keep],c['query'].ID)
    np.testing.assert_array_equal(z['truth'][keep],c['query'][LABELS])
    return z['anchored'][keep],z['cand2'][keep]


def sources():
    files=[HERE/n for n in ['study_run.py','partial_model.py','check_math.py','parent_inputs.npz',
        'parent_reported_result.json','provenance.json']]
    files+=[P1/'anchored_solution.py',P1/'TESTING_POLICY.md',HELPER/'run.py',HELPER/'models.py']
    files+=list((P1/'data').glob('*.csv'))
    files+=[P1/'experiments/current_leaders_grouped_v1/snapshots'/f'{name}.json'
        for name in ['train3','train5','pooled3','pooled5','chronology']]
    return {str(p.relative_to(ROOT)):sha(p) for p in sorted(files)}


def register():
    p=HERE/'protocol.json';assert not p.exists()
    assert len(cases())==84
    save(p,dict(version='partial-secondary-v1',registered_at=datetime.now(timezone.utc).isoformat(),
        base_main='766d16d3bc74248cb7d8f4c1e7240094ebd5a55e',
        parent='Scored .89935 arithmetic anchored80/Cand2_inductive5_20. Original anchored .89636 is secondary control.',
        hypothesis='Confident primary pseudo-labels may incorrectly imply absent Other. Keep unselected Other unknown, with gold negatives still observed. Targets the rare Recruitment+Other combination without changing supplied labels.',
        motivation='Prior full235 anchored fit generated 757 pseudo rows:756 single-label and1 two-label. Previous corrections/specialists failed the complete rule. This is a distinct hypothesis, not proof those pseudo labels are wrong.',
        candidates={
            'neighbor_unknown':'Only Other neighbor voting changes. Pseudo Other=0 is unknown and excluded from Other neighbor retrieval; gold0/1 and pseudo Other1 remain observed. Retrieve ten eligible Other neighbors. Original multiclass LR/LDA and cluster teacher unchanged. Relearn pseudo labels using altered neighbor votes each round.',
            'binary_complete':'Architectural control. Replace only the Other output of LR/LDA with binary observed-label classifiers; all pseudo Other0/1 remain complete. Neighbor and cluster mapping retain complete labels.',
            'binary_unknown':'Matched to binary_complete: exclude pseudo Other0 from binary Other LR/LDA fits, Other neighbor retrieval, and Other cluster-rate numerator/denominator/prior. Supplied negatives remain observed. Other four class outputs use original multiclass LR/LDA/votes; they can change through regenerated pseudo labels.',
            'binary_complete_half':'Fixed conservative architectural control: .4 original anchored +.4 binary_complete anchored +.2 unchanged Cand2.',
            'binary_unknown_half':'Fixed conservative partial-label variant: .4 original anchored +.4 binary_unknown anchored +.2 unchanged Cand2.'},
        binary_settings='Other LR C.1,max_iter3000,tol1e-4. Raw gold weight1,pseudo.25. Rebalance observed class-weight masses to .8 negative/.2 positive and total gold-row count each fit, preventing removal of unknown negatives from changing effective prior/regularization mass. Other LDA uses raw1/.25 weights, weighted class means/shrinkage covariance, fixed .8/.2 priors. These match the original equal-five-class .2 Other prior in intent, not mathematical identity; binary_complete is required to isolate architecture from unknown treatment.',
        unchanged='Original scaler,PCA30,GMM30,n_init3,seeds0,C.1,LR class weights, two pseudo refreshes, row acceptance max>=.8 and positive label cutoff.3+argmax. Only Other can be unknown; other four pseudo labels retain current treatment. Final80/20 blend and cutoff.3+argmax; no score normalization, thresholds or blend weights selected on outcomes.',
        schedule='All five variants on80 saved grouped folds plus validation78 and three chronology windows. Fresh unchanged anchored and three altered model fits per case; half blends reuse scores. Cached strict five-seed Cand2 reused with matching IDs/truth/plans and frozen source provenance. No new Cand2 fits needed.',
        supplementary_exception='Evaluate all variants on all four supplementary cases regardless of CV, predeclared to diagnose the hypothesis.',
        access='Inductive: only eligible fitting gold/masked inputs; exact saved order and group purges. Original validation retains historical no-outer-purge boundary. Query functions receive embeddings only. No actual Kaggle test inputs loaded during benchmarking.',
        advancement='Positive mean sample-F1 delta and>=5/8 paired wins in BOTH pools against .89935 parent. Also compare to anchored and binary_unknown versus matched binary_complete (likewise half). Keep every supplementary regression visible. No automatic promotion or upload. A qualifying variant with validation/chronology regressions no worse than .005 may receive a final235 candidate export.',
        scoring='Complete OOF sample-F1 per repeat, population SD, paired deltas/wins/ties/losses, macro F1, every label precision/recall/F1/support, Other-only and Recruitment+Other recall/false positives, each chronology window and pooled98.',
        checks='Fresh anchored matches all saved scores within1e-7 and decisions exactly; no-change subclass matches on first case. Unit-weight binary LDA equals sklearn shrinkage LDA, unknown pseudo rows cannot vote negative, known gold negatives retained, binary weighted prior masses fixed. Independently rescore outputs and verify IDs/groups/coverage/source hashes.',
        limitations='Reused development labels/splits, only nine gold Recruitment+Other examples in pooled235, unknown-label omission may add false positives or broaden neighbor radius. Binary architecture changes require its complete-label control. No outer label/threshold tuning.',
        retention='Keep source/protocol, complete compact summaries/negatives and independent verification. Keep detailed predictions only for qualifying/useful comparisons after checking all summaries; remove failed fold caches.',
        versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__),sources=sources()))
    print('Registered five fixed variants, including matched controls and half-strength pairs.',flush=True)


def guard():assert read(HERE/'protocol.json')['sources']==sources(),'Registered sources changed'


def evaluate(c,output):
    t=time.monotonic();fit,query=c['fit'],c['query']
    X,Y,Q=fit[EMB].to_numpy(float),fit[LABELS].to_numpy(int),query[EMB].to_numpy(float)
    original=Y.copy();a_ref,cc=reference(c);audits={}
    with threadpool_limits(limits=1),warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        parent=AnchoredReasonClassifier().fit(X,Y);a=parent.predict_proba(Q)
        np.testing.assert_allclose(a,a_ref,atol=1e-7,rtol=0)
        np.testing.assert_array_equal(select_reasons(a),select_reasons(a_ref))
        changed={}
        for mode in ['neighbor_unknown','binary_complete','binary_unknown']:
            m=PartialOtherAnchored(mode).fit(X,Y);changed[mode]=m.predict_proba(Q)
            audits[mode]=m.round_audit_
        control_error=None
        if c['key']=='train_20261126_0':
            b=PartialOtherAnchored('control').fit(X,Y).predict_proba(Q)
            np.testing.assert_allclose(b,a,atol=1e-10,rtol=0)
            np.testing.assert_array_equal(select_reasons(b),select_reasons(a))
            control_error=float(np.abs(b-a).max())
    np.testing.assert_array_equal(Y,original)
    scores=dict(parent=.8*a_ref+.2*cc,anchored=a_ref)
    scores.update({name:.8*s+.2*cc for name,s in changed.items()})
    for name in ['binary_complete','binary_unknown']:scores[name+'_half']=.4*a_ref+.4*changed[name]+.2*cc
    S=np.stack([scores[name] for name in NAMES]);P=np.stack([select_reasons(s) for s in S])
    assert np.isfinite(S).all() and (P.sum(2)>0).all()
    dest=Path(output)/'predictions'/(c['key']+'.npz')
    np.savez_compressed(dest,names=np.array(NAMES),ids=query.ID.to_numpy(str),truth=query[LABELS].to_numpy(int),scores=S,predictions=P)
    audit=dict(key=c['key'],pool=c['pool'],seed=c['seed'],fit_ids_sha256=idhash(fit.ID),query_ids_sha256=idhash(query.ID),
        gold=int((Y>=0).all(1).sum()),masked=int((Y<0).all(1).sum()),parent_max_error=float(np.abs(a-a_ref).max()),
        no_change_control_max_error=control_error,rounds=audits,seconds=time.monotonic()-t,sha256=sha(dest))
    save(dest.with_suffix('.json'),audit)
    return dict(key=c['key'],seconds=round(audit['seconds'],2))


def metric_rows(y,p):
    precision,recall,f1,support=precision_recall_fscore_support(y,p,zero_division=0)
    m=dict(sample_f1=float(f1_score(y,p,average='samples')),macro_f1=float(f1.mean()))
    lab=[dict(label=name,precision=float(precision[j]),recall=float(recall[j]),f1=float(f1[j]),support=int(support[j])) for j,name in enumerate(LABELS)]
    return m,lab


def collect(output):
    guard();output=Path(output);groups={};audits=[]
    for c in cases():
        p=output/'predictions'/(c['key']+'.npz');z=loadnp(p);a=read(p.with_suffix('.json'))
        assert a['sha256']==sha(p);audits.append(a)
        np.testing.assert_array_equal(z['ids'],c['query'].ID);np.testing.assert_array_equal(z['truth'],c['query'][LABELS])
        key=(c['pool'],c['seed']) if c['pool']!='supplementary' else (c['key'],-1)
        groups.setdefault(key,[]).append(z)
        if c['key'].startswith('through_'):groups.setdefault(('chronology98',-1),[]).append(z)
    rows=[];labels=[];subgroups=[]
    for (pool,seed),zs in groups.items():
        y=np.concatenate([z['truth'] for z in zs]);ids=np.concatenate([z['ids'] for z in zs])
        assert len(ids)==len(set(ids))
        if pool in ['train','pooled']:assert len(zs)==5
        for j,name in enumerate(NAMES):
            p=np.concatenate([z['predictions'][j] for z in zs]);m,ls=metric_rows(y,p)
            base=dict(pool=pool,seed=seed,recipe=name,n=len(y));rows.append(dict(**base,**m));labels.extend(dict(**base,**l) for l in ls)
            for group,mask in [('other_alone',(y[:,4]==1)&(y.sum(1)==1)),('other_with_recruitment',(y[:,4]==1)&(y[:,3]==1)),('not_other',y[:,4]==0)]:
                subgroups.append(dict(**base,subgroup=group,support=int(mask.sum()),other_predicted=int(p[mask,4].sum())))
    out=output/'summary';out.mkdir(exist_ok=True);d=pd.DataFrame(rows)
    d.to_csv(out/'repeat_scores.csv',index=False);pd.DataFrame(labels).to_csv(out/'per_label_scores.csv',index=False)
    pd.DataFrame(subgroups).to_csv(out/'other_subgroups.csv',index=False)
    summary=d.groupby(['pool','recipe']).agg(mean_sample_f1=('sample_f1','mean'),population_sd=('sample_f1',lambda x:x.std(ddof=0)),mean_macro_f1=('macro_f1','mean')).reset_index()
    summary.to_csv(out/'overview.csv',index=False);pairs=[];deltas=[];qualifies={n:True for n in CANDIDATES}
    comparisons=[(n,control) for n in CANDIDATES for control in ['parent','anchored']]+[
        ('binary_unknown','binary_complete'),('binary_unknown_half','binary_complete_half')]
    for pool in ['train','pooled']:
        pivot=d[d.pool==pool].pivot(index='seed',columns='recipe',values='sample_f1')
        for name,control in comparisons:
            v=pivot[name]-pivot[control];wins=int((v>1e-12).sum());ties=int((v.abs()<=1e-12).sum())
            passed=bool(v.mean()>1e-12 and wins>=5)
            if control=='parent':qualifies[name]&=passed
            pairs.append(dict(pool=pool,recipe=name,control=control,mean_delta=float(v.mean()),wins=wins,ties=ties,losses=8-wins-ties,passes=passed))
            deltas.extend(dict(pool=pool,recipe=name,control=control,seed=int(s),delta=float(t)) for s,t in v.items())
    pd.DataFrame(pairs).to_csv(out/'paired_summary.csv',index=False);pd.DataFrame(deltas).to_csv(out/'paired_repeat_deltas.csv',index=False)
    save(out/'qualification.json',qualifies)
    save(out/'fit_summary.json',dict(cases=len(audits),fresh_anchored_fits=84,altered_fits=252,no_change_control_fits=1,
        cached_cand2_cases=84,total_fit_seconds=sum(a['seconds'] for a in audits),parent_max_error=max(a['parent_max_error'] for a in audits),
        control_max_error=[a['no_change_control_max_error'] for a in audits if a['no_change_control_max_error'] is not None]))
    audit_rows=[]
    for a in audits:
        for mode,rounds in a['rounds'].items():
            audit_rows.extend(dict(case=a['key'],pool=a['pool'],seed=a['seed'],recipe=mode,**r) for r in rounds)
    pd.DataFrame(audit_rows).to_csv(out/'training_audits.csv',index=False)
    print(summary[summary.pool.isin(['train','pooled','validation','chronology98'])].to_string(index=False),flush=True)
    print('QUALIFIED',qualifies,flush=True)


def run(output,workers):
    guard();output=Path(output);assert not output.exists(),'Use a fresh output directory.'
    (output/'predictions').mkdir(parents=True)
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futures=[pool.submit(evaluate,c,str(output)) for c in cases()]
        for i,f in enumerate(as_completed(futures),1):print(json.dumps(dict(done=i,total=84,**f.result())),flush=True)
    collect(output)


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('command',choices=['register','run','collect']);p.add_argument('--output',type=Path,default=HERE/'results');p.add_argument('--workers',type=int,default=4)
    args=p.parse_args()
    if args.command=='register':register()
    elif args.command=='run':run(args.output,args.workers)
    else:collect(args.output)

Overwriting _source_export/part1/research/partial_secondary_v1/study_run.py


## part1/research/partial_secondary_v1/verify_partial.py

In [129]:
%%writefile _source_export/part1/research/partial_secondary_v1/verify_partial.py
"""Independent arithmetic, boundary and partial-label contract verification."""
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import study_run as study


def metrics(y,p):
    tp=(y*p).sum(0);fp=((1-y)*p).sum(0);fn=(y*(1-p)).sum(0)
    pr=np.divide(tp,tp+fp,out=np.zeros(5),where=(tp+fp)>0)
    re=np.divide(tp,tp+fn,out=np.zeros(5),where=(tp+fn)>0)
    f=np.divide(2*tp,2*tp+fp+fn,out=np.zeros(5),where=(2*tp+fp+fn)>0)
    sample=(2*(y*p).sum(1)/(y.sum(1)+p.sum(1))).mean()
    return float(sample),float(f.mean()),pr,re,f,y.sum(0)


def verify(output):
    study.guard();output=Path(output)
    recorded=pd.read_csv(output/'summary/repeat_scores.csv')
    labs=pd.read_csv(output/'summary/per_label_scores.csv');pairs=pd.read_csv(output/'summary/paired_summary.csv')
    grouped={};n=0;max_error=0.;unknown_rounds=0
    for c in study.cases():
        path=output/'predictions'/(c['key']+'.npz');z=study.loadnp(path);audit=study.read(path.with_suffix('.json'))
        assert study.sha(path)==audit['sha256']
        np.testing.assert_array_equal(z['ids'],c['query'].ID);np.testing.assert_array_equal(z['truth'],c['query'][study.LABELS])
        np.testing.assert_array_equal(z['names'],study.NAMES)
        assert study.idhash(c['fit'].ID)==audit['fit_ids_sha256'] and study.idhash(c['query'].ID)==audit['query_ids_sha256']
        assert set(c['fit'].ID).isdisjoint(c['query'].ID)
        if c['key']!='validation':assert not set(study.benchmark.groups(c['fit']))&set(study.benchmark.groups(c['query']))
        Y=c['fit'][study.LABELS].to_numpy(int);gold=(Y>=0).all(1);neg=int((Y[gold,4]==0).sum())
        for mode,rounds in audit['rounds'].items():
            assert len(rounds)==3
            for r in rounds:
                assert r['gold']==int(gold.sum())
                assert r['observed_other_positive']+r['observed_other_negative']+r['unknown_other']==r['gold']+r['generated']
                if mode in ['neighbor_unknown','binary_unknown']:
                    assert r['observed_other_negative']==neg
                    assert r['unknown_other']>=0
                    unknown_rounds+=1
                else:assert r['unknown_other']==0
                if mode.startswith('binary_'):
                    assert r['binary_training_rows']==r['observed_other_positive']+r['observed_other_negative']
                    np.testing.assert_allclose(r['binary_weight_by_class'],np.array([.8,.2])*r['gold'],atol=1e-10,rtol=0)
        a,cc=study.reference(c)
        np.testing.assert_array_equal(z['scores'][0],.8*a+.2*cc)
        for name in ['binary_complete','binary_unknown']:
            np.testing.assert_allclose(z['scores'][study.NAMES.index(name+'_half')],
                .5*z['scores'][0]+.5*z['scores'][study.NAMES.index(name)],atol=1e-12,rtol=0)
        for j in range(len(study.NAMES)):
            p=(z['scores'][j]>=.3).astype(int);p[np.arange(len(p)),z['scores'][j].argmax(1)]=1
            np.testing.assert_array_equal(p,z['predictions'][j])
        key=(c['pool'],c['seed']) if c['pool']!='supplementary' else (c['key'],-1)
        grouped.setdefault(key,[]).append(z)
        if c['key'].startswith('through_'):grouped.setdefault(('chronology98',-1),[]).append(z)
        n+=1;max_error=max(max_error,audit['parent_max_error'])
    checked=0
    for (pool,seed),zs in grouped.items():
        y=np.concatenate([z['truth'] for z in zs]);ids=np.concatenate([z['ids'] for z in zs])
        assert len(ids)==len(set(ids))
        if pool in ['train','pooled']:assert len(zs)==5 and len(y)==(157 if pool=='train' else 235)
        if pool=='chronology98':assert len(y)==98
        for j,name in enumerate(study.NAMES):
            p=np.concatenate([z['predictions'][j] for z in zs]);s,m,pr,re,f,support=metrics(y,p)
            r=recorded[(recorded.pool==pool)&(recorded.seed==seed)&(recorded.recipe==name)].iloc[0]
            np.testing.assert_allclose([s,m],[r.sample_f1,r.macro_f1],atol=1e-12,rtol=0)
            l=labs[(labs.pool==pool)&(labs.seed==seed)&(labs.recipe==name)].set_index('label').loc[study.LABELS]
            np.testing.assert_allclose(np.c_[pr,re,f,support],l[['precision','recall','f1','support']],atol=1e-12,rtol=0)
            checked+=1
    for _,r in pairs.iterrows():
        d=recorded[recorded.pool==r.pool].pivot(index='seed',columns='recipe',values='sample_f1')
        v=d[r.recipe]-d[r.control]
        np.testing.assert_allclose(v.mean(),r.mean_delta,atol=1e-12,rtol=0)
        assert int((v>1e-12).sum())==r.wins and int((v.abs()<=1e-12).sum())==r.ties
    expected={'train':.935244161358811,'pooled':.9216312056737589,'validation':.8910256410256411,'chronology98':.8877551020408163}
    for pool,value in expected.items():
        np.testing.assert_allclose(recorded[(recorded.pool==pool)&(recorded.recipe=='parent')].sample_f1.mean(),value,atol=1e-12,rtol=0)
    receipt=dict(passed=True,outer_cases=n,metric_rows_independently_checked=checked,
        partial_label_rounds_checked=unknown_rounds,gold_negative_labels_preserved=True,
        class_prior_weight_mass_fixed=True,parent_max_score_error=max_error,
        parent_four_headlines_reproduced=True,ids_groups_and_coverage_checked=True,
        half_blend_formulas_checked=True,all_source_hashes_unchanged=True,
        protocol_sha256=study.sha(study.HERE/'protocol.json'))
    study.save(output/'summary/verification.json',receipt);print(receipt)


if __name__=='__main__':verify(Path(sys.argv[1]) if len(sys.argv)>1 else study.HERE/'results')

Overwriting _source_export/part1/research/partial_secondary_v1/verify_partial.py


## part1/research/pysr_blend_submission_v1/export.py

In [130]:
%%writefile _source_export/part1/research/pysr_blend_submission_v1/export.py
"""Freeze and prepare the user-authorized 80/20 PySR trial on all 235 labels."""
import os
for key in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','JULIA_NUM_THREADS','JULIA_NUM_PRECOMPILE_TASKS']:os.environ[key]='1'
from datetime import datetime,timezone
from pathlib import Path
import argparse
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

HERE=Path(__file__).resolve().parent
STUDY=HERE.parent/'pysr_v1';P1=HERE.parents[1];ROOT=P1.parent
sys.path.insert(0,str(STUDY))
import study
from model import SymbolicReasons,centered,decide,FEATURE_NAMES
from common import LABELS,FEATURES,sha,save
PARENT=HERE.parent/'anchored_lda_blend_submission_v1'
CSV=P1/'submissions/ready/part1_current80_pysr20_trainval235.csv'


def sources():
    paths=[Path(__file__),HERE/'verify.py',STUDY/'protocol.json',STUDY/'outcome.json',STUDY/'retention.json',PARENT/'prediction_scores.npz',PARENT/'manifest.json',P1/'submissions/results/anchored75-lda25-trainval235.json',P1/'submissions/ready/part1_anchored75_lda25_trainval235.csv']
    return study.sources()|{str(p.relative_to(ROOT)):sha(p) for p in paths}


def guard():
    study.guard();assert sources()==json.loads((HERE/'protocol.json').read_text())['sources']


def register():
    assert not (HERE/'protocol.json').exists()
    save(HERE/'protocol.json',dict(registered_utc=datetime.now(timezone.utc).isoformat(),
        authorization='User requested trying the PySR blend on Kaggle despite its weaker full local comparison.',
        evaluation_source_commit='80f713fb',recipe='current80_pysr20',
        formula='0.8 * existing anchored75_lda25 centered score + 0.2 * center(PySR sigmoid score, native cutoff=.3). Decode at .5 plus nonempty argmax.',
        component_weights=dict(existing_89850_blend=.8,pysr=.2),effective_weights=dict(anchored=.60,lda_ensemble=.20,pysr=.20),
        fitting_population='All 1171 original training rows followed by78 validation rows:235 observed gold +1014 originally masked. Fit exactly the registered PySR model. Existing75/25 component uses its already verified full235 fits.',
        access='Inductive: actual test inputs used only after fitting. No labels from test. No threshold, equation-selection, seed, search-budget or weight changes.',
        policy_exception='User-authorized manual trial. Local both-pool rule failed and all four headline checks regressed. This export does not change that finding or promote a leader.',
        selection='Exactly one requested candidate; no further search based on test outputs. Retain full final model state, equations, generated labels and predictions.',
        status='Registered preparation, not an upload or reported score.',sources=sources()))


def prepare():
    guard();assert not CSV.exists() and not (HERE/'manifest.json').exists()
    train=pd.read_csv(P1/'data/part1_train.csv');val=pd.read_csv(P1/'data/part1_val.csv');fit=pd.concat([train,val],ignore_index=True)
    x=fit[FEATURES].to_numpy(float);y=fit[LABELS].to_numpy(int);known=np.isin(y,[0,1]).all(1)
    assert known.sum()==235 and (~known).sum()==1014 and fit.ID.is_unique
    print('Fitting unchanged PySR recipe:235 gold +1014 masked rows',flush=True);start=time.monotonic()
    with threadpool_limits(limits=1):model=SymbolicReasons().fit(x,y)
    # Actual test inputs first enter after model fitting is complete.
    test=pd.read_csv(P1/'data/part1_test.csv');sample=pd.read_csv(P1/'data/part1_sample_submission.csv')
    assert set(fit.ID).isdisjoint(test.ID);np.testing.assert_array_equal(test.ID,sample.ID)
    q=test[FEATURES].to_numpy(float)
    with threadpool_limits(limits=1):raw,_=model.predict_scores(q)
    np.testing.assert_allclose(model.predict_scores(q[:1])[0],raw[:1],atol=1e-12,rtol=0)
    with np.load(PARENT/'prediction_scores.npz') as z:
        np.testing.assert_array_equal(z['ids'],test.ID);parent=z['scores'];parent_pred=z['predictions']
    record=json.loads((P1/'submissions/results/anchored75-lda25-trainval235.json').read_text())
    assert sha(ROOT/record['submission_file'])==record['submission_sha256']
    encode=lambda p:[' '.join(label for label,v in zip(LABELS,row) if v) for row in p]
    assert pd.read_csv(ROOT/record['submission_file']).Predicted.tolist()==encode(parent_pred)
    np.testing.assert_array_equal(decide(parent,.5),parent_pred)
    scores=.8*parent+.2*centered(raw);pred=decide(scores,.5)
    frame=pd.DataFrame(dict(ID=test.ID,Predicted=encode(pred)));assert len(frame)==312 and frame.ID.is_unique and frame.Predicted.str.len().gt(0).all()
    frame.to_csv(CSV,index=False);pd.testing.assert_frame_equal(frame,pd.read_csv(CSV))
    np.savez_compressed(HERE/'prediction_scores.npz',ids=test.ID.to_numpy(str),labels=np.array(LABELS),parent_scores=parent,parent_predictions=parent_pred,pysr_raw=raw,pysr_centered=centered(raw),scores=scores,predictions=pred)
    g=model.geometry
    np.savez_compressed(HERE/'model_state.npz',fit_ids=fit.ID.to_numpy(str),gold_mask=known,embedding_mean=g.scaler.mean_,embedding_scale=g.scaler.scale_,
        cluster_centers=g.cluster.cluster_centers_,fit_cluster=g.cluster.labels_,bank=g.bank,bank_y=g.y,bank_groups=g.bank_groups,bank_cluster=g.bank_cluster,
        augmented_labels=g.augmented,sample_weights=g.weights,feature_means=np.stack([s.mean_ for s in model.scales]),feature_scales=np.stack([s.scale_ for s in model.scales]))
    save(HERE/'equations.json',dict(feature_names=FEATURE_NAMES,equations=model.equations,model_audit=model.audit))
    augmented=fit[['ID']].copy();augmented['source']=np.where(known,'observed','kmeans_generated')
    for j,label in enumerate(LABELS):augmented[label]=g.augmented[:,j]
    augmented.to_csv(HERE/'augmented_labels.csv',index=False)
    changed=(pred!=parent_pred).any(1)
    pd.DataFrame(dict(ID=test.ID[changed],previous=np.array(encode(parent_pred))[changed],candidate=np.array(encode(pred))[changed])).to_csv(HERE/'changes_vs_89850.csv',index=False)
    save(HERE/'manifest.json',dict(status='Prepared for user-requested manual Kaggle trial; not uploaded',recipe='current80_pysr20_trainval235',created_utc=datetime.now(timezone.utc).isoformat(),
        submission_file=str(CSV.relative_to(ROOT)),submission_sha256=sha(CSV),rows=312,verified_training_labels=235,originally_masked_training_rows=1014,
        new_pysr_full235_fits=1,new_symbolic_heads=5,existing_parent_fits_reused=True,test_inputs_used_for_fit=False,observed_labels_preserved=True,query_batch_invariance_passed=True,
        weights=dict(existing_89850_blend=.8,pysr=.2),changed_labelsets_vs_89850=int(changed.sum()),minimum_margin_from_cutoff=float(abs(scores-.5).min()),
        upload_performed=False,reported_kaggle_score=None,preparation_seconds=time.monotonic()-start,protocol_sha256=sha(HERE/'protocol.json'),
        artifacts={p.name:sha(p) for p in [HERE/'prediction_scores.npz',HERE/'model_state.npz',HERE/'equations.json',HERE/'augmented_labels.csv',HERE/'changes_vs_89850.csv']}))
    guard();print(json.dumps(json.loads((HERE/'manifest.json').read_text()),indent=2),flush=True)


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('command',choices=['register','prepare']);a=p.parse_args()
    register() if a.command=='register' else prepare()

Overwriting _source_export/part1/research/pysr_blend_submission_v1/export.py


## part1/research/pysr_blend_submission_v1/verify.py

In [131]:
%%writefile _source_export/part1/research/pysr_blend_submission_v1/verify.py
"""Reconstruct the final PySR predictions from portable arrays and equations."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import expit,logit
from sklearn.preprocessing import normalize
import sympy as sp
from export import HERE,P1,ROOT,PARENT,FEATURES,LABELS,sha,guard


def verify():
    guard();manifest=json.loads((HERE/'manifest.json').read_text())
    for name,digest in manifest['artifacts'].items():assert sha(HERE/name)==digest,name
    csv=ROOT/manifest['submission_file'];assert sha(csv)==manifest['submission_sha256']
    frame=pd.read_csv(csv);sample=pd.read_csv(P1/'data/part1_sample_submission.csv');test=pd.read_csv(P1/'data/part1_test.csv')
    fit=pd.concat([pd.read_csv(P1/'data/part1_train.csv'),pd.read_csv(P1/'data/part1_val.csv')],ignore_index=True)
    with np.load(HERE/'model_state.npz') as z:state={k:z[k] for k in z.files}
    with np.load(HERE/'prediction_scores.npz') as z:preds={k:z[k] for k in z.files}
    np.testing.assert_array_equal(state['fit_ids'],fit.ID);np.testing.assert_array_equal(frame.ID,sample.ID);np.testing.assert_array_equal(frame.ID,test.ID)
    gold=np.isin(fit[LABELS].to_numpy(),[0,1]).all(1);np.testing.assert_array_equal(gold,state['gold_mask']);assert gold.sum()==235 and (~gold).sum()==1014
    np.testing.assert_array_equal(state['augmented_labels'][gold],fit.loc[gold,LABELS]);assert set(fit.ID).isdisjoint(test.ID)
    assert np.all(state['augmented_labels'].sum(1)>0) and abs(state['sample_weights'][~gold].sum()-23.5)<1e-12
    all_z=normalize((fit[FEATURES].to_numpy(float)-state['embedding_mean'])/state['embedding_scale'])
    np.testing.assert_allclose(all_z[gold],state['bank'],atol=1e-12,rtol=0)
    q=normalize((test[FEATURES].to_numpy(float)-state['embedding_mean'])/state['embedding_scale'])
    # Independently assign nearest centroids, then recompute all five inputs.
    centers=state['cluster_centers'];clusters=np.argmin(((q[:,None,:]-centers[None,:,:])**2).sum(2),axis=1)
    bank=state['bank'];y=state['bank_y'];bank_clusters=state['bank_cluster'];prior=y.mean(0)
    pos=normalize(y.T@bank);neg=normalize((1-y).T@bank);inputs=[]
    for row,k in zip(q,clusters):
        sims=bank@row;ix=np.argsort(-sims,kind='stable')[:10];w=np.maximum(sims[ix],0)**2
        vote=(w@y[ix]+1e-3*prior)/(w.sum()+1e-3)
        rates=(y[bank_clusters==k].sum(0)+prior)/(int((bank_clusters==k).sum())+1)
        margin=(pos-neg)@row;features=[]
        for j in range(5):
            other=np.arange(5)!=j
            features.append([logit(np.clip(vote[j],.01,.99)),margin[j],logit(np.clip(rates[j],.01,.99)),logit(np.clip(vote[other].max(),.01,.99)),margin[other].max()])
        inputs.append(features)
    inputs=(np.asarray(inputs)-state['feature_means'])/state['feature_scales']
    formulas=json.loads((HERE/'equations.json').read_text());symbols=sp.symbols(formulas['feature_names']);raw=[]
    for j,e in enumerate(formulas['equations']):
        expression=sp.sympify(e['expression'],locals=dict(zip(formulas['feature_names'],symbols)))
        fn=sp.lambdify(symbols,expression,modules='numpy');values=fn(*inputs[:,j].T)
        raw.append(expit(np.broadcast_to(values,(312,))))
    raw=np.column_stack(raw);np.testing.assert_allclose(raw,preds['pysr_raw'],atol=1e-10,rtol=0)
    with np.load(PARENT/'prediction_scores.npz') as z:
        np.testing.assert_array_equal(z['ids'],frame.ID);np.testing.assert_array_equal(z['scores'],preds['parent_scores'])
    raw=np.clip(raw,1e-9,1-1e-9);centered=.7*raw/(.7*raw+.3*(1-raw));scores=.8*preds['parent_scores']+.2*centered
    np.testing.assert_allclose(scores,preds['scores'],atol=1e-10,rtol=0)
    pred=(scores>=.5).astype(int)
    for i in range(312):
        if not pred[i].any():pred[i,scores[i].argmax()]=1
    np.testing.assert_array_equal(pred,preds['predictions'])
    assert frame.Predicted.tolist()==[' '.join(label for label,v in zip(LABELS,row) if v) for row in pred]
    assert len(frame)==312 and frame.ID.is_unique and frame.Predicted.str.len().gt(0).all()
    changed=int((pred!=preds['parent_predictions']).any(1).sum());assert changed==manifest['changed_labelsets_vs_89850']
    result=dict(passed=True,rows=312,verified_training_labels=235,originally_masked_rows=1014,independent_feature_equation_and_csv_reconstruction=True,
                maximum_pysr_score_error=float(abs(raw-preds['pysr_raw']).max()),changed_labelsets_vs_89850=changed,submission_sha256=sha(csv))
    print(json.dumps(result,indent=2));return result


if __name__=='__main__':verify()

Overwriting _source_export/part1/research/pysr_blend_submission_v1/verify.py


## part1/research/pysr_blend_submission_v1/verify_retained.py

In [132]:
%%writefile _source_export/part1/research/pysr_blend_submission_v1/verify_retained.py
"""Verify the compact scored record after nonwinning model caches are removed."""
import csv
import hashlib
import json
from pathlib import Path
import subprocess

HERE=Path(__file__).resolve().parent
ROOT=HERE.parents[2]
LABELS=['Safety','Funding','Efficacy','Recruitment','Other']
REMOVED={'model_state.npz','prediction_scores.npz','augmented_labels.csv'}

def read(path):return json.loads(Path(path).read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def rows(path):
    with Path(path).open(newline='') as f:return list(csv.DictReader(f))

def verify():
    retained=read(HERE/'retention.json')
    for path,digest in retained['files'].items():assert sha(ROOT/path)==digest,path
    protocol=read(HERE/'protocol.json')
    for path,digest in protocol['sources'].items():assert sha(ROOT/path)==digest,path
    manifest=read(HERE/'manifest.json');result_path='part1/submissions/results/current80-pysr20-trainval235.json';result=read(ROOT/result_path)
    assert result['reported_kaggle_score']==.89423
    assert result['submission_sha256']==manifest['submission_sha256']
    assert sha(ROOT/result['submission_file'])==result['submission_sha256']
    for name,digest in manifest['artifacts'].items():
        if name in REMOVED:assert not (HERE/name).exists()
        else:assert sha(HERE/name)==digest,name
    sample=rows(ROOT/'part1/data/part1_sample_submission.csv');submitted=rows(ROOT/result['submission_file'])
    ids=[r['ID'] for r in submitted];assert ids==[r['ID'] for r in sample] and len(ids)==len(set(ids))==312
    for row in submitted:
        labels=row['Predicted'].split();assert labels and len(labels)==len(set(labels)) and set(labels)<=set(LABELS)
        assert row['Predicted']==' '.join(label for label in LABELS if label in labels)
    parent_result=read(ROOT/'part1/submissions/results/anchored75-lda25-trainval235.json')
    parent=rows(ROOT/parent_result['submission_file']);assert sha(ROOT/parent_result['submission_file'])==parent_result['submission_sha256']
    assert [r['ID'] for r in parent]==ids
    changes=[dict(ID=a['ID'],previous=a['Predicted'],candidate=b['Predicted']) for a,b in zip(parent,submitted) if a['Predicted']!=b['Predicted']]
    assert changes==rows(HERE/'changes_vs_89850.csv') and len(changes)==manifest['changed_labelsets_vs_89850']==9
    eq=read(HERE/'equations.json');assert [e['label'] for e in eq['equations']]==LABELS
    assert eq['model_audit']['gold_rows']==235 and eq['model_audit']['masked_rows']==1014
    assert eq['model_audit']['observed_labels_preserved'] and not eq['model_audit']['query_inputs_used_for_fit']
    audit=read(HERE/'retention_audit.json');assert audit['pre_cleanup_verification']['full_prediction_reconstruction_passed']
    assert audit['candidate_artifacts_removed']==3 and not retained['candidate_model_or_score_arrays_retained']
    git=lambda *args:subprocess.check_output(['git','-C',str(ROOT),*args])
    tag=result['tag'];assert git('cat-file','-t','refs/tags/'+tag).strip()==b'tag'
    assert git('rev-parse',tag+'^{commit}').decode().strip()==audit['immutable_result_commit']
    assert git('show',tag+':'+result_path)==(ROOT/result_path).read_bytes()
    assert git('show',tag+':'+result['submission_file'])==(ROOT/result['submission_file']).read_bytes()
    report=dict(passed=True,retained_files_checked=len(retained['files']),protected_sources_checked=len(protocol['sources']),
        scored_csv_rows=312,changed_labelsets_vs_parent=len(changes),score_tag_and_result_unchanged=True,
        candidate_arrays_present=False,full_prediction_reconstruction_performed=False,
        limitation='Full equation/model reconstruction passed before pruning. Current verification checks compact records, sources, exact CSV and tag; a refit or scored checkout is required for full reconstruction.')
    print(json.dumps(report,indent=2));return report

if __name__=='__main__':verify()

Overwriting _source_export/part1/research/pysr_blend_submission_v1/verify_retained.py


## part1/research/pysr_v1/build_controls.py

In [133]:
%%writefile _source_export/part1/research/pysr_v1/build_controls.py
"""Snapshot matched controls from a frozen Git commit, never a moving worktree."""
import io
import importlib.util
import json
import subprocess
import sys
import hashlib
import numpy as np
from common import HERE,PART1,ROOT,benchmark,sha,save

COMMIT='cde3d500e1304f2afeeb6dc4c7f50e49190c6a4b'
BASE='part1/experiments/cand2_followup_v1/'
blob=lambda path:subprocess.check_output(['git','-C',str(ROOT),'show',COMMIT+':'+path])
prior=PART1/'research/anchored_neighbor_blend_v1'
sys.path.insert(0,str(prior));spec=importlib.util.spec_from_file_location('pysr_lda_control',prior/'study.py');ref=importlib.util.module_from_spec(spec);spec.loader.exec_module(ref)
plan={c['key']:c for c in json.loads(blob(BASE+'case_plan.json'))}
ids,truth,keys,pools,seeds,scores=[],[],[],[],[],[];origins={}
for c in benchmark.cases():
    key='strict_inductive__'+('grouped_' if c['pool']!='supplementary' else '')+c['key']
    old=plan[key];q=c['query'];assert old['query_ids']==q.ID.tolist()
    assert set(old['fit_ids'])==set(c['fit'].ID)
    path=BASE+'inputs/'+key+'.npz';data=blob(path);origins[path]=hashlib.sha256(data).hexdigest()
    with np.load(io.BytesIO(data)) as z:
        np.testing.assert_array_equal(z['ids'],q.ID);np.testing.assert_array_equal(z['truth'],q[benchmark.LABELS])
        anchor=z['anchored_scores'];cand=z['cand2_scores']
        center=lambda s:.7*s/(.7*s+.3*(1-s))
        a=center(anchor);lda=ref.lda_for(c)
        scores.append(np.stack([a,.75*a+.25*lda,center(.8*anchor+.2*cand)]))
    ids.extend(q.ID);truth.extend(q[benchmark.LABELS].to_numpy(int));keys.extend([c['key']]*len(q));pools.extend([c['pool']]*len(q));seeds.extend([c['seed']]*len(q))
np.savez_compressed(HERE/'controls.npz',modes=np.array(['anchored','current75_25','anchored80_cand2_20']),scores=np.concatenate(scores,axis=1),ids=np.array(ids),truth=np.array(truth),case=np.array(keys),pool=np.array(pools),seed=np.array(seeds))
result_path='part1/submissions/results/anchored80-cand2-inductive20-trainval235.json'
(HERE/'best_control_result.json').write_bytes(blob(result_path))
save(HERE/'control_provenance.json',dict(source_commit=COMMIT,source_sha256=origins,source_result_sha256=sha(HERE/'best_control_result.json'),retained_sha256=sha(HERE/'controls.npz'),cases=84,all_case_keys_fit_ids_query_ids_truth_matched=True,score_convention='All .5-centered; new best is .8 raw anchored + .2 Cand2 then centered from cutoff .3.',lda_source_helper='part1/research/anchored_neighbor_blend_v1/study.py::lda_for',lda_helper_sha256=sha(prior/'study.py')))
print('Frozen controls: all 84 exact case keys, fitting populations, query IDs and truth match.')

Overwriting _source_export/part1/research/pysr_v1/build_controls.py


## part1/research/pysr_v1/check_geometry.py

In [134]:
%%writefile _source_export/part1/research/pysr_v1/check_geometry.py
"""Synthetic check that training feature construction excludes the entire own group."""
import numpy as np
from model import ReasonGeometry
rng=np.random.default_rng(9)
x=rng.normal(size=(120,150));x[1]=x[0]
y=(rng.uniform(size=(120,5))>.6).astype(int);y[np.arange(120),rng.integers(5,size=120)]=1;y[1]=y[0];y[40:]=-100
changed=y.copy();changed[:2]=1-changed[:2]
a=ReasonGeometry().fit(x,y);b=ReasonGeometry().fit(x,changed)
np.testing.assert_allclose(a.transform(x[:2],training=True),b.transform(x[:2],training=True),atol=0,rtol=0)
np.testing.assert_array_equal(a.augmented[:40],y[:40]);assert np.all(a.augmented[40:].sum(1)>0)
assert abs(a.weights[40:].sum()-4)<1e-12
np.testing.assert_array_equal(a.transform(x[:1]),a.transform(x[:7])[:1])
print('Whole-group exclusion, observed-label preservation, pseudo-weight total and query-batch invariance passed on synthetic data.')

Overwriting _source_export/part1/research/pysr_v1/check_geometry.py


## part1/research/pysr_v1/common.py

In [135]:
%%writefile _source_export/part1/research/pysr_v1/common.py
import importlib.util
import json
import hashlib
from pathlib import Path
import sys

HERE=Path(__file__).resolve().parent
PART1=HERE.parents[1];ROOT=PART1.parent
OLD=PART1/'research/anchored_transductive_v1'
sys.path[:0]=[str(PART1),str(OLD)]
spec=importlib.util.spec_from_file_location('pysr_saved_benchmark',OLD/'run.py')
benchmark=importlib.util.module_from_spec(spec);spec.loader.exec_module(benchmark)
LABELS=benchmark.LABELS;FEATURES=benchmark.FEATURES

def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save(path,value):Path(path).write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')

Overwriting _source_export/part1/research/pysr_v1/common.py


## part1/research/pysr_v1/model.py

In [136]:
%%writefile _source_export/part1/research/pysr_v1/model.py
"""Small symbolic classifiers over fold-local reason similarity signals."""
import tempfile
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, normalize
from threadpoolctl import threadpool_limits

LABELS=['Safety','Funding','Efficacy','Recruitment','Other']
FEATURE_NAMES=['neighbor_logodds','centroid_margin','cluster_logodds','rival_neighbor_logodds','rival_centroid_margin']
SETTINGS=dict(clusters=12,neighbors=10,pseudo_mass=.10,cutoff=.30,seed=42)
SEARCH=dict(niterations=80,populations=8,population_size=27,ncycles_per_iteration=100,
    maxsize=15,maxdepth=5,binary_operators=['+','-','*'],unary_operators=[],
    model_selection='best',parsimony=.003,batching=True,batch_size=128,
    parallelism='serial',deterministic=True,precision=64,progress=False,verbosity=0,
    update_verbosity=0,elementwise_loss='weighted_logistic(x, y, w) = w * (max(x, zero(x)) - x*y + log1p(exp(-abs(x))))')


def groups(x):
    return pd.util.hash_pandas_object(pd.DataFrame(np.round(x,10)),index=False).to_numpy()


def decide(scores,cutoff=.3):
    p=(scores>=cutoff).astype(int)
    p[np.arange(len(p)),scores.argmax(1)]=1
    return p


def centered(s):
    s=np.clip(s,1e-9,1-1e-9)
    return .7*s/(.7*s+.3*(1-s))


class ReasonGeometry:
    def fit(self,x,y):
        self.known=np.isin(y,[0,1]).all(1)
        assert (self.known | (y==-100).all(1)).all()
        assert self.known.any() and (~self.known).any()
        self.scaler=StandardScaler().fit(x)
        z=normalize(self.scaler.transform(x))
        self.cluster=KMeans(n_clusters=SETTINGS['clusters'],n_init=10,random_state=SETTINGS['seed']).fit(z)
        self.bank=z[self.known];self.y=y[self.known].copy();self.bank_groups=groups(x[self.known])
        self.bank_cluster=self.cluster.labels_[self.known]
        prior=self.y.mean(0)
        cluster_rates=np.stack([(self.y[self.bank_cluster==k].sum(0)+prior)/(int((self.bank_cluster==k).sum())+1) for k in range(SETTINGS['clusters'])])
        generated=decide(cluster_rates[self.cluster.labels_[~self.known]])
        self.augmented=y.copy();self.augmented[~self.known]=generated
        np.testing.assert_array_equal(self.augmented[self.known],y[self.known])
        self.weights=np.where(self.known,1.,SETTINGS['pseudo_mass']*self.known.sum()/(~self.known).sum())
        return self

    def transform(self,x,training=False):
        z=normalize(self.scaler.transform(x));g=groups(x);cl=self.cluster.predict(z)
        similarities=z@self.bank.T
        result=[]
        for i,v in enumerate(z):
            # Training features exclude the row's whole raw-embedding group from
            # all label-derived signals. Query features use the frozen gold bank.
            keep=(self.bank_groups!=g[i]) if training else np.ones(len(self.y),dtype=bool)
            bank=self.bank[keep];y=self.y[keep];clusters=self.bank_cluster[keep]
            assert len(y)>0
            sim=similarities[i,keep];ix=np.argsort(-sim,kind='stable')[:SETTINGS['neighbors']]
            w=np.maximum(sim[ix],0)**2
            vote=(w@y[ix]+1e-3*y.mean(0))/(w.sum()+1e-3)
            rates=(y[clusters==cl[i]].sum(0)+y.mean(0))/(int((clusters==cl[i]).sum())+1)
            pos=normalize(y.T@bank);neg=normalize((1-y).T@bank)
            margin=(pos-neg)@v
            row=[]
            for j in range(5):
                other=np.arange(5)!=j
                row.append([logit(np.clip(vote[j],.01,.99)),margin[j],logit(np.clip(rates[j],.01,.99)),
                            logit(np.clip(vote[other].max(),.01,.99)),margin[other].max()])
            result.append(row)
        return np.asarray(result,float)


class SymbolicReasons:
    def __init__(self,search_override=None):
        self.search_override=search_override or {}

    def fit(self,x,y):
        from pysr import PySRRegressor
        x,y=np.asarray(x,float),np.asarray(y,int)
        self.geometry=ReasonGeometry().fit(x,y)
        inputs=self.geometry.transform(x,training=True)
        self.heads=[];self.linear=[];self.scales=[];self.equations=[]
        with tempfile.TemporaryDirectory(prefix='aml-pysr-') as tmp,threadpool_limits(limits=1):
            for j,label in enumerate(LABELS):
                scale=StandardScaler().fit(inputs[:,j]);features=scale.transform(inputs[:,j])
                target=self.geometry.augmented[:,j].astype(float)
                self.scales.append(scale)
                if np.unique(target).size<2:
                    prior=float((target.sum()+.5)/(len(target)+1))
                    self.heads.append(prior);self.linear.append(prior)
                    self.equations.append(dict(label=label,expression=str(logit(prior)),complexity=1,loss=None,constant_fallback=True))
                    continue
                linear=LogisticRegression(C=1.,max_iter=3000,random_state=42).fit(features,target,sample_weight=self.geometry.weights)
                sr=PySRRegressor(**(SEARCH|self.search_override),random_state=42+j,output_directory=tmp,run_id=label)
                sr.fit(features,target,weights=self.geometry.weights,variable_names=FEATURE_NAMES)
                best=sr.get_best()
                # SymPy's exported callable works after the temporary Julia/model
                # files have been deleted; retain only this small Python expression.
                func=best['lambda_format']
                test=np.asarray(func(features),float)
                assert np.isfinite(test).all()
                np.testing.assert_allclose(test,sr.predict(features),atol=1e-10,rtol=1e-10)
                self.heads.append(func);self.linear.append(linear)
                self.equations.append(dict(label=label,expression=str(best['sympy_format']),complexity=int(best['complexity']),loss=float(best['loss']),constant_fallback=False))
        self.audit=dict(gold_rows=int(self.geometry.known.sum()),masked_rows=int((~self.geometry.known).sum()),
            generated_positive_counts=self.geometry.augmented[~self.geometry.known].sum(0).tolist(),
            generated_weight=float(self.geometry.weights[~self.geometry.known].sum()),observed_labels_preserved=True,
            training_feature_group_exclusion=True,query_inputs_used_for_fit=False,search=SEARCH|self.search_override)
        return self

    def predict_scores(self,x):
        inputs=self.geometry.transform(np.asarray(x,float));symbolic=[];linear=[]
        for j,(head,control,scale) in enumerate(zip(self.heads,self.linear,self.scales)):
            f=scale.transform(inputs[:,j])
            symbolic.append(np.full(len(f),head) if isinstance(head,float) else expit(np.broadcast_to(head(f),(len(f),))))
            linear.append(np.full(len(f),control) if isinstance(control,float) else control.predict_proba(f)[:,1])
        s,l=np.column_stack(symbolic),np.column_stack(linear)
        assert np.isfinite(s).all() and np.isfinite(l).all()
        return s,l

Overwriting _source_export/part1/research/pysr_v1/model.py


## part1/research/pysr_v1/retained.py

In [137]:
%%writefile _source_export/part1/research/pysr_v1/retained.py
"""Compact sufficient-statistic audit; raw candidate predictions can be discarded."""
import argparse
import json
import shutil
import numpy as np
import pandas as pd
from common import HERE,ROOT,benchmark,LABELS,sha,save
import study


def collect():
    study.report(HERE/'run',write=False)
    rows=[]
    for c in benchmark.cases():
        with np.load(HERE/'run/predictions'/(c['key']+'.npz')) as z:
            y=z['truth']
            for i,mode in enumerate(study.MODES):
                p=z['predictions'][i]
                row=dict(mode=mode,case=c['key'],pool=c['pool'],seed=c['seed'],rows=len(y),sample_f1_sum=float((2*(y*p).sum(1)/(y.sum(1)+p.sum(1))).sum()))
                for j,label in enumerate(LABELS):
                    row.update({f'tp_{label}':int((y[:,j]*p[:,j]).sum()),f'fp_{label}':int(((1-y[:,j])*p[:,j]).sum()),f'fn_{label}':int((y[:,j]*(1-p[:,j])).sum())})
                rows.append(row)
    assert len(rows)==84*len(study.MODES)
    dest=HERE/'results';assert not dest.exists();shutil.copytree(HERE/'run/results',dest)
    pd.DataFrame(rows).to_csv(dest/'sufficient_statistics.csv',index=False)
    verify()


def metric(frame):
    result=dict(sample_f1=float(frame.sample_f1_sum.sum()/frame.rows.sum()))
    fs=[]
    for label in LABELS:
        tp,fp,fn=[float(frame[f'{s}_{label}'].sum()) for s in ['tp','fp','fn']]
        pr=tp/(tp+fp) if tp+fp else 0.;re=tp/(tp+fn) if tp+fn else 0.;f=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
        result.update({f'precision_{label}':pr,f'recall_{label}':re,f'f1_{label}':f,f'support_{label}':tp+fn});fs.append(f)
    result['macro_f1']=float(np.mean(fs));return result


def verify():
    study.guard();out=HERE/'results';stats=pd.read_csv(out/'sufficient_statistics.csv');count=0
    assert len(stats)==504 and not stats.duplicated(['mode','case']).any()
    cases=benchmark.cases();assert set(stats['case'])=={c['key'] for c in cases}
    for c in cases:
        rows=stats[stats['case']==c['key']];assert len(rows)==6 and (rows.rows==len(c['query'])).all()
    for filename in ['repeat_scores','supplementary_scores']:
        table=pd.read_csv(out/(filename+'.csv'))
        for row in table.to_dict('records'):
            frame=stats[stats['mode']==row['mode']]
            if 'pool' in row:frame=frame[(frame.pool==row['pool'])&(frame.seed==row['seed'])]
            elif row['case']=='chronology98':frame=frame[(frame.pool=='supplementary')&(frame['case']!='validation')]
            else:frame=frame[frame['case']==row['case']]
            for name,value in metric(frame).items():assert abs(value-row[name])<1e-12,(filename,row['mode'],name);count+=1
    repeats=pd.read_csv(out/'repeat_scores.csv')
    for row in pd.read_csv(out/'cv_summary.csv').to_dict('records'):
        part=repeats[(repeats.pool==row['pool'])&(repeats['mode']==row['mode'])];assert len(part)==8
        for key,value in [('mean_sample_f1',part.sample_f1.mean()),('population_sd',part.sample_f1.std(ddof=0)),('mean_macro_f1',part.macro_f1.mean())]:assert abs(row[key]-value)<1e-12
    for row in pd.read_csv(out/'paired_summary.csv').to_dict('records'):
        p=repeats[repeats.pool==row['pool']].pivot(index='seed',columns='mode',values='sample_f1');d=p[row['mode']]-p[row['baseline']]
        assert abs(d.mean()-row['mean_delta'])<1e-12
        assert [int((d>1e-12).sum()),int((abs(d)<=1e-12).sum()),int((d< -1e-12).sum())]==[row[k] for k in ['wins','ties','losses']]
    # Matched controls remain detailed: independently check their sufficient stats.
    ref=study.controls()
    for mode in study.MODES[:3]:
        index=study.MODES.index(mode)
        for c in cases:
            mask=ref['case']==c['key'];values=study.arithmetic(ref['truth'][mask],study.decide(ref['scores'][index,mask],.5))
            stored=metric(stats[(stats['mode']==mode)&(stats['case']==c['key'])])
            for name,value in values.items():assert abs(value-stored[name])<1e-12
    if (HERE/'retention.json').exists():
        retained=json.loads((HERE/'retention.json').read_text())
        for path,digest in retained['files'].items():assert sha(ROOT/path)==digest,path
    print(json.dumps(dict(passed=True,matched_cases=84,metric_values_reconstructed=count,full_controls_rechecked=True,all_repeat_averages_and_paired_counts_match=True),indent=2))


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('command',choices=['collect','verify']);a=p.parse_args()
    collect() if a.command=='collect' else verify()

Overwriting _source_export/part1/research/pysr_v1/retained.py


## part1/research/pysr_v1/study.py

In [138]:
%%writefile _source_export/part1/research/pysr_v1/study.py
"""Registered PySR experiment on all 84 frozen cases; no Kaggle upload."""
import os
for key in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','JULIA_NUM_THREADS','JULIA_NUM_PRECOMPILE_TASKS']:
    os.environ[key]='1'
import argparse
from concurrent.futures import ProcessPoolExecutor,as_completed
from datetime import datetime,timezone
import importlib.metadata
import multiprocessing
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from common import HERE,PART1,ROOT,OLD,benchmark,LABELS,FEATURES,sha,save
sys.path.insert(0,str(HERE))
from model import SymbolicReasons,SETTINGS,SEARCH,FEATURE_NAMES,centered,decide,groups

MODES=['anchored','current75_25','anchored80_cand2_20','pysr','linear_geometry','current80_pysr20']
CANDIDATES=MODES[3:]


def sources():
    paths=[HERE/n for n in ['common.py','model.py','study.py','build_controls.py','controls.npz','control_provenance.json','best_control_result.json','requirements-lock.txt','JuliaProject.toml','JuliaManifest.toml']]
    paths += [OLD/'run.py',OLD/'models.py',PART1/'anchored_solution.py',PART1/'TESTING_POLICY.md',ROOT/'Midterm_Project.pdf']
    paths += list((PART1/'data').glob('*.csv'))
    paths += [PART1/'experiments/current_leaders_grouped_v1/snapshots'/(s+'.json') for s in ['train3','train5','pooled3','pooled5','chronology']]
    return {str(p.relative_to(ROOT)):sha(p) for p in paths}


def guard():
    assert sources()==__import__('json').loads((HERE/'protocol.json').read_text())['sources']


def register():
    assert not (HERE/'protocol.json').exists()
    save(HERE/'protocol.json',dict(registered_utc=datetime.now(timezone.utc).isoformat(),
        request='User specifically requested trying PySR after closing prior blend experiments.',
        parent='Anchored 0.89636 policy parent; own submitted 0.89850 blend and current cross-agent 0.89935 blend are matched practical controls.',
        hypothesis='Short symbolic formulas can combine nonlinear interactions between neighbor support, class-centroid affinity, cluster label support and competing reasons, without searching individual randomly rotated coordinates.',
        feature_access='Only 150 supplied reason embeddings. Fit StandardScaler, normalization and KMeans(12) on eligible fitting rows. Gold-derived nearest neighbors, positive-minus-negative centroid cosine margins and smoothed cluster label rates use fitting gold only. Each training row excludes its complete embedding group from these label-derived signals; held-out queries use the frozen fitting bank. Five features per reason: its neighbor logodds, centroid margin, cluster logodds and strongest competing neighbor logodds/centroid margin. Clip rate inputs to [.01,.99]; scale derived features using fitting rows only.',
        augmentation='Generate every originally masked fitting labelset using KMeans cluster gold rates with one global-prior pseudocount, .3 threshold and argmax. Preserve observed gold exactly. Pseudo rows have total weight .10 times gold-row count. IDs serve only split/group audits.',
        settings=SETTINGS,search=SEARCH,search_seeds=[42,43,44,45,46],feature_names=FEATURE_NAMES,
        search_boundary='Each PySR search is a fresh fit wholly inside the outer training partition. Serial deterministic evolution; no wall-clock early cutoff. Formula selected using PySR best training loss/complexity rule, never held-out labels. At most 15 nodes and depth5, operators +,-,* only. Weighted logistic loss; sigmoid gives label scores. No later threshold or weight search.',
        recipes={'pysr':'Five independent PySR binary heads; sigmoid, .3 cutoff, nonempty argmax. Constant smoothed prevalence fallback only if fitting target has one class.',
                 'linear_geometry':'Diagnostic LogisticRegression(C=1) heads on precisely the same features, generated labels, weights and cutoff. Distinguishes signal quality from symbolic-search value; not another hyperparameter search.',
                 'current80_pysr20':'Fixed .8 submitted75_25 centered scores + .2 PySR scores centered from .3; .5 cutoff plus argmax. No fitting of blending weights.'},
        evaluation='All eight saved five-fold repeats in both Train157 and Pooled235, validation78 and all three chronology windows for every recipe, irrespective of screening result. 84 fresh geometry pipelines and up to420 symbolic searches plus420 linear diagnostic heads. Controls are reused exact case-matched predictions, not new fits.',
        advancement='Positive paired mean and >=5/8 repeat wins in BOTH pools versus anchored; separately report same rule against both submitted blends. Keep every supplementary regression visible; no automatic promotion or Kaggle upload.',
        boundaries='Strict inductive. Existing saved split generator enforces masked duplicate purges and chronological fitting years. Query labels never passed to fit or feature generation. Train157 excludes all validation-source rows. Actual Kaggle test inputs hashed only; final fit only if local evidence merits a candidate.',
        setup_checks='Synthetic-only API, weighted-loss, query-batch-invariance and timing checks before evaluation; no real-label pilot or tuning.',
        retention='Source, fixed settings, dependencies, all compact scores, paired deltas, equations and fit audits retained. Full predictions only for qualifying candidates/useful controls. Independently rescore before deleting failed caches. No fitted PySR models or search logs retained.',
        references=['https://pysr.ai/api','https://github.com/astroautomata/PySR','https://arxiv.org/abs/2305.01582'],
        limitations='A compact fixed PySR search budget and one feature construction, not an exhaustive test of symbolic regression. Existing validation data have influenced prior research. Label-dependent similarity features are part of the model, so these are not equations directly in original physical variables.',
        versions={n:importlib.metadata.version(n) for n in ['pysr','juliacall','numpy','pandas','scipy','scikit-learn']},julia='1.13.3',sources=sources()))


def controls():
    with np.load(HERE/'controls.npz') as z:return {k:z[k] for k in z.files}


def evaluate(c):
    start=time.monotonic();fit,q=c['fit'],c['query']
    x=fit[FEATURES].to_numpy(float);y=fit[LABELS].to_numpy(int);qx=q[FEATURES].to_numpy(float)
    with threadpool_limits(limits=1):
        model=SymbolicReasons().fit(x,y);s,l=model.predict_scores(qx)
        np.testing.assert_allclose(model.predict_scores(qx[:1])[0],s[:1],atol=1e-12,rtol=0)
    ref=controls();mask=ref['case']==c['key'];np.testing.assert_array_equal(ref['ids'][mask],q.ID);np.testing.assert_array_equal(ref['truth'][mask],q[LABELS])
    old=ref['scores'][:,mask];scores=np.concatenate([old,np.stack([centered(s),centered(l),.8*old[1]+.2*centered(s)])])
    pred=np.stack([decide(v,.5) for v in scores]);assert scores.shape==(6,len(q),5)
    packet=dict(ids=q.ID.to_numpy(str),truth=q[LABELS].to_numpy(int),scores=scores,predictions=pred)
    audit=dict(case=c['key'],pool=c['pool'],seed=c['seed'],fold=c['fold'],fit_rows=len(fit),query_rows=len(q),purged_masked=c['purged_masked'],
        fit_ids_sha256=__import__('hashlib').sha256('\n'.join(fit.ID).encode()).hexdigest(),query_batch_invariant=True,
        fitting_query_ids_disjoint=set(fit.ID).isdisjoint(q.ID),fitting_query_groups_disjoint=set(groups(x)).isdisjoint(groups(qx)),
        validation_group_overlap_allowed=c['key']=='validation',seconds=time.monotonic()-start,**model.audit)
    assert audit['fitting_query_ids_disjoint']
    if c['key']!='validation':assert audit['fitting_query_groups_disjoint']
    return c['key'],packet,audit,model.equations


def run(output,workers):
    guard();output=Path(output);(output/'predictions').mkdir(parents=True,exist_ok=True)
    jobs=benchmark.cases();pending=[]
    for c in jobs:
        files=[output/'predictions'/(c['key']+ext) for ext in ['.npz','.json']]
        if all(p.exists() for p in files):continue
        assert not any(p.exists() for p in files),'Incomplete case: inspect failure before resuming.'
        pending.append(c)
    print(f'Running {len(pending)} remaining of84 cases, workers={workers}',flush=True)
    with ProcessPoolExecutor(max_workers=workers,mp_context=multiprocessing.get_context('spawn')) as pool:
        futures=[pool.submit(evaluate,c) for c in pending]
        for future in as_completed(futures):
            key,packet,audit,equations=future.result()
            np.savez_compressed(output/'predictions'/(key+'.npz'),**packet)
            save(output/'predictions'/(key+'.json'),dict(audit=audit,equations=equations,protocol_sha256=sha(HERE/'protocol.json')))
            print(__import__('json').dumps(dict(completed=len(list((output/'predictions').glob('*.npz'))),total=84,case=key,seconds=round(audit['seconds'],1))),flush=True)
    guard();report(output)


def arithmetic(y,p):
    tp=(y*p).sum(0);fp=((1-y)*p).sum(0);fn=(y*(1-p)).sum(0)
    div=lambda a,b:np.divide(a,b,out=np.zeros_like(a,dtype=float),where=b!=0)
    pr=div(tp,tp+fp);re=div(tp,tp+fn);f=div(2*tp,2*tp+fp+fn)
    return dict(sample_f1=float(div(2*(y*p).sum(1),y.sum(1)+p.sum(1)).mean()),macro_f1=float(f.mean()),
        **{f'{k}_{label}':float(v[j]) for k,v in [('precision',pr),('recall',re),('f1',f),('support',y.sum(0))] for j,label in enumerate(LABELS)})


def report(output,write=True):
    guard();output=Path(output);ref=controls();packets=[];audits=[];equations=[];errors=[]
    for c in benchmark.cases():
        with np.load(output/'predictions'/(c['key']+'.npz')) as z:p={k:z[k] for k in z.files}
        mask=ref['case']==c['key'];np.testing.assert_array_equal(p['ids'],ref['ids'][mask]);np.testing.assert_array_equal(p['truth'],ref['truth'][mask]);np.testing.assert_array_equal(p['scores'][:3],ref['scores'][:,mask])
        np.testing.assert_allclose(p['scores'][5],.8*p['scores'][1]+.2*p['scores'][3],atol=1e-15,rtol=0)
        np.testing.assert_array_equal(p['predictions'],np.stack([decide(s,.5) for s in p['scores']]))
        meta=__import__('json').loads((output/'predictions'/(c['key']+'.json')).read_text());assert meta['protocol_sha256']==sha(HERE/'protocol.json')
        audits.append(meta['audit']);equations += [dict(case=c['key'],**e) for e in meta['equations']];packets.append(p)
    y=np.concatenate([p['truth'] for p in packets]);pred=np.concatenate([p['predictions'] for p in packets],axis=1)
    np.testing.assert_array_equal(y,ref['truth']);rows=[];supp=[]
    masks=[(dict(pool=pool,seed=int(seed)),(ref['pool']==pool)&(ref['seed']==seed)) for pool in ['train','pooled'] for seed in np.unique(ref['seed'][ref['pool']==pool])]
    masks +=[(dict(case=case),ref['case']==case) for case in np.unique(ref['case'][ref['pool']=='supplementary'])]
    masks +=[(dict(case='chronology98'),(ref['pool']=='supplementary')&(ref['case']!='validation'))]
    for i,mode in enumerate(MODES):
        for meta,mask in masks:
            values=benchmark.metrics(y[mask],pred[i,mask]);manual=arithmetic(y[mask],pred[i,mask])
            errors.extend(abs(values[k]-manual[k]) for k in values)
            (rows if 'pool' in meta else supp).append(dict(mode=mode,**meta,**values))
    assert max(errors)<1e-12
    rows,supp=pd.DataFrame(rows),pd.DataFrame(supp)
    summary=rows.groupby(['pool','mode']).agg(mean_sample_f1=('sample_f1','mean'),population_sd=('sample_f1',lambda a:a.std(ddof=0)),mean_macro_f1=('macro_f1','mean')).reset_index()
    pairs=[];deltas=[]
    for parent in MODES[:3]:
        for pool in ['train','pooled']:
            pivot=rows[rows.pool==pool].pivot(index='seed',columns='mode',values='sample_f1')
            for mode in CANDIDATES:
                d=pivot[mode]-pivot[parent];w,t,l=int((d>1e-12).sum()),int((abs(d)<=1e-12).sum()),int((d< -1e-12).sum())
                pairs.append(dict(baseline=parent,pool=pool,mode=mode,mean_delta=float(d.mean()),wins=w,ties=t,losses=l,meets_pool_rule=bool(d.mean()>0 and w>=5)))
                deltas +=[dict(baseline=parent,pool=pool,mode=mode,seed=int(seed),sample_f1_delta=float(v)) for seed,v in d.items()]
    tables=dict(repeat_scores=rows,supplementary_scores=supp,cv_summary=summary,paired_summary=pd.DataFrame(pairs),paired_repeat_deltas=pd.DataFrame(deltas),equations=pd.DataFrame(equations),fit_audits=pd.DataFrame([{k:(__import__('json').dumps(v) if isinstance(v,(list,dict)) else v) for k,v in a.items() if k!='search'} for a in audits]))
    check=dict(passed=True,matched_cases=84,new_geometry_fits=84,symbolic_heads=len(equations),metrics_independently_checked=len(errors),max_metric_error=max(errors),controls_reused=True,all_queries_excluded_from_fit=all(a['fitting_query_ids_disjoint'] for a in audits),all_observed_labels_preserved=all(a['observed_labels_preserved'] for a in audits),source_hashes_unchanged=True)
    if write:
        results=output/'results';results.mkdir(exist_ok=True)
        for name,frame in tables.items():frame.to_csv(results/(name+'.csv'),index=False)
        save(results/'verification.json',check)
    else:
        for name,frame in tables.items():
            saved=pd.read_csv(output/'results'/(name+'.csv'))
            pd.testing.assert_frame_equal(frame,saved,check_dtype=False,rtol=1e-12,atol=1e-12)
    print(summary.to_string(index=False));print(supp[['mode','case','sample_f1']].to_string(index=False));print(__import__('json').dumps(check,indent=2))
    return tables,check


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('command',choices=['register','run','report','verify']);p.add_argument('--output',type=Path,default=HERE/'run');p.add_argument('--workers',type=int,default=2);a=p.parse_args()
    if a.command=='register':register()
    elif a.command=='run':run(a.output,a.workers)
    elif a.command=='report':report(a.output)
    else:report(a.output,write=False)

Overwriting _source_export/part1/research/pysr_v1/study.py


## handin.py

In [139]:
%%writefile _source_export/handin.py
"""Portable Part 1 notebook helpers. Frozen model implementations stay unchanged."""

from pathlib import Path
import hashlib
import importlib.util
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support
from threadpoolctl import threadpool_limits

ROOT = Path(__file__).resolve().parent
P1 = ROOT / "part1"
LABELS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMB = [f"emb_{i}" for i in range(150)]
WINNER_SHA = "c20aa937fa8b1e5aea55bcd6564c6acd952f0d74030cc7c1a437eac54a1ee257"
sys.path.insert(0, str(P1))
from anchored_solution import AnchoredReasonClassifier, select_reasons


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def arrays(path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k].copy() for k in z.files}


def data():
    return [
        pd.read_csv(P1 / "data" / f"part1_{name}.csv")
        for name in ["train", "val", "test"]
    ]


def metrics(y, prediction):
    precision, recall, f, support = precision_recall_fscore_support(
        y, prediction, zero_division=0
    )
    return dict(
        sample_f1=float(f1_score(y, prediction, average="samples", zero_division=0)),
        macro_f1=float(f.mean()),
    ), pd.DataFrame(
        dict(label=LABELS, precision=precision, recall=recall, f1=f, support=support)
    )


def frame(ids, prediction):
    return pd.DataFrame(
        dict(
            ID=ids,
            Predicted=[
                " ".join(l for l, yes in zip(LABELS, row) if yes) for row in prediction
            ],
        )
    )


def verify_resources():
    manifest = json.loads((ROOT / "PACKAGE_MANIFEST.json").read_text())
    for name, record in manifest["files"].items():
        assert sha(ROOT / name) == record["sha256"], name
    return dict(
        files=len(manifest["files"]),
        hashes_match=True,
        external_checkout_required=False,
    )


def fit_winner(validation=True, output=None):
    """Fresh anchored and all five Cand2 seeds; no stored scores enter training."""
    started = time.monotonic()
    study = P1 / "experiments/cand2_submission_v1"
    sys.path[:0] = [str(study / "source"), str(study)]
    spec = importlib.util.spec_from_file_location(
        "_frozen_cand2_export", study / "build.py"
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    train, val, test = data()
    fit = train if validation else pd.concat([train, val], ignore_index=True)
    query = val if validation else test
    with threadpool_limits(limits=1):
        anchored = AnchoredReasonClassifier().fit(
            fit[EMB].to_numpy(float), fit[LABELS].to_numpy(int)
        )
        a = anchored.predict_proba(query[EMB].to_numpy(float))
        outputs = [module.one_seed((validation, seed)) for seed in range(5)]
        D = module.load(str(P1 / "data"))
        _, Q, _ = module.average(outputs, D.CM)
    c = Q @ D.CM
    score = 0.8 * a + 0.2 * c
    prediction = select_reasons(score)
    if validation:
        ref = arrays(
            P1 / "experiments/cand2_followup_v1/inputs/strict_inductive__validation.npz"
        )
        expected = select_reasons(
            0.8 * ref["anchored_scores"] + 0.2 * ref["cand2_scores"]
        )
    else:
        ref = arrays(P1 / "experiments/cand2_followup_v1/blend_final_predictions.npz")
        expected = ref["predictions"]
    np.testing.assert_array_equal(query.ID.to_numpy(str), ref["ids"])
    np.testing.assert_allclose(a, ref["anchored_scores"], atol=1e-7, rtol=0)
    np.testing.assert_allclose(c, ref["cand2_scores"], atol=1e-7, rtol=0)
    np.testing.assert_array_equal(prediction, expected)
    receipt = dict(
        validation=validation,
        gold=int(fit[LABELS].ge(0).all(axis=1).sum()),
        originally_masked=int(fit[LABELS].eq(-100).all(axis=1).sum()),
        rows=len(query),
        seeds=list(range(5)),
        fresh_anchored=True,
        fresh_cand2=True,
        test_inputs_in_fit=False,
        max_anchored_error=float(np.abs(a - ref["anchored_scores"]).max()),
        max_cand2_error=float(np.abs(c - ref["cand2_scores"]).max()),
        identical_predictions=True,
        seconds=time.monotonic() - started,
        anchored_refreshes=anchored.round_audit_,
    )
    result = frame(query.ID, prediction)
    if not validation:
        digest = hashlib.sha256(
            result.to_csv(index=False, lineterminator="\n").encode()
        ).hexdigest()
        assert digest == WINNER_SHA
        receipt["submission_sha256"] = digest
    if output is not None:
        output = Path(output)
        output.mkdir(parents=True, exist_ok=True)
        result.to_csv(
            output
            / ("validation_predictions.csv" if validation else "fresh_submission.csv"),
            index=False,
        )
        (output / "receipt.json").write_text(json.dumps(receipt, indent=2) + "\n")
        # Preserve gold; retain -100 where anchored did not accept a pseudo-label.
        augmented = fit[["ID"] + LABELS].copy()
        missing = np.flatnonzero(fit[LABELS].eq(-100).all(axis=1))
        accepted = missing[anchored.generated_mask_]
        augmented.loc[accepted, LABELS] = anchored.generated_labels_
        augmented["label_source"] = np.where(
            fit[LABELS].ge(0).all(axis=1), "supplied", "unaccepted"
        )
        augmented.loc[accepted, "label_source"] = "anchored_pseudo"
        augmented.to_csv(output / "anchored_augmented_labels.csv", index=False)
    return receipt, result


def winner_repeats():
    """Recompute complete-repeat F1, never an unweighted mean of fold F1."""
    buckets = {}
    cases = []
    for path in sorted(
        (P1 / "experiments/cand2_followup_v1/inputs").glob("strict_inductive__*.npz")
    ):
        key = path.stem.removeprefix("strict_inductive__")
        if key.startswith("grouped_"):
            _, pool, seed, fold = key.split("_")
            repeat = int(seed)
        elif key == "validation":
            pool, repeat = "validation", -1
        elif key.startswith("through_"):
            pool, repeat = "chronology98", -1
        else:
            continue
        z = arrays(path)
        for recipe, pred in [
            ("anchored", z["anchored_predictions"]),
            ("cand2", z["cand2_predictions"]),
            (
                "winner",
                select_reasons(0.8 * z["anchored_scores"] + 0.2 * z["cand2_scores"]),
            ),
        ]:
            buckets.setdefault((pool, repeat, recipe), []).append(
                (z["ids"], z["truth"], pred)
            )
        cases.append(key)
    assert len(cases) == 84
    rows = []
    for (pool, seed, recipe), chunks in buckets.items():
        ids, y, pred = [np.concatenate([c[j] for c in chunks]) for j in range(3)]
        assert len(ids) == len(set(ids))
        assert (
            len(ids)
            == {"train": 157, "pooled": 235, "validation": 78, "chronology98": 98}[pool]
        )
        scores, _ = metrics(y, pred)
        rows.append(dict(pool=pool, seed=seed, recipe=recipe, rows=len(ids), **scores))
    return pd.DataFrame(rows)


def compact_rescore(relative):
    """Rescore useful matched predictions; omitted variants keep their full summaries."""
    z = arrays(P1 / relative)
    rows = []
    for pool in np.unique(z["pool"]):
        for seed in np.unique(z["seed"][z["pool"] == pool]):
            keep = (z["pool"] == pool) & (z["seed"] == seed)
            for j, name in enumerate(z["names"]):
                m, _ = metrics(z["truth"][keep], z["predictions"][j, keep])
                rows.append(dict(pool=str(pool), seed=int(seed), recipe=str(name), **m))
    chrono = np.char.startswith(z["pool"], "through_")
    if chrono.any():
        for j, name in enumerate(z["names"]):
            m, _ = metrics(z["truth"][chrono], z["predictions"][j, chrono])
            rows.append(dict(pool="chronology98", seed=-1, recipe=str(name), **m))
    return pd.DataFrame(rows)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--fresh", choices=["validation", "final"], required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    print(json.dumps(fit_winner(args.fresh == "validation", args.output)[0], indent=2))

Overwriting _source_export/handin.py


In [140]:
for name in source_names:
    assert h.sha(EXPORT/name)==h.sha(RES/name),name
print("Verified visible source copies:",len(source_names))

Verified visible source copies: 137


## Original notebook-only study source

The historical baseline/feature studies below were authored directly as notebooks. Their full code is reproduced in Markdown code blocks for review; executing this source-export notebook does not replay their searches. The original `.ipynb` files, settings and compact results are also included as source records.

## part1/additional_methods.ipynb

```python
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
SVM_C = [0.1, 1.0, 10.0]
SVM_GAMMA = [0.001, "scale", 0.03]
NB_SMOOTHING = [1e-9, 1e-3, 1e-1]
DBSCAN_MIN_SAMPLES = [5, 10]
DBSCAN_RADIUS_QUANTILES = [0.25, 0.5, 0.75]
GRAPH_NEIGHBORS = [5, 10, 20]
GRAPH_ALPHA = [0.2, 0.8]
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import DBSCAN
from sklearn.semi_supervised import LabelSpreading
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from scipy.sparse.csgraph import connected_components
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"additional_methods"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")
```

```python
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")
```

```python
gmm_source=DATA/"experiments/method_comparison/datasets/gmm_tied_k10.csv"
gmm_frame=pd.read_csv(gmm_source,float_precision="round_trip")
assert gmm_frame.ID.tolist()==train.ID.tolist()
gmm_y=gmm_frame[REASONS].to_numpy(dtype=int)
gmm_w=gmm_frame[[f"weight_{r}" for r in REASONS]].to_numpy()
save_dataset("frozen_gmm_tied_k10",gmm_y,gmm_w)
incumbent_path=DATA/"experiments/method_comparison/classifier_models/gmm_tied_k10__logistic_C1000.joblib"
incumbent=joblib.load(incumbent_path)
incumbent_metrics=save_predictions("current_validation_best",predict_reason_models(incumbent,X_valid))
np.testing.assert_allclose(incumbent_metrics["sample_f1"],0.905982905982906,atol=1e-14)
reference_rows.append({"experiment":"current_validation_best","dataset":"frozen_gmm_tied_k10","classifier":"logistic",
                       "parameter":1000.0,"generated_trials":696,**incumbent_metrics,"kaggle_score":np.nan})
protected_hashes.update({str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in [gmm_source,incumbent_path]})
RESULTS=[]
MODELS={"current_validation_best":incumbent}
LABELER_ROWS=[]
GRAPH_MODELS={}
DB_MODELS={}


def fit_additional_classifier(dataset_name,kind,parameters):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert included[known].all() and np.isin(y[included],[0,1]).all()
        key=(kind,json.dumps(parameters,sort_keys=True),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="rbf_svm":
                estimator=SVC(kernel="rbf",C=parameters["C"],gamma=parameters["gamma"],random_state=SEED)
                model=CalibratedClassifierCV(
                    estimator=estimator,method="sigmoid",
                    cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED),
                    ensemble=False,n_jobs=1,
                )
            else:model=GaussianNB(var_smoothing=parameters["var_smoothing"])
            normalized=w[included]*known.sum()/w[included].sum()
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                model.fit(XS[included],y[included],sample_weight=normalized)
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameters":parameters.copy(),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "fitting_weight_normalization":157}


def predict_graph_model(model,raw):
    scaled=model["scaler"].transform(np.asarray(raw))
    fallback=model["fallback_model"].predict_proba(raw)
    probabilities=fallback.copy()
    fallback_count=0
    for j,graph in enumerate(model["graphs"]):
        neighbors=graph.nn_fit.kneighbors(scaled,return_distance=False)
        mass=graph.label_distributions_[neighbors].sum(axis=1)
        total=mass.sum(axis=1)
        reachable=np.isfinite(mass).all(axis=1)&(total>0)
        pos=list(graph.classes_).index(1)
        probabilities[reachable,j]=mass[reachable,pos]/total[reachable]
        fallback_count+=int((~reachable).sum())
    return np.clip(probabilities,0,1),fallback_count


def predict_candidate(model,raw):
    return predict_graph_model(model,raw)[0] if model["kind"]=="graph_direct" else predict_reason_models(model,raw)


def record_candidate(name,model,dataset_name,category,parameters,extra=None):
    probabilities=predict_candidate(model,X_valid)
    metrics=save_predictions(name,probabilities)
    joblib.dump(model,OUT/"classifier_models"/f"{name}.joblib",compress=3)
    row={"experiment":name,"category":category,"classifier":model["kind"],"dataset":dataset_name,
         "parameters":json.dumps(parameters,sort_keys=True),
         "generated_trials":np.nan if model["kind"]=="graph_direct" else DATASETS[dataset_name]["generated_trials"],
         "unlabeled_trials_used":1014 if model["kind"]=="graph_direct" else DATASETS[dataset_name]["generated_trials"],
         **metrics,"delta_vs_current_best":metrics["sample_f1"]-incumbent_metrics["sample_f1"],"kaggle_score":np.nan}
    if extra:row.update(extra)
    RESULTS.append(row);MODELS[name]=model
    print(f"{name}: sample F1={metrics['sample_f1']:.4f}, macro F1={metrics['macro_f1']:.4f}, "
          f"change={row['delta_vs_current_best']:+.4f}",flush=True)
    return row


def targets_from_probabilities(probabilities,support,valid_probability=None):
    valid_probability=np.ones_like(probabilities,dtype=bool) if valid_probability is None else valid_probability
    targets=np.full_like(original_y,-100)
    weights=np.zeros(original_y.shape)
    targets[known]=original_y[known];weights[known]=1.0
    eligible=(~known)[:,None]&(np.asarray(support)>=LABEL_RULE["minimum_support"])[:,None]&valid_probability
    for j in range(5):
        positive=eligible[:,j]&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible[:,j]&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        np_,nn=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not np_ or not nn:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],LABEL_RULE["maximum_generated_weight"]*np_/kp,
                     LABEL_RULE["maximum_generated_weight"]*nn/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/np_;weights[negative,j]=fraction*kn/nn
        assert weights[~known,j].max()<=0.25+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    return targets,weights


def safe_name(value):return str(value).replace(".","p").replace("-","m")

print("Current validation reference:",incumbent_metrics)
print("SVM calibration: sigmoid, 3 stratified training-only folds, final estimator fitted on all included training rows.")
```

```python
def run_supervised_family(kind):
    if kind=="rbf_svm":
        configs=[{"C":c,"gamma":gamma} for c in SVM_C for gamma in SVM_GAMMA]
    else:configs=[{"var_smoothing":smooth} for smooth in NB_SMOOTHING]
    with threadpool_limits(limits=1):
        for dataset_name in ["known_only","frozen_gmm_tied_k10"]:
            for parameters in configs:
                suffix="_".join(f"{k}{safe_name(v)}" for k,v in parameters.items())
                name=f"{dataset_name}__{kind}_{suffix}"
                model=fit_additional_classifier(dataset_name,kind,parameters)
                record_candidate(name,model,dataset_name,"supervised_comparison",parameters)
```

```python
run_supervised_family("rbf_svm")
```

```python
run_supervised_family("gaussian_nb")
```

```python
def run_dbscan(min_samples):
    with threadpool_limits(limits=1):
        distances=NearestNeighbors(n_neighbors=min_samples,algorithm="brute").fit(XC).kneighbors(XC)[0][:,-1]
        for quantile in DBSCAN_RADIUS_QUANTILES:
            eps=max(float(np.quantile(distances,quantile)),1e-6)
            name=f"dbscan_m{min_samples}_q{round(quantile*100):02d}"
            density=DBSCAN(eps=eps,min_samples=min_samples,metric="euclidean",algorithm="brute",n_jobs=1).fit(XC)
            ids=density.labels_
            unique=np.unique(ids[ids>=0])
            if len(unique):
                responsibilities=np.column_stack([ids==cluster for cluster in unique]).astype(float)
                counts=responsibilities[known].sum(axis=0)
                positive=responsibilities[known].T@original_y[known]
                prior=original_y[known].mean(axis=0)
                rates=(positive+LABEL_RULE["prior_strength"]*prior)/(counts[:,None]+LABEL_RULE["prior_strength"])
                probabilities=responsibilities@rates
                support=responsibilities@counts
                clusters=pd.DataFrame({"cluster":unique,"trials":responsibilities.sum(axis=0),"known_support":counts})
                for j,r in enumerate(REASONS):clusters[f"probability_{r}"]=rates[:,j]
            else:
                probabilities=np.tile(original_y[known].mean(axis=0),(len(train),1));support=np.zeros(len(train))
                clusters=pd.DataFrame(columns=["cluster","trials","known_support"])
            targets,weights=targets_from_probabilities(probabilities,support)
            valid_mask=np.repeat((ids>=0)[:,None],5,axis=1)
            save_dataset(name,targets,weights,np.where(valid_mask,probabilities,np.nan),ids,support)
            assert (weights[(ids<0)&~known]==0).all()
            clusters.to_csv(OUT/"clusters"/f"{name}.csv",index=False)
            joblib.dump({"model":density,"scaler":cluster_scaler,"eps_quantile":quantile,"label_rule":LABEL_RULE},
                        OUT/"labeling_models"/f"{name}.joblib",compress=3)
            DB_MODELS[name]=density
            model=fit_reason_models(name,"logistic",1000.0)
            extra={"labeler_family":"dbscan","eps":eps,"eps_quantile":quantile,"min_samples":min_samples,
                   "clusters":len(unique),"noise_trials":int((ids==-1).sum())}
            row=record_candidate(name+"__logistic_C1000",model,name,"label_generation",{"C":1000.0},extra)
            LABELER_ROWS.append({**row,**extra})
```

```python
run_dbscan(5)
```

```python
run_dbscan(10)
```

```python
def run_graph(neighbors,alpha):
    name=f"graph_n{neighbors}_a{safe_name(alpha)}"
    with threadpool_limits(limits=1):
        nn=NearestNeighbors(n_neighbors=neighbors,algorithm="brute",n_jobs=1).fit(XC)
        adjacency=nn.kneighbors_graph(XC,mode="connectivity")
        component_count,component=connected_components(adjacency.maximum(adjacency.T),directed=False)
        component_support=np.bincount(component[known],minlength=component_count)
        support=component_support[component]
        probabilities=np.zeros((len(train),5));reachable=np.zeros((len(train),5),dtype=bool)
        graphs=[]
        for j in range(5):
            labels=np.full(len(train),-1,dtype=int);labels[known]=original_y[known,j]
            graph=LabelSpreading(kernel="knn",n_neighbors=neighbors,alpha=alpha,max_iter=1000,tol=1e-4,n_jobs=1)
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                graph.fit(XC,labels)
            mass=graph.label_distributions_.sum(axis=1)
            reachable[:,j]=np.isfinite(graph.label_distributions_).all(axis=1)&(mass>0)
            pos=list(graph.classes_).index(1)
            probabilities[:,j]=np.divide(graph.label_distributions_[:,pos],mass,out=np.zeros(len(train)),where=mass>0)
            graphs.append(graph)
        targets,weights=targets_from_probabilities(probabilities,support,reachable)
        save_dataset(name,targets,weights,np.where(reachable,probabilities,np.nan),component,support)
        pd.DataFrame({"component":np.arange(component_count),"known_support":component_support,
                      "trials":np.bincount(component,minlength=component_count)}).to_csv(OUT/"clusters"/f"{name}.csv",index=False)
        graph_model={"kind":"graph_direct","graphs":graphs,"scaler":cluster_scaler,"fallback_model":baseline,
                     "features":FEATURES,"reasons":REASONS,"dataset":name,"parameters":{"neighbors":neighbors,"alpha":alpha}}
        GRAPH_MODELS[name]=graph_model
        joblib.dump(graph_model,OUT/"labeling_models"/f"{name}.joblib",compress=3)
        extra={"labeler_family":"label_spreading","neighbors":neighbors,"alpha":alpha,
               "graph_components":component_count,"training_reason_slots_without_label_mass":int((~reachable).sum()),
               "maximum_iterations_used":max(int(g.n_iter_) for g in graphs)}
        model=fit_reason_models(name,"logistic",1000.0)
        row=record_candidate(name+"__logistic_C1000",model,name,"label_generation",{"C":1000.0},extra)
        LABELER_ROWS.append({**row,**extra})
        _,fallback_slots=predict_graph_model(graph_model,X_valid)
        record_candidate(name+"__direct",graph_model,name,"graph_direct",graph_model["parameters"],
                         {**extra,"validation_reason_slots_using_supervised_fallback":fallback_slots})
```

```python
run_graph(5,0.2)
```

```python
run_graph(5,0.8)
```

```python
run_graph(10,0.2)
```

```python
run_graph(10,0.8)
```

```python
run_graph(20,0.2)
```

```python
run_graph(20,0.8)
```

```python
dbscan_candidates=pd.DataFrame(LABELER_ROWS)
dbscan_best=dbscan_candidates.loc[dbscan_candidates.labeler_family=="dbscan"].sort_values(
    ["sample_f1","experiment"],ascending=[False,True],kind="stable").iloc[0]
dbscan_dataset=DATASETS[dbscan_best.dataset]
gmm_dataset=DATASETS["frozen_gmm_tied_k10"]
hybrid_y=known_y.copy();hybrid_w=known_w.copy();hybrid_sources={}
for j,r in enumerate(REASONS):
    gm=bool((gmm_dataset["weights"][~known,j]>0).any())
    db=bool((dbscan_dataset["weights"][~known,j]>0).any())
    if gm:chosen=gmm_dataset;hybrid_sources[r]="frozen_gmm_tied_k10"
    elif db:chosen=dbscan_dataset;hybrid_sources[r]=dbscan_best.dataset
    else:chosen=DATASETS["known_only"];hybrid_sources[r]="known_only"
    hybrid_y[:,j]=chosen["targets"][:,j]
    hybrid_w[:,j]=chosen["weights"][:,j]
save_dataset("hybrid_gmm_dbscan",hybrid_y,hybrid_w)
with threadpool_limits(limits=1):
    hybrid_model=fit_reason_models("hybrid_gmm_dbscan","logistic",1000.0)
    hybrid_probabilities=predict_reason_models(hybrid_model,X_valid)
    db_probabilities=predict_candidate(MODELS[dbscan_best.experiment],X_valid)
    gm_probabilities=predict_reason_models(incumbent,X_valid)
    known_probabilities=baseline.predict_proba(X_valid)
for j,r in enumerate(REASONS):
    expected=(gm_probabilities if hybrid_sources[r]=="frozen_gmm_tied_k10" else
              db_probabilities if hybrid_sources[r]==dbscan_best.dataset else known_probabilities)
    np.testing.assert_allclose(hybrid_probabilities[:,j],expected[:,j],rtol=1e-8,atol=1e-10)
record_candidate("hybrid_gmm_dbscan__logistic_C1000",hybrid_model,"hybrid_gmm_dbscan","hybrid_label_generation",
                 {"C":1000.0,"label_sources":hybrid_sources})
print("Hybrid label sources:",hybrid_sources)
```

```python
initial=pd.DataFrame(RESULTS)
best_parameters={}
for kind in ["rbf_svm","gaussian_nb"]:
    row=initial.loc[(initial.classifier==kind)&(initial.category=="supervised_comparison")].sort_values(
        ["sample_f1","experiment"],ascending=[False,True],kind="stable").iloc[0]
    best_parameters[kind]=json.loads(row.parameters)
labeler_comparison=pd.DataFrame(LABELER_ROWS)
labeler_shortlist=[]
for family in ["dbscan","label_spreading"]:
    row=labeler_comparison.loc[labeler_comparison.labeler_family==family].sort_values(
        ["sample_f1","experiment"],ascending=[False,True],kind="stable").iloc[0]
    labeler_shortlist.append(row.dataset)
with threadpool_limits(limits=1):
    for dataset_name in labeler_shortlist:
        for kind,parameters in best_parameters.items():
            name=f"{dataset_name}__crossover_{kind}"
            model=fit_additional_classifier(dataset_name,kind,parameters)
            record_candidate(name,model,dataset_name,"classifier_crossover",parameters)
# Follow-up requested after the initial method comparison: KNN on the new label sets.
knn_followup_datasets=list(dict.fromkeys(labeler_shortlist+["hybrid_gmm_dbscan"]))
knn_neighbor_counts=[1,3,7,15]
knn_dbscan_reference=float(pd.DataFrame(LABELER_ROWS).loc[lambda d:d.labeler_family=="dbscan","sample_f1"].max())
with threadpool_limits(limits=1):
    for dataset_name in knn_followup_datasets:
        for neighbors in knn_neighbor_counts:
            name=f"{dataset_name}__weighted_knn_k{neighbors}"
            model=fit_reason_models(dataset_name,"weighted_knn",neighbors)
            row=record_candidate(name,model,dataset_name,"knn_followup",{"neighbors":neighbors})
            row["delta_vs_dbscan_lr"]=row["sample_f1"]-knn_dbscan_reference
results=pd.DataFrame(RESULTS)
results.to_csv(OUT/"new_comparison.csv",index=False)
results.loc[results.category=="knn_followup"].to_csv(OUT/"knn_comparison.csv",index=False)
labeler_comparison.to_csv(OUT/"labeler_comparison.csv",index=False)
pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
all_results=pd.concat([pd.DataFrame(reference_rows),results],ignore_index=True)
all_results.to_csv(OUT/"comparison.csv",index=False)
ranked=results.sort_values(["sample_f1","experiment"],ascending=[False,True],kind="stable")
best_new=ranked.iloc[0]
best_model_name=best_new.experiment if best_new.sample_f1>incumbent_metrics["sample_f1"]+1e-12 else "current_validation_best"
print("Leading model after this round:",best_model_name)
display(ranked.head(12)[["experiment","classifier","sample_f1","macro_f1","delta_vs_current_best"]].round(4))
```

```python
# Verify the graph predictor against sklearn on rows where normalization is defined.
first_graph=next(iter(GRAPH_MODELS.values()))
scaled_valid=first_graph["scaler"].transform(X_valid)
for j,graph in enumerate(first_graph["graphs"]):
    ids=graph.nn_fit.kneighbors(scaled_valid,return_distance=False)
    mass=graph.label_distributions_[ids].sum(axis=1)
    good=mass.sum(axis=1)>0
    if good.any():
        native=graph.predict_proba(scaled_valid[good])[:,list(graph.classes_).index(1)]
        custom=predict_graph_model(first_graph,X_valid[good])[0][:,j]
        np.testing.assert_allclose(native,custom,rtol=1e-12,atol=1e-12)

selected_names=[best_new.experiment]
best_label_aug=ranked.loc[ranked.category=="label_generation"].iloc[0]
if best_label_aug.experiment not in selected_names:selected_names.append(best_label_aug.experiment)
for name in selected_names:
    reloaded=joblib.load(OUT/"classifier_models"/f"{name}.joblib")
    expected=predict_candidate(MODELS[name],X_valid)
    np.testing.assert_allclose(predict_candidate(reloaded,X_valid),expected,rtol=1e-12,atol=1e-12)
    if reloaded["kind"]!="graph_direct":
        frame=pd.read_csv(OUT/"datasets"/f"{reloaded['dataset']}.csv",float_precision="round_trip")
        assert frame.ID.tolist()==train.ID.tolist()
        for j,r in enumerate(REASONS):
            np.testing.assert_array_equal(frame[r].to_numpy(),DATASETS[reloaded["dataset"]]["targets"][:,j])
            np.testing.assert_allclose(frame[f"weight_{r}"].to_numpy(),DATASETS[reloaded["dataset"]]["weights"][:,j],atol=1e-14)
        # Independently refit from exported features, labels, and weights.
        from sklearn.base import clone
        replay={**reloaded,"estimators":[]}
        saved_scaled=reloaded["scaler"].transform(frame[FEATURES].to_numpy())
        with threadpool_limits(limits=1):
            for j,r in enumerate(REASONS):
                y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy()
                included=w>0
                if reloaded["kind"]=="weighted_knn":
                    estimator={"neighbors":NearestNeighbors(n_neighbors=min(int(reloaded["parameter"]),int(included.sum())),algorithm="brute").fit(saved_scaled[included]),
                               "target":y[included],"sample_weight":w[included]}
                else:
                    estimator=clone(reloaded["estimators"][j])
                    with warnings.catch_warnings():
                        warnings.simplefilter("error",ConvergenceWarning)
                        estimator.fit(saved_scaled[included],y[included],sample_weight=w[included]*known.sum()/w[included].sum())
                replay["estimators"].append(estimator)
        np.testing.assert_array_equal(label_predictions(predict_candidate(replay,X_valid))[0],label_predictions(expected)[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

record={"features":FEATURES,"deep_learning":False,"new_features_added":False,
    "svm_grid":{"C":SVM_C,"gamma":SVM_GAMMA,"calibration":"sigmoid; 3 training-only stratified folds; ensemble=False"},
    "gaussian_nb_smoothing":NB_SMOOTHING,"dbscan_min_samples":DBSCAN_MIN_SAMPLES,
    "dbscan_radius_quantiles":DBSCAN_RADIUS_QUANTILES,"graph_neighbors":GRAPH_NEIGHBORS,"graph_alpha":GRAPH_ALPHA,
    "graph_max_iter":1000,"graph_tol":1e-4,"graph_direct_fallback":"known-only logistic for zero label mass",
    "knn_followup_datasets":knn_followup_datasets,"knn_followup_neighbors":knn_neighbor_counts,
    "knn_followup_reference_sample_f1":knn_dbscan_reference,
    "label_rule":LABEL_RULE,"classifier_scaler_fit_rows":157,"graph_cluster_scaler_fit_rows":1171,
    "fitting_weight_normalization":157,"threshold":0.5,"nonempty_fallback":True,
    "validation_rows":78,"validation_used_for_fit":False,"best_new":best_new.experiment,
    "best_overall":best_model_name,"hybrid_label_sources":hybrid_sources,"labeler_crossover_datasets":labeler_shortlist,"crossover_parameters":best_parameters,
    "protected_input_sha256":protected_hashes,
    "dataset_sha256":{str(q.relative_to(OUT)):hashlib.sha256(q.read_bytes()).hexdigest() for q in sorted((OUT/"datasets").glob("*.csv"))},
    "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},
    "kaggle_scores_new_methods":None,"reported_kaggle_best":0.87393,
}
(OUT/"settings.json").write_text(json.dumps(record,indent=2)+"\n")
lines=["# Additional classical ML methods","",
    "All models use only the same 150 embeddings, the same 78 validation trials, threshold 0.5, and nonempty fallback. "
    "No neural networks, extra PCA, year, text, or drug features were used. No Kaggle submission was made.","",
    f"Current validation reference: **{incumbent_metrics['sample_f1']:.4f}**. Best new model: **{best_new.experiment}**, "
    f"sample-wise F1 **{best_new.sample_f1:.4f}**. Leading model after this round: **{best_model_name}**.","",
    "## Best result by method","",
    "| Method | Configuration | Sample-wise F1 | Macro F1 | Change from 0.9060 reference |","|---|---|---:|---:|---:|"]
for classifier in ["rbf_svm","gaussian_nb","logistic","graph_direct","weighted_knn"]:
    group=ranked.loc[ranked.classifier==classifier]
    if len(group):
        row=group.iloc[0]
        lines.append(f"| {classifier} | {row.experiment} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.delta_vs_current_best:+.4f} |")
lines.extend(["","## Leading new configurations","",
    "| Configuration | Sample-wise F1 | Macro F1 |","|---|---:|---:|"])
for row in ranked.head(12).itertuples(index=False):lines.append(f"| {row.experiment} | {row.sample_f1:.4f} | {row.macro_f1:.4f} |")
lines.extend(["","## Method details and limits","",
    "RBF SVM compares C=0.1,1,10 and gamma=0.001,scale,0.03 on known-only and frozen GMM labels. "
    "Its probabilities are calibrated with sigmoid calibration and three stratified training-only folds; validation is excluded. "
    "GaussianNB compares smoothing 1e-9,0.001,0.1 on the same two datasets. It accepts continuous signed features; "
    "MultinomialNB is not appropriate for these embeddings.","",
    "DBSCAN compares min_samples=5,10 and radii taken from the 25th/50th/75th percentiles of training neighbor distances. "
    "Noise is excluded from generated targets. LabelSpreading compares 5,10,20 neighbors and alpha=0.2,0.8, "
    "one graph per reason. Missing labels are -1 internally for sklearn and remain -100 in saved uncertain targets. "
    "For augmentation, graph components need at least 3 known-label members and nonzero propagated mass. "
    "Both labelers reuse the fixed confidence thresholds, original-label preservation, and generated-influence budgets. "
    "Their augmented datasets train C=1000 logistic regression.","",
    "Direct graph prediction is also evaluated from each LabelSpreading fit. It sums class distributions over training neighbors "
    "and uses known-only logistic probabilities only if propagated mass is zero. It is separate from generating labels and fitting LR. "
    "The direct graph model uses the training-only graph scaler and does not use augmentation weight budgets.","",
    "A small crossover refits the best SVM and GaussianNB parameter settings on the best DBSCAN and LabelSpreading datasets. "
    "One adaptive follow-up combines GMM-generated Recruitment labels with DBSCAN-generated Efficacy labels, retaining known-only labels for the other reasons and preserving all source weights. "
    "This staged search does not establish a global maximum. All new scores are validation selections, not independently measured Kaggle scores. "
    "The reported best submitted Kaggle model remains K=2 without year or Safety boost at 0.87393.","",
    "## Artifacts and controls","",
    "`datasets/` and `label_audits/` preserve each training table and target/weight mask; `clusters/` records density or graph components. "
    "`labeling_models/` and `classifier_models/` save fitted models. `validation_predictions/` preserves probabilities and labels. "
    "`comparison.csv`, `new_comparison.csv`, `labeler_comparison.csv`, and `label_counts.csv` retain all results. "
    "`settings.json` records grids, selected models, source/data checksums, and software versions.","",
    "Controls reproduced the old baseline, preserved all known labels and protected artifacts, excluded noise and unreachable graph nodes "
    "from generated targets, matched direct graph predictions to sklearn where defined, reloaded selected models with matching predictions, and refitted selected classifiers from their saved datasets.",""])
hybrid_result=results.set_index("experiment").loc["hybrid_gmm_dbscan__logistic_C1000"]
lines.extend([
    "", "## Complementary-label hybrid", "",
    f"The one additional GMM-Recruitment/DBSCAN-Efficacy hybrid scored sample-wise F1 **{hybrid_result.sample_f1:.4f}** "
    f"and macro F1 **{hybrid_result.macro_f1:.4f}**, below both leading individual label-generation approaches. "
    "The source-probability controls confirmed that each reason's probabilities matched its selected source classifier. "
    "Sample-wise F1 and the nonempty fallback couple the five outputs, so changes that help separately need not help when combined. "
    "Retain this negative result; do not use the hybrid as the leading candidate.",
])
lines.extend(["","## KNN follow-up on the new label sets","",
    "The earlier KNN results used older datasets. A separate follow-up tests k=1,3,7,15 on the best DBSCAN and LabelSpreading datasets "
    "and the complementary-label hybrid, using the same frozen classifier scaler and distance/sample weights.","",
    "| Dataset | Best neighbor count | Sample-wise F1 | Macro F1 | Difference from DBSCAN + LR |",
    "|---|---:|---:|---:|---:|"])
knn_rows=results.loc[results.category=="knn_followup"]
for dataset_name in knn_followup_datasets:
    row=knn_rows.loc[knn_rows.dataset==dataset_name].sort_values(["sample_f1","experiment"],ascending=[False,True]).iloc[0]
    neighbors=json.loads(row.parameters)["neighbors"]
    lines.append(f"| {dataset_name} | {neighbors} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.delta_vs_dbscan_lr:+.4f} |")
lines.extend(["","All twelve follow-up configurations are saved in `knn_comparison.csv`. "
    "The results only compare these tested settings; earlier losses alone did not establish that KNN could not benefit from new labels."])
summary="\n".join(lines).rstrip()+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary)
```

```python
new_candidate_record=None
if best_model_name!="current_validation_best":
    candidate=joblib.load(OUT/"classifier_models"/f"{best_model_name}.joblib")
    test=pd.read_csv(INPUT_DIR/"part1_test.csv")
    sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    probability=predict_candidate(candidate,test[FEATURES].to_numpy())
    predicted,fallback=label_predictions(probability)
    frame=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in predicted]})
    frame=frame.set_index("ID").loc[sample.ID].reset_index()
    assert frame.shape==(312,2) and frame.Predicted.str.len().gt(0).all()
    assert frame.Predicted.map(lambda s:set(s.split()).issubset(REASONS)).all()
    path=OUT/"submissions"/f"part1_{best_model_name.replace('__','_')}_submission.csv"
    frame.to_csv(path,index=False)
    pd.testing.assert_frame_equal(pd.read_csv(path),frame)
    selected=results.set_index("experiment").loc[best_model_name]
    new_candidate_record={"experiment":best_model_name,"validation_sample_f1":float(selected.sample_f1),
        "validation_macro_f1":float(selected.macro_f1),"test_rows":len(frame),"test_fallback_trials":int(fallback.sum()),
        "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
        "model_file":f"classifier_models/{best_model_name}.joblib",
        "model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{best_model_name}.joblib").read_bytes()).hexdigest(),
        "kaggle_score":None,"kaggle_status":"prepared locally; not submitted","reported_kaggle_comparator":0.87393}
    (OUT/"submission_manifest.json").write_text(json.dumps(new_candidate_record,indent=2)+"\n")
    summary_path=OUT/"SUMMARY.md"
    text=summary_path.read_text().split("## Prepared candidate")[0].rstrip()
    text+=(f"\n\n## Prepared candidate\n\n`{path.relative_to(OUT)}` contains 312 predictions from **{best_model_name}**. "
           f"Validation sample-wise F1: **{selected.sample_f1:.4f}**; macro F1: **{selected.macro_f1:.4f}**. "
           "The file is prepared locally, not submitted. No Kaggle slot was used, and the submitted-best tag is unchanged. "
           "The model and file checksums are recorded in `submission_manifest.json`.\n")
    summary_path.write_text(text)
    print(json.dumps(new_candidate_record,indent=2))
else:
    print("No new candidate exceeded the current validation reference; retain its existing submission file.")
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
```

## part1/classifier_feature_tuning.ipynb

```python
from pathlib import Path
import ast
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
C_GRID=[1.0,10.0,100.0,1000.0]
BLOCK_WEIGHTS=[0.25,1.0,4.0]
REPRESENTATIONS=["diseases_min2","diseases_min3","diseases_min5","criteria_min3","criteria_min50","criteria_min100"]
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/classifier_features"
OUT=DATA/"experiments/classifier_feature_tuning"
for folder in ["datasets","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert len(train)==1171 and len(valid)==78 and known.sum()==157
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
fixed=pd.read_csv(SOURCE/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_targets_weights.csv").write_bytes((SOURCE/"datasets/fixed_targets_weights.csv").read_bytes())
source_settings=json.loads((SOURCE/"settings.json").read_text())
reference_sample=float(source_settings["reference_sample_f1"])
reference_macro=float(source_settings["reference_macro_f1"])
source_names=["embeddings_only"]+REPRESENTATIONS
SOURCE_MODELS={name:joblib.load(SOURCE/"models"/f"{name}.joblib") for name in source_names}
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",SOURCE/"settings.json",SOURCE/"comparison.csv",SOURCE/"datasets/fixed_targets_weights.csv"]
for name in source_names:protected_paths.extend([SOURCE/"models"/f"{name}.joblib",SOURCE/"datasets"/f"{name}.npz"])
additional=DATA/"experiments/additional_methods"
for rel in ["datasets/dbscan_m5_q25.csv","labeling_models/dbscan_m5_q25.joblib","classifier_models/dbscan_m5_q25__logistic_C1000.joblib"]:
    protected_paths.append(additional/rel)
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Frozen targets, weights, embeddings, and vocabularies; 76 classifier settings declared.")

```

```python
def disease_lists(frame):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        values=ast.literal_eval(value)
        assert isinstance(values,list)
        return sorted({str(v).strip().lower() for v in values if str(v).strip()})
    return frame.diseases_cleaned.map(parse).tolist()


def pathway_lists(frame):
    return frame.drug_pathways.fillna("").map(lambda s:sorted({v.strip() for v in str(s).split(";") if v.strip()})).tolist()


def missing_pathways(frame):
    return frame.drug_pathways.fillna("").astype(str).str.strip().eq("").to_numpy(dtype=float)[:,None]


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_pathways(frame))
    if kind=="criteria":return block["vectorizer"].transform(frame.criteria.fillna("")).tocsr()
    values=disease_lists(frame) if kind=="diseases" else pathway_lists(frame)
    allowed=set(block["encoder"].classes_)
    result=block["encoder"].transform([[v for v in row if v in allowed] for row in values]).astype(float).tocsr()
    if kind=="pathways":result=sparse.hstack([result,sparse.csr_matrix(missing_pathways(frame))],format="csr")
    assert result.shape[1]==len(block["feature_names"])
    assert np.isin(result.data,[0,1]).all()
    return result


def transform_model(model,frame):
    embeddings=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embeddings
    return sparse.hstack([sparse.csr_matrix(embeddings),transform_block(model["block"],frame)*model.get("feature_block_weight",1.0)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int)
    empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

reference_probability=predict_model(SOURCE_MODELS["embeddings_only"],valid)
reference_prediction,_=label_predictions(reference_probability)
np.testing.assert_allclose(f1_score(truth,reference_prediction,average="samples",zero_division=0),reference_sample,atol=1e-14)
ROWS=[];MATRIX_RECORDS={};BASELINE_SCORES={}


def prepare_matrix(representation,block_weight):
    source_matrix=sparse.load_npz(SOURCE/"datasets"/f"{representation}.npz")
    source_model=SOURCE_MODELS[representation]
    np.testing.assert_allclose(source_matrix[:,:150].toarray(),source_model["embedding_scaler"].transform(train[EMBEDDINGS].to_numpy()),rtol=1e-12,atol=1e-12)
    matrix=source_matrix.copy()
    is_block=matrix.indices>=150
    matrix.data[is_block]*=block_weight
    matrix_name=f"{representation}_weight{block_weight:g}".replace(".","p")
    sparse.save_npz(OUT/"datasets"/f"{matrix_name}.npz",matrix)
    (OUT/"datasets"/f"{matrix_name}_columns.json").write_bytes((SOURCE/"datasets"/f"{representation}_columns.json").read_bytes())
    MATRIX_RECORDS[matrix_name]={"representation":representation,"block_weight":block_weight,"shape":list(matrix.shape),
                               "source_file":str((SOURCE/"datasets"/f"{representation}.npz").relative_to(DATA)),
                               "file_sha256":hashlib.sha256((OUT/"datasets"/f"{matrix_name}.npz").read_bytes()).hexdigest()}
    return matrix,matrix_name


def fit_and_evaluate(representation,block_weight,C,matrix,matrix_name):
    name=f"{representation}_weight{block_weight:g}_C{C:g}".replace(".","p")
    template=SOURCE_MODELS[representation]
    model={**template,"C":C,"feature_block_weight":block_weight,"estimators":[],
           "training_matrix_file":f"datasets/{matrix_name}.npz","source_targets_file":"datasets/fixed_targets_weights.csv",
           "source_feature_model_file":str((SOURCE/"models"/f"{representation}.joblib").relative_to(DATA))}
    # Preserve dense input for the embeddings-only control, as in the reference experiment.
    fit_matrix=matrix.toarray() if representation=="embeddings_only" else matrix
    with threadpool_limits(limits=1):
        for j in range(5):
            estimator=clone(template["estimators"][j]).set_params(C=C)
            included=W[:,j]>0
            normalized=W[included,j]*157/W[included,j].sum()
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(fit_matrix[included],Y[included,j],sample_weight=normalized)
            model["estimators"].append(estimator)
    probabilities=predict_model(model,valid)
    if C==1000 and block_weight==1:
        np.testing.assert_allclose(probabilities,predict_model(template,valid),rtol=1e-8,atol=1e-10)
    predicted,empty=label_predictions(probabilities)
    sample=f1_score(truth,predicted,average="samples",zero_division=0)
    macro=f1_score(truth,predicted,average="macro",zero_division=0)
    if representation=="embeddings_only":BASELINE_SCORES[C]=sample
    known_predicted,_=label_predictions(predict_model(model,train.loc[known]))
    row={"experiment":name,"representation":representation,"feature_group":template["block"]["kind"],
         "additional_features":len(template["block"]["feature_names"]),"total_features":matrix.shape[1],
         "C":C,"block_weight":block_weight,"matrix_file":f"datasets/{matrix_name}.npz",
         "sample_f1":sample,"macro_f1":macro,"delta_vs_current_leader":sample-reference_sample,
         "delta_vs_same_C_embeddings":sample-BASELINE_SCORES[C],
         "known_training_sample_f1":f1_score(original_y[known],known_predicted,average="samples",zero_division=0),
         "fallback_trials":int(empty.sum()),"changed_trials_vs_current":int((predicted!=reference_prediction).any(axis=1).sum()),
         **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0))),
         "is_current":bool(representation=="embeddings_only" and C==1000),
         "distance_from_current_settings":abs(float(np.log10(C/1000)))+abs(float(np.log2(block_weight))),"kaggle_score":np.nan}
    ROWS.append(row)
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=predicted[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=2*(truth*predicted).sum(axis=1)/(truth.sum(axis=1)+predicted.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
    pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
    return row


def run_representation(representation):
    weights=[1.0] if representation=="embeddings_only" else BLOCK_WEIGHTS
    for weight in weights:
        matrix,matrix_name=prepare_matrix(representation,weight)
        for C in C_GRID:fit_and_evaluate(representation,weight,C,matrix,matrix_name)
    subset=pd.DataFrame(ROWS).query("representation==@representation")
    best_row=subset.sort_values("sample_f1",ascending=False).iloc[0]
    print(f"{representation}: {len(subset)} configurations, best sample F1={best_row.sample_f1:.6f}, "
          f"macro F1={best_row.macro_f1:.6f}, C={best_row.C:g}, weight={best_row.block_weight:g}, "
          f"change vs current={best_row.delta_vs_current_leader:+.6f}",flush=True)

run_representation("embeddings_only")

```

```python
for representation in ["diseases_min2","diseases_min3","diseases_min5"]:run_representation(representation)

```

```python
for representation in ["criteria_min3","criteria_min50","criteria_min100"]:run_representation(representation)

```

```python
comparison=pd.DataFrame(ROWS);assert len(comparison)==76
ranked=comparison.sort_values(["sample_f1","is_current","additional_features","distance_from_current_settings","experiment"],
                             ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_representation=ranked.groupby("representation",sort=False).head(1)
best_by_group=ranked.groupby("feature_group",sort=False).head(1)
best_by_representation.to_csv(OUT/"best_by_representation.csv",index=False)
best_by_group.to_csv(OUT/"best_by_group.csv",index=False)
comparison.query("representation=='embeddings_only'").to_csv(OUT/"embedding_controls.csv",index=False)
print("Selected:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(best_by_representation[["representation","C","block_weight","additional_features","sample_f1","macro_f1","delta_vs_current_leader","delta_vs_same_C_embeddings"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
selected_prediction,_=label_predictions(selected_probability)
# Reload and independently rescore every saved label set.
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    predictions=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy()
    np.testing.assert_array_equal(label_predictions(probs)[0],predictions)
    np.testing.assert_allclose(f1_score(truth,predictions,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predictions,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    if row.block_weight==1 and row.C==1000:
        np.testing.assert_allclose(probs,predict_model(SOURCE_MODELS[row.representation],valid),rtol=1e-8,atol=1e-10)
# Winner preprocessing must recreate its exact saved matrix.
persisted=sparse.load_npz(OUT/best.matrix_file)
expected=sparse.csr_matrix(transform_model(selected,train))
difference=persisted-expected
assert not difference.nnz or abs(difference.data).max()<1e-12
# Independently refit winner from persisted matrix and targets/weights.
fixed_replay=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed_replay.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(fixed_replay[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(fixed_replay[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
fit_matrix=persisted.toarray() if selected["block"]["kind"]=="none" else persisted
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=fixed_replay[r].to_numpy(dtype=int);w=fixed_replay[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(fit_matrix[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],selected_prediction)
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
print("All 76 saved label sets rescored; original settings reproduced; winner preprocessing/refit and source checks passed.")

```

```python
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"C_grid":C_GRID,"block_weight_grid":BLOCK_WEIGHTS,"representations":REPRESENTATIONS,
          "configurations":76,"reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "source_experiment":"classifier_features","generated_labels_changed":False,"generated_weights_changed":False,
          "preprocessing_refitted":False,"features_added_to":"classifier only","new_feature_combinations":False,
          "embedding_scaler_fit_rows":157,"original_vocabulary_idf_fit_rows":1171,"validation_rows":78,"validation_used_for_fit":False,
          "fitting_weight_total_per_reason":157,"cutoff":0.5,"fallback":True,
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,"matrices":MATRIX_RECORDS,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "selection":"sample F1; ties prefer current leader, fewer features, then original settings",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(frame):
    lines=["| Representation | C | Block weight | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |",
           "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for row in frame.itertuples(index=False):
        lines.append(f"| {row.representation} | {row.C:g} | {row.block_weight:g} | {row.additional_features} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_current_leader:+.6f} | {row.delta_vs_same_C_embeddings:+.6f} |")
    return "\n".join(lines)
lines=["# Regularization and added-feature weighting","",
       "Grid: C=1,10,100,1000; added-feature multipliers=0.25,1,4. Six existing representations (three diseases, three criteria) "
       "each have 12 settings, plus four embeddings-only controls: 76 configurations. No pathway or feature-combination tests in this round.","",
       "All embeddings, vocabularies, IDF, generated targets and weights are frozen from the previous experiment. "
       "Only classifier C and the added-column multiplier vary. Smaller C means stronger regularization. "
       "The multiplier changes relative regularization cost, not a prescribed multiplier of the feature's prediction effect. "
       "Cutoff 0.5, highest-probability fallback, and fitting-weight total 157 per reason stay fixed.","",
       f"Reference sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best by representation","",table(best_by_representation),"",
       "## Matched embeddings-only controls","",table(comparison.query("representation=='embeddings_only'")),"",
       "## All configurations","",table(ranked),"",
       "## Verification and interpretation","",
       "All 76 saved prediction sets were reloaded and rescored. Every C=1000, weight=1 setting reproduces its previous "
       "probabilities. The winner's preprocessing recreates its exact saved feature matrix, and independent refitting "
       "from that matrix and the fixed targets/weights gives matching validation probabilities. Protected earlier artifacts retain their hashes.","",
       "For every added-feature setting, delta_vs_same_C_embeddings distinguishes an improvement due to added information "
       "from one due to changing regularization alone. Selection uses the same repeatedly consulted 78 validation trials; "
       "it does not establish independent test improvement. Models remain alternatives using one new feature group at a time.","",
       "Exact weighted sparse matrices and ordered feature names are saved once per representation/weight in datasets/, "
       "with the shared trial IDs, targets, and raw weights. Each model references its matrix and persists the original "
       "native sklearn preprocessing plus the new block weight. comparison.csv, best_by_representation.csv, "
       "best_by_group.csv, embedding_controls.csv, and settings.json preserve results and checksums.","",
       "No Kaggle submission, commit, or tag was made.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No configuration improves the primary validation metric; retain the current embeddings-only leader.")
summary="\n".join(lines)+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print("Selected:",best.experiment)
print(table(best_by_representation))
print("Candidate:",candidate)

```

## part1/classifier_features.ipynb

```python
from pathlib import Path
import ast
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
DISEASE_MIN_COUNTS=[2,3,5]
PATHWAY_MIN_COUNTS=[1,5,30]
TEXT_MIN_COUNTS=[3,50,100]
TEXT_MAX_FEATURES=2000
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT=DATA/"experiments/classifier_features"
for folder in ["profiles","vocabularies","datasets","models","validation_predictions","submissions"]:
    (OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
catalog=pd.read_csv(INPUT_DIR/"pathway_catalog.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
SOURCE=DATA/"experiments/additional_methods"
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
source_model_path=SOURCE/source_record["model_file"]
source_model=joblib.load(source_model_path)
embedding_scaler=source_model["scaler"]
assert embedding_scaler.n_samples_seen_==157
fixed_frame=pd.read_csv(SOURCE/"datasets/dbscan_m5_q25.csv",float_precision="round_trip")
assert fixed_frame.ID.tolist()==train.ID.tolist()
np.testing.assert_allclose(fixed_frame[EMBEDDINGS].to_numpy(),train[EMBEDDINGS].to_numpy(),rtol=1e-12,atol=1e-15)
Y=fixed_frame[REASONS].to_numpy(dtype=int)
W=fixed_frame[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known])
np.testing.assert_array_equal(W[known],np.ones((known.sum(),5)))
assert np.array_equal(W>0,Y!=-100)
assert np.isin(Y,[-100,0,1]).all()
targets_frame=fixed_frame[["ID"]+[c for r in REASONS for c in [r,f"weight_{r}"]]].copy()
targets_frame.to_csv(OUT/"datasets/fixed_targets_weights.csv",index=False)
assert int(((W>0).any(axis=1)&~known).sum())==133
protected_paths=[INPUT_DIR/f for f in ["part1_train.csv","part1_val.csv","pathway_catalog.csv"]]+[
    source_model_path,SOURCE/"datasets/dbscan_m5_q25.csv",SOURCE/"labeling_models/dbscan_m5_q25.joblib",
    SOURCE/source_record["submission_file"],SOURCE/"submission_manifest.json"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
reference_scaled=embedding_scaler.transform(valid[EMBEDDINGS].to_numpy())
reference_probability=np.column_stack([e.predict_proba(reference_scaled)[:,list(e.classes_).index(1)] for e in source_model["estimators"]])
print("Frozen targets and weights: 157 fully labeled trials +133 trials with generated Efficacy labels only.")

```

```python
def disease_lists(frame):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        values=ast.literal_eval(value)
        assert isinstance(values,list)
        return sorted({str(v).strip().lower() for v in values if str(v).strip()})
    return frame.diseases_cleaned.map(parse).tolist()


def pathway_lists(frame):
    return frame.drug_pathways.fillna("").map(lambda s:sorted({v.strip() for v in str(s).split(";") if v.strip()})).tolist()


def missing_pathways(frame):
    return frame.drug_pathways.fillna("").astype(str).str.strip().eq("").to_numpy(dtype=float)[:,None]

train_diseases=disease_lists(train)
disease_counts=pd.Series([v for row in train_diseases for v in row]).value_counts().rename_axis("disease").rename("training_trials")
disease_known_counts=pd.Series([v for row in np.array(train_diseases,dtype=object)[known] for v in row]).value_counts()
disease_profile=disease_counts.to_frame()
disease_profile["known_label_trials"]=disease_known_counts.reindex(disease_counts.index,fill_value=0).astype(int)
disease_profile.to_csv(OUT/"profiles/disease_frequencies.csv")
train_pathways=pathway_lists(train)
all_catalog_ids=set(catalog.reactome_id)
assert set(v for row in train_pathways for v in row)<=all_catalog_ids
broad_catalog=catalog.loc[catalog.pathway_level==0].sort_values("reactome_id")
assert len(broad_catalog)==29
pathway_profile=broad_catalog[["reactome_id","pathway_name","pathway_level"]].copy()
pathway_profile["training_trials"]=[sum(pid in row for row in train_pathways) for pid in pathway_profile.reactome_id]
pathway_profile["known_label_trials"]=[sum(pid in train_pathways[i] for i in np.flatnonzero(known)) for pid in pathway_profile.reactome_id]
pathway_profile.to_csv(OUT/"profiles/broad_pathway_frequencies.csv",index=False)
count_vectorizer=CountVectorizer(lowercase=True,strip_accents="unicode",ngram_range=(1,2),stop_words=None)
counts=count_vectorizer.fit_transform(train.criteria.fillna(""))
text_df=np.asarray((counts>0).sum(axis=0)).ravel()
text_profile=pd.DataFrame({"term":count_vectorizer.get_feature_names_out(),"training_trials":text_df,
                           "total_occurrences":np.asarray(counts.sum(axis=0)).ravel()})
text_profile.to_csv(OUT/"profiles/criteria_term_frequencies.csv",index=False)
profile_rows=[]
for minimum in [1,2,3,5,10,20,30,50,75,100]:
    profile_rows.append({"minimum_training_trials":minimum,"disease_features":int((disease_counts>=minimum).sum()),
                         "broad_pathway_features":int((pathway_profile.training_trials>=minimum).sum()),
                         "criteria_terms_before_cap":int(((text_df>=minimum)&(text_df<=0.95*len(train))).sum())})
pd.DataFrame(profile_rows).to_csv(OUT/"profiles/frequency_cutoff_sizes.csv",index=False)
print("Disease vocabulary counts:",{m:int((disease_counts>=m).sum()) for m in DISEASE_MIN_COUNTS})
print("Broad pathway counts:",{m:int((pathway_profile.training_trials>=m).sum()) for m in PATHWAY_MIN_COUNTS})
print("Criteria raw vocabulary:",len(text_df),"terms in one trial:",int((text_df==1).sum()))
del counts,count_vectorizer

```

```python
def make_block(kind,minimum=None):
    if kind=="none":return {"kind":"none","feature_names":[]}
    if kind=="diseases":
        classes=sorted(disease_counts[disease_counts>=minimum].index.tolist())
        encoder=MultiLabelBinarizer(classes=classes,sparse_output=True)
        encoder.fit(train_diseases)
        return {"kind":kind,"minimum_training_trials":minimum,"encoder":encoder,
                "feature_names":[f"disease__{v}" for v in classes]}
    if kind=="pathways":
        classes=pathway_profile.loc[pathway_profile.training_trials>=minimum,"reactome_id"].tolist()
        encoder=MultiLabelBinarizer(classes=classes,sparse_output=True)
        encoder.fit(train_pathways)
        return {"kind":kind,"minimum_training_trials":minimum,"encoder":encoder,
                "feature_names":[f"pathway__{v}" for v in classes]+["pathway_annotation_missing"]}
    if kind=="missing_only":return {"kind":kind,"feature_names":["pathway_annotation_missing"]}
    if kind=="criteria":
        vectorizer=TfidfVectorizer(lowercase=True,strip_accents="unicode",ngram_range=(1,2),stop_words=None,
                    min_df=minimum,max_df=0.95,max_features=TEXT_MAX_FEATURES,sublinear_tf=True,norm="l2")
        vectorizer.fit(train.criteria.fillna(""))
        return {"kind":kind,"minimum_training_trials":minimum,"vectorizer":vectorizer,
                "feature_names":[f"criteria__{v}" for v in vectorizer.get_feature_names_out()]}
    raise ValueError(kind)


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_pathways(frame))
    if kind=="criteria":return block["vectorizer"].transform(frame.criteria.fillna("")).tocsr()
    values=disease_lists(frame) if kind=="diseases" else pathway_lists(frame)
    allowed=set(block["encoder"].classes_)
    result=block["encoder"].transform([[v for v in row if v in allowed] for row in values]).astype(float).tocsr()
    if kind=="pathways":result=sparse.hstack([result,sparse.csr_matrix(missing_pathways(frame))],format="csr")
    assert result.shape[1]==len(block["feature_names"])
    assert np.isin(result.data,[0,1]).all()
    return result


def transform_model(model,frame):
    embeddings=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embeddings
    return sparse.hstack([sparse.csr_matrix(embeddings),transform_block(model["block"],frame)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int)
    empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

reference_prediction,_=label_predictions(reference_probability)
reference_sample=f1_score(truth,reference_prediction,average="samples",zero_division=0)
reference_macro=f1_score(truth,reference_prediction,average="macro",zero_division=0)
np.testing.assert_allclose(reference_sample,source_record["validation_sample_f1"],atol=1e-14)
np.testing.assert_allclose(reference_macro,source_record["validation_macro_f1"],atol=1e-14)
ROWS=[];MODELS={};FEATURE_HASHES={}


def run_configuration(kind,minimum=None):
    name="embeddings_only" if kind=="none" else ("pathway_missing_only" if kind=="missing_only" else f"{kind}_min{minimum}")
    block=make_block(kind,minimum)
    model={"kind":"embeddings_plus_features_logistic","C":1000.0,"embedding_scaler":embedding_scaler,
           "embedding_features":EMBEDDINGS,"reasons":REASONS,"block":block,"estimators":[],
           "classifier_weight_normalization_total":157,"cutoff":0.5,"nonempty_fallback":True,
           "source_model_file":str(source_model_path.relative_to(DATA)),
           "source_targets_file":"datasets/fixed_targets_weights.csv","feature_block_weight":1.0}
    matrix=transform_model(model,train)
    saved_matrix=sparse.csr_matrix(matrix)
    sparse.save_npz(OUT/"datasets"/f"{name}.npz",saved_matrix)
    feature_names=EMBEDDINGS+block["feature_names"]
    (OUT/"datasets"/f"{name}_columns.json").write_text(json.dumps(feature_names,indent=2)+"\n")
    assert matrix.shape==(len(train),len(feature_names)) and np.isfinite(saved_matrix.data).all()
    assert len(set(feature_names))==len(feature_names)
    with threadpool_limits(limits=1):
        for j in range(5):
            estimator=clone(source_model["estimators"][j]);included=W[:,j]>0
            assert estimator.C==1000.0 and included[known].all()
            normalized=W[included,j]*157/W[included,j].sum()
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(matrix[included],Y[included,j],sample_weight=normalized)
            model["estimators"].append(estimator)
    probabilities=predict_model(model,valid)
    if kind=="none":np.testing.assert_allclose(probabilities,reference_probability,rtol=1e-8,atol=1e-10)
    predicted,empty=label_predictions(probabilities)
    metrics={"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
             "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
             **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))}
    # These are apparent training scores on the original, fully labeled trials only.
    known_probability=predict_model(model,train.loc[known])
    known_prediction,_=label_predictions(known_probability)
    known_train_f1=f1_score(original_y[known],known_prediction,average="samples",zero_division=0)
    feature_hash=hashlib.sha256(saved_matrix.data.tobytes()+saved_matrix.indices.tobytes()+saved_matrix.indptr.tobytes()+str(saved_matrix.shape).encode()).hexdigest()
    FEATURE_HASHES[name]=feature_hash
    ROWS.append({"experiment":name,"feature_group":kind,"minimum_training_trials":minimum,
                 "additional_features":len(block["feature_names"]),"total_features":len(feature_names),
                 "training_preprocessing_rows":1171,"known_training_sample_f1":known_train_f1,
                 **metrics,"fallback_trials":int(empty.sum()),"changed_validation_trials":int((predicted!=reference_prediction).any(axis=1).sum()),
                 "delta_sample_f1":metrics["sample_f1"]-reference_sample,"delta_macro_f1":metrics["macro_f1"]-reference_macro,
                 "matrix_sha256":feature_hash,"kaggle_score":np.nan})
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=predicted[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=np.divide(2*(truth*predicted).sum(axis=1),truth.sum(axis=1)+predicted.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3);MODELS[name]=model
    if kind in ["diseases","pathways"]:
        key="disease" if kind=="diseases" else "reactome_id"
        vocabulary=pd.DataFrame({key:block["encoder"].classes_})
        if kind=="diseases":vocabulary=vocabulary.merge(disease_profile.reset_index(),on=key,how="left")
        else:vocabulary=vocabulary.merge(pathway_profile,on=key,how="left")
        vocabulary.to_csv(OUT/"vocabularies"/f"{name}.csv",index=False)
    elif kind=="criteria":
        terms=block["vectorizer"].get_feature_names_out()
        vocabulary=pd.DataFrame({"term":terms,"idf":block["vectorizer"].idf_}).merge(text_profile,on="term",how="left")
        vocabulary.to_csv(OUT/"vocabularies"/f"{name}.csv",index=False)
    print(f"{name}: +{len(block['feature_names'])} features, sample F1={metrics['sample_f1']:.6f}, "
          f"macro F1={metrics['macro_f1']:.6f}, change={metrics['sample_f1']-reference_sample:+.6f}",flush=True)

run_configuration("none")

```

```python
for minimum in DISEASE_MIN_COUNTS:run_configuration("diseases",minimum)

```

```python
run_configuration("missing_only")
for minimum in PATHWAY_MIN_COUNTS:run_configuration("pathways",minimum)

```

```python
for minimum in TEXT_MIN_COUNTS:run_configuration("criteria",minimum)

```

```python
comparison=pd.DataFrame(ROWS)
assert len(comparison)==11
comparison.to_csv(OUT/"comparison.csv",index=False)
ranked=comparison.assign(is_reference=comparison.experiment.eq("embeddings_only")).sort_values(
    ["sample_f1","is_reference","additional_features","experiment"],ascending=[False,False,True,True],kind="stable")
best=ranked.iloc[0]
best_by_group=ranked.groupby("feature_group",sort=False).head(1)
best_by_group.to_csv(OUT/"best_by_group.csv",index=False)
print("Best overall:",best.experiment,"sample F1:",best.sample_f1)
display(comparison[["experiment","additional_features","sample_f1","macro_f1","delta_sample_f1","fallback_trials"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
np.testing.assert_allclose(selected_probability,predict_model(MODELS[best.experiment],valid),rtol=1e-12,atol=1e-12)
# Every saved matrix must exactly reproduce its model's training preprocessing.
for name,model in MODELS.items():
    persisted=sparse.load_npz(OUT/"datasets"/f"{name}.npz")
    expected=sparse.csr_matrix(transform_model(model,train))
    difference=persisted-expected
    assert not difference.nnz or abs(difference.data).max()<1e-12
    assert persisted.shape[0]==len(targets_frame)
    frame=pd.read_csv(OUT/"validation_predictions"/f"{name}.csv")
    predicted=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    expected_prediction=label_predictions(predict_model(joblib.load(OUT/"models"/f"{name}.joblib"),valid))[0]
    np.testing.assert_array_equal(predicted,expected_prediction)
    row=comparison.loc[comparison.experiment==name].iloc[0]
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
# Independently refit the selected model from its saved matrix and raw targets/weights.
replay_matrix=sparse.load_npz(OUT/"datasets"/f"{best.experiment}.npz")
if selected["block"]["kind"]=="none":replay_matrix=replay_matrix.toarray()
fixed_replay=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed_replay.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(fixed_replay[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(fixed_replay[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=fixed_replay[r].to_numpy(dtype=int);w=fixed_replay[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(replay_matrix[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],label_predictions(selected_probability)[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
print("All saved matrices, reloaded predictions, fixed targets/weights, reference reproduction, and independent winner refit passed.")

```

```python
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    test_probability=predict_model(selected,test)
    prediction,empty=label_predictions(test_probability)
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_logistic_C1000_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"reference_experiment":source_record["experiment"],"reference_model_file":str(source_model_path.relative_to(DATA)),
          "reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "features_changed_in":"classifier only","generated_labels_changed":False,"generated_weights_changed":False,
          "embedding_scaler_fit_rows":157,"new_preprocessing_fit_rows":1171,"preprocessing_uses_validation":False,
          "disease_minimum_training_counts":DISEASE_MIN_COUNTS,"pathway_minimum_training_counts":PATHWAY_MIN_COUNTS,
          "pathway_level":0,"pathway_missing_indicator":True,"pathway_missing_only_control":True,
          "text_minimum_training_counts":TEXT_MIN_COUNTS,"text_max_features":TEXT_MAX_FEATURES,
          "text_ngram_range":[1,2],"text_max_df":0.95,"text_stop_words":None,"text_sublinear_tf":True,"text_norm":"l2",
          "binary_feature_scaling":"0/1; no centering or variance scaling","new_feature_block_weight":1.0,
          "classifier_C":1000,"fitting_weight_total_per_reason":157,"prediction_cutoff":0.5,"nonempty_fallback":True,
          "validation_rows":78,"configurations":len(comparison),"selection":"sample-wise F1; ties favor reference, then fewer added features",
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,"matrix_sha256":FEATURE_HASHES,
          "saved_dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Classifier feature additions with frozen DBSCAN pseudo-labels","",
       f"Reference: sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       "Each feature group is added separately to the same 150 embeddings. No clustering, target, or generated-weight changes. "
       "The same 157 known-label trials and 133 trials with generated Efficacy labels are used; each reason retains its own targets/weights.","",
       "## Training-only frequency inspection","",
       f"Diseases: {len(disease_counts)} distinct normalized names; minimum counts 2/3/5 retain "
       f"{int((disease_counts>=2).sum())}/{int((disease_counts>=3).sum())}/{int((disease_counts>=5).sum())} indicators. "
       "Counts are numbers of distinct training trials, not repeated mentions.","",
       "Pathways: 29 broad level-0 categories. Count thresholds 1/5/30 retain 29/27/26 categories; each adds a missing-annotation indicator. "
       "That indicator is also evaluated alone. Missing annotation does not imply no drug or no biological activity.","",
       f"Criteria: {len(text_profile)} unigram/bigram terms, including {int((text_profile.training_trials==1).sum())} occurring in one trial. "
       "With a 2,000-term cap, low cutoffs such as 3/5/10 would select nearly identical vocabularies. "
       "Minimum document counts 3/50/100 were chosen to test progressively stronger filtering. All criteria vocabularies use "
       "maximum DF 95%, sublinear TF, row L2 normalization, and preserved negation words. No stop-word list is used.","",
       "## Validation results","",
       "| Configuration | Added features | Sample F1 | Macro F1 | Change in sample F1 | Fallback trials |",
       "|---|---:|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    lines.append(f"| {row.experiment} | {row.additional_features} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_sample_f1:+.6f} | {row.fallback_trials} |")
lines.extend(["",f"Selected: **{best.experiment}**, sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Controls and interpretation","",
       "The embedding scaler stays frozen from the 157 originally labeled trials. New feature vocabularies and IDF are "
       "fit only on all 1,171 training trials, using no validation/test documents or labels. Binary features stay 0/1; "
       "TF-IDF uses L2 row normalization. New blocks have weight 1. C=1000, normalized fitting-weight total 157, "
       "final cutoff 0.5, and highest-probability fallback stay fixed. No feature combinations are evaluated in this round.","",
       "All saved sparse feature matrices reproduce their preprocessing. Saved models reload with matching predictions, "
       "the embeddings-only control reproduces the original probabilities, and the selected model independently refits "
       "from its persisted matrix and targets/weights with matching probabilities. Protected original artifacts retain their hashes.","",
       "Performance differences reflect these specific encodings/filtering settings at fixed C and feature-block scale; "
       "a negative result does not establish that a raw feature can never help. This is exploratory selection on the same "
       "78 validation trials, not an independent test estimate.","",
       "`profiles/` preserves frequency distributions and counts; `vocabularies/` records accepted terms/categories. "
       "`datasets/` saves each exact sparse classifier matrix with ordered feature names, and the shared trial IDs/targets/weights. "
       "`models/` saves native sklearn preprocessing and estimators; `validation_predictions/` saves per-trial outputs. "
       "`comparison.csv`, `best_by_group.csv`, and `settings.json` preserve scores, settings, and checksums.","",
       "No Kaggle submission, commit, or tag was made.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No addition improved the primary metric; retain the embeddings-only DBSCAN model as the leader.")
summary="\n".join(lines)+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary[:4500])

```

## part1/method_comparison.ipynb

```python
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
K_VALUES = [2, 5, 8, 10, 15]
LABEL_RULE = {"minimum_support": 3.0, "prior_strength": 2.0,
              "positive_threshold": 0.6, "negative_threshold": 0.1,
              "maximum_generated_weight": 0.25, "maximum_class_fraction": 0.5}
GMM_RULE = {"n_init": 3, "max_iter": 500, "tol": 1e-3,
            "reg_covar": 1e-3, "random_state": SEED, "init_params": "kmeans"}
CLASSIFIER_GRID = [("logistic", c) for c in [100.0, 1000.0, 10000.0]] + [
    ("weighted_knn", k) for k in [1, 3, 7, 15]
]
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"method_comparison"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")
```

```python
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"]=="logistic":
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
legacy=pd.read_csv(DATA/"experiments/kmeans/datasets/k02_train.csv").set_index("ID")
legacy_y=known_y.copy();legacy_w=known_w.copy()
positions=pd.Index(train.ID).get_indexer(legacy.index)
assert (positions>=0).all()
legacy_y[positions]=legacy[REASONS].to_numpy(dtype=int)
legacy_w[positions]=legacy.training_weight.to_numpy()[:,None]
save_dataset("legacy_k02",legacy_y,legacy_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")
```

```python
cluster_rows=[]
cluster_models={}
stage1_models={}


def run_clustering_candidate(family,k):
    name=f"{family}_k{k:02d}"
    with threadpool_limits(limits=1):
        if family=="kmeans":
            packet=joblib.load(DATA/"experiments/kmeans/models"/f"k{k:02d}.joblib")
            clustering=packet["kmeans"]
            ids=clustering.predict(XC)
            audit=pd.read_csv(DATA/"experiments/kmeans/label_audits"/f"k{k:02d}_labels.csv")
            assert audit.ID.tolist()==train.ID.tolist()
            np.testing.assert_array_equal(ids,audit.kmeans_cluster.to_numpy())
            responsibilities=np.eye(k)[ids]
            fit_info={"converged":True,"iterations":int(clustering.n_iter_),"inertia":float(clustering.inertia_)}
        else:
            covariance="diag" if family=="gmm_diag" else "tied"
            clustering=GaussianMixture(n_components=k,covariance_type=covariance,**GMM_RULE)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always",ConvergenceWarning)
                clustering.fit(XC)
            fit_info={"converged":bool(clustering.converged_),"iterations":int(clustering.n_iter_),
                      "bic":float(clustering.bic(XC)),"lower_bound":float(clustering.lower_bound_)}
            if not clustering.converged_:
                cluster_rows.append({"experiment":name,"family":family,"k":k,**fit_info,"status":"excluded_unconverged"})
                print(name,"excluded because GMM did not converge.",flush=True)
                return
            responsibilities=clustering.predict_proba(XC)
        targets,weights,probabilities,support,components=partial_labels(responsibilities)
        dataset=save_dataset(name,targets,weights,probabilities,responsibilities.argmax(axis=1),support)
        components.to_csv(OUT/"clusters"/f"{name}.csv",index=False)
        model=fit_reason_models(name,"logistic",1000.0)
        metrics=save_predictions("stage1_"+name,predict_reason_models(model,X_valid))
        stage1_models[name]=model
        cluster_models[name]=clustering
        joblib.dump({"clustering":clustering,"scaler":cluster_scaler,"family":family,"k":k,"label_rule":LABEL_RULE},
                    OUT/"clustering_models"/f"{name}.joblib",compress=3)
        joblib.dump(model,OUT/"classifier_models"/f"stage1_{name}.joblib",compress=3)
        cluster_rows.append({"experiment":name,"family":family,"k":k,"generated_trials":dataset["generated_trials"],
                             **fit_info,**metrics,"status":"ok"})
        print(f"{name}: generated trials={dataset['generated_trials']}, sample F1={metrics['sample_f1']:.4f}, "
              f"macro F1={metrics['macro_f1']:.4f}",flush=True)
```

```python
for k in K_VALUES:
    run_clustering_candidate("kmeans",k)
```

```python
run_clustering_candidate("gmm_diag",2)
```

```python
run_clustering_candidate("gmm_diag",5)
```

```python
run_clustering_candidate("gmm_diag",8)
```

```python
run_clustering_candidate("gmm_diag",10)
```

```python
run_clustering_candidate("gmm_diag",15)
```

```python
run_clustering_candidate("gmm_tied",2)
```

```python
run_clustering_candidate("gmm_tied",5)
```

```python
run_clustering_candidate("gmm_tied",8)
```

```python
run_clustering_candidate("gmm_tied",10)
```

```python
run_clustering_candidate("gmm_tied",15)
```

```python
cluster_comparison=pd.DataFrame(cluster_rows)
cluster_comparison.to_csv(OUT/"clustering_comparison.csv",index=False)
label_counts=pd.DataFrame(LABEL_COUNTS)
label_counts.to_csv(OUT/"label_counts.csv",index=False)
shortlist=["known_only","legacy_k02"]
for family in ["kmeans","gmm_diag","gmm_tied"]:
    candidates=cluster_comparison.loc[(cluster_comparison.family==family)&(cluster_comparison.status=="ok")]
    if len(candidates):
        best=candidates.sort_values(["sample_f1","k"],ascending=[False,True],kind="stable").iloc[0]
        shortlist.append(best.experiment)
print("Datasets shortlisted for classifier comparison:",shortlist)
display(cluster_comparison[["experiment","generated_trials","sample_f1","macro_f1","status"]].round(4))
display(label_counts.loc[label_counts.dataset.isin(shortlist)].pivot(index="dataset",columns="reason",values="generated_positives"))
```

```python
classifier_rows=[]
classifier_models={}
with threadpool_limits(limits=1):
    for dataset_name in shortlist:
        for kind,parameter in CLASSIFIER_GRID:
            suffix=f"C{int(parameter)}" if kind=="logistic" else f"k{int(parameter)}"
            name=f"{dataset_name}__{kind}_{suffix}"
            model=fit_reason_models(dataset_name,kind,parameter)
            probabilities=predict_reason_models(model,X_valid)
            metrics=save_predictions(name,probabilities)
            classifier_models[name]=model
            joblib.dump(model,OUT/"classifier_models"/f"{name}.joblib",compress=3)
            classifier_rows.append({"experiment":name,"dataset":dataset_name,"classifier":kind,"parameter":parameter,
                                    "generated_trials":DATASETS[dataset_name]["generated_trials"],**metrics,"kaggle_score":np.nan})
            print(f"{name}: sample F1={metrics['sample_f1']:.4f}, macro F1={metrics['macro_f1']:.4f}",flush=True)
classifier_comparison=pd.DataFrame(classifier_rows)
classifier_comparison.to_csv(OUT/"classifier_comparison.csv",index=False)
ranking=classifier_comparison.sort_values(["sample_f1","generated_trials","experiment"],ascending=[False,True,True],kind="stable")
all_results=pd.concat([pd.DataFrame(reference_rows),classifier_comparison],ignore_index=True)
all_results.to_csv(OUT/"comparison.csv",index=False)
display(ranking.head(12)[["experiment","sample_f1","macro_f1","generated_trials"]].round(4))
```

```python
# Controls for the weighted-neighbor implementation and saved-data replay.
with threadpool_limits(limits=1):
    native=KNeighborsClassifier(n_neighbors=7,weights="distance",algorithm="brute").fit(XS[known],original_y[known,0])
    custom=fit_reason_models("known_only","weighted_knn",7)
    np.testing.assert_allclose(predict_reason_models(custom,X_valid)[:,0],native.predict_proba(VS)[:,list(native.classes_).index(1)],rtol=1e-10,atol=1e-10)
best_overall=ranking.iloc[0]
semi_candidates=ranking.loc[ranking.generated_trials>0]
best_semi=semi_candidates.iloc[0] if len(semi_candidates) else None
selected_names=[best_overall.experiment]
if best_semi is not None and best_semi.experiment not in selected_names:selected_names.append(best_semi.experiment)
for name in selected_names:
    model=joblib.load(OUT/"classifier_models"/f"{name}.joblib")
    expected=predict_reason_models(classifier_models[name],X_valid)
    np.testing.assert_allclose(predict_reason_models(model,X_valid),expected,rtol=1e-12,atol=1e-12)
    frame=pd.read_csv(OUT/"datasets"/f"{model['dataset']}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==train.ID.tolist()
    for j,r in enumerate(REASONS):
        np.testing.assert_array_equal(frame[r].to_numpy(),DATASETS[model["dataset"]]["targets"][:,j])
        np.testing.assert_allclose(frame[f"weight_{r}"].to_numpy(),DATASETS[model["dataset"]]["weights"][:,j],atol=1e-14)
    # Refit from the saved feature/target/weight table, not in-memory label generation.
    replay={**model,"estimators":[]}
    saved_scaled=model["scaler"].transform(frame[FEATURES].to_numpy())
    with threadpool_limits(limits=1):
        for j,r in enumerate(REASONS):
            y=frame[r].to_numpy(dtype=int)
            w=frame[f"weight_{r}"].to_numpy()
            included=w>0
            if model["kind"]=="logistic":
                estimator=LogisticRegression(C=model["parameter"],solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    estimator.fit(saved_scaled[included],y[included],sample_weight=w[included]*known.sum()/w[included].sum())
            else:
                estimator={"neighbors":NearestNeighbors(n_neighbors=min(int(model["parameter"]),int(included.sum())),algorithm="brute").fit(saved_scaled[included]),
                           "target":y[included],"sample_weight":w[included]}
            replay["estimators"].append(estimator)
    np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(expected)[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

settings={
    "features":FEATURES,"extra_features_added":False,"extra_pca":False,"year_included":False,
    "k_values":K_VALUES,"label_rule":LABEL_RULE,"gmm_rule":GMM_RULE,
    "stage1_classifier":{"kind":"logistic","C":1000.0},"classifier_grid":CLASSIFIER_GRID,
    "shortlisted_datasets":shortlist,"classifier_scaler_fit_rows":157,"cluster_scaler_fit_rows":1171,
    "logistic_fitting_weight_normalization":157,"threshold":0.5,"nonempty_fallback":True,
    "validation_rows":78,"validation_used_for_fit":False,
    "best_overall":best_overall.experiment,"best_semi_supervised":None if best_semi is None else best_semi.experiment,
    "reported_kaggle_references":{"submitted_supervised":0.87072,"submitted_k02":0.87393},
    "new_kaggle_scores":None,"protected_input_sha256":protected_hashes,
    "dataset_sha256":{str(q.relative_to(OUT)):hashlib.sha256(q.read_bytes()).hexdigest() for q in sorted((OUT/"datasets").glob("*.csv"))},
    "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},
}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Labeling and classifier comparison: 150 embeddings","",
       "No year, additional PCA, text, or drug features were used. All validation scores use the same 78 trials and the same 0.5 threshold with nonempty fallback.","",
       "## Frozen submitted references","",
       "| Model | Sample-wise F1 | Macro F1 | Reported Kaggle |","|---|---:|---:|---:|"]
for row in reference_rows:lines.append(f"| {row['experiment']} | {row['sample_f1']:.4f} | {row['macro_f1']:.4f} | {row['kaggle_score']:.5f} |")
lines.extend(["","## Best new classifier configurations","",
              "| Configuration | Sample-wise F1 | Macro F1 | Generated trials |","|---|---:|---:|---:|"])
for row in ranking.head(12).itertuples(index=False):lines.append(f"| {row.experiment} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.generated_trials} |")
lines.extend(["",f"Best new configuration: **{best_overall.experiment}**, validation sample-wise F1 **{best_overall.sample_f1:.4f}**."])
if best_semi is not None:lines.append(f"Best augmented configuration: **{best_semi.experiment}**, validation sample-wise F1 **{best_semi.sample_f1:.4f}**.")
lines.extend(["","## Clustering and generated labels","",
              "| Family | Best K | Sample-wise F1 with C=1000 | Macro F1 | Generated trials |","|---|---:|---:|---:|---:|"])
for family in ["kmeans","gmm_diag","gmm_tied"]:
    rows=cluster_comparison.loc[(cluster_comparison.family==family)&(cluster_comparison.status=="ok")].sort_values(["sample_f1","k"],ascending=[False,True])
    if len(rows):
        row=rows.iloc[0]
        lines.append(f"| {family} | {int(row.k)} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {int(row.generated_trials)} |")
lines.extend(["","For the selected augmented dataset, generated-label counts are:","",
              "| Reason | Generated positives | Generated negatives | Positive weight | Negative weight |","|---|---:|---:|---:|---:|"])
if best_semi is not None:
    for row in label_counts.loc[label_counts.dataset==best_semi.dataset].itertuples(index=False):
        lines.append(f"| {row.reason} | {row.generated_positives} | {row.generated_negatives} | {row.generated_positive_weight:.3f} | {row.generated_negative_weight:.3f} |")
lines.extend(["","## Method and limits","",
    "Stage 1 compares 15 clustering settings with the same partial-label rule and C=1000 classifier. "
    "Stage 2 shortlists one dataset per clustering family plus known-only and legacy K=2 labels, then evaluates seven classifier settings each. "
    "Some low-component labelers abstain entirely and reproduce the known-only dataset; those are not augmentation improvements. "
    "The result is the best of this staged search, not a global optimum over every method or parameter combination.","",
    "Each reason is assigned independently. Confident positives need estimated prevalence >=0.6; confident negatives need <=0.1; "
    "at least 3 effective labeled members are required. Component estimates use prior strength 2. "
    "Generated positive and negative weights are added in the same fraction of their respective known classes, capped at 0.5, "
    "with per-example weight <=0.25. If either generated class is absent, neither is used for that reason. "
    "This prevents generated negatives alone from drowning rare positive classes, without forcing equal class frequencies. "
    "Preserving known class balance is an influence-control choice, not an assertion about population prevalence.","",
    "All new classifiers use the same frozen scaler fitted on 157 known-label trials. Logistic fitting weights are normalized "
    "to a total of 157 per reason to hold effective regularization comparable. Weighted nearest-neighbor votes include both "
    "inverse distance and sample weights. Exact-distance matches vote alone. The frozen submitted K=2 model used its original "
    "scaler and weight total; its controlled re-evaluation is a separate candidate.","",
    "Unknown targets stay -100 with weight zero and are excluded independently per reason. Original known labels are preserved. "
    "No validation trial participates in fitting. Validation has been used to select among many candidates, and the new Kaggle "
    "scores remain unmeasured; small validation gains should not be presented as confirmed test improvements.","",
    "## Artifacts and verification","",
    "`datasets/` saves each exact feature/target/weight table. `label_audits/`, `clusters/`, and `label_counts.csv` expose generated labels and budgets. "
    "`clustering_models/` and `classifier_models/` save fitted models. `validation_predictions/` saves probabilities and labels. "
    "`clustering_comparison.csv`, `classifier_comparison.csv`, and `comparison.csv` retain every evaluated configuration. "
    "`settings.json` records settings, selection, versions, and checksums.","",
    "Checks reproduced baseline probabilities, matched weighted-neighbor predictions to scikit-learn when all weights equal one, "
    "preserved known labels and source artifacts, enforced partial-label/weight masks and budgets, and replayed saved selected models and data.",""])
summary="\n".join(lines).rstrip()+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary)
```

```python
test=pd.read_csv(INPUT_DIR/"part1_test.csv")
sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
assert test.ID.is_unique and sample.ID.is_unique
assert set(test.ID)==set(sample.ID) and set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
assert np.isfinite(test[FEATURES].to_numpy()).all()
old_submission=pd.read_csv(DATA/"submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv").set_index("ID")
export_rows=[]
for prefix in ["gmm_tied_","kmeans_"]:
    candidates=ranking.loc[ranking.dataset.str.startswith(prefix)]
    if not len(candidates):continue
    selected=candidates.iloc[0]
    name=selected.experiment
    model=joblib.load(OUT/"classifier_models"/f"{name}.joblib")
    probability=predict_reason_models(model,test[FEATURES].to_numpy())
    predicted,fallback=label_predictions(probability)
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in predicted]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    assert output.Predicted.map(lambda s:set(s.split()).issubset(REASONS)).all()
    path=OUT/"submissions"/f"part1_{name.replace('__','_')}_submission.csv"
    output.to_csv(path,index=False)
    pd.testing.assert_frame_equal(pd.read_csv(path),output)
    changed=sum(set(row.Predicted.split())!=set(old_submission.loc[row.ID,"Predicted"].split()) for row in output.itertuples(index=False))
    export_rows.append({
        "experiment":name,"dataset":selected.dataset,"classifier":selected.classifier,"parameter":float(selected.parameter),
        "validation_sample_f1":float(selected.sample_f1),"validation_macro_f1":float(selected.macro_f1),
        "submission_file":str(path.relative_to(OUT)),"test_rows":len(output),"test_fallback_trials":int(fallback.sum()),
        "test_trials_changed_from_submitted_k02":int(changed),
        "submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
        "model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{name}.joblib").read_bytes()).hexdigest(),
        "kaggle_score":None,"kaggle_status":"pending manual submission",
    })
exports=pd.DataFrame(export_rows)
exports.to_csv(OUT/"submissions.csv",index=False)
(OUT/"submission_manifest.json").write_text(json.dumps({"submissions":export_rows,"comparator_score":0.87393,
    "comparator_source":"reported by Michael for original K=2; current-best-kaggle tag unchanged"},indent=2)+"\n")
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

summary_path=OUT/"SUMMARY.md"
text=summary_path.read_text().split("## Candidate Kaggle files")[0].rstrip()
export_lines=["","","## Candidate Kaggle files","",
              "These are local files for manual upload; neither candidate has a measured Kaggle score. Compare against **0.87393**.","",
              "| Candidate | Validation F1 | Submission file | Changed test trials from original K=2 |",
              "|---|---:|---|---:|"]
for row in export_rows:
    export_lines.append(f"| {row['experiment']} | {row['validation_sample_f1']:.4f} | `{row['submission_file']}` | {row['test_trials_changed_from_submitted_k02']} |")
summary_path.write_text((text+"\n".join(export_lines)).rstrip()+"\n")
display(exports.drop(columns=["submission_sha256","model_sha256"]).round(4))
print("Candidate files saved under:",OUT/"submissions")
```

## part1/remaining_classifier_features.ipynb

```python
from pathlib import Path
from functools import lru_cache
import ast
import hashlib
import json
import platform
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer,StandardScaler
from threadpoolctl import threadpool_limits
from IPython.display import display
from rdkit import Chem,rdBase
from rdkit.Chem import Descriptors,Lipinski,rdMolDescriptors

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
C_GRID=[100.0,1000.0]
MIN_COUNTS=[3,5]
CATEGORICAL_FIELDS=["drug_actions","icdcodes","drug_targets","drugs","matched_drugs"]
ANNOTATION_FIELDS={"drug_actions","drug_targets","matched_drugs"}
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/classifier_features"
OUT=DATA/"experiments/remaining_classifier_features"
for folder in ["profiles","datasets","vocabularies","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert len(train)==1171 and len(valid)==78 and known.sum()==157
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
fixed=pd.read_csv(SOURCE/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_targets_weights.csv").write_bytes((SOURCE/"datasets/fixed_targets_weights.csv").read_bytes())
source_model=joblib.load(SOURCE/"models/embeddings_only.joblib")
embedding_scaler=source_model["embedding_scaler"]
source_settings=json.loads((SOURCE/"settings.json").read_text())
reference_sample=float(source_settings["reference_sample_f1"])
reference_macro=float(source_settings["reference_macro_f1"])
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",SOURCE/"models/embeddings_only.joblib",
                 SOURCE/"datasets/fixed_targets_weights.csv",SOURCE/"settings.json"]
additional=DATA/"experiments/additional_methods"
for rel in ["datasets/dbscan_m5_q25.csv","labeling_models/dbscan_m5_q25.joblib","classifier_models/dbscan_m5_q25__logistic_C1000.joblib"]:
    protected_paths.append(additional/rel)
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Frozen DBSCAN targets and weights; 30 classifier configurations; RDKit",rdBase.rdkitVersion)

```

```python
def normalize_name(value):return " ".join(str(value).strip().lower().split())


def parse_icd(value):
    def flatten(item):
        if isinstance(item,(list,tuple,set)):
            return [z for child in item for z in flatten(child)]
        if isinstance(item,str) and item.strip().startswith("["):
            return flatten(ast.literal_eval(item))
        return [str(item).strip().upper()]
    if pd.isna(value) or not str(value).strip():return []
    return sorted({re.sub(r"[^A-Z0-9]","",v)[:3] for v in flatten(ast.literal_eval(value)) if v})


def categorical_lists(frame,field):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        if field=="icdcodes":return parse_icd(value)
        raw=ast.literal_eval(value) if field=="drugs" else str(value).split(";")
        if field=="drug_targets":return sorted({str(z).strip().upper() for z in raw if str(z).strip()})
        return sorted({normalize_name(z) for z in raw if str(z).strip()})
    return frame[field].map(parse).tolist()


def missing_annotation(frame,field):
    return np.array([not row for row in categorical_lists(frame,field)],dtype=float)[:,None]

CATEGORIES={field:categorical_lists(train,field) for field in CATEGORICAL_FIELDS}
COUNTS={};PROFILE_ROWS=[]
for field,rows in CATEGORIES.items():
    counts=pd.Series([v for row in rows for v in row]).value_counts().rename_axis("category").rename("training_trials")
    known_counts=pd.Series([v for i in np.flatnonzero(known) for v in rows[i]]).value_counts()
    profile=counts.to_frame();profile["known_label_trials"]=known_counts.reindex(counts.index,fill_value=0).astype(int)
    profile.to_csv(OUT/"profiles"/f"{field}_frequencies.csv")
    COUNTS[field]=counts
    for minimum in MIN_COUNTS:
        PROFILE_ROWS.append({"field":field,"minimum_training_trials":minimum,"retained_categories":int((counts>=minimum).sum()),
                             "training_rows_with_no_annotation":sum(not row for row in rows)})
# These three database fields share the same annotation-missing pattern in the training data.
np.testing.assert_array_equal(missing_annotation(train,"matched_drugs"),missing_annotation(train,"drug_targets"))
np.testing.assert_array_equal(missing_annotation(train,"matched_drugs"),train.drug_smiles.fillna("").str.strip().eq("").to_numpy(dtype=float)[:,None])
pd.DataFrame(PROFILE_ROWS).to_csv(OUT/"profiles/categorical_cutoff_sizes.csv",index=False)
display(pd.DataFrame(PROFILE_ROWS))

```

```python
DESCRIPTOR_NAMES=["molecular_weight","logP","polar_surface_area","hydrogen_donors","hydrogen_acceptors","rotatable_bonds","ring_count","fraction_sp3"]
NUMERIC_SMILES_NAMES=[f"{aggregation}_{name}" for aggregation in ["mean","max"] for name in DESCRIPTOR_NAMES]+["valid_structure_count"]
SMILES_NAMES=NUMERIC_SMILES_NAMES+["no_valid_structure","has_invalid_structure"]


@lru_cache(maxsize=None)
def molecule_descriptors(smiles):
    with rdBase.BlockLogs():mol=Chem.MolFromSmiles(smiles)
    if mol is None:return None
    values=np.array([Descriptors.MolWt(mol),Descriptors.MolLogP(mol),rdMolDescriptors.CalcTPSA(mol),
                     Lipinski.NumHDonors(mol),Lipinski.NumHAcceptors(mol),Lipinski.NumRotatableBonds(mol),
                     rdMolDescriptors.CalcNumRings(mol),rdMolDescriptors.CalcFractionCSP3(mol)],dtype=float)
    if not np.isfinite(values).all():return None
    return tuple(values)


def raw_smiles_features(frame):
    features=[];audit=[]
    for trial_id,value in zip(frame.ID,frame.drug_smiles):
        parts=[] if pd.isna(value) else sorted({v.strip() for v in str(value).split(";") if v.strip()})
        valid_values=[];invalid=[]
        for part in parts:
            values=molecule_descriptors(part)
            if values is None:invalid.append(part)
            else:valid_values.append(values)
        if valid_values:
            a=np.asarray(valid_values);summaries=np.r_[a.mean(axis=0),a.max(axis=0)]
        else:summaries=np.full(16,np.nan)
        features.append(np.r_[summaries,len(valid_values),float(not valid_values),float(bool(invalid))])
        audit.append({"ID":trial_id,"listed_distinct_structures":len(parts),"valid_structures":len(valid_values),
                      "invalid_structures":len(invalid),"invalid_smiles":";".join(invalid)})
    return np.asarray(features,dtype=float),pd.DataFrame(audit)

TRAIN_SMILES,train_molecule_audit=raw_smiles_features(train)
train_molecule_audit.to_csv(OUT/"profiles/training_smiles_parse_audit.csv",index=False)
assert TRAIN_SMILES.shape==(1171,19)
assert np.isfinite(TRAIN_SMILES[:,16:]).all()
print("SMILES training trials with usable structures:",int((TRAIN_SMILES[:,17]==0).sum()),
      "trials containing invalid structures:",int(TRAIN_SMILES[:,18].sum()))
print("Parsed valid distinct molecular structures:",molecule_descriptors.cache_info())

```

```python
def make_block(kind,minimum=None,field=None):
    if kind=="none":return {"kind":"none","feature_names":[]}
    if kind=="categorical":
        classes=sorted(COUNTS[field][COUNTS[field]>=minimum].index.tolist())
        encoder=MultiLabelBinarizer(classes=classes,sparse_output=True);encoder.fit(CATEGORIES[field])
        missing_flag=field in ANNOTATION_FIELDS
        names=[f"{field}__{v}" for v in classes]+([f"{field}_annotation_missing"] if missing_flag else [])
        return {"kind":kind,"field":field,"minimum_training_trials":minimum,"encoder":encoder,
                "missing_flag":missing_flag,"feature_names":names}
    if kind=="missing_only":return {"kind":kind,"field":field,"feature_names":[f"{field}_annotation_missing"]}
    if kind=="year":
        scaler=StandardScaler().fit(train[["start_year"]].to_numpy())
        return {"kind":kind,"scaler":scaler,"feature_names":["standardized_start_year"]}
    if kind=="smiles":
        medians=np.nanmedian(TRAIN_SMILES[:,:17],axis=0)
        assert np.isfinite(medians).all()
        numeric=np.where(np.isnan(TRAIN_SMILES[:,:17]),medians[None,:],TRAIN_SMILES[:,:17])
        scaler=StandardScaler().fit(numeric)
        return {"kind":kind,"medians":medians,"scaler":scaler,"descriptor_names":DESCRIPTOR_NAMES,
                "numeric_feature_names":NUMERIC_SMILES_NAMES,"feature_names":SMILES_NAMES}
    raise ValueError(kind)


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_annotation(frame,block["field"]))
    if kind=="year":return sparse.csr_matrix(block["scaler"].transform(frame[["start_year"]].to_numpy()))
    if kind=="smiles":
        raw,_=raw_smiles_features(frame)
        numeric=np.where(np.isnan(raw[:,:17]),block["medians"][None,:],raw[:,:17])
        return sparse.csr_matrix(np.column_stack([block["scaler"].transform(numeric),raw[:,17:]]))
    rows=categorical_lists(frame,block["field"]);allowed=set(block["encoder"].classes_)
    encoded=block["encoder"].transform([[v for v in row if v in allowed] for row in rows]).astype(float).tocsr()
    if block["missing_flag"]:encoded=sparse.hstack([encoded,sparse.csr_matrix(missing_annotation(frame,block["field"]))],format="csr")
    assert encoded.shape[1]==len(block["feature_names"]) and np.isin(encoded.data,[0,1]).all()
    return encoded


def transform_model(model,frame):
    embedding=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embedding
    return sparse.hstack([sparse.csr_matrix(embedding),transform_block(model["block"],frame)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int);empty=prediction.sum(axis=1)==0
    rows=np.flatnonzero(empty);prediction[rows,probabilities[rows].argmax(axis=1)]=1
    return prediction,empty

reference_probability=predict_model(source_model,valid)
reference_prediction,_=label_predictions(reference_probability)
np.testing.assert_allclose(f1_score(truth,reference_prediction,average="samples",zero_division=0),reference_sample,atol=1e-14)
ROWS=[];BASELINE_SCORES={};MATRIX_RECORDS={}


def run_representation(kind,minimum=None,field=None):
    name="embeddings_only" if kind=="none" else (f"{field}_min{minimum}" if kind=="categorical" else (f"{field}_missing_only" if kind=="missing_only" else kind))
    block=make_block(kind,minimum,field)
    template={**source_model,"kind":"remaining_features_logistic","block":block,"estimators":[],
              "source_targets_file":"datasets/fixed_targets_weights.csv","training_matrix_file":f"datasets/{name}.npz",
              "feature_block_weight":1.0}
    matrix=transform_model(template,train);stored=sparse.csr_matrix(matrix)
    assert np.isfinite(stored.data).all()
    names=EMBEDDINGS+block["feature_names"]
    assert stored.shape==(len(train),len(names)) and len(names)==len(set(names))
    sparse.save_npz(OUT/"datasets"/f"{name}.npz",stored)
    (OUT/"datasets"/f"{name}_columns.json").write_text(json.dumps(names,indent=2)+"\n")
    MATRIX_RECORDS[name]={"shape":list(stored.shape),"kind":kind,"field":field,"minimum_training_trials":minimum}
    if kind=="categorical":
        vocabulary=pd.DataFrame({"category":block["encoder"].classes_})
        vocabulary["training_trials"]=vocabulary.category.map(COUNTS[field])
        vocabulary.to_csv(OUT/"vocabularies"/f"{name}.csv",index=False)
    for C in C_GRID:
        experiment=f"{name}_C{C:g}"
        model={**template,"C":C,"estimators":[]}
        with threadpool_limits(limits=1):
            for j in range(5):
                estimator=clone(source_model["estimators"][j]).set_params(C=C)
                included=W[:,j]>0;normalized=W[included,j]*157/W[included,j].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    estimator.fit(matrix[included],Y[included,j],sample_weight=normalized)
                model["estimators"].append(estimator)
        probability=predict_model(model,valid)
        prediction,empty=label_predictions(probability)
        sample=f1_score(truth,prediction,average="samples",zero_division=0)
        macro=f1_score(truth,prediction,average="macro",zero_division=0)
        if kind=="none":BASELINE_SCORES[C]=sample
        if kind=="none" and C==1000:np.testing.assert_allclose(probability,reference_probability,rtol=1e-8,atol=1e-10)
        row={"experiment":experiment,"representation":name,"feature_group":field if kind=="categorical" else kind,
             "minimum_training_trials":minimum,"C":C,"additional_features":len(block["feature_names"]),
             "total_features":len(names),"sample_f1":sample,"macro_f1":macro,
             "delta_vs_current_leader":sample-reference_sample,"delta_vs_same_C_embeddings":sample-BASELINE_SCORES[C],
             "fallback_trials":int(empty.sum()),"changed_trials_vs_current":int((prediction!=reference_prediction).any(axis=1).sum()),
             **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,prediction,average=None,zero_division=0))),
             "is_current":kind=="none" and C==1000,"kaggle_score":np.nan}
        ROWS.append(row)
        frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
        for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probability[:,j];frame[f"predicted_{r}"]=prediction[:,j]
        frame["used_fallback"]=empty
        frame["trial_f1"]=2*(truth*prediction).sum(axis=1)/(truth.sum(axis=1)+prediction.sum(axis=1))
        frame.to_csv(OUT/"validation_predictions"/f"{experiment}.csv",index=False)
        joblib.dump(model,OUT/"models"/f"{experiment}.joblib",compress=3)
        pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
        print(f"{experiment}: +{len(block['feature_names'])} features, sample F1={sample:.6f}, macro F1={macro:.6f}, change={sample-reference_sample:+.6f}",flush=True)

run_representation("none")

```

```python
for field in CATEGORICAL_FIELDS:
    for minimum in MIN_COUNTS:run_representation("categorical",minimum,field)

```

```python
run_representation("missing_only",field="matched_drugs")
run_representation("missing_only",field="drug_actions")
run_representation("year")
run_representation("smiles")

```

```python
comparison=pd.DataFrame(ROWS);assert len(comparison)==30
ranked=comparison.assign(distance_from_current_C=abs(np.log10(comparison.C/1000))).sort_values(
    ["sample_f1","is_current","additional_features","distance_from_current_C","experiment"],ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_group=ranked.groupby("feature_group",sort=False).head(1)
best_by_group.to_csv(OUT/"best_by_group.csv",index=False)
comparison.query("representation=='embeddings_only'").to_csv(OUT/"embedding_controls.csv",index=False)
display(best_by_group[["feature_group","representation","C","additional_features","sample_f1","macro_f1","delta_vs_current_leader"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy()
    predicted=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_array_equal(label_predictions(probs)[0],predicted)
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predicted,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    loaded=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    np.testing.assert_allclose(predict_model(loaded,valid),probs,rtol=1e-12,atol=1e-12)
    saved=sparse.load_npz(OUT/loaded["training_matrix_file"])
    expected=sparse.csr_matrix(transform_model(loaded,train));diff=saved-expected
    assert not diff.nnz or abs(diff.data).max()<1e-12
# Independent selected-model refit from exact persisted features and target weights.
replayed=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert replayed.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(replayed[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(replayed[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
matrix=sparse.load_npz(OUT/selected["training_matrix_file"])
if selected["block"]["kind"]=="none":matrix=matrix.toarray()
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j in range(5):
        included=W[:,j]>0;estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(matrix[included],Y[included,j],sample_weight=W[included,j]*157/W[included,j].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
valid_smiles,valid_parse_audit=raw_smiles_features(valid)
valid_parse_audit.to_csv(OUT/"profiles/validation_smiles_parse_audit.csv",index=False)
print("All 30 saved models/predictions/matrices, reference control, independent winner refit, and protected hashes verified.")

```

```python
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"classifier":"logistic regression","C_grid":C_GRID,"categorical_minimum_counts":MIN_COUNTS,
          "categorical_fields":CATEGORICAL_FIELDS,"configurations":30,"reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "generated_labels_changed":False,"generated_weights_changed":False,"clustering_changed":False,"embedding_scaler_fit_rows":157,
          "new_preprocessing_fit_rows":1171,"categorical_binary_features":"unscaled 0/1","feature_block_weight":1.0,
          "icd_encoding":"three-character alphanumeric category prefixes from safely parsed nested code lists",
          "smiles_descriptors":DESCRIPTOR_NAMES,"smiles_aggregation":["mean","max"],"smiles_extra_features":["valid_structure_count","no_valid_structure","has_invalid_structure"],
          "numeric_imputation":"training medians for missing descriptors","numeric_scaling":"training-only StandardScaler; binary flags unscaled",
          "numeric_imputation_fit_rows":1171,"validation_used_for_fit":False,"validation_rows":78,
          "cutoff":0.5,"fallback":True,"fitting_weight_total_per_reason":157,"matrices":MATRIX_RECORDS,
          "year_scaler_mean":float(joblib.load(OUT/"models/year_C1000.joblib")["block"]["scaler"].mean_[0]),
          "year_scaler_standard_deviation":float(joblib.load(OUT/"models/year_C1000.joblib")["block"]["scaler"].scale_[0]),
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "selection":"sample F1; ties prefer current leader, fewer features, then original C",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__,"rdkit":rdBase.rdkitVersion},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(frame):
    lines=["| Representation | C | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |",
           "|---|---:|---:|---:|---:|---:|---:|"]
    for row in frame.itertuples(index=False):
        lines.append(f"| {row.representation} | {row.C:g} | {row.additional_features} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_current_leader:+.6f} | {row.delta_vs_same_C_embeddings:+.6f} |")
    return "\n".join(lines)
lines=["# Remaining classifier feature screen","",
       "30 configurations: five categorical feature fields at minimum training counts 3/5 and C=100/1000, "
       "plus standardized year, molecular descriptors, two missing-annotation controls, and embeddings-only at both C settings. "
       "Each group is added independently to the same 150 embeddings with fixed DBSCAN targets and raw weights.","",
       f"Reference sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best by feature group","",table(best_by_group),"",
       "## Preprocessing","",
       "Category counts measure distinct training trials. Drug names/actions are lowercased with normalized whitespace; "
       "protein target symbols are uppercased; ICD codes are safely unpacked from nested lists and reduced to three-character "
       "category prefixes. Categories below the selected training frequency are omitted. Multi-hot columns remain 0/1. "
       "Each matched-drug, target, and action block includes its annotation-missing flag. The matched-drug missing indicator "
       "covers the same training rows as missing targets/SMILES; actions have a slightly different pattern. Both patterns "
       "are tested alone so missingness can be separated from biological content.","",
       "Eight RDKit molecular descriptors (molecular weight, logP, polar surface area, hydrogen donors/acceptors, rotatable "
       "bonds, ring count, fraction sp3 carbons) are aggregated by mean and maximum over valid distinct matched SMILES. "
       "Valid-structure count is added; absent descriptors are imputed with training medians. These 17 numeric columns are "
       "standardized using all training trials; no-valid-structure and invalid-structure flags stay binary. No fingerprints "
       "or chemical text bag-of-words is used in this initial screen. Parsing failures are saved in the molecule audit. "
       "Start year is standardized on all 1,171 training trials; the original embedding scaler remains frozen from 157 known-label trials.","",
       "All new preprocessing uses training rows only. Validation/test never supplies vocabulary, imputation, scaling, or fit labels. "
       "The classifier keeps normalized fitting weight 157 per reason, final cutoff 0.5, highest-probability fallback, and block weight 1.","",
       "## All configurations","",table(comparison),"",
       "## Verification and limits","",
       "Every saved model reloads with matching probabilities; all label sets rescore to the recorded sample and macro F1. "
       "Each model recreates its exact saved sparse matrix. The C=1000 embedding control reproduces the original leader, "
       "and the selected model independently refits from its persisted matrix and targets/weights with matching probabilities. "
       "Protected sources retain their checksums.","",
       "This is a bounded screen of specific encodings on the repeatedly consulted 78-trial validation set. It does not "
       "establish independent test improvement or exhaust all encodings, feature combinations, fingerprints, or regularization settings.","",
       "profiles/ and vocabularies/ preserve training frequency distributions, filters, and molecular parsing audits; "
       "datasets/ saves exact matrices, ordered feature names, and shared trial IDs/targets/weights; models/ and "
       "validation_predictions/ preserve all fits and outputs. comparison.csv, best_by_group.csv, embedding_controls.csv, "
       "and settings.json preserve results, settings, and checksums. RDKit was installed in the workspace virtual environment.","",
       "No Kaggle submission, commit, or tag was made.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No configuration improves the primary validation metric; retain the current embeddings-only leader.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print(table(best_by_group))
print("Selected:",best.experiment,"Candidate:",candidate)

```

## part1/research/lda_ensemble/research/ani_snapshots/a548721/part1_best_model.ipynb

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.mixture import GaussianMixture
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog('rdApp.*')
SEED = 0
DATA = '../part1/'
LABELS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']
EMB = [f'emb_{i}' for i in range(150)]

train = pd.read_csv(DATA + 'part1_train.csv')
val = pd.read_csv(DATA + 'part1_val.csv')
test = pd.read_csv(DATA + 'part1_test.csv')
is_lab = (train[LABELS] != -100).all(axis=1).values

print(f'train {train.shape}, labeled {is_lab.sum()}, unlabeled {(~is_lab).sum()} | val {val.shape} | test {test.shape}')
pd.DataFrame({'labeled train': train.loc[is_lab, LABELS].mean(), 'val': val[LABELS].mean()}).T.round(3)
```

```python
n_tr, n_va, n_te = len(train), len(val), len(test)
emb_all = np.vstack([train[EMB].values, val[EMB].values, test[EMB].values])
X_all = StandardScaler().fit_transform(emb_all)
X_tr, X_va, X_te = X_all[:n_tr], X_all[n_tr:n_tr + n_va], X_all[n_tr + n_va:]
Y_tr, Y_va = train[LABELS].values, val[LABELS].values

pca30 = PCA(30, random_state=SEED).fit(X_all)
gmm = GaussianMixture(30, covariance_type='diag', n_init=3, random_state=SEED).fit(pca30.transform(X_all))
print('GMM converged:', gmm.converged_, '| iterations:', gmm.n_iter_, '| tol:', gmm.tol)
```

```python
def decide(P, thr=0.3):
    Y = (P >= thr).astype(int)
    Y[np.arange(len(P)), P.argmax(1)] = 1
    return Y


def expand(X, Y, w):
    rows, ys, ws = [], [], []
    for i, y in enumerate(Y):
        pos = np.flatnonzero(y)
        rows += [i] * len(pos); ys += list(pos); ws += [w[i] / len(pos)] * len(pos)
    return X[rows], np.array(ys), np.array(ws)


def multinomial_lr(Xl, Yl, Xq, C=0.3, balanced=True):
    X, y, sw = expand(Xl, Yl, np.ones(len(Yl)))
    if balanced:
        cnt = np.bincount(y, weights=sw, minlength=5) + 1e-9
        sw = sw * cnt.mean() / cnt[y]
    m = LogisticRegression(C=C, max_iter=3000).fit(X, y, sample_weight=sw)
    P = np.zeros((len(Xq), 5))
    P[:, m.classes_] = m.predict_proba(Xq)
    return P


def knn(Xl, Yl, Xq, k=10):
    S = normalize(Xq) @ normalize(Xl).T
    idx = np.argsort(-S, 1)[:, :k]
    w = np.take_along_axis(S, idx, 1).clip(0) ** 2
    T = Yl / Yl.sum(1, keepdims=True)
    return (w[:, :, None] * T[idx]).sum(1) / (w.sum(1, keepdims=True) + 1e-9)


def gmm_cluster_label(Xl, Yl, Xq, a=1.0):
    Rl, Rq = (gmm.predict_proba(pca30.transform(M[:, :len(EMB)])) for M in (Xl, Xq))
    T = Yl / Yl.sum(1, keepdims=True)
    cluster_dist = (Rl.T @ T + a * T.mean(0)) / (Rl.sum(0)[:, None] + a)
    return Rq @ cluster_dist


def base_model(Xl, Yl, Xq, w_knn=0.3):
    return (1 - w_knn) * multinomial_lr(Xl, Yl, Xq) + w_knn * knn(Xl, Yl, Xq)


def final_model(Xl, Yl, Xq, Xu, w_gmm=0.2, conf=0.8, thr=0.3, rounds=2, return_aug=False):
    XL, YL = Xl, Yl
    for _ in range(rounds):
        Pu = (1 - w_gmm) * base_model(XL, YL, Xu) + w_gmm * gmm_cluster_label(XL, YL, Xu)
        keep = Pu.max(1) >= conf
        XL, YL = np.vstack([Xl, Xu[keep]]), np.vstack([Yl, decide(Pu, thr)[keep]])
    P = base_model(XL, YL, Xq)
    return (P, keep.sum()) if return_aug else P
```

```python
X_pool, Y_pool = np.vstack([X_tr[is_lab], X_va]), np.vstack([Y_tr[is_lab], Y_va])
X_unl = np.vstack([X_tr[~is_lab], X_te])


def metrics(Y, Yh):
    per = f1_score(Y, Yh, average=None, zero_division=0)
    return {'sample_F1': f1_score(Y, Yh, average='samples', zero_division=0), 'macro_F1': per.mean(),
            **{f'F1_{l}': v for l, v in zip(LABELS, per)}}


def repeated_cv(fn, X=X_pool, Xu=X_unl, reps=8, thr=0.3):
    scores = []
    for r in range(reps):
        for a, b in KFold(5, shuffle=True, random_state=100 + r).split(X):
            scores.append(metrics(Y_pool[b], decide(fn(X[a], Y_pool[a], X[b], Xu), thr)))
    return pd.DataFrame(scores).mean()


def evaluate(name, fn, Xl=X_tr[is_lab], Xq=X_va, X=X_pool, Xu=X_unl):
    v = metrics(Y_va, decide(fn(Xl, Y_tr[is_lab], Xq, Xu)))
    c = repeated_cv(fn, X, Xu)
    return {'model': name, 'val_sample_F1': v['sample_F1'], 'val_macro_F1': v['macro_F1'],
            'cv_sample_F1': c['sample_F1'], 'cv_macro_F1': c['macro_F1']}
```

```python
def ovr_lr(Xl, Yl, Xq, C=1.0):
    return np.column_stack([LogisticRegression(C=C, max_iter=2000).fit(Xl, Yl[:, j]).predict_proba(Xq)[:, 1]
                            for j in range(5)])


candidates = {
    'One-vs-rest LR (C=1)': lambda a, b, c, u: ovr_lr(a, b, c),
    'kNN cosine (k=10)': lambda a, b, c, u: knn(a, b, c),
    'GMM-30 cluster-then-label': lambda a, b, c, u: gmm_cluster_label(a, b, c),
    'Multinomial LR, unbalanced': lambda a, b, c, u: multinomial_lr(a, b, c, balanced=False),
    'Balanced multinomial LR + kNN': lambda a, b, c, u: base_model(a, b, c),
    'Self-training, no GMM': lambda a, b, c, u: final_model(a, b, c, u, w_gmm=0.0),
    'FINAL: GMM-assisted self-training': final_model,
}
comparison = pd.DataFrame([evaluate(n, f) for n, f in candidates.items()]).set_index('model')
comparison.round(3)
```

```python
def svd_block(M, k):
    return StandardScaler().fit_transform(TruncatedSVD(k, random_state=SEED).fit_transform(M))


def text_col(col):
    return pd.concat([train[col], val[col], test[col]]).fillna('').astype(str).values


fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def trial_fingerprint(smiles):
    if not isinstance(smiles, str):
        return np.zeros(1024)
    fps = [fp_gen.GetFingerprintAsNumPy(m) for s in smiles.split(';') if (m := Chem.MolFromSmiles(s)) is not None]
    return np.max(fps, axis=0) if fps else np.zeros(1024)


year = pd.concat([train.start_year, val.start_year, test.start_year]).values.astype(float)
extra = {
    '+ start_year': ((year - year.mean()) / year.std())[:, None],
    '+ criteria TF-IDF (SVD 50)': svd_block(TfidfVectorizer(min_df=3, ngram_range=(1, 2), sublinear_tf=True,
                                                           max_features=20000).fit_transform(text_col('criteria')), 50),
    '+ diseases TF-IDF (SVD 20)': svd_block(TfidfVectorizer(min_df=2).fit_transform(text_col('diseases_cleaned')), 20),
    '+ Morgan fingerprints (SVD 20)': svd_block(np.vstack([trial_fingerprint(s) for s in text_col('drug_smiles')
                                                           .astype(object)]).astype(float), 20),
}


def split_feats(F):
    return F[:n_tr], F[n_tr:n_tr + n_va], F[n_tr + n_va:]


rows = [evaluate('embeddings only', final_model)]
for name, F in extra.items():
    a, b, c = split_feats(np.hstack([X_all, F]))
    rows.append(evaluate(name, final_model, Xl=a[is_lab], Xq=b,
                         X=np.vstack([a[is_lab], b]), Xu=np.vstack([a[~is_lab], c])))
ablation = pd.DataFrame(rows).set_index('model')
ablation.round(3)
```

```python
P_va, n_pseudo = final_model(X_tr[is_lab], Y_tr[is_lab], X_va, X_unl, return_aug=True)
val_metrics = pd.Series(metrics(Y_va, decide(P_va)))
print(f'pseudo-labeled trials added: {n_pseudo} / {len(X_unl)}')
val_metrics.round(3)
```

```python
X_lab_all, Y_lab_all = X_pool, Y_pool
Y_unl_tr = decide(final_model(X_lab_all, Y_lab_all, X_tr[~is_lab], X_unl))
Y_te = decide(final_model(X_lab_all, Y_lab_all, X_te, X_unl))

submission = pd.DataFrame({'ID': test.ID,
                           'Predicted': [' '.join(l for l, y in zip(LABELS, row) if y) for row in Y_te]})
submission.to_csv('part1_best_submission.csv', index=False)

completed = pd.concat([
    train.loc[is_lab, ['ID']].assign(**dict(zip(LABELS, Y_tr[is_lab].T)), split='train', source='true'),
    train.loc[~is_lab, ['ID']].assign(**dict(zip(LABELS, Y_unl_tr.T)), split='train', source='predicted'),
    val[['ID']].assign(**dict(zip(LABELS, Y_va.T)), split='val', source='true'),
    test[['ID']].assign(**dict(zip(LABELS, Y_te.T)), split='test', source='predicted'),
], ignore_index=True)
completed.to_csv('part1_best_reason_labels.csv', index=False)

print(submission.head())
print(f'\nwrote {len(submission)} test predictions and {len(completed)} reason-label rows')
```

```python
freq = pd.DataFrame({
    'labeled (train+val)': Y_lab_all.mean(0),
    'imputed train (unlabeled)': Y_unl_tr.mean(0),
    'predicted test (2010-2016)': Y_te.mean(0),
}, index=LABELS)
ax = freq.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel('fraction of trials')
ax.set_title('Reason-label frequency by split')
plt.tight_layout()
plt.savefig('part1_label_frequency.png', dpi=150)
freq.round(3)
```

## part1/self_training.ipynb

```python
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
CONFIDENCES=[0.90,0.95]
SELF_WEIGHT_CAPS=[0.05,0.10]
ROUNDS=2
MAXIMUM_ADDED_CLASS_FRACTION=0.25
from sklearn.base import clone
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"self_training"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")
```

```python
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")
```

```python
SOURCE=DATA/"experiments/additional_methods"
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
seed_model=joblib.load(SOURCE/source_record["model_file"])
seed_frame=pd.read_csv(SOURCE/"datasets/dbscan_m5_q25.csv",float_precision="round_trip")
assert seed_frame.ID.tolist()==train.ID.tolist()
seed_y=seed_frame[REASONS].to_numpy(dtype=int)
seed_w=seed_frame[[f"weight_{r}" for r in REASONS]].to_numpy()
save_dataset("dbscan_seed",seed_y,seed_w)
with threadpool_limits(limits=1):reproduced_seed=fit_reason_models("dbscan_seed")
seed_probability=predict_reason_models(seed_model,X_valid)
np.testing.assert_allclose(predict_reason_models(reproduced_seed,X_valid),seed_probability,rtol=1e-8,atol=1e-10)
seed_metrics=save_predictions("dbscan_seed",seed_probability)
np.testing.assert_allclose(seed_metrics["sample_f1"],0.9102564102564102,atol=1e-14)
seed_prediction,_=label_predictions(seed_probability)
for p in [SOURCE/source_record["model_file"],SOURCE/"datasets/dbscan_m5_q25.csv",SOURCE/source_record["submission_file"],SOURCE/"labeling_models/dbscan_m5_q25.joblib"]:
    protected_hashes[str(p.relative_to(DATA))]=hashlib.sha256(p.read_bytes()).hexdigest()
ROWS=[];ACCEPTANCE=[];MODELS={"dbscan_seed":seed_model}


def build_self_targets(teacher_probability,confidence,cap):
    targets=seed_y.copy();weights=seed_w.copy()
    unknown=(~known)[:,None]&(seed_w==0)
    for j,r in enumerate(REASONS):
        positive=unknown[:,j]&(teacher_probability[:,j]>=confidence)
        negative=unknown[:,j]&(teacher_probability[:,j]<=1-confidence)
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        seeded_pos=seed_w[(~known)&(seed_y[:,j]==1),j].sum()/kp
        seeded_neg=seed_w[(~known)&(seed_y[:,j]==0),j].sum()/kn
        np.testing.assert_allclose(seeded_pos,seeded_neg,atol=1e-12)
        fraction=0.0
        if npos and nneg:
            fraction=min(MAXIMUM_ADDED_CLASS_FRACTION,max(0.0,0.5-max(seeded_pos,seeded_neg)),cap*npos/kp,cap*nneg/kn)
            if fraction>0:
                targets[positive,j]=1;targets[negative,j]=0
                weights[positive,j]=fraction*kp/npos;weights[negative,j]=fraction*kn/nneg
        added=(weights[:,j]>0)&(seed_w[:,j]==0)
        assert weights[added,j].max(initial=0)<=cap+1e-12
        pos_added=added&(targets[:,j]==1);neg_added=added&(targets[:,j]==0)
        np.testing.assert_allclose(weights[pos_added,j].sum()/kp,weights[neg_added,j].sum()/kn,atol=1e-12)
        assert seeded_pos+fraction<=0.5+1e-12 and seeded_neg+fraction<=0.5+1e-12
        ACCEPTANCE.append({"reason":r,"positive_candidates":npos,"negative_candidates":nneg,
                           "accepted_positives":int(pos_added.sum()),"accepted_negatives":int(neg_added.sum()),
                           "added_positive_weight":float(weights[pos_added,j].sum()),"added_negative_weight":float(weights[neg_added,j].sum()),
                           "added_class_fraction":fraction})
    np.testing.assert_array_equal(targets[seed_w>0],seed_y[seed_w>0])
    np.testing.assert_array_equal(weights[seed_w>0],seed_w[seed_w>0])
    return targets,weights


def run_chain(confidence,cap):
    teacher=seed_model;previous_targets=seed_y.copy()
    for round_number in range(1,ROUNDS+1):
        name=f"confidence{confidence:g}_cap{cap:g}_round{round_number}".replace(".","p")
        teacher_probability=predict_reason_models(teacher,X)
        start=len(ACCEPTANCE)
        y,w=build_self_targets(teacher_probability,confidence,cap)
        for row in ACCEPTANCE[start:]:row.update({"experiment":name,"confidence":confidence,"weight_cap":cap,"round":round_number})
        dataset=save_dataset(name,y,w,teacher_probability)
        # Distinguish original verified labels, frozen DBSCAN seeds, and new self-training targets.
        audit=pd.read_csv(OUT/"label_audits"/f"{name}.csv")
        for j,r in enumerate(REASONS):
            audit[f"label_source_{r}"]=np.where(known,"original",np.where(seed_w[:,j]>0,"dbscan_seed",np.where(w[:,j]>0,"self_training","unknown")))
        audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
        with threadpool_limits(limits=1):model=fit_reason_models(name,"logistic",1000)
        model["self_training"]={"confidence":confidence,"maximum_added_example_weight":cap,"rounds":round_number,
                                "maximum_added_class_fraction":MAXIMUM_ADDED_CLASS_FRACTION,"seed_dataset":"dbscan_seed"}
        probabilities=predict_reason_models(model,X_valid)
        metrics=save_predictions(name,probabilities)
        joblib.dump(model,OUT/"classifier_models"/f"{name}.joblib",compress=3);MODELS[name]=model
        extra=(w>0)&(seed_w==0)
        previous_extra=(previous_targets!=-100)&(seed_w==0)
        flips=extra&previous_extra&(y!=previous_targets)
        prediction,_=label_predictions(probabilities)
        ROWS.append({"experiment":name,"confidence":confidence,"weight_cap":cap,"round":round_number,
                     "generated_trials_total":dataset["generated_trials"],"self_labeled_trials":int(extra.any(axis=1).sum()),
                     "self_labeled_reason_slots":int(extra.sum()),"dropped_reason_slots_vs_previous":int((previous_extra&~extra).sum()),
                     "flipped_reason_slots_vs_previous":int(flips.sum()),**metrics,
                     "changed_trials_vs_seed":int((prediction!=seed_prediction).any(axis=1).sum()),
                     "delta_vs_seed":metrics["sample_f1"]-seed_metrics["sample_f1"],"kaggle_score":np.nan})
        pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
        pd.DataFrame(ACCEPTANCE).to_csv(OUT/"acceptance_audit.csv",index=False)
        pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
        print(f"{name}: sample F1={metrics['sample_f1']:.6f}, macro F1={metrics['macro_f1']:.6f}, "
              f"new reason slots={int(extra.sum())}, change={ROWS[-1]['delta_vs_seed']:+.6f}",flush=True)
        teacher=model;previous_targets=y

```

```python
for cap in SELF_WEIGHT_CAPS:run_chain(0.90,cap)

```

```python
for cap in SELF_WEIGHT_CAPS:run_chain(0.95,cap)

```

```python
comparison=pd.DataFrame(ROWS);assert len(comparison)==8
ranked=comparison.sort_values(["sample_f1","round","weight_cap","confidence"],ascending=[False,True,True,False],kind="stable")
best=ranked.iloc[0]
selected=joblib.load(OUT/"classifier_models"/f"{best.experiment}.joblib")
selected_probability=predict_reason_models(selected,X_valid)
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"datasets"/f"{row.experiment}.csv",float_precision="round_trip")
    y=frame[REASONS].to_numpy(dtype=int);w=frame[[f"weight_{r}" for r in REASONS]].to_numpy()
    assert frame.ID.tolist()==train.ID.tolist()
    np.testing.assert_array_equal(y[seed_w>0],seed_y[seed_w>0]);np.testing.assert_array_equal(w[seed_w>0],seed_w[seed_w>0])
    prediction_file=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert prediction_file.ID.tolist()==valid.ID.tolist()
    probs=prediction_file[[f"probability_{r}" for r in REASONS]].to_numpy()
    predicted=prediction_file[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_reason_models(joblib.load(OUT/"classifier_models"/f"{row.experiment}.joblib"),X_valid),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],predicted)
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predicted,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
frame=pd.read_csv(OUT/"datasets"/f"{best.experiment}.csv",float_precision="round_trip")
scaled=classifier_scaler.transform(frame[FEATURES].to_numpy());replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_reason_models(replay,X_valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("All seed-preservation, saved-model, score, independent-refit, and protected-input checks passed.")
display(comparison[["confidence","weight_cap","round","self_labeled_trials","self_labeled_reason_slots","sample_f1","macro_f1","delta_vs_seed"]].round(6))

```

```python
candidate=None
if best.sample_f1>seed_metrics["sample_f1"]+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_reason_models(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_self_training_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"classifier_models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"confidences":CONFIDENCES,"additional_weight_caps":SELF_WEIGHT_CAPS,"rounds":ROUNDS,"configurations":8,
          "maximum_added_class_fraction":MAXIMUM_ADDED_CLASS_FRACTION,"maximum_total_pseudo_class_fraction":0.5,
          "seed_targets_preserved":True,"seed_weights_preserved":True,"refresh_from_original_seed_unknown_slots_each_round":True,
          "verified_training_labels":157,"validation_labels_added_to_training":False,"validation_used_for_fit":False,"test_used_for_fit":False,
          "features":FEATURES,"classifier_C":1000,"classifier_scaler_fit_rows":157,"fitting_weight_total_per_reason":157,
          "prediction_cutoff":0.5,"fallback":True,"reference_sample_f1":seed_metrics["sample_f1"],"reference_macro_f1":seed_metrics["macro_f1"],
          "best_self_training_experiment":best.experiment,"best_self_training_sample_f1":float(best.sample_f1),"best_self_training_macro_f1":float(best.macro_f1),
          "leader_changed":bool(best.sample_f1>seed_metrics["sample_f1"]+1e-12),"candidate":candidate,
          "protected_input_sha256":protected_hashes,"dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Conservative self-training from the DBSCAN seed","",
       "Eight settings cross confidence 0.90/0.95, added-example caps 0.05/0.10, and one/two rounds. "
       "Only original seed-unknown reason slots in unlabeled training rows can receive new targets. Known and DBSCAN seed "
       "targets/weights remain unchanged. Both new pseudo classes are required; their total added fractions preserve the "
       "known class ratio, capped at 0.25 per class and total seed+self influence 0.50. Each round refreshes its additions "
       "from the original seed using the previous classifier, allowing uncertain self-generated targets to be dropped.","",
       f"Seed sample F1 **{seed_metrics['sample_f1']:.6f}**, macro F1 **{seed_metrics['macro_f1']:.6f}**. "
       f"Best self-training setting **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "| Confidence | Added weight cap | Round | Self-labeled trials | Self-labeled reason slots | Sample F1 | Macro F1 | Change |",
       "|---:|---:|---:|---:|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    lines.append(f"| {row.confidence:g} | {row.weight_cap:g} | {row.round} | {row.self_labeled_trials} | {row.self_labeled_reason_slots} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_seed:+.6f} |")
lines.extend(["","Classifier/scaler, 150 embeddings, C=1000, fitting-weight total 157, cutoff 0.5, and fallback stay fixed. "
       "Validation and test do not participate in training. No final refit with validation labels was performed. "
       "These are exploratory selections on the same 78 validation trials; no independent test improvement is established.","",
       "Exact datasets/weights, source audits distinguishing original/DBSCAN/self-training targets, teacher probabilities, "
       "classifiers, validation predictions, per-reason acceptance counts, settings, and checksums are saved. "
       "The seed reproduces the current leader. Every saved model rescored correctly and preserves all seed targets/weights; "
       "the selected model independently refits from its saved data with matching probabilities. Protected sources retain their hashes.","",
       "No Kaggle submission, commit, or tag was made.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No self-training setting improves the current leader; keep the seed model.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Best self-training:",best.experiment,"score:",best.sample_f1,"Leader changed:",settings['leader_changed'])

```

## legacy_feature_model.ipynb

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.mixture import GaussianMixture
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog('rdApp.*')
SEED = 0
DATA = '../part1/'
LABELS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']
EMB = [f'emb_{i}' for i in range(150)]

train = pd.read_csv(DATA + 'part1_train.csv')
val = pd.read_csv(DATA + 'part1_val.csv')
test = pd.read_csv(DATA + 'part1_test.csv')
is_lab = (train[LABELS] != -100).all(axis=1).values

print(f'train {train.shape}, labeled {is_lab.sum()}, unlabeled {(~is_lab).sum()} | val {val.shape} | test {test.shape}')
pd.DataFrame({'labeled train': train.loc[is_lab, LABELS].mean(), 'val': val[LABELS].mean()}).T.round(3)
```

```python
n_tr, n_va, n_te = len(train), len(val), len(test)
emb_all = np.vstack([train[EMB].values, val[EMB].values, test[EMB].values])
X_all = StandardScaler().fit_transform(emb_all)
X_tr, X_va, X_te = X_all[:n_tr], X_all[n_tr:n_tr + n_va], X_all[n_tr + n_va:]
Y_tr, Y_va = train[LABELS].values, val[LABELS].values

pca30 = PCA(30, random_state=SEED).fit(X_all)
gmm = GaussianMixture(30, covariance_type='diag', n_init=3, random_state=SEED).fit(pca30.transform(X_all))
print('GMM converged:', gmm.converged_, '| iterations:', gmm.n_iter_, '| tol:', gmm.tol)
```

```python
def decide(P, thr=0.3):
    Y = (P >= thr).astype(int)
    Y[np.arange(len(P)), P.argmax(1)] = 1
    return Y


def expand(X, Y, w):
    rows, ys, ws = [], [], []
    for i, y in enumerate(Y):
        pos = np.flatnonzero(y)
        rows += [i] * len(pos); ys += list(pos); ws += [w[i] / len(pos)] * len(pos)
    return X[rows], np.array(ys), np.array(ws)


def multinomial_lr(Xl, Yl, Xq, C=0.3, balanced=True):
    X, y, sw = expand(Xl, Yl, np.ones(len(Yl)))
    if balanced:
        cnt = np.bincount(y, weights=sw, minlength=5) + 1e-9
        sw = sw * cnt.mean() / cnt[y]
    m = LogisticRegression(C=C, max_iter=3000).fit(X, y, sample_weight=sw)
    P = np.zeros((len(Xq), 5))
    P[:, m.classes_] = m.predict_proba(Xq)
    return P


def knn(Xl, Yl, Xq, k=10):
    S = normalize(Xq) @ normalize(Xl).T
    idx = np.argsort(-S, 1)[:, :k]
    w = np.take_along_axis(S, idx, 1).clip(0) ** 2
    T = Yl / Yl.sum(1, keepdims=True)
    return (w[:, :, None] * T[idx]).sum(1) / (w.sum(1, keepdims=True) + 1e-9)


def gmm_cluster_label(Xl, Yl, Xq, a=1.0):
    Rl, Rq = (gmm.predict_proba(pca30.transform(M[:, :len(EMB)])) for M in (Xl, Xq))
    T = Yl / Yl.sum(1, keepdims=True)
    cluster_dist = (Rl.T @ T + a * T.mean(0)) / (Rl.sum(0)[:, None] + a)
    return Rq @ cluster_dist


def base_model(Xl, Yl, Xq, w_knn=0.3):
    return (1 - w_knn) * multinomial_lr(Xl, Yl, Xq) + w_knn * knn(Xl, Yl, Xq)


def final_model(Xl, Yl, Xq, Xu, w_gmm=0.2, conf=0.8, thr=0.3, rounds=2, return_aug=False):
    XL, YL = Xl, Yl
    for _ in range(rounds):
        Pu = (1 - w_gmm) * base_model(XL, YL, Xu) + w_gmm * gmm_cluster_label(XL, YL, Xu)
        keep = Pu.max(1) >= conf
        XL, YL = np.vstack([Xl, Xu[keep]]), np.vstack([Yl, decide(Pu, thr)[keep]])
    P = base_model(XL, YL, Xq)
    return (P, keep.sum()) if return_aug else P
```

```python
X_pool, Y_pool = np.vstack([X_tr[is_lab], X_va]), np.vstack([Y_tr[is_lab], Y_va])
X_unl = np.vstack([X_tr[~is_lab], X_te])


def metrics(Y, Yh):
    per = f1_score(Y, Yh, average=None, zero_division=0)
    return {'sample_F1': f1_score(Y, Yh, average='samples', zero_division=0), 'macro_F1': per.mean(),
            **{f'F1_{l}': v for l, v in zip(LABELS, per)}}


def repeated_cv(fn, X=X_pool, Xu=X_unl, reps=8, thr=0.3):
    scores = []
    for r in range(reps):
        for a, b in KFold(5, shuffle=True, random_state=100 + r).split(X):
            scores.append(metrics(Y_pool[b], decide(fn(X[a], Y_pool[a], X[b], Xu), thr)))
    return pd.DataFrame(scores).mean()


def evaluate(name, fn, Xl=X_tr[is_lab], Xq=X_va, X=X_pool, Xu=X_unl):
    v = metrics(Y_va, decide(fn(Xl, Y_tr[is_lab], Xq, Xu)))
    c = repeated_cv(fn, X, Xu)
    return {'model': name, 'val_sample_F1': v['sample_F1'], 'val_macro_F1': v['macro_F1'],
            'cv_sample_F1': c['sample_F1'], 'cv_macro_F1': c['macro_F1']}
```

```python
def ovr_lr(Xl, Yl, Xq, C=1.0):
    return np.column_stack([LogisticRegression(C=C, max_iter=2000).fit(Xl, Yl[:, j]).predict_proba(Xq)[:, 1]
                            for j in range(5)])


candidates = {
    'One-vs-rest LR (C=1)': lambda a, b, c, u: ovr_lr(a, b, c),
    'kNN cosine (k=10)': lambda a, b, c, u: knn(a, b, c),
    'GMM-30 cluster-then-label': lambda a, b, c, u: gmm_cluster_label(a, b, c),
    'Multinomial LR, unbalanced': lambda a, b, c, u: multinomial_lr(a, b, c, balanced=False),
    'Balanced multinomial LR + kNN': lambda a, b, c, u: base_model(a, b, c),
    'Self-training, no GMM': lambda a, b, c, u: final_model(a, b, c, u, w_gmm=0.0),
    'FINAL: GMM-assisted self-training': final_model,
}
comparison = pd.DataFrame([evaluate(n, f) for n, f in candidates.items()]).set_index('model')
comparison.round(3)
```

```python
def svd_block(M, k):
    return StandardScaler().fit_transform(TruncatedSVD(k, random_state=SEED).fit_transform(M))


def text_col(col):
    return pd.concat([train[col], val[col], test[col]]).fillna('').astype(str).values


fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def trial_fingerprint(smiles):
    if not isinstance(smiles, str):
        return np.zeros(1024)
    fps = [fp_gen.GetFingerprintAsNumPy(m) for s in smiles.split(';') if (m := Chem.MolFromSmiles(s)) is not None]
    return np.max(fps, axis=0) if fps else np.zeros(1024)


year = pd.concat([train.start_year, val.start_year, test.start_year]).values.astype(float)
extra = {
    '+ start_year': ((year - year.mean()) / year.std())[:, None],
    '+ criteria TF-IDF (SVD 50)': svd_block(TfidfVectorizer(min_df=3, ngram_range=(1, 2), sublinear_tf=True,
                                                           max_features=20000).fit_transform(text_col('criteria')), 50),
    '+ diseases TF-IDF (SVD 20)': svd_block(TfidfVectorizer(min_df=2).fit_transform(text_col('diseases_cleaned')), 20),
    '+ Morgan fingerprints (SVD 20)': svd_block(np.vstack([trial_fingerprint(s) for s in text_col('drug_smiles')
                                                           .astype(object)]).astype(float), 20),
}


def split_feats(F):
    return F[:n_tr], F[n_tr:n_tr + n_va], F[n_tr + n_va:]


rows = [evaluate('embeddings only', final_model)]
for name, F in extra.items():
    a, b, c = split_feats(np.hstack([X_all, F]))
    rows.append(evaluate(name, final_model, Xl=a[is_lab], Xq=b,
                         X=np.vstack([a[is_lab], b]), Xu=np.vstack([a[~is_lab], c])))
ablation = pd.DataFrame(rows).set_index('model')
ablation.round(3)
```

```python
P_va, n_pseudo = final_model(X_tr[is_lab], Y_tr[is_lab], X_va, X_unl, return_aug=True)
val_metrics = pd.Series(metrics(Y_va, decide(P_va)))
print(f'pseudo-labeled trials added: {n_pseudo} / {len(X_unl)}')
val_metrics.round(3)
```

```python
X_lab_all, Y_lab_all = X_pool, Y_pool
Y_unl_tr = decide(final_model(X_lab_all, Y_lab_all, X_tr[~is_lab], X_unl))
Y_te = decide(final_model(X_lab_all, Y_lab_all, X_te, X_unl))

submission = pd.DataFrame({'ID': test.ID,
                           'Predicted': [' '.join(l for l, y in zip(LABELS, row) if y) for row in Y_te]})
submission.to_csv('part1_best_submission.csv', index=False)

completed = pd.concat([
    train.loc[is_lab, ['ID']].assign(**dict(zip(LABELS, Y_tr[is_lab].T)), split='train', source='true'),
    train.loc[~is_lab, ['ID']].assign(**dict(zip(LABELS, Y_unl_tr.T)), split='train', source='predicted'),
    val[['ID']].assign(**dict(zip(LABELS, Y_va.T)), split='val', source='true'),
    test[['ID']].assign(**dict(zip(LABELS, Y_te.T)), split='test', source='predicted'),
], ignore_index=True)
completed.to_csv('part1_best_reason_labels.csv', index=False)

print(submission.head())
print(f'\nwrote {len(submission)} test predictions and {len(completed)} reason-label rows')
```

```python
freq = pd.DataFrame({
    'labeled (train+val)': Y_lab_all.mean(0),
    'imputed train (unlabeled)': Y_unl_tr.mean(0),
    'predicted test (2010-2016)': Y_te.mean(0),
}, index=LABELS)
ax = freq.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel('fraction of trials')
ax.set_title('Reason-label frequency by split')
plt.tight_layout()
plt.savefig('part1_label_frequency.png', dpi=150)
freq.round(3)
```